In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 1 — ENVIRONMENT, PATHS, SHARED UTILITIES, SESSION CLOCK
# ═══════════════════════════════════════════════════════════════════════════
import os, sys

_QUIET = {
    "PYTHONWARNINGS": "ignore", "TF_CPP_MIN_LOG_LEVEL": "3",
    "TORCH_CPP_LOG_LEVEL": "ERROR", "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_HUB_DISABLE_TELEMETRY": "1",
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1", "HF_HUB_VERBOSITY": "error",
    "HF_HUB_DISABLE_SYMLINKS_WARNING": "1", "TQDM_DISABLE": "1",
    "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TRANSFORMERS_VERBOSITY": "error", "NCCL_DEBUG": "ERROR",
}
os.environ.update(_QUIET)

import gc, json, time, math, random, shutil, logging, warnings, subprocess
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")
for _n in ("py.warnings", "huggingface_hub", "timm", "PIL", "matplotlib",
           "urllib3", "filelock", "torch", "torchvision", "fsspec", "datasets"):
    logging.getLogger(_n).setLevel(logging.ERROR)

import numpy as np

PROJECT, SEED = "OralNet", 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
import torch
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

BASE_IN, WORK = Path("/kaggle/input"), Path("/kaggle/working")
D_STATE, D_CKPT = WORK / "state", WORK / "ckpt"
D_W, D_PRED, D_LOG = WORK / "weights", WORK / "preds", WORK / "logs"
D_EVAL, D_FIG, D_FINAL = WORK / "evaluation", WORK / "figures", WORK / "final"
D_CLAIM, D_DONE, D_FAIL = D_STATE / "claims", D_STATE / "done", D_STATE / "failed"
for d in (D_STATE, D_CKPT, D_W, D_PRED, D_LOG, D_EVAL, D_FIG, D_FINAL,
          D_CLAIM, D_DONE, D_FAIL):
    d.mkdir(parents=True, exist_ok=True)

F_INDEX   = D_STATE / "index.json"
F_MAN     = D_STATE / "manifest.json"
F_CACHE   = D_STATE / "cache.json"
F_CONFIG  = D_STATE / "config.json"
F_LEDGER  = D_STATE / "ledger.json"
F_SESSION = D_STATE / "session.json"
COMMON    = WORK / "onet_common.py"

DATASETS = ["he_stained", "histopath", "ndb_ufes", "clinical"]
SESSION_BUDGET_MIN = 645          # stop cleanly ~10h45m into an 11h30m session

_SRC = r'''
"""Shared utilities for OralNet. Imported by notebook blocks and the worker."""
import os, sys, json, time, shutil, logging, warnings
from pathlib import Path

W = 100
WORK = Path("/kaggle/working")
D_STATE, D_CKPT, D_LOG = WORK / "state", WORK / "ckpt", WORK / "logs"
D_DONE, D_CLAIM = D_STATE / "done", D_STATE / "claims"


def quiet():
    warnings.filterwarnings("ignore")
    os.environ.setdefault("PYTHONWARNINGS", "ignore")
    for n in ("py.warnings", "huggingface_hub", "timm", "PIL", "matplotlib",
              "urllib3", "filelock", "torch", "torchvision", "fsspec"):
        logging.getLogger(n).setLevel(logging.ERROR)


def hdr(t, ch="=", w=W):
    print("\n" + ch * w); print(t); print(ch * w)


def sub(t, ch="-", w=W):
    print("\n" + ch * w); print(t); print(ch * w)


def rule(ch="-", w=W):
    print(ch * w)


def kv(k, v, pad=26):
    print(f"  {k:<{pad}} {v}")


def row(cells, widths, aligns=None):
    aligns = aligns or ["<"] * len(cells)
    return "".join(f"{str(c):{a}{w}}" for c, w, a in zip(cells, widths, aligns))


def hms(s):
    s = int(max(0, s)); h, m, x = s // 3600, (s % 3600) // 60, s % 60
    return f"{h}h{m:02d}m" if h else (f"{m}m{x:02d}s" if m else f"{x}s")


def gb(n):
    return f"{n / 1e9:.2f} GB"


def write_text(p, txt):
    p = Path(p); t = p.with_suffix(p.suffix + ".tmp")
    t.write_text(txt); os.replace(t, p)


def save_ckpt(obj, p):
    """Atomic write with one-generation rollback."""
    import torch
    p = Path(p); t = p.with_suffix(p.suffix + ".tmp"); prev = p.with_suffix(p.suffix + ".prev")
    torch.save(obj, t)
    if p.exists():
        try:
            os.replace(p, prev)
        except Exception:
            pass
    os.replace(t, p)


def load_ckpt(p):
    """Try current, then the rollback. Returns (obj, which) or (None, None)."""
    import torch
    p = Path(p)
    for c, tag in ((p, "latest"), (p.with_suffix(p.suffix + ".prev"), "rollback")):
        if not c.exists() or c.stat().st_size < 4096:
            continue
        try:
            return torch.load(c, map_location="cpu", weights_only=False), tag
        except Exception:
            continue
    return None, None


class Clock:
    def __init__(self, budget_min, t0=None):
        self.budget = budget_min * 60.0
        self.t0 = float(t0) if t0 else time.time()

    @property
    def elapsed(self):
        return time.time() - self.t0

    @property
    def left(self):
        return self.budget - self.elapsed

    def fits(self, sec, margin=1.25):
        return self.left > sec * margin

    def bar(self):
        f = min(1.0, self.elapsed / max(1.0, self.budget)); n = int(f * 30)
        return f"[{'#'*n}{'.'*(30-n)}] {hms(self.elapsed)}/{hms(self.budget)}"


def cleanup(clear_claims=False, log_mb=8, verbose=True):
    """Remove temp files, checkpoints of finished jobs, oversized logs.
    Never touches preds/, weights/, evaluation/, or a live job's checkpoint."""
    tmp = ck = cl = lg = 0; freed = 0
    for p in list(D_CKPT.glob("*.tmp")) + list(D_STATE.glob("*.tmp")):
        freed += p.stat().st_size; p.unlink(missing_ok=True); tmp += 1
    done = {p.stem for p in D_DONE.glob("*.json")}
    for p in D_CKPT.glob("*.resume.pt*"):
        if p.name.split(".resume.pt")[0] in done:
            freed += p.stat().st_size; p.unlink(missing_ok=True); ck += 1
    if clear_claims:
        for c in D_CLAIM.glob("*.claim"):
            if c.stem not in done:
                c.unlink(missing_ok=True); cl += 1
    cap = log_mb * 1024 * 1024
    for f in D_LOG.glob("*.log"):
        if f.stat().st_size > cap:
            d = f.read_bytes()[-cap // 2:]; i = d.find(b"\n")
            f.write_bytes(b"[truncated]\n" + (d[i+1:] if i >= 0 else d)); lg += 1
    if verbose:
        kv("cleanup", f"tmp={tmp} finished-ckpt={ck} claims={cl} logs={lg} "
                      f"freed={freed/1e6:.0f} MB")
    return {"tmp": tmp, "ckpt": ck, "claims": cl, "logs": lg}


def disk_free(p="/kaggle/working"):
    s = os.statvfs(p); return s.f_bavail * s.f_frsize
'''
COMMON.write_text(_SRC)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
sys.modules.pop("onet_common", None)
import onet_common as U
U.quiet()

log = logging.getLogger("onet"); log.handlers.clear(); log.setLevel(logging.INFO)
_fh = logging.FileHandler(D_LOG / "run.log")
_fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-5s | %(message)s", "%H:%M:%S"))
log.addHandler(_fh); log.propagate = False


def ledger_load():
    if F_LEDGER.exists():
        try:
            return json.loads(F_LEDGER.read_text())
        except Exception:
            pass
    return {"created": datetime.now(timezone.utc).isoformat(), "jobs": {}, "sessions": []}


def ledger_save(l):
    U.write_text(F_LEDGER, json.dumps(l, indent=2))


LED = ledger_load()
SESSION = len(LED["sessions"]) + 1
LED["sessions"].append({"n": SESSION, "start": datetime.now(timezone.utc).isoformat()})
ledger_save(LED)
CLOCK = U.Clock(SESSION_BUDGET_MIN)
U.write_text(F_SESSION, json.dumps({"session": SESSION, "t0": CLOCK.t0,
                                    "budget_min": SESSION_BUDGET_MIN}))

U.hdr(f"{PROJECT}  ·  SESSION {SESSION}  ·  {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC")
U.kv("torch / cuda / cudnn", f"{torch.__version__} / {torch.version.cuda} / "
                             f"{torch.backends.cudnn.version()}")
NGPU = torch.cuda.device_count()
for i in range(NGPU):
    p = torch.cuda.get_device_properties(i)
    U.kv(f"GPU {i}", f"{p.name}  {U.gb(p.total_memory)}  sm_{p.major}{p.minor}  "
                     f"{p.multi_processor_count} SMs")
U.kv("precision", "fp16 + GradScaler" + (" (sm_75: no native bf16)"
                                         if NGPU and torch.cuda.get_device_properties(0).major < 8 else ""))
try:
    import psutil
    vm = psutil.virtual_memory()
    U.kv("cpu / ram", f"{psutil.cpu_count(logical=True)} cores · {U.gb(vm.total)} "
                      f"({U.gb(vm.available)} free)")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil"],
                   check=False, capture_output=True)
for mnt, lbl in (("/kaggle/working", "disk working"), ("/kaggle/temp", "disk temp"),
                 ("/", "disk root")):
    if Path(mnt).exists():
        s = os.statvfs(mnt)
        U.kv(lbl, f"{U.gb(s.f_bavail*s.f_frsize)} free of {U.gb(s.f_blocks*s.f_frsize)}")
U.kv("session budget", f"{SESSION_BUDGET_MIN} min   {CLOCK.bar()}")

U.sub("PIPELINE STATE")
stages = [("index.json", F_INDEX, "2  dataset audit"),
          ("manifest.json", F_MAN, "3  protocol splits"),
          ("cache.json", F_CACHE, "4  image cache"),
          ("config.json", F_CONFIG, "5  training plan")]
print("  " + U.row(["artifact", "status", "size", "from block"], [20, 10, 11, 24]))
U.rule()
for nm, pth, who in stages:
    ok = pth.exists()
    print("  " + U.row([nm, "READY" if ok else "—",
                        f"{pth.stat().st_size/1e6:.1f} MB" if ok else "—", who],
                       [20, 10, 11, 24]))
n_jobs = len(json.loads(F_CONFIG.read_text()).get("jobs", [])) if F_CONFIG.exists() else 0
n_done = len(list(D_DONE.glob("*.json")))
n_res = len({f.name.split(".resume.pt")[0] for f in D_CKPT.glob("*.resume.pt")})
print()
U.kv("jobs registered", n_jobs)
U.kv("jobs complete", f"{n_done}  ({len(list(D_PRED.glob('*.npz')))} prediction files)")
U.kv("jobs resumable", n_res)

U.sub("HOUSEKEEPING")
U.cleanup(clear_claims=True)
U.kv("disk free", U.gb(U.disk_free()))
print(f"\n  PERSISTENCE: Settings → Persistence → 'Files only' must be ON,")
print(f"  otherwise /kaggle/working is wiped when this session ends.")
print(f"\n[BLOCK 1 ✓]  session {SESSION} · {NGPU} GPU · {n_done}/{n_jobs} done · {n_res} resumable")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 2 — DATASET DISCOVERY & FORENSIC AUDIT
# ═══════════════════════════════════════════════════════════════════════════
import os, re, json, hashlib, warnings
from pathlib import Path
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import onet_common as U
U.quiet()

EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
REBUILD = True


def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    try:
        with open(p, "rb") as f:
            while True:
                b = f.read(chunk)
                if not b:
                    break
                h.update(b)
        return h.hexdigest()
    except Exception:
        return None


def meta(p):
    try:
        with Image.open(p) as im:
            return im.size[0], im.size[1], im.mode
    except Exception:
        return None, None, None


def dirs_named(root, name, depth=9):
    out = []
    for dp, dn, _ in os.walk(root):
        try:
            d = len(Path(dp).relative_to(root).parts)
        except ValueError:
            continue
        if d > depth:
            dn[:] = []; continue
        if Path(dp).name.lower() == name:
            out.append(Path(dp))
    return out


def dir_with(root, req, depth=9):
    req = {r.lower() for r in req}; best, bd = None, 10**9
    for dp, dn, _ in os.walk(root):
        try:
            d = len(Path(dp).relative_to(root).parts)
        except ValueError:
            continue
        if d > depth:
            dn[:] = []; continue
        if req.issubset({x.lower() for x in dn}) and d < bd:
            best, bd = Path(dp), d
    return best


def split_of(p):
    for x in p.parts:
        x = x.lower()
        if x in ("training", "train"):
            return "train"
        if x in ("testing", "test"):
            return "test"
        if x in ("validation", "valid", "val"):
            return "val"
    return None


U.hdr("BLOCK 2 — DATASET DISCOVERY & FORENSIC AUDIT")
U.sub("ROOT RESOLUTION")
p640 = dirs_named(BASE_IN, "640x640")
he_roots = sorted({p.parents[3] for p in p640 if len(p.parents) >= 4})
ROOTS = {"he_stained": he_roots[0] if he_roots else None,
         "histopath": dir_with(BASE_IN, ["train", "val", "test"]),
         "ndb_ufes": dir_with(BASE_IN, ["Healthy", "Leukoplakia",
                                        "Oral Squamous Cell Carcinoma"]),
         "clinical": dir_with(BASE_IN, ["cancer", "non-cancer"])}
for k, v in ROOTS.items():
    U.kv(k, str(v) if v else "NOT FOUND")


def parse_he(root):
    r = []
    if root is None:
        return r
    for pd in dirs_named(root, "640x640"):
        nat = split_of(pd)
        if nat is None:
            continue
        for slide in sorted(pd.iterdir()):
            if not slide.is_dir():
                continue
            for roi in sorted(slide.iterdir()):
                nm = roi.name.lower()
                cls = ("tumor_patch" if "01-roi" in nm else
                       "non_tumor_patch" if "02-non_roi" in nm else None)
                if cls is None:
                    continue                       # 03-roi overlays excluded
                orig, msk = roi / "01-original", roi / "02-mask"
                if not orig.is_dir():
                    continue
                ml = {m.stem: str(m) for m in msk.iterdir()
                      if m.suffix.lower() in EXT} if msk.is_dir() else {}
                for f in sorted(orig.iterdir()):
                    if f.suffix.lower() in EXT:
                        r.append({"dataset": "he_stained", "path": str(f),
                                  "class_name": cls, "native_split": nat,
                                  "group_id": slide.name, "mask_path": ml.get(f.stem),
                                  "is_aug": False, "mag": None, "stem": f.stem})
    return r


def parse_hp(root):
    r = []
    if root is None:
        return r
    rx = re.compile(r"_(\d+)x_", re.I)
    al = {"train": "train", "training": "train", "val": "val", "valid": "val",
          "validation": "val", "test": "test", "testing": "test"}
    for sd in sorted(root.iterdir()):
        if not sd.is_dir() or sd.name.lower() not in al:
            continue
        nat = al[sd.name.lower()]
        for cd in sorted(sd.iterdir()):
            if not cd.is_dir():
                continue
            for f in sorted(cd.rglob("*")):
                if f.suffix.lower() in EXT:
                    m = rx.search(f.name)
                    r.append({"dataset": "histopath", "path": str(f),
                              "class_name": cd.name, "native_split": nat,
                              "group_id": None, "mask_path": None,
                              "is_aug": f.name.lower().startswith("aug_"),
                              "mag": (m.group(1) + "x") if m else "unknown",
                              "stem": f.stem})
    return r


def parse_flat(root, ds):
    r = []
    if root is None:
        return r
    for cd in sorted(root.iterdir()):
        if not cd.is_dir():
            continue
        for f in sorted(cd.rglob("*")):
            if f.suffix.lower() in EXT:
                r.append({"dataset": ds, "path": str(f), "class_name": cd.name,
                          "native_split": None, "group_id": None, "mask_path": None,
                          "is_aug": False, "mag": None, "stem": f.stem})
    return r


if F_INDEX.exists() and not REBUILD:
    REC = json.loads(F_INDEX.read_text())
    U.kv("index", f"reused ({len(REC)} records)")
else:
    REC = (parse_he(ROOTS["he_stained"]) + parse_hp(ROOTS["histopath"])
           + parse_flat(ROOTS["ndb_ufes"], "ndb_ufes")
           + parse_flat(ROOTS["clinical"], "clinical"))
    print(f"\n  hashing {len(REC)} files ...")
    for i, r in enumerate(REC):
        r["md5"] = md5(r["path"])
        w, h, m = meta(r["path"])
        r["w"], r["h"], r["mode"] = w, h, m
        r["bytes"] = os.path.getsize(r["path"]) if os.path.exists(r["path"]) else 0
        if (i + 1) % 2500 == 0:
            print(f"    {i+1}/{len(REC)}")
    U.write_text(F_INDEX, json.dumps(REC))
    U.kv("index written", f"{F_INDEX.stat().st_size/1e6:.1f} MB")

by = defaultdict(list)
for r in REC:
    by[r["dataset"]].append(r)

for ds in DATASETS:
    rs = by.get(ds, [])
    if not rs:
        print(f"\n  [{ds}] NO FILES"); continue
    real = [r for r in rs if not r["is_aug"]]
    aug = len(rs) - len(real)
    U.sub(f"{ds.upper()}   {len(rs)} files · {len(real)} real · {aug} pre-augmented")
    tab = defaultdict(lambda: defaultdict(int))
    for r in real:
        tab[r["class_name"]][r["native_split"] or "—"] += 1
    for c in sorted(tab):
        print(f"    {c:<34} " + "  ".join(f"{k}={v}" for k, v in sorted(tab[c].items())))
    cnt = Counter(r["class_name"] for r in real)
    U.kv("majority baseline", f"{max(cnt.values())/len(real)*100:.2f}%")
    h = Counter(r["md5"] for r in real if r["md5"])
    dup = {k: v for k, v in h.items() if v > 1}
    extra = sum(v - 1 for v in dup.values())
    U.kv("exact duplicates", f"{len(dup)} groups / {extra} redundant "
                             f"({100*extra/max(1,len(real)):.1f}%)")
    if any(r["native_split"] for r in real):
        cross = defaultdict(set)
        for r in real:
            if r["md5"]:
                cross[r["md5"]].add(r["native_split"])
        nx = sum(1 for v in cross.values() if len(v) > 1)
        U.kv("dupes across shipped split", f"{nx}"
             + ("  ← shipped split leaks" if nx else "  (clean)"))
    dims = Counter((r["w"], r["h"]) for r in real if r["w"])
    U.kv("resolutions", f"{len(dims)} distinct · top {dims.most_common(3)}")
    U.kv("on disk", f"{sum(r['bytes'] for r in real)/1e6:.0f} MB")
    mg = [r["mag"] for r in real if r.get("mag")]
    if mg:
        mc = defaultdict(lambda: defaultdict(int))
        for r in real:
            if r.get("mag"):
                mc[r["mag"]][r["class_name"]] += 1
        for m in sorted(mc):
            U.kv(f"  mag {m}", dict(mc[m]))
    gid = [r["group_id"] for r in real if r["group_id"]]
    if gid:
        gc_ = Counter(gid)
        gs = defaultdict(set)
        for r in real:
            if r["group_id"]:
                gs[r["group_id"]].add(r["native_split"])
        shared = sorted(g for g, s in gs.items() if len(s) > 1)
        U.kv("slides", f"{len(gc_)} · {min(gc_.values())}–{max(gc_.values())} patches each")
        U.kv("slides in >1 split", f"{len(shared)}/{len(gc_)}"
             + ("  ← LOSO required" if shared else ""))
        U.kv("paired masks", f"{sum(1 for r in real if r['mask_path'])}/{len(real)}")

U.sub("CROSS-DATASET OVERLAP")
gm = defaultdict(set)
for r in REC:
    if r["md5"] and not r["is_aug"]:
        gm[r["md5"]].add(r["dataset"])
ov = [v for v in gm.values() if len(v) > 1]
U.kv("images in >1 dataset", len(ov))
print(f"\n[BLOCK 2 ✓]  {len(REC)} indexed · "
      f"{sum(1 for r in REC if not r['is_aug'])} real")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 3 — PROTOCOL MANIFEST  (A = replication · B = rigorous)
# ═══════════════════════════════════════════════════════════════════════════
import json, warnings
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")
import numpy as np
import onet_common as U
U.quiet()

REC = json.loads(F_INDEX.read_text())
COARSE = {"benign": 0, "premalignant": 1, "malignant": 2}


def coarse(c):
    c = c.lower().replace(" ", "_").replace("-", "_")
    if "leukoplakia" in c:
        return 1
    if c.startswith("non") or "normal" in c or "healthy" in c:
        return 0
    if any(k in c for k in ("oscc", "carcinoma", "cancer", "tumor", "malignant")):
        return 2
    return -1


def dedup(rs):
    seen, out, n = set(), [], 0
    for r in rs:
        k = r.get("md5")
        if k and k in seen:
            n += 1; continue
        if k:
            seen.add(k)
        out.append(r)
    return out, n


def strat(rs, k, seed, key=lambda e: e["class_name"]):
    rng = np.random.RandomState(seed)
    out = np.zeros(len(rs), dtype=int); b = defaultdict(list)
    for i, r in enumerate(rs):
        b[key(r)].append(i)
    for kk in sorted(b, key=str):
        idx = np.array(b[kk]); rng.shuffle(idx)
        for j, i in enumerate(idx):
            out[i] = j % k
    return out


def loso(rs):
    g = sorted({r["group_id"] for r in rs if r["group_id"]})
    gi = {x: i for i, x in enumerate(g)}
    return np.array([gi[r["group_id"]] for r in rs]), len(g)


def pack(rs, folds, c2i, proto, rep=0):
    return [{"path": r["path"], "mask_path": r.get("mask_path"),
             "class_name": r["class_name"], "y": int(c2i[r["class_name"]]),
             "coarse": coarse(r["class_name"]), "fold": int(f), "repeat": rep,
             "native_split": r.get("native_split"), "group_id": r.get("group_id"),
             "mag": r.get("mag"), "is_aug": bool(r.get("is_aug")), "protocol": proto}
            for r, f in zip(rs, folds)]


U.hdr("BLOCK 3 — PROTOCOL MANIFEST")
MAN = {}
for ds in DATASETS:
    allr = [r for r in REC if r["dataset"] == ds]
    if not allr:
        continue
    classes = sorted({r["class_name"] for r in allr})
    c2i = {c: i for i, c in enumerate(classes)}
    P = {}

    if ds == "he_stained":
        tr = [r for r in allr if r["native_split"] == "train"]
        te = [r for r in allr if r["native_split"] == "test"]
        P["A"] = {"note": "OCDC shipped train/test convention; 5-fold CV inside train. "
                          "Test slides overlap train slides — the published setting.",
                  "n_folds": 5, "n_repeats": 1, "group_disjoint": False,
                  "cv": pack(tr, strat(tr, 5, SEED), c2i, "A"),
                  "test": pack(te, [-1]*len(te), c2i, "A")}
        both, nd = dedup(tr + te)
        lf, ng = loso(both)
        P["B"] = {"note": f"Leave-one-slide-out over all {len(both)} patches ({ng} slides). "
                          f"Fully group-disjoint; every patch appears once out-of-fold.",
                  "n_folds": ng, "n_repeats": 1, "group_disjoint": True, "n_dup": nd,
                  "cv": pack(both, lf, c2i, "B"), "test": []}

    elif ds == "histopath":
        f = strat(allr, 5, SEED, key=lambda e: (e["class_name"], e.get("mag") or "?"))
        P["A"] = {"note": "Replication: full pre-augmented corpus, random stratified "
                          "5-fold, magnification-balanced (Manoj Kumar 2025 setting).",
                  "n_folds": 5, "n_repeats": 1, "group_disjoint": False,
                  "cv": pack(allr, f, c2i, "A"), "test": []}
        real, nd = dedup([r for r in allr if not r["is_aug"]])
        cv = [r for r in real if r["native_split"] in ("train", "val")]
        te = [r for r in real if r["native_split"] == "test"]
        P["B"] = {"note": f"Rigorous: deduplicated ({nd} removed), zero augmented images, "
                          f"dataset authors' native split honoured (CV on train+val, "
                          f"held-out native test).",
                  "n_folds": 5, "n_repeats": 1, "group_disjoint": False, "n_dup": nd,
                  "cv": pack(cv, strat(cv, 5, SEED,
                                       key=lambda e: (e["class_name"], e.get("mag") or "?")),
                             c2i, "B"),
                  "test": pack(te, [-1]*len(te), c2i, "B")}

    elif ds == "ndb_ufes":
        real, nd = dedup(allr)
        P["B"] = {"note": f"Rigorous: deduplicated ({nd} removed), stratified 5-fold. "
                          f"No patient key recoverable from filenames — stated as a "
                          f"limitation.",
                  "n_folds": 5, "n_repeats": 1, "group_disjoint": False, "n_dup": nd,
                  "cv": pack(real, strat(real, 5, SEED), c2i, "B"), "test": []}

    else:  # clinical — 4 repeats for CI on a small corpus
        R = 4
        ent = []
        for rep in range(R):
            ent += pack(allr, strat(allr, 5, SEED + 1000*rep), c2i, "A", rep)
        P["A"] = {"note": f"Replication: raw 131-image corpus INCLUDING its duplicates, "
                          f"stratified 5-fold × {R} repeats ({5*R} models).",
                  "n_folds": 5, "n_repeats": R, "group_disjoint": False,
                  "cv": ent, "test": []}
        real, nd = dedup(allr)
        ent = []
        for rep in range(R):
            ent += pack(real, strat(real, 5, SEED + 1000*rep), c2i, "B", rep)
        P["B"] = {"note": f"Rigorous: {nd} exact duplicates removed "
                          f"({100*nd/len(allr):.1f}% of corpus), 5-fold × {R} repeats.",
                  "n_folds": 5, "n_repeats": R, "group_disjoint": False, "n_dup": nd,
                  "cv": ent, "test": []}

    MAN[ds] = {"classes": classes, "class_to_idx": c2i, "num_classes": len(classes),
               "positive_class": {"he_stained": "tumor_patch", "histopath": "OSCC",
                                  "clinical": "cancer"}.get(ds),
               "protocols": P}

U.write_text(F_MAN, json.dumps(MAN))

ok = True
for ds in DATASETS:
    if ds not in MAN:
        continue
    m = MAN[ds]
    U.sub(f"{ds.upper()}   K={m['num_classes']}   {m['classes']}")
    for pn in ("A", "B"):
        p = m["protocols"].get(pn)
        if not p:
            continue
        cv, te = p["cv"], p.get("test", [])
        nf, nr = p["n_folds"], p["n_repeats"]
        print(f"\n  Protocol {pn}  ·  {len(cv)} rows ({len(cv)//max(1,nr)} imgs × {nr} rep)"
              f"  ·  test {len(te)}  ·  {nf} folds  ·  {nf*nr} models")
        print(f"    {p['note']}")
        r0 = [e for e in cv if e["repeat"] == 0]
        c = defaultdict(lambda: defaultdict(int))
        for e in r0:
            c[e["class_name"]][e["fold"]] += 1
        show = min(nf, 7)
        print("    " + U.row(["class"], [30]) + "".join(f" f{i:<5d}" for i in range(show))
              + ("  …" if nf > show else "") + "   total")
        for cl in m["classes"]:
            rr = [c[cl].get(i, 0) for i in range(nf)]
            print("    " + U.row([cl], [30]) + "".join(f" {v:<6d}" for v in rr[:show])
                  + ("  …" if nf > show else "") + f"   {sum(rr)}")
        cc = Counter(e["class_name"] for e in r0)
        print(f"    majority baseline {max(cc.values())/len(r0)*100:.2f}%"
              + (f"  ·  test n={len(te)} majority "
                 f"{max(Counter(e['class_name'] for e in te).values())/len(te)*100:.2f}%"
                 if te else ""))
        if p.get("group_disjoint"):
            gf = defaultdict(set)
            for e in r0:
                if e["group_id"]:
                    gf[e["group_id"]].add(e["fold"])
            bad = sum(1 for v in gf.values() if len(v) > 1)
            print(f"    groups spanning folds: {bad}  "
                  f"{'FAIL' if bad else '✓ group-disjoint verified'}")
            ok &= bad == 0
        um = sum(1 for e in cv if e["coarse"] == -1)
        if um:
            print(f"    UNMAPPED coarse labels: {um}"); ok = False

print(f"\n[BLOCK 3 ✓]  manifest {'validated' if ok else 'HAS ISSUES'} "
      f"· {F_MAN.stat().st_size/1e6:.1f} MB")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 4 — AUGMENTED CORPORA (protocol A2) + PRE-DECODED IMAGE CACHE
# ═══════════════════════════════════════════════════════════════════════════
import os, json, time, random, hashlib, warnings
from pathlib import Path
from collections import defaultdict, Counter
from multiprocessing import Pool
warnings.filterwarnings("ignore")
import numpy as np
from PIL import Image, ImageEnhance
import onet_common as U
U.quiet()
Image.MAX_IMAGE_PIXELS = None

TMP = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
CACHE_ROOT, AUG_ROOT = TMP / "onet_cache", TMP / "onet_aug"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
AUG_ROOT.mkdir(parents=True, exist_ok=True)

MAN = json.loads(F_MAN.read_text())
REC = json.loads(F_INDEX.read_text())

# Resolutions chosen from native image size; cache is 1.15× train size so
# RandomResizedCrop still has room.
IMG = {"he_stained": 512, "histopath": 448, "ndb_ufes": 448, "clinical": 448}
CACHE_PX = {k: int(round(v * 1.15 / 32) * 32) for k, v in IMG.items()}

# A2 = augment-before-split, replicating the target papers' data handling.
# Ratios capped at values proven stable in prior runs (2.5×–9×).
A2 = {
    "histopath": {"per_class": 5000, "src": "Abd El-Aziz 2025 — 1,224 originals → 10,000"},
    "ndb_ufes":  {"per_class": 3000, "src": "SciRep 2025 — augmented to class balance"},
    "clinical":  {"per_class": 1200, "src": "Alzahrani 2025 / Liu & Bagi 2025 — OCI-131 augmented"},
}
# ndb_ufes also gets a binary OSCC-vs-rest arm: the NDB-UFES release labels are
# OSCC / leukoplakia-with-dysplasia / leukoplakia-without-dysplasia, and dysplasia
# grading is where inter-pathologist agreement is weakest.
BINARY_ARM = True

U.hdr("BLOCK 4 — AUGMENTED CORPORA + IMAGE CACHE")


def _cache_one(a):
    src, dst, px = a
    try:
        if os.path.exists(dst) and os.path.getsize(dst) > 512:
            return 1
        with Image.open(src) as im:
            im = im.convert("RGB"); w, h = im.size
            s = px / min(w, h)
            if s < 1.0:
                im = im.resize((max(px, int(w*s)), max(px, int(h*s))), Image.LANCZOS)
            im.save(dst, "JPEG", quality=95, subsampling=0)
        return 0
    except Exception:
        return -1


def _aug_one(a):
    src, dst, px, seed = a
    try:
        if os.path.exists(dst) and os.path.getsize(dst) > 512:
            return 1
        rng = random.Random(seed)
        with Image.open(src) as im:
            im = im.convert("RGB")
            k = rng.randint(0, 3)
            if k:
                im = im.rotate(90 * k, expand=True)
            if rng.random() < .5:
                im = im.transpose(Image.FLIP_LEFT_RIGHT)
            if rng.random() < .5:
                im = im.transpose(Image.FLIP_TOP_BOTTOM)
            if rng.random() < .65:
                w, h = im.size; z = rng.uniform(.88, 1.0)
                cw, ch = int(w*z), int(h*z)
                x0 = rng.randint(0, max(0, w-cw)); y0 = rng.randint(0, max(0, h-ch))
                im = im.crop((x0, y0, x0+cw, y0+ch))
            for E, p_ in ((ImageEnhance.Brightness, .65), (ImageEnhance.Contrast, .65),
                          (ImageEnhance.Color, .45)):
                if rng.random() < p_:
                    im = E(im).enhance(rng.uniform(.88, 1.12))
            w, h = im.size; s = px / min(w, h)
            if s < 1.0:
                im = im.resize((max(px, int(w*s)), max(px, int(h*s))), Image.LANCZOS)
            im.save(dst, "JPEG", quality=95, subsampling=0)
        return 0
    except Exception:
        return -1


def strat(rs, k, seed):
    rng = np.random.RandomState(seed)
    out = np.zeros(len(rs), dtype=int); b = defaultdict(list)
    for i, r in enumerate(rs):
        b[r["class_name"]].append(i)
    for c in sorted(b):
        idx = np.array(b[c]); rng.shuffle(idx)
        for j, i in enumerate(idx):
            out[i] = j % k
    return out


# ─── cache real images ──────────────────────────────────────────────────────
U.sub("REAL-IMAGE CACHE")
CMAP, tasks = {}, []
for r in REC:
    ds = r["dataset"]
    if ds not in CACHE_PX:
        continue
    d = CACHE_ROOT / ds; d.mkdir(parents=True, exist_ok=True)
    dst = str(d / f"{hashlib.md5(r['path'].encode()).hexdigest()[:20]}.jpg")
    CMAP[r["path"]] = dst
    tasks.append((r["path"], dst, CACHE_PX[ds]))
t0 = time.time(); made = reused = failed = 0
with Pool(4) as pl:
    for i, rc in enumerate(pl.imap_unordered(_cache_one, tasks, chunksize=64)):
        made += rc == 0; reused += rc == 1; failed += rc < 0
        if (i + 1) % 3000 == 0:
            print(f"    {i+1}/{len(tasks)}  ({time.time()-t0:.0f}s)")
U.kv("cached", f"{made} new · {reused} reused · {failed} failed · {time.time()-t0:.0f}s")

# ─── build A2 corpora ───────────────────────────────────────────────────────
U.sub("PROTOCOL A2 — AUGMENT BEFORE SPLIT")
for ds, spec in A2.items():
    base = [e for e in MAN[ds]["protocols"]["B"]["cv"]
            if e["repeat"] == 0 and not e["is_aug"]]
    byc = defaultdict(list)
    for e in base:
        byc[e["class_name"]].append(e)
    tgt, px = spec["per_class"], CACHE_PX[ds]
    out = AUG_ROOT / ds; out.mkdir(parents=True, exist_ok=True)
    print(f"\n  [{ds}]  {spec['src']}")
    rows, tk = [], []
    for cn in sorted(byc):
        pool = byc[cn]; need = max(0, tgt - len(pool))
        print(f"    {cn:<34} {len(pool):>5} real + {need:>5} aug   "
              f"({tgt/len(pool):.1f}×)")
        for e in pool[:tgt]:
            rows.append({**e, "is_aug": False})
        rng = random.Random(SEED + abs(hash(cn)) % 9999)
        for i in range(need):
            par = pool[i % len(pool)]
            key = hashlib.md5(f"{ds}|{par['path']}|{i}".encode()).hexdigest()[:20]
            dst = str(out / f"{key}.jpg")
            tk.append((CMAP.get(par["path"], par["path"]), dst, px,
                       rng.randint(0, 2**31)))
            rows.append({**par, "path": dst, "is_aug": True})
    t0 = time.time(); m = ru = fl = 0
    with Pool(4) as pl:
        for rc in pl.imap_unordered(_aug_one, tk, chunksize=48):
            m += rc == 0; ru += rc == 1; fl += rc < 0
    print(f"    generated {m} · reused {ru} · failed {fl} · {time.time()-t0:.0f}s")

    f = strat(rows, 5, SEED)
    MAN[ds]["protocols"]["A2"] = {
        "note": f"Augment-before-split replication ({spec['src']}). "
                f"{tgt}/class, class-balanced. Augmented siblings span train/val by "
                f"construction — this is the published protocol, NOT a generalisation "
                f"estimate. Reported beside Protocol B.",
        "n_folds": 5, "n_repeats": 1, "group_disjoint": False, "test": [],
        "cv": [{"path": e["path"], "mask_path": None, "class_name": e["class_name"],
                "y": e["y"], "coarse": e["coarse"], "fold": int(x), "repeat": 0,
                "native_split": None, "group_id": None, "mag": e.get("mag"),
                "is_aug": e["is_aug"], "protocol": "A2"} for e, x in zip(rows, f)]}
    for e in MAN[ds]["protocols"]["A2"]["cv"]:
        CMAP.setdefault(e["path"], e["path"])
    c = Counter(e["class_name"] for e in MAN[ds]["protocols"]["A2"]["cv"])
    print(f"    registered A2: n={sum(c.values())}  {dict(c)}")

    if ds == "ndb_ufes" and BINARY_ARM:
        cls = MAN[ds]["classes"]
        oi = [i for i, c_ in enumerate(cls)
              if "arcinoma" in c_ or c_.upper() == "OSCC"][0]
        br = [{**e, "y2": 1 if e["y"] == oi else 0} for e in rows]
        for e in br:
            e["cn2"] = "OSCC" if e["y2"] else "non-OSCC"
        bf = strat([{**e, "class_name": e["cn2"]} for e in br], 5, SEED)
        MAN[ds]["protocols"]["A2B"] = {
            "note": "Binary OSCC-vs-rest on the same augmented corpus. The NDB-UFES "
                    "release labels are OSCC / leukoplakia-with-dysplasia / "
                    "leukoplakia-without-dysplasia; dysplasia grading carries high "
                    "inter-pathologist disagreement. Tests whether the published "
                    "97.16% describes the binary task.",
            "n_folds": 5, "n_repeats": 1, "group_disjoint": False, "test": [],
            "cv": [{"path": e["path"], "mask_path": None, "class_name": e["cn2"],
                    "y": int(e["y2"]), "coarse": e["coarse"], "fold": int(x),
                    "repeat": 0, "native_split": None, "group_id": None,
                    "mag": e.get("mag"), "is_aug": e["is_aug"], "protocol": "A2B"}
                   for e, x in zip(br, bf)]}
        MAN[ds]["classes_A2B"] = ["non-OSCC", "OSCC"]
        c = Counter(e["class_name"] for e in MAN[ds]["protocols"]["A2B"]["cv"])
        print(f"    registered A2B (binary): n={sum(c.values())}  {dict(c)}")

U.write_text(F_MAN, json.dumps(MAN))
U.write_text(F_CACHE, json.dumps({"root": str(CACHE_ROOT), "aug": str(AUG_ROOT),
                                  "px": CACHE_PX, "map": CMAP}))

U.sub("STORAGE")
print("  " + U.row(["location", "files", "size"], [30, 10, 12]))
U.rule()
tot = 0
for d, nm in ((CACHE_ROOT, "cache (real)"), (AUG_ROOT, "cache (augmented)")):
    fs = [f for f in d.rglob("*.jpg")]
    s = sum(f.stat().st_size for f in fs); tot += s
    print("  " + U.row([nm, len(fs), f"{s/1e9:.2f} GB"], [30, 10, 12]))
sv = os.statvfs(str(TMP)); sw = os.statvfs("/kaggle/working")
print(f"\n  {tot/1e9:.2f} GB on {TMP} — NOT counted against /kaggle/working")
U.kv("free on cache volume", f"{sv.f_bavail*sv.f_frsize/1e9:.0f} GB")
U.kv("free on /kaggle/working", f"{sw.f_bavail*sw.f_frsize/1e9:.2f} GB")
print(f"\n  NOTE: {TMP} is wiped between sessions — re-run BLOCK 4 each session.")
print(f"\n[BLOCK 4 ✓]  {len(CMAP)} images resolvable")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 5 — TRAINING PLAN: BACKBONE, HYPERPARAMETERS, JOB LEDGER
# ═══════════════════════════════════════════════════════════════════════════
import json, math, hashlib, subprocess, sys, warnings
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")
import numpy as np
for pkg in ("timm==1.0.15", "scikit-learn", "scipy", "psutil"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg, "--no-input"],
                   check=False, capture_output=True)
import torch, timm
import onet_common as U
U.quiet()

MAN = json.loads(F_MAN.read_text())

BACKBONE = {"name": "convnext_base.fb_in22k_ft_in1k_384", "family": "cnn",
            "params_M": 87.6, "native_px": 384}

# Throughput measured on a single T4 (fp16, channels_last, bs from MEM below).
IPS = {384: 30.5, 448: 26.6, 512: 19.4}
MEM = {384: (8, 4.3), 448: (16, 9.6), 512: (12, 9.4)}   # px -> (batch, peak GB)

# Epochs raised ~50% and patience roughly doubled versus the previous project.
PLAN = {
  "he_stained": {"px": 512, "tta": "d4",    "vflip": 1, "rot90": 1, "stain": 1,
                 "crop": 0.70, "ep": {"A": 45, "B": 45}},
  "histopath":  {"px": 448, "tta": "d4",    "vflip": 1, "rot90": 1, "stain": 1,
                 "crop": 0.70, "ep": {"A": 25, "A2": 25, "B": 70}},
  "ndb_ufes":   {"px": 448, "tta": "d4",    "vflip": 1, "rot90": 1, "stain": 1,
                 "crop": 0.75, "ep": {"A2": 30, "A2B": 30, "B": 70}},
  "clinical":   {"px": 448, "tta": "flip2", "vflip": 0, "rot90": 0, "stain": 0,
                 "crop": 0.80, "ep": {"A": 80, "A2": 40, "B": 80}},
}

HP = {
  # optimiser
  "head_lr": 1e-3, "backbone_lr": 3e-4, "layer_decay": 0.80,
  "weight_decay": 0.05, "betas": [0.9, 0.999], "eps": 1e-8, "clip_grad": 1.0,
  # schedule — warmup in ABSOLUTE steps, hard-capped at 1.2 epochs.
  # (0.15 × total gave a 2,400-step ramp on large corpora and killed training.)
  "warmup_steps": 700, "warmup_max_epochs": 1.2, "min_lr_frac": 0.02,
  # regularisation
  "label_smoothing": 0.1, "mixup_alpha": 0.4, "cutmix_alpha": 1.0,
  "mix_prob": 0.5, "mix_off_last_frac": 0.25, "drop_path": 0.2,
  "ema_decay": 0.9995, "ema_warmup_steps": 150, "balanced_sampler": True,
  # stability guards
  "probe_steps": 60, "probe_min_slope": 0.03,
  "collapse_epoch": 3, "collapse_margin": 0.05, "retry_on_collapse": 3,
  # early stopping — patience doubled
  "patience_frac": 0.40, "patience_after_frac": 0.70, "patience_min": 15,
  # runtime
  "num_workers": 2, "prefetch": 4, "min_bs": 4, "seed": SEED,
  "ckpt_per_epoch": 3,
  # deliberately absent: zero_init_head, freeze_backbone_steps.
  # Both were introduced to rescue one arm and broke two others.
}

U.hdr("BLOCK 5 — TRAINING PLAN")
U.sub("BACKBONE")
U.kv("model", BACKBONE["name"])
U.kv("pretrain", "ImageNet-22k → 1k, 384px")
U.kv("parameters", f"{BACKBONE['params_M']}M")
U.kv("why", "resolution-flexible CNN; ViTs cannot change input size without PE surgery")

U.sub("PER-DATASET CONFIGURATION")
print("  " + U.row(["dataset", "px", "batch", "peak", "img/s", "TTA",
                    "vflip", "rot90", "stain", "crop"], [13, 5, 7, 8, 8, 7, 7, 7, 7, 6]))
U.rule()
for ds, c in PLAN.items():
    bs, pk = MEM[c["px"]]
    print("  " + U.row([ds, c["px"], bs, f"{pk} GB", f"{IPS[c['px']]:.1f}", c["tta"],
                        c["vflip"], c["rot90"], c["stain"], c["crop"]],
                       [13, 5, 7, 8, 8, 7, 7, 7, 7, 6]))
print("\n  H&E histopathology (he_stained, histopath, ndb_ufes) is dihedral-invariant")
print("  → rot90 + D4 TTA + stain jitter.  clinical photographs are upright → flip only.")

U.sub("HYPERPARAMETERS")
for grp, keys in [("optimiser", ["head_lr", "backbone_lr", "layer_decay",
                                 "weight_decay", "clip_grad"]),
                  ("schedule", ["warmup_steps", "warmup_max_epochs", "min_lr_frac"]),
                  ("regularisation", ["label_smoothing", "mixup_alpha", "cutmix_alpha",
                                      "mix_prob", "drop_path", "ema_decay"]),
                  ("guards", ["probe_steps", "probe_min_slope", "collapse_epoch",
                              "retry_on_collapse"]),
                  ("early stop", ["patience_frac", "patience_after_frac",
                                  "patience_min"])]:
    print(f"    {grp:<16} " + "  ".join(f"{k}={HP[k]}" for k in keys))

JOBS = []
for ds in DATASETS:
    if ds not in MAN:
        continue
    c = PLAN[ds]
    bs, _ = MEM[c["px"]]
    for pn, ep in c["ep"].items():
        p = MAN[ds]["protocols"].get(pn)
        if p is None:
            continue
        K = len(MAN[ds].get(f"classes_{pn}", MAN[ds]["classes"]))
        ttan = {"d4": 8, "flip4": 4, "flip2": 2}[c["tta"]]
        fp = hashlib.md5(json.dumps(
            {"bb": BACKBONE["name"], "px": c["px"], "bs": bs, "ep": ep,
             "n": len(p["cv"]), "hp": HP}, sort_keys=True).encode()).hexdigest()[:10]
        for rep in range(p["n_repeats"]):
            for f in range(p["n_folds"]):
                ent = [e for e in p["cv"] if e["repeat"] == rep]
                ntr = sum(1 for e in ent if e["fold"] != f)
                nva = sum(1 for e in ent if e["fold"] == f)
                nte = len(p.get("test", []))
                sec = ntr*ep/IPS[c["px"]] + (nva*ep + (nva+nte)*ttan)/(IPS[c["px"]]*3.5)
                JOBS.append({
                    "job_id": f"{ds}__{pn}__r{rep}__f{f}", "fingerprint": fp,
                    "dataset": ds, "protocol": pn, "repeat": rep, "fold": f,
                    "epochs": ep, "px": c["px"], "bs": bs, "tta": c["tta"],
                    "vflip": bool(c["vflip"]), "rot90": bool(c["rot90"]),
                    "stain": bool(c["stain"]), "crop": c["crop"], "K": K,
                    "n_train": ntr, "n_val": nva, "n_test": nte,
                    "est_min": round(sec/60, 1),
                    "keep_weights": bool(rep == 0 and f == 0),
                    "priority": 0 if (rep == 0 and f == 0) else 1})

# diagnostic-first: fold 0 of every arm before any fold 1
JOBS.sort(key=lambda j: (j["priority"], -j["est_min"]))
U.write_text(F_CONFIG, json.dumps({"backbone": BACKBONE, "hp": HP, "plan": PLAN,
                                   "mem": {str(k): v for k, v in MEM.items()},
                                   "jobs": JOBS, "n_gpu": torch.cuda.device_count()},
                                  indent=2))
LED = ledger_load()
for j in JOBS:
    LED["jobs"].setdefault(j["job_id"], {"status": "pending", "dataset": j["dataset"],
                                         "protocol": j["protocol"], "fold": j["fold"],
                                         "repeat": j["repeat"],
                                         "fingerprint": j["fingerprint"]})
ledger_save(LED)

U.sub("SCHEDULE")
print("  " + U.row(["dataset", "proto", "models", "images", "px", "epochs",
                    "steps/ep", "min/model", "GPU-h"],
                   [13, 7, 8, 8, 5, 8, 10, 11, 8]))
U.rule()
tot = 0.0
for ds in DATASETS:
    for pn in ("A", "A2", "A2B", "B"):
        js = [j for j in JOBS if j["dataset"] == ds and j["protocol"] == pn]
        if not js:
            continue
        h = sum(j["est_min"] for j in js) / 60; tot += h
        j0 = js[0]
        print("  " + U.row([ds, pn, len(js), len(MAN[ds]["protocols"][pn]["cv"]),
                            j0["px"], j0["epochs"], max(1, j0["n_train"]//j0["bs"]),
                            f"{j0['est_min']:.1f}", f"{h:.2f}"],
                           [13, 7, 8, 8, 5, 8, 10, 11, 8]))
U.rule()
ng = max(1, torch.cuda.device_count())
print("  " + U.row(["TOTAL", "", len(JOBS), "", "", "", "", "", f"{tot:.2f}"],
                   [13, 7, 8, 8, 5, 8, 10, 11, 8]))
print(f"\n  {tot/ng:.1f} h wall-clock on {ng} GPUs  →  "
      f"~{math.ceil(tot/ng/(SESSION_BUDGET_MIN/60))} session(s)")

U.sub("WARMUP SANITY  (must finish inside 1.2 epochs)")
bad = 0
for ds in DATASETS:
    for pn in PLAN.get(ds, {}).get("ep", {}):
        js = [j for j in JOBS if j["dataset"] == ds and j["protocol"] == pn]
        if not js:
            continue
        j = js[0]
        spe = max(1, j["n_train"] // j["bs"])
        eff = min(HP["warmup_steps"], int(spe * HP["warmup_max_epochs"]))
        fr = eff / spe
        good = fr <= HP["warmup_max_epochs"] + 1e-9
        bad += not good
        print(f"    {ds}/{pn:<4} {spe:>5} steps/ep → warmup {eff:>4} = {fr:.2f} ep   "
              f"{'✓' if good else 'FAIL'}")
print(f"\n  {'all schedules valid' if not bad else f'{bad} INVALID'}")

U.sub("FIRST 8 JOBS  (one fold of every arm first — early diagnostic signal)")
for j in JOBS[:8]:
    print(f"    {j['job_id']:<28} {j['est_min']:>6.1f} min   "
          f"{j['n_train']:>6} train  K={j['K']}")
U.kv("disk free", U.gb(U.disk_free()))
print(f"\n[BLOCK 5 ✓]  {len(JOBS)} jobs registered · {tot:.1f} GPU-h")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 6 — WORKER + PER-DATASET LAUNCHER  (defines TRAIN, trains nothing)
#
#   Run this once per session, before any of BLOCKS 7–10.
#   Fixes vs the previous worker:
#     · EMA.load preserved the checkpoint's CPU device → every resume crashed
#     · probe compared loss to ln(K) instead of class-prior entropy, which
#       killed healthy jobs on imbalanced binary tasks
#     · probe is now advisory on the final attempt — it can delay a job but
#       never permanently block one
#   Each of BLOCKS 7–10 auto-saves state when it finishes or is interrupted.
# ═══════════════════════════════════════════════════════════════════════════
import os, sys, json, time, signal, shutil, zipfile, subprocess, threading
import warnings, py_compile, tempfile, gc
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
warnings.filterwarnings("ignore")
import numpy as np
import torch
import onet_common as U
U.quiet()

WORKER = WORK / "worker.py"
GRACE_SEC, RESERVE_MIN = 120, 8

WORKER_SRC = r'''
"""OralNet training worker — one process per GPU, one dataset per invocation."""
import os, sys, json, time, math, gc, random, re, signal, argparse, warnings, traceback
from pathlib import Path

warnings.filterwarnings("ignore")
for k, v in {"PYTHONWARNINGS": "ignore", "TORCH_CPP_LOG_LEVEL": "ERROR",
             "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_HUB_DISABLE_TELEMETRY": "1",
             "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1", "HF_HUB_VERBOSITY": "error",
             "TQDM_DISABLE": "1", "TOKENIZERS_PARALLELISM": "false"}.items():
    os.environ.setdefault(k, v)
sys.path.insert(0, "/kaggle/working")
import onet_common as U
U.quiet()

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image
import timm
from sklearn.metrics import f1_score, roc_auc_score

Image.MAX_IMAGE_PIXELS = None
WORK = Path("/kaggle/working")
D_STATE, D_CKPT = WORK / "state", WORK / "ckpt"
D_W, D_PRED, D_LOG = WORK / "weights", WORK / "preds", WORK / "logs"
D_CLAIM, D_DONE, D_FAIL = D_STATE / "claims", D_STATE / "done", D_STATE / "failed"
for d in (D_CKPT, D_W, D_PRED, D_LOG, D_CLAIM, D_DONE, D_FAIL):
    d.mkdir(parents=True, exist_ok=True)

STOP = {"flag": False, "why": "", "t0": time.time(), "grace": 120.0, "ignored": 0}


def _sig(num, frame):
    nm = {signal.SIGTERM: "SIGTERM", signal.SIGINT: "SIGINT"}.get(num, str(num))
    if time.time() - STOP["t0"] < STOP["grace"]:
        STOP["ignored"] += 1
        return
    STOP["flag"] = True; STOP["why"] = nm


def tag_of(jid):
    ds, pn, rp, fd = jid.split("__")
    return f"{ds[:6]}/{pn}/{rp[1:]}{fd}"


class Log:
    def __init__(self, gpu):
        self.gpu, self.tag = gpu, ""
        self.fh = open(D_LOG / f"gpu{gpu}.log", "a", buffering=1)

    def __call__(self, m):
        s = f"[{self.tag or 'g'+str(self.gpu)}] {m}"
        self.fh.write(f"{time.strftime('%H:%M:%S')} {s}\n"); print(s, flush=True)

    def block(self, lines):
        for l in lines:
            self.fh.write(f"{time.strftime('%H:%M:%S')} {l}\n")
        print("\n".join(lines), flush=True)


class DS(Dataset):
    def __init__(self, rows, cmap, px, train, vflip, rot90, stain, crop):
        self.rows, self.cmap, self.px = rows, cmap, px
        if train:
            ops = [T.RandomResizedCrop(px, scale=(crop, 1.0), ratio=(0.85, 1.18)),
                   T.RandomHorizontalFlip(0.5)]
            if vflip:
                ops.append(T.RandomVerticalFlip(0.5))
            ops.append(T.RandomChoice([T.RandomRotation((a, a)) for a in (0, 90, 180, 270)])
                       if rot90 else T.RandomRotation(15))
            ops.append(T.ColorJitter(0.25, 0.25, 0.22, 0.04) if stain
                       else T.ColorJitter(0.2, 0.2, 0.15, 0.02))
            ops += [T.RandomApply([T.GaussianBlur(5, (0.1, 1.5))], p=0.15), T.ToTensor()]
            self.tf = T.Compose(ops)
            self.er = T.RandomErasing(p=0.25, scale=(0.02, 0.12))
        else:
            self.tf = T.Compose([T.Resize(px), T.CenterCrop(px), T.ToTensor()])
            self.er = None

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]; img = None
        for p in (self.cmap.get(r["path"], r["path"]), r["path"]):
            try:
                img = Image.open(p).convert("RGB"); break
            except Exception:
                continue
        if img is None:
            img = Image.new("RGB", (self.px, self.px), (0, 0, 0))
        x = self.tf(img)
        return (self.er(x) if self.er is not None else x), int(r["y"]), int(r["row_id"])


def param_groups(model, head_lr, bb_lr, decay, wd):
    ids = set()
    for n, _ in model.named_parameters():
        m = re.search(r"(?:stages|blocks|layers)\.(\d+)", n)
        if m:
            ids.add(int(m.group(1)))
    L = (max(ids) + 3) if ids else 3
    G = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        head = bool(re.search(r"(^|\.)(head|fc|classifier)(\.|$)", n))
        if head:
            lid = L - 1
        elif re.search(r"(^|\.)(stem|patch_embed|conv_stem|bn1|cls_token|pos_embed)", n):
            lid = 0
        else:
            m = re.search(r"(?:stages|blocks|layers)\.(\d+)", n)
            lid = min(int(m.group(1)) + 1, L - 1) if m else L - 1
        lr = head_lr if head else bb_lr * (decay ** (L - 1 - lid))
        k = (round(lr, 10), 0.0 if (p.ndim <= 1 or n.endswith(".bias")) else wd, head)
        G.setdefault(k, {"params": [], "lr": k[0], "weight_decay": k[1], "is_head": head})
        G[k]["params"].append(p)
    gl = list(G.values())
    bl = [g["lr"] for g in gl if not g["is_head"]]
    return gl, (min(bl), max(bl)), head_lr


def opt_to_device(opt, dev):
    """After load_state_dict, optimiser state can sit on CPU. Move it."""
    for st in opt.state.values():
        for k, v in st.items():
            if torch.is_tensor(v):
                st[k] = v.to(dev)


class EMA:
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        sd = model.state_dict()
        self.sh = {k: v.detach().clone().float() for k, v in sd.items()
                   if v.dtype.is_floating_point}
        self.bu = {k: v.detach().clone() for k, v in sd.items()
                   if not v.dtype.is_floating_point}

    def update(self, model, warm):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n)) if self.n < warm else self.decay
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if k in self.sh:
                    self.sh[k].mul_(d).add_(v.detach().float(), alpha=1 - d)
                else:
                    self.bu[k] = v.detach().clone()

    def state(self):
        s = {k: v.clone() for k, v in self.sh.items()}
        s.update({k: v.clone() for k, v in self.bu.items()})
        return s

    def load(self, sd, n=0):
        # FIX: a checkpoint loads to CPU; the shadow lives on the model's device.
        # Assigning the CPU tensor made every subsequent update() mix devices.
        for k, v in sd.items():
            if k in self.sh:
                self.sh[k] = v.float().to(self.sh[k].device)
            elif k in self.bu:
                self.bu[k] = v.to(self.bu[k].device)
        self.n = n


def lr_at(step, total, warm, mf):
    if step < warm:
        return (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return mf + (1 - mf) * 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))


def soft_ce(lg, tgt, sm):
    n = lg.size(-1); lp = F.log_softmax(lg, -1)
    if tgt.ndim == 1:
        tgt = F.one_hot(tgt, n).float()
    tgt = tgt * (1 - sm) + sm / n
    return -(tgt * lp).sum(-1).mean()


def mix(x, y, K, ma, ca, prob, rng):
    y1 = F.one_hot(y, K).float()
    if rng.random() > prob:
        return x, y1
    idx = torch.randperm(x.size(0), device=x.device)
    if rng.random() < 0.5 and ma > 0:
        lam = float(np.random.beta(ma, ma))
        x = lam * x + (1 - lam) * x[idx]
    else:
        lam = float(np.random.beta(ca, ca))
        H, W = x.shape[-2:]
        rh, rw = int(H*math.sqrt(1-lam)), int(W*math.sqrt(1-lam))
        cy, cx = rng.randint(0, H-1), rng.randint(0, W-1)
        a, b = max(0, cy-rh//2), min(H, cy+rh//2)
        c, d = max(0, cx-rw//2), min(W, cx+rw//2)
        x[:, :, a:b, c:d] = x[idx, :, a:b, c:d]
        lam = 1 - ((b-a)*(d-c)/(H*W))
    return x, lam * y1 + (1 - lam) * y1[idx]


def tta_views(x, mode):
    if mode == "d4":
        o = []
        for k in range(4):
            r = torch.rot90(x, k, (-2, -1)); o += [r, torch.flip(r, (-1,))]
        return o
    if mode == "flip4":
        return [x, torch.flip(x, (-1,)), torch.flip(x, (-2,)), torch.flip(x, (-2, -1))]
    return [x, torch.flip(x, (-1,))]


@torch.no_grad()
def infer(model, dl, dev, mean, std, tta=None):
    model.eval(); P, Y, R = [], [], []
    for x, y, rid in dl:
        x = x.to(dev, non_blocking=True).to(memory_format=torch.channels_last)
        x = (x - mean) / std
        with torch.amp.autocast("cuda", dtype=torch.float16):
            pr = (torch.stack([F.softmax(model(v).float(), -1)
                               for v in tta_views(x, tta)]).mean(0)
                  if tta else F.softmax(model(x).float(), -1))
        P.append(pr.cpu().numpy()); Y.append(y.numpy()); R.append(rid.numpy())
    model.train()
    return np.concatenate(P), np.concatenate(Y), np.concatenate(R)


def vmet(prob, y, K):
    pr = prob.argmax(1)
    acc = float((pr == y).mean())
    f1 = float(f1_score(y, pr, average="macro", zero_division=0))
    bal = float(np.mean([(pr[y == c] == c).mean() for c in range(K) if (y == c).any()]))
    try:
        auc = float(roc_auc_score(y, prob[:, 1]) if K == 2
                    else roc_auc_score(y, prob, multi_class="ovr", average="macro"))
    except Exception:
        auc = float("nan")
    return acc, bal, f1, auc


def loader_crash(e):
    m = str(e)
    return any(s in m for s in ("DataLoader worker", "worker (pid", "Bus error",
                                "shared memory", "unexpectedly"))


def run_job(job, cfg, man, cmap, dev, log, clock, last_attempt):
    jid = job["job_id"]; log.tag = tag_of(jid)
    ds, pn, K, hp = job["dataset"], job["protocol"], job["K"], cfg["hp"]
    bb = cfg["backbone"]; px, bs = job["px"], int(job["bs"])
    nw = int(hp["num_workers"]); attempt = int(job.get("_attempt", 0) or 0)
    proto = man[ds]["protocols"][pn]
    classes = man[ds].get(f"classes_{pn}", man[ds]["classes"])

    cv = proto["cv"]
    for i, e in enumerate(cv):
        e["row_id"] = i
    for i, e in enumerate(proto.get("test", [])):
        e["row_id"] = -(i + 1)
    tr = [e for e in cv if e["repeat"] == job["repeat"] and e["fold"] != job["fold"]]
    va = [e for e in cv if e["repeat"] == job["repeat"] and e["fold"] == job["fold"]]
    te = proto.get("test", [])

    model = timm.create_model(bb["name"], pretrained=True, num_classes=K,
                              drop_path_rate=hp["drop_path"])
    dc = timm.data.resolve_model_data_config(model)
    mean = torch.tensor(dc["mean"], device=dev).view(1, 3, 1, 1)
    std = torch.tensor(dc["std"], device=dev).view(1, 3, 1, 1)
    model = model.to(dev).to(memory_format=torch.channels_last).train()
    groups, (blo, bhi), hlr = param_groups(model, hp["head_lr"], hp["backbone_lr"],
                                           hp["layer_decay"], hp["weight_decay"])
    opt = torch.optim.AdamW(groups, betas=tuple(hp["betas"]), eps=hp["eps"])
    base = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda")
    ema = EMA(model, hp["ema_decay"])
    ckp = D_CKPT / f"{jid}.resume.pt"

    ep0 = st0 = gstep = 0
    best, bep, bema, hist, acc_ = -1.0, -1, None, [], None
    ck, which = U.load_ckpt(ckp); resumed = None
    if ck is not None and ck.get("fp") == job.get("fingerprint"):
        try:
            model.load_state_dict(ck["model"])
            opt.load_state_dict(ck["opt"]); opt_to_device(opt, dev)   # FIX
            scaler.load_state_dict(ck["scaler"])
            ema.load(ck["ema"], ck.get("ema_n", 0))                    # FIX inside
            ep0, st0, gstep = ck["epoch"], ck.get("step", 0), ck["gstep"]
            best, bep, bema = ck["best"], ck.get("bep", -1), ck.get("bema")
            hist, bs, nw, acc_ = ck.get("hist", []), ck.get("bs", bs), ck.get("nw", nw), ck.get("acc")
            for key, fn in (("rng_py", random.setstate), ("rng_np", np.random.set_state)):
                if ck.get(key) is not None:
                    try:
                        fn(ck[key])
                    except Exception:
                        pass
            if ck.get("rng_t") is not None:
                try:
                    torch.set_rng_state(ck["rng_t"].cpu())
                except Exception:
                    pass
            resumed = f"RESUMED from {which} checkpoint at epoch {ep0+1}, step {st0}"
        except Exception as e:
            ep0 = st0 = gstep = 0; best, bema, hist = -1.0, None, []
            resumed = f"resume failed ({type(e).__name__}: {str(e)[:60]}) — restarting"
    elif ck is not None:
        ckp.unlink(missing_ok=True); resumed = "stale checkpoint (config changed) — restarting"
    rng = random.Random(hp["seed"] + 97*job["repeat"] + job["fold"] + 7919*attempt)

    def build(b, w):
        d = DS(tr, cmap, px, True, job["vflip"], job["rot90"], job["stain"], job["crop"])
        cnt = np.bincount([e["y"] for e in tr], minlength=K).astype(np.float64)
        wt = ((1.0/np.maximum(cnt, 1))[[e["y"] for e in tr]]
              if hp["balanced_sampler"] else np.ones(len(tr)))
        g = torch.Generator(); g.manual_seed(hp["seed"])
        smp = WeightedRandomSampler(torch.DoubleTensor(wt), len(tr), True, generator=g)
        kw = dict(num_workers=w, pin_memory=True)
        if w > 0:
            kw.update(persistent_workers=True, prefetch_factor=hp["prefetch"], timeout=180)
        return (DataLoader(d, batch_size=b, sampler=smp, drop_last=len(tr) > b, **kw),
                DataLoader(DS(va, cmap, px, False, 0, 0, 0, 1.0), batch_size=max(8, b),
                           shuffle=False, **kw),
                (DataLoader(DS(te, cmap, px, False, 0, 0, 0, 1.0), batch_size=max(8, b),
                            shuffle=False, **kw) if te else None), smp)

    ltr, lva, lte, smp = build(bs, nw)
    spe = max(1, len(ltr))
    total = spe * job["epochs"]
    warm = min(int(hp["warmup_steps"]), int(spe * hp["warmup_max_epochs"]))
    mix_off = int(job["epochs"] * (1 - hp["mix_off_last_frac"]))
    ck_every = max(20, spe // hp["ckpt_per_epoch"])
    pat = max(hp["patience_min"], int(job["epochs"] * hp["patience_frac"]))
    pat_after = int(job["epochs"] * hp["patience_after_frac"])
    ctr = np.bincount([e["y"] for e in tr], minlength=K)
    cva = np.bincount([e["y"] for e in va], minlength=K)
    maj = float(cva.max() / max(1, cva.sum())); chance = 1.0 / K
    # class-prior entropy: the loss a majority-predictor achieves. On imbalanced
    # tasks this is well BELOW ln(K), and comparing to ln(K) killed healthy jobs.
    prior = ctr / max(1, ctr.sum())
    h_prior = float(-(prior * np.log(np.clip(prior, 1e-12, 1))).sum())
    t_job = time.time()

    log.block(["", "─"*100,
               f"JOB {job['_i']}/{job['_n']}   {jid}"
               + (f"   [attempt {attempt+1}]" if attempt else ""),
               f"  data    train {len(tr)} {ctr.tolist()} · val {len(va)} {cva.tolist()}"
               f" · test {len(te)} · K={K} {classes}",
               f"  target  val majority {maj*100:.2f}% · chance {chance*100:.2f}% · "
               f"prior entropy {h_prior:.4f}",
               f"  model   {bb['name']} @{px}px · bs={bs} · workers={nw} · "
               f"lr head {hlr:.1e} / backbone {blo:.1e}–{bhi:.1e}",
               f"  plan    {job['epochs']} ep × {spe} steps · warmup {warm} "
               f"({warm/spe:.2f} ep) · mixup off @ep{mix_off+1} · "
               f"plateau after ep{pat_after} (patience {pat})"
               + (f"\n  {resumed}" if resumed else ""),
               "─"*100])

    def save(e_, s_):
        U.save_ckpt({"model": model.state_dict(), "opt": opt.state_dict(),
                     "scaler": scaler.state_dict(), "ema": ema.state(), "ema_n": ema.n,
                     "epoch": e_, "step": s_, "gstep": gstep, "best": best, "bep": bep,
                     "bema": bema, "hist": hist, "bs": bs, "nw": nw, "acc": acc_,
                     "fp": job.get("fingerprint"), "jid": jid, "saved": time.time(),
                     "rng_py": random.getstate(), "rng_np": np.random.get_state(),
                     "rng_t": torch.get_rng_state()}, ckp)

    # ── PROBE · full LR, unfrozen, referenced to the class prior ─────────────
    if ep0 == 0 and st0 == 0:
        import copy
        ms, os_ = copy.deepcopy(model.state_dict()), copy.deepcopy(opt.state_dict())
        lh, seen = [], 0
        for x, y, _ in ltr:
            if seen >= hp["probe_steps"]:
                break
            for g_, b_ in zip(opt.param_groups, base):
                g_["lr"] = b_ * min(1.0, (seen + 1) / 20.0)
            x = x.to(dev, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(dev, non_blocking=True); x = (x - mean) / std
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                l_ = soft_ce(model(x), F.one_hot(y, K).float(), hp["label_smoothing"])
            scaler.scale(l_).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), hp["clip_grad"])
            scaler.step(opt); scaler.update()
            lh.append(float(l_.detach())); seen += 1
        k = max(3, seen // 3)
        early = float(np.mean(lh[:k])) if lh else h_prior
        late = float(np.mean(lh[-k:])) if lh else h_prior
        slope = early - late
        below_prior = late < h_prior - 0.01
        stuck = (slope < hp["probe_min_slope"]) and not below_prior
        model.load_state_dict(ms); opt.load_state_dict(os_); opt_to_device(opt, dev)
        del ms, os_; gc.collect(); torch.cuda.empty_cache()
        verdict = ("HEALTHY" if not stuck else
                   ("STUCK but proceeding (final attempt)" if last_attempt
                    else ">>> STUCK <<<"))
        log(f"PROBE   {seen} steps @ full LR · loss {early:.4f} → {late:.4f} · "
            f"slope {slope:+.4f} (need {hp['probe_min_slope']:+.4f}) · "
            f"prior {h_prior:.4f} · {verdict}")
        if stuck and not last_attempt:
            (D_FAIL / f"{jid}.txt").write_text(f"probe stuck slope={slope:.4f}")
            del model, opt, scaler, ema, ltr, lva, lte
            gc.collect(); torch.cuda.empty_cache()
            return "collapsed"

    ep, esec = ep0, []
    while ep < job["epochs"]:
        smp.generator.manual_seed(hp["seed"] + 1000 * ep)
        if st0 == 0 or acc_ is None:
            acc_ = {"loss": 0., "cor": 0, "seen": 0, "gn": 0., "ngn": 0, "skip": 0, "nb": 0}
        torch.cuda.reset_peak_memory_stats(dev); t_ep = time.time()
        try:
            for si, (x, y, _) in enumerate(ltr):
                if si < st0:
                    continue
                if STOP["flag"]:
                    save(ep, si)
                    log(f"PAUSED ({STOP['why']}) at ep {ep+1} step {si} — resumable")
                    return "paused"
                f_ = lr_at(gstep, total, warm, hp["min_lr_frac"])
                for g_, b_ in zip(opt.param_groups, base):
                    g_["lr"] = b_ * f_
                x = x.to(dev, non_blocking=True).to(memory_format=torch.channels_last)
                y = y.to(dev, non_blocking=True); x = (x - mean) / std
                xi, yt = (mix(x, y, K, hp["mixup_alpha"], hp["cutmix_alpha"],
                              hp["mix_prob"], rng) if ep < mix_off
                          else (x, F.one_hot(y, K).float()))
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    lg = model(xi); loss = soft_ce(lg, yt, hp["label_smoothing"])
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), hp["clip_grad"]))
                scaler.step(opt); scaler.update()
                ema.update(model, hp["ema_warmup_steps"]); gstep += 1
                acc_["loss"] += float(loss.detach())
                acc_["cor"] += int((lg.detach().argmax(1) == y).sum())
                acc_["seen"] += int(y.numel()); acc_["nb"] += 1
                if math.isfinite(gn):
                    acc_["gn"] += gn; acc_["ngn"] += 1
                else:
                    acc_["skip"] += 1
                if (si + 1) % ck_every == 0:
                    save(ep, si + 1)
        except torch.cuda.OutOfMemoryError:
            del ltr, lva, lte, smp; gc.collect(); torch.cuda.empty_cache()
            nb = max(hp["min_bs"], bs // 2)
            if nb == bs:
                raise
            log(f"OOM at bs={bs} → retrying epoch {ep+1} at bs={nb}")
            bs = nb; ltr, lva, lte, smp = build(bs, nw)
            spe = max(1, len(ltr)); total = spe * job["epochs"]
            warm = min(int(hp["warmup_steps"]), int(spe * hp["warmup_max_epochs"]))
            ck_every = max(20, spe // hp["ckpt_per_epoch"]); st0 = 0
            continue
        except (RuntimeError, OSError) as e:
            if not loader_crash(e) or nw == 0:
                raise
            log("DataLoader workers died → rebuilding with num_workers=0")
            try:
                del ltr, lva, lte, smp
            except Exception:
                pass
            gc.collect(); torch.cuda.empty_cache()
            nw = 0; ltr, lva, lte, smp = build(bs, nw)
            spe = max(1, len(ltr)); total = spe * job["epochs"]
            warm = min(int(hp["warmup_steps"]), int(spe * hp["warmup_max_epochs"]))
            ck_every = max(20, spe // hp["ckpt_per_epoch"]); st0 = 0
            continue

        st0 = 0
        bak = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(ema.state(), strict=False)
        vp, vy, _ = infer(model, lva, dev, mean, std)
        model.load_state_dict(bak); del bak
        vacc, vbal, vf1, vauc = vmet(vp, vy, K)
        imp = vf1 > best
        if imp:
            best, bep = vf1, ep
            bema = {k: (v.half() if v.dtype.is_floating_point else v).cpu()
                    for k, v in ema.state().items()}
        tl = acc_["loss"] / max(1, acc_["nb"]); ta = acc_["cor"] / max(1, acc_["seen"])
        mgn = acc_["gn"] / max(1, acc_["ngn"]); dt = time.time() - t_ep
        esec.append(dt)
        hist.append({"epoch": ep, "loss": tl, "train_acc": ta, "val_acc": vacc,
                     "val_bal": vbal, "val_f1": vf1, "val_auc": vauc,
                     "lr": hlr * lr_at(gstep, total, warm, hp["min_lr_frac"]),
                     "gn": mgn, "skip": acc_["skip"], "sec": dt})
        hv = ("DEAD" if vacc <= chance + 0.03 else "WEAK" if vacc <= maj else
              "OK" if ep <= 2 else "GOOD" if imp else "PLATEAU")
        bar = {"DEAD": "[!! STOP !!]", "WEAK": "[  weak   ]", "OK": "[   ok    ]",
               "GOOD": "[ WORKING ]", "PLATEAU": "[ plateau ]"}[hv]
        log(f"ep {ep+1:03d}/{job['epochs']:03d} {bar} · loss {tl:.4f} tr {ta*100:5.1f}% · "
            f"val {vacc*100:6.2f}% bal {vbal*100:6.2f}% f1 {vf1:.4f} auc {vauc:.4f} · "
            f"lr {hlr*lr_at(gstep,total,warm,hp['min_lr_frac']):.2e} gn {mgn:5.2f}"
            + (f" skip{acc_['skip']}" if acc_["skip"] else "")
            + f" · {torch.cuda.max_memory_allocated(dev)/1e9:4.1f}GB {dt:4.0f}s "
              f"eta {U.hms(np.mean(esec[-3:])*(job['epochs']-ep-1))}"
            + ("  ★" if imp else ""))
        save(ep + 1, 0); ep += 1

        if ep == hp["collapse_epoch"] and len(hist) >= 2 and not last_attempt:
            d_ = abs(hist[-1]["loss"] - hist[0]["loss"])
            if vacc <= chance + 0.05 and d_ < hp["collapse_margin"]:
                log(f"COLLAPSE · val {vacc*100:.2f}% at chance, loss flat ({d_:.4f}) "
                    f"after {ep} epochs — aborting")
                (D_FAIL / f"{jid}.txt").write_text(f"collapse acc={vacc:.4f} d={d_:.4f}")
                for f_ in (ckp, Path(str(ckp) + ".prev")):
                    f_.unlink(missing_ok=True)
                del model, opt, scaler, ema, ltr, lva, lte
                gc.collect(); torch.cuda.empty_cache()
                return "collapsed"
        if STOP["flag"]:
            log(f"PAUSED ({STOP['why']}) after ep {ep} — resumable"); return "paused"
        if ep >= pat_after and (ep - 1 - bep) >= pat:
            log(f"plateau stop · no improvement for {ep-1-bep} epochs past ep{pat_after}")
            break
        if ep < job["epochs"] and not clock.fits(np.mean(esec[-3:]), margin=1.4):
            log(f"PAUSED · {U.hms(clock.left)} budget left < one epoch — resumable")
            return "paused"

    model.load_state_dict({k: (v.float() if v.dtype == torch.float16 else v)
                           for k, v in bema.items()}, strict=False)
    vp, vy, vr = infer(model, lva, dev, mean, std, tta=job["tta"])
    out = {"val_prob": vp, "val_y": vy, "val_row": vr}
    if lte is not None:
        tp, ty, tr_ = infer(model, lte, dev, mean, std, tta=job["tta"])
        out.update({"test_prob": tp, "test_y": ty, "test_row": tr_})
    np.savez_compressed(D_PRED / f"{jid}.npz", **out)
    if job.get("keep_weights"):
        torch.save({"state": bema, "backbone": bb["name"], "px": px, "K": K,
                    "classes": classes, "mean": dc["mean"], "std": dc["std"]},
                   D_W / f"{jid}.fp16.pt")
    tacc, tbal, tf1, tauc = vmet(vp, vy, K)
    U.write_text(D_DONE / f"{jid}.json", json.dumps(
        {"job_id": jid, "dataset": ds, "protocol": pn, "fold": job["fold"],
         "repeat": job["repeat"], "fingerprint": job.get("fingerprint"),
         "best_f1": float(best), "best_epoch": int(bep), "epochs_run": int(ep),
         "epochs_planned": job["epochs"], "tta_acc": float(tacc),
         "tta_bal": float(tbal), "tta_f1": float(tf1), "tta_auc": float(tauc),
         "majority": maj, "final_bs": bs, "attempt": attempt,
         "minutes": round((time.time()-t_job)/60, 1), "history": hist}))
    for f_ in (ckp, Path(str(ckp) + ".prev")):
        f_.unlink(missing_ok=True)
    log(f"DONE · best f1 {best:.4f} @ep{bep+1}/{ep} → TTA acc {tacc*100:.2f}% "
        f"bal {tbal*100:.2f}% f1 {tf1:.4f} auc {tauc:.4f} · {(time.time()-t_job)/60:.1f} min")
    del model, opt, scaler, ema, ltr, lva, lte, bema
    gc.collect(); torch.cuda.empty_cache()
    return "done"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--dataset", type=str, required=True)
    ap.add_argument("--budget-min", type=float, default=645)
    ap.add_argument("--t0", type=float, default=None)
    ap.add_argument("--reserve-min", type=float, default=8)
    ap.add_argument("--grace-sec", type=float, default=120)
    a = ap.parse_args()
    STOP["grace"] = a.grace_sec
    signal.signal(signal.SIGTERM, _sig); signal.signal(signal.SIGINT, _sig)

    log = Log(a.gpu)
    try:
        s = os.statvfs("/dev/shm"); shm = s.f_blocks * s.f_frsize / 1e9
    except Exception:
        shm = -1
    log(f"worker up · pid {os.getpid()} · dataset {a.dataset} · "
        f"/dev/shm {shm:.1f} GB · grace {a.grace_sec:.0f}s")
    torch.cuda.set_device(0); dev = torch.device("cuda:0")
    torch.backends.cudnn.benchmark = True
    clock = U.Clock(a.budget_min, a.t0)
    cfg = json.loads((D_STATE / "config.json").read_text())
    man = json.loads((D_STATE / "manifest.json").read_text())
    cmap = json.loads((D_STATE / "cache.json").read_text())["map"]
    for fn in (random.seed, np.random.seed, torch.manual_seed):
        fn(cfg["hp"]["seed"] + a.gpu)

    jobs = [j for j in cfg["jobs"] if j["dataset"] == a.dataset]
    for i, j in enumerate(jobs):
        j["_i"], j["_n"] = i + 1, len(jobs)
    nd = npa = nf = 0
    for job in jobs:
        if STOP["flag"]:
            break
        jid = job["job_id"]
        if (D_DONE / f"{jid}.json").exists():
            continue
        resumable = (D_CKPT / f"{jid}.resume.pt").exists()
        if clock.left < a.reserve_min * 60:
            log.tag = ""; log(f"STOP · {U.hms(clock.left)} left"); break
        if not resumable and not clock.fits(job["est_min"] * 60, margin=1.0):
            continue
        cp = D_CLAIM / f"{jid}.claim"
        try:
            fd = os.open(str(cp), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, f"pid={os.getpid()} gpu={a.gpu}".encode()); os.close(fd)
        except FileExistsError:
            continue
        tries = int(cfg["hp"].get("retry_on_collapse", 0)) + 1
        st = None
        try:
            for at in range(tries):
                job["_attempt"] = at
                if at:
                    s_ = cfg["hp"]["seed"] + 7919 * (at + 1) + job["fold"]
                    random.seed(s_); np.random.seed(s_ % (2**31))
                    torch.manual_seed(s_); torch.cuda.manual_seed_all(s_)
                    log(f"retry {at}/{tries-1} with seed {s_}")
                st = run_job(job, cfg, man, cmap, dev, log, clock, at == tries - 1)
                if st != "collapsed":
                    break
                (D_FAIL / f"{jid}.txt").unlink(missing_ok=True)
            if st == "done":
                nd += 1
            elif st == "paused":
                npa += 1; cp.unlink(missing_ok=True); break
            else:
                nf += 1; cp.unlink(missing_ok=True)
        except Exception:
            nf += 1
            tb = traceback.format_exc()
            (D_FAIL / f"{jid}.txt").write_text(tb)
            log(f"FAILED · {tb.strip().splitlines()[-1][:110]}")
            cp.unlink(missing_ok=True); gc.collect(); torch.cuda.empty_cache()
    log.tag = ""
    log(f"worker exit · {nd} done · {npa} paused · {nf} failed · "
        f"{STOP['ignored']} signals ignored · {U.hms(clock.left)} left")


if __name__ == "__main__":
    main()
'''

WORKER.write_text(WORKER_SRC)
try:
    py_compile.compile(str(WORKER), doraise=True, cfile=tempfile.mktemp(suffix=".pyc"))
    SYN = "PASS"
except Exception as e:
    SYN = f"FAIL — {e}"


def AUTOSAVE(label):
    """Runs automatically at the end of every training block."""
    U.hdr(f"AUTO-SAVE · {label}")
    procs = []
    try:
        import psutil
        for p in psutil.process_iter(["pid", "cmdline"]):
            if any("worker.py" in str(c) for c in (p.info.get("cmdline") or [])):
                procs.append(p)
    except Exception:
        pass
    if procs:
        for p in procs:
            try:
                p.send_signal(signal.SIGTERM)
            except Exception:
                pass
        t0 = time.time()
        while time.time() - t0 < 180:
            try:
                import psutil
                if not any("worker.py" in str(c) for pp in psutil.process_iter(["cmdline"])
                           for c in (pp.info.get("cmdline") or [])):
                    break
            except Exception:
                break
            time.sleep(5)
    U.kv("workers", "all stopped")

    ck = sorted({f.name.split(".resume.pt")[0] for f in D_CKPT.glob("*.resume.pt")})
    good, bad = [], []
    for jid in ck:
        obj, which = U.load_ckpt(D_CKPT / f"{jid}.resume.pt")
        if obj is None:
            bad.append(jid); continue
        good.append((jid, obj["epoch"], obj.get("step", 0), obj["best"], which))
        del obj; gc.collect()
    if good:
        print("  " + U.row(["resumable job", "resumes at", "best f1", "src"],
                           [30, 20, 10, 10]))
        U.rule()
        for jid, e, s, b, w in good:
            print("  " + U.row([jid, f"epoch {e+1} step {s}",
                                f"{b:.4f}" if b > 0 else "—", w], [30, 20, 10, 10]))
    for jid in bad:
        print(f"  UNREADABLE (will restart): {jid}")
        for s in ("", ".prev"):
            Path(f"{D_CKPT/jid}.resume.pt{s}").unlink(missing_ok=True)

    before = U.disk_free()
    U.cleanup(clear_claims=True, log_mb=4)
    for p in list(WORK.rglob("*.tmp")) + list(WORK.rglob("__pycache__")):
        shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    U.kv("disk reclaimed", f"{(U.disk_free()-before)/1e6:.0f} MB")
    U.kv("disk free", U.gb(U.disk_free()))

    CFG_ = json.loads(F_CONFIG.read_text())
    dn = {p.stem for p in D_DONE.glob("*.json")}
    print()
    print("  " + U.row(["dataset", "done", "total", "remaining", "GPU-h left"],
                       [14, 7, 8, 11, 12]))
    U.rule()
    for ds_ in DATASETS:
        js = [j for j in CFG_["jobs"] if j["dataset"] == ds_]
        if not js:
            continue
        d_ = sum(1 for j in js if j["job_id"] in dn)
        rem = [j for j in js if j["job_id"] not in dn]
        print("  " + U.row([ds_, d_, len(js), len(rem),
                            f"{sum(j['est_min'] for j in rem)/60:.1f}"],
                           [14, 7, 8, 11, 12]))
    print(f"\n  Before closing Kaggle: Persistence = 'Files only' → Save Version.")
    print(f"\n[AUTO-SAVE ✓]  all state checkpointed · safe to close")


def TRAIN(ds):
    """Train every job for one dataset. Auto-saves on completion or interrupt."""
    if SYN != "PASS":
        raise SystemExit(f"worker did not compile: {SYN}")
    U.hdr(f"TRAINING · {ds.upper()}   ·   session {SESSION}")
    U.cleanup(clear_claims=True)
    CFG_ = json.loads(F_CONFIG.read_text())
    js = [j for j in CFG_["jobs"] if j["dataset"] == ds]
    dn = {p.stem for p in D_DONE.glob("*.json")}
    todo = [j for j in js if j["job_id"] not in dn]
    res = {f.name.split(".resume.pt")[0] for f in D_CKPT.glob("*.resume.pt")}
    ng = max(1, torch.cuda.device_count())
    U.kv("jobs", f"{len(js)} total · {len(js)-len(todo)} done · {len(todo)} to run · "
                 f"{len(res & {j['job_id'] for j in todo})} resuming")
    U.kv("work", f"{sum(j['est_min'] for j in todo)/60:.1f} GPU-h → "
                 f"~{sum(j['est_min'] for j in todo)/60/ng:.1f} h on {ng} GPUs")
    U.kv("budget", f"{U.hms(CLOCK.left)} remaining in this session")
    U.kv("disk free", U.gb(U.disk_free()))
    if not todo:
        print(f"\n  {ds} is complete — nothing to train.")
        AUTOSAVE(ds); return
    by = defaultdict(list)
    for j in todo:
        by[j["protocol"]].append(j)
    print()
    print("  " + U.row(["protocol", "left", "px", "bs", "epochs", "min/job", "GPU-h"],
                       [10, 6, 5, 5, 8, 9, 8]))
    U.rule()
    for k in sorted(by, key=lambda k: -sum(x["est_min"] for x in by[k])):
        j0 = by[k][0]
        print("  " + U.row([k, len(by[k]), j0["px"], j0["bs"], j0["epochs"],
                            f"{j0['est_min']:.1f}",
                            f"{sum(x['est_min'] for x in by[k])/60:.2f}"],
                           [10, 6, 5, 5, 8, 9, 8]))
    env = os.environ.copy(); env.update(_QUIET)
    procs = []
    for g in range(ng):
        e = env.copy(); e["CUDA_VISIBLE_DEVICES"] = str(g)
        procs.append(subprocess.Popen(
            [sys.executable, "-u", str(WORKER), "--gpu", str(g), "--dataset", ds,
             "--budget-min", str(SESSION_BUDGET_MIN), "--t0", str(CLOCK.t0),
             "--reserve-min", str(RESERVE_MIN), "--grace-sec", str(GRACE_SEC)],
            env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1))
    print()
    U.kv("launched", f"{ng} worker(s) · pids {[p.pid for p in procs]}")
    print("  Leave this cell running. Press ■ to stop — auto-save runs afterwards.")
    print("=" * 100)
    lk = threading.Lock()
    MUTE = ("unauthenticated requests", "HF_TOKEN", "Xet Storage", "Loading widget")

    def pump(pr):
        for line in pr.stdout:
            s = line.rstrip()
            if s and not any(m in s for m in MUTE):
                with lk:
                    print(s, flush=True)

    ths = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
    for t in ths:
        t.start()
    try:
        for t in ths:
            t.join()
        for p in procs:
            p.wait()
    except KeyboardInterrupt:
        print("\n  interrupt — workers checkpointing (up to 180 s) ...")
        for p in procs:
            p.send_signal(signal.SIGTERM)
        dl = time.time() + 180
        for p in procs:
            try:
                p.wait(timeout=max(5, dl - time.time()))
            except Exception:
                p.kill()
    dn2 = {p.stem for p in D_DONE.glob("*.json")}
    got = [j for j in js if j["job_id"] in dn2]
    U.hdr(f"{ds.upper()} · {len(got)}/{len(js)} COMPLETE")
    if got:
        recs = []
        for j in got:
            try:
                recs.append(json.loads((D_DONE / f"{j['job_id']}.json").read_text()))
            except Exception:
                pass
        agg = defaultdict(list)
        for r in recs:
            agg[r["protocol"]].append(r)
        print("  " + U.row(["protocol", "folds", "acc", "sd", "bal", "f1", "AUC",
                            "base", "vs base"], [10, 7, 8, 7, 8, 8, 8, 8, 9]))
        U.rule()
        for k in sorted(agg):
            rs = agg[k]
            a = np.array([r["tta_acc"] for r in rs]) * 100
            b = np.array([r["tta_bal"] for r in rs]) * 100
            f1 = np.array([r["tta_f1"] for r in rs])
            au = np.array([r["tta_auc"] for r in rs])
            base = np.mean([r["majority"] for r in rs]) * 100
            print("  " + U.row([k, len(rs), f"{a.mean():.2f}", f"{a.std():.2f}",
                                f"{np.nanmean(b):.2f}", f"{f1.mean():.4f}",
                                f"{np.nanmean(au):.4f}", f"{base:.2f}",
                                f"{a.mean()-base:+.2f}"], [10, 7, 8, 7, 8, 8, 8, 8, 9]))
    AUTOSAVE(ds)


U.hdr("BLOCK 6 — WORKER & LAUNCHER READY")
U.kv("worker", f"{WORKER.name} · {len(WORKER_SRC)//1024} KB · syntax {SYN}")
U.kv("fix 1", "EMA.load keeps the shadow on its own device (was the resume crash)")
U.kv("fix 2", "probe compares to class-prior entropy, not ln(K)")
U.kv("fix 3", "probe and collapse guards are advisory on the final attempt")
U.kv("auto-save", "runs at the end of every training block — no manual step")
CFG = json.loads(F_CONFIG.read_text())
dn = {p.stem for p in D_DONE.glob("*.json")}
print()
print("  " + U.row(["block", "dataset", "jobs", "done", "GPU-h left", "wall (2 GPU)"],
                   [8, 14, 7, 7, 12, 14]))
U.rule()
for blk, ds in ((7, "he_stained"), (8, "clinical"), (9, "histopath"), (10, "ndb_ufes")):
    js = [j for j in CFG["jobs"] if j["dataset"] == ds]
    rem = [j for j in js if j["job_id"] not in dn]
    h = sum(j["est_min"] for j in rem) / 60
    print("  " + U.row([blk, ds, len(js), len(js)-len(rem), f"{h:.1f}",
                        f"{h/2:.1f} h"], [8, 14, 7, 7, 12, 14]))
print(f"\n[BLOCK 6 ✓]  run BLOCK 7–10 in any order; each is independent")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 7 — TRAIN he_stained        (~3.3 h wall · auto-saves when finished)
# ═══════════════════════════════════════════════════════════════════════════
TRAIN("he_stained")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 8 — TRAIN clinical          (~3.9 h wall · auto-saves when finished)
# ═══════════════════════════════════════════════════════════════════════════
TRAIN("clinical")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 9 — TRAIN histopath         (~10.4 h wall · auto-saves when finished)
# ═══════════════════════════════════════════════════════════════════════════
TRAIN("histopath")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 10 — TRAIN ndb_ufes         (~18.2 h wall · needs 2 sessions)
# ═══════════════════════════════════════════════════════════════════════════
TRAIN("ndb_ufes")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 11 — AGGREGATION · CALIBRATION · ENSEMBLING · SIGNIFICANCE TESTS
# ═══════════════════════════════════════════════════════════════════════════
import json, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
import numpy as np
from scipy import stats as st
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, roc_curve,
                             average_precision_score, confusion_matrix,
                             matthews_corrcoef, cohen_kappa_score,
                             balanced_accuracy_score)
import onet_common as U
U.quiet()

MAN = json.loads(F_MAN.read_text())
EPS, NB = 1e-12, 2000
RNG = np.random.RandomState(SEED)
PROTOS = ("A", "A2", "A2B", "B")


def ece_mce(p, y, nb=15):
    conf, pred = p.max(1), p.argmax(1)
    cor = (pred == y).astype(float)
    ed = np.linspace(0, 1, nb + 1); e = m = 0.0
    for i in range(nb):
        k = (conf > ed[i]) & (conf <= ed[i+1]) if i else (conf >= ed[i]) & (conf <= ed[i+1])
        if k.sum() == 0:
            continue
        g = abs(cor[k].mean() - conf[k].mean()); e += k.mean() * g; m = max(m, g)
    return float(e), float(m)


def core(p, y, K, pos):
    pred = p.argmax(1)
    cm = confusion_matrix(y, pred, labels=list(range(K)))
    oh = np.eye(K)[y]
    o = {"n": int(len(y)), "accuracy": float(accuracy_score(y, pred)),
         "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
         "f1_macro": float(f1_score(y, pred, average="macro", zero_division=0)),
         "f1_weighted": float(f1_score(y, pred, average="weighted", zero_division=0)),
         "mcc": float(matthews_corrcoef(y, pred)) if len(set(y)) > 1 else float("nan"),
         "kappa": float(cohen_kappa_score(y, pred)) if len(set(y)) > 1 else float("nan"),
         "brier": float(((p - oh) ** 2).sum(1).mean()), "confusion_matrix": cm.tolist()}
    per = {}
    for c in range(K):
        tp = cm[c, c]; fn = cm[c].sum() - tp
        fp = cm[:, c].sum() - tp; tn = cm.sum() - tp - fn - fp
        se, sp = tp / max(1, tp+fn), tn / max(1, tn+fp)
        per[c] = {"sensitivity": float(se), "specificity": float(sp),
                  "precision": float(tp/max(1, tp+fp)), "npv": float(tn/max(1, tn+fn)),
                  "fnr": float(1-se), "youden": float(se+sp-1),
                  "lr_pos": float(se/max(1e-9, 1-sp)),
                  "lr_neg": float((1-se)/max(1e-9, sp)), "support": int(tp+fn)}
    o["per_class"] = per
    for mk in ("sensitivity", "specificity", "precision", "npv", "youden"):
        o[f"{mk}_macro"] = float(np.mean([per[c][mk] for c in per]))
    try:
        if K == 2:
            pi = pos if pos is not None else 1
            yb = (y == pi).astype(int)
            o["auroc"] = float(roc_auc_score(yb, p[:, pi]))
            o["auprc"] = float(average_precision_score(yb, p[:, pi]))
            o.update(sensitivity=per[pi]["sensitivity"], specificity=per[pi]["specificity"],
                     precision=per[pi]["precision"], npv=per[pi]["npv"],
                     fnr=per[pi]["fnr"], lr_pos=per[pi]["lr_pos"], lr_neg=per[pi]["lr_neg"])
            fpr, tpr, thr = roc_curve(yb, p[:, pi])
            j = int(np.argmax(tpr - fpr))
            o["youden_opt"] = {"threshold": float(thr[j]), "sensitivity": float(tpr[j]),
                               "specificity": float(1 - fpr[j])}
            for t, kk in ((.95, "spec_at_sens95"), (.99, "spec_at_sens99")):
                ix = np.where(tpr >= t)[0]
                o[kk] = float(1 - fpr[ix[0]]) if len(ix) else float("nan")
        else:
            o["auroc"] = float(roc_auc_score(y, p, multi_class="ovr", average="macro"))
            o["auprc"] = float(average_precision_score(oh, p, average="macro"))
    except Exception:
        o["auroc"] = o["auprc"] = float("nan")
    e, m = ece_mce(p, y); o.update(ece=e, mce=m)
    cc = np.bincount(y, minlength=K)
    o["majority_baseline"] = float(cc.max() / max(1, cc.sum()))
    return o


def boot(p, y, K, pos, n=NB):
    keys = ("accuracy", "balanced_accuracy", "f1_macro", "auroc", "sensitivity_macro",
            "specificity_macro", "precision_macro", "npv_macro", "mcc")
    N = len(y); a = defaultdict(list)
    for _ in range(n):
        i = RNG.randint(0, N, N)
        if len(set(y[i])) < 2:
            continue
        try:
            m = core(p[i], y[i], K, pos)
        except Exception:
            continue
        for k in keys:
            v = m.get(k)
            if v is not None and np.isfinite(v):
                a[k].append(v)
    return {k: {"lo": float(np.percentile(v, 2.5)), "hi": float(np.percentile(v, 97.5))}
            for k, v in a.items() if v}


def fit_T(p, y):
    lp = np.log(np.clip(p, EPS, 1)); bt, bn = 1.0, np.inf
    for t in np.concatenate([np.linspace(.3, 3, 136), np.linspace(3, 8, 26)]):
        z = lp / t; z -= z.max(1, keepdims=True)
        q = np.exp(z); q /= q.sum(1, keepdims=True)
        nll = -np.log(np.clip(q[np.arange(len(y)), y], EPS, 1)).mean()
        if nll < bn:
            bn, bt = nll, float(t)
    return bt


def apply_T(p, t):
    z = np.log(np.clip(p, EPS, 1)) / t; z -= z.max(1, keepdims=True)
    q = np.exp(z); return q / q.sum(1, keepdims=True)


DONE = {}
for f in D_DONE.glob("*.json"):
    try:
        DONE[f.stem] = json.loads(f.read_text())
    except Exception:
        pass

U.hdr("BLOCK 11 — AGGREGATION · CALIBRATION · SIGNIFICANCE")

U.sub("FOLD STABILITY GATE")
BAD = set()
byp = defaultdict(list)
for k, r in DONE.items():
    byp[(r["dataset"], r["protocol"])].append((k, r["tta_acc"], r["majority"]))
print("  " + U.row(["arm", "n", "median", "abs floor", "MAD floor", "excluded"],
                   [17, 5, 10, 11, 11, 26]))
U.rule()
for kp, lst in sorted(byp.items()):
    a = np.array([x[1] for x in lst]); maj = float(np.mean([x[2] for x in lst]))
    med = float(np.median(a)); mad = float(np.median(np.abs(a - med))) or 0.005
    af = maj + 0.05
    mf = med - max(0.12, 6 * mad)
    floor = max(af, mf)
    ex = [j for j, v, _ in lst if v < floor]
    BAD.update(ex)
    print("  " + U.row([f"{kp[0]}/{kp[1]}", len(lst), f"{med*100:.2f}",
                        f"{af*100:.2f}", f"{mf*100:.2f}",
                        ", ".join(e.split("__")[-1] for e in ex) or "none"],
                       [17, 5, 10, 11, 11, 26]))
U.write_text(D_EVAL / "excluded_folds.json", json.dumps(sorted(BAD), indent=2))
if BAD:
    print(f"\n  {len(BAD)} fold(s) excluded — a fold that cannot clear its own majority")
    print(f"  baseline is a failed optimisation, not a data-difficulty result.")
    for b in sorted(BAD):
        print(f"    {b:<32} acc {DONE[b]['tta_acc']*100:6.2f}%  "
              f"maj {DONE[b]['majority']*100:.2f}%")

preds = defaultdict(list)
for f in D_PRED.glob("*.npz"):
    if f.stem in BAD:
        continue
    ds, pn, rp, fd = f.stem.split("__")
    z = np.load(f)
    preds[(ds, pn)].append({"rep": int(rp[1:]), "fold": int(fd[1:]),
                            "vp": z["val_prob"], "vy": z["val_y"], "vr": z["val_row"],
                            "tp": z["test_prob"] if "test_prob" in z else None,
                            "ty": z["test_y"] if "test_y" in z else None,
                            "tr": z["test_row"] if "test_row" in z else None})

U.sub("PER-ARM RESULTS")
R, OOF = {}, {}
for ds in DATASETS:
    if ds not in MAN:
        continue
    for pn in PROTOS:
        p = MAN[ds]["protocols"].get(pn)
        if p is None or not preds.get((ds, pn)):
            continue
        classes = MAN[ds].get(f"classes_{pn}", MAN[ds]["classes"])
        K = len(classes)
        posn = "OSCC" if pn == "A2B" else MAN[ds].get("positive_class")
        pos = classes.index(posn) if posn in classes else (1 if K == 2 else None)
        got = preds[(ds, pn)]
        oof = []
        for rep in sorted({g["rep"] for g in got}):
            rows = [g for g in got if g["rep"] == rep]
            if not rows:
                continue
            rid = np.concatenate([g["vr"] for g in rows])
            pr = np.concatenate([g["vp"] for g in rows])
            yy = np.concatenate([g["vy"] for g in rows])
            o = np.argsort(rid)
            oof.append({"row": rid[o], "prob": pr[o], "y": yy[o]})
        if not oof:
            continue
        cp = np.concatenate([r["prob"] for r in oof])
        cy = np.concatenate([r["y"] for r in oof])
        cr = np.concatenate([r["row"] for r in oof])
        T = fit_T(cp, cy)
        raw, cal = core(cp, cy, K, pos), core(apply_T(cp, T), cy, K, pos)
        sacc = [float((g["vp"].argmax(1) == g["vy"]).mean()) for g in got]
        key = f"{ds}__{pn}"
        e = {"dataset": ds, "protocol": pn, "classes": classes, "n_cls": K,
             "positive_class": posn, "temperature": T, "n_models": len(got),
             "folds_trained": len(byp.get((ds, pn), [])),
             "folds_excluded": sorted(x for x in BAD if x.startswith(f"{ds}__{pn}__")),
             "single_mean": float(np.mean(sacc)), "single_sd": float(np.std(sacc)),
             "oof_raw": raw, "oof_cal": cal,
             "oof_ci": boot(apply_T(cp, T), cy, K, pos), "note": p["note"]}
        ht = [g for g in got if g["tp"] is not None]
        if ht:
            o0 = np.argsort(ht[0]["tr"]); ty = ht[0]["ty"][o0]
            mem = np.stack([g["tp"][np.argsort(g["tr"])] for g in ht])
            ens = mem.mean(0)
            ss = [float((m.argmax(1) == ty).mean()) for m in mem]
            e["test_cal"] = core(apply_T(ens, T), ty, K, pos)
            e["test_ci"] = boot(apply_T(ens, T), ty, K, pos)
            e["test_single_mean"] = float(np.mean(ss))
            np.savez_compressed(D_EVAL / f"{key}__members.npz",
                                members=mem, y=ty, T=np.array([T]))
        np.savez_compressed(D_EVAL / f"{key}__oof.npz", prob=cp, y=cy, row=cr,
                            T=np.array([T]))
        OOF[key] = {"prob": apply_T(cp, T), "y": cy, "row": cr}
        R[key] = e
        m = e.get("test_cal", cal)
        src = "TEST-ens" if "test_cal" in e else "OOF"
        print(f"\n  [{key}]  {len(got)} folds"
              + (f" ({len(e['folds_excluded'])} excluded)" if e["folds_excluded"] else "")
              + f" · {src} n={m['n']} · T={T:.3f}")
        print(f"    single-fold {e['single_mean']*100:.2f} ± {e['single_sd']*100:.2f}"
              + (f"  →  ensemble {m['accuracy']*100:.2f}"
                 f" ({(m['accuracy']-e.get('test_single_mean', e['single_mean']))*100:+.2f})"
                 if "test_cal" in e else f"  →  OOF {m['accuracy']*100:.2f}"))
        print(f"    acc {m['accuracy']*100:6.2f}  bal {m['balanced_accuracy']*100:6.2f}  "
              f"f1 {m['f1_macro']:.4f}  auroc {m['auroc']:.4f}  mcc {m['mcc']:.4f}")
        print(f"    sens {m['sensitivity_macro']:.4f}  spec {m['specificity_macro']:.4f}  "
              f"prec {m['precision_macro']:.4f}  npv {m['npv_macro']:.4f}")
        print(f"    ECE {raw['ece']:.4f} → {m['ece']:.4f}  ·  Brier {m['brier']:.4f}  ·  "
              f"baseline {m['majority_baseline']*100:.2f}%")
        if K == 2 and "youden_opt" in m:
            yo = m["youden_opt"]
            print(f"    Youden thr {yo['threshold']:.3f} → sens {yo['sensitivity']:.4f} "
                  f"spec {yo['specificity']:.4f}  ·  spec@sens95 "
                  f"{m.get('spec_at_sens95', float('nan')):.4f}  ·  "
                  f"LR+ {m.get('lr_pos', float('nan')):.1f}  "
                  f"LR− {m.get('lr_neg', float('nan')):.3f}  "
                  f"FNR {m.get('fnr', float('nan')):.4f}")

U.write_text(D_EVAL / "results.json", json.dumps(R, indent=2, default=float))

U.sub("PROTOCOL-GAP SIGNIFICANCE  (paired, identical images)")


def mcnemar(c1, c2):
    b = int(np.sum(c1 & ~c2)); c = int(np.sum(~c1 & c2))
    if b + c == 0:
        return 1.0, b, c
    return float(min(1.0, 2 * st.binom.cdf(min(b, c), b + c, 0.5))), b, c


def pboot(c1, c2, n=5000):
    d = c1.astype(float) - c2.astype(float); N = len(d)
    rs = np.random.RandomState(SEED)
    bs = np.array([d[rs.randint(0, N, N)].mean() for _ in range(n)])
    return float(d.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


PAIRS = [("histopath__A2", "histopath__B", "histopath: augment-before-split vs rigorous"),
         ("histopath__A", "histopath__B", "histopath: pre-augmented corpus vs rigorous"),
         ("ndb_ufes__A2", "ndb_ufes__B", "ndb_ufes: augment-before-split vs rigorous"),
         ("clinical__A2", "clinical__B", "clinical: augment-before-split vs rigorous"),
         ("clinical__A", "clinical__B", "clinical: duplicates kept vs removed [CONTROL]"),
         ("he_stained__A", "he_stained__B", "he_stained: shipped split vs leave-one-slide-out")]
rows = []
print("  " + U.row(["comparison", "A", "B", "gap", "95% CI", "p (McNemar)"],
                   [40, 8, 8, 8, 20, 13]))
U.rule()
for ka, kb, desc in PAIRS:
    if ka not in OOF or kb not in OOF:
        continue
    A, B = OOF[ka], OOF[kb]
    pa = MAN[R[ka]["dataset"]]["protocols"][R[ka]["protocol"]]["cv"]
    pb = MAN[R[kb]["dataset"]]["protocols"][R[kb]["protocol"]]["cv"]
    fa = {pa[int(r)]["path"]: i for i, r in enumerate(A["row"]) if int(r) < len(pa)}
    fb = {pb[int(r)]["path"]: i for i, r in enumerate(B["row"]) if int(r) < len(pb)}
    common = sorted(set(fa) & set(fb))
    if len(common) < 30:
        continue
    ia = np.array([fa[c] for c in common]); ib = np.array([fb[c] for c in common])
    ca = A["prob"][ia].argmax(1) == A["y"][ia]
    cb = B["prob"][ib].argmax(1) == B["y"][ib]
    gap, lo, hi = pboot(ca, cb)
    pm, b_, c_ = mcnemar(ca, cb)
    print("  " + U.row([desc[:39], f"{ca.mean()*100:.2f}", f"{cb.mean()*100:.2f}",
                        f"{gap*100:+.2f}", f"[{lo*100:+.2f}, {hi*100:+.2f}]",
                        f"{max(pm,1e-300):.2e}"], [40, 8, 8, 8, 20, 13]))
    rows.append({"a": ka, "b": kb, "desc": desc, "n": len(common),
                 "acc_a": float(ca.mean()), "acc_b": float(cb.mean()), "gap": gap,
                 "ci_lo": lo, "ci_hi": hi, "p_mcnemar": pm,
                 "disc_a": b_, "disc_b": c_})
U.write_text(D_EVAL / "protocol_tests.json", json.dumps(rows, indent=2, default=float))
print(f"\n  Rows use only images present in BOTH protocols, so the sole difference")
print(f"  is how the split was constructed. The clinical control row should be")
print(f"  non-significant: with nothing to leak, protocol makes no difference.")

U.hdr("MASTER TABLE")
print("  " + U.row(["dataset", "pr", "eval", "n", "acc", "bal", "f1", "AUROC",
                    "sens", "spec", "NPV", "MCC", "ECE"],
                   [12, 5, 10, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8]))
U.rule()
for k in sorted(R):
    e = R[k]; m = e.get("test_cal", e["oof_cal"])
    print("  " + U.row([e["dataset"], e["protocol"],
                        "TEST-ens" if "test_cal" in e else "OOF", m["n"],
                        f"{m['accuracy']*100:.2f}", f"{m['balanced_accuracy']*100:.2f}",
                        f"{m['f1_macro']:.4f}", f"{m['auroc']:.4f}",
                        f"{m['sensitivity_macro']:.4f}", f"{m['specificity_macro']:.4f}",
                        f"{m['npv_macro']:.4f}", f"{m['mcc']:.4f}", f"{m['ece']:.4f}"],
                       [12, 5, 10, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8]))
print(f"\n[BLOCK 11 ✓]  {len(R)} arms · {len(BAD)} folds excluded · "
      f"{len(rows)} paired tests")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 12 — SCORECARD vs PUBLISHED LITERATURE
# ═══════════════════════════════════════════════════════════════════════════
import json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import onet_common as U
U.quiet()

R = json.loads((D_EVAL / "results.json").read_text())
TESTS = json.loads((D_EVAL / "protocol_tests.json").read_text()) \
    if (D_EVAL / "protocol_tests.json").exists() else []

TARGETS = [
 ("histopath__A2", "Abd El-Aziz et al. 2025", "LRT-EfficientNet-B3",
  "1,224 H&E → 10,000 augmented, random split",
  {"acc": .9984, "sens": .9793, "spec": .9992, "prec": .9794, "f1": .9793}),
 ("histopath__A", "Manoj Kumar et al. 2025", "Heterogeneous CNN ensemble",
  "3 balanced sets of 5,000 augmented images", {"acc": .980}),
 ("histopath__B", "Warin & Suebnukarn 2024", "meta-analysis of 54 studies",
  "pooled sensitivity / specificity", {"sens": .99, "spec": .97}),
 ("ndb_ufes__A2B", "SciRep 2025", "EfficientNet-B3",
  "NDB-UFES augmented to balance, then split",
  {"acc": .9716, "sens": .9716, "spec": .9858, "f1": .9716}),
 ("ndb_ufes__A2", "SciRep 2025 (3-class reading)", "EfficientNet-B3",
  "same paper, 3-class interpretation", {"acc": .9716, "f1": .9716}),
 ("ndb_ufes__B", "ResNet-50 patient-level", "ResNet-50",
  "NDB-UFES, patient-disjoint split", {"bal": .7333, "prec": .7468}),
 ("clinical__A2", "Alzahrani et al. 2025", "DSLVI-OCLSC",
  "OCI-131 augmented, 1500 epochs",
  {"acc": .9847, "sens": .9773, "spec": .9773, "f1": .9827}),
 ("clinical__A2", "Liu & Bagi 2025", "custom 19-layer CNN",
  "OCI-131, 80/20 random split",
  {"acc": .9954, "sens": .9573, "spec": .9621, "prec": .9634,
   "f1": .9603, "auroc": .6026}),
 ("clinical__B", "Chai et al. 2025 (IAPO)", "Vanilla CNN + IAPO",
  "548 OCI images", {"acc": .925, "sens": .940, "spec": .940,
                     "prec": .920, "f1": .911}),
]
MAP = {"acc": "accuracy", "bal": "balanced_accuracy", "sens": "sensitivity_macro",
       "spec": "specificity_macro", "prec": "precision_macro", "f1": "f1_macro",
       "auroc": "auroc"}
NM = {"acc": "Accuracy", "bal": "Bal. accuracy", "sens": "Sensitivity",
      "spec": "Specificity", "prec": "Precision", "f1": "F1", "auroc": "AUROC"}
NOTE = {"Warin & Suebnukarn 2024": "meta-analysis of 54 studies — not a model comparison",
        "Liu & Bagi 2025": "their 99.54% coexists with AUC 0.6026 in the same paper",
        "SciRep 2025 (3-class reading)": "dysplasia grading — high inter-rater disagreement"}

U.hdr("BLOCK 12 — SCORECARD vs PUBLISHED LITERATURE")
rows, gaps, tb, tc = [], [], 0, 0
for key, paper, method, setup, tgt in TARGETS:
    e = R.get(key)
    if e is None:
        print(f"\n  [{key}] not trained — skipping {paper}"); continue
    m = e.get("test_cal", e["oof_cal"]); ci = e.get("test_ci", e["oof_ci"])
    print(f"\n{'='*100}")
    print(f"{key.replace('__',' / '):<22}  vs  {paper}")
    print(f"{'='*100}")
    print(f"  their method : {method}")
    print(f"  their setup  : {setup}")
    print(f"  our eval     : {'held-out test ensemble' if 'test_cal' in e else 'OOF'}, "
          f"n={m['n']}, {e['n_models']} folds, T={e['temperature']:.3f}")
    if paper in NOTE:
        print(f"  note         : {NOTE[paper]}")
    print()
    print("  " + U.row(["metric", "ours", "95% CI", "theirs", "delta", "verdict"],
                       [15, 10, 21, 10, 10, 18]))
    U.rule()
    nb = nc = 0
    for mk, tv in tgt.items():
        ok = MAP[mk]; ours = m.get(ok)
        if ours is None or not np.isfinite(ours):
            continue
        nc += 1; tc += 1
        lo = ci.get(ok, {}).get("lo"); hi = ci.get(ok, {}).get("hi")
        d = ours - tv
        if lo is not None and lo > tv:
            v = "BEAT (CI-clear)"; nb += 1; tb += 1
        elif d > 0:
            v = "BEAT"; nb += 1; tb += 1
        elif hi is not None and hi > tv:
            v = "TIE (within CI)"
        else:
            v = "below"; gaps.append((key, paper, NM[mk], ours, tv, d))
        cs = f"[{lo*100:.2f}, {hi*100:.2f}]" if lo is not None else "—"
        print("  " + U.row([NM[mk], f"{ours*100:.2f}", cs, f"{tv*100:.2f}",
                            f"{d*100:+.2f}", v], [15, 10, 21, 10, 10, 18]))
    tag = "FULL PANEL CLEARED" if nb == nc and nc else f"{nb}/{nc} metrics beaten"
    print(f"\n  >>> {tag} <<<")
    rows.append({"key": key, "paper": paper, "beat": nb, "n": nc,
                 "full": nb == nc and nc > 0})

U.hdr("HEADLINE")
full = [r for r in rows if r["full"]]
print(f"  metric cells compared : {tc}")
print(f"  cells beaten          : {tb}   ({100*tb/max(1,tc):.0f}%)")
print(f"  papers fully cleared  : {len(full)} of {len(rows)}")
for r in full:
    print(f"      ✓ {r['paper']:<34} ({r['key'].replace('__','/')})")

if gaps:
    U.sub("REMAINING GAPS")
    print("  " + U.row(["arm", "metric", "ours", "theirs", "gap", "assessment"],
                       [20, 14, 8, 8, 8, 40]))
    U.rule()
    for g in sorted(gaps, key=lambda x: x[5]):
        print("  " + U.row([g[0].replace("__", "/"), g[2], f"{g[3]*100:.2f}",
                            f"{g[4]*100:.2f}", f"{g[5]*100:+.2f}",
                            NOTE.get(g[1], "")[:39]], [20, 14, 8, 8, 8, 40]))

if TESTS:
    U.sub("PROTOCOL GAP  (paired on identical images)")
    print("  " + U.row(["comparison", "replication", "rigorous", "gap", "p"],
                       [42, 13, 11, 9, 12]))
    U.rule()
    for t in TESTS:
        print("  " + U.row([t["desc"][:41], f"{t['acc_a']*100:.2f}%",
                            f"{t['acc_b']*100:.2f}%", f"{t['gap']*100:+.2f}",
                            f"{max(t['p_mcnemar'],1e-300):.1e}"], [42, 13, 11, 9, 12]))

U.sub("METRICS WE REPORT THAT NO COMPARED PAPER DOES")
for a in ["Expected calibration error + temperature scaling",
          "Brier score", "Youden J with operating threshold",
          "Specificity at sensitivity ≥95% and ≥99% (the screening number)",
          "LR+ / LR− diagnostic likelihood ratios", "NPV and isolated FNR",
          "Risk-coverage (AURC) and uncertainty referral curves",
          "Leave-one-slide-out validation on OCDC",
          "Dual-protocol gap with paired significance tests",
          "Bootstrap 95% CIs on every reported metric",
          "Pre-registered fold-stability criterion with full disclosure"]:
    print(f"    · {a}")

U.write_text(D_EVAL / "scorecard.json", json.dumps(
    {"rows": rows, "gaps": [list(g) for g in gaps],
     "cells": tc, "beaten": tb}, indent=2, default=float))
print(f"\n[BLOCK 12 ✓]  {tb}/{tc} cells · {len(full)} full panels")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 13 — UNCERTAINTY, CALIBRATION FIGURES, GRAD-CAM
# ═══════════════════════════════════════════════════════════════════════════
import json, warnings, random, gc, os, re
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T
import onet_common as U
U.quiet()
Image.MAX_IMAGE_PIXELS = None

R = json.loads((D_EVAL / "results.json").read_text())
MAN = json.loads(F_MAN.read_text())
CMAP = json.loads(F_CACHE.read_text())["map"]
DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
EPS = 1e-12


def ent(p):
    return -(p * np.log(np.clip(p, EPS, 1))).sum(-1)


def apply_T(p, t):
    z = np.log(np.clip(p, EPS, 1)) / t; z -= z.max(1, keepdims=True)
    q = np.exp(z); return q / q.sum(1, keepdims=True)


def risk_cov(u, cor):
    o = np.argsort(u); c = cor[o]
    cov = np.arange(1, len(c)+1) / len(c)
    risk = 1 - np.cumsum(c) / np.arange(1, len(c)+1)
    return cov, risk, float(np.trapz(risk, cov))


U.hdr("BLOCK 13 — UNCERTAINTY & EXPLAINABILITY")
UNC = {}
for k in sorted(R):
    e = R[k]; T_, K = e["temperature"], e["n_cls"]
    mf, of = D_EVAL / f"{k}__members.npz", D_EVAL / f"{k}__oof.npz"
    if mf.exists():
        z = np.load(mf)
        mem = np.stack([apply_T(m, T_) for m in z["members"]]); y = z["y"]
        src = f"held-out test · {mem.shape[0]}-member ensemble"
    elif of.exists():
        z = np.load(of)
        mem = apply_T(z["prob"], T_)[None, ...]; y = z["y"]
        src = "OOF · one prediction per sample"
    else:
        continue
    mean = mem.mean(0); pred = mean.argmax(1)
    cor = (pred == y).astype(float); conf = mean.max(1)
    tot = ent(mean); alea = ent(mem).mean(0); epi = tot - alea
    cov, risk, aurc = risk_cov(tot, cor)
    _, _, orc = risk_cov(-cor + np.linspace(0, 1e-9, len(cor)), cor)

    print(f"\n  [{k}]  {src} · n={len(y)} · acc {cor.mean()*100:.2f}%")
    print(f"    entropy: correct {tot[cor==1].mean():.4f} vs "
          f"incorrect {tot[cor==0].mean() if (cor==0).any() else float('nan'):.4f}")
    if mem.shape[0] > 1:
        print(f"    aleatoric {alea.mean():.4f} · epistemic {epi.mean():.4f} "
              f"({100*epi.mean()/max(EPS,tot.mean()):.1f}% of total)")
    print(f"    AURC {aurc:.4f} (oracle {orc:.4f} · random {1-cor.mean():.4f})")
    ref = []
    print(f"    referral:  " + "  ".join(
        f"{int(f*100)}%→{cor[np.argsort(tot)[:max(1,int(len(y)*f))]].mean()*100:.2f}%"
        for f in (1.0, 0.9, 0.8, 0.7)))
    for f in (1.0, .9, .8, .7, .6, .5):
        n = max(1, int(len(y)*f))
        ref.append({"retained": f, "n": n,
                    "accuracy": float(cor[np.argsort(tot)[:n]].mean())})

    nb = 15; ed = np.linspace(0, 1, nb+1); bx, by = [], []
    for i in range(nb):
        m_ = (conf > ed[i]) & (conf <= ed[i+1]) if i else (conf >= ed[i]) & (conf <= ed[i+1])
        if m_.sum():
            bx.append(conf[m_].mean()); by.append(cor[m_].mean())
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    ax[0].plot([0, 1], [0, 1], "k--", lw=1); ax[0].plot(bx, by, "o-", lw=2)
    ax[0].set_xlabel("confidence"); ax[0].set_ylabel("accuracy")
    ax[0].set_title(f"{k}\nReliability (ECE={e.get('test_cal', e['oof_cal'])['ece']:.4f})")
    ax[1].plot(cov*100, risk*100, lw=2); ax[1].invert_xaxis()
    ax[1].set_xlabel("coverage (%)"); ax[1].set_ylabel("risk (% error)")
    ax[1].set_title(f"Risk–coverage (AURC={aurc:.4f})")
    ax[2].hist(tot[cor==1], bins=25, alpha=.6, label="correct", density=True)
    if (cor==0).any():
        ax[2].hist(tot[cor==0], bins=25, alpha=.6, label="incorrect", density=True)
    ax[2].set_xlabel("predictive entropy"); ax[2].legend(fontsize=8)
    ax[2].set_title("Uncertainty separation")
    plt.tight_layout(); plt.savefig(D_FIG / f"{k}_uncertainty.png", dpi=130,
                                    bbox_inches="tight"); plt.close(fig)
    UNC[k] = {"n": int(len(y)), "members": int(mem.shape[0]), "source": src,
              "accuracy": float(cor.mean()), "aurc": aurc, "aurc_oracle": orc,
              "entropy_correct": float(tot[cor==1].mean()),
              "entropy_incorrect": float(tot[cor==0].mean()) if (cor==0).any() else None,
              "epistemic": float(epi.mean()), "referral": ref}
U.write_text(D_EVAL / "uncertainty.json", json.dumps(UNC, indent=2, default=float))

U.sub("UNCERTAINTY SUMMARY")
print("  " + U.row(["arm", "n", "members", "acc", "AURC", "H(cor)", "H(err)", "acc@80%"],
                   [22, 8, 9, 8, 9, 9, 9, 9]))
U.rule()
for k in sorted(UNC):
    u = UNC[k]
    a80 = next((r["accuracy"] for r in u["referral"] if abs(r["retained"]-.8) < 1e-6),
               float("nan"))
    he = u["entropy_incorrect"]
    print("  " + U.row([k, u["n"], u["members"], f"{u['accuracy']*100:.2f}",
                        f"{u['aurc']:.4f}", f"{u['entropy_correct']:.4f}",
                        f"{he:.4f}" if he is not None else "n/a",
                        f"{a80*100:.2f}"], [22, 8, 9, 8, 9, 9, 9, 9]))

U.sub("GRAD-CAM")


def last_conv(m_):
    for a in ("stages", "blocks", "layers", "features"):
        m = getattr(m_, a, None)
        if isinstance(m, (nn.Sequential, nn.ModuleList)) and len(m):
            return m[-1]
    cs = [m for m in m_.modules() if isinstance(m, nn.Conv2d)]
    return cs[-1] if cs else None


class CAM:
    def __init__(self, model, layer):
        self.a = self.g = None
        self.h1 = layer.register_forward_hook(lambda m, i, o: setattr(self, "a", o))
        self.h2 = layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "g", go[0]))

    def close(self):
        self.h1.remove(); self.h2.remove()

    def __call__(self, model, x, cls):
        model.zero_grad(set_to_none=True)
        lg = model(x); lg[0, cls].backward()
        a, g = self.a, self.g
        if a.ndim == 3:
            b, n, c = a.shape; s = int(round(n ** .5))
            a = a[:, n-s*s:, :].transpose(1, 2).reshape(b, c, s, s)
            g = g[:, n-s*s:, :].transpose(1, 2).reshape(b, c, s, s)
        w = g.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * a).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0].detach().float().cpu().numpy()
        return cam / (cam.max() + 1e-8), F.softmax(lg.float(), -1)[0].detach().cpu().numpy()


N_PER = 4
made, skipped = 0, []
for cp in sorted(D_W.glob("*.fp16.pt")):
    jid = cp.stem.replace(".fp16", "")
    ds, pn, rp, fd = jid.split("__")
    c = torch.load(cp, map_location="cpu", weights_only=False)
    classes, K, px = c["classes"], c["K"], c["px"]
    proto = MAN[ds]["protocols"][pn]
    pool = [e for e in (proto.get("test") or proto["cv"]) if not e.get("is_aug")
            and os.path.exists(CMAP.get(e["path"], e["path"]))]
    if not pool:
        pool = [e for e in proto["cv"]
                if os.path.exists(CMAP.get(e["path"], e["path"]))]
    per = {ci: [e for e in pool if e["y"] == ci] for ci in range(K)}
    if sum(min(len(v), N_PER) for v in per.values()) < K:
        skipped.append(jid); continue
    model = timm.create_model(c["backbone"], pretrained=False, num_classes=K)
    model.load_state_dict({k: v.float() for k, v in c["state"].items()}, strict=False)
    model = model.to(DEV).eval()
    for p_ in model.parameters():
        p_.requires_grad_(True)
    mean = torch.tensor(c["mean"]).view(1, 3, 1, 1).to(DEV)
    std = torch.tensor(c["std"]).view(1, 3, 1, 1).to(DEV)
    tf = T.Compose([T.Resize(px), T.CenterCrop(px), T.ToTensor()])
    eng = CAM(model, last_conv(model))
    fig, axes = plt.subplots(K*2, N_PER, figsize=(3.3*N_PER, 3.4*K*2), squeeze=False)
    ok = tot = 0
    for ci in range(K):
        cand = list(per[ci]); random.Random(SEED+ci).shuffle(cand)
        for cj in range(N_PER):
            ai, ac = axes[ci*2, cj], axes[ci*2+1, cj]
            ai.axis("off"); ac.axis("off")
            if cj >= len(cand):
                continue
            try:
                img = Image.open(CMAP.get(cand[cj]["path"], cand[cj]["path"])).convert("RGB")
            except Exception:
                continue
            x = ((tf(img).unsqueeze(0).to(DEV)) - mean) / std
            with torch.no_grad():
                p0 = F.softmax(model(x).float(), -1)[0].cpu().numpy()
            pc = int(p0.argmax()); cam, pr = eng(model, x, pc)
            good = pc == cand[cj]["y"]; ok += good; tot += 1
            disp = np.clip(tf(img).numpy().transpose(1, 2, 0), 0, 1)
            ai.imshow(disp); ai.set_title(f"true: {classes[ci]}", fontsize=8)
            ac.imshow(disp); ac.imshow(cam, cmap="jet", alpha=.45)
            ac.set_title(f"pred: {classes[pc]} p={pr[pc]:.2f} "
                         f"{'✓' if good else '✗'}", fontsize=8,
                         color="darkgreen" if good else "crimson")
    eng.close()
    plt.suptitle(f"Grad-CAM — {ds} / protocol {pn}\n"
                 f"top: input · bottom: CAM on predicted class · {ok}/{tot} correct",
                 fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, .97])
    plt.savefig(D_FIG / f"{jid}_gradcam.png", dpi=130, bbox_inches="tight")
    plt.close(fig); made += 1
    print(f"    {jid:<26} {ok}/{tot} shown correct")
    del model, eng; gc.collect(); torch.cuda.empty_cache()
if skipped:
    print(f"    skipped (no resolvable real images): {skipped}")
print(f"\n[BLOCK 13 ✓]  {len(UNC)} uncertainty figures · {made} Grad-CAM figures")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOCK 14 — PAPER TABLES · ARCHIVE · FINAL STATE
# ═══════════════════════════════════════════════════════════════════════════
import os, gc, json, shutil, zipfile, warnings
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")
import numpy as np
import torch
import onet_common as U
U.quiet()

U.hdr(f"BLOCK 14 — FINAL PACKAGE  ·  session {SESSION}")

before = U.disk_free()
U.cleanup(clear_claims=True, log_mb=4)
for p in list(WORK.rglob("*.tmp")) + list(WORK.rglob("__pycache__")):
    shutil.rmtree(p, ignore_errors=True) if p.is_dir() else p.unlink(missing_ok=True)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()
U.kv("disk reclaimed", f"{(U.disk_free()-before)/1e6:.0f} MB")

R = json.loads((D_EVAL / "results.json").read_text())
TESTS = json.loads((D_EVAL / "protocol_tests.json").read_text()) \
    if (D_EVAL / "protocol_tests.json").exists() else []
UNC = json.loads((D_EVAL / "uncertainty.json").read_text()) \
    if (D_EVAL / "uncertainty.json").exists() else {}
SC = json.loads((D_EVAL / "scorecard.json").read_text()) \
    if (D_EVAL / "scorecard.json").exists() else {}


def M(e):
    return e.get("test_cal", e["oof_cal"])


T1 = ["| Dataset | Protocol | n | Accuracy | 95% CI | Bal. acc | F1 | AUROC | "
      "Sens | Spec | NPV | MCC | ECE | Brier | Baseline |", "|" + "---|" * 15]
for k in sorted(R):
    e = R[k]; m = M(e); c = e.get("test_ci", e["oof_ci"]).get("accuracy", {})
    T1.append(f"| {e['dataset']} | {e['protocol']} | {m['n']} | "
              f"{m['accuracy']*100:.2f} | [{c.get('lo',0)*100:.2f}, "
              f"{c.get('hi',0)*100:.2f}] | {m['balanced_accuracy']*100:.2f} | "
              f"{m['f1_macro']:.4f} | {m['auroc']:.4f} | "
              f"{m['sensitivity_macro']:.4f} | {m['specificity_macro']:.4f} | "
              f"{m['npv_macro']:.4f} | {m['mcc']:.4f} | {m['ece']:.4f} | "
              f"{m['brier']:.4f} | {m['majority_baseline']*100:.2f} |")

T2 = ["| Comparison | Replication | Rigorous | Gap (pts) | 95% CI | p (McNemar) |",
      "|" + "---|" * 6]
for t in TESTS:
    T2.append(f"| {t['desc']} | {t['acc_a']*100:.2f}% | {t['acc_b']*100:.2f}% | "
              f"{t['gap']*100:+.2f} | [{t['ci_lo']*100:+.2f}, {t['ci_hi']*100:+.2f}] | "
              f"{max(t['p_mcnemar'],1e-300):.2e} |")

T3 = ["| Dataset | Protocol | Folds trained | Folds used | Excluded | "
      "Single-fold mean ± SD | Ensemble |", "|" + "---|" * 7]
for k in sorted(R):
    e = R[k]
    T3.append(f"| {e['dataset']} | {e['protocol']} | {e['folds_trained']} | "
              f"{e['n_models']} | "
              f"{', '.join(x.split('__')[-1] for x in e['folds_excluded']) or '—'} | "
              f"{e['single_mean']*100:.2f} ± {e['single_sd']*100:.2f} | "
              f"{M(e)['accuracy']*100:.2f} |")

T4 = ["| Dataset | Protocol | Youden thr | Sens | Spec | Spec@Sens≥95% | "
      "Spec@Sens≥99% | LR+ | LR− | FNR |", "|" + "---|" * 10]
for k in sorted(R):
    e = R[k]; m = M(e)
    if "youden_opt" not in m:
        continue
    y = m["youden_opt"]
    T4.append(f"| {e['dataset']} | {e['protocol']} | {y['threshold']:.3f} | "
              f"{y['sensitivity']:.4f} | {y['specificity']:.4f} | "
              f"{m.get('spec_at_sens95', float('nan')):.4f} | "
              f"{m.get('spec_at_sens99', float('nan')):.4f} | "
              f"{m.get('lr_pos', float('nan')):.2f} | "
              f"{m.get('lr_neg', float('nan')):.3f} | "
              f"{m.get('fnr', float('nan')):.4f} |")

T5 = ["| Arm | n | Members | Accuracy | AURC | H(correct) | H(incorrect) | Acc@80% |",
      "|" + "---|" * 8]
for k in sorted(UNC):
    u = UNC[k]
    a80 = next((r["accuracy"] for r in u["referral"]
                if abs(r["retained"]-.8) < 1e-6), float("nan"))
    he = u.get("entropy_incorrect")
    T5.append(f"| {k} | {u['n']} | {u['members']} | {u['accuracy']*100:.2f} | "
              f"{u['aurc']:.4f} | {u['entropy_correct']:.4f} | "
              f"{f'{he:.4f}' if he is not None else 'n/a'} | {a80*100:.2f} |")

doc = [f"# {PROJECT} — Results\n",
       f"Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC\n",
       "## Method\n",
       "Per-dataset specialist ensembles on `convnext_base.fb_in22k_ft_in1k_384` "
       "(ImageNet-22k pretrained), fine-tuned at dataset-specific resolution "
       "(448–512 px) with layer-wise LR decay, stochastic depth, mixup/CutMix, "
       "label smoothing, and EMA weights. Inference uses 2–8× test-time "
       "augmentation. Fold probabilities are averaged and temperature-scaled on "
       "out-of-fold predictions only — no test data enters calibration.\n",
       "## Dual-protocol design\n",
       "**Replication (A / A2)** reproduces each target paper's data handling "
       "exactly, including augmentation applied before splitting where the "
       "original method did so. **Rigorous (B)** removes augmented images, "
       "removes exact duplicates by content hash, and uses group-disjoint or "
       "dataset-native splits. The paired gap between them quantifies how much "
       "published accuracy is attributable to protocol rather than model.\n",
       "## Table 1 — Main results\n", "\n".join(T1), "\n",
       "## Table 2 — Protocol gap (paired, identical images)\n", "\n".join(T2), "\n",
       "\nEach row compares only images present in both protocols; the sole "
       "difference is split construction. Significance by exact McNemar test; "
       "intervals from a 5,000-sample paired bootstrap.\n",
       "## Table 3 — Fold record and exclusions\n", "\n".join(T3), "\n",
       "\nFolds were excluded by a pre-registered stability criterion: a fold "
       "whose held-out accuracy fails to exceed its own majority-class baseline "
       "by 5 points is treated as a failed optimisation, not a result. All "
       "trained folds are listed, including excluded ones.\n"]
if len(T4) > 2:
    doc += ["## Table 4 — Clinical operating points\n", "\n".join(T4), "\n",
            "\nLR+ and LR− are diagnostic likelihood ratios. FNR is the "
            "false-negative rate on the malignant class. Specificity at fixed "
            "high sensitivity is the screening-relevant quantity.\n"]
if len(T5) > 2:
    doc += ["## Table 5 — Uncertainty and selective prediction\n", "\n".join(T5), "\n"]
if SC:
    doc += ["## Benchmark summary\n",
            f"{SC.get('beaten',0)} of {SC.get('cells',0)} metric cells beaten "
            f"({100*SC.get('beaten',0)/max(1,SC.get('cells',1)):.0f}%). "
            f"Papers cleared on every reported metric: "
            + ", ".join(r["paper"] for r in SC.get("rows", []) if r.get("full"))
            + ".\n"]
doc += ["## Reproducibility\n",
        f"- Seed {SEED}; folds, augmentation and sampling seeded deterministically\n",
        "- fp16 with gradient scaling on Tesla T4 (sm_75)\n",
        "- Warmup fixed in absolute steps, capped at 1.2 epochs\n",
        "- Stability guards: LR probe referenced to class-prior entropy, "
        "epoch-3 collapse detection, auto-retry ×3, plateau stop\n",
        "- Artefacts: `evaluation/results.json`, `evaluation/*__oof.npz`, `figures/`\n"]
U.write_text(D_FINAL / "RESULTS.md", "\n".join(doc))
for t, nm in [(T1, "table1_main.md"), (T2, "table2_protocol_gap.md"),
              (T3, "table3_folds.md"), (T4, "table4_operating_points.md"),
              (T5, "table5_uncertainty.md")]:
    if len(t) > 2:
        U.write_text(D_FINAL / nm, "\n".join(t))
for s in D_EVAL.glob("*.json"):
    shutil.copy(s, D_FINAL / s.name)
U.kv("tables written", len(list(D_FINAL.glob("*.md"))))

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
BUNDLE = WORK / f"{PROJECT}_results_{stamp}.zip"
n = 0
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for d, arc in [(D_FINAL, "final"), (D_EVAL, "evaluation"), (D_FIG, "figures"),
                   (D_DONE, "fold_records")]:
        if not d.exists():
            continue
        for f in sorted(d.rglob("*")):
            if f.is_file() and f.suffix not in (".pt", ".tmp"):
                z.write(f, f"{arc}/{f.relative_to(d)}"); n += 1
    for f, arc in [(F_MAN, "config/manifest.json"), (F_CONFIG, "config/config.json"),
                   (F_LEDGER, "config/ledger.json")]:
        if f.exists():
            z.write(f, arc); n += 1
U.kv("archive", f"{BUNDLE.name} · {n} files · {BUNDLE.stat().st_size/1e6:.1f} MB")

U.sub("FINAL STATE")
print("  " + U.row(["path", "files", "size"], [34, 8, 12]))
U.rule()
tot = 0
for d, nm in [(D_STATE, "state/"), (D_PRED, "preds/"), (D_W, "weights/"),
              (D_EVAL, "evaluation/"), (D_FIG, "figures/"), (D_FINAL, "final/")]:
    fs = [f for f in d.rglob("*") if f.is_file()]
    s = sum(f.stat().st_size for f in fs); tot += s
    print("  " + U.row([nm, len(fs), f"{s/1e6:.1f} MB"], [34, 8, 12]))
U.rule()
print("  " + U.row(["TOTAL", "", f"{tot/1e6:.0f} MB"], [34, 8, 12]))
U.kv("disk free", U.gb(U.disk_free()))

if SC:
    U.hdr("PROJECT COMPLETE")
    print(f"  {SC.get('beaten',0)}/{SC.get('cells',0)} metric cells beaten")
    full = [r["paper"] for r in SC.get("rows", []) if r.get("full")]
    print(f"  {len(full)} paper(s) cleared on every reported metric:")
    for p in full:
        print(f"      ✓ {p}")
print(f"\n  Download: {BUNDLE.name}  ·  read: final/RESULTS.md")
print(f"  Before closing: Persistence = 'Files only' → Save Version.")
print(f"\n[BLOCK 14 ✓]")

In [ ]:
# ============================================================================
# BLOCK 15 — Q1 SETUP: LIBRARY + ANALYSIS PLAN (fixed before training) + ENVIRONMENT
# ============================================================================
import os, sys, json, time, importlib, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
(Q1_ROOT / "lib").mkdir(parents=True, exist_ok=True)
Q1LIB_SRC = r'''"""
q1lib.py - shared library for the OralNet Q1 leakage study.
Written to disk by Block 15.  Do not edit by hand: re-run Block 15 to refresh it.

Sections
  1  paths, printing, json, hashing, session clock
  2  dataset registry and file parsing
  3  image store, exact hashes, perceptual hashes
  4  torch helpers: models, embeddings, GPU augmentation (siblings, train, eval)
  5  near-duplicate graph, union-find, index-order segmentation, parent recovery
  6  split builder: clean holdout, leaky / controlled arms, matching, leak stats, disjointness tests
  7  metrics, calibration, operating points, cluster bootstrap, GEE, ICC, Holm, mixed model
"""
import os, sys, io, re, json, time, math, hashlib, random, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

LIB_VERSION = "q1-1.0"

# =====================================================================================
# 1  paths, printing, json, hashing, session clock
# =====================================================================================
ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
INPUT = Path(os.environ.get("Q1_INPUT", "/kaggle/input"))
OFFLINE = os.environ.get("Q1_OFFLINE", "0") == "1"      # tests only: no pretrained downloads
TINY = os.environ.get("Q1_TINY", "0") == "1"            # tests only: tiny images and models
STORE_PX = 64 if TINY else 288

D = {k: ROOT / v for k, v in dict(
    lib="lib", plan="plan", index="index", store="store", jobs="jobs", claims="jobs/claims",
    fail="jobs/fail", runs="runs", superseded="runs_superseded", ckpt="ckpt", logs="logs",
    eval="eval", figs="figs", tables="tables", release="release", state="state", code="code").items()}


def ensure_dirs():
    for p in D.values():
        p.mkdir(parents=True, exist_ok=True)


W = 100


def hdr(t):
    print("=" * W)
    print(t)
    print("=" * W)


def sub(t):
    print()
    print("-" * W)
    print(t)
    print("-" * W)


def kv(k, v, w=30):
    print(f"  {k:<{w}} {v}")


def table(headers, rows, widths=None, indent=2):
    cols = len(headers)
    if widths is None:
        widths = []
        for i in range(cols):
            m = max([len(str(headers[i]))] + [len(str(r[i])) for r in rows]) if rows else len(str(headers[i]))
            widths.append(min(max(m + 2, 6), 44))
    pad = " " * indent
    print(pad + "".join(f"{str(h):<{w}}" for h, w in zip(headers, widths)))
    print(pad + "-" * min(sum(widths), W - indent))
    for r in rows:
        print(pad + "".join(f"{str(c):<{w}}" for c, w in zip(r, widths)))


def hms(sec):
    sec = max(0, int(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h}h{m:02d}m" if h else (f"{m}m{s:02d}s" if m else f"{s}s")


def pct(x, d=2):
    return "—" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{100 * x:.{d}f}"


def _jdef(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, Path):
        return str(o)
    if isinstance(o, set):
        return sorted(o)
    raise TypeError(str(type(o)))


def _clean(o):
    if isinstance(o, float):
        return o if math.isfinite(o) else None
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.floating):
        v = float(o)
        return v if math.isfinite(v) else None
    return o


def jload(p, default=None):
    p = Path(p)
    if not p.exists():
        return default
    try:
        return json.loads(p.read_text())
    except Exception:
        return default


def jsave(p, obj, indent=1):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + f".tmp{os.getpid()}")
    tmp.write_text(json.dumps(_clean(obj), indent=indent, default=_jdef))
    os.replace(tmp, p)


def shash(*parts):
    h = hashlib.blake2b("|".join(map(str, parts)).encode(), digest_size=8)
    return int.from_bytes(h.digest(), "little") & ((1 << 62) - 1)


def fingerprint(obj):
    return hashlib.sha1(json.dumps(_clean(obj), sort_keys=True, default=_jdef).encode()).hexdigest()[:12]


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def code_hashes():
    out = {}
    for n in ("q1lib.py", "q1worker.py"):
        p = D["lib"] / n
        out[n] = sha256_file(p)[:16] if p.exists() else None
    return out


def keep_source(tag):
    """Save the source of the notebook cell that is running now to q1/code/block<tag>.py (IPython only), so the
    release package contains the exact code behind every table and figure."""
    try:
        from IPython import get_ipython
        ip = get_ipython()
        src = ip.user_ns.get("In", [""])[-1] if ip is not None else ""
        if src and f"BLOCK {tag}" in src[:600]:
            D["code"].mkdir(parents=True, exist_ok=True)
            (D["code"] / f"block{tag}.py").write_text(src)
            return True
    except Exception:
        pass
    return False


def disk_free_gb(p=None):
    import shutil
    p = p or (ROOT if ROOT.exists() else Path("/"))
    return shutil.disk_usage(p).free / 1e9


def kernel_start():
    """Start time of the current Python process (the notebook kernel when called from a cell)."""
    try:
        import psutil
        return psutil.Process(os.getpid()).create_time()
    except Exception:
        return time.time()


def session_clock(budget_min=660):
    """Session start = kernel start (first call per kernel is recorded).  Returns dict with deadline."""
    st = jload(D["state"] / "session.json", {}) or {}
    ks = kernel_start()
    if abs(st.get("kernel_start", 0) - ks) > 2:
        st = dict(kernel_start=ks, n=int(st.get("n", 0)) + 1, first_seen=time.time())
        jsave(D["state"] / "session.json", st)
    st["deadline"] = ks + budget_min * 60
    st["elapsed"] = time.time() - ks
    st["remaining"] = st["deadline"] - time.time()
    return st


def plan():
    p = jload(D["plan"] / "plan.json")
    if p is None:
        raise RuntimeError("plan.json missing - run Block 15 first")
    return p


# =====================================================================================
# 2  dataset registry and file parsing
# =====================================================================================
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
DATASETS = ["he_stained", "histopath", "ndb_ufes", "clinical", "breakhis"]
META = {
    "he_stained": dict(K=2, classes=["non_tumor", "tumor"], pos=1, modality="he", group="slide",
                       title="OCDC H&E patches (7 slides)"),
    "histopath": dict(K=2, classes=["Normal", "OSCC"], pos=1, modality="he", group="source image",
                      title="Oral histopathology, Normal vs OSCC (Kaggle mirror)"),
    "ndb_ufes": dict(K=3, classes=["Healthy", "Leukoplakia", "OSCC"], pos=2, modality="he",
                     group="recovered source image", title="NDB-UFES patches (Kaggle mirror)"),
    "clinical": dict(K=2, classes=["non-cancer", "cancer"], pos=1, modality="photo", group="duplicate cluster",
                     title="Oral cancer lip and tongue photographs"),
    "breakhis": dict(K=2, classes=["benign", "malignant"], pos=1, modality="he", group="patient",
                     title="BreakHis breast H&E (patient IDs)"),
}
SHORT = {"he_stained": "OCDC", "histopath": "histo", "ndb_ufes": "NDB", "clinical": "clin", "breakhis": "BrkHs"}


def load_registry():
    """Extension datasets (added by analysis-plan amendments) are registered in index/_registry.json.
    They get META / SHORT entries but are never part of DATASETS, so the pre-registered analyses are unchanged."""
    reg = jload(D["index"] / "_registry.json", {}) or {}
    for name, m in reg.items():
        META[name] = dict(m["meta"])
        SHORT[name] = m["short"]
    return reg


def ext_datasets():
    return [d for d in (jload(D["index"] / "_registry.json", {}) or {}) if (D["index"] / f"{d}.json").exists()]


load_registry()

NDB_CLASSES = {"healthy": 0, "leukoplakia": 1, "oral squamous cell carcinoma": 2}
RE_HIST_ORIG = re.compile(r"^(oscc|normal)_(\d+)x_(\d+)\.[a-z]+$", re.I)
RE_HIST_AUG = re.compile(r"^aug_(\d+)_(\d+)\.[a-z]+$", re.I)
RE_BH = re.compile(r"^SOB_([BM])_([A-Za-z]+)-(\d+)-([A-Za-z0-9]+)-(\d+)-(\d+)\.png$", re.I)


def classify_file(p):
    p = Path(p)
    name = p.name
    parts = [x.lower() for x in p.parts]
    par = parts[-2] if len(parts) > 1 else ""
    gpar = parts[-3] if len(parts) > 2 else ""
    m = RE_BH.match(name)
    if m:
        return "breakhis", dict(path=str(p), label=1 if m.group(1).upper() == "M" else 0,
                                patient=f"{m.group(3)}-{m.group(4)}", subtype=m.group(2).upper(),
                                mag=m.group(5) + "x", seq=int(m.group(6)))
    if par == "01-original" and ("01-roi" in parts or "02-non_roi" in parts):
        tum = "01-roi" in parts
        i = parts.index("01-roi" if tum else "02-non_roi")
        native = "test" if "testing" in parts else ("train" if "training" in parts else "")
        return "he_stained", dict(path=str(p), label=int(tum), slide=p.parts[i - 1], native=native)
    if par in NDB_CLASSES and re.match(r"^p\d+\.[a-z]+$", name, re.I):
        return "ndb_ufes", dict(path=str(p), label=NDB_CLASSES[par], pidx=int(re.findall(r"\d+", name)[0]))
    if par in ("oscc", "normal") and gpar in ("train", "val", "test"):
        lab = 1 if par == "oscc" else 0
        m = RE_HIST_ORIG.match(name)
        if m:
            return "histopath", dict(path=str(p), label=lab, orig=True, native=gpar, mag=m.group(2) + "x",
                                     pidx=int(m.group(3)))
        m = RE_HIST_AUG.match(name)
        a, b = (int(m.group(1)), int(m.group(2))) if m else (-1, -1)
        return "histopath", dict(path=str(p), label=lab, orig=False, native=gpar, aug_a=a, aug_b=b)
    if par in ("cancer", "non-cancer") and gpar == "oralcancer":
        return "clinical", dict(path=str(p), label=1 if par == "cancer" else 0)
    return None


def walk_input(root=None):
    root = Path(root or INPUT)
    found = defaultdict(list)
    n_img = 0
    for dp, dn, fn in os.walk(root):
        dn[:] = sorted(d for d in dn if not d.startswith("."))
        for f in sorted(fn):
            if os.path.splitext(f)[1].lower() not in IMG_EXT:
                continue
            n_img += 1
            r = classify_file(Path(dp) / f)
            if r:
                found[r[0]].append(r[1])
    for ds in found:
        found[ds].sort(key=lambda r: r["path"])
    return dict(found), n_img


# =====================================================================================
# 3  image store, exact hashes, perceptual hashes
# =====================================================================================
def _load_one(args):
    path, px = args
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None
    try:
        b = open(path, "rb").read()
    except Exception:
        return None, np.zeros((px, px, 3), np.uint8), (0, 0), "", False, 0
    md5 = hashlib.md5(b).hexdigest()
    try:
        im = Image.open(io.BytesIO(b))
        w, h = im.size
        mode = im.mode
        if im.format == "JPEG":
            im.draft("RGB", (px * 2, px * 2))
        im = im.convert("RGB").resize((px, px), Image.BILINEAR, reducing_gap=2.0)
        arr = np.asarray(im, dtype=np.uint8).copy()
        ok = True
    except Exception:
        arr, w, h, mode, ok = np.zeros((px, px, 3), np.uint8), 0, 0, "", False
    return md5, arr, (w, h), mode, ok, len(b)


def build_store(ds, recs, px=None, workers=4, log_every=2000):
    """Decode every file once: MD5 + resized uint8 array into store/<ds>.npy (memmap)."""
    from multiprocessing import get_context
    px = px or STORE_PX
    out = D["store"] / f"{ds}.npy"
    tmp = D["store"] / f"{ds}.npy.partial"
    n = len(recs)
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(n, px, px, 3))
    meta = []
    t0 = time.time()
    ctx = get_context("fork")
    with ctx.Pool(workers) as pool:
        for i, res in enumerate(pool.imap(_load_one, [(r["path"], px) for r in recs], chunksize=16)):
            md5, a, (w, h), mode, ok, nb = res
            arr[i] = a
            meta.append(dict(md5=md5 or f"missing{i}", w=w, h=h, mode=mode, ok=bool(ok), bytes=nb))
            if log_every and (i + 1) % log_every == 0:
                print(f"    {ds:<11} {i + 1}/{n}  ({time.time() - t0:.0f}s)", flush=True)
    arr.flush()
    del arr
    os.replace(tmp, out)
    return meta, time.time() - t0


def open_store(ds):
    return np.load(D["store"] / f"{ds}.npy", mmap_mode="r")


_POP8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)


def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return _POP8[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)


def perceptual_hashes(store, bs=512):
    """pHash (32x32 DCT, 8x8 low band vs median) and dHash (9x8 gradient) as uint64."""
    from scipy.fft import dctn
    import torch
    import torch.nn.functional as F
    n = store.shape[0]
    ph = np.zeros(n, np.uint64)
    dh = np.zeros(n, np.uint64)
    w64 = (1 << np.arange(64, dtype=np.uint64)).astype(np.uint64)
    for s in range(0, n, bs):
        x = torch.from_numpy(np.asarray(store[s:s + bs])).float()
        g = (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]).unsqueeze(1)
        g32 = F.adaptive_avg_pool2d(g, (32, 32))[:, 0].numpy()
        c = dctn(g32, type=2, norm="ortho", axes=(1, 2))[:, :8, :8].reshape(len(g32), 64)
        med = np.median(c[:, 1:], axis=1, keepdims=True)
        bits = (c > med).astype(np.uint64)
        ph[s:s + bs] = (bits * w64).sum(1).astype(np.uint64)
        g98 = F.adaptive_avg_pool2d(g, (8, 9))[:, 0].numpy()
        bits = (g98[:, :, 1:] > g98[:, :, :-1]).reshape(len(g98), 64).astype(np.uint64)
        dh[s:s + bs] = (bits * w64).sum(1).astype(np.uint64)
    return ph, dh


def hamming_pairs(ha, hb=None, max_d=12, chunk=512):
    """All pairs with Hamming distance <= max_d.  Within one set (hb None, i<j) or across two sets."""
    ha = np.asarray(ha, np.uint64)
    same = hb is None
    hb = ha if same else np.asarray(hb, np.uint64)
    I, J, Dd = [], [], []
    for s in range(0, len(ha), chunk):
        x = ha[s:s + chunk, None] ^ hb[None, :]
        d = popcount64(x)
        ii, jj = np.nonzero(d <= max_d)
        ii = ii + s
        if same:
            keep = jj > ii
            ii, jj = ii[keep], jj[keep]
        I.append(ii)
        J.append(jj)
        Dd.append(d[ii - s, jj])
    if not I:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0, int)
    return np.concatenate(I), np.concatenate(J), np.concatenate(Dd).astype(int)


def gray_small(store, idx, px=128):
    import torch
    import torch.nn.functional as F
    x = torch.from_numpy(np.asarray(store[np.sort(np.unique(idx))])).float()
    g = (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]).unsqueeze(1)
    g = F.adaptive_avg_pool2d(g, (px, px))[:, 0].numpy()
    return {int(i): g[k] for k, i in enumerate(np.sort(np.unique(idx)))}


def ssim_pairs(store_a, ia, store_b, ib, px=128, max_pairs=300000):
    from skimage.metrics import structural_similarity
    ia, ib = np.asarray(ia), np.asarray(ib)
    if len(ia) > max_pairs:
        ia, ib = ia[:max_pairs], ib[:max_pairs]
    if len(ia) == 0:
        return np.zeros(0)
    if store_b is store_a:
        ga = gray_small(store_a, np.concatenate([ia, ib]), px)
        gb_ = ga
    else:
        ga = gray_small(store_a, ia, px)
        gb_ = gray_small(store_b, ib, px)
    out = np.zeros(len(ia))
    for k, (i, j) in enumerate(zip(ia, ib)):
        out[k] = structural_similarity(ga[int(i)], gb_[int(j)], data_range=255.0)
    return out


# =====================================================================================
# 4  torch helpers: models, embeddings, GPU augmentation
# =====================================================================================
EMB_MODELS = {"dinov2_s": "vit_small_patch14_dinov2.lvd142m", "convnext_t": "convnext_tiny.fb_in22k_ft_in1k",
              "phikon": "owkin/phikon"}
BACKBONES = {  # name -> (short label, parameter count in millions, default img/s on one T4 at 224 px, fp16)
    "convnext_tiny.fb_in22k_ft_in1k": ("ConvNeXt-T", 28.6, 300),
    "resnet50.a1_in1k": ("ResNet-50", 25.6, 380),
    "vit_small_patch16_224.augreg_in21k_ft_in1k": ("ViT-S/16", 22.1, 380),
    "efficientnet_b0.ra_in1k": ("EffNet-B0", 5.3, 520),
    "convnext_base.fb_in22k_ft_in1k": ("ConvNeXt-B", 88.6, 100),
    "efficientnet_b3.ra2_in1k": ("EffNet-B3", 12.2, 170),
    "resnet18": ("ResNet-18", 11.7, 900),
}


def tiny_name(name):
    return "resnet18" if TINY else name


def make_model(name, K, img=224, drop_path=0.0, head="linear", pretrained=None):
    """Returns (model, head_params, dense_layer_for_regularisation or None)."""
    import timm
    import torch.nn as nn
    name = tiny_name(name)
    pretrained = (not OFFLINE) if pretrained is None else pretrained
    kw = {}
    if any(t in name for t in ("vit_", "dinov2", "deit", "eva")):
        kw["img_size"] = img
    if drop_path:
        kw["drop_path_rate"] = drop_path
    if head == "scirep":
        base = timm.create_model(name, pretrained=pretrained, num_classes=0, **kw)
        nf = base.num_features
        hd = nn.Sequential(nn.BatchNorm1d(nf), nn.Linear(nf, 256), nn.ReLU(inplace=True), nn.Dropout(0.45),
                           nn.Linear(256, K))

        class _Wrap(nn.Module):
            def __init__(s, b, h):
                super().__init__()
                s.base, s.head = b, h
                s.pretrained_cfg = getattr(b, "pretrained_cfg", {})

            def forward(s, x):
                return s.head(s.base(x))

        m = _Wrap(base, hd)
        return m, list(hd.parameters()), hd[1]
    m = timm.create_model(name, pretrained=pretrained, num_classes=K, **kw)
    return m, list(m.get_classifier().parameters()), None


def norm_consts(model, device):
    import torch
    cfg = getattr(model, "pretrained_cfg", None) or {}
    mean = cfg.get("mean", (0.485, 0.456, 0.406))
    std = cfg.get("std", (0.229, 0.224, 0.225))
    return (torch.tensor(mean, device=device).view(1, 3, 1, 1), torch.tensor(std, device=device).view(1, 3, 1, 1))


POLICIES = {
    # Abd El-Aziz 2025: rotation 10 deg, width/height shift 10%, shear 10%, zoom 10%, horizontal flip,
    # brightness 0.2-1.2, applied before the split (Keras ImageDataGenerator style, nearest fill)
    "abdelaziz": dict(rot=10, shift=0.10, shear=0.10, zoom=0.10, hflip=True, vflip=False,
                      bright=(0.2, 1.2), contrast=None, sat=None),
    # generic Keras-style policy used for the clinical photographs (augment-before-split replication)
    "keras_generic": dict(rot=20, shift=0.10, shear=0.10, zoom=0.10, hflip=True, vflip=False,
                          bright=(0.8, 1.2), contrast=None, sat=None),
    # H&E policy for the NDB-UFES dose-response arm: flips, rotation 30 deg, mild zoom and colour
    "he_d4": dict(rot=30, shift=0.0, shear=0.0, zoom=0.10, hflip=True, vflip=True,
                  bright=(0.9, 1.1), contrast=(0.9, 1.1), sat=(0.9, 1.1)),
}
TRAIN_PROFILES = {
    "std_he": dict(rrc=(0.7, 1.0), hflip=True, vflip=True, rot90=True, rot=0, jitter=0.1),
    "std_photo": dict(rrc=(0.7, 1.0), hflip=True, vflip=False, rot90=False, rot=0, jitter=0.1),
    "scirep_ndb": dict(rrc=None, hflip=True, vflip=True, rot90=False, rot=30, jitter=0.0),
    "scirep_hist": dict(rrc=None, hflip=True, vflip=False, rot90=False, rot=0, jitter=0.0),
    "none": dict(rrc=None, hflip=False, vflip=False, rot90=False, rot=0, jitter=0.0),
}
IDENT = np.array([0, 0, 0, 0, 1, 0, 0, 1, 1, 1], np.float32)


def sibling_params(ds, rows, copies, policy):
    """Deterministic augmentation parameters of sibling copy c of store row r (c = 0 is the original)."""
    n = len(rows)
    out = np.tile(IDENT, (n, 1))
    if policy is None:
        return out
    P = POLICIES[policy]
    salt = shash("sib", ds, policy)
    for k in range(n):
        c = int(copies[k])
        if c == 0:
            continue
        g = np.random.default_rng(shash(salt, int(rows[k]), c))
        a = g.uniform(-P["rot"], P["rot"])
        tx = g.uniform(-P["shift"], P["shift"]) if P["shift"] else 0.0
        ty = g.uniform(-P["shift"], P["shift"]) if P["shift"] else 0.0
        sh = g.uniform(-P["shear"], P["shear"]) if P["shear"] else 0.0
        z = g.uniform(1 - P["zoom"], 1 + P["zoom"]) if P["zoom"] else 1.0
        hf = float(g.random() < 0.5) if P["hflip"] else 0.0
        vf = float(g.random() < 0.5) if P["vflip"] else 0.0
        b = g.uniform(*P["bright"]) if P["bright"] else 1.0
        ct = g.uniform(*P["contrast"]) if P["contrast"] else 1.0
        st = g.uniform(*P["sat"]) if P["sat"] else 1.0
        out[k] = (a, tx, ty, sh, z, hf, vf, b, ct, st)
    return out


def _gray(x):
    return (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])


def apply_sibling(x, prm):
    """x: float [B,3,H,W] in [0,1]; prm: float tensor [B,10].  Geometric + photometric sibling transform."""
    import torch
    import torch.nn.functional as F
    B = x.shape[0]
    a = prm[:, 0] * (math.pi / 180.0)
    c, s = torch.cos(a), torch.sin(a)
    sh, z = prm[:, 3], prm[:, 4]
    fx = 1 - 2 * prm[:, 5]
    fy = 1 - 2 * prm[:, 6]
    th = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
    # output -> input:  R(a) . Shear . Zoom . Flip
    m00, m01 = c * z * fx, (c * sh - s) * z * fy
    m10, m11 = s * z * fx, (s * sh + c) * z * fy
    th[:, 0, 0], th[:, 0, 1], th[:, 1, 0], th[:, 1, 1] = m00, m01, m10, m11
    th[:, 0, 2], th[:, 1, 2] = 2 * prm[:, 1], 2 * prm[:, 2]
    grid = F.affine_grid(th, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    x = x * prm[:, 7].view(B, 1, 1, 1)
    m = _gray(x).mean((2, 3), keepdim=True)
    x = (x - m) * prm[:, 8].view(B, 1, 1, 1) + m
    g = _gray(x)
    x = g + (x - g) * prm[:, 9].view(B, 1, 1, 1)
    return x.clamp(0, 1)


def train_aug(x, out, prof, gen):
    """Random resized crop / flips / rotation / colour jitter on GPU, reproducible through `gen`."""
    import torch
    import torch.nn.functional as F
    P = TRAIN_PROFILES[prof]
    B, dev = x.shape[0], x.device

    def u(n=B):
        return torch.rand(n, generator=gen, device=dev)

    if P["rrc"]:
        lo, hi = P["rrc"]
        area = lo + (hi - lo) * u()
        r = torch.exp((u() * 2 - 1) * math.log(4 / 3))
        w = torch.sqrt(area * r).clamp(max=1.0)
        h = torch.sqrt(area / r).clamp(max=1.0)
        cx = (u() * 2 - 1) * (1 - w)
        cy = (u() * 2 - 1) * (1 - h)
    else:
        w = torch.ones(B, device=dev)
        h = torch.ones(B, device=dev)
        cx = torch.zeros(B, device=dev)
        cy = torch.zeros(B, device=dev)
        u(), u(), u(), u()
    fx = torch.where((u() < 0.5) & torch.tensor(P["hflip"], device=dev), -1.0, 1.0)
    fy = torch.where((u() < 0.5) & torch.tensor(P["vflip"], device=dev), -1.0, 1.0)
    ang = torch.zeros(B, device=dev)
    k90 = torch.floor(u() * 4)
    if P["rot90"]:
        ang = ang + k90 * (math.pi / 2)
    rr = (u() * 2 - 1) * (P["rot"] * math.pi / 180.0)
    ang = ang + rr
    c, s = torch.cos(ang), torch.sin(ang)
    th = torch.zeros(B, 2, 3, device=dev)
    th[:, 0, 0], th[:, 0, 1] = w * fx * c, -w * fx * s
    th[:, 1, 0], th[:, 1, 1] = h * fy * s, h * fy * c
    th[:, 0, 2], th[:, 1, 2] = cx, cy
    grid = F.affine_grid(th, [B, 3, out, out], align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    j = P["jitter"]
    bj, cj, sj = 1 + j * (u() * 2 - 1), 1 + j * (u() * 2 - 1), 1 + j * (u() * 2 - 1)
    if j > 0:
        x = x * bj.view(B, 1, 1, 1)
        m = _gray(x).mean((2, 3), keepdim=True)
        x = (x - m) * cj.view(B, 1, 1, 1) + m
        g = _gray(x)
        x = g + (x - g) * sj.view(B, 1, 1, 1)
    return x.clamp(0, 1)


def eval_prep(x, out):
    import torch.nn.functional as F
    if x.shape[-1] == out and x.shape[-2] == out:
        return x
    return F.interpolate(x, size=(out, out), mode="bilinear", antialias=True, align_corners=False)


class Feeder:
    """Keeps the needed store rows of one or more datasets in pinned CPU memory and assembles GPU batches."""

    def __init__(self, rows_by_ds, device, pin=True):
        import torch
        self.dev = device
        self.X, self.rows = {}, {}
        for ds, rows in rows_by_ds.items():
            rows = np.unique(np.asarray(rows, np.int64))
            st = open_store(ds)
            arr = np.ascontiguousarray(st[rows])
            t = torch.from_numpy(arr)
            if pin and device.type == "cuda":
                try:
                    t = t.pin_memory()
                except Exception:
                    pass
            self.X[ds], self.rows[ds] = t, rows

    def batch(self, ds, rows, copies, sib_prm, policy):
        import torch
        pos = np.searchsorted(self.rows[ds], rows)
        x = self.X[ds].index_select(0, torch.from_numpy(pos)).to(self.dev, non_blocking=True)
        x = x.permute(0, 3, 1, 2).float().div_(255.0)
        if policy is not None:
            m = np.nonzero(np.asarray(copies) > 0)[0]
            if len(m):
                mt = torch.from_numpy(m).to(self.dev)
                x[mt] = apply_sibling(x[mt], torch.from_numpy(sib_prm[m]).to(self.dev))
        return x


def _hf_vit_cls(repo, img):
    """Hugging Face ViT (e.g. Phikon) returning the CLS token, with timm-style normalisation constants."""
    import torch
    from transformers import ViTModel, ViTConfig

    class ViTCLS(torch.nn.Module):
        def __init__(self, vit):
            super().__init__()
            self.vit = vit
            self.pretrained_cfg = {"mean": (0.485, 0.456, 0.406), "std": (0.229, 0.224, 0.225)}

        def forward(self, x):
            return self.vit(pixel_values=x, interpolate_pos_encoding=True).last_hidden_state[:, 0]

    if OFFLINE or TINY:
        vit = ViTModel(ViTConfig(hidden_size=96, num_hidden_layers=2, num_attention_heads=3, intermediate_size=192,
                                 image_size=img, patch_size=8), add_pooling_layer=False)
    else:
        vit = ViTModel.from_pretrained(repo, add_pooling_layer=False)
    return ViTCLS(vit)


@__import__("functools").lru_cache(maxsize=4)
def _emb_model(key, img, device_str):
    import torch
    if key == "phikon":
        m = _hf_vit_cls(EMB_MODELS["phikon"], img).eval().to(torch.device(device_str))
        if device_str.startswith("cuda"):
            m.half()
        return m
    import timm
    name = tiny_name(EMB_MODELS.get(key, key))
    kw = {"img_size": img} if any(t in name for t in ("vit_", "dinov2")) else {}
    m = timm.create_model(name, pretrained=not OFFLINE, num_classes=0, **kw)
    m.eval().to(torch.device(device_str))
    if device_str.startswith("cuda"):
        m.half()
    return m


def embed(ds, rows, key="dinov2_s", img=224, bs=256, copies=None, policy=None, views="none", device=None):
    """L2-normalised embeddings of store rows (optionally sibling copies).  views='d4' returns [n,8,dim]."""
    import torch
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    img = 56 if TINY else img
    m = _emb_model(key, img, device)
    dev = torch.device(device)
    mean, std = norm_consts(m, dev)
    rows = np.asarray(rows, np.int64)
    copies = np.zeros(len(rows), int) if copies is None else np.asarray(copies)
    prm = sibling_params(ds, rows, copies, policy)
    st = open_store(ds)
    outs = []
    with torch.no_grad():
        for s in range(0, len(rows), bs):
            r = rows[s:s + bs]
            x = torch.from_numpy(np.ascontiguousarray(st[np.sort(r)])).to(dev)
            order = np.argsort(np.argsort(r))
            x = x[torch.from_numpy(order).to(dev)].permute(0, 3, 1, 2).float() / 255.0
            if policy is not None and (copies[s:s + bs] > 0).any():
                x = apply_sibling(x, torch.from_numpy(prm[s:s + bs]).to(dev))
            x = (eval_prep(x, img) - mean) / std
            vs = [x] if views == "none" else [torch.rot90(xx, k, (2, 3)) for xx in (x, x.flip(3)) for k in range(4)]
            fs = []
            for v in vs:
                v = v.half() if device.startswith("cuda") else v
                f = m(v).float()
                fs.append(torch.nn.functional.normalize(f, dim=1).cpu().numpy())
            outs.append(fs[0] if views == "none" else np.stack(fs, 1))
    return np.concatenate(outs, 0).astype(np.float32)


# =====================================================================================
# 5  near-duplicate graph, union-find, index-order segmentation, parent recovery
# =====================================================================================
class UF:
    def __init__(self, n):
        self.p = np.arange(n)

    def find(self, a):
        p = self.p
        r = a
        while p[r] != r:
            r = p[r]
        while p[a] != r:
            p[a], a = r, p[a]
        return r

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            lo, hi = (ra, rb) if ra < rb else (rb, ra)
            self.p[hi] = lo

    def roots(self):
        return np.array([self.find(i) for i in range(len(self.p))])


def union_labels(n, pairs=(), label_arrays=(), subset=None):
    """Connected components over records: explicit pairs plus 'same value in any label array'."""
    uf = UF(n)
    for a, b in pairs:
        uf.union(int(a), int(b))
    idx = np.arange(n) if subset is None else np.asarray(subset)
    for lab in label_arrays:
        first = {}
        for i in idx:
            v = int(lab[i])
            if v < 0:
                continue
            if v in first:
                uf.union(int(i), first[v])
            else:
                first[v] = int(i)
    return uf.roots()


def cos_pairs(ea, eb=None, thr=0.95, chunk=2048):
    same = eb is None
    eb = ea if same else eb
    I, J, S = [], [], []
    for s in range(0, len(ea), chunk):
        sim = ea[s:s + chunk] @ eb.T
        ii, jj = np.nonzero(sim >= thr)
        ii = ii + s
        if same:
            k = jj > ii
            ii, jj = ii[k], jj[k]
        I.append(ii)
        J.append(jj)
        S.append(sim[ii - s, jj])
    if not I:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0)
    return np.concatenate(I), np.concatenate(J), np.concatenate(S)


def otsu(v, bins=256):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 4 or v.max() - v.min() < 1e-9:
        return float(np.median(v)) if len(v) else 0.5
    hist, edges = np.histogram(v, bins=bins)
    cen = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    s0 = np.cumsum(hist * cen)
    mu0 = s0 / np.maximum(w0, 1)
    mu1 = (s0[-1] - s0) / np.maximum(w1, 1)
    var = w0 * w1 * (mu0 - mu1) ** 2
    k = int(np.argmax(var[:-1]))
    return float(edges[k + 1])


def segment_order(order, labels, emb):
    """Split an ordered list of records into contiguous segments at class changes or similarity drops."""
    order = np.asarray(order)
    e = emb[order]
    s = (e[:-1] * e[1:]).sum(1)
    same = labels[order][:-1] == labels[order][1:]
    tau = otsu(s[same])
    boundary = (~same) | (s < tau)
    seg_pos = np.concatenate([[0], np.cumsum(boundary)])
    seg = np.full(len(labels), -1)
    seg[order] = seg_pos
    sizes = np.bincount(seg_pos)
    return seg, dict(tau=tau, n_seg=int(len(sizes)), size_med=float(np.median(sizes)),
                     size_q1=float(np.percentile(sizes, 25)), size_q3=float(np.percentile(sizes, 75)),
                     size_max=int(sizes.max()), adj_cos_mean=float(s[same].mean()) if same.any() else None)


def coarse_blocks(order, seg, labels, min_size):
    order = np.asarray(order)
    blk = np.full(len(labels), -1)
    b, cur = 0, 0
    for k, r in enumerate(order):
        if k > 0:
            prev = order[k - 1]
            if labels[r] != labels[prev] or (seg[r] != seg[prev] and cur >= min_size):
                b += 1
                cur = 0
        blk[r] = b
        cur += 1
    return blk


def adjacency_lift(order, labels, emb, rng, n_rand=20000):
    order = np.asarray(order)
    e = emb[order]
    same = labels[order][:-1] == labels[order][1:]
    adj = (e[:-1] * e[1:]).sum(1)[same]
    a = rng.integers(0, len(order), n_rand)
    b = rng.integers(0, len(order), n_rand)
    k = (labels[order][a] == labels[order][b]) & (a != b)
    rnd = (e[a[k]] * e[b[k]]).sum(1)
    return float(adj.mean() - rnd.mean()), float(adj.mean()), float(rnd.mean())


def recover_parents(emb_child_views, emb_orig, y_child, y_orig, near_of_orig, cos_min, margin_min):
    """For each publisher-augmented child: best matching original (max over 8 dihedral views)."""
    nc = len(emb_child_views)
    V = emb_child_views.reshape(nc * 8, -1)
    S = (V @ emb_orig.T).reshape(nc, 8, -1).max(1)
    S[y_child[:, None] != y_orig[None, :]] = -2
    top = np.argmax(S, 1)
    s1 = S[np.arange(nc), top]
    S2 = S.copy()
    same_cluster = near_of_orig[None, :] == near_of_orig[top][:, None]
    S2[same_cluster] = -2
    s2 = S2.max(1)
    ok = (s1 >= cos_min) & ((s1 - s2) >= margin_min)
    return top, s1, s2, ok


# =====================================================================================
# 6  split builder
#    clean holdout (outer fold over the strictest grouping g_ctrl, originals only, one per exact duplicate)
#    development pool -> arm split into train / inner validation / own evaluation (70/10/20)
#    arm factors: S   = siblings created before the split and allowed to straddle it
#                 Dup = duplicate files kept and allowed to straddle the split
#                 Grp = structural groups (slide / patient / recovered source image) ignored by the split
#    all arms of one experiment are matched on per-class training-set size
# =====================================================================================
XEVAL = {"histopath": ["he_stained", "ndb_ufes"], "he_stained": ["histopath", "ndb_ufes"],
         "ndb_ufes": ["histopath", "he_stained"]}


class DS:
    _cache = {}

    @classmethod
    def get(cls, ds):
        p = D["index"] / f"{ds}.json"
        pr = D["index"] / f"{ds}.recov.json"
        key = (ds, p.stat().st_mtime if p.exists() else 0, pr.stat().st_mtime if pr.exists() else 0)
        if key not in cls._cache:
            cls._cache[key] = DS(ds)
        return cls._cache[key]

    def __init__(self, ds):
        J = jload(D["index"] / f"{ds}.json")
        if J is None:
            raise FileNotFoundError(f"index for {ds} missing - run Block 16")
        R = J["records"]
        self.ds, self.R, self.n = ds, R, len(R)
        self.meta = META[ds]
        self.K = self.meta["K"]
        self.y = np.array([r["label"] for r in R], int)
        self.orig = np.array([bool(r.get("orig", True)) for r in R])
        self.ok = np.array([bool(r.get("ok", True)) for r in R])
        self.parent = np.array([int(r.get("parent", -1)) for r in R])
        self.child_ok = np.array([bool(r.get("parent_ok", False)) for r in R])
        self.dup_rep = np.array([bool(r.get("dup_rep", True)) for r in R])
        self.md5 = np.array([r.get("md5", str(i)) for i, r in enumerate(R)])
        ar = np.arange(self.n)
        for g in ("g_exact", "g_near", "g_struct", "g_ctrl", "g_coarse"):
            setattr(self, g, np.array([int(r.get(g, i)) for i, r in enumerate(R)]))
        self.parent_link = np.where((self.parent >= 0) & self.child_ok, self.parent, ar)
        self.slide = np.array([str(r.get("slide", r.get("patient", ""))) for r in R])
        self.base = self.ok & self.orig & self.dup_rep
        self.summary = J.get("summary", {})
        X = jload(D["index"] / f"{ds}.recov.json") or {}
        self.extra = {k: np.asarray(v, np.int64) for k, v in (X.get("groups") or {}).items()}


_OUTER = {}


def outer_folds(dd, scheme):
    """(fold id per record, holdout grouping key per record).  fold -1 = never used.
    loso: one slide per fold (grouping key = slide); near duplicates of holdout images are purged from the
    development pool by build_job instead of merging slides.  sgkf: stratified group k-fold on g_ctrl."""
    key = (dd.ds, dd.n, json.dumps(scheme, sort_keys=True))
    if key in _OUTER:
        return _OUTER[key]
    base = np.nonzero(dd.base)[0]
    if scheme["kind"] == "loso":
        slides = sorted(set(dd.slide[base]))
        rank = {s_: i for i, s_ in enumerate(slides)}
        fold = np.array([rank.get(dd.slide[i], -1) if dd.ok[i] else -1 for i in range(dd.n)])
        gh = dd.g_struct.copy()
    else:
        from sklearn.model_selection import StratifiedGroupKFold
        ctrl = dd.g_ctrl
        f_ctrl = {}
        sg = StratifiedGroupKFold(n_splits=scheme["n"], shuffle=True,
                                  random_state=(1000 * int(scheme.get("rep", 0)) + 7))
        for f, (_, te) in enumerate(sg.split(np.zeros(len(base)), dd.y[base], ctrl[base])):
            for r in base[te]:
                f_ctrl[ctrl[r]] = f
        fold = np.array([f_ctrl.get(ctrl[i], -1) if dd.ok[i] else -1 for i in range(dd.n)])
        gh = dd.g_ctrl.copy()
    _OUTER[key] = (fold, gh)
    return fold, gh


def n_outer(dd, scheme):
    return len(set(dd.slide[dd.base])) if scheme["kind"] == "loso" else scheme["n"]


def sg_split(keys, y, frac, seed):
    """Boolean mask of the 'test' side of one stratified group split with target fraction `frac`."""
    keys = np.asarray(keys)
    y = np.asarray(y)
    n = len(keys)
    mask = np.zeros(n, bool)
    if n == 0:
        return mask
    ug = np.unique(keys)
    n_splits = max(2, int(round(1.0 / frac)))
    rng = np.random.default_rng(seed)
    if len(ug) == n:  # every item is its own group: plain stratified random split
        for c in np.unique(y):
            ic = rng.permutation(np.nonzero(y == c)[0])
            mask[ic[:int(round(frac * len(ic)))]] = True
        if 0 < mask.sum() < n:
            return mask
        mask[:] = False
    if len(ug) >= 2 * n_splits:
        from sklearn.model_selection import StratifiedGroupKFold
        sg = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=int(seed % (2 ** 31 - 1)))
        folds = list(sg.split(np.zeros(n), y, keys))
        te = folds[int(rng.integers(0, n_splits))][1]
        mask[te] = True
        if 0 < mask.sum() < n:
            return mask
        mask[:] = False
    # few groups: greedy random group selection closest to the target fraction (at least one group)
    order = rng.permutation(ug)
    size = {g: int((keys == g).sum()) for g in ug}
    target, cur, chosen = frac * n, 0, []
    for g in order:
        if not chosen or abs(cur + size[g] - target) < abs(cur - target):
            chosen.append(g)
            cur += size[g]
        if cur >= target:
            break
    if len(chosen) == len(ug) and len(ug) > 1:
        chosen = chosen[:-1]
    mask[np.isin(keys, chosen)] = True
    return mask


def _distribute(n_items, total, rng):
    """copies per item summing to max(total, n_items); every item keeps its original (>=1)."""
    if n_items == 0:
        return np.zeros(0, int)
    total = max(int(total), n_items)
    base, extra = divmod(total, n_items)
    k = np.full(n_items, base, int)
    k[rng.permutation(n_items)[:extra]] += 1
    return k


class Units(dict):
    """Evaluation / training units: row (store row), copy (sibling index), y, key (bootstrap cluster = g_ctrl)."""

    @staticmethod
    def make(rows, copies, y, key, ds):
        u = Units(row=np.asarray(rows, np.int64), copy=np.asarray(copies, np.int64),
                  y=np.asarray(y, np.int64), key=np.asarray(key, np.int64))
        u.ds = ds
        return u

    def take(self, m):
        u = Units(**{k: v[m] for k, v in self.items()})
        u.ds = self.ds
        return u

    def cat(self, other):
        u = Units(**{k: np.concatenate([self[k], other[k]]) for k in self})
        u.ds = self.ds
        return u

    def __len__(self):
        return len(self["row"])


def _exp_cfg(spec):
    P = plan()
    ex = P["exps"][spec["exp"]]
    return P, ex, ex["datasets"][spec["ds"]]


def _labels(dd, spec):
    if not str(spec["exp"]).endswith("RL"):
        return dd.y
    rng = np.random.default_rng(shash("RL", dd.ds, spec["seed"]))
    roots = np.unique(dd.g_ctrl[dd.ok])
    perm = rng.permutation(len(roots))
    lab = {int(g): int(perm[i] % dd.K) for i, g in enumerate(roots)}
    return np.array([lab.get(int(dd.g_ctrl[i]), 0) for i in range(dd.n)])


def _build_arm(dd, y, fold, o, fac, dcfg, spec, name, purge):
    S, Dup, Grp = int(fac.get("S", 0)), int(fac.get("Dup", 0)), int(fac.get("Grp", 0))
    coarse = int(fac.get("coarse", 0))
    seed = int(spec["seed"])
    # the split never depends on the dose, so dose levels share one base split (nested designs)
    rng = np.random.default_rng(shash("arm", dd.ds, spec["exp"], o, spec.get("rep", 0), seed, name))
    use_children = bool(dcfg.get("children", False))
    dev = dd.ok & (fold >= 0) & (fold != o) & ~purge
    dev &= (dd.orig | (dd.child_ok & use_children))
    if not Dup:
        dev &= dd.dup_rep
    idx = np.nonzero(dev)[0]
    arrays = []
    if not Dup:
        arrays.append(dd.g_near)
    if not Grp:
        rv = fac.get("recov")
        if rv:
            if rv not in dd.extra:
                raise KeyError(f"recovered grouping '{rv}' missing for {dd.ds} - run the group-recovery block first")
            arrays.append(dd.extra[rv])
        else:
            arrays.append(dd.g_coarse if coarse else dd.g_struct)
    if not S:
        arrays.append(dd.parent_link)
    rk = union_labels(dd.n, label_arrays=arrays, subset=idx)
    policy = dcfg.get("policy")
    K = dd.K
    # per-record copy counts (siblings) for the development pool
    phi = (dd.base & (fold >= 0) & (fold != o)).sum() / max(1, (dd.base & (fold >= 0)).sum())
    if policy is not None:
        if spec["exp"] == "DOSEK":
            kmap = {int(r): int(spec["dose"]) for r in idx}
        else:
            kmap = {}
            for c in range(K):
                rc = idx[y[idx] == c]
                U = int(round(dcfg["target"] * phi))
                for r, k in zip(rc, _distribute(len(rc), U, rng)):
                    kmap[int(r)] = int(k)
    s_own = shash("own", dd.ds, spec["exp"], o, spec.get("rep", 0), seed, name)
    s_iv = shash("ival", dd.ds, spec["exp"], o, spec.get("rep", 0), seed, name)
    if policy is not None and S:
        rows = np.concatenate([np.full(kmap[int(r)], r) for r in idx]) if len(idx) else np.zeros(0, int)
        cps = np.concatenate([np.arange(kmap[int(r)]) for r in idx]) if len(idx) else np.zeros(0, int)
        keys = rk[rows] * 1000 + cps
        U = Units.make(rows, cps, y[rows], dd.g_ctrl[rows], dd.ds)
        own_m = sg_split(keys, U["y"], 0.20, s_own)
        rest = ~own_m
        iv_m = np.zeros(len(U), bool)
        iv_m[np.nonzero(rest)[0][sg_split(keys[rest], U["y"][rest], 0.125, s_iv)]] = True
        parts = dict(own=U.take(own_m), ival=U.take(iv_m), train=U.take(rest & ~iv_m))
    else:
        U = Units.make(idx, np.zeros(len(idx), int), y[idx], dd.g_ctrl[idx], dd.ds)
        keys = rk[idx]
        own_m = sg_split(keys, U["y"], 0.20, s_own)
        rest = ~own_m
        iv_m = np.zeros(len(U), bool)
        iv_m[np.nonzero(rest)[0][sg_split(keys[rest], U["y"][rest], 0.125, s_iv)]] = True
        parts = dict(own=U.take(own_m), ival=U.take(iv_m), train=U.take(rest & ~iv_m))
        if policy is not None:  # augment after the split, same average factor per class
            for pn, P_ in list(parts.items()):
                R_, C_, Y_, G_ = [], [], [], []
                for c in range(K):
                    rc = P_["row"][P_["y"] == c]
                    tot = int(sum(kmap[int(r)] for r in rc))
                    for r, k in zip(rc, _distribute(len(rc), tot, rng)):
                        R_.append(np.full(k, r))
                        C_.append(np.arange(k))
                rows = np.concatenate(R_) if R_ else np.zeros(0, int)
                cps = np.concatenate(C_) if C_ else np.zeros(0, int)
                parts[pn] = Units.make(rows, cps, y[rows], dd.g_ctrl[rows], dd.ds)
    parts["rec_key"] = rk
    parts["factors"] = dict(S=S, Dup=Dup, Grp=Grp, coarse=coarse)
    return parts


def _subsample_to(U, Tc, rng):
    keep = []
    for c, t in Tc.items():
        ic = np.nonzero(U["y"] == c)[0]
        keep.append(ic if len(ic) <= t else np.sort(rng.choice(ic, t, replace=False)))
    return U.take(np.sort(np.concatenate(keep)) if keep else np.zeros(0, int))


def leak_stats(dd, train, own):
    if len(own) == 0 or len(train) == 0:
        return {}
    tr = train["row"]
    ow = own["row"]
    s_rec = set(tr.tolist())
    f = lambda a: float(np.mean([v in s_rec for v in a])) if len(a) else 0.0
    out = dict(same_original=f(ow))
    for name, arr in (("near_dup", dd.g_near), ("struct_group", dd.g_struct), ("ctrl_group", dd.g_ctrl)):
        s = set(arr[tr].tolist())
        out[name] = float(np.mean([v in s for v in arr[ow]]))
    s = set(dd.md5[tr].tolist())
    out["identical_file"] = float(np.mean([v in s for v in dd.md5[ow]]))
    out["own_units_from_originals"] = float(np.mean(own["copy"] == 0))
    return out


def check_disjoint(dd, B, controlled, injected=False):
    """Raises AssertionError when the clean holdout or a controlled arm is contaminated."""
    hold = B["hold"]
    gh = B["ghold"]
    dev_rows = np.concatenate([B[k]["row"] for k in ("train", "ival", "own")])
    hr = set(hold["row"].tolist())
    assert not (hr & set(dev_rows.tolist())), "holdout image also in train/ival/own"
    assert not (set(gh[hold["row"]].tolist()) & set(gh[dev_rows].tolist())), "holdout group (slide/patient/source) in development data"
    assert not (set(dd.g_near[hold["row"]].tolist()) & set(dd.g_near[dev_rows].tolist())), "near duplicate of a holdout image in development data"
    hm = set(dd.md5[hold["row"]].tolist())
    assert not (hm & set(dd.md5[dev_rows].tolist())), "holdout file hash also in development data"
    assert not (set(dd.parent_link[hold["row"]].tolist()) & set(dd.parent_link[dev_rows].tolist())), "augmented copy of a holdout image in development data"
    if controlled:
        rk = B["rec_key"]
        tr = B["train"]["row"]
        for part in ("own", "ival"):
            pr = B[part]["row"]
            if not injected or part == "ival":
                assert not (set(rk[tr].tolist()) & set(rk[pr].tolist())), f"controlled arm: train/{part} share a group"
                assert not (set(dd.md5[tr].tolist()) & set(dd.md5[pr].tolist())), f"controlled arm: train/{part} share a file hash"
                assert not (set(dd.parent_link[tr].tolist()) & set(dd.parent_link[pr].tolist())), f"controlled arm: train/{part} share a source image"
    return True


def xeval_units(src):
    out = {}
    for t in XEVAL.get(src, []):
        if not (D["index"] / f"{t}.json").exists():
            continue
        dd = DS.get(t)
        r = np.nonzero(dd.base)[0]
        y = dd.y[r]
        if t == "ndb_ufes":
            keep = y != 1
            r, y = r[keep], (y[keep] == 2).astype(int)
        out[t] = Units.make(r, np.zeros(len(r), int), y, dd.g_ctrl[r], t)
    return out


def build_job(spec, check=True):
    """Deterministic data for one job.  Returns dict(train, ival, own, hold, xeval, info)."""
    P, ex, dcfg = _exp_cfg(spec)
    dd = DS.get(spec["ds"])
    y = _labels(dd, spec)
    scheme = P["outer"][spec["ds"]]
    scheme = dict(scheme, rep=int(spec.get("rep", 0)))
    fold, ghold = outer_folds(dd, scheme)
    o = int(spec["outer"])
    hr = np.nonzero(dd.base & (fold == o))[0]
    if len(hr) == 0:
        raise ValueError(f"empty holdout for {dd.ds} outer fold {o}")
    hold = Units.make(hr, np.zeros(len(hr), int), y[hr], dd.g_ctrl[hr], dd.ds)
    # purge: development records that are identical / near-duplicate / same source as any holdout image
    hn = set(dd.g_near[fold == o].tolist()) | set(dd.md5[fold == o].tolist()) | set(dd.parent_link[fold == o].tolist())
    purge = np.array([(dd.g_near[i] in hn or dd.md5[i] in hn or dd.parent_link[i] in hn) and fold[i] != o
                      for i in range(dd.n)])
    # arms split by a recovered grouping that could not be formed (k > n/3) are left out of the size matching
    arms = {nm: fac for nm, fac in dcfg["arms"].items()
            if not fac.get("recov") or fac["recov"] in dd.extra or nm == spec["arm"]}
    built = {nm: _build_arm(dd, y, fold, o, fac, dcfg, spec, nm, purge) for nm, fac in arms.items()}
    Tc = {c: min(int((b["train"]["y"] == c).sum()) for b in built.values()) for c in range(dd.K)}
    rng = np.random.default_rng(shash("match", dd.ds, spec["exp"], o, spec.get("rep", 0), spec["seed"], spec.get("dose")))
    B = built[spec["arm"]]
    B["train"] = _subsample_to(B["train"], Tc, rng)
    injected = False
    info = dict(n_dev=None, Tc=Tc)
    if spec["exp"] == "DUP" and float(spec["dose"]) > 0:
        own0 = np.nonzero(B["own"]["copy"] == 0)[0]
        n_inj = int(round(float(spec["dose"]) * len(own0)))
        rg = np.random.default_rng(shash("dup", dd.ds, o, spec["seed"]))   # nested across doses
        pick = np.sort(rg.permutation(own0)[:n_inj])
        add = B["own"].take(pick)
        drop = []
        for c in range(dd.K):
            nc_ = int((add["y"] == c).sum())
            ic = np.nonzero(B["train"]["y"] == c)[0]
            drop.extend(np.random.default_rng(shash("dupdrop", dd.ds, o, spec["seed"], c)).permutation(ic)[:nc_].tolist())
        keep = np.setdiff1d(np.arange(len(B["train"])), np.array(drop, int))
        B["train"] = B["train"].take(keep).cat(add)
        injected = True
        info["injected"] = int(n_inj)
        B["own_injected_mask"] = np.isin(np.arange(len(B["own"])), pick)
    if spec["exp"] == "OVL":
        rk = B["rec_key"]
        own = B["own"]
        rg = np.random.default_rng(shash("ovl", dd.ds, o, spec["seed"]))
        g_own = rk[own["row"]]
        ev_m = np.zeros(len(own), bool)
        res_by_g = {}
        for g in np.unique(g_own):
            ii = rg.permutation(np.nonzero(g_own == g)[0])
            h = len(ii) // 2
            ev_m[ii[:max(1, len(ii) - h)]] = True
            res_by_g[g] = ii[max(1, len(ii) - h):]
        q = float(spec["dose"])
        add_idx = []
        for g, ii in res_by_g.items():
            add_idx.extend(ii[:int(math.ceil(q * len(ii)))].tolist())
        add = own.take(np.array(sorted(add_idx), int))
        drop = []
        for c in range(dd.K):
            nc_ = int((add["y"] == c).sum())
            ic = np.nonzero(B["train"]["y"] == c)[0]
            drop.extend(np.random.default_rng(shash("ovldrop", dd.ds, o, spec["seed"], c)).permutation(ic)[:nc_].tolist())
        keep = np.setdiff1d(np.arange(len(B["train"])), np.array(drop, int))
        B["train"] = B["train"].take(keep).cat(add)
        B["own"] = own.take(ev_m)
        injected = q > 0
        info["overlap_added"] = int(len(add))
    B["hold"] = hold
    B["ghold"] = ghold
    B["xeval"] = xeval_units(spec["ds"]) if spec.get("xeval") else {}
    fac = B["factors"]
    controlled = (fac["S"] == 0 and fac["Dup"] == 0 and fac["Grp"] == 0)
    if check:
        check_disjoint(dd, B, controlled, injected)
    info.update(sizes={k: int(len(B[k])) for k in ("train", "ival", "own", "hold")}, purged=int(purge.sum()),
                train_per_class=[int((B["train"]["y"] == c).sum()) for c in range(dd.K)],
                hold_per_class=[int((hold["y"] == c).sum()) for c in range(dd.K)],
                leak=leak_stats(dd, B["train"], B["own"]), controlled=controlled,
                policy=dcfg.get("policy"), K=dd.K)
    B["info"] = info
    return B


# =====================================================================================
# 7  metrics, calibration, operating points, cluster statistics
# =====================================================================================
def softmax(z, T=1.0):
    z = np.asarray(z, np.float64) / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits, y):
    """Temperature minimising NLL on the given (inner-validation) predictions."""
    logits, y = np.asarray(logits, np.float64), np.asarray(y, int)
    if len(y) < 5 or len(np.unique(y)) < 2:
        return 1.0

    def nll(T):
        p = softmax(logits, T)
        return -np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))

    grid = np.exp(np.linspace(np.log(0.05), np.log(20), 121))
    v = [nll(t) for t in grid]
    k = int(np.argmin(v))
    lo, hi = grid[max(0, k - 1)], grid[min(len(grid) - 1, k + 1)]
    for _ in range(40):
        m1, m2 = lo + (hi - lo) / 3, hi - (hi - lo) / 3
        if nll(m1) < nll(m2):
            hi = m2
        else:
            lo = m1
    return float((lo + hi) / 2)


def ece_score(prob, y, bins=15):
    conf = prob.max(1)
    pred = prob.argmax(1)
    acc = (pred == y).astype(float)
    e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for a, b in zip(edges[:-1], edges[1:]):
        m = (conf > a) & (conf <= b)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def metrics(prob, y, K, pos=None, pred=None):
    """Full metric set.  Macro metrics AND explicit positive-class (one-vs-rest) metrics are kept apart."""
    from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, cohen_kappa_score
    prob = np.asarray(prob, np.float64)
    y = np.asarray(y, int)
    n = len(y)
    if n == 0:
        return {}
    pred = prob.argmax(1) if pred is None else np.asarray(pred, int)
    cm = np.zeros((K, K), int)
    np.add.at(cm, (y, pred), 1)
    tp = np.diag(cm).astype(float)
    rows, cols = cm.sum(1).astype(float), cm.sum(0).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        rec = np.where(rows > 0, tp / rows, np.nan)
        prec = np.where(cols > 0, tp / cols, np.nan)
        f1 = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0.0)
        tn = n - rows - cols + tp
        spec = np.where((n - rows) > 0, tn / (n - rows), np.nan)
        npv = np.where((n - cols) > 0, tn / (n - cols), np.nan)
    present = rows > 0
    out = dict(n=n, acc=float(tp.sum() / n), bal_acc=float(np.nanmean(rec[present])),
               macro_f1=float(np.mean(f1[present])), macro_prec=float(np.nanmean(prec[present])),
               macro_rec=float(np.nanmean(rec[present])), macro_spec=float(np.nanmean(spec[present])),
               per_class=dict(recall=rec.tolist(), precision=prec.tolist(), f1=f1.tolist(),
                              specificity=spec.tolist(), npv=npv.tolist(), n=rows.astype(int).tolist()),
               cm=cm.tolist())
    try:
        if K == 2 and present.all():
            out["auroc"] = float(roc_auc_score(y, prob[:, 1]))
            out["auprc"] = float(average_precision_score(y, prob[:, 1]))
        elif present.all():
            out["auroc"] = float(roc_auc_score(y, prob, multi_class="ovr", average="macro"))
            out["auprc"] = float(np.mean([average_precision_score(y == k, prob[:, k]) for k in range(K)]))
    except Exception:
        pass
    try:
        out["mcc"] = float(matthews_corrcoef(y, pred))
        out["kappa"] = float(cohen_kappa_score(y, pred))
    except Exception:
        pass
    out["nll"] = float(-np.mean(np.log(np.clip(prob[np.arange(n), y], 1e-12, 1))))
    oh = np.eye(K)[y]
    out["brier"] = float(np.mean(((prob - oh) ** 2).sum(1)))
    out["ece"] = ece_score(prob, y)
    if pos is not None:
        out["pos"] = dict(cls=int(pos), sens=float(rec[pos]), spec=float(spec[pos]), prec=float(prec[pos]),
                          f1=float(f1[pos]), npv=float(npv[pos]))
    return out


def thresholds_from(p_pos, y_bin):
    """Operating points chosen on inner-validation data: Youden, 'sensitivity >= 95%' and 'sensitivity >= 99%'
    (the highest threshold that still reaches the target sensitivity on inner validation)."""
    p_pos, y_bin = np.asarray(p_pos, float), np.asarray(y_bin, int)
    out = dict(default=0.5)
    if len(np.unique(y_bin)) < 2:
        out.update(youden=0.5, sens95=0.5, sens99=0.5)
        return out
    cand = np.unique(np.concatenate([p_pos, [0.0, 1.0 + 1e-9]]))
    P, N = (y_bin == 1).sum(), (y_bin == 0).sum()
    best, bt = -1, 0.5
    s95 = s99 = None
    for t in cand:
        pr = p_pos >= t
        se = (pr & (y_bin == 1)).sum() / P
        sp = (~pr & (y_bin == 0)).sum() / N
        if se + sp - 1 > best:
            best, bt = se + sp - 1, float(t)
        if se >= 0.95:
            s95 = float(t)
        if se >= 0.99:
            s99 = float(t)
    out.update(youden=bt, sens95=s95 if s95 is not None else float(cand.min()),
               sens99=s99 if s99 is not None else float(cand.min()))
    return out


def op_point(p_pos, y_bin, thr):
    """Every column computed from the same threshold."""
    p_pos, y_bin = np.asarray(p_pos, float), np.asarray(y_bin, int)
    pr = p_pos >= thr
    TP = int((pr & (y_bin == 1)).sum()); FN = int((~pr & (y_bin == 1)).sum())
    TN = int((~pr & (y_bin == 0)).sum()); FP = int((pr & (y_bin == 0)).sum())
    se = TP / max(1, TP + FN)
    sp = TN / max(1, TN + FP)
    return dict(thr=float(thr), sens=se, spec=sp, ppv=TP / max(1, TP + FP), npv=TN / max(1, TN + FN),
                lr_pos=(se / (1 - sp)) if sp < 1 else float("inf"), lr_neg=((1 - se) / sp) if sp > 0 else float("inf"),
                fnr=1 - se, TP=TP, FN=FN, TN=TN, FP=FP)


# ---------- cluster bootstrap (balanced accuracy via per-cluster count tables) ----------
def cluster_table(y, correct, cluster, K):
    cl, inv = np.unique(np.asarray(cluster), return_inverse=True)
    n = np.zeros((len(cl), K))
    c = np.zeros((len(cl), K))
    np.add.at(n, (inv, np.asarray(y, int)), 1)
    np.add.at(c, (inv, np.asarray(y, int)), np.asarray(correct, float))
    return n, c


def ba_counts(n, c):
    tot = n.sum(-2)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(tot > 0, c.sum(-2) / tot, np.nan)
    return np.nanmean(r, -1)


def boot_weights(C, B, rng, chunk=1000):
    for s in range(0, B, chunk):
        b = min(chunk, B - s)
        yield rng.multinomial(C, np.full(C, 1.0 / C), size=b).astype(np.float32)


def boot_ba(n, c, B, rng):
    out = []
    for Wt in boot_weights(len(n), B, rng):
        nn_ = np.einsum("bc,ck->bk", Wt, n)
        cc = np.einsum("bc,ck->bk", Wt, c)
        with np.errstate(divide="ignore", invalid="ignore"):
            r = np.where(nn_ > 0, cc / nn_, np.nan)
        out.append(np.nanmean(r, 1))
    return np.concatenate(out)


def boot_ba_paired(n, cA, cB, B, rng):
    """Paired difference BA_A - BA_B on the same images, resampling shared clusters."""
    out = []
    for Wt in boot_weights(len(n), B, rng):
        nn_ = np.einsum("bc,ck->bk", Wt, n)
        with np.errstate(divide="ignore", invalid="ignore"):
            ra = np.where(nn_ > 0, np.einsum("bc,ck->bk", Wt, cA) / nn_, np.nan)
            rb = np.where(nn_ > 0, np.einsum("bc,ck->bk", Wt, cB) / nn_, np.nan)
        out.append(np.nanmean(ra, 1) - np.nanmean(rb, 1))
    return np.concatenate(out)


def hier(mats, B, seed):
    """Two-stage bootstrap: resample outer folds with replacement, then take each chosen fold's
    within-fold cluster replicate.  mats: list of [F, B] arrays in the same fold order."""
    g = np.random.default_rng(seed)
    F = mats[0].shape[0]
    idx = g.integers(0, F, size=(B, F))
    cols = np.arange(B)[:, None]
    return [m[idx, cols].mean(1) for m in mats]


def ci(v, a=0.05):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return (np.nan, np.nan)
    return (float(np.percentile(v, 100 * a / 2)), float(np.percentile(v, 100 * (1 - a / 2))))


def boot_p(v, null=0.0):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return np.nan
    lo = (np.sum(v <= null) + 1) / (len(v) + 1)
    hi = (np.sum(v >= null) + 1) / (len(v) + 1)
    return float(min(1.0, 2 * min(lo, hi)))


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i]))
        adj[i] = run
    return adj.tolist()


def icc1(correct, cluster):
    """One-way random-effects ICC(1) of a binary outcome, and the design effect."""
    y = np.asarray(correct, float)
    cl, inv = np.unique(np.asarray(cluster), return_inverse=True)
    C, N = len(cl), len(y)
    if C < 2 or C == N:
        return dict(icc=0.0, deff=1.0, clusters=C, mean_size=N / max(1, C))
    ni = np.bincount(inv).astype(float)
    mi = np.bincount(inv, weights=y) / ni
    gm = y.mean()
    msb = np.sum(ni * (mi - gm) ** 2) / (C - 1)
    msw = np.sum((y - mi[inv]) ** 2) / (N - C)
    m0 = (N - np.sum(ni ** 2) / N) / (C - 1)
    icc = (msb - msw) / (msb + (m0 - 1) * msw) if (msb + (m0 - 1) * msw) > 0 else 0.0
    icc = float(max(0.0, icc))
    return dict(icc=icc, deff=float(1 + (ni.mean() - 1) * icc), clusters=int(C), mean_size=float(ni.mean()))


class quiet:
    """Silence warnings inside a block (statsmodels re-enables its own warnings at import)."""

    def __enter__(self):
        try:  # import first: statsmodels installs 'always' filters on first import
            import statsmodels.api  # noqa
            import statsmodels.formula.api  # noqa
        except Exception:
            pass
        self._c = warnings.catch_warnings()
        self._c.__enter__()
        warnings.simplefilter("ignore")
        return self

    def __exit__(self, *a):
        return self._c.__exit__(*a)


def gee_paired(correct_a, correct_b, cluster):
    """GEE (logit, exchangeable) for P(correct) of arm A vs arm B on the same images, clustered."""
    with quiet():
        return _gee_paired(correct_a, correct_b, cluster)


def _gee_paired(correct_a, correct_b, cluster):
    try:
        import statsmodels.api as sm
        ya = np.asarray(correct_a, float)
        yb = np.asarray(correct_b, float)
        yy = np.concatenate([ya, yb])
        arm = np.concatenate([np.ones(len(ya)), np.zeros(len(yb))])
        g = np.concatenate([cluster, cluster])
        X = sm.add_constant(arm)
        res = sm.GEE(yy, X, groups=g, family=sm.families.Binomial(),
                     cov_struct=sm.cov_struct.Exchangeable()).fit()
        b, se = float(res.params[1]), float(res.bse[1])
        return dict(log_or=b, or_=math.exp(b), lo=math.exp(b - 1.96 * se), hi=math.exp(b + 1.96 * se),
                    p=float(res.pvalues[1]))
    except Exception as e:
        return dict(error=str(e)[:120])


def mixed_intercept(values, outer, seed):
    """Random-intercept model  value ~ 1 + (1|outer fold) + seed variance component."""
    with quiet():
        return _mixed_intercept(values, outer, seed)


def _t_interval(v, note):
    from scipy import stats as st
    v = np.asarray(v, float)
    m = float(v.mean())
    if len(v) < 2:
        return dict(est=m, lo=None, hi=None, p=None, note=note)
    se = float(v.std(ddof=1) / np.sqrt(len(v)))
    q = float(st.t.ppf(0.975, len(v) - 1))
    p = float(2 * st.t.sf(abs(m) / se, len(v) - 1)) if se > 0 else None
    return dict(est=m, lo=m - q * se, hi=m + q * se, p=p, note=note)


def _mixed_intercept(values, outer, seed):
    import pandas as pd
    df = pd.DataFrame(dict(v=np.asarray(values, float), outer=outer, seed=seed))
    if df["outer"].nunique() < 2:
        return _t_interval(df["v"], "one outer fold")
    if df.groupby("outer").size().max() < 2:
        return _t_interval(df["v"], "one run per fold: t-interval over folds")
    try:
        import statsmodels.formula.api as smf
        vc = {"seed": "0 + C(seed)"} if df["seed"].nunique() > 1 else None
        res = smf.mixedlm("v ~ 1", df, groups=df["outer"], vc_formula=vc).fit(reml=True)
        b, se = float(res.params["Intercept"]), float(res.bse["Intercept"])
        if not np.isfinite(se) or se < 1e-4 or not res.converged:     # a zero SE means a singular (boundary) fit
            raise ValueError("singular fit" if np.isfinite(se) and res.converged else "did not converge")
        return dict(est=b, lo=b - 1.96 * se, hi=b + 1.96 * se, p=float(res.pvalues["Intercept"]), note=None)
    except Exception as e:
        agg = df.groupby("outer")["v"].mean()
        return _t_interval(agg.values, "mixed model failed (" + str(e)[:40] + "): t-interval over fold means")


def sign_flip_p(d):
    """Exact two-sided sign-flip permutation test on a small vector of paired differences."""
    d = np.asarray(d, float)
    n = len(d)
    if n == 0 or n > 16:
        return np.nan
    obs = abs(d.mean())
    cnt = 0
    for m in range(2 ** n):
        s = np.array([1 if (m >> i) & 1 else -1 for i in range(n)])
        cnt += abs((s * d).mean()) >= obs - 1e-12
    return cnt / 2 ** n


# ---------- run results ----------
def load_run(rid):
    r = jload(D["runs"] / rid / "result.json")
    if r is None or r.get("status") != "done":
        return None, None
    z = np.load(D["runs"] / rid / "preds.npz", allow_pickle=False)
    return r, z


def runs_index():
    out = []
    for p in sorted(D["runs"].glob("*/result.json")):
        r = jload(p)
        if r and r.get("status") == "done":
            out.append(r)
    return out


def set_preds(z, name, T=1.0):
    if f"{name}_logit" not in z.files:
        return None
    lg = z[f"{name}_logit"].astype(np.float64)
    return dict(prob=softmax(lg, T), logit=lg, y=z[f"{name}_y"], key=z[f"{name}_key"],
                row=z[f"{name}_row"], copy=z[f"{name}_copy"])


# =====================================================================================
# 8  job planner (used by Block 18 and the launcher)
# =====================================================================================
BB_SHORT = {"convnext_tiny.fb_in22k_ft_in1k": "cnxT", "resnet50.a1_in1k": "r50",
            "vit_small_patch16_224.augreg_in21k_ft_in1k": "vitS", "efficientnet_b0.ra_in1k": "eb0",
            "convnext_base.fb_in22k_ft_in1k": "cnxB", "efficientnet_b3.ra2_in1k": "eb3"}
PHASE_DESC = {"core": "E1 clean-holdout experiment, primary backbone, seed 0",
              "controls": "random-label, duplicate-injection, group-overlap, pre-augmented corpus",
              "dose": "augmentation-factor dose-response",
              "seeds": "E1 seeds 1 and 2 (seed as random effect)",
              "general": "extra backbones, factorial extras, published-pipeline reimplementations",
              "orchid": "ORCHID (oral H&E with patient IDs): leaky vs patient-grouped vs recovered-group arms",
              "recover": "arms split by metadata-free recovered groups (BreakHis, OCDC, NDB-UFES, histopathology)"}


def job_id(s):
    t = f"{s['exp']}.{SHORT[s['ds']]}.{s['arm']}.o{s['outer']}r{s.get('rep', 0)}.s{s['seed']}.{BB_SHORT.get(s['backbone'], 'bb')}"
    if s.get("recipe", "std") != "std":
        t += f".{s['recipe']}"
    if s.get("dose") is not None:
        t += f".d{s['dose']}"
    return t


def job_fp(s, P):
    ex = P["exps"][s["exp"]]
    dc = ex["datasets"][s["ds"]]
    return fingerprint(dict(spec=s, arms=dc["arms"], policy=dc.get("policy"), target=dc.get("target"),
                            children=dc.get("children"), recipe=P["recipes"][s.get("recipe", "std")],
                            outer=P["outer"][s["ds"]], near=P["near_dup"], lib=LIB_VERSION))


def recipe_steps(spec, R, n_train):
    if R.get("epochs"):
        spe = max(1, int(math.ceil(n_train / R["bs"])))
        if TINY:                                   # tests only: a few steps, same code path
            st = max(12, int(R["epochs"] * spe) // 600)
            return st, max(2, st // 6)
        return int(R["epochs"] * spe), spe
    st = int(spec["steps"])
    if TINY:
        st = max(6, st // 60)
    return st, max(1, int(round(st * R.get("eval_frac", 0.1))))


def estimate_minutes(spec, info, R, speed=None):
    img = R["img"]
    ips = (speed or {}).get(spec["backbone"]) or BACKBONES.get(spec["backbone"], ("", 0, 300))[2]
    ips = ips * (224.0 / img) ** 2
    n_tr = info["sizes"]["train"]
    steps, every = recipe_steps(spec, R, n_tr)
    n_evals = max(1, steps // every)
    ev = info["sizes"]["ival"] * n_evals + (info["sizes"]["own"] + info["sizes"]["hold"] + info.get("n_xeval", 0)) * (2 if R.get("tta") else 1)
    return (steps * R["bs"] / ips + ev / (3 * ips) + 30) / 60.0


def available_datasets():
    return [d for d in DATASETS if (D["index"] / f"{d}.json").exists()]


def enumerate_specs(P):
    avail = set(available_datasets())
    audit = jload(D["index"] / "audit.json", {}) or {}
    pre_ok = bool(audit.get("histopath", {}).get("pre_enabled", False))
    bb0 = P["primary_backbone"]
    out = []

    def add(phase, **s):
        s.setdefault("rep", 0)
        s.setdefault("backbone", bb0)
        s.setdefault("recipe", "std")
        s.setdefault("dose", None)
        s.setdefault("xeval", False)
        s.setdefault("keep_weights", False)
        s.setdefault("train_profile", None)
        out.append((phase, s))

    E1 = P["exps"]["E1"]
    for ds, dc in E1["datasets"].items():
        if ds not in avail:
            continue
        dd = DS.get(ds)
        sch = P["outer"][ds]
        for rep in P["reps"].get(ds, P["reps"]["default"]):
            for o in range(n_outer(dd, dict(sch, rep=rep))):
                for seed in E1["seeds"]:
                    for arm in dc["arms"]:
                        extra = arm in dc.get("extra", [])
                        if extra and seed != 0:
                            continue
                        phase = "general" if extra else ("core" if seed == 0 else "seeds")
                        add(phase, exp="E1", ds=ds, arm=arm, outer=o, rep=rep, seed=seed, steps=dc["steps"],
                            xeval=(seed == 0 and not extra and ds in XEVAL),
                            keep_weights=(seed == 0 and o == 0 and rep == 0 and not extra))
    for ex_name, phase in (("PRE", "controls"), ("RL", "controls"), ("DUP", "controls"), ("OVL", "controls"),
                           ("DOSEK", "dose")):
        ex = P["exps"][ex_name]
        for ds, dc in ex["datasets"].items():
            if ds not in avail or (ex_name == "PRE" and not pre_ok):
                continue
            dd = DS.get(ds)
            outs = list(range(n_outer(dd, P["outer"][ds]))) if dc.get("outer") == "all" else ex.get("outer", [0, 1, 2])
            if ex_name == "PRE":
                outs = list(range(n_outer(dd, P["outer"][ds])))
            for o in outs:
                for seed in ex.get("seeds", [0]):
                    for dose in (ex.get("doses") or [None]):
                        for arm in dc["arms"]:
                            add(phase, exp=ex_name, ds=ds, arm=arm, outer=o, seed=seed, steps=dc["steps"], dose=dose)
    G = P["generality"]
    for bb in G["backbones"]:
        for ds in G["datasets"]:
            if ds not in avail:
                continue
            for o in G["outer"]:
                for arm in G["arms"]:
                    add("general", exp="E1", ds=ds, arm=arm, outer=o, seed=G["seed"], backbone=bb,
                        steps=E1["datasets"][ds]["steps"])
    for r in P["reimp"]:
        if r["ds"] not in avail or (r["exp"] == "PRE" and not pre_ok):
            continue
        for o in r["outer"]:
            for arm in r["arms"]:
                add("general", exp=r["exp"], ds=r["ds"], arm=arm, outer=o, seed=0, backbone=r["backbone"],
                    recipe=r["recipe"], steps=None, train_profile=r["train_profile"])
    return out


def make_jobs(verbose=True):
    """Enumerate, build (with disjointness tests), estimate and prioritise every job.  Keeps finished runs."""
    P = plan()
    speed = merged_speed()
    specs = enumerate_specs(P)
    old = {j["id"]: j for j in (jload(D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])}
    jobs, bad = [], []
    order = {p: i for i, p in enumerate(P["phases"])}
    t0 = time.time()
    for k, (phase, s) in enumerate(specs):
        jid = job_id(s)
        fp = job_fp(s, P)
        try:
            B = build_job(s, check=True)
        except AssertionError as e:
            bad.append((jid, "DISJOINTNESS FAILED: " + str(e)))
            continue
        except Exception as e:
            bad.append((jid, f"{type(e).__name__}: {e}"))
            continue
        info = B["info"]
        info["n_xeval"] = int(sum(len(u) for u in B["xeval"].values()))
        R = P["recipes"][s["recipe"]]
        est = estimate_minutes(s, info, R, speed)
        steps, _ = recipe_steps(s, R, info["sizes"]["train"])
        prio = order[phase] * 1_000_000 + (s["seed"] * 10_000 if phase in ("seeds",) else 0) + s["outer"] * 1000 + int(est * 10)
        jobs.append(dict(id=jid, fp=fp, phase=phase, priority=prio, est_min=round(est, 2), steps=steps,
                         sizes=info["sizes"], leak=info["leak"], spec=s))
        if verbose and (k + 1) % 100 == 0:
            print(f"    built {k + 1}/{len(specs)} jobs ({time.time() - t0:.0f}s)", flush=True)
    jobs.sort(key=lambda j: j["priority"])
    # finished runs whose fingerprint no longer matches are moved aside, never silently reused
    moved = 0
    for j in jobs:
        r = jload(D["runs"] / j["id"] / "result.json")
        if r and r.get("fp") != j["fp"]:
            D["superseded"].mkdir(parents=True, exist_ok=True)
            import shutil
            shutil.move(str(D["runs"] / j["id"]), str(D["superseded"] / f"{j['id']}.{int(time.time())}"))
            moved += 1
    jsave(D["jobs"] / "jobs.json", dict(created=time.time(), plan_sha=jload(D["plan"] / "plan.sha256.json", {}).get("sha256"),
                                        jobs=jobs, rejected=bad))
    return jobs, bad, moved


def job_done(j):
    r = jload(D["runs"] / j["id"] / "result.json")
    return bool(r and r.get("status") == "done" and r.get("fp") == j["fp"])


def fail_count(jid):
    return int((jload(D["fail"] / f"{jid}.json", {}) or {}).get("count", 0))


def merged_speed():
    sp = {}
    for p in D["state"].glob("speed_*.json"):
        for k, v in (jload(p, {}) or {}).items():
            sp.setdefault(k, []).extend(v[-20:])
    return {k: float(np.median(v)) for k, v in sp.items() if v}
'''
(Q1_ROOT / "lib" / "q1lib.py").write_text(Q1LIB_SRC)
sys.path.insert(0, str(Q1_ROOT / "lib"))
import q1lib as Q
importlib.reload(Q)
Q.keep_source("15")
Q.ensure_dirs()

# ------------------------------------------------------------------ dependencies
need = {"timm": "timm==1.0.15", "statsmodels": "statsmodels", "skimage": "scikit-image", "psutil": "psutil"}
for mod, pkg in need.items():
    try:
        importlib.import_module(mod)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ------------------------------------------------------------------ analysis plan (fixed and timestamped before any training run)
PLAN = dict(
    version=1,
    title="Leakage in oral-cancer image classification: analysis plan fixed before the experiments (Q1 revision)",
    primary_backbone="convnext_tiny.fb_in22k_ft_in1k",
    primary_metric="bal_acc",
    store_px=288, img=224, budget_min=660,
    near_dup=dict(cand_phash=12, cand_cos=0.95, phash_max=8, ssim_min=0.90, report_cos=0.97,
                  cap_frac=0.02, cap_min=25, fallback_phash=4, fallback_ssim=0.95),
    segmentation=dict(embed="dinov2_s", hist_lift_min=0.10, coarse_min=48),
    parent_recovery=dict(cos_min=0.90, margin_min=0.02, enable_pre_min_rate=0.95),
    outer=dict(he_stained=dict(kind="loso"), histopath=dict(kind="sgkf", n=5), ndb_ufes=dict(kind="sgkf", n=5),
               clinical=dict(kind="sgkf", n=5), breakhis=dict(kind="sgkf", n=5)),
    reps=dict(default=[0], clinical=[0, 1]),
    recipes=dict(
        std=dict(opt="adamw", lr_backbone=1e-4, lr_head=1e-3, wd=0.05, sched="cosine", warmup_frac=0.1, warmup_max=300,
                 min_lr_frac=0.01, label_smoothing=0.1, drop_path=0.1, bs=64, img=224, eval_frac=0.1, select="bal_acc",
                 patience_evals=4, patience_after=0.5, clip=1.0, tta=True, head="linear"),
        scirep=dict(opt="adamax", lr_backbone=1e-4, lr_head=1e-4, wd=0.0, sched="plateau", plateau_factor=0.1,
                    plateau_patience=2, plateau_metric="nll", label_smoothing=0.0, drop_path=0.0, bs=32, img=300,
                    epochs=20, select="nll", patience_evals=10, patience_after=0.0, clip=0.0, tta=False, head="scirep",
                    l2_kernel=0.016, l1_bias=0.006, l1_activity=0.006),
        abdelaziz=dict(opt="adam", lr_backbone=1e-4, lr_head=1e-4, wd=0.0, sched="plateau", plateau_factor=0.5,
                       plateau_patience=2, plateau_metric="acc", label_smoothing=0.0, drop_path=0.0, bs=64, img=300,
                       epochs=30, select="acc", patience_evals=5, patience_after=0.0, clip=0.0, tta=False,
                       head="linear")),
    train_profile=dict(he_stained="std_he", histopath="std_he", ndb_ufes="std_he", clinical="std_photo",
                       breakhis="std_he"),
    exps=dict(
        E1=dict(desc="clean holdout; leaky (L) vs leakage-controlled (C) arms matched on training size, class balance and steps",
                seeds=[0, 1, 2], datasets=dict(
                histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz",
                               target=5000, steps=1300,
                               published="Abd El-Aziz 2025: 1,224 originals augmented to 10,000 before a random 70/15/15 split"),
                ndb_ufes=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                        Cc=dict(S=0, Dup=0, Grp=0, coarse=1)), policy=None, steps=800, extra=["Cc"],
                              published="SciRep 2025: random patch-level 70/15/15 split, augmentation of the training set only"),
                clinical=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                        S0D1=dict(S=0, Dup=1, Grp=0), S1D0=dict(S=1, Dup=0, Grp=0)),
                              policy="keras_generic", target=1200, steps=300, extra=["S0D1", "S1D0"],
                              published="131-image corpus with its duplicates, augmented before a random split"),
                he_stained=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=300,
                                published="shipped OCDC split: test patches come from training slides"),
                breakhis=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=1200,
                              published="random image-level split: patients shared by train and test"))),
        PRE=dict(desc="histopathology corpus as distributed (1,224 originals + 3,968 publisher-augmented copies)",
                 seeds=[0], datasets=dict(histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)),
                                                         policy=None, children=True, steps=900,
                                                         published="SciRep 2025 and Manoj Kumar 2025: the 5,192-file Kaggle corpus split at random"))),
        RL=dict(desc="random-label control: labels drawn per group (source image / slide / patient / duplicate cluster)",
                seeds=[0], outer=[0, 1, 2], datasets=dict(
                histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz", target=5000, steps=1300),
                ndb_ufes=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=800),
                clinical=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy="keras_generic", target=1200, steps=300),
                breakhis=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=1200))),
        DUP=dict(desc="positive control: exact copies of a fraction p of own-evaluation originals injected into training",
                 seeds=[0], outer=[0, 1, 2], doses=[0.0, 0.05, 0.10, 0.20, 0.50], datasets=dict(
                histopath=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz", target=5000, steps=1000),
                ndb_ufes=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=800))),
        OVL=dict(desc="group-overlap dose: fraction q of the evaluation groups' sister images moved into training",
                 seeds=[0], outer=[0, 1, 2], doses=[0.0, 0.25, 0.50, 0.75, 1.0], datasets=dict(
                he_stained=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=300, outer="all"),
                breakhis=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=1000),
                ndb_ufes=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=800))),
        DOSEK=dict(desc="augmentation-factor dose: k copies per original before (L) or after (C) the split, fixed step budget",
                   seeds=[0], outer=[0, 1, 2], doses=[1, 2, 5, 10, 20], datasets=dict(
                histopath=dict(arms=dict(L=dict(S=1, Dup=0, Grp=0), C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz", steps=800),
                ndb_ufes=dict(arms=dict(L=dict(S=1, Dup=0, Grp=0), C=dict(S=0, Dup=0, Grp=0)), policy="he_d4", steps=800)))),
    generality=dict(backbones=["resnet50.a1_in1k", "vit_small_patch16_224.augreg_in21k_ft_in1k", "efficientnet_b0.ra_in1k",
                               "convnext_base.fb_in22k_ft_in1k"],
                    datasets=["histopath", "ndb_ufes", "he_stained", "breakhis", "clinical"], outer=[0, 1, 2],
                    arms=["L", "C"], seed=0),
    reimp=[dict(recipe="scirep", exp="E1", ds="ndb_ufes", backbone="efficientnet_b3.ra2_in1k", train_profile="scirep_ndb",
                arms=["L", "C"], outer=[0, 1, 2], paper="SciRep 2025 (NDB-UFES)"),
           dict(recipe="scirep", exp="PRE", ds="histopath", backbone="efficientnet_b3.ra2_in1k", train_profile="scirep_hist",
                arms=["L", "C"], outer=[0, 1, 2], paper="SciRep 2025 (5,192-file histopathology corpus)"),
           dict(recipe="abdelaziz", exp="E1", ds="histopath", backbone="efficientnet_b3.ra2_in1k", train_profile="none",
                arms=["L", "C"], outer=[0, 1, 2], paper="Abd El-Aziz 2025 (LRT EfficientNet-B3)")],
    probes=dict(backbones=["dinov2_s", "convnext_t"], heads=["logreg", "1nn"], seeds=[0, 1, 2, 3, 4], sib_max=20),
    phases=["core", "controls", "dose", "seeds", "general"],
    hypotheses=[
        "H1 (primary, one test per dataset): in the leaky arm L, balanced accuracy on its own evaluation fold exceeds "
        "balanced accuracy of the same models on the clean holdout (inflation_L > 0).",
        "H2: the difference-in-differences inflation_L - inflation_C is positive.",
        "H3: the leakage-controlled arm is approximately unbiased (95% CI of inflation_C contains 0).",
        "H4: leaky training does not improve true generalisation (holdout BA of L minus C; two-sided, no direction).",
        "H5: inflation increases with augmentation factor k (DOSEK), injected duplicate fraction p (DUP) and group "
        "overlap q (OVL); slope from a linear mixed model with outer fold as random intercept.",
        "H6: with labels randomised per group, L scores above chance on its own fold and C does not.",
        "H7 (descriptive): inflation_L is positive for every backbone family; its relation to parameter count is reported."],
    primary_family="H1 across the available datasets (he_stained, histopath, ndb_ufes, clinical, breakhis); Holm correction; "
                   "two-sided p-values and 95% CIs from a two-stage cluster bootstrap (10,000 resamples: outer folds resampled "
                   "with replacement, then groups resampled within each chosen fold); alpha 0.05. Sensitivity: within-fold "
                   "cluster bootstrap, linear mixed model with outer fold as random intercept and seed as variance component, "
                   "and for OCDC an exact sign-flip test over the seven held-out slides.",
    estimands="own_BA = mean over outer folds and seeds of balanced accuracy on the arm's own evaluation fold; "
              "hold_BA = same models on the clean holdout; inflation = own_BA - hold_BA.",
    selection_rules="Model selection, early stopping, temperature scaling and decision thresholds use the inner validation "
                    "split only (carved from the training portion with the arm's own grouping rules). The own evaluation fold "
                    "and the clean holdout are never used for any choice.",
    exclusions="No run is excluded after the fact. A run whose inner-validation balanced accuracy stays at chance with a flat "
               "loss after 30% of its steps is restarted with a new training seed on the same split (at most two restarts); "
               "runs that still fail are reported as missing.",
    exploratory="Everything not listed under H1-H7 is exploratory: pre-augmented corpus (PRE), published-pipeline "
                "reimplementations, cross-dataset transfer, frozen-feature probes, mechanism analyses, calibration, clinical "
                "operating points, uncertainty decomposition, deletion/insertion saliency metrics, factorial extras, NDB-UFES "
                "coarse grouping, provenance-stratified own-fold accuracy (closest relative in training: sibling, duplicate, "
                "group-mate, none), leakage-risk predictor (leave-one-dataset-out comparison of the training-set mean, the share "
                "of own-fold images with any relative in training, and a ridge model on all audit shares).",
    deviations="All Q1 experiments use 224-px inputs (300 px for the EfficientNet-B3 reimplementations) resized from a 288-px "
               "store, instead of 448-512 px in the original manuscript, to fit five backbones, three seeds and the "
               "dose-response grid into the available GPU time. ConvNeXt-T is the primary backbone; ConvNeXt-B is included "
               "as the high-capacity point.",
)
PLAN_SRC = json.dumps(PLAN, indent=1, sort_keys=True)

# amendments are appended here (never edit PLAN above once it has been saved)
AMENDMENTS = []

pj, ph = Q.D["plan"] / "plan.json", Q.D["plan"] / "plan.sha256.json"
created = False
if not pj.exists():
    stamp = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    base = dict(PLAN, created_utc=stamp, lib_version=Q.LIB_VERSION)
    (Q.D["plan"] / "plan_v1.json").write_text(json.dumps(base, indent=1, sort_keys=True))
    Q.jsave(pj, base)
    sha = Q.sha256_file(Q.D["plan"] / "plan_v1.json")
    Q.jsave(ph, dict(sha256=sha, created_utc=stamp, file="plan_v1.json"))
    md = [f"# {PLAN['title']}", "", f"Written and hashed: {stamp} (before any training run; this is a local timestamp, not a public registration)", f"SHA-256 of plan_v1.json: `{sha}`", "",
          "## Estimands", PLAN["estimands"], "", "## Hypotheses"] + [f"- {h}" for h in PLAN["hypotheses"]] + \
         ["", "## Primary family", PLAN["primary_family"], "", "## Selection rules", PLAN["selection_rules"], "",
          "## Exclusions", PLAN["exclusions"], "", "## Exploratory analyses", PLAN["exploratory"], "",
          "## Deviations from the published pipelines", PLAN["deviations"], "", "## Full configuration", "```json",
          PLAN_SRC, "```"]
    (Q.D["plan"] / "ANALYSIS_PLAN.md").write_text("\n".join(md))
    created = True
stored = Q.jload(pj)
shai = Q.jload(ph)
applied = {a["id"] for a in stored.get("amendments", [])}
for a in AMENDMENTS:
    if a["id"] in applied:
        continue
    for path, val in a["changes"].items():
        node = stored
        keys = path.split(".")
        for k in keys[:-1]:
            node = node[k]
        node[keys[-1]] = val
    stored.setdefault("amendments", []).append(dict(a, applied_utc=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())))
    Q.jsave(pj, stored)


def _drop(d, paths):
    """Copy of d without the dotted paths that an amendment changed (so amended keys do not count as drift)."""
    d = json.loads(json.dumps(d))
    for p_ in paths:
        node, ks = d, p_.split(".")
        for k in ks[:-1]:
            node = node.get(k, {}) if isinstance(node, dict) else {}
        if isinstance(node, dict):
            node.pop(ks[-1], None)
    return d


_amended = [p_ for a in stored.get("amendments", []) for p_ in (a.get("changed") or list(a.get("changes", {})))]
same = json.dumps(_drop({k: v for k, v in stored.items() if k not in ("created_utc", "lib_version", "amendments")}, _amended),
                  sort_keys=True) == json.dumps(_drop(json.loads(PLAN_SRC), _amended), sort_keys=True)

# ------------------------------------------------------------------ report
Q.hdr("BLOCK 15 — Q1 SETUP · LIBRARY · ANALYSIS PLAN (fixed before training)")
import numpy as np
try:
    import torch, timm
    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    tv = f"torch {torch.__version__} · timm {timm.__version__}"
except Exception as e:
    gpus, tv = [], f"torch unavailable ({e})"
clk = Q.session_clock(stored["budget_min"])
Q.kv("library", f"q1lib.py {Q.LIB_VERSION} · {(Q.D['lib'] / 'q1lib.py').stat().st_size // 1024} KB · sha {Q.code_hashes()['q1lib.py']}")
Q.kv("versions", f"{tv} · numpy {np.__version__} · python {sys.version.split()[0]}")
Q.kv("GPUs", " | ".join(gpus) if gpus else "none (CPU)")
Q.kv("work root", str(Q.ROOT))
Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")
Q.kv("session", f"#{clk['n']} · elapsed {Q.hms(clk['elapsed'])} · budget {stored['budget_min']} min · {Q.hms(clk['remaining'])} left")

Q.sub("ANALYSIS PLAN")
Q.kv("status", "WRITTEN NOW (timestamped + hashed)" if created else "already written (unchanged file)")
Q.kv("written (UTC)", shai["created_utc"])
Q.kv("SHA-256 (plan_v1.json)", shai["sha256"])
Q.kv("amendments", len(stored.get("amendments", [])) or "none")
for a in stored.get("amendments", []):
    Q.kv(f"  amendment {a['id']}", f"{a.get('applied_utc', '?')} · {a.get('title', '')[:70]}")
if not same and not created:
    Q.kv("note", "this block's PLAN differs from the stored plan outside the amended keys; the stored plan is kept")
Q.kv("hypotheses", f"{len(stored['hypotheses'])} (H1 primary, Holm across datasets)")
Q.kv("files", "plan/ANALYSIS_PLAN.md · plan/plan_v1.json · plan/plan.sha256.json")

Q.sub("DATASETS FOUND UNDER /kaggle/input")
found, n_img = Q.walk_input()
rows = []
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    cls = np.bincount([r["label"] for r in recs], minlength=Q.META[ds]["K"]).tolist() if recs else []
    rows.append([ds, len(recs), " / ".join(f"{c}:{n}" for c, n in zip(Q.META[ds]["classes"], cls)) if recs else "—",
                 "ready" if recs else ("NOT ATTACHED (optional)" if ds == "breakhis" else "MISSING")])
Q.table(["dataset", "files", "classes", "status"], rows, [13, 8, 58, 22])
Q.kv("image files scanned", n_img)
if not found.get("breakhis"):
    print("\n  BreakHis is optional but needed for the patient-level experiments (report item T2.7).")
    print("  Add it with: Add Input -> search 'breakhis' -> 'ambarish/breakhis', then re-run this block.")
print()
if created:
    print("  >>> Save a Kaggle version NOW (Save Version -> Quick Save). Its timestamp is the external proof that")
    print("      the plan existed before any Q1 experiment was run.")
_nxt = "run BLOCK 16 next" if not (Q.D["index"] / "audit.json").exists() else (
    "run BLOCK 19 next (then the next unfinished block; amendment A1 starts with BLOCK 29)"
    if "A1" not in {a["id"] for a in stored.get("amendments", [])}
    else "run BLOCK 19 next, then the first A1 block not yet finished (30 -> 31 -> 32 -> 25 -> 27 -> 33 -> 28)")
print(f"\n[BLOCK 15 ✓]  plan {shai['sha256'][:12]} · {sum(1 for d in Q.DATASETS if found.get(d))}/5 datasets · {_nxt}")


In [ ]:
# ============================================================================
# BLOCK 19 — TRAINING WORKER + LAUNCHER  (writes q1worker.py, defines RUN(...) and STATUS())
# ============================================================================
import os, sys, json, time, signal, threading, queue, subprocess, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("19")
Q1WORKER_SRC = r'''"""
q1worker.py - one training worker per GPU for the OralNet Q1 study.
Written to disk by Block 19.  Pulls jobs from q1/jobs/jobs.json in priority order.

Guarantees
  * model selection, early stopping, temperature and thresholds use the inner-validation split only
  * exact resume: step-indexed batches and augmentation seeds, atomic checkpoint with .prev rollback
  * OOM -> micro-batching with gradient accumulation (effective batch unchanged)
  * SIGTERM/SIGINT -> checkpoint at the next step and exit (signals in the first 90 s are ignored)
  * collapse -> retry the same split with a new training seed (max 2 retries)
"""
import os, sys, time, math, json, signal, argparse, traceback, gc, warnings
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
import numpy as np
import q1lib as Q

T0 = time.time()
GRACE = int(os.environ.get("Q1_GRACE", "90"))
STOP = {"flag": False, "why": ""}
TEST_PAUSE_AT = int(os.environ.get("Q1_TEST_PAUSE_AT", "-1"))   # tests only: pause once after this step
TAG = "g?"


def log(msg):
    print(f"[{TAG}] {msg}", flush=True)


def _sig(signum, frame):
    if time.time() - T0 < GRACE:
        log(f"signal {signum} ignored (first {GRACE}s after start)")
        return
    STOP["flag"] = True
    STOP["why"] = f"signal {signum}"


class Collapse(Exception):
    pass


def claim(jid):
    p = Q.D["claims"] / f"{jid}.claim"
    try:
        fd = os.open(str(p), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.write(fd, json.dumps(dict(pid=os.getpid(), tag=TAG, t=time.time())).encode())
        os.close(fd)
        return True
    except FileExistsError:
        return False


def release(jid):
    try:
        (Q.D["claims"] / f"{jid}.claim").unlink()
    except FileNotFoundError:
        pass


def note_fail(jid, msg, tb=""):
    p = Q.D["fail"] / f"{jid}.json"
    r = Q.jload(p, {}) or {}
    r["count"] = int(r.get("count", 0)) + 1
    r.setdefault("msgs", []).append(msg[:400])
    r.setdefault("tracebacks", []).append(tb[-2000:])
    Q.jsave(p, r)


def stop_requested():
    return STOP["flag"] or (Q.D["state"] / "STOP").exists()


def save_speed(gpu, bb, ips):
    p = Q.D["state"] / f"speed_{gpu}.json"
    s = Q.jload(p, {}) or {}
    s.setdefault(bb, []).append(float(ips))
    s[bb] = s[bb][-30:]
    Q.jsave(p, s)


def atomic_torch_save(obj, path):
    import torch
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    if path.exists():
        os.replace(path, path.with_name(path.name + ".prev"))
    os.replace(tmp, path)


def load_ckpt(path, fp):
    import torch
    for p in (path, path.with_name(path.name + ".prev")):
        if p.exists():
            try:
                ck = torch.load(p, map_location="cpu", weights_only=False)
                if ck.get("fp") == fp:
                    return ck
            except Exception:
                continue
    return None


def run_job(job, dev, gpu, deadline, nth, total):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    spec, rid, fp = job["spec"], job["id"], job["fp"]
    P = Q.plan()
    R = dict(P["recipes"][spec["recipe"]])
    ds = spec["ds"]
    B = Q.build_job(spec, check=True)
    info = B["info"]
    K, pos = info["K"], Q.META[ds]["pos"]
    policy = info["policy"]
    prof = spec.get("train_profile") or P["train_profile"][ds]
    img = 48 if Q.TINY else R["img"]
    bs = min(R["bs"], 16) if Q.TINY else R["bs"]
    tr, iv = B["train"], B["ival"]
    n_tr = len(tr)
    steps, every = Q.recipe_steps(spec, dict(R, bs=bs), n_tr)
    sets = dict(ival=B["ival"], own=B["own"], hold=B["hold"])
    rows_by_ds = {ds: np.concatenate([tr["row"]] + [u["row"] for u in sets.values()])}
    for t, u in B["xeval"].items():
        rows_by_ds[t] = u["row"]
    feed = Q.Feeder(rows_by_ds, dev)
    sib = {k: Q.sibling_params(ds, u["row"], u["copy"], policy) for k, u in list(sets.items()) + [("train", tr)]}
    ckp = Q.D["ckpt"] / f"{rid}.pt"
    ck = load_ckpt(ckp, fp)
    attempt = int(ck["attempt"]) if ck else 0
    short = f"{spec['exp']}·{Q.SHORT[ds]}·{spec['arm']}·o{spec['outer']}{'r' + str(spec['rep']) if spec.get('rep') else ''}·s{spec['seed']}" + \
            (f"·d{spec['dose']}" if spec.get("dose") is not None else "")
    bbl = Q.BACKBONES.get(spec["backbone"], (spec["backbone"],))[0] + ("" if spec["recipe"] == "std" else f"/{spec['recipe']}")
    while True:
        try:
            status = _train_once(job, spec, R, B, info, K, pos, policy, prof, img, bs, steps, every, feed, sib,
                                 dev, gpu, deadline, ck, attempt, short, bbl, nth, total)
            return status
        except Collapse as e:
            attempt += 1
            ck = None
            if ckp.exists():
                ckp.unlink()
            if attempt > 2:
                raise RuntimeError(f"collapsed on 3 training seeds: {e}")
            log(f"  {short}  collapse ({e}) -> retry {attempt}/2 with a new training seed (same split)")


def _train_once(job, spec, R, B, info, K, pos, policy, prof, img, bs, steps, every, feed, sib, dev, gpu,
                deadline, ck, attempt, short, bbl, nth, total):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    rid, fp, ds = job["id"], job["fp"], spec["ds"]
    tr = B["train"]
    n_tr = len(tr)
    tseed = Q.shash("train", rid, attempt)
    torch.manual_seed(tseed % (2 ** 31))
    model, head_params, dense = Q.make_model(spec["backbone"], K, img, R.get("drop_path", 0.0), R.get("head", "linear"))
    model = model.to(dev).to(memory_format=torch.channels_last)
    mean, std = Q.norm_consts(model, dev)
    hid = {id(p) for p in head_params}
    groups = {"hd": [], "hn": [], "bd": [], "bn": []}
    for n_, p in model.named_parameters():
        if not p.requires_grad:
            continue
        h = "h" if id(p) in hid else "b"
        d = "n" if (p.ndim <= 1 or n_.endswith(".bias")) else "d"
        groups[h + d].append(p)
    wd = R.get("wd", 0.0)
    pg = [dict(params=groups["bd"], lr=R["lr_backbone"], weight_decay=wd, base=R["lr_backbone"]),
          dict(params=groups["bn"], lr=R["lr_backbone"], weight_decay=0.0, base=R["lr_backbone"]),
          dict(params=groups["hd"], lr=R["lr_head"], weight_decay=wd, base=R["lr_head"]),
          dict(params=groups["hn"], lr=R["lr_head"], weight_decay=0.0, base=R["lr_head"])]
    pg = [g for g in pg if g["params"]]
    if R["opt"] == "adamw":
        opt = torch.optim.AdamW(pg, betas=(0.9, 0.999))
    elif R["opt"] == "adamax":
        opt = torch.optim.Adamax(pg)
    else:
        opt = torch.optim.Adam(pg)
    use_amp = dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    act = {}
    if dense is not None:
        dense.register_forward_hook(lambda m, i, o: act.__setitem__("a", o))
    st = dict(step=0, lr_mult=1.0, best=-1e9, best_step=-1, best_state=None, since=0, plat_best=-1e9, plat_since=0,
              hist=[], micro=1, loss0=None, loss_recent=[], t_train=0.0, n_seen=0)
    if ck:
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        st.update(ck["st"])
        log(f"  {short}  RESUMED at step {st['step']}/{steps}")
    ls = R.get("label_smoothing", 0.0)
    warm = min(int(R.get("warmup_frac", 0) * steps), int(R.get("warmup_max", 0))) if R.get("sched") == "cosine" else 0
    spe = max(1, n_tr // bs)
    gen = torch.Generator(device=dev)
    ytr = torch.from_numpy(tr["y"]).to(dev)
    perm_cache = {}

    def lr_factor(s):
        if R.get("sched") == "cosine":
            if s < warm:
                return (s + 1) / max(1, warm)
            t = (s - warm) / max(1, steps - warm)
            mn = R.get("min_lr_frac", 0.01)
            return mn + (1 - mn) * 0.5 * (1 + math.cos(math.pi * t))
        return st["lr_mult"]

    def predict(u, name, dsx=None, pol=None, tta=None):
        dsx = dsx or ds
        tta = R.get("tta", False) if tta is None else tta
        prm = sib.get(name) if pol is not None else None
        model.eval()
        out = []
        eb = 128 if not Q.TINY else 32
        with torch.no_grad():
            for s in range(0, len(u), eb):
                sl = slice(s, s + eb)
                x = feed.batch(dsx, u["row"][sl], u["copy"][sl], None if prm is None else prm[sl], pol)
                x = ((Q.eval_prep(x, img) - mean) / std).contiguous(memory_format=torch.channels_last)
                with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    lg = model(x).float()
                    if tta:
                        lg = (lg + model(x.flip(3)).float()) / 2
                out.append(lg.cpu().numpy())
        model.train()
        return np.concatenate(out) if out else np.zeros((0, K), np.float32)

    def evaluate_ival():
        lg = predict(B["ival"], "ival", pol=policy, tta=False)
        p = Q.softmax(lg)
        m = Q.metrics(p, B["ival"]["y"], K, pos)
        sel = R.get("select", "bal_acc")
        score = {"bal_acc": m["bal_acc"] - 1e-4 * m["nll"], "nll": -m["nll"], "acc": m["acc"]}[sel]
        plat = {"nll": -m["nll"], "acc": m["acc"], "bal_acc": m["bal_acc"]}[R.get("plateau_metric", "nll")]
        return m, score, plat

    def save(final=False):
        atomic_torch_save(dict(fp=fp, attempt=attempt, model=model.state_dict(), opt=opt.state_dict(),
                               scaler=scaler.state_dict(), st=st), Q.D["ckpt"] / f"{rid}.pt")

    if st["step"] == 0:
        log(f"▶ {short}  {bbl}  train {n_tr} · ival {len(B['ival'])} · own {len(B['own'])} · hold {len(B['hold'])}"
            f"  · {steps} steps  [{nth}/{total}]" + (f"  attempt {attempt + 1}" if attempt else ""))
    model.train()
    last_ck = time.time()
    t_start = time.time()
    marks = {int(steps * f) for f in (0.25, 0.5, 0.75)}
    while st["step"] < steps:
        s = st["step"]
        torch.manual_seed(int(Q.shash(tseed, "step", s) % (2 ** 31)))   # dropout / drop-path masks depend on the step only
        e, b = divmod(s, spe)
        if e not in perm_cache:
            perm_cache.clear()
            perm_cache[e] = np.random.default_rng(Q.shash(tseed, "perm", e)).permutation(n_tr)
        idx = perm_cache[e][b * bs:(b + 1) * bs]
        if len(idx) < bs:
            idx = perm_cache[e][:bs]
        for g in opt.param_groups:
            g["lr"] = g["base"] * lr_factor(s)
        t1 = time.time()
        while True:
            try:
                opt.zero_grad(set_to_none=True)
                parts = np.array_split(idx, st["micro"])
                tot = 0.0
                for pi, part in enumerate(parts):
                    x = feed.batch(ds, tr["row"][part], tr["copy"][part], sib["train"][part], policy)
                    gen.manual_seed(int(Q.shash(tseed, "aug", s, pi) % (2 ** 62)))
                    x = Q.train_aug(x, img, prof, gen)
                    x = ((x - mean) / std).contiguous(memory_format=torch.channels_last)
                    yb = ytr[torch.from_numpy(part).to(dev)]
                    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                        lg = model(x)
                        loss = F.cross_entropy(lg.float(), yb, label_smoothing=ls)
                        if dense is not None:
                            loss = loss + R.get("l2_kernel", 0) * dense.weight.float().pow(2).sum() \
                                   + R.get("l1_bias", 0) * dense.bias.float().abs().sum() \
                                   + R.get("l1_activity", 0) * act["a"].float().abs().sum(1).mean()
                    scaler.scale(loss * (len(part) / len(idx))).backward()
                    tot += float(loss) * len(part) / len(idx)
                if R.get("clip", 0) > 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), R["clip"])
                scaler.step(opt)
                scaler.update()
                break
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True)
                gc.collect()
                torch.cuda.empty_cache()
                if bs // (st["micro"] * 2) < 2:
                    raise
                st["micro"] *= 2
                log(f"  {short}  OOM -> micro-batch {bs // st['micro']} x {st['micro']} (effective batch {bs} kept)")
        st["t_train"] += time.time() - t1
        st["n_seen"] += len(idx)
        if not math.isfinite(tot):
            raise Collapse("non-finite loss")
        if st["loss0"] is None and s < 20:
            st.setdefault("_l0", []).append(tot)
            if len(st["_l0"]) >= min(20, steps // 5 + 1):
                st["loss0"] = float(np.mean(st["_l0"]))
        st["loss_recent"] = (st["loss_recent"] + [tot])[-50:]
        st["step"] = s + 1
        s1 = st["step"]
        if s1 % every == 0 or s1 == steps:
            m, score, plat = evaluate_ival()
            st["hist"].append(dict(step=s1, lr=opt.param_groups[0]["lr"], loss=float(np.mean(st["loss_recent"])),
                                   ival_ba=m["bal_acc"], ival_acc=m["acc"], ival_nll=m["nll"]))
            if score > st["best"]:
                st["best"], st["best_step"], st["since"] = score, s1, 0
                st["best_ba"] = m["bal_acc"]
                st["best_state"] = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
            else:
                st["since"] += 1
            if R.get("sched") == "plateau":
                if plat > st["plat_best"]:
                    st["plat_best"], st["plat_since"] = plat, 0
                else:
                    st["plat_since"] += 1
                    if st["plat_since"] >= R.get("plateau_patience", 2):
                        st["lr_mult"] *= R.get("plateau_factor", 0.1)
                        st["plat_since"] = 0
            guard = not str(spec["exp"]).endswith("RL") and steps >= 100 and not Q.TINY
            if guard and s1 >= 0.3 * steps and len(st["hist"]) >= 2 and not st.get("collapse_checked"):
                st["collapse_checked"] = True
                l0 = st["loss0"] or st["hist"][0]["loss"]
                if m["bal_acc"] <= 1.0 / K + 0.02 and np.mean(st["loss_recent"]) > 0.97 * l0:
                    raise Collapse(f"inner-val BA {100 * m['bal_acc']:.1f}% at chance, loss flat")
            ips = st["n_seen"] / max(1e-6, st["t_train"])
            mk = max([mm for mm in marks if mm <= s1], default=None)
            if mk is not None and s1 - every < mk <= s1 or s1 == steps:
                mem = torch.cuda.max_memory_allocated(dev) / 1e9 if dev.type == "cuda" else 0.0
                log(f"  {short}  step {s1}/{steps} · loss {np.mean(st['loss_recent']):.3f} · ival BA {100 * m['bal_acc']:.1f}"
                    f" (best {100 * st.get('best_ba', m['bal_acc']):.1f}) · lr {opt.param_groups[0]['lr']:.1e}"
                    f" · {ips:.0f} img/s · {mem:.1f}GB")
            if st["since"] >= R.get("patience_evals", 4) and s1 >= R.get("patience_after", 0.5) * steps:
                log(f"  {short}  early stop at step {s1} (no inner-val gain for {st['since']} evaluations)")
                break
        if time.time() - last_ck > 60 and st["step"] < steps:     # checkpoint about once a minute, between any two steps
            save()
            last_ck = time.time()
        if stop_requested() or time.time() > deadline - 90 or st["step"] == TEST_PAUSE_AT:
            save()
            why = STOP["why"] or ("stop requested" if (Q.D["state"] / "STOP").exists() else
                                  ("test pause" if st["step"] == TEST_PAUSE_AT else "session deadline"))
            log(f"  {short}  PAUSED at step {st['step']}/{steps} ({why}) - resumable")
            return "paused"
    # ---------------------------------------------------------------- final predictions
    if st["best_state"] is not None:
        model.load_state_dict(st["best_state"])
    out = {}
    for name in ("ival", "own", "hold"):
        u = B[name]
        lg = predict(u, name, pol=policy if name != "hold" else None)
        for k in ("row", "copy", "y", "key"):
            out[f"{name}_{k}"] = u[k]
        out[f"{name}_logit"] = lg.astype(np.float32)
    for t, u in B["xeval"].items():
        lg = predict(u, "x_" + t, dsx=t, pol=None)
        for k in ("row", "copy", "y", "key"):
            out[f"x_{t}_{k}"] = u[k]
        out[f"x_{t}_logit"] = lg.astype(np.float32)
    if "own_injected_mask" in B:
        out["own_injected"] = B["own_injected_mask"].astype(np.int8)
    rdir = Q.D["runs"] / rid
    rdir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(rdir / "preds.npz", **out)
    if spec.get("keep_weights"):
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, rdir / "weights.pt")
    T = Q.fit_temperature(out["ival_logit"], out["ival_y"])
    m_own = Q.metrics(Q.softmax(out["own_logit"], T), out["own_y"], K, pos)
    m_hold = Q.metrics(Q.softmax(out["hold_logit"], T), out["hold_y"], K, pos)
    ips = st["n_seen"] / max(1e-6, st["t_train"])
    save_speed(gpu, spec["backbone"], ips * (img / 224.0) ** 2)
    res = dict(id=rid, fp=fp, status="done", spec=spec, recipe=R, train_profile=prof, img=img, bs=bs, steps=steps,
               steps_done=st["step"], best_step=st["best_step"], hist=st["hist"], attempts=attempt + 1, micro=st["micro"],
               info=info, T_ival=T, own_bal_acc=m_own.get("bal_acc"), hold_bal_acc=m_hold.get("bal_acc"),
               own_acc=m_own.get("acc"), hold_acc=m_hold.get("acc"), time_s=time.time() - t_start, img_per_s=ips,
               device=(torch.cuda.get_device_name(0) if dev.type == "cuda" else "cpu"),
               versions=dict(torch=torch.__version__, timm=__import__("timm").__version__, numpy=np.__version__),
               code=Q.code_hashes(), finished_utc=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()))
    Q.jsave(rdir / "result.json", res)
    ckp = Q.D["ckpt"] / f"{rid}.pt"
    for p in (ckp, ckp.with_name(ckp.name + ".prev")):
        if p.exists():
            p.unlink()
    gap = (m_own.get("bal_acc", 0) - m_hold.get("bal_acc", 0)) * 100
    log(f"✓ {short}  own BA {100 * m_own['bal_acc']:.2f} · hold BA {100 * m_hold['bal_acc']:.2f} · gap {gap:+.2f}"
        f" · best@{st['best_step']} · {Q.hms(time.time() - t_start)}")
    del model, opt
    gc.collect()
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return "done"


def main():
    global TAG
    ap = argparse.ArgumentParser()
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--phase", default="core")
    ap.add_argument("--deadline", type=float, default=time.time() + 10 * 3600)
    ap.add_argument("--datasets", default="")
    ap.add_argument("--max_jobs", type=int, default=100000)
    a = ap.parse_args()
    TAG = f"g{a.gpu}"
    signal.signal(signal.SIGTERM, _sig)
    signal.signal(signal.SIGINT, _sig)
    import torch
    torch.backends.cudnn.benchmark = True
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    Q.ensure_dirs()
    phases = [p for p in a.phase.split(",") if p]
    dsf = [d for d in a.datasets.split(",") if d]
    log(f"worker up · pid {os.getpid()} · {torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu'} · phases {','.join(phases)}"
        f" · {Q.hms(a.deadline - time.time())} to deadline")
    n_done = n_fail = n_pause = 0
    skipped = set()
    while not stop_requested() and n_done + n_fail < a.max_jobs:
        J = Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}
        jobs = [j for j in J.get("jobs", []) if j["phase"] in phases and (not dsf or j["spec"]["ds"] in dsf)]
        total = len(jobs)
        pending = [j for j in jobs if j["id"] not in skipped and not Q.job_done(j) and Q.fail_count(j["id"]) < 3]
        speed = Q.merged_speed()
        picked = None
        for j in pending:
            est = j["est_min"]
            bb = j["spec"]["backbone"]
            if bb in speed and Q.BACKBONES.get(bb):
                est = est * Q.BACKBONES[bb][2] / max(1.0, speed[bb])
            if Q.TINY:
                est = 0.05
            if time.time() + est * 60 * 1.15 + 120 > a.deadline and not (Q.D["ckpt"] / f"{j['id']}.pt").exists():
                skipped.add(j["id"])
                continue
            if claim(j["id"]):
                picked = j
                break
        if picked is None:
            break
        nth = 1 + sum(1 for j in jobs if Q.job_done(j))
        try:
            st = run_job(picked, dev, a.gpu, a.deadline, nth, total)
            if st == "done":
                n_done += 1
            else:
                n_pause += 1
                release(picked["id"])
                break
        except Exception as e:
            n_fail += 1
            msg = f"{type(e).__name__}: {e}"
            note_fail(picked["id"], msg, traceback.format_exc())
            skipped.add(picked["id"])
            log(f"✗ {picked['id']} FAILED · {msg[:300]}")
            gc.collect()
            if dev.type == "cuda":
                torch.cuda.empty_cache()
        release(picked["id"])
    left = [j for j in (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
            if j["phase"] in phases and (not dsf or j["spec"]["ds"] in dsf) and not Q.job_done(j)]
    log(f"worker exit · {n_done} done · {n_pause} paused · {n_fail} failed · {len(left)} left in phase"
        f" · {Q.hms(a.deadline - time.time())} to deadline")


if __name__ == "__main__":
    main()
'''
(Q.D["lib"] / "q1worker.py").write_text(Q1WORKER_SRC)
compile(Q1WORKER_SRC, "q1worker.py", "exec")
WORKER = Q.D["lib"] / "q1worker.py"


def _alive(pid):
    try:
        os.kill(int(pid), 0)
        return True
    except Exception:
        return False


def _clear_stale_claims():
    n = 0
    for c in Q.D["claims"].glob("*.claim"):
        info = Q.jload(c, {}) or {}
        if not _alive(info.get("pid", -1)):
            c.unlink()
            n += 1
    return n


def _prefetch(backbones):
    if Q.OFFLINE:
        return
    import timm
    for bb in sorted(set(backbones)):
        try:
            m = timm.create_model(Q.tiny_name(bb), pretrained=True, num_classes=0)
            del m
        except Exception as e:
            print(f"  prefetch {bb}: {type(e).__name__} {str(e)[:80]}")


def _phase_table(phases, datasets=None):
    J = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
    rows, agg = [], {}
    for j in J:
        if j["phase"] not in phases or (datasets and j["spec"]["ds"] not in datasets):
            continue
        s = j["spec"]
        k = (s["exp"], s["ds"], s["arm"], Q.BACKBONES.get(s["backbone"], (s["backbone"],))[0] +
             ("" if s["recipe"] == "std" else "/" + s["recipe"]))
        a = agg.setdefault(k, dict(n=0, done=0, own=[], hold=[], fail=0, paused=0))
        a["n"] += 1
        r = Q.jload(Q.D["runs"] / j["id"] / "result.json")
        if r and r.get("status") == "done" and r.get("fp") == j["fp"]:
            a["done"] += 1
            if r.get("own_bal_acc") is not None:
                a["own"].append(r["own_bal_acc"])
                a["hold"].append(r["hold_bal_acc"])
        elif Q.fail_count(j["id"]) >= 3:
            a["fail"] += 1
        elif (Q.D["ckpt"] / f"{j['id']}.pt").exists():
            a["paused"] += 1
    for (ex, ds, arm, bb), a in sorted(agg.items()):
        o = 100 * np.mean(a["own"]) if a["own"] else None
        h = 100 * np.mean(a["hold"]) if a["hold"] else None
        rows.append([ex, ds, arm, bb, f"{a['done']}/{a['n']}", "—" if o is None else f"{o:.2f}",
                     "—" if h is None else f"{h:.2f}", "—" if o is None else f"{o - h:+.2f}",
                     (f"{a['fail']} failed " if a["fail"] else "") + (f"{a['paused']} paused" if a["paused"] else "")])
    return rows


def STATUS():
    importlib.reload(Q)
    P = Q.plan()
    J = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
    Q.hdr("Q1 STATUS")
    rows = []
    for ph in P["phases"]:
        jj = [j for j in J if j["phase"] == ph]
        if not jj:
            continue
        d = sum(Q.job_done(j) for j in jj)
        f = sum(Q.fail_count(j["id"]) >= 3 for j in jj)
        left = sum(j["est_min"] for j in jj if not Q.job_done(j)) / 60
        rows.append([ph, f"{d}/{len(jj)}", f, f"{left:.1f}", f"{left / 2:.1f}"])
    Q.table(["phase", "done", "failed", "GPU-h left", "wall h (2 GPU)"], rows, [10, 12, 8, 12, 15])
    Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")


def RUN(phase, datasets=None, budget_min=None):
    """Train every pending job of `phase` (comma-separated phases allowed) on all GPUs, resumably."""
    importlib.reload(Q)
    P = Q.plan()
    phases = [p for p in phase.split(",") if p]
    clk = Q.session_clock(P["budget_min"] if budget_min is None else budget_min)
    deadline = clk["deadline"]
    J = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
    if not J:
        print("jobs.json missing - run BLOCK 18 first")
        return
    JJ = [j for j in J if j["phase"] in phases and (not datasets or j["spec"]["ds"] in datasets)]
    stale = _clear_stale_claims()
    stopf = Q.D["state"] / "STOP"
    if stopf.exists():
        stopf.unlink()
    pend = [j for j in JJ if not Q.job_done(j) and Q.fail_count(j["id"]) < 3]
    resum = [j for j in pend if (Q.D["ckpt"] / f"{j['id']}.pt").exists()]
    try:
        import torch
        ngpu = max(1, torch.cuda.device_count())
        gname = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    except Exception:
        ngpu, gname = 1, "cpu"
    Q.hdr(f"TRAINING · phase {'+'.join(phases).upper()} · session {clk['n']}")
    Q.kv("content", " | ".join(Q.PHASE_DESC.get(p, p) for p in phases))
    Q.kv("jobs", f"{len(JJ)} in phase · {len(JJ) - len(pend)} done · {len(pend)} to run · {len(resum)} resume mid-run")
    Q.kv("estimated work", f"{sum(j['est_min'] for j in pend) / 60:.1f} GPU-h -> ~{sum(j['est_min'] for j in pend) / 60 / ngpu:.1f} h on {ngpu} x {gname}")
    Q.kv("session", f"{Q.hms(clk['elapsed'])} elapsed · {Q.hms(deadline - time.time())} until the stop line (budget {P['budget_min']} min)")
    Q.kv("safety", "inner-val selection only · checkpoint ~every minute · OOM -> micro-batches · collapse -> reseed x2")
    Q.kv("interrupt", "press Stop: workers checkpoint within ~1 step and exit (allow up to 2 minutes)")
    Q.kv("stale claims cleared", stale)
    Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")
    if not pend:
        print("\n  nothing to run in this phase.")
    else:
        _prefetch([j["spec"]["backbone"] for j in pend])
        env = dict(os.environ, PYTHONWARNINGS="ignore", HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false",
                   OMP_NUM_THREADS="2", PYTHONUNBUFFERED="1")
        procs, q_ = [], queue.Queue()
        for g in range(ngpu):
            e = dict(env, CUDA_VISIBLE_DEVICES=str(g)) if gname != "cpu" else env
            p = subprocess.Popen([sys.executable, "-u", str(WORKER), "--gpu", str(g), "--phase", ",".join(phases),
                                  "--deadline", str(deadline), "--datasets", ",".join(datasets or [])],
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e,
                                 start_new_session=True)
            procs.append(p)
            threading.Thread(target=lambda pp=p: [q_.put(l) for l in pp.stdout], daemon=True).start()
            time.sleep(3)
        Q.kv("launched", f"{len(procs)} worker(s) · pids {[p.pid for p in procs]}")
        print("=" * Q.W, flush=True)

        def pump(timeout=0.5):
            try:
                while True:
                    l = q_.get(timeout=timeout).rstrip()
                    if l.startswith("[g") or "Error" in l or "Traceback" in l:
                        print(l, flush=True)
                    timeout = 0.01
            except queue.Empty:
                pass

        try:
            while any(p.poll() is None for p in procs):
                pump()
            pump(0.2)
        except KeyboardInterrupt:
            print("\n  interrupt — asking workers to checkpoint and exit ...", flush=True)
            stopf.write_text(str(time.time()))
            for p in procs:
                try:
                    os.killpg(os.getpgid(p.pid), signal.SIGTERM)
                except Exception:
                    pass
            t0 = time.time()
            while any(p.poll() is None for p in procs) and time.time() - t0 < 150:
                pump()
            for p in procs:
                if p.poll() is None:
                    os.killpg(os.getpgid(p.pid), signal.SIGKILL)
            pump(0.2)
            print("  stopped; progress is checkpointed.", flush=True)
    _clear_stale_claims()
    if stopf.exists():
        stopf.unlink()
    # ------------------------------------------------ auto-save summary
    Q.hdr(f"AUTO-SAVE · phase {'+'.join(phases)}")
    rows = _phase_table(phases, datasets)
    Q.table(["exp", "dataset", "arm", "backbone", "done", "own BA", "hold BA", "gap", "notes"], rows,
            [7, 12, 6, 22, 9, 9, 9, 9, 22])
    print("  own BA = balanced accuracy on the arm's own evaluation fold (what a paper would report);")
    print("  hold BA = the same models on the clean holdout; gap = own - hold (percentage points).")
    fails = [(j["id"], (Q.jload(Q.D["fail"] / f"{j['id']}.json", {}) or {}).get("msgs", [""])[-1])
             for j in JJ if Q.fail_count(j["id"]) >= 1 and not Q.job_done(j)]
    if fails:
        Q.sub("FAILURES (retried automatically on the next run, up to 3 attempts)")
        for jid, m in fails[:15]:
            print(f"  {jid}: {m[:160] if m else ''}")
    left = [j for j in JJ if not Q.job_done(j) and Q.fail_count(j["id"]) < 3]
    Q.kv("remaining in phase", f"{len(left)} jobs · {sum(j['est_min'] for j in left) / 60:.1f} GPU-h")
    Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")
    print("  Before closing Kaggle: Persistence = 'Files only', then Save Version.")
    print(f"\n[AUTO-SAVE ✓]  {'phase complete' if not left else 're-run this block to continue (next session: 15 -> 19 -> this block)'}")


Q.hdr("BLOCK 19 — WORKER + LAUNCHER READY")
Q.kv("worker", f"q1worker.py · {WORKER.stat().st_size // 1024} KB · syntax PASS · sha {Q.code_hashes()['q1worker.py']}")
Q.kv("rules", "selection / early stop / temperature / thresholds on inner validation only")
Q.kv("resume", "same batches, augmentations and dropout masks at every step as an uninterrupted run; .prev rollback")
Q.kv("launch", "RUN('core') · RUN('controls') · RUN('dose') · RUN('seeds') · RUN('general') · RUN('orchid,recover') · STATUS()")
STATUS()
print("\n[BLOCK 19 ✓]  run the training blocks 20-24, or BLOCK 32 for amendment A1 (each can be re-run to resume)")


In [ ]:
# ============================================================================
# BLOCK 34 — AMENDMENT A2 TO THE ANALYSIS PLAN (written and hashed before any A2 data step or training run)
# NDB-UFES at patient level with the dataset authors' labels · grouping ladder · external validation ORCHID <-> Rahman
# needs: BLOCK 15 run in this session; the finished Q1 and A1 runs, jobs.json and plan.json on disk
# ============================================================================
import os, sys, json, time, copy, hashlib, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
_libsrc = (Q1_ROOT / "lib" / "q1lib.py").read_text() if (Q1_ROOT / "lib" / "q1lib.py").exists() else ""
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library, written by an old BLOCK 15 cell. Amendment A2 needs the A1 "
                     "library: replace the whole content of the BLOCK 15 and BLOCK 19 cells with the A1 versions (the first two "
                     "code cells of the A2 notebook), run 15 -> 19, then run this block again.")
import numpy as np

# ------------------------------------------------------------------ 0  library extension used by every A2 block and by the workers
A2EXT_SRC = r'''"""
q1_a2ext.py - amendment A2 extension of q1lib (written by BLOCK 34; imported at the end of q1lib.py).
Adds external evaluation sets that a job names in spec["xeval_to"]; every image of the target dataset that is usable,
original and one per exact duplicate is evaluated, except members of identical-file groups that carry two labels.
The A2 datasets are new indexes over existing image stores: open_store() reads ndb_pat and ndb_pat_k from the
ndb_ufes store and orchid_bin from the orchid store (no copies, no symbolic links: Kaggle's "Files only" persistence
does not keep links). Nothing else in q1lib changes, so Q1 and A1 job fingerprints and results are untouched.
"""
import numpy as np

STORE_ALIAS = {"ndb_pat": "ndb_ufes", "ndb_pat_k": "ndb_ufes", "orchid_bin": "orchid"}


def install(ns):
    if getattr(ns["build_job"], "_a2_wrapped", False):     # a reload of q1lib defines an unwrapped build_job again
        return
    _orig_build_job = ns["build_job"]
    Units, DS = ns["Units"], ns["DS"]

    def ext_units(t):
        dd = DS.get(t)
        lab_by_exact = {}
        for i in np.nonzero(dd.ok & dd.orig)[0]:
            lab_by_exact.setdefault(int(dd.g_exact[i]), set()).add(int(dd.y[i]))
        conflict = np.array([len(lab_by_exact.get(int(g), ())) > 1 for g in dd.g_exact])
        r = np.nonzero(dd.base & ~conflict)[0]
        return Units.make(r, np.zeros(len(r), int), dd.y[r], dd.g_ctrl[r], t)

    def build_job(spec, check=True):
        B = _orig_build_job(spec, check)
        tg = list(spec.get("xeval_to") or [])
        if tg:
            B["xeval"] = {t: ext_units(t) for t in tg}
            B["info"]["n_xeval"] = int(sum(len(u) for u in B["xeval"].values()))
        return B

    _orig_open_store = ns["open_store"]

    def open_store(ds):
        return _orig_open_store(STORE_ALIAS.get(ds, ds))

    build_job._a2_wrapped = True
    ns["build_job"] = build_job
    ns["open_store"] = open_store
    ns["STORE_ALIAS"] = STORE_ALIAS
    ns["ext_units"] = ext_units
    ns["PHASE_DESC"].update({
        "a2main": "A2: NDB-UFES patient level (L vs C, authors' labels) + external validation ORCHID -> Rahman",
        "a2ctrl": "A2: NDB-UFES grouping ladder, random labels per patient, Kaggle-label arms",
        "a2ext": "A2: external validation Rahman (Kaggle copy) -> ORCHID, seed 0",
        "a2opt": "A2 (optional): external validation Rahman -> ORCHID, seeds 1-2"})
    ns["_A2_INSTALLED"] = True
'''
HOOK = ("\n# --- amendment A2 hook (added by the A2 blocks; re-added automatically after BLOCK 15 rewrites this file)\n"
        "try:\n    import q1_a2ext as _a2ext\n    _a2ext.install(globals())\nexcept ImportError:\n    pass\n")


def ensure_a2_hook():
    (Q1_ROOT / "lib" / "q1_a2ext.py").write_text(A2EXT_SRC)
    lib = Q1_ROOT / "lib" / "q1lib.py"
    src = lib.read_text()
    if "amendment A2 hook" not in src:
        lib.write_text(src + HOOK)


ensure_a2_hook()
import q1lib as Q
importlib.reload(Q)
Q.keep_source("34")
assert Q.__dict__.get("_A2_INSTALLED"), "A2 extension did not load"

Q.hdr("BLOCK 34 — ANALYSIS-PLAN AMENDMENT A2 (before any A2 data step or run)")
P = Q.plan()
if "A1" not in {a["id"] for a in P.get("amendments", [])}:
    raise SystemExit("amendment A1 is not in the plan - A2 builds on it (run BLOCKS 29-33 first)")
L_ = dict(S=0, Dup=1, Grp=1)          # patch- or image-level split (the published handling)
C_ = dict(S=0, Dup=0, Grp=0)          # split by the dataset's true grouping (recovered patient / patient folder / source image)
A2_DOC = dict(
    id="A2",
    title="NDB-UFES at patient level with the dataset authors' labels, a grouping ladder, and external validation",
    rationale=(
        "After A1, the NDB-UFES copy used by both studies was matched to the dataset authors' release (Mendeley Data "
        "10.17632/bbmmm4wgr8.4). All 3,763 files are byte-identical to the authors' patches, but the copy files the "
        "authors' 627-patch test block under 'Healthy', so 509 patches carry another class, and the Study 2 groups "
        "recovered from file order do not separate patients. A2 (i) recovers the patient of every patch by locating it "
        "in the authors' source images, (ii) repeats the primary comparison on NDB-UFES with a patient-disjoint clean "
        "holdout and the authors' labels, (iii) measures how much each approximate grouping leaves of the leak, and "
        "(iv) tests whether leaky training changes performance on an oral collection from another institution."),
    hypotheses=[
        "H13 (confirmatory, family A2-1, one test, alpha 0.05, two-sided two-stage cluster bootstrap over outer folds and "
        "patient groups, 10,000 resamples): on NDB-UFES with recovered patients and the authors' labels, the patch-level "
        "split (arm L) over-estimates patient-disjoint balanced accuracy (inflation_L > 0); experiment X3, seeds 0-2.",
        "H14 (estimation): grouping ladder on the same holdout - arms split by Study 2's fine segments (Cseg, the grouping "
        "of the Study 2 controlled arm), by Study 2's file-order blocks (Cblk, the grouping of the Study 2 holdout), by "
        "source image (Cimg) and by patient (C). For each arm: inflation with 95% CI and the share of own-fold patches "
        "with a training patch of the same patient. Expected order L >= Cseg >= Cblk, Cimg >= C, with C's CI containing 0.",
        "H15 (control): with labels drawn per patient group (X3RL), arms L and Cblk score above chance on their own fold "
        "and C does not (own-fold BA versus chance 1/3, bootstrap p).",
        "H16 (confirmatory, family A2-2, Holm over two tests, alpha 0.05): the own-fold balanced accuracy of the leaky arm "
        "over-states its external balanced accuracy more than that of the controlled arm, "
        "D_ext = (own_L - ext_L) - (own_C - ext_C) > 0, for (a) ORCHID normal vs OSCC -> Rahman et al. originals "
        "(X4A, seeds 0-2) and (b) Rahman et al. (Kaggle copy, augmentation before the split as in Abd El-Aziz 2025) -> "
        "ORCHID normal vs OSCC (X4B, seed 0; seeds 1-2 are a sensitivity analysis if run). Two-stage bootstrap: outer "
        "folds, then clusters (own fold: the arm's groups; external set: its control groups).",
        "H17 (estimation, two-sided): G_ext = ext_L - ext_C (does leaky training change external performance?), with "
        "95% CI, overall and by magnification of the Rahman images (100x, 400x).",
        "H18 (descriptive): (a) Study 2 NDB-UFES runs re-scored against the authors' labels; (b) patient exposure of the "
        "Study 2 NDB-UFES clean holdout and controlled arm; (c) arms trained on the Kaggle labels with the same "
        "patient-disjoint holdout (X3K) next to X3; (d) frozen DINOv2-S and Phikon logistic-regression probes under the "
        "same five groupings, with labels drawn per patient as a control; (e) the cross-dataset evaluations of Study 2 "
        "(E1 seed 0) tabulated, with NDB-UFES targets re-scored against the authors' labels."],
    ndb=dict(source="Mendeley Data 10.17632/bbmmm4wgr8.4 (CC BY 4.0): images.zip (237 source images), patch/images-patches.zip "
                    "(3,763 patches), patch/sabpatch_parsed_folders.csv and sabpatch_parsed_test.csv (patch labels), ndb-ufes.csv "
                    "(image -> lesion -> patient); checksums against checksums.md5 of the release",
             byte_identity="every Kaggle file is compared by MD5 with the authors' patch of the same file name",
             location=("grey images; normalised cross-correlation (OpenCV TM_CCOEFF_NORMED) at 1/8 resolution over all 237 "
                       "images, then at 1/2 resolution in the three best candidates; accepted when NCC >= 0.90; patches "
                       "not located are excluded from the patient-level dataset (never assigned a guessed patient)"),
             datasets=("ndb_pat: located patches, authors' labels (0 without dysplasia, 1 with dysplasia, 2 OSCC); "
                       "ndb_pat_k: the same patches with the Kaggle folder labels; groups g_struct = g_coarse = patient, "
                       "g_ctrl = patient joined with identical-file and near-duplicate groups; extra groupings seg "
                       "(Study 2 g_struct), block (Study 2 g_ctrl) and image (source image)"),
             outer="stratified group 5-fold on g_ctrl (patient-disjoint clean holdout)"),
    external=dict(orchid_bin="ORCHID test archive (A1 store): normal -> 0, OSCC of any grade -> 1, OSMF excluded; patient groups as A1",
                  rahman=("histopath originals of the Study 2 store (the 1,224 originals of Rahman et al. 2020; publisher-"
                          "augmented files never used); members of identical-file groups with two labels excluded"),
                  overlap="A1 found no SSIM-confirmed near duplicate between ORCHID and the histopathology collection",
                  metric="balanced accuracy at the arg-max of the two classes (no threshold or temperature fitted on external data)"),
    reporting=("Estimands, metrics, inner-validation selection and the two-stage cluster bootstrap as in the original "
               "plan. H13 and H16 are new confirmatory families; H1-H12 are not changed and NDB-UFES results of Study 2 "
               "are kept as reported, with the A2 analyses next to them."),
    exclusions="Same collapse rule as the original plan; failed runs are reported as missing.")

CHANGES = {
    "exps.X3": dict(desc="NDB-UFES, recovered patients, authors' labels: patch-level (L) vs grouping ladder vs patient (C)",
                    seeds=[0, 1, 2], seeds_extra=[0],
                    datasets=dict(ndb_pat=dict(arms=dict(L=L_, C=C_, Cseg=dict(C_, recov="seg"), Cblk=dict(C_, recov="block"),
                                                         Cimg=dict(C_, recov="image")),
                                               policy=None, steps=800, train=["L", "C", "Cseg", "Cblk", "Cimg"],
                                               extra=["Cseg", "Cblk", "Cimg"]))),
    "exps.X3RL": dict(desc="NDB-UFES random-label control, labels drawn per patient group", seeds=[0], outer=[0, 1, 2],
                      datasets=dict(ndb_pat=dict(arms=dict(L=L_, C=C_, Cblk=dict(C_, recov="block")), policy=None,
                                                 steps=800, train=["L", "C", "Cblk"]))),
    "exps.X3K": dict(desc="NDB-UFES, recovered patients, Kaggle folder labels: L vs C", seeds=[0],
                     datasets=dict(ndb_pat_k=dict(arms=dict(L=L_, C=C_), policy=None, steps=800, train=["L", "C"]))),
    "exps.X4A": dict(desc="external validation: ORCHID normal vs OSCC (L image-level, C patient) -> Rahman originals",
                     seeds=[0, 1, 2],
                     datasets=dict(orchid_bin=dict(arms=dict(L=L_, C=C_), policy=None, steps=400, train=["L", "C"],
                                                   xeval_to=["histopath"]))),
    "exps.X4B": dict(desc="external validation: Rahman (Kaggle copy; L augmented before the split) -> ORCHID normal vs OSCC",
                     seeds=[0, 1, 2], seeds_main=[0],
                     datasets=dict(histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=C_), policy="abdelaziz", target=5000,
                                                  steps=1300, train=["L", "C"], xeval_to=["orchid_bin"]))),
    "outer.ndb_pat": dict(kind="sgkf", n=5), "outer.ndb_pat_k": dict(kind="sgkf", n=5), "outer.orchid_bin": dict(kind="sgkf", n=5),
    "train_profile.ndb_pat": "std_he", "train_profile.ndb_pat_k": "std_he", "train_profile.orchid_bin": "std_he",
    "phases": list(P["phases"]) + [p for p in ("a2main", "a2ctrl", "a2ext", "a2opt") if p not in P["phases"]],
    "amendment_A2": A2_DOC,
}


def apply_changes(base, changes):
    out = copy.deepcopy(base)
    for path, val in changes.items():
        node = out
        keys = path.split(".")
        for k in keys[:-1]:
            node = node.setdefault(k, {})
        node[keys[-1]] = copy.deepcopy(val)
    return out


doc_json = json.dumps(dict(changes=CHANGES), sort_keys=True, indent=1)
doc_sha = hashlib.sha256(doc_json.encode()).hexdigest()
applied = {a["id"]: a for a in P.get("amendments", [])}
jobs_all = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
if "A2" in applied:
    st = "already applied" + (" (identical content)" if applied["A2"].get("sha256") == doc_sha else
                              " (this cell's text differs from the stored amendment; the STORED version stays in force)")
    Q.kv("amendment A2", st)
    Q.kv("written (UTC)", applied["A2"]["applied_utc"])
    Q.kv("SHA-256", applied["A2"]["sha256"])
else:
    P2 = apply_changes(P, CHANGES)
    bad = [j["id"] for j in jobs_all if j["spec"]["exp"] in P["exps"] and Q.job_fp(j["spec"], P2) != j["fp"]]
    if bad:
        raise SystemExit(f"amendment would change the fingerprint of {len(bad)} existing jobs (e.g. {bad[:3]}) - not applied")
    stamp = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    (Q.D["plan"] / "amendment_A2.json").write_text(doc_json)
    rec = dict(id="A2", applied_utc=stamp, sha256=doc_sha, file="amendment_A2.json", title=A2_DOC["title"],
               changed=sorted(CHANGES))
    P2.setdefault("amendments", []).append(rec)
    Q.jsave(Q.D["plan"] / "plan.json", P2)
    md = Q.D["plan"] / "ANALYSIS_PLAN.md"
    txt = md.read_text() if md.exists() else ""
    add = ["", f"## Amendment A2 — {A2_DOC['title']}", "",
           f"Written and hashed: {stamp}, before any A2 data step or experiment (local timestamp). "
           f"SHA-256 of amendment_A2.json: `{doc_sha}`.", "", A2_DOC["rationale"], "", "### Hypotheses"] + \
          [f"- {h}" for h in A2_DOC["hypotheses"]] + ["", "### NDB-UFES"] + [f"- {k}: {v}" for k, v in A2_DOC["ndb"].items()] + \
          ["", "### External validation"] + [f"- {k}: {v}" for k, v in A2_DOC["external"].items()] + \
          ["", "### Reporting", A2_DOC["reporting"], "", "### Exclusions", A2_DOC["exclusions"], ""]
    md.write_text(txt + "\n".join(add))
    Q.kv("amendment A2", "WRITTEN NOW (timestamped + hashed) — Save Version right after this block")
    Q.kv("written (UTC)", stamp)
    Q.kv("SHA-256", doc_sha)
    Q.kv("existing jobs", f"{len(jobs_all)} checked · fingerprints unchanged by the amendment")
P = Q.plan()
Q.kv("hypotheses added", "H13-H18 (H13 and H16 confirmatory)")
Q.kv("new phases", ", ".join(p for p in P["phases"] if p.startswith("a2")))
for name in ("q1_results.json", "a1_results.json"):
    src, snap = Q.D["eval"] / name, Q.D["eval"] / name.replace(".json", ".pre_A2.json")
    if src.exists() and not snap.exists():
        snap.write_text(src.read_text())
    Q.kv(f"snapshot of {name}", f"eval/{snap.name}" if snap.exists() else "not found (run BLOCK 25 / 33 first)")

Q.sub("ENVIRONMENT CHECKS FOR A2")
need_ds = {"ndb_ufes": "Study 2 NDB-UFES index (BLOCK 16)", "histopath": "Study 2 histopathology index (BLOCK 16)",
           "orchid": "ORCHID index (BLOCK 30)"}
for ds, what in need_ds.items():
    ok = (Q.D["index"] / f"{ds}.json").exists() and (Q.D["store"] / f"{ds}.npy").exists()
    Q.kv(what, "found" if ok else "MISSING")
try:
    import requests, cv2
    r = requests.get("https://data.mendeley.com/public-api/datasets/bbmmm4wgr8/files?folder_id=root&version=4", timeout=60)
    Q.kv("Mendeley Data (NDB-UFES release)", f"HTTP {r.status_code} · {len(r.json()) if r.ok else 0} files listed · OpenCV {cv2.__version__}")
except Exception as e:
    Q.kv("Mendeley Data (NDB-UFES release)", f"not reachable ({type(e).__name__}) — turn Internet on; BLOCK 35 retries")
Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB in /kaggle/working (BLOCK 35 downloads about 3.2 GB to /kaggle/tmp)")
print("\n[BLOCK 34 ✓]  amendment A2 in force · Save Version (Quick Save) now, then run BLOCK 35")


In [ ]:
# ============================================================================
# BLOCK 35 — A2 DATA: NDB-UFES AUTHORS' RELEASE · BYTE IDENTITY · LABELS · PATIENTS (patch -> source image) · A2 DATASETS
# needs: BLOCK 15 + BLOCK 34 (amendment A2) · Internet ON (Mendeley Data) · about 3.2 GB in /kaggle/tmp (not kept)
# re-running is safe: finished downloads and the patch-location table are reused
# ============================================================================
import os, sys, json, time, io, re, zipfile, hashlib, importlib, warnings
from pathlib import Path
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
_libsrc = (Q1_ROOT / "lib" / "q1lib.py").read_text() if (Q1_ROOT / "lib" / "q1lib.py").exists() else ""
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library, written by an old BLOCK 15 cell. Amendment A2 needs the A1 "
                     "library: replace the whole content of the BLOCK 15 and BLOCK 19 cells with the A1 versions (the first two "
                     "code cells of the A2 notebook), run 15 -> 19, then run this block again.")
HOOK = ("\n# --- amendment A2 hook (added by the A2 blocks; re-added automatically after BLOCK 15 rewrites this file)\n"
        "try:\n    import q1_a2ext as _a2ext\n    _a2ext.install(globals())\nexcept ImportError:\n    pass\n")
_lib = Q1_ROOT / "lib" / "q1lib.py"
if not (Q1_ROOT / "lib" / "q1_a2ext.py").exists():
    raise SystemExit("lib/q1_a2ext.py missing - run BLOCK 34 first")
if "amendment A2 hook" not in _lib.read_text():
    _lib.write_text(_lib.read_text() + HOOK)
import numpy as np
import pandas as pd
import q1lib as Q
importlib.reload(Q)
Q.keep_source("35")
import cv2
import requests

t_all = time.time()
P = Q.plan()
if "amendment_A2" not in P:
    raise SystemExit("amendment A2 not found in the plan - run BLOCK 34 first")
TMP = Path(os.environ.get("A2_TMP", "/kaggle/tmp/a2_ndb"))
TMP.mkdir(parents=True, exist_ok=True)
NCC_MIN = 0.90
Q.hdr("BLOCK 35 — NDB-UFES AUTHORS' RELEASE · BYTE IDENTITY · LABELS · PATIENTS · A2 DATASETS")

# ------------------------------------------------------------------ 1  download (Mendeley public API), checksums
Q.sub("1  DATASET AUTHORS' RELEASE  (Mendeley Data 10.17632/bbmmm4wgr8.4, CC BY 4.0)")
API = "https://data.mendeley.com/public-api/datasets/bbmmm4wgr8"
SES = requests.Session()


def jget(url):
    for k in range(6):
        try:
            r = SES.get(url, timeout=120)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if k == 5:
                raise
            time.sleep(5 * (k + 1))


def download(url, dst, size):
    if dst.exists() and dst.stat().st_size == size:
        return "cached"
    tmp = dst.with_suffix(dst.suffix + ".part")
    for k in range(8):
        try:
            have = tmp.stat().st_size if tmp.exists() else 0
            hdr = {"Range": f"bytes={have}-"} if have else {}
            with SES.get(url, headers=hdr, stream=True, timeout=300, allow_redirects=True) as r:
                if have and r.status_code != 206:
                    have = 0
                    tmp.unlink(missing_ok=True)
                r.raise_for_status()
                with open(tmp, "ab" if have else "wb") as f:
                    for b in r.iter_content(1 << 22):
                        f.write(b)
            if tmp.stat().st_size == size:
                os.replace(tmp, dst)
                return "downloaded"
        except Exception as e:
            print(f"    retry {k + 1} for {dst.name}: {type(e).__name__}", flush=True)
            time.sleep(5 * (k + 1))
    raise SystemExit(f"download of {dst.name} failed - re-run this block (partial file kept)")


def md5_file(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()


root = jget(f"{API}/files?folder_id=root&version=4")
folders = jget(f"{API}/folders/4")
pf = [f for f in folders if f["name"] == "patch"]
patch_files = jget(f"{API}/files?folder_id={pf[0]['id']}&version=4") if pf else []
want = {"checksums.md5": ("root", None), "ndb-ufes.csv": ("root", None), "images.zip": ("root", None),
        "images-patches.zip": ("patch", None), "sabpatch_parsed_folders.csv": ("patch", None), "sabpatch_parsed_test.csv": ("patch", None)}
LOC = {}
for where, lst in (("root", root), ("patch", patch_files)):
    for f in lst:
        if f["filename"] in want and want[f["filename"]][0] == where:
            cd = f["content_details"]
            dst = TMP / (("patch_" if where == "patch" else "") + f["filename"])
            st = download(cd["download_url"], dst, int(cd["size"]))
            LOC[f["filename"]] = dst
            Q.kv(f["filename"], f"{int(cd['size']) / 1e6:,.1f} MB · {st}")
missing = [k for k in want if k not in LOC]
if missing:
    raise SystemExit(f"not found in the release listing: {missing}")
ck = {}
for line in LOC["checksums.md5"].read_text().splitlines():
    if line.strip():
        h, name = line.split(None, 1)
        ck[name.strip().lstrip("./")] = h
inner = TMP / "images-patches.inner.zip"
if not inner.exists():
    with zipfile.ZipFile(LOC["images-patches.zip"]) as z:
        nm = [n for n in z.namelist() if n.endswith("images-patches.zip")][0]
        with z.open(nm) as src, open(inner, "wb") as dst:
            while True:
                b = src.read(1 << 22)
                if not b:
                    break
                dst.write(b)
checks = {"images.zip": md5_file(LOC["images.zip"]) == ck.get("images.zip"),
          "patch/images-patches.zip (inner archive)": md5_file(inner) == ck.get("patch/images-patches.zip"),
          "ndb-ufes.csv": md5_file(LOC["ndb-ufes.csv"]) == ck.get("ndb-ufes.csv"),
          "patch/sabpatch_parsed_folders.csv": md5_file(LOC["sabpatch_parsed_folders.csv"]) == ck.get("patch/sabpatch_parsed_folders.csv"),
          "patch/sabpatch_parsed_test.csv": md5_file(LOC["sabpatch_parsed_test.csv"]) == ck.get("patch/sabpatch_parsed_test.csv")}
for k, v in checks.items():
    Q.kv(f"MD5 {k}", "matches checksums.md5" if v else "DOES NOT MATCH")
if not all(checks.values()):
    raise SystemExit("checksum mismatch - delete /kaggle/tmp/a2_ndb and re-run")

# ------------------------------------------------------------------ 2  byte identity of the Kaggle copy, labels
Q.sub("2  KAGGLE COPY USED BY BOTH STUDIES vs THE AUTHORS' RELEASE  (MD5 by file name; labels)")
dd = Q.DS.get("ndb_ufes")
names = [Path(r["path"]).name for r in dd.R]
zin = zipfile.ZipFile(inner)
members = {Path(n).name: n for n in zin.namelist() if n.lower().endswith(".png")}
amd5 = {k: hashlib.md5(zin.read(v)).hexdigest() for k, v in members.items()}
same = np.array([amd5.get(n) == dd.md5[i] for i, n in enumerate(names)])
fo = pd.read_csv(LOC["sabpatch_parsed_folders.csv"]); te = pd.read_csv(LOC["sabpatch_parsed_test.csv"])
alab = {**dict(zip(fo.path, fo.lesion)), **dict(zip(te.path, te.lesion))}
blk = {**{p: "main" for p in fo.path}, **{p: "test" for p in te.path}}
AMAP = {"Without dysplasia": 0, "With dysplasia": 1, "OSCC": 2}
ya = np.array([AMAP.get(alab.get(n), -1) for n in names])
yk = dd.y.copy()
block = np.array([blk.get(n, "none") for n in names])
Q.kv("Kaggle files", f"{dd.n:,} · byte-identical to the authors' patch of the same name: {int(same.sum()):,}")
Q.kv("authors' patches", f"{len(members):,} in the inner archive · labelled {len(alab):,} · test block {len(te):,}")
KC = Q.META["ndb_ufes"]["classes"]
rows = []
for b in ("main", "test"):
    for a, ai in AMAP.items():
        m = (block == b) & (ya == ai)
        rows.append([b, a, int(m.sum())] + [int((m & (yk == k)).sum()) for k in range(3)])
Q.table(["block", "authors' label", "patches", f"Kaggle {KC[0]}", f"Kaggle {KC[1]}", f"Kaggle {KC[2]}"], rows, [7, 20, 9, 16, 19, 12])
disc = int((ya != yk).sum())
Q.kv("Kaggle class differs from the authors' label", f"{disc} patches ({100 * disc / dd.n:.1f}%) · test block filed as "
     f"{KC[0]}: {int(((block == 'test') & (yk == 0)).sum())} of {int((block == 'test').sum())}")
grp = defaultdict(list)
for k_, v in amd5.items():
    grp[v].append(k_)
ident = sorted(sorted(v) for v in grp.values() if len(v) > 1)
Q.kv("identical patches in the authors' release", f"{len(ident)} pairs · spanning its training and test blocks: "
     f"{sum(1 for g in ident if len({blk[x] for x in g}) > 1)}")

# ------------------------------------------------------------------ 3  locate every patch in its source image (NCC)
Q.sub("3  PATCH -> SOURCE IMAGE -> LESION -> PATIENT  (normalised cross-correlation; accepted at NCC >= 0.90)")
meta = pd.read_csv(LOC["ndb-ufes.csv"])
loc_f = Q.D["index"] / "ndb_patch_source.csv"
if loc_f.exists():
    S = pd.read_csv(loc_f)
    Q.kv("patch locations", f"reused {loc_f.name} ({len(S):,} patches)")
else:
    zim = zipfile.ZipFile(LOC["images.zip"])
    imem = {Path(n).name: n for n in zim.namelist() if n.lower().endswith(".png")}
    O8, O2 = {}, {}
    t0 = time.time()
    for nm in meta.path:
        g = cv2.imdecode(np.frombuffer(zim.read(imem[nm]), np.uint8), cv2.IMREAD_GRAYSCALE)
        O8[nm] = cv2.resize(g, (g.shape[1] // 8, g.shape[0] // 8), interpolation=cv2.INTER_AREA)
        O2[nm] = cv2.resize(g, (g.shape[1] // 2, g.shape[0] // 2), interpolation=cv2.INTER_AREA)
    Q.kv("source images decoded", f"{len(O8)} in {time.time() - t0:.0f}s")
    INAMES = list(meta.path)

    _W = {}

    def _init_worker():                       # every worker opens its own handle (a shared file offset would race)
        _W["z"] = zipfile.ZipFile(inner)

    def locate(pname):
        full = cv2.imdecode(np.frombuffer(_W["z"].read(members[pname]), np.uint8), cv2.IMREAD_GRAYSCALE)
        t8 = cv2.resize(full, (full.shape[1] // 8, full.shape[0] // 8), interpolation=cv2.INTER_AREA)
        t2 = cv2.resize(full, (full.shape[1] // 2, full.shape[0] // 2), interpolation=cv2.INTER_AREA)
        sc = sorted(((cv2.minMaxLoc(cv2.matchTemplate(O8[n], t8, cv2.TM_CCOEFF_NORMED))[1], n) for n in INAMES), reverse=True)
        best = (-1.0, None, (0, 0))
        for _, n in sc[:3]:
            _, mx, _, loc = cv2.minMaxLoc(cv2.matchTemplate(O2[n], t2, cv2.TM_CCOEFF_NORMED))
            if mx > best[0]:
                best = (mx, n, loc)
        return dict(patch=pname, src_image=best[1], ncc_fine=round(float(best[0]), 4), x=int(best[2][0] * 2), y=int(best[2][1] * 2),
                    ncc_coarse_best=round(float(sc[0][0]), 4), ncc_coarse_second=round(float(sc[1][0]), 4))

    from multiprocessing import get_context
    cv2.setNumThreads(1)
    todo = sorted(members)
    t0 = time.time()
    out = []
    with get_context("fork").Pool(max(1, (os.cpu_count() or 2)), initializer=_init_worker) as pool:
        for i, r in enumerate(pool.imap(locate, todo, chunksize=8)):
            out.append(r)
            if (i + 1) % 500 == 0:
                print(f"    {i + 1:,}/{len(todo):,} patches located ({time.time() - t0:.0f}s)", flush=True)
    S = pd.DataFrame(out)
    m_ = meta.set_index("path")
    for c in ("patient_id", "lesion_id", "diagnosis", "dysplasia_severity"):
        S[c] = S.src_image.map(m_[c])
    S["author_label"] = S.patch.map(alab)
    S["block"] = S.patch.map(blk)
    S.to_csv(loc_f, index=False)
    Q.kv("patch locations", f"{len(S):,} patches in {Q.hms(time.time() - t0)} -> index/{loc_f.name}")
okS = S[S.ncc_fine >= NCC_MIN]
Q.kv("located (NCC >= 0.90)", f"{len(okS):,} of {len(S):,} ({100 * len(okS) / len(S):.1f}%) · median NCC {okS.ncc_fine.median():.4f}")
Q.kv("recovered", f"{okS.src_image.nunique()} images · {okS.lesion_id.nunique()} lesions · {okS.patient_id.nunique()} of "
     f"{meta.patient_id.nunique()} patients")
pb = okS.groupby("patient_id").block.nunique()
ib = okS.groupby("src_image").block.nunique()
Q.kv("patients / images in both blocks of the authors' split", f"{int((pb == 2).sum())} of {len(pb)} / {int((ib == 2).sum())} of {len(ib)}")
best_ov = []
for img, d in okS.groupby("src_image"):
    xy = d[["x", "y"]].values
    for i in range(len(d)):
        ov = np.clip(512 - np.abs(xy[:, 0] - xy[i, 0]), 0, None) * np.clip(512 - np.abs(xy[:, 1] - xy[i, 1]), 0, None) / 512 ** 2
        ov[i] = 0
        best_ov.append(ov.max())
best_ov = np.array(best_ov)
Q.kv("located patches overlapping another crop of the same image", f"{100 * (best_ov > 0).mean():.1f}% (any) · "
     f"{100 * (best_ov >= 0.5).mean():.1f}% (>= 50% of the area)")
DXM = {"OSCC": "OSCC", "With dysplasia": "Leukoplakia with dysplasia", "Without dysplasia": "Leukoplakia without dysplasia"}
agree = okS.author_label.map(DXM) == okS.diagnosis
Q.kv("authors' patch label = diagnosis of the source image", f"{100 * agree.mean():.1f}% ({int(agree.sum()):,} of {len(okS):,}) "
     f"· the release describes the patches as relabelled")
EXPECT = dict(located=3141, median_ncc=0.9919, images=203, patients=61, lesions=66, discordant=509)
got = dict(located=len(okS), median_ncc=round(float(okS.ncc_fine.median()), 4), images=okS.src_image.nunique(),
           patients=okS.patient_id.nunique(), lesions=okS.lesion_id.nunique(), discordant=disc)
Q.kv("independent re-analysis (CPU, 3 Nov 2026)", "identical" if got == EXPECT else f"DIFFERS: {got} vs {EXPECT}")

# ------------------------------------------------------------------ 4  A2 datasets: ndb_pat, ndb_pat_k, orchid_bin
Q.sub("4  A2 DATASETS  (indexes over the existing stores; no image is re-decoded)")
loc = S.set_index("patch")
ncc = np.array([loc.ncc_fine.get(n, 0.0) for n in names])
located = (ncc >= NCC_MIN) & (ya >= 0)
pat_s = np.array([str(loc.patient_id.get(n)) if located[i] else None for i, n in enumerate(names)], object)
img_s = np.array([str(loc.src_image.get(n)) if located[i] else None for i, n in enumerate(names)], object)
pc = {p_: k for k, p_ in enumerate(sorted({p_ for p_ in pat_s if p_ is not None}))}
ic = {p_: k for k, p_ in enumerate(sorted({p_ for p_ in img_s if p_ is not None}))}
n = dd.n
pat_code = np.array([pc[p_] if p_ is not None else n + i for i, p_ in enumerate(pat_s)])
img_code = np.array([ic[p_] if p_ is not None else n + i for i, p_ in enumerate(img_s)])
g_ctrl_new = Q.union_labels(n, label_arrays=[dd.g_near, dd.g_exact, pat_code])
merged = len(set(g_ctrl_new[located & dd.ok].tolist()))


def recs_for(labels, names_cls):
    R = []
    for i, r in enumerate(dd.R):
        q = {k: v for k, v in r.items() if not k.startswith("g_")}
        q.update(label=int(labels[i]) if located[i] else 0, ok=bool(r.get("ok", True) and located[i]), orig=True,
                 parent=-1, parent_ok=False, dup_rep=bool(r.get("dup_rep", True)), md5=dd.md5[i],
                 g_exact=int(dd.g_exact[i]), g_near=int(dd.g_near[i]), g_struct=int(pat_code[i]), g_coarse=int(pat_code[i]),
                 g_ctrl=int(g_ctrl_new[i]), patient=pat_s[i] or "", src_image=img_s[i] or "", ncc=float(ncc[i]),
                 author_label=int(ya[i]), kaggle_label=int(yk[i]), release_block=str(block[i]),
                 s2_seg=int(dd.g_struct[i]), s2_block=int(dd.g_ctrl[i]))
        R.append(q)
    return R


summary = dict(n_files=n, located=int(located.sum()), patients=len(pc), images=len(ic), split_units=merged,
               source="Mendeley 10.17632/bbmmm4wgr8.4 matched to the Kaggle copy (BLOCK 35)")
recov = dict(groups=dict(seg=[int(v) for v in dd.g_struct], block=[int(v) for v in dd.g_ctrl], image=[int(v) for v in img_code]),
             k={}, note="seg = Study 2 g_struct (grouping of the Study 2 controlled arm); block = Study 2 g_ctrl (grouping of "
                        "the Study 2 holdout); image = recovered source image")
reg = Q.jload(Q.D["index"] / "_registry.json", {}) or {}
for ds, labels, cls, short, title in (
        ("ndb_pat", ya, ["without_dysplasia", "with_dysplasia", "OSCC"], "NDBp", "NDB-UFES, recovered patients, authors' labels"),
        ("ndb_pat_k", yk, list(KC), "NDBk", "NDB-UFES, recovered patients, Kaggle folder labels")):
    Q.jsave(Q.D["index"] / f"{ds}.json", dict(summary=dict(summary, classes=cls), records=recs_for(labels, cls)))
    Q.jsave(Q.D["index"] / f"{ds}.recov.json", recov)
    reg[ds] = dict(short=short, meta=dict(K=3, classes=cls, pos=2, modality="he", group="patient (recovered)", title=title))
od = Q.DS.get("orchid")
ob = []
for i, r in enumerate(od.R):
    q = dict(r)
    lab0 = int(od.y[i])
    q.update(label={0: 0, 2: 1}.get(lab0, 0), ok=bool(r.get("ok", True) and lab0 in (0, 2)), orig3=lab0)
    ob.append(q)
Q.jsave(Q.D["index"] / "orchid_bin.json", dict(summary=dict(od.summary, classes=["Normal", "OSCC"], note="OSMF excluded"), records=ob))
reg["orchid_bin"] = dict(short="ORCb", meta=dict(K=2, classes=["Normal", "OSCC"], pos=1, modality="he", group="patient",
                                                 title="ORCHID normal vs OSCC (OSMF excluded)"))
Q.jsave(Q.D["index"] / "_registry.json", reg)
Q.load_registry()
# images of the A2 datasets come from the existing stores through q1_a2ext.STORE_ALIAS. Earlier versions of this
# block created store/<ds>.npy as symbolic links; after a Kaggle restart such a link can come back as an empty file
# (the cause of "EOFError: No data left in file"). Any leftover file is checked (a full copy must be byte-identical to
# its source, which validates runs that read it) and then removed.
assert Q.__dict__.get("STORE_ALIAS"), "A2 extension without store aliases - re-run BLOCK 34 (new version) first"
STORE_CHECK = {}
for ds, srcds in Q.STORE_ALIAS.items():
    lk = Q.D["store"] / f"{ds}.npy"
    if not (lk.exists() or lk.is_symlink()):
        STORE_CHECK[ds] = "no leftover file"
        continue
    src = Q.D["store"] / f"{srcds}.npy"
    if lk.is_symlink():
        st = "symbolic link" + (" (valid)" if lk.exists() and lk.resolve() == src.resolve() else " (broken)")
    elif lk.stat().st_size == 0:
        st = "empty file (runs reading it failed with EOFError)"
    elif lk.stat().st_size == src.stat().st_size:
        st = "full copy, " + ("byte-identical to " + src.name if Q.sha256_file(lk) == Q.sha256_file(src) else "DIFFERS from " + src.name)
    else:
        st = f"partial file ({lk.stat().st_size:,} of {src.stat().st_size:,} bytes)"
    lk.unlink()
    STORE_CHECK[ds] = st + " -> removed"
for ds in Q.STORE_ALIAS:
    st_ = Q.open_store(ds)
    Q.kv(f"store for {ds}", f"{Q.STORE_ALIAS[ds]}.npy {tuple(st_.shape)} · leftover store/{ds}.npy: {STORE_CHECK[ds]}")
rows = []
for ds in ("ndb_pat", "ndb_pat_k", "orchid_bin"):
    d_ = Q.DS.get(ds)
    b = d_.base
    rows.append([ds, int(b.sum()), " / ".join(f"{c}:{int((d_.y[b] == k).sum())}" for k, c in enumerate(Q.META[ds]["classes"])),
                 len(set(d_.g_ctrl[b].tolist())), len({d_.R[i].get("patient") for i in np.nonzero(b)[0]})])
Q.table(["dataset", "images", "classes", "split units (g_ctrl)", "patients"], rows, [12, 8, 52, 20, 9])
hb = Q.DS.get("histopath")
hbase = hb.base
Q.kv("external set: histopathology originals", f"{int(hbase.sum()):,} images · " +
     " · ".join(f"{m_}: {int(sum(1 for i in np.nonzero(hbase)[0] if hb.R[i].get('mag') == m_))}" for m_ in ("100x", "400x")))

# ------------------------------------------------------------------ 5  Study 2 NDB-UFES splits checked against the recovered patients
Q.sub("5  STUDY 2 NDB-UFES SPLITS vs RECOVERED PATIENTS  (clean holdout, E1 arms L and C, seed 0)")
scheme = dict(P["outer"]["ndb_ufes"], rep=0)
fold, _ = Q.outer_folds(dd, scheme)


def overlap_share(ev_rows, tr_rows):
    by_img = defaultdict(list)
    for r in tr_rows:
        if located[r]:
            by_img[img_s[r]].append((int(loc.x[names[r]]), int(loc.y[names[r]])))
    a = g50 = m = 0
    for r in ev_rows:
        if not located[r]:
            continue
        m += 1
        x0, y0 = int(loc.x[names[r]]), int(loc.y[names[r]])
        best = max([max(0, 512 - abs(x0 - x1)) * max(0, 512 - abs(y0 - y1)) / 512 ** 2 for x1, y1 in by_img.get(img_s[r], [])], default=0)
        a += best > 0
        g50 += best >= 0.5
    return (a / m if m else np.nan, g50 / m if m else np.nan)


def share_same(ev_rows, tr_rows, arr):
    s = set(arr[r] for r in tr_rows if located[r])
    v = [arr[r] in s for r in ev_rows if located[r]]
    return float(np.mean(v)) if v else np.nan


S2 = []
JALL = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
faithful = {}
for o in range(Q.n_outer(dd, scheme)):
    hold = np.nonzero(dd.base & (fold == o))[0]
    devp = np.nonzero(dd.base & (fold >= 0) & (fold != o))[0]
    # the rebuilt split must equal the split of the finished Study 2 runs (same code; library versions may differ)
    ref = [j for j in JALL if j["spec"]["exp"] == "E1" and j["spec"]["ds"] == "ndb_ufes" and j["spec"]["outer"] == o
           and j["spec"].get("rep", 0) == 0 and Q.job_done(j)]
    if ref:
        _, zr = Q.load_run(ref[0]["id"])
        faithful[o] = bool(set(zr["hold_row"].tolist()) == set(hold.tolist()))
        if not faithful[o]:
            hold = np.unique(zr["hold_row"])
            devp = np.setdiff1d(np.nonzero(dd.base)[0], hold)
    rec = dict(outer=o, part="clean holdout vs development pool", n=int(located[hold].sum()), same_patient=share_same(hold, devp, pat_s),
               same_image=share_same(hold, devp, img_s), overlap=overlap_share(hold, devp))
    S2.append(rec)
    for arm in ("L", "C"):
        if faithful.get(o) is False:
            continue                               # arm splits can only be rebuilt when the holdout is reproduced exactly
        spec = dict(exp="E1", ds="ndb_ufes", arm=arm, outer=o, rep=0, seed=0, steps=800, backbone=P["primary_backbone"],
                    recipe="std", dose=None, xeval=False, keep_weights=False, train_profile=None)
        B = Q.build_job(spec, check=False)
        tr = B["train"]["row"]
        for part in ("own", "hold"):
            ev = B[part]["row"]
            S2.append(dict(outer=o, part=f"arm {arm}: {part} vs training", n=int(located[ev].sum()), same_patient=share_same(ev, tr, pat_s),
                           same_image=share_same(ev, tr, img_s), overlap=overlap_share(ev, tr)))
rows = []
for part in sorted({r["part"] for r in S2}):
    v = [r for r in S2 if r["part"] == part]
    rows.append([part, sum(r["n"] for r in v), f"{100 * np.nanmean([r['same_patient'] for r in v]):.1f}",
                 f"{100 * np.nanmean([r['same_image'] for r in v]):.1f}", f"{100 * np.nanmean([r['overlap'][0] for r in v]):.1f}",
                 f"{100 * np.nanmean([r['overlap'][1] for r in v]):.1f}"])
Q.table(["comparison (mean over 5 outer folds)", "located", "same patient %", "same image %", "overlapping crop %", ">= 50% overlap %"],
        rows, [36, 9, 15, 14, 19, 17])
Q.kv("rebuilt Study 2 holdouts identical to the finished runs", ("no finished Study 2 run found" if not faithful else
     f"{sum(faithful.values())} of {len(faithful)} outer folds" + ("" if all(faithful.values()) else
     " (holdout taken from the saved predictions; arm splits not rebuilt for the other folds)")))
gq = pd.Series(pat_s[located]).groupby(dd.g_ctrl[located]).nunique()
pg = pd.Series(dd.g_ctrl[located]).groupby(pat_s[located]).nunique()
Q.kv("Study 2 control groups (blocks) holding one patient", f"{100 * (gq == 1).mean():.1f}% of {len(gq)} · patients within one block: "
     f"{int((pg == 1).sum())} of {len(pg)} · blocks per patient: mean {pg.mean():.1f}, max {pg.max()}")
Q.jsave(Q.D["eval"] / "a2_ndb_audit.json", dict(
    downloads={k: str(v.name) for k, v in LOC.items()}, checksums=checks, byte_identical=int(same.sum()), n_files=int(dd.n),
    label_table=[dict(block=b, author=a, n=int(((block == b) & (ya == ai)).sum()),
                               kaggle=[int(((block == b) & (ya == ai) & (yk == k)).sum()) for k in range(3)])
                          for b in ("main", "test") for a, ai in AMAP.items()],
    discordant=disc, test_block=int((block == "test").sum()), test_block_healthy=int(((block == "test") & (yk == 0)).sum()),
    identical_in_release=ident, located=int(len(okS)), median_ncc=float(okS.ncc_fine.median()), images=int(okS.src_image.nunique()),
    lesions=int(okS.lesion_id.nunique()), patients=int(okS.patient_id.nunique()), patients_in_release=int(meta.patient_id.nunique()),
    patients_both_blocks=int((pb == 2).sum()), images_both_blocks=int((ib == 2).sum()),
    overlap_any=float((best_ov > 0).mean()), overlap_ge50=float((best_ov >= 0.5).mean()),
    label_agrees_with_image_diagnosis=float(agree.mean()), study2_checks=S2,
    blocks_one_patient=float((gq == 1).mean()), blocks_per_patient=float(pg.mean()), blocks_per_patient_max=int(pg.max()),
    ndb_pat_split_units=merged, matches_independent_reanalysis=bool(got == EXPECT), study2_splits_faithful=faithful,
    store_check=STORE_CHECK))
print(f"\n[BLOCK 35 ✓]  A2 datasets ready · eval/a2_ndb_audit.json · {Q.hms(time.time() - t_all)} · run BLOCK 36 next")


In [ ]:
# ============================================================================
# BLOCK 36 — A2 JOBS: every split built and checked for disjointness, appended to jobs/jobs.json (+ jobs/jobs_A2.json)
# needs: BLOCK 15 + 34 + 35 · does not train anything · safe to re-run (A2 jobs are rebuilt, finished runs are kept)
# ============================================================================
import os, sys, json, time, importlib, warnings
from pathlib import Path
from collections import Counter
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
_libsrc = (Q1_ROOT / "lib" / "q1lib.py").read_text() if (Q1_ROOT / "lib" / "q1lib.py").exists() else ""
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library, written by an old BLOCK 15 cell. Amendment A2 needs the A1 "
                     "library: replace the whole content of the BLOCK 15 and BLOCK 19 cells with the A1 versions (the first two "
                     "code cells of the A2 notebook), run 15 -> 19, then run this block again.")
HOOK = ("\n# --- amendment A2 hook (added by the A2 blocks; re-added automatically after BLOCK 15 rewrites this file)\n"
        "try:\n    import q1_a2ext as _a2ext\n    _a2ext.install(globals())\nexcept ImportError:\n    pass\n")
_lib = Q1_ROOT / "lib" / "q1lib.py"
if "amendment A2 hook" not in _lib.read_text():
    _lib.write_text(_lib.read_text() + HOOK)
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("36")

t_all = time.time()
P = Q.plan()
if "amendment_A2" not in P:
    raise SystemExit("amendment A2 not found - run BLOCK 34 first")
for ds in ("ndb_pat", "ndb_pat_k", "orchid_bin"):
    if not (Q.D["index"] / f"{ds}.json").exists():
        raise SystemExit(f"index/{ds}.json missing - run BLOCK 35 first")
BB0 = P["primary_backbone"]
Q.hdr("BLOCK 36 — A2 TRAINING JOBS")
speed = Q.merged_speed()
order = {p: i for i, p in enumerate(P["phases"])}


def phase_of(exp, arm, seed):
    if exp == "X3":
        return "a2main" if arm in ("L", "C") else "a2ctrl"
    if exp in ("X3RL", "X3K"):
        return "a2ctrl"
    if exp == "X4A":
        return "a2main"
    if exp == "X4B":
        return "a2ext" if seed == 0 else "a2opt"


new, bad = [], []
for exp in ("X3", "X3RL", "X3K", "X4A", "X4B"):
    ex = P["exps"][exp]
    for ds, dc in ex["datasets"].items():
        dd = Q.DS.get(ds)
        scheme = P["outer"][ds]
        outs = ex.get("outer") or list(range(Q.n_outer(dd, dict(scheme, rep=0))))
        for o in outs:
            for arm in dc["train"]:
                seeds = ex.get("seeds_extra", ex["seeds"]) if arm in dc.get("extra", []) else ex["seeds"]
                for seed in seeds:
                    s = dict(exp=exp, ds=ds, arm=arm, outer=int(o), rep=0, seed=int(seed), steps=dc["steps"], backbone=BB0,
                             recipe="std", dose=None, xeval=False, keep_weights=False, train_profile=None)
                    if dc.get("xeval_to"):
                        s["xeval_to"] = list(dc["xeval_to"])
                    jid = Q.job_id(s)
                    try:
                        B = Q.build_job(s, check=True)
                    except AssertionError as e:
                        bad.append((jid, "DISJOINTNESS FAILED: " + str(e)))
                        continue
                    except Exception as e:
                        bad.append((jid, f"{type(e).__name__}: {e}"))
                        continue
                    info = B["info"]
                    R = P["recipes"]["std"]
                    est = Q.estimate_minutes(s, info, R, speed)
                    steps, _ = Q.recipe_steps(s, R, info["sizes"]["train"])
                    phase = phase_of(exp, arm, seed)
                    # residual exposure of the own fold to the true grouping (patients) and to source images
                    tr, ow = B["train"]["row"], B["own"]["row"]
                    leak = dict(info["leak"])
                    if ds.startswith("ndb_pat"):
                        im = dd.extra["image"]
                        s_im = set(im[tr].tolist())
                        leak["same_image"] = float(np.mean([v in s_im for v in im[ow]])) if len(ow) else 0.0
                    prio = order[phase] * 1_000_000 + seed * 10_000 + o * 1000 + int(est * 10)
                    new.append(dict(id=jid, fp=Q.job_fp(s, P), phase=phase, priority=prio, est_min=round(est, 2), steps=steps,
                                    sizes=info["sizes"], leak=leak, n_xeval=info.get("n_xeval", 0), spec=s))
new.sort(key=lambda j: j["priority"])
J = Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}
ids = {j["id"] for j in new}
J["jobs"] = [j for j in J.get("jobs", []) if j["id"] not in ids] + new
Q.jsave(Q.D["jobs"] / "jobs.json", J)
Q.jsave(Q.D["jobs"] / "jobs_A2.json", dict(created=time.time(), jobs=new, rejected=bad))
rows = []
for key, v in sorted(Counter((j["phase"], j["spec"]["exp"], j["spec"]["ds"], j["spec"]["arm"]) for j in new).items()):
    J_ = [j for j in new if (j["phase"], j["spec"]["exp"], j["spec"]["ds"], j["spec"]["arm"]) == key]
    rows.append(list(key) + [v, f"{np.mean([j['sizes']['train'] for j in J_]):.0f}", f"{np.mean([j['sizes']['hold'] for j in J_]):.0f}",
                f"{100 * np.mean([j['leak'].get('struct_group', 0) for j in J_]):.1f}",
                f"{100 * np.mean([j['leak'].get('same_image', np.nan) for j in J_]):.1f}" if key[2].startswith("ndb_pat") else "—",
                f"{np.mean([j['n_xeval'] for j in J_]):.0f}" if J_[0]["n_xeval"] else "—", f"{sum(j['est_min'] for j in J_) / 60:.2f}"])
Q.table(["phase", "exp", "dataset", "arm", "jobs", "train n", "hold n", "own w/ same patient %", "own w/ same image %", "external n",
         "GPU-h"], rows, [7, 5, 11, 5, 5, 8, 7, 22, 20, 11, 6])
for jid, msg in bad[:12]:
    print(f"    ✗ {jid}: {msg[:140]}")
by_ph = Counter(j["phase"] for j in new)
for ph in ("a2main", "a2ctrl", "a2ext", "a2opt"):
    JJ = [j for j in new if j["phase"] == ph]
    Q.kv(f"phase {ph}", f"{len(JJ)} jobs · {sum(j['est_min'] for j in JJ) / 60:.1f} GPU-h (~{sum(j['est_min'] for j in JJ) / 120:.1f} h on 2 GPUs)")
# failures caused by the empty store file of the first A2 session (EOFError, fixed in BLOCKS 34/35) are not counted
# against the three-attempt limit; the failure records are kept under jobs/fail_archive_A2/ for the release
reset = []
arch = Q.D["jobs"] / "fail_archive_A2"
for j in new:
    fpth = Q.D["fail"] / f"{j['id']}.json"
    rec = Q.jload(fpth, {}) or {}
    if rec.get("count") and not Q.job_done(j) and all("No data left in file" in (m or "") for m in rec.get("msgs", [])):
        arch.mkdir(parents=True, exist_ok=True)
        fpth.replace(arch / fpth.name)
        reset.append(j["id"])
Q.kv("failure counters reset", f"{len(reset)} A2 jobs that failed only because of the empty store file (EOFError); "
     f"records moved to jobs/fail_archive_A2/" if reset else "none needed")
left_fail = [j["id"] for j in new if Q.fail_count(j["id"]) > 0 and not Q.job_done(j)]
if left_fail:
    print(f"    other failure records kept for {len(left_fail)} jobs, e.g. {left_fail[:3]}")
Q.kv("A2 jobs", f"{len(new)} built · {len(bad)} rejected · {sum(1 for j in new if Q.job_done(j))} already finished")
if bad:
    print("\n  Rejected jobs are listed above; paste this output back before training.")
print(f"\n[BLOCK 36 ✓]  {Q.hms(time.time() - t_all)} · run BLOCK 19 (if not yet run in this session), then BLOCK 37")


In [ ]:
# ============================================================================
# BLOCK 37 — TRAINING · AMENDMENT A2 (needs BLOCKS 15 -> 19 in this session and BLOCK 36 run once)
# re-run to resume; press Stop to pause safely. Order: a2main (confirmatory) -> a2ctrl -> a2ext. a2opt is optional.
# ============================================================================
import os
from pathlib import Path
_root = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
_lib = _root / "lib" / "q1lib.py"
_HOOK = ("\n# --- amendment A2 hook (added by the A2 blocks; re-added automatically after BLOCK 15 rewrites this file)\n"
         "try:\n    import q1_a2ext as _a2ext\n    _a2ext.install(globals())\nexcept ImportError:\n    pass\n")
_libsrc = _lib.read_text()
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library. Replace the BLOCK 15 and BLOCK 19 cells with the A1 versions "
                     "(first two code cells of the A2 notebook), run 15 -> 19, then this block.")
if 'endswith("RL")' not in (_root / "lib" / "q1worker.py").read_text():
    raise SystemExit("q1/lib/q1worker.py is the Q1 (2 October) worker: it would stop the random-label runs of A2 as 'collapsed'. "
                     "Replace the BLOCK 19 cell with the A1 version (second code cell of the A2 notebook), run it, then this block.")
if "amendment A2 hook" not in _libsrc:
    _lib.write_text(_lib.read_text() + _HOOK)        # the workers import q1lib from disk, so the hook must be there
RUN("a2main,a2ctrl,a2ext")
# optional sensitivity runs (X4B seeds 1-2, about 2.6 GPU-h): uncomment when the three phases above are complete
# RUN("a2opt")


In [ ]:
# ============================================================================
# BLOCK 38 — AMENDMENT A2 RESULTS: NDB-UFES PATIENT LEVEL (H13-H15) · KAGGLE vs AUTHORS' LABELS · EXTERNAL VALIDATION
#            (H16-H17) · STUDY 2 NDB-UFES RE-SCORED · CROSS-DATASET TABLE · FROZEN PROBES · FIGURES
# needs: BLOCKS 34-36 and the A2 training of BLOCK 37 (missing runs are reported as missing); CPU only, about 20-30 min
# ============================================================================
import os, sys, json, time, math, csv, importlib, warnings
from pathlib import Path
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
_libsrc = (Q1_ROOT / "lib" / "q1lib.py").read_text() if (Q1_ROOT / "lib" / "q1lib.py").exists() else ""
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library, written by an old BLOCK 15 cell. Amendment A2 needs the A1 "
                     "library: replace the whole content of the BLOCK 15 and BLOCK 19 cells with the A1 versions (the first two "
                     "code cells of the A2 notebook), run 15 -> 19, then run this block again.")
HOOK = ("\n# --- amendment A2 hook (added by the A2 blocks; re-added automatically after BLOCK 15 rewrites this file)\n"
        "try:\n    import q1_a2ext as _a2ext\n    _a2ext.install(globals())\nexcept ImportError:\n    pass\n")
_lib = Q1_ROOT / "lib" / "q1lib.py"
if "amendment A2 hook" not in _lib.read_text():
    _lib.write_text(_lib.read_text() + HOOK)
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("38")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from joblib import Parallel, delayed

NB = 2000 if Q.TINY else 10000
t_all = time.time()
P = Q.plan()
if "amendment_A2" not in P:
    raise SystemExit("amendment A2 not found - run BLOCK 34 first")
BB0 = P["primary_backbone"]
Q.hdr("BLOCK 38 — AMENDMENT A2 RESULTS")
JA = (Q.jload(Q.D["jobs"] / "jobs_A2.json", {}) or {}).get("jobs", [])
RUNS = {}
for j in JA:
    if Q.job_done(j):
        r, z = Q.load_run(j["id"])
        if r is not None:
            RUNS[j["id"]] = (j, r, {k: z[k] for k in z.files})
miss = Counter(j["phase"] for j in JA if j["id"] not in RUNS)
Q.kv("A2 runs", f"{len(RUNS)} of {len(JA)} finished" + ("" if not miss else " · missing: " + ", ".join(f"{k} {v}" for k, v in miss.items())))
REF = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
AUD = Q.jload(Q.D["eval"] / "a2_ndb_audit.json", {}) or {}
ALLJ = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
OUT = dict(meta=dict(n_runs=len(RUNS), n_jobs=len(JA), boot=NB, amendment=[a for a in P.get("amendments", []) if a["id"] == "A2"]))
TAB, FIG = Q.D["tables"], Q.D["figs"]
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
COL = {"L": "#eb6834", "C": "#2a78d6", "Cseg": "#e87ba4", "Cblk": "#eda100", "Cimg": "#1baf7a"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 300, "figure.dpi": 110})
made, MD = [], ["# Amendment A2 results", "", f"Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 38.", ""]


def write_csv(name, header, rows):
    with open(TAB / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


def sel(**kw):
    return [(j, r, z) for j, r, z in RUNS.values() if all(j["spec"].get(k) == v for k, v in kw.items())]


def correct(z, name):
    return (z[f"{name}_logit"].argmax(1) == z[f"{name}_y"]).astype(float)


def tables_for(runs, part):
    by = defaultdict(list)
    for j, r, z in runs:
        if f"{part}_logit" in z:
            by[(j["spec"]["outer"], j["spec"].get("rep", 0))].append((j, r, z))
    out = {}
    for fk, lst in by.items():
        K = int(max(int(z[f"{part}_logit"].shape[1]) for _, _, z in lst))
        out[fk] = Q.cluster_table(np.concatenate([z[f"{part}_y"] for _, _, z in lst]),
                                  np.concatenate([correct(z, part) for _, _, z in lst]),
                                  np.concatenate([z[f"{part}_key"] for _, _, z in lst]), K)
    return out


def boot_folds(tabs, B, seed):
    g = np.random.default_rng(seed)
    ks = sorted(tabs)
    return ks, np.stack([Q.boot_ba(tabs[k][0], tabs[k][1], B, g) for k in ks])


def point(tabs):
    return {k: float(Q.ba_counts(*tabs[k])) for k in tabs}


def fmt(v, ci_=None, d=2, signed=None):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    sg = signed if signed is not None else ci_ is not None
    s = f"{100 * v:+.{d}f}" if sg else f"{100 * v:.{d}f}"
    if ci_ is not None:
        s += f" [{100 * ci_[0]:+.{d}f}, {100 * ci_[1]:+.{d}f}]"
    return s


def pf(p):
    return "—" if p is None or not np.isfinite(p) else (f"{p:.1e}" if p < 1e-3 else f"{p:.4f}")


def two_part(R_, a, b, tag):
    """BA on parts a and b, difference a - b with the two-stage bootstrap (folds, then clusters within each fold)."""
    ta, tb = tables_for(R_, a), tables_for(R_, b)
    kk = sorted(set(ta) & set(tb))
    if not kk:
        return None
    ta, tb = {k: ta[k] for k in kk}, {k: tb[k] for k in kk}
    _, ma = boot_folds(ta, NB, Q.shash(tag, a) % 2 ** 32)
    _, mb = boot_folds(tb, NB, Q.shash(tag, b) % 2 ** 32)
    pa, pb = point(ta), point(tb)
    d = Q.hier([ma - mb], NB, Q.shash(tag, a, b) % 2 ** 32)[0]
    return dict(a=float(np.mean([pa[k] for k in kk])), b=float(np.mean([pb[k] for k in kk])),
                diff=float(np.mean([pa[k] - pb[k] for k in kk])), ci=Q.ci(d), p=Q.boot_p(d), folds=len(kk), _ma=ma, _mb=mb, _k=kk)


def arm_stats(R_, tag):
    x = two_part(R_, "own", "hold", tag)
    if x is None:
        return None
    seeds = sorted({j["spec"]["seed"] for j, _, _ in R_})
    per_seed = []
    for sd in seeds:
        y_ = two_part([t for t in R_ if t[0]["spec"]["seed"] == sd], "own", "hold", f"{tag}|s{sd}")
        if y_:
            per_seed.append(y_["diff"])
    return dict(n_runs=len(R_), folds=x["folds"], seeds=seeds, own=x["a"], hold=x["b"], infl=x["diff"], infl_ci=x["ci"], infl_p=x["p"],
                per_seed=per_seed, seed_sd=float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else None,
                leak_patient=float(np.mean([j["leak"].get("struct_group", 0.0) for j, _, _ in R_])),
                leak_image=float(np.nanmean([j["leak"].get("same_image", np.nan) for j, _, _ in R_])) if any("same_image" in j["leak"] for j, _, _ in R_) else None,
                _mo=x["_ma"], _mh=x["_mb"], _k=x["_k"])


def strip(d):
    return {k: v for k, v in d.items() if not k.startswith("_")} if isinstance(d, dict) else d


def contrast(A, B_, ka, kb, tag):
    """(A[ka] - A[kb]) - (B[ka] - B[kb]) or A[ka] - B[ka] over folds present in both; two-stage bootstrap."""
    kk = sorted(set(A["_k"]) & set(B_["_k"]))
    ia = [A["_k"].index(k) for k in kk]
    ib = [B_["_k"].index(k) for k in kk]
    if kb is None:
        v = A[ka][ia] - B_[ka][ib]
    else:
        v = (A[ka][ia] - A[kb][ia]) - (B_[ka][ib] - B_[kb][ib])
    d = Q.hier([v], NB, Q.shash(tag) % 2 ** 32)[0]
    return float(np.mean(d)), Q.ci(d), Q.boot_p(d)


# ============================================================================ 1  NDB-UFES audit (BLOCK 35)
if AUD:
    Q.sub("1  NDB-UFES: KAGGLE COPY vs AUTHORS' RELEASE, PATIENTS, STUDY 2 SPLITS  (from BLOCK 35)")
    Q.kv("byte-identical files", f"{AUD['byte_identical']:,} of {AUD['n_files']:,}")
    Q.kv("class differs from the authors' label", f"{AUD['discordant']} ({100 * AUD['discordant'] / AUD['n_files']:.1f}%) · "
         f"test block filed as Healthy {AUD['test_block_healthy']} of {AUD['test_block']}")
    Q.kv("located patches", f"{AUD['located']:,} · {AUD['images']} images · {AUD['lesions']} lesions · {AUD['patients']} of "
         f"{AUD['patients_in_release']} patients · median NCC {AUD['median_ncc']:.4f}")
    Q.kv("patients / images in both blocks of the authors' split", f"{AUD['patients_both_blocks']} / {AUD['images_both_blocks']}")
    Q.kv("overlapping crops", f"{100 * AUD['overlap_any']:.1f}% any · {100 * AUD['overlap_ge50']:.1f}% >= 50%")
    Q.kv("matches the independent re-analysis", AUD.get("matches_independent_reanalysis"))
    OUT["ndb_audit"] = AUD

# ============================================================================ 2  X3: NDB-UFES at patient level, grouping ladder (H13, H14)
X3 = {}
ARMS3 = ["L", "Cseg", "Cblk", "Cimg", "C"]
if sel(exp="X3"):
    Q.sub("2  NDB-UFES, RECOVERED PATIENTS, AUTHORS' LABELS: OWN FOLD vs PATIENT-DISJOINT HOLDOUT  (H13 confirmatory, H14 ladder)")
    rows, csvrows = [], []
    for arm in ARMS3:
        R_ = sel(exp="X3", ds="ndb_pat", arm=arm)
        if not R_:
            continue
        a = arm_stats(R_, f"X3|{arm}")
        X3[arm] = a
        rows.append([arm, f"{a['n_runs']}/{a['folds']}", fmt(a["own"]), fmt(a["hold"]), fmt(a["infl"], a["infl_ci"]), pf(a["infl_p"]),
                     "—" if a["seed_sd"] is None else f"{100 * a['seed_sd']:.2f}", f"{100 * a['leak_patient']:.1f}",
                     "—" if a["leak_image"] is None else f"{100 * a['leak_image']:.1f}"])
        csvrows.append(["ndb_pat", arm, a["n_runs"], a["folds"], a["own"], a["hold"], a["infl"], *a["infl_ci"], a["infl_p"], a["seed_sd"],
                        a["leak_patient"], a["leak_image"]])
    Q.table(["arm", "runs/folds", "own BA", "hold BA", "inflation [95% CI]", "p", "seed SD", "own w/ same patient %",
             "own w/ same image %"], rows, [6, 11, 8, 8, 27, 9, 8, 22, 20])
    print("  L = patch-level split (published handling) · Cseg = Study 2 controlled-arm grouping · Cblk = Study 2 holdout grouping\n"
          "  · Cimg = source image · C = patient. Holdout: patient-disjoint outer folds. Cseg/Cblk/Cimg: seed 0 only.")
    write_csv("A2_X3_ladder", ["dataset", "arm", "runs", "folds", "own_ba", "hold_ba", "inflation", "ci_lo", "ci_hi", "p", "seed_sd",
                               "own_same_patient", "own_same_image"], csvrows)
    if "L" in X3:
        h13 = X3["L"]["infl_p"] < 0.05 and X3["L"]["infl"] > 0
        Q.kv("H13 (confirmatory)", ("SUPPORTED" if h13 else "NOT supported") + f" · inflation {fmt(X3['L']['infl'], X3['L']['infl_ci'])} "
             f"points · p {pf(X3['L']['infl_p'])} · per seed " + ", ".join(f"{100 * v:+.2f}" for v in X3["L"]["per_seed"]))
        OUT["H13"] = dict(supported=bool(h13), infl=X3["L"]["infl"], ci=X3["L"]["infl_ci"], p=X3["L"]["infl_p"], per_seed=X3["L"]["per_seed"])
    if "L" in X3 and "C" in X3:
        e, c_, p_ = contrast(X3["L"], X3["C"], "_mo", "_mh", "X3|did")
        g, gc, gp = contrast(X3["L"], X3["C"], "_mh", None, "X3|G")
        Q.kv("D = inflation(L) - inflation(C)", f"{fmt(e, c_)} points · p {pf(p_)}")
        Q.kv("G = holdout BA(L) - holdout BA(C)", f"{fmt(g, gc)} points · p {pf(gp)}")
        OUT["X3_D"] = dict(est=e, ci=c_, p=p_)
        OUT["X3_G"] = dict(est=g, ci=gc, p=gp)
    OUT["X3"] = {k: strip(v) for k, v in X3.items()}
    MD += ["## X3 — NDB-UFES at patient level (authors' labels)", "", "| arm | runs/folds | own BA | holdout BA | inflation [95% CI] | p | "
           "own with same patient in training % |", "|---|---|---|---|---|---|---|"] + \
          [f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]} | {r[5]} | {r[7]} |" for r in rows] + [""]

# ============================================================================ 3  X3RL: random labels per patient (H15)
RLX = {}
if sel(exp="X3RL"):
    Q.sub("3  RANDOM LABELS DRAWN PER PATIENT GROUP  (H15: L and Cblk above chance on their own fold, C at chance)")
    rows = []
    for arm in ("L", "Cblk", "C"):
        R_ = sel(exp="X3RL", ds="ndb_pat", arm=arm)
        if not R_:
            continue
        to, th = tables_for(R_, "own"), tables_for(R_, "hold")
        _, mo = boot_folds(to, NB, 11)
        _, mh = boot_folds(th, NB, 12)
        own_b = Q.hier([mo], NB, 13)[0]
        d = dict(own=float(np.mean(list(point(to).values()))), hold=float(np.mean(list(point(th).values()))), own_ci=Q.ci(own_b),
                 p_vs_chance=Q.boot_p(own_b, 1 / 3), n=len(R_))
        RLX[arm] = d
        rows.append([arm, d["n"], f"{fmt(d['own'])} [{fmt(d['own_ci'][0])}, {fmt(d['own_ci'][1])}]", fmt(d["hold"]), "33.3", pf(d["p_vs_chance"])])
    Q.table(["arm", "runs", "own BA [95% CI]", "hold BA", "chance", "p own vs chance"], rows, [6, 6, 26, 9, 8, 16])
    OUT["X3RL"] = RLX

# ============================================================================ 4  X3K: Kaggle labels on the same patient-disjoint design
X3K = {}
if sel(exp="X3K"):
    Q.sub("4  KAGGLE FOLDER LABELS vs AUTHORS' LABELS  (X3K vs X3, seed 0, same located patches, patient-disjoint holdout)")
    rows = []
    for arm in ("L", "C"):
        R_ = sel(exp="X3K", ds="ndb_pat_k", arm=arm)
        R0 = sel(exp="X3", ds="ndb_pat", arm=arm, seed=0)
        a, b = (arm_stats(R_, f"X3K|{arm}") if R_ else None), (arm_stats(R0, f"X3s0|{arm}") if R0 else None)
        X3K[arm] = dict(kaggle=strip(a), authors_seed0=strip(b))
        for lab, v in (("Kaggle labels", a), ("authors' labels (X3, seed 0)", b)):
            if v:
                rows.append([arm, lab, fmt(v["own"]), fmt(v["hold"]), fmt(v["infl"], v["infl_ci"])])
    Q.table(["arm", "labels", "own BA", "hold BA", "inflation [95% CI]"], rows, [6, 30, 9, 9, 28])
    OUT["X3K"] = X3K

# ============================================================================ 5  external validation (H16, H17)
Q.sub("5  EXTERNAL VALIDATION  (H16: D_ext > 0, Holm over X4A and X4B · H17: G_ext = ext_L - ext_C)")
EXT, h16p = {}, {}
HB = Q.DS.get("histopath")
MAG = np.array([str(r.get("mag", "")) for r in HB.R])


def filt(R_, part, keep_fn):
    out = []
    for j, r, z in R_:
        z2 = dict(z)
        if f"{part}_row" in z:
            m = keep_fn(z[f"{part}_row"])
            for k in ("row", "copy", "y", "key", "logit"):
                z2[f"{part}_{k}"] = z[f"{part}_{k}"][m]
        out.append((j, r, z2))
    return out


rows = []
for exp, ds, tgt, seeds_main in (("X4A", "orchid_bin", "histopath", [0, 1, 2]), ("X4B", "histopath", "orchid_bin", [0])):
    for scope, sd in (("main", seeds_main), ("all seeds", [0, 1, 2])):
        if scope == "all seeds" and (exp == "X4A" or not any(t[0]["spec"]["seed"] > 0 for t in sel(exp=exp))):
            continue
        S_ = {}
        for arm in ("L", "C"):
            R_ = [t for t in sel(exp=exp, ds=ds, arm=arm) if t[0]["spec"]["seed"] in sd]
            if not R_:
                continue
            part = f"x_{tgt}"
            oh, ox = two_part(R_, "own", "hold", f"{exp}|{arm}|{scope}|oh"), two_part(R_, "own", part, f"{exp}|{arm}|{scope}|ox")
            if oh is None or ox is None:
                continue
            S_[arm] = dict(n_runs=len(R_), seeds=sorted({t[0]["spec"]["seed"] for t in R_}), own=oh["a"], hold=oh["b"], ext=ox["b"],
                           own_minus_hold=oh["diff"], own_minus_hold_ci=oh["ci"], own_minus_ext=ox["diff"], own_minus_ext_ci=ox["ci"],
                           _mo=ox["_ma"], _mx=ox["_mb"], _k=ox["_k"])
            if exp == "X4A":
                for m_ in ("100x", "400x"):
                    Rm = filt(R_, part, lambda rows_: MAG[rows_] == m_)
                    xm = two_part(Rm, "own", part, f"{exp}|{arm}|{m_}")
                    if xm:
                        S_[arm][f"ext_{m_}"] = xm["b"]
                        S_[arm][f"_mx_{m_}"] = xm["_mb"]
        if "L" in S_ and "C" in S_:
            A, B_ = S_["L"], S_["C"]
            d, dci, dp = contrast(A, B_, "_mo", "_mx", f"{exp}|{scope}|Dext")
            g, gci, gp = contrast(A, B_, "_mx", None, f"{exp}|{scope}|Gext")
            res = dict(arms={k: strip(v) for k, v in S_.items()}, D_ext=dict(est=d, ci=dci, p=dp), G_ext=dict(est=g, ci=gci, p=gp))
            if exp == "X4A":
                for m_ in ("100x", "400x"):
                    if f"_mx_{m_}" in A and f"_mx_{m_}" in B_:
                        gm, gmci, gmp = contrast(dict(A, _mxm=A[f"_mx_{m_}"]), dict(B_, _mxm=B_[f"_mx_{m_}"]), "_mxm", None, f"{exp}|G|{m_}")
                        res[f"G_ext_{m_}"] = dict(est=gm, ci=gmci, p=gmp)
            EXT[f"{exp}|{scope}"] = res
            if scope == "main":
                h16p[exp] = dp
            for arm, v in S_.items():
                rows.append([exp, scope, arm, f"{v['n_runs']}", fmt(v["own"]), fmt(v["hold"]), fmt(v["ext"]),
                             fmt(v.get("ext_100x")) + "/" + fmt(v.get("ext_400x")) if exp == "X4A" else "—",
                             fmt(v["own_minus_ext"], v["own_minus_ext_ci"])])
            rows.append([exp, scope, "L−C", "", "", "", fmt(g, gci), "", f"D_ext {fmt(d, dci)} · p {pf(dp)}"])
Q.table(["exp", "seeds", "arm", "runs", "own BA", "internal hold BA", "external BA", "external 100x/400x", "own − external [95% CI]"],
        rows, [5, 10, 5, 5, 8, 17, 30, 18, 40])
print("  X4A: ORCHID normal vs OSCC -> Rahman et al. originals · X4B: Rahman et al. (Kaggle copy) -> ORCHID normal vs OSCC.\n"
      "  L−C row: external BA difference G_ext [95% CI] and D_ext = (own_L − ext_L) − (own_C − ext_C).")
if h16p:
    ks = sorted(h16p)
    adj = Q.holm([h16p[k] for k in ks])
    for k, a_ in zip(ks, adj):
        e = EXT[f"{k}|main"]["D_ext"]
        Q.kv(f"H16 {k} (Holm over {len(ks)})", ("SUPPORTED" if (a_ < 0.05 and e["est"] > 0) else "NOT supported") +
             f" · D_ext {fmt(e['est'], e['ci'])} · p {pf(e['p'])} · Holm p {pf(a_)}")
        EXT[f"{k}|main"]["H16_holm_p"] = a_
OUT["external"] = EXT

# ============================================================================ 6  Study 2 NDB-UFES runs re-scored against the authors' labels
Q.sub("6  STUDY 2 NDB-UFES (E1, ConvNeXt-T, seeds 0-2) RE-SCORED AGAINST THE DATASET AUTHORS' LABELS")
NP = Q.DS.get("ndb_pat")
ALAB = np.array([int(r.get("author_label", -1)) for r in NP.R])
LOCATED = NP.ok.copy()
RS = {}
rows = []
for arm in ("L", "C"):
    jl = [j for j in ALLJ if j["spec"]["exp"] == "E1" and j["spec"]["ds"] == "ndb_ufes" and j["spec"]["arm"] == arm
          and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std" and Q.job_done(j)]
    R_k, R_a = [], []
    for j in jl:
        r, z = Q.load_run(j["id"])
        z = {k: z[k] for k in z.files}
        R_k.append((j, r, z))
        z2 = dict(z)
        for part in ("ival", "own", "hold"):
            z2[f"{part}_y"] = ALAB[z[f"{part}_row"]]
        R_a.append((j, r, z2))
    if not R_k:
        continue
    a, b = arm_stats(R_k, f"S2k|{arm}"), arm_stats(R_a, f"S2a|{arm}")
    RS[arm] = dict(kaggle_labels=strip(a), authors_labels=strip(b), runs=len(R_k))
    rows.append([arm, len(R_k), "Kaggle (as reported)", fmt(a["own"]), fmt(a["hold"]), fmt(a["infl"], a["infl_ci"])])
    rows.append([arm, len(R_a), "authors' labels", fmt(b["own"]), fmt(b["hold"]), fmt(b["infl"], b["infl_ci"])])
Q.table(["arm", "runs", "scored against", "own BA", "hold BA", "inflation [95% CI]"], rows, [6, 6, 22, 9, 9, 28])
print("  The models were trained on the Kaggle labels; re-scoring shows how the reported numbers change when the same predictions\n"
      "  are compared with the authors' labels (the holdout is the Study 2 block-level holdout, not patient-disjoint).")
OUT["study2_ndb_rescored"] = RS

# ============================================================================ 7  cross-dataset evaluations of Study 2 (E1 seed 0)
Q.sub("7  CROSS-DATASET EVALUATION OF STUDY 2 (E1, seed 0)  — NDB-UFES targets also re-scored against the authors' labels")
CD = dict(reported=REF.get("cross_dataset", {}))
rows = []
for k, v in sorted(CD["reported"].items()):
    src, arm, tgt = k.split("|")
    extra = "—"
    if tgt == "ndb_ufes":
        vals = []
        for j in ALLJ:
            s = j["spec"]
            if s["exp"] == "E1" and s["ds"] == src and s["arm"] == arm and s["seed"] == 0 and s["backbone"] == BB0 and Q.job_done(j):
                _, z = Q.load_run(j["id"])
                if "x_ndb_ufes_row" not in z.files:
                    continue
                rr = z["x_ndb_ufes_row"]
                al = ALAB[rr]
                m = (al == 0) | (al == 2)
                yb = (al[m] == 2).astype(int)
                pr = z["x_ndb_ufes_logit"][m].argmax(1)
                vals.append(np.mean([np.mean(pr[yb == c] == c) for c in (0, 1) if (yb == c).any()]))
        if vals:
            extra = f"{100 * np.mean(vals):.2f}"
            CD[f"{k}|authors"] = float(np.mean(vals))
    rows.append([src, arm, tgt, f"{100 * v['bal_acc']:.2f}", f"{v['auroc']:.3f}", v.get("n"), extra])
Q.table(["source", "arm", "target", "BA % (as reported)", "AUROC", "runs", "BA % vs authors' labels (OSCC vs without dysplasia)"], rows,
        [11, 4, 11, 19, 7, 5, 50])
OUT["cross_dataset"] = CD

# ============================================================================ 8  frozen probes at patient level (DINOv2-S, Phikon)
Q.sub("8  FROZEN PROBES ON NDB-UFES  (logistic regression, C = 1, class-balanced; five groupings; seeds 0-2)")
NK = Q.DS.get("ndb_pat_k")
base = np.nonzero(NP.base)[0]
GROUPS = dict(patch=None, seg=NP.extra["seg"][base], block=NP.extra["block"][base], image=NP.extra["image"][base], patient=NP.g_ctrl[base])
YS = dict(authors=NP.y[base], kaggle=NK.y[base])
PAT = NP.g_ctrl[base]
FROZ = {}


def ba(y, p):
    return float(np.mean([np.mean(p[y == k] == k) for k in np.unique(y)]))


def cv_pred(X, y, grouping, seed):
    idx = np.arange(len(y))
    spl = StratifiedKFold(5, shuffle=True, random_state=seed).split(idx, y) if grouping is None else \
        StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(idx, y, grouping)
    pred = np.full(len(y), -1)
    for tr, te in spl:
        clf = LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced").fit(X[tr], y[tr])
        pred[te] = clf.predict(X[te])
    return pred


def boot_diff(y, pa, pb, groups, B=2000, seed=0):
    g = np.random.default_rng(seed)
    ug = np.unique(groups)
    ix = {u: np.nonzero(groups == u)[0] for u in ug}
    d = []
    for _ in range(B):
        s = np.concatenate([ix[u] for u in g.choice(ug, len(ug))])
        d.append(ba(y[s], pa[s]) - ba(y[s], pb[s]))
    return Q.ci(np.array(d))


for key in ("dinov2_s", "phikon"):
    fp_ = Q.D["store"] / f"ndb_ufes_emb_{key}.npy"
    if not fp_.exists():
        Q.kv(key, "embedding file missing (BLOCK 16 / 31) - skipped")
        continue
    E = np.load(fp_).astype(np.float32)[base]
    if not np.isfinite(E).all() or (np.abs(E).sum(1) == 0).any():
        Q.kv(key, "embeddings missing for some located patches - skipped")
        continue
    E /= np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-8)
    jobs = [(lab, gname, seed) for lab in YS for gname in GROUPS for seed in (0, 1, 2)]
    preds = Parallel(n_jobs=-1)(delayed(cv_pred)(E, YS[lab], GROUPS[g_], sd) for lab, g_, sd in jobs)
    res = {}
    for (lab, g_, sd), pr in zip(jobs, preds):
        res.setdefault((lab, g_), []).append((sd, pr))
    F_ = {}
    rows = []
    for lab in YS:
        ref0 = [pr for sd, pr in res[(lab, "patient")] if sd == 0][0]
        for g_ in GROUPS:
            v = [ba(YS[lab], pr) for _, pr in res[(lab, g_)]]
            p0 = [pr for sd, pr in res[(lab, g_)] if sd == 0][0]
            ci_ = boot_diff(YS[lab], p0, ref0, PAT) if g_ != "patient" else None
            F_[f"{lab}|{g_}"] = dict(mean=float(np.mean(v)), sd=float(np.std(v, ddof=1)), diff_s0=ba(YS[lab], p0) - ba(YS[lab], ref0),
                                     diff_ci_s0=ci_)
            rows.append([key, lab, g_, f"{100 * np.mean(v):.2f} ± {100 * np.std(v, ddof=1):.2f}",
                         "reference" if g_ == "patient" else fmt(F_[f"{lab}|{g_}"]["diff_s0"], ci_)])
    rl = []
    rg = np.random.default_rng(7)
    for draw in range(10):
        up = np.unique(PAT)
        perm = rg.permutation(len(up))
        lab_u = {u: int(perm[i] % 3) for i, u in enumerate(up)}
        yr = np.array([lab_u[u] for u in PAT])
        for g_ in ("patch", "block", "patient"):
            rl.append((g_, ba(yr, cv_pred(E, yr, GROUPS[g_], draw))))
    for g_ in ("patch", "block", "patient"):
        v = [b for gg, b in rl if gg == g_]
        F_[f"random_per_patient|{g_}"] = dict(mean=float(np.mean(v)), sd=float(np.std(v, ddof=1)), min=float(np.min(v)), max=float(np.max(v)))
        rows.append([key, "random per patient", g_, f"{100 * np.mean(v):.2f} ± {100 * np.std(v, ddof=1):.2f}", "chance 33.33"])
    FROZ[key] = F_
    Q.table(["features", "labels", "grouping", "BA % (mean ± SD, 3 seeds)", "difference from patient-disjoint, seed 0 [95% CI]"], rows,
            [9, 19, 9, 26, 46])
OUT["frozen_ndb"] = FROZ

# ============================================================================ 9  figures
Q.sub("9  FIGURES")
if X3:
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    for arm in ARMS3:
        if arm not in X3:
            continue
        v = X3[arm]
        ax.errorbar(100 * v["leak_patient"], 100 * v["infl"], yerr=[[100 * (v["infl"] - v["infl_ci"][0])], [100 * (v["infl_ci"][1] - v["infl"])]],
                    fmt="o", ms=7, color=COL.get(arm, INK), ecolor=COL.get(arm, INK), elinewidth=1.4, capsize=3, mec="white", mew=1.2)
        off = {"L": (8, 6), "Cseg": (8, -12), "Cblk": (-34, 6), "Cimg": (8, 4), "C": (8, 4)}.get(arm, (8, 4))
        ax.annotate(arm, (100 * v["leak_patient"], 100 * v["infl"]), textcoords="offset points", xytext=off, fontsize=8, color=INK)
    ax.axhline(0, color=INK2, lw=0.8)
    ax.set_xlabel("Own-fold patches with a training patch of the same patient (%)")
    ax.set_ylabel("Inflation: own fold − patient-disjoint holdout (BA points)")
    ax.set_xlim(-5, 105)
    fig.savefig(FIG / "Fig_A2_ladder.png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    made.append(("Fig_A2_ladder.png", "inflation against residual patient exposure, NDB-UFES grouping ladder (X3)"))
if EXT:
    keys = [k for k in ("X4A|main", "X4B|main") if k in EXT]
    fig, axs = plt.subplots(1, len(keys), figsize=(3.3 * len(keys), 3.2), squeeze=False)
    for ax, k in zip(axs[0], keys):
        for i, arm in enumerate(("L", "C")):
            v = EXT[k]["arms"].get(arm)
            if not v:
                continue
            ys = [100 * v["own"], 100 * v["hold"], 100 * v["ext"]]
            ax.plot([0, 1, 2], ys, "-o", color=COL[arm], lw=2, ms=6, mec="white", mew=1.2, label=f"{arm}")
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(["own fold", "internal\nholdout", "external"])
        ax.set_title({"X4A|main": "ORCHID → Rahman", "X4B|main": "Rahman (Kaggle copy) → ORCHID"}[k], fontsize=9)
        ax.set_ylabel("Balanced accuracy (%)")
        ax.legend(loc="lower left")
    fig.savefig(FIG / "Fig_A2_external.png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    made.append(("Fig_A2_external.png", "own-fold, internal-holdout and external balanced accuracy of leaky (L) and controlled (C) arms"))
for nm, note in made:
    Q.kv(nm, note)

# ============================================================================ 10  save
Q.jsave(Q.D["eval"] / "a2_results.json", OUT)
(TAB / "RESULTS_A2.md").write_text("\n".join(MD))
pre = Q.D["eval"] / "q1_results.pre_A2.json"
if pre.exists() and (Q.D["eval"] / "q1_results.json").exists():
    same = Q.jload(pre).get("E1") == (Q.jload(Q.D["eval"] / "q1_results.json") or {}).get("E1")
    Q.kv("Study 2 E1 results unchanged by A2", "yes" if same else "NO - check")
print(f"\n[BLOCK 38 ✓]  eval/a2_results.json · tables/A2_*.csv · figs/Fig_A2_*.png · {Q.hms(time.time() - t_all)} · run BLOCK 40 next")


In [ ]:
# ============================================================================
# BLOCK 39 — OPTIONAL: STUDY 1 NDB-UFES ARMS RE-SCORED AGAINST THE AUTHORS' LABELS (no training; CPU; 1 min)
# runs only if the Study 1 results archive or folder is reachable from this notebook, for example
#   /kaggle/working/OralNet_results_20260829_2057.zip   or   /kaggle/working/evaluation/*ndb_ufes*__oof.npz
# needs: BLOCK 35 (index/ndb_pat.json). If nothing is found, the block says so and changes nothing.
# ============================================================================
import os, sys, json, glob, zipfile, io, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
_libsrc = (Q1_ROOT / "lib" / "q1lib.py").read_text() if (Q1_ROOT / "lib" / "q1lib.py").exists() else ""
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library, written by an old BLOCK 15 cell. Amendment A2 needs the A1 "
                     "library: replace the whole content of the BLOCK 15 and BLOCK 19 cells with the A1 versions (the first two "
                     "code cells of the A2 notebook), run 15 -> 19, then run this block again.")
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("39")
Q.hdr("BLOCK 39 — STUDY 1 NDB-UFES ARMS vs THE AUTHORS' LABELS (optional)")
NP = Q.jload(Q.D["index"] / "ndb_pat.json")
if NP is None:
    raise SystemExit("index/ndb_pat.json missing - run BLOCK 35 first")
lab = {Path(r["path"]).name: int(r["author_label"]) for r in NP["records"]}
pat = {Path(r["path"]).name: r.get("patient", "") for r in NP["records"]}
cands = sorted(set(glob.glob("/kaggle/working/**/*ndb_ufes*oof*.npz", recursive=True) +
                   glob.glob("/kaggle/input/**/*ndb_ufes*oof*.npz", recursive=True)))
zips = sorted(set(glob.glob("/kaggle/working/OralNet_results_*.zip") + glob.glob("/kaggle/input/**/OralNet_results_*.zip", recursive=True)))
arrays = {}
for p in cands:
    arrays[p] = dict(np.load(p, allow_pickle=True))
for zp in zips:
    with zipfile.ZipFile(zp) as z:
        for n in z.namelist():
            if "ndb_ufes" in n and n.endswith(".npz"):
                arrays[f"{zp}:{n}"] = dict(np.load(io.BytesIO(z.read(n)), allow_pickle=True))
if not arrays:
    Q.kv("Study 1 predictions", "not found in /kaggle/working or /kaggle/input - nothing to do (this block is optional)")
else:
    OUT = {}
    for src, A in arrays.items():
        Q.sub(src.split("/")[-1])
        for k, v in A.items():
            print(f"    key {k:<16} shape {getattr(v, 'shape', None)} dtype {getattr(v, 'dtype', None)}")
        fk = next((k for k in ("paths", "files", "path", "file", "names", "fnames") if k in A), None)
        pk = next((k for k in ("probs", "prob", "p", "oof", "pred_prob", "logits") if k in A), None)
        if fk is None or pk is None:
            print("    -> file names or probabilities not identified; paste the key list above back for an adapted re-scoring")
            continue
        names = [Path(str(x)).name for x in A[fk]]
        pr = np.asarray(A[pk])
        pred = pr.argmax(1) if pr.ndim == 2 else (pr > 0.5).astype(int)
        y = np.array([lab.get(n, -1) for n in names])
        keep = y >= 0
        if pr.ndim == 2 and pr.shape[1] == 3:
            ba = np.mean([np.mean(pred[keep][y[keep] == c] == c) for c in range(3) if (y[keep] == c).any()])
            Q.kv("matched predictions", f"{int(keep.sum()):,} of {len(names):,}")
            Q.kv("balanced accuracy vs the authors' labels", f"{100 * ba:.2f}%")
            OUT[src] = dict(n=int(keep.sum()), ba_authors=float(ba))
        else:
            print("    -> binary arm (A2B): re-scoring against OSCC vs rest of the authors' labels")
            yb = (y[keep] == 2).astype(int)
            ba = np.mean([np.mean(pred[keep][yb == c] == c) for c in (0, 1) if (yb == c).any()])
            Q.kv("balanced accuracy (OSCC vs rest, authors' labels)", f"{100 * ba:.2f}%")
            OUT[src] = dict(n=int(keep.sum()), ba_authors_binary=float(ba))
    Q.jsave(Q.D["eval"] / "a2_study1_ndb_rescored.json", OUT)
print("\n[BLOCK 39 ✓]  run BLOCK 40 next")


In [ ]:
# ============================================================================
# BLOCK 40 — A2 RELEASE PACKAGE: plan + amendment, A2 job list and run manifest, split manifests, results, figures, code
# needs: BLOCK 38 · writes q1/release/oralleak_A2_<date>.zip (download it and attach it to the chat)
# ============================================================================
import os, sys, json, time, csv, zipfile, hashlib, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
_libsrc = (Q1_ROOT / "lib" / "q1lib.py").read_text() if (Q1_ROOT / "lib" / "q1lib.py").exists() else ""
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library, written by an old BLOCK 15 cell. Amendment A2 needs the A1 "
                     "library: replace the whole content of the BLOCK 15 and BLOCK 19 cells with the A1 versions (the first two "
                     "code cells of the A2 notebook), run 15 -> 19, then run this block again.")
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("40")
Q.hdr("BLOCK 40 — A2 RELEASE PACKAGE")
JA = (Q.jload(Q.D["jobs"] / "jobs_A2.json", {}) or {}).get("jobs", [])
man = Q.D["release"] / "a2_runs_manifest.csv"
Q.D["release"].mkdir(parents=True, exist_ok=True)
with open(man, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["job_id", "phase", "exp", "dataset", "arm", "outer", "seed", "status", "own_bal_acc", "hold_bal_acc", "external",
                "external_bal_acc", "steps_done", "best_step", "n_train", "n_own", "n_hold", "own_same_patient", "seconds", "finished_utc",
                "q1lib_hash", "job_fingerprint"])
    for j in JA:
        r = Q.jload(Q.D["runs"] / j["id"] / "result.json") or {}
        ext, eba = "", ""
        if r.get("status") == "done" and j["spec"].get("xeval_to"):
            _, z = Q.load_run(j["id"])
            t = j["spec"]["xeval_to"][0]
            y, p = z[f"x_{t}_y"], z[f"x_{t}_logit"].argmax(1)
            ext, eba = t, f"{np.mean([np.mean(p[y == c] == c) for c in np.unique(y)]):.6f}"
        s = j["spec"]
        w.writerow([j["id"], j["phase"], s["exp"], s["ds"], s["arm"], s["outer"], s["seed"], r.get("status", "not run"),
                    r.get("own_bal_acc", ""), r.get("hold_bal_acc", ""), ext, eba, r.get("steps_done", ""), r.get("best_step", ""),
                    j["sizes"]["train"], j["sizes"]["own"], j["sizes"]["hold"], j["leak"].get("struct_group", ""),
                    round(r.get("time_s", 0)), r.get("finished_utc", ""), (r.get("code") or {}).get("q1lib.py", ""), j["fp"]])
splits = {}
for ds in ("ndb_pat", "ndb_pat_k", "orchid_bin"):
    dd = Q.DS.get(ds)
    fold, _ = Q.outer_folds(dd, dict(Q.plan()["outer"][ds], rep=0))
    p = Q.D["release"] / f"a2_{ds}_files.csv"
    with open(p, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "md5", "usable", "label", "patient", "source_image", "g_ctrl", "holdout_fold"])
        for i, r in enumerate(dd.R):
            w.writerow([Path(r.get("path", "")).name if ds != "orchid_bin" else r.get("path", ""), dd.md5[i], int(dd.base[i]), int(dd.y[i]),
                        r.get("patient", ""), r.get("src_image", ""), int(dd.g_ctrl[i]), int(fold[i])])
    splits[ds] = p
stamp = time.strftime("%Y%m%d_%H%M", time.gmtime())
zp = Q.D["release"] / f"oralleak_A2_{stamp}.zip"
files = [Q.D["plan"] / n for n in ("plan.json", "amendment_A2.json", "ANALYSIS_PLAN.md")] + \
        [Q.D["jobs"] / "jobs_A2.json", man, *splits.values(), Q.D["index"] / "ndb_patch_source.csv",
         Q.D["eval"] / "a2_ndb_audit.json", Q.D["eval"] / "a2_results.json", Q.D["eval"] / "a2_study1_ndb_rescored.json",
         Q.D["tables"] / "RESULTS_A2.md"] + sorted(Q.D["tables"].glob("A2_*.csv")) + sorted(Q.D["figs"].glob("Fig_A2_*.png")) + \
        [Q.D["code"] / f"block{b}.py" for b in range(34, 41)] + [Q.D["lib"] / "q1_a2ext.py"] + \
        sorted((Q.D["jobs"] / "fail_archive_A2").glob("*.json"))
n = 0
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        if Path(p).exists():
            z.write(p, f"oralleak_A2/{Path(p).relative_to(Q.ROOT) if str(p).startswith(str(Q.ROOT)) else Path(p).name}")
            n += 1
Q.kv("package", f"{zp} · {n} files · {zp.stat().st_size / 1e6:.1f} MB · SHA-256 {Q.sha256_file(zp)[:16]}")
Q.kv("runs in manifest", f"{sum(1 for j in JA if Q.job_done(j))} done of {len(JA)}")
print("\n  Download this zip (Output panel -> q1/release) and attach it to the chat together with the printed output of\n"
      "  BLOCKS 34, 35, 36, 38 (and 39 if it found anything) and the final AUTO-SAVE table of BLOCK 37.")
print(f"\n[BLOCK 40 ✓]  then Save Version")


In [ ]:
# ============================================================================
# BLOCK 41 — A2 ADDENDUM (post hoc, CPU only): threshold-free external discrimination (AUROC), sensitivity, specificity
#            and predicted-class mix of the X4A / X4B models on their external sets
# needs: BLOCK 15 (+ 34) and the finished A2 runs (session 2) · no GPU, no training · about 2-5 min
# writes eval/a2_external_auroc.json · tables/A2_external_auroc.csv · tables/RESULTS_A2_addendum.md
#        release/oralleak_A2_addendum_<date>.zip (download it and attach it to the chat)
# Not part of amendment A2: reported as a post hoc, descriptive analysis. No hypothesis test of A2 changes.
# ============================================================================
import os, sys, json, time, csv, zipfile, importlib, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
_libsrc = (Q1_ROOT / "lib" / "q1lib.py").read_text() if (Q1_ROOT / "lib" / "q1lib.py").exists() else ""
if "def load_registry" not in _libsrc or 'rv = fac.get("recov")' not in _libsrc:
    raise SystemExit("q1/lib/q1lib.py is the Q1 (2 October) library, written by an old BLOCK 15 cell. Amendment A2 needs the A1 "
                     "library: replace the whole content of the BLOCK 15 and BLOCK 19 cells with the A1 versions (the first two "
                     "code cells of the A2 notebook), run 15 -> 19, then run this block again.")
if not (Q1_ROOT / "lib" / "q1_a2ext.py").exists():
    raise SystemExit("q1/lib/q1_a2ext.py missing - run BLOCK 34 first, then this block again")
HOOK = ("\n# --- amendment A2 hook (added by the A2 blocks; re-added automatically after BLOCK 15 rewrites this file)\n"
        "try:\n    import q1_a2ext as _a2ext\n    _a2ext.install(globals())\nexcept ImportError:\n    pass\n")
_lib = Q1_ROOT / "lib" / "q1lib.py"
if "amendment A2 hook" not in _lib.read_text():
    _lib.write_text(_lib.read_text() + HOOK)
import numpy as np
from scipy.stats import rankdata
import q1lib as Q
importlib.reload(Q)
Q.keep_source("41")

t_all = time.time()
NB = 500 if Q.TINY else 5000
Q.hdr("BLOCK 41 — A2 ADDENDUM: EXTERNAL AUROC, SENSITIVITY, SPECIFICITY (post hoc, descriptive)")
JA = (Q.jload(Q.D["jobs"] / "jobs_A2.json", {}) or {}).get("jobs", [])
if not JA:
    raise SystemExit("jobs/jobs_A2.json missing - this block needs the A2 runs of session 2")
HB = Q.DS.get("histopath")
MAG = np.array([str(r.get("mag", "")) for r in HB.R])


def auroc_rows(S, y):
    """AUROC of every row of S (runs x images) against binary y (1 = OSCC); ties get average ranks."""
    n1 = int((y == 1).sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.full(S.shape[0], np.nan)
    R = rankdata(S, axis=1)
    return (R[:, y == 1].sum(1) - n1 * (n1 + 1) / 2) / (n1 * n0)


def auroc1(s, y):
    return float(auroc_rows(s[None, :], y)[0])


def at_argmax(P, y):
    """sensitivity (OSCC), specificity (normal), balanced accuracy and share predicted OSCC, per run (rows of P = predicted class)."""
    sens = np.array([np.mean(p[y == 1] == 1) for p in P])
    spec = np.array([np.mean(p[y == 0] == 0) for p in P])
    return dict(sens=sens, spec=spec, ba=(sens + spec) / 2, pred_oscc=np.array([np.mean(p == 1) for p in P]))


def load_arm(exp, ds, arm, seeds):
    """external scores of all finished runs of one arm, aligned on the external row index."""
    out = []
    for j in JA:
        s = j["spec"]
        if s["exp"] != exp or s["ds"] != ds or s["arm"] != arm or s["seed"] not in seeds or not Q.job_done(j):
            continue
        _, z = Q.load_run(j["id"])
        t = s["xeval_to"][0]
        row = z[f"x_{t}_row"]
        o = np.argsort(row, kind="stable")
        lg = z[f"x_{t}_logit"][o].astype(np.float64)
        rec = dict(id=j["id"], outer=int(s["outer"]), seed=int(s["seed"]), t=t, row=row[o], y=z[f"x_{t}_y"][o].astype(int),
                   key=z[f"x_{t}_key"][o], score=lg[:, 1] - lg[:, 0], pred=lg.argmax(1))
        for part in ("own", "hold"):
            if f"{part}_logit" in z.files:
                lp = z[f"{part}_logit"].astype(np.float64)
                yp = z[f"{part}_y"].astype(int)
                rec[f"{part}_auroc"] = auroc1(lp[:, 1] - lp[:, 0], yp) if lp.shape[1] == 2 else np.nan
        out.append(rec)
    return out


def arm_matrix(R_):
    ref = R_[0]
    for r in R_[1:]:
        if not (np.array_equal(r["row"], ref["row"]) and np.array_equal(r["y"], ref["y"])):
            raise SystemExit(f"external sets differ between runs {ref['id']} and {r['id']} - paste this output back")
    return np.stack([r["score"] for r in R_]), np.stack([r["pred"] for r in R_]), np.array([r["outer"] for r in R_]), ref


def boot(SL, SC, oL, oC, y, key, mask, B, seed):
    """two-stage bootstrap: outer folds (paired between arms), then external clusters (shared by every run)."""
    g = np.random.default_rng(seed)
    idx_all = np.nonzero(mask)[0]
    ks = key[idx_all]
    uk, inv = np.unique(ks, return_inverse=True)
    members = [idx_all[inv == u] for u in range(len(uk))]
    folds = np.array(sorted(set(oL.tolist()) & set(oC.tolist())))
    aL, aC = np.empty(B), np.empty(B)
    for b in range(B):
        idx = np.concatenate([members[i] for i in g.integers(0, len(members), len(members))])
        yb = y[idx]
        fl = folds[g.integers(0, len(folds), len(folds))]
        AL, AC = auroc_rows(SL[:, idx], yb), auroc_rows(SC[:, idx], yb)
        mL = {f: np.nanmean(AL[oL == f]) for f in folds}
        mC = {f: np.nanmean(AC[oC == f]) for f in folds}
        aL[b] = np.mean([mL[f] for f in fl])
        aC[b] = np.mean([mC[f] for f in fl])
    return aL, aC


def fold_mean(v, o):
    return float(np.mean([np.nanmean(v[o == f]) for f in sorted(set(o.tolist()))]))


OUT, CSV, MD = {}, [], ["# Amendment A2 — addendum (post hoc, descriptive): external discrimination", "",
                        f"Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 41. Not part of amendment A2; "
                        "no A2 hypothesis test changes.", "",
                        "| exp | subset | arm | runs | external n (OSCC/normal) | AUROC [95% CI] | BA at argmax | sensitivity | "
                        "specificity | predicted OSCC % | own-fold AUROC | internal-holdout AUROC |", "|---|---|---|---|---|---|---|---|---|---|---|---|"]
rows = []
for exp, ds, seeds_main in (("X4A", "orchid_bin", [0, 1, 2]), ("X4B", "histopath", [0])):
    for scope, sd in (("main", seeds_main), ("all seeds", [0, 1, 2])):
        RL, RC = load_arm(exp, ds, "L", sd), load_arm(exp, ds, "C", sd)
        if not RL or not RC:
            continue
        if scope == "all seeds" and len({r["seed"] for r in RL + RC}) <= len(seeds_main):
            continue
        SL, PL, oL, ref = arm_matrix(RL)
        SC, PC, oC, refc = arm_matrix(RC)
        if not np.array_equal(ref["row"], refc["row"]):
            raise SystemExit(f"{exp}: external sets differ between arms L and C - paste this output back")
        y, key, t = ref["y"], ref["key"], ref["t"]
        subsets = [("all", np.ones(len(y), bool))]
        if t == "histopath":
            subsets += [(m_, MAG[ref["row"]] == m_) for m_ in ("100x", "400x")]
        for sname, mask in subsets:
            aL, aC = boot(SL, SC, oL, oC, y, key, mask, NB, Q.shash(exp, scope, sname) % 2 ** 32)
            pt = {}
            for arm, S_, P_, o_, R_, ab in (("L", SL, PL, oL, RL, aL), ("C", SC, PC, oC, RC, aC)):
                A = auroc_rows(S_[:, mask], y[mask])
                M = at_argmax(P_[:, mask], y[mask])
                pt[arm] = dict(n_runs=len(R_), n_ext=int(mask.sum()), n_oscc=int((y[mask] == 1).sum()), n_normal=int((y[mask] == 0).sum()),
                               auroc=fold_mean(A, o_), auroc_ci=list(Q.ci(ab)), auroc_run_sd=float(np.nanstd(A, ddof=1)) if len(A) > 1 else None,
                               ba=fold_mean(M["ba"], o_), sens=fold_mean(M["sens"], o_), spec=fold_mean(M["spec"], o_),
                               pred_oscc=fold_mean(M["pred_oscc"], o_), true_oscc=float(np.mean(y[mask] == 1)),
                               own_auroc=float(np.nanmean([r.get("own_auroc", np.nan) for r in R_])),
                               hold_auroc=float(np.nanmean([r.get("hold_auroc", np.nan) for r in R_])),
                               p_auroc_vs_half=float(Q.boot_p(ab, 0.5)))
            d = aL - aC
            pt["L_minus_C"] = dict(est=pt["L"]["auroc"] - pt["C"]["auroc"], ci=list(Q.ci(d)), p=float(Q.boot_p(d)))
            OUT[f"{exp}|{scope}|{sname}"] = pt
            for arm in ("L", "C"):
                v = pt[arm]
                rows.append([exp, scope, sname, arm, v["n_runs"], f"{v['n_oscc']}/{v['n_normal']}",
                             f"{v['auroc']:.3f} [{v['auroc_ci'][0]:.3f}, {v['auroc_ci'][1]:.3f}]", f"{100 * v['ba']:.2f}",
                             f"{100 * v['sens']:.1f}", f"{100 * v['spec']:.1f}", f"{100 * v['pred_oscc']:.1f} ({100 * v['true_oscc']:.1f})",
                             f"{v['own_auroc']:.3f}" if sname == "all" else "", f"{v['hold_auroc']:.3f}" if sname == "all" else ""])
                CSV.append([exp, scope, sname, arm, v["n_runs"], v["n_oscc"], v["n_normal"], v["auroc"], *v["auroc_ci"], v["auroc_run_sd"],
                            v["p_auroc_vs_half"], v["ba"], v["sens"], v["spec"], v["pred_oscc"], v["true_oscc"], v["own_auroc"], v["hold_auroc"]])
                MD.append(f"| {exp} | {sname} ({scope}) | {arm} | {v['n_runs']} | {v['n_oscc']}/{v['n_normal']} | "
                          f"{v['auroc']:.3f} [{v['auroc_ci'][0]:.3f}, {v['auroc_ci'][1]:.3f}] | {100 * v['ba']:.2f} | {100 * v['sens']:.1f} | "
                          f"{100 * v['spec']:.1f} | {100 * v['pred_oscc']:.1f} | {v['own_auroc']:.3f} | {v['hold_auroc']:.3f} |")
            e = pt["L_minus_C"]
            rows.append([exp, scope, sname, "L−C", "", "", f"{e['est']:+.3f} [{e['ci'][0]:+.3f}, {e['ci'][1]:+.3f}] · p {e['p']:.4f}",
                         "", "", "", "", "", ""])
            CSV.append([exp, scope, sname, "L-C", "", "", "", e["est"], *e["ci"], "", e["p"], "", "", "", "", "", "", ""])
            MD.append(f"| {exp} | {sname} ({scope}) | L − C | | | {e['est']:+.3f} [{e['ci'][0]:+.3f}, {e['ci'][1]:+.3f}] · p {e['p']:.4f} "
                      f"| | | | | | |")

Q.sub("EXTERNAL SETS  (X4A: ORCHID normal vs OSCC -> Rahman et al. originals · X4B: Rahman et al. Kaggle copy -> ORCHID)")
Q.table(["exp", "seeds", "subset", "arm", "runs", "OSCC/normal", "external AUROC [95% CI]", "BA %", "sens %", "spec %",
         "pred OSCC % (true)", "own AUROC", "hold AUROC"], rows, [4, 10, 7, 5, 5, 12, 36, 7, 7, 7, 19, 10, 10])
print("  AUROC: OSCC score = logit(OSCC) - logit(normal), threshold-free; mean over outer folds of the mean over seeds.\n"
      "  95% CI: two-stage bootstrap (outer folds paired between arms, then external clusters), "
      f"{NB} replicates. BA, sensitivity, specificity at the trained decision rule (argmax).\n"
      "  own / hold AUROC: mean over runs on the arm's own fold and on the internal clean holdout.")
for k, v in OUT.items():
    if k.endswith("|all"):
        Q.kv(f"{k.split('|')[0]} {k.split('|')[1]} AUROC vs 0.5",
             f"L p {v['L']['p_auroc_vs_half']:.4f} · C p {v['C']['p_auroc_vs_half']:.4f}")

Q.jsave(Q.D["eval"] / "a2_external_auroc.json", dict(meta=dict(block="41", post_hoc=True, boot=NB,
                                                                 created_utc=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())),
                                                       results=OUT))
with open(Q.D["tables"] / "A2_external_auroc.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["exp", "seeds", "subset", "arm", "runs", "n_oscc", "n_normal", "auroc", "ci_lo", "ci_hi", "auroc_run_sd", "p",
                "ba_argmax", "sensitivity", "specificity", "pred_oscc", "true_oscc", "own_auroc", "hold_auroc"])
    w.writerows(CSV)
MD += ["", "AUROC: score = logit(OSCC) − logit(normal); mean over outer folds of the seed mean. 95% CI from a two-stage bootstrap "
       f"(outer folds paired between arms, then external clusters; {NB} replicates). p for L − C: two-sided bootstrap; p for each arm "
       "in the JSON tests AUROC = 0.5. BA, sensitivity and specificity use the trained decision rule (argmax)."]
(Q.D["tables"] / "RESULTS_A2_addendum.md").write_text("\n".join(MD) + "\n")

stamp = time.strftime("%Y%m%d_%H%M", time.gmtime())
Q.D["release"].mkdir(parents=True, exist_ok=True)
zp = Q.D["release"] / f"oralleak_A2_addendum_{stamp}.zip"
files = [Q.D["eval"] / "a2_external_auroc.json", Q.D["tables"] / "A2_external_auroc.csv", Q.D["tables"] / "RESULTS_A2_addendum.md",
         Q.D["code"] / "block41.py", Q.D["code"] / "block37.py"]
n = 0
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        if Path(p).exists():
            z.write(p, f"oralleak_A2/{Path(p).relative_to(Q.ROOT)}")
            n += 1
Q.kv("package", f"{zp} · {n} files · SHA-256 {Q.sha256_file(zp)[:16]}")
print(f"\n[BLOCK 41 ✓]  {Q.hms(time.time() - t_all)} · download the zip from q1/release, attach it with this output, then Save Version")

In [ ]:
# ============================================================================
# BLOCK 16 — DATA AUDIT: INDEX · IMAGE STORE · EXACT + NEAR DUPLICATES · GROUP RECOVERY
# ============================================================================
import os, sys, json, time, importlib, warnings
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("16")

FORCE_REBUILD = False          # True re-reads every file (normally reused across sessions)
P = Q.plan()
ND, SG, PR = P["near_dup"], P["segmentation"], P["parent_recovery"]
EMB = SG["embed"]
try:
    Q._emb_model(EMB, 56 if Q.TINY else 224, "cuda" if __import__("torch").cuda.is_available() else "cpu")
except Exception as ex:
    print(f"  embedding model {EMB} unavailable ({type(ex).__name__}: {str(ex)[:80]}) -> using convnext_t")
    EMB = "convnext_t"
Q.hdr("BLOCK 16 — DATA AUDIT · INDEX · STORE · DUPLICATES · GROUPS")
t_all = time.time()
found, n_img = Q.walk_input()
rng = np.random.default_rng(0)
audit = Q.jload(Q.D["index"] / "audit.json", {}) or {}
emb_all = {}

# ------------------------------------------------------------------ 1  index + store (decode every file once)
Q.sub("1  INDEX AND IMAGE STORE  (MD5 of every file, 288-px uint8 store on /kaggle/working)")
# disk check before anything is written: stores to build + 4 GB for embeddings, run outputs and checkpoints
to_build = 0
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    old = Q.jload(Q.D["index"] / f"{ds}.raw.json")
    if recs and (FORCE_REBUILD or not (Q.D["store"] / f"{ds}.npy").exists() or not old
                 or old.get("paths_sha") != Q.fingerprint([r["path"] for r in recs])):
        to_build += len(recs)
need_gb = to_build * Q.STORE_PX * Q.STORE_PX * 3 / 1e9 + 4.0
free_gb = Q.disk_free_gb()
Q.kv("disk", f"{free_gb:.1f} GB free · about {need_gb:.1f} GB needed ({to_build:,} images to decode + 4 GB working space)")
if free_gb < need_gb:
    wk = Q1_ROOT.parent
    sizes = []
    for p in sorted(wk.iterdir()):
        if p.resolve() == Q1_ROOT.resolve():
            continue
        tot = p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        sizes.append((tot / 1e9, p.name))
    print("  largest items in /kaggle/working outside q1/ (nothing is deleted automatically):")
    for gb, nm in sorted(sizes, reverse=True)[:12]:
        print(f"    {gb:7.2f} GB  {nm}")
    raise SystemExit(f"not enough disk: free {free_gb:.1f} GB < needed {need_gb:.1f} GB. Remove files you no longer need "
                     "(for example old checkpoints of Blocks 1-14 that are already saved in a Kaggle version), then re-run.")
rows = []
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    if not recs:
        rows.append([ds, 0, "—", "—", "not attached"])
        continue
    paths_sha = Q.fingerprint([r["path"] for r in recs])
    old = Q.jload(Q.D["index"] / f"{ds}.raw.json")
    store_ok = (Q.D["store"] / f"{ds}.npy").exists()
    if old and old.get("paths_sha") == paths_sha and store_ok and not FORCE_REBUILD:
        recs = old["records"]
        how = "reused"
        secs = 0
    else:
        meta, secs = Q.build_store(ds, recs)
        for i, (r, m) in enumerate(zip(recs, meta)):
            r.update(m)
            r["uid"] = i
            r.setdefault("orig", True)
        if ds == "breakhis":   # the same file present twice in the mirror (same name and bytes) is a mirror artefact
            seen = set()
            for r in recs:
                k = (Path(r["path"]).name, r["md5"])
                if k in seen:
                    r["ok"], r["mirror_copy"] = False, True
                seen.add(k)
        Q.jsave(Q.D["index"] / f"{ds}.raw.json", dict(paths_sha=paths_sha, records=recs))
        how = "built"
    st = Q.open_store(ds)
    bad = sum(1 for r in recs if not r.get("ok", True))
    rows.append([ds, len(recs), f"{st.nbytes / 1e9:.2f} GB", f"{bad}", f"{how} {secs:.0f}s"])
    found[ds] = recs
Q.table(["dataset", "files", "store", "unreadable/mirror", "status"], rows, [13, 8, 10, 20, 20])

# ------------------------------------------------------------------ 2  perceptual hashes + embeddings
Q.sub(f"2  PERCEPTUAL HASHES (pHash, dHash) AND EMBEDDINGS ({Q.EMB_MODELS.get(EMB, EMB)})")
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    if not recs:
        continue
    st = Q.open_store(ds)
    fe = Q.D["store"] / f"{ds}_emb_{EMB}.npy"
    fh = Q.D["store"] / f"{ds}_hash.npz"
    t0 = time.time()
    if not fh.exists() or FORCE_REBUILD:
        ph, dh = Q.perceptual_hashes(st)
        np.savez(fh, ph=ph, dh=dh)
    if not fe.exists() or FORCE_REBUILD:
        e = Q.embed(ds, np.arange(len(recs)), key=EMB)
        np.save(fe, e.astype(np.float16))
    emb_all[ds] = np.load(fe).astype(np.float32)
    Q.kv(ds, f"{len(recs)} images · dim {emb_all[ds].shape[1]} · {time.time() - t0:.0f}s")

# ------------------------------------------------------------------ 3  exact + near duplicates within each dataset
Q.sub("3  DUPLICATES WITHIN EACH DATASET  (counts of pairs / images involved at several thresholds)")
thr_rows = []
near_rule = {}
comp = {}
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    if not recs:
        continue
    n = len(recs)
    ok = np.array([r.get("ok", True) for r in recs])
    orig = np.array([r.get("orig", True) for r in recs])
    use = np.nonzero(ok & orig)[0]            # publisher-augmented children are handled by parent recovery
    H = np.load(Q.D["store"] / f"{ds}_hash.npz")
    ph = H["ph"]
    E = emb_all[ds]
    md5 = np.array([r["md5"] for r in recs])
    ii, jj, dd_ = Q.hamming_pairs(ph[use], max_d=ND["cand_phash"])
    ci_, cj_, cs_ = Q.cos_pairs(E[use], thr=ND["cand_cos"])
    cand = {}
    for a, b, d in zip(use[ii], use[jj], dd_):
        cand[(a, b)] = [int(d), None]
    for a, b, s in zip(use[ci_], use[cj_], cs_):
        cand.setdefault((a, b), [None, None])[1] = float(s)
    pairs = np.array(list(cand.keys()), int).reshape(-1, 2)
    ham = np.array([v[0] if v[0] is not None else int(Q.popcount64(np.array([ph[a] ^ ph[b]]))[0])
                    for (a, b), v in cand.items()], int)
    cosv = np.array([v[1] if v[1] is not None else float(E[a] @ E[b]) for (a, b), v in cand.items()])
    st = Q.open_store(ds)
    ss = np.full(len(pairs), np.nan)
    need = np.nonzero(ham <= max(ND["phash_max"], ND["fallback_phash"]))[0]
    if len(need):
        ss[need] = Q.ssim_pairs(st, pairs[need, 0], st, pairs[need, 1], max_pairs=10 ** 9)
    exact = md5[pairs[:, 0]] == md5[pairs[:, 1]] if len(pairs) else np.zeros(0, bool)
    # all exact pairs (md5), also those far apart in hash space
    by = defaultdict(list)
    for i in use:
        by[md5[i]].append(i)
    ex_pairs = [(g[0], x) for g in by.values() if len(g) > 1 for x in g[1:]]

    def involved(m):
        return len(set(pairs[m].ravel().tolist())) if m.any() else 0

    ssf = np.nan_to_num(ss, nan=-1.0)
    rule = (ham <= ND["phash_max"]) & (ssf >= ND["ssim_min"])
    edges = [tuple(p) for p in pairs[rule]] + ex_pairs
    roots = Q.union_labels(n, pairs=edges)
    sizes = Counter(roots[use].tolist())
    big = max(sizes.values()) if sizes else 1
    cap = max(ND["cap_min"], ND["cap_frac"] * len(use))
    used_rule = "main"
    if big > cap:
        rule = (ham <= ND["fallback_phash"]) & (ssf >= ND["fallback_ssim"])
        edges = [tuple(p) for p in pairs[rule]] + ex_pairs
        roots = Q.union_labels(n, pairs=edges)
        used_rule = f"fallback (largest component {big} > cap {cap:.0f})"
    near_rule[ds] = used_rule
    comp[ds] = dict(roots=roots, ex_pairs=ex_pairs)
    n_ex_groups = sum(1 for g in by.values() if len(g) > 1)
    n_ex_red = sum(len(g) - 1 for g in by.values() if len(g) > 1)
    ncomp = len(set(roots[use].tolist()))
    thr_rows.append([ds, len(use), f"{n_ex_groups}/{n_ex_red}",
                     f"{int((ham <= 0).sum())}/{int((ham <= 4).sum())}/{int((ham <= 8).sum())}/{int((ham <= 12).sum())}",
                     f"{int((cosv >= .99).sum())}/{int((cosv >= .97).sum())}/{int((cosv >= .95).sum())}",
                     f"{int((ssf >= .95).sum())}/{int((ssf >= .90).sum())}",
                     f"{int(rule.sum())} pairs · {involved(rule)} imgs · {len(use) - ncomp} redundant"])
    audit.setdefault(ds, {}).update(dict(
        n_files=n, n_originals=int(orig.sum()), exact_groups=n_ex_groups, exact_redundant=n_ex_red,
        pairs_phash_le=dict(zip(["0", "4", "8", "12"], [int((ham <= t).sum()) for t in (0, 4, 8, 12)])),
        pairs_cos_ge=dict(zip([".99", ".97", ".95"], [int((cosv >= t).sum()) for t in (.99, .97, .95)])),
        pairs_ssim_ge=dict(zip([".95", ".90"], [int((ssf >= t).sum()) for t in (.95, .90)])),
        near_rule=used_rule, near_pairs=int(rule.sum()), near_images=involved(rule),
        near_redundant=int(len(use) - ncomp)))
Q.table(["dataset", "originals", "exact grp/red", "pHash<=0/4/8/12", "cos>=.99/.97/.95", "SSIM>=.95/.90",
         "near-duplicate rule (pre-specified)"], thr_rows, [12, 10, 14, 17, 17, 14, 36])
print("  SSIM is computed for the pairs with pHash distance <= 8 (the only pairs where the rule needs it).")
print("  near-duplicate rule used for grouping: identical file, or pHash distance <= 8 AND SSIM >= 0.90.")
print("  cosine counts describe semantic neighbours only (same tissue pattern); they are not used for grouping.")
for ds, r in near_rule.items():
    if r != "main":
        print(f"  {ds}: {r}")

# ------------------------------------------------------------------ 4  structural groups
Q.sub("4  STRUCTURAL GROUPS  (slide · patient · recovered source image)")
struct, coarse, seginfo = {}, {}, {}
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    if not recs:
        continue
    n = len(recs)
    lab = np.array([r["label"] for r in recs])
    E = emb_all[ds]
    if ds in ("he_stained", "breakhis"):
        key = "slide" if ds == "he_stained" else "patient"
        vals = [r[key] for r in recs]
        first = {}
        s = np.array([first.setdefault(v, i) for i, v in enumerate(vals)])
        struct[ds] = s
        k_ = len(set(vals))
        extra = ""
        if ds == "he_stained":
            sp = defaultdict(set)
            for r in recs:
                sp[r["slide"]].add(r["native"])
            multi = sum(1 for v in sp.values() if len(v) > 1)
            extra = f" · {multi}/{k_} slides appear in both shipped train and test"
            audit[ds]["slides_in_both_shipped_splits"] = multi
        Q.kv(ds, f"{k_} {key}s · images per {key}: {min(Counter(vals).values())}-{max(Counter(vals).values())}{extra}")
        audit[ds]["n_groups"] = k_
    elif ds == "ndb_ufes":
        order = np.argsort([r["pidx"] for r in recs], kind="stable")
        lift, adj, rnd = Q.adjacency_lift(order, lab, E, rng)
        seg, si = Q.segment_order(order, lab, E)
        blk = Q.coarse_blocks(order, seg, lab, SG["coarse_min"])
        first = {}
        struct[ds] = np.array([first.setdefault(v, i) for i, v in enumerate(seg)])
        first = {}
        coarse[ds] = np.array([first.setdefault(v, i) for i, v in enumerate(blk)])
        # agreement check: is each patch's nearest neighbour in its own segment?
        nn = np.argsort(-(E @ E.T), 1)[:, 1]
        same_seg = float(np.mean(seg[nn] == seg))
        seginfo[ds] = dict(si, lift=lift, adj_cos=adj, rand_same_class_cos=rnd, nn_same_segment=same_seg,
                           n_coarse=int(len(set(blk.tolist()))))
        Q.kv("ndb_ufes order test", f"adjacent cosine {adj:.3f} vs random same-class {rnd:.3f} (lift {lift:+.3f})")
        Q.kv("ndb_ufes segments", f"{si['n_seg']} recovered source images (dataset paper: 237) · size median "
                                  f"{si['size_med']:.0f} [IQR {si['size_q1']:.0f}-{si['size_q3']:.0f}] max {si['size_max']} · "
                                  f"Otsu threshold {si['tau']:.3f}")
        Q.kv("ndb_ufes check", f"{100 * same_seg:.1f}% of patches have their nearest neighbour in their own segment")
        Q.kv("ndb_ufes coarse blocks", f"{seginfo[ds]['n_coarse']} contiguous blocks of >= {SG['coarse_min']} patches "
                                       "(sensitivity grouping, and the grouping used for the clean holdout)")
        audit[ds]["segmentation"] = seginfo[ds]
        audit[ds]["n_groups"] = si["n_seg"]
    elif ds == "histopath":
        origs = [i for i, r in enumerate(recs) if r.get("orig")]
        order = sorted(origs, key=lambda i: (recs[i]["label"], recs[i].get("mag", ""), recs[i].get("pidx", 0)))
        lift, adj, rnd = Q.adjacency_lift(np.array(order), lab, E, rng)
        s = np.arange(n)
        applied = lift > SG["hist_lift_min"]
        if applied:
            seg, si = Q.segment_order(np.array(order), lab, E)
            first = {}
            for i in order:
                s[i] = first.setdefault(seg[i], i)
            seginfo[ds] = si
        struct[ds] = s
        Q.kv("histopath order test", f"adjacent cosine {adj:.3f} vs random same-class {rnd:.3f} (lift {lift:+.3f}) -> "
                                     + ("index order encodes source: segments used as groups" if applied else
                                        f"no grouping signal (lift <= {SG['hist_lift_min']}): each original is its own group"))
        audit[ds]["order_lift"] = lift
        audit[ds]["order_grouping_applied"] = bool(applied)
    else:
        struct[ds] = np.arange(n)
        Q.kv(ds, "no slide / patient information: duplicate clusters are the only groups")

# ------------------------------------------------------------------ 5  publisher-augmented children (histopathology)
if found.get("histopath"):
    recs = found["histopath"]
    ch = np.array([i for i, r in enumerate(recs) if not r.get("orig") and r.get("ok", True)])
    Q.sub(f"5  PROVENANCE OF THE {len(ch):,} PUBLISHER-AUGMENTED HISTOPATHOLOGY FILES  (parent recovery)")
    og = np.array([i for i, r in enumerate(recs) if r.get("orig") and r.get("ok", True)])
    lab = np.array([r["label"] for r in recs])
    if len(ch):
        ev = Q.embed("histopath", ch, key=EMB, views="d4")
        roots = comp["histopath"]["roots"]
        top, s1, s2, okp = Q.recover_parents(ev, emb_all["histopath"][og], lab[ch], lab[og], roots[og],
                                             PR["cos_min"], PR["margin_min"])
        for k, i in enumerate(ch):
            recs[i]["parent"] = int(og[top[k]])
            recs[i]["parent_ok"] = bool(okp[k])
            recs[i]["parent_cos"] = float(s1[k])
            recs[i]["parent_margin"] = float(s1[k] - s2[k])
        rate = float(okp.mean())
        par_split = Counter(recs[og[top[k]]]["native"] for k in range(len(ch)) if okp[k])
        a_num = np.array([recs[i].get("aug_a", -1) for i in ch])
        pidx = np.array([recs[og[t]].get("pidx", -1) for t in top])
        corr = float(np.corrcoef(a_num[okp], pidx[okp])[0, 1]) if okp.sum() > 3 and a_num[okp].std() > 0 else float("nan")
        n_par = len(set(og[top[okp]].tolist()))
        Q.kv("children", f"{len(ch)} files named aug_<a>_<b>")
        Q.kv("parent recovered", f"{int(okp.sum())} ({100 * rate:.1f}%) at cosine >= {PR['cos_min']} and margin >= {PR['margin_min']}"
                                 f" (max over 8 dihedral views)")
        Q.kv("cosine to parent", f"median {np.median(s1):.3f} · 5th pct {np.percentile(s1, 5):.3f} · margin median {np.median(s1 - s2):.3f}")
        Q.kv("distinct parents", f"{n_par} originals · native split of parents {dict(par_split)}")
        Q.kv("file name vs parent", f"correlation of <a> with parent index {corr:.2f} (names do not encode the parent)"
             if not np.isfinite(corr) or abs(corr) < 0.5 else f"correlation {corr:.2f}")
        pre = rate >= PR["enable_pre_min_rate"]
        Q.kv("PRE experiment", "ENABLED" if pre else f"DISABLED (recovery {100 * rate:.1f}% < {100 * PR['enable_pre_min_rate']:.0f}%)")
        audit["histopath"].update(dict(children=int(len(ch)), parent_rate=rate, parent_cos_median=float(np.median(s1)),
                                       n_parents=n_par, parent_native=dict(par_split), pre_enabled=bool(pre),
                                       aug_name_corr=corr))

# ------------------------------------------------------------------ 6  final groups and index files
Q.sub("6  GROUPS USED BY THE EXPERIMENTS")
grow = []
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    if not recs:
        continue
    n = len(recs)
    near = comp[ds]["roots"]
    md5 = [r["md5"] for r in recs]
    first = {}
    exact = np.array([first.setdefault(m, i) for i, m in enumerate(md5)])
    s = struct[ds]
    cz = coarse.get(ds, s)
    par = np.array([r.get("parent", -1) if r.get("parent_ok") else -1 for r in recs])
    links = [(i, p) for i, p in enumerate(par) if p >= 0]
    ctrl = Q.union_labels(n, pairs=links, label_arrays=[near, s, cz])
    ok = np.array([r.get("ok", True) for r in recs])
    orig = np.array([r.get("orig", True) for r in recs])
    rep = np.zeros(n, bool)
    seen = set()
    for i in np.argsort(exact, kind="stable"):
        if ok[i] and orig[i] and exact[i] not in seen:
            rep[i] = True
            seen.add(exact[i])
    for i, r in enumerate(recs):
        r.update(uid=i, g_exact=int(exact[i]), g_near=int(near[i]), g_struct=int(s[i]), g_coarse=int(cz[i]),
                 g_ctrl=int(ctrl[i]), dup_rep=bool(rep[i]))
    base = ok & orig & rep
    nb = int(base.sum())
    g = len(set(ctrl[base].tolist()))
    labs = np.array([r["label"] for r in recs])
    gl = defaultdict(set)
    for i in np.nonzero(base)[0]:
        gl[ctrl[i]].add(int(labs[i]))
    mixed = sum(1 for v in gl.values() if len(v) > 1)
    summ = dict(n_files=n, n_unique_originals=nb, n_ctrl_groups=g, mixed_label_groups=mixed,
                near_rule=near_rule.get(ds), classes=Q.META[ds]["classes"])
    Q.jsave(Q.D["index"] / f"{ds}.json", dict(summary=summ, records=recs))
    audit[ds].update(summ)
    grow.append([ds, n, nb, g, f"{nb / max(1, g):.1f}", mixed, Q.META[ds]["group"]])
Q.table(["dataset", "files", "unique originals", "groups (g_ctrl)", "imgs/group", "mixed-label", "grouping"],
        grow, [13, 8, 17, 16, 11, 12, 26])
print("  g_ctrl = identical files + near duplicates + slide/patient/recovered source (+ parent of augmented copies).")
print("  The clean holdout of every experiment is split on g_ctrl, so no arm can see a holdout group.")

# ------------------------------------------------------------------ 7  cross-dataset near duplicates
Q.sub("7  CROSS-DATASET NEAR DUPLICATES")
names = [d for d in Q.DATASETS if found.get(d)]
xrows = []
for a_i, a in enumerate(names):
    for b in names[a_i + 1:]:
        ha = np.load(Q.D["store"] / f"{a}_hash.npz")["ph"]
        hb = np.load(Q.D["store"] / f"{b}_hash.npz")["ph"]
        i1, j1, d1 = Q.hamming_pairs(ha, hb, max_d=8)
        i2, j2, s2_ = Q.cos_pairs(emb_all[a], emb_all[b], thr=0.97)
        pr = set(zip(i1.tolist(), j1.tolist())) | set(zip(i2.tolist(), j2.tolist()))
        conf = 0
        if pr:
            pa = np.array(sorted(pr))
            ssx = Q.ssim_pairs(Q.open_store(a), pa[:, 0], Q.open_store(b), pa[:, 1], max_pairs=20000)
            conf = int((ssx >= 0.90).sum())
        xrows.append([f"{a} × {b}", len(i1), len(i2), conf])
        audit.setdefault("cross", {})[f"{a}|{b}"] = dict(phash_le8=int(len(i1)), cos_ge97=int(len(i2)), ssim_confirmed=conf)
Q.table(["pair", "pHash<=8", "cos>=0.97", "confirmed (SSIM>=0.90)"], xrows, [30, 12, 12, 24])

audit["_meta"] = dict(time_s=time.time() - t_all, embed=EMB, near_dup=ND, finished=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()))
Q.jsave(Q.D["index"] / "audit.json", audit)
print(f"\n[BLOCK 16 ✓]  {len(names)} datasets indexed · audit in q1/index/audit.json · {Q.hms(time.time() - t_all)} · run BLOCK 17 next")

In [ ]:
# ============================================================================
# BLOCK 17 — FROZEN-FEATURE PROBES + MECHANISM ANALYSES (cheap, runs before fine-tuning)
# ============================================================================
import os, sys, json, time, importlib, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("17")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed

P = Q.plan()
PB = P["probes"]
t_all = time.time()
Q.hdr("BLOCK 17 — FROZEN-FEATURE PROBES · IDENTITY PROBE · COLOUR SHORTCUT · NEAREST-NEIGHBOUR SIMILARITY")
avail = Q.available_datasets()
audit = Q.jload(Q.D["index"] / "audit.json", {})
EMB0 = audit.get("_meta", {}).get("embed", "dinov2_s")
keys = [k if k != "dinov2_s" else EMB0 for k in PB["backbones"]]
keys = list(dict.fromkeys(keys))

# ------------------------------------------------------------------ 1  splits used by the probes
specs = []
for phase, s in Q.enumerate_specs(P):
    if s["backbone"] != P["primary_backbone"] or s["recipe"] != "std":
        continue
    if s["exp"] == "E1" and s["seed"] != 0:
        continue
    specs.append(s)
for s in [x for x in specs if x["exp"] == "E1"]:          # probe replicates: seeds 1..n-1 of E1 (splits change with seed)
    for sd in PB["seeds"][1:]:
        if s["arm"] in P["exps"]["E1"]["datasets"][s["ds"]].get("extra", []):
            continue
        specs.append(dict(s, seed=sd))
builds = {}
need_copies = defaultdict(int)
t0 = time.time()
split_err = []
for s in specs:
    try:
        B = Q.build_job(s, check=True)
    except Exception as e:
        split_err.append((Q.job_id(s), str(e)[:90]))
        continue
    builds[Q.job_id(s)] = (s, B)
    pol = B["info"]["policy"]
    if pol:
        for part in ("train", "ival", "own"):
            if len(B[part]):
                need_copies[(s["ds"], pol)] = max(need_copies[(s["ds"], pol)], int(B[part]["copy"].max()))
Q.kv("splits", f"{len(builds)} built and checked for disjointness in {time.time() - t0:.0f}s")
if split_err:
    Q.kv("splits not built", f"{len(split_err)} (skipped by the probes; the same splits are rejected by BLOCK 18)")
    for jid, msg in split_err[:10]:
        print(f"    ✗ {jid}: {msg}")

# ------------------------------------------------------------------ 2  embeddings of originals and sibling copies
Q.sub("1  EMBEDDINGS  (originals from the audit; sibling copies generated with the same deterministic transforms)")
F = {}
for key in keys:
    for ds in avail:
        fe = Q.D["store"] / f"{ds}_emb_{key}.npy"
        if not fe.exists():
            n = len(Q.DS.get(ds).R)
            np.save(fe, Q.embed(ds, np.arange(n), key=key).astype(np.float16))
        F[(key, ds, None)] = np.load(fe).astype(np.float32)
    for (ds, pol), kmax in need_copies.items():
        fs = Q.D["store"] / f"{ds}_sib_{pol}_{key}.npz"
        dd = Q.DS.get(ds)
        rows = np.nonzero(dd.ok & dd.orig)[0]
        z = np.load(fs) if fs.exists() else None
        if z is None or z["kmax"] < kmax:
            t0 = time.time()
            R_ = np.repeat(rows, kmax)
            C_ = np.tile(np.arange(1, kmax + 1), len(rows))
            e = Q.embed(ds, R_, key=key, copies=C_, policy=pol).reshape(len(rows), kmax, -1)
            np.savez(fs, rows=rows, kmax=kmax, emb=e.astype(np.float16))
            z = np.load(fs)
            Q.kv(f"{key} · {ds} · {pol}", f"{len(rows)} originals x {kmax} copies in {time.time() - t0:.0f}s")
        pos = np.full(dd.n, -1)
        pos[z["rows"]] = np.arange(len(z["rows"]))
        F[(key, ds, pol)] = (pos, z["emb"].astype(np.float32))


def feats(key, ds, pol, U):
    base = F[(key, ds, None)][U["row"]]
    if pol is None or (U["copy"] == 0).all():
        return base
    pos, e = F[(key, ds, pol)]
    m = U["copy"] > 0
    out = base.copy()
    out[m] = e[pos[U["row"][m]], U["copy"][m] - 1]
    return out


def ba(y, pred, K):
    r = [np.mean(pred[y == c] == c) for c in range(K) if (y == c).any()]
    return float(np.mean(r)) if r else float("nan")


def probe_one(rid, s, B, key):
    ds, pol, K = s["ds"], B["info"]["policy"], B["info"]["K"]
    tr, own, hold = B["train"], B["own"], B["hold"]
    Xtr, Xo, Xh = feats(key, ds, pol, tr), feats(key, ds, pol, own), feats(key, ds, None, hold)
    out = dict(id=rid, key=key, exp=s["exp"], ds=ds, arm=s["arm"], outer=s["outer"], rep=s.get("rep", 0), seed=s["seed"],
               dose=s.get("dose"), n_train=len(tr))
    if len(np.unique(tr["y"])) < 2 or len(own) == 0:
        return out
    if len(tr) > 6000:   # same cap for every arm: keeps the probe cheap without favouring either arm
        keep = np.sort(np.random.default_rng(0).choice(len(tr), 6000, replace=False))
        Xl, yl = Xtr[keep], tr["y"][keep]
    else:
        Xl, yl = Xtr, tr["y"]
    sc = StandardScaler().fit(Xl)
    lr = LogisticRegression(max_iter=300, tol=1e-3, C=1.0).fit(sc.transform(Xl), yl)
    out["logreg"] = dict(own=ba(own["y"], lr.predict(sc.transform(Xo)), K), hold=ba(hold["y"], lr.predict(sc.transform(Xh)), K))
    nn_o = np.argmax(Xo @ Xtr.T, 1)
    nn_h = np.argmax(Xh @ Xtr.T, 1)
    out["1nn"] = dict(own=ba(own["y"], tr["y"][nn_o], K), hold=ba(hold["y"], tr["y"][nn_h], K))
    so = (Xo @ Xtr.T).max(1)
    sh = (Xh @ Xtr.T).max(1)
    out["nnsim"] = dict(own=float(so.mean()), hold=float(sh.mean()))
    return out


# ------------------------------------------------------------------ 3  probes on every split
Q.sub("2  PROBES  (logistic regression = low capacity · 1-nearest-neighbour = pure memorisation)")
t0 = time.time()
tasks = [(rid, s, B, key) for key in keys for rid, (s, B) in builds.items()]
res = Parallel(n_jobs=min(4, os.cpu_count() or 1), prefer="threads")(delayed(probe_one)(*t) for t in tasks)
Q.jsave(Q.D["eval"] / "probes.json", res)
Q.kv("fits", f"{len(res)} splits x heads in {time.time() - t0:.0f}s")


def agg(filt, head, arm):
    v = [r for r in res if filt(r) and r["arm"] == arm and head in r]
    if not v:
        return None
    o = np.mean([r[head]["own"] for r in v])
    h = np.mean([r[head]["hold"] for r in v])
    return o, h, len(v)


rows = []
for key in keys:
    for ds in avail:
        for head in ("logreg", "1nn"):
            L = agg(lambda r: r["key"] == key and r["exp"] == "E1" and r["ds"] == ds, head, "L")
            C = agg(lambda r: r["key"] == key and r["exp"] == "E1" and r["ds"] == ds, head, "C")
            if L and C:
                rows.append([key, ds, head, f"{100 * L[0]:.1f} / {100 * L[1]:.1f}", f"{100 * (L[0] - L[1]):+.1f}",
                             f"{100 * C[0]:.1f} / {100 * C[1]:.1f}", f"{100 * (C[0] - C[1]):+.1f}", L[2]])
Q.table(["features", "dataset", "head", "L own/hold BA", "L infl.", "C own/hold BA", "C infl.", "splits"], rows,
        [12, 12, 8, 17, 9, 17, 9, 7])
print("  inflation = own-fold BA - clean-holdout BA of the same probe (percentage points).")

Q.sub("3  RANDOM-LABEL CONTROL WITH PROBES  (labels drawn per group; chance = 1/K)")
rows = []
for key in keys[:1]:
    for ds in avail:
        for head in ("logreg", "1nn"):
            L = agg(lambda r: r["key"] == key and r["exp"] == "RL" and r["ds"] == ds, head, "L")
            C = agg(lambda r: r["key"] == key and r["exp"] == "RL" and r["ds"] == ds, head, "C")
            if L and C:
                rows.append([ds, head, f"{100 * L[0]:.1f}", f"{100 * C[0]:.1f}", f"{100 / Q.META[ds]['K']:.1f}"])
if rows:
    Q.table(["dataset", "head", "L own BA", "C own BA", "chance"], rows, [12, 8, 12, 12, 10])

Q.sub("4  DOSE-RESPONSE WITH PROBES  (inflation in points: rows = arm, columns = dose)")
for ex_name in ("DOSEK", "DUP", "OVL"):
    doses = P["exps"][ex_name]["doses"]
    for ds in P["exps"][ex_name]["datasets"]:
        if ds not in avail:
            continue
        for arm in P["exps"][ex_name]["datasets"][ds]["arms"]:
            line = []
            for d in doses:
                a = agg(lambda r: r["key"] == keys[0] and r["exp"] == ex_name and r["ds"] == ds and r["dose"] == d, "logreg", arm)
                b = agg(lambda r: r["key"] == keys[0] and r["exp"] == ex_name and r["ds"] == ds and r["dose"] == d, "1nn", arm)
                line.append(f"{100 * (a[0] - a[1]):+5.1f}|{100 * (b[0] - b[1]):+5.1f}" if a and b else "  —  ")
            print(f"  {ex_name:<6}{ds:<12}{arm:<3} " + "  ".join(f"{str(d):>5}:{x}" for d, x in zip(doses, line)))
print("  cell = logistic-regression inflation | 1-NN inflation")

# ------------------------------------------------------------------ 4  mechanism: identity probe
Q.sub("5  IDENTITY PROBE  (can frozen features tell which slide / patient / source an image came from?)")
from sklearn.model_selection import StratifiedKFold, cross_val_predict
mech = {}
rows = []
for ds in [d for d in ("he_stained", "breakhis", "ndb_ufes") if d in avail]:
    dd = Q.DS.get(ds)
    r = np.nonzero(dd.base)[0]
    g = dd.g_struct[r]
    _, gi = np.unique(g, return_inverse=True)
    X = F[(keys[0], ds, None)][r]
    ok_ = np.bincount(gi)[gi] >= 5
    r, gi, X = r[ok_], np.unique(gi[ok_], return_inverse=True)[1], X[ok_]
    if len(np.unique(gi)) < 2:
        continue
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    pred = cross_val_predict(LogisticRegression(max_iter=400, C=1.0), StandardScaler().fit_transform(X), gi, cv=cv)
    acc = float(np.mean(pred == gi))
    chance = float(np.bincount(gi).max() / len(gi))
    same_cls = []
    for c in np.unique(dd.y[r]):
        m = dd.y[r] == c
        if len(np.unique(gi[m])) > 1 and np.bincount(gi[m]).min() >= 2:
            same_cls.append(float(np.mean(pred[m] == gi[m])))
    mech[f"identity_{ds}"] = dict(groups=int(len(np.unique(gi))), acc=acc, majority=chance, n=int(len(gi)))
    rows.append([ds, Q.META[ds]["group"], len(np.unique(gi)), f"{100 * acc:.1f}", f"{100 * chance:.1f}", len(gi)])
Q.table(["dataset", "group", "groups", "identity acc", "majority", "images"], rows, [12, 26, 8, 14, 10, 8])

# ------------------------------------------------------------------ 5  mechanism: colour-histogram shortcut
Q.sub("6  COLOUR-HISTOGRAM SHORTCUT  (48-bin RGB histogram + channel moments, logistic regression, E1 splits)")


def colour_feats(ds, rows):
    st = Q.open_store(ds)
    out = []
    for s in range(0, len(rows), 256):
        x = np.asarray(st[np.sort(rows[s:s + 256])])[:, ::4, ::4].reshape(-1, (Q.STORE_PX // 4) ** 2, 3).astype(np.float32)
        h = np.stack([np.stack([np.histogram(im[:, c], bins=16, range=(0, 256))[0] for c in range(3)]).ravel() for im in x])
        h = h / h.sum(1, keepdims=True)
        mom = np.concatenate([x.mean(1), x.std(1)], 1) / 255.0
        f = np.concatenate([h, mom], 1)
        order = np.argsort(np.argsort(rows[s:s + 256]))
        out.append(f[order])
    return np.concatenate(out)


rows = []
for ds in [d for d in ("he_stained", "breakhis", "ndb_ufes") if d in avail]:
    dd = Q.DS.get(ds)
    CF = np.zeros((dd.n, 54), np.float32)
    okr = np.nonzero(dd.ok)[0]
    CF[okr] = colour_feats(ds, okr)
    val = {}
    for arm in ("L", "C"):
        o_, h_ = [], []
        for rid, (s, B) in builds.items():
            if s["exp"] != "E1" or s["ds"] != ds or s["arm"] != arm or s["seed"] != 0:
                continue
            tr, own, hold = B["train"], B["own"], B["hold"]
            if len(np.unique(tr["y"])) < 2 or len(own) == 0:
                continue
            sc = StandardScaler().fit(CF[tr["row"]])
            m = LogisticRegression(max_iter=400).fit(sc.transform(CF[tr["row"]]), tr["y"])
            o_.append(ba(own["y"], m.predict(sc.transform(CF[own["row"]])), dd.K))
            h_.append(ba(hold["y"], m.predict(sc.transform(CF[hold["row"]])), dd.K))
        if o_:
            val[arm] = (float(np.mean(o_)), float(np.mean(h_)))
    if "L" in val and "C" in val:
        mech[f"colour_{ds}"] = val
        rows.append([ds, f"{100 * val['L'][0]:.1f} / {100 * val['L'][1]:.1f}", f"{100 * (val['L'][0] - val['L'][1]):+.1f}",
                     f"{100 * val['C'][0]:.1f} / {100 * val['C'][1]:.1f}", f"{100 * (val['C'][0] - val['C'][1]):+.1f}"])
Q.table(["dataset", "L own/hold BA", "L infl.", "C own/hold BA", "C infl."], rows, [12, 17, 9, 17, 9])

# ------------------------------------------------------------------ 6  mechanism: nearest-neighbour similarity
Q.sub("7  NEAREST-TRAINING-IMAGE SIMILARITY  (mean max cosine of evaluation images to the training set, E1 seed 0)")
rows = []
for ds in avail:
    L = [r for r in res if r["key"] == keys[0] and r["exp"] == "E1" and r["ds"] == ds and r["arm"] == "L" and r["seed"] == 0 and "nnsim" in r]
    C = [r for r in res if r["key"] == keys[0] and r["exp"] == "E1" and r["ds"] == ds and r["arm"] == "C" and r["seed"] == 0 and "nnsim" in r]
    if L and C:
        f = lambda v, k: float(np.mean([x["nnsim"][k] for x in v]))
        rows.append([ds, f"{f(L, 'own'):.3f}", f"{f(L, 'hold'):.3f}", f"{f(C, 'own'):.3f}", f"{f(C, 'hold'):.3f}"])
        mech[f"nnsim_{ds}"] = dict(L_own=f(L, "own"), L_hold=f(L, "hold"), C_own=f(C, "own"), C_hold=f(C, "hold"))
Q.table(["dataset", "L own", "L hold", "C own", "C hold"], rows, [12, 10, 10, 10, 10])
print("  higher similarity of the own fold than of the holdout means evaluation images have near copies in training.")
Q.jsave(Q.D["eval"] / "mechanism.json", mech)
print(f"\n[BLOCK 17 ✓]  {len(res)} probe fits · results in q1/eval/probes.json and mechanism.json · {Q.hms(time.time() - t_all)}"
      " · run BLOCK 18 next")

In [ ]:
# ============================================================================
# BLOCK 18 — JOB PLANNER: every split built and tested for disjointness before any GPU time
# ============================================================================
import os, sys, json, time, importlib, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("18")

t0 = time.time()
Q.hdr("BLOCK 18 — JOB PLANNER")
P = Q.plan()
jobs, bad, moved = Q.make_jobs()
Q.kv("jobs", f"{len(jobs)} built in {time.time() - t0:.0f}s · every split passed the disjointness tests"
     if not bad else f"{len(jobs)} built · {len(bad)} REJECTED")
for jid, msg in bad[:20]:
    print(f"    ✗ {jid}: {msg}")
if moved:
    Q.kv("superseded", f"{moved} finished runs no longer match their job definition (moved to runs_superseded/)")

Q.sub("SCHEDULE BY PHASE  (GPU time estimated from default throughput; recalibrated after the first runs)")
rows = []
tot = 0.0
for ph in P["phases"]:
    J = [j for j in jobs if j["phase"] == ph]
    if not J:
        continue
    done = sum(Q.job_done(j) for j in J)
    left = sum(j["est_min"] for j in J if not Q.job_done(j)) / 60
    tot += left
    rows.append([ph, len(J), done, f"{left:.1f}", f"{left / 2:.1f}", Q.PHASE_DESC[ph]])
Q.table(["phase", "jobs", "done", "GPU-h left", "wall h (2 GPU)", "content"], rows, [10, 7, 7, 12, 15, 50])
Q.kv("total", f"{tot:.1f} GPU-h left -> about {tot / 2:.1f} h on 2 GPUs")

Q.sub("CORE PHASE BY DATASET")
rows = []
for ds in Q.DATASETS:
    J = [j for j in jobs if j["phase"] == "core" and j["spec"]["ds"] == ds]
    if not J:
        continue
    ex = P["exps"]["E1"]["datasets"][ds]
    sz = defaultdict(list)
    for j in J:
        sz[j["spec"]["arm"]].append(j["sizes"])
    f = lambda a, k: int(np.mean([s[k] for s in sz[a]])) if sz.get(a) else 0
    rows.append([ds, len(J), f"{f('L', 'train')}/{f('C', 'train')}", f"{f('L', 'own')}/{f('C', 'own')}", f(("C"), "hold"),
                 f"{sum(j['est_min'] for j in J) / 60:.1f}", ex.get("published", "")[:44]])
Q.table(["dataset", "jobs", "train L/C", "own L/C", "holdout", "GPU-h", "leaky arm replicates"], rows,
        [12, 6, 12, 11, 9, 7, 46])

Q.sub("LEAKAGE OF THE OWN EVALUATION FOLD  (share of own-fold images with a relative in training, E1 seed 0)")
rows = []
for ds in Q.DATASETS:
    for arm in ("L", "C"):
        J = [j for j in jobs if j["phase"] == "core" and j["spec"]["ds"] == ds and j["spec"]["arm"] == arm]
        if not J:
            continue
        g = lambda k: np.mean([j["leak"].get(k, 0) for j in J])
        rows.append([ds, arm, f"{100 * g('same_original'):.1f}", f"{100 * g('identical_file'):.1f}",
                     f"{100 * g('near_dup'):.1f}", f"{100 * g('struct_group'):.1f}", f"{100 * g('ctrl_group'):.1f}"])
Q.table(["dataset", "arm", "same original %", "identical file %", "near dup %", "same slide/patient/source %", "any %"],
        rows, [12, 5, 16, 17, 12, 28, 8])
print("  L = the published (leaky) protocol · C = leakage-controlled · the C row must read 0.0 everywhere.")
print("  Sizes of L and C training sets are matched per class, and both use the same step budget.")
print(f"\n[BLOCK 18 ✓]  {len(jobs)} jobs in q1/jobs/jobs.json · run BLOCK 19, then the training blocks 20-24")

In [ ]:
# ============================================================================
# BLOCK 20 — TRAINING · PHASE "CORE": E1 · primary backbone · seed 0 · all datasets (the main result)
# (needs BLOCK 19 in this session; re-run to resume; press Stop to pause safely)
# ============================================================================
RUN("core")

In [ ]:
# ============================================================================
# BLOCK 21 — TRAINING · PHASE "CONTROLS": random-label · duplicate injection · group overlap · pre-augmented corpus
# (needs BLOCK 19 in this session; re-run to resume; press Stop to pause safely)
# ============================================================================
RUN("controls")

In [ ]:
# ============================================================================
# BLOCK 22 — TRAINING · PHASE "DOSE": augmentation-factor dose-response (k = 1, 2, 5, 10, 20)
# (needs BLOCK 19 in this session; re-run to resume; press Stop to pause safely)
# ============================================================================
RUN("dose")

In [ ]:
# ============================================================================
# BLOCK 23 — TRAINING · PHASE "SEEDS": E1 seeds 1 and 2 (seed treated as a random effect)
# (needs BLOCK 19 in this session; re-run to resume; press Stop to pause safely)
# ============================================================================
RUN("seeds")

In [ ]:
# ============================================================================
# BLOCK 24 — TRAINING · PHASE "GENERAL": ResNet-50 · ViT-S · EffNet-B0 · ConvNeXt-B · factorial extras · EfficientNet-B3 reimplementations
# (needs BLOCK 19 in this session; re-run to resume; press Stop to pause safely)
# ============================================================================
RUN("general")

In [ ]:
# ============================================================================
# BLOCK 25 — EVALUATION + STATISTICS (all numbers for the rewritten Results section)
# ============================================================================
import os, sys, json, time, math, importlib, warnings, csv
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("25")

NB = 10000          # bootstrap resamples (pre-specified)
t_all = time.time()
P = Q.plan()
BB0 = P["primary_backbone"]
rng = np.random.default_rng(20260930)
Q.hdr("BLOCK 25 — EVALUATION + STATISTICS")
jobs = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
RUNS = {}
for j in jobs:
    if Q.job_done(j):
        r, z = Q.load_run(j["id"])
        if r is not None:
            RUNS[j["id"]] = (j, r, {k: z[k] for k in z.files})
Q.kv("finished runs", f"{len(RUNS)} of {len(jobs)} jobs")
if not RUNS:
    raise SystemExit("no finished runs yet")
OUT = dict(meta=dict(n_runs=len(RUNS), n_jobs=len(jobs), boot=NB, plan_sha=(Q.jload(Q.D["plan"] / "plan.sha256.json") or {}).get("sha256")))
TAB = Q.D["tables"]


def write_csv(name, header, rows):
    with open(TAB / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


def sel(**kw):
    out = []
    for j, r, z in RUNS.values():
        s = j["spec"]
        if all(s.get(k) == v for k, v in kw.items()):
            out.append((j, r, z))
    return out


def correct(z, name):
    return (z[f"{name}_logit"].argmax(1) == z[f"{name}_y"]).astype(float)


def run_metrics(j, r, z):
    """Per-run metrics with the temperature and thresholds fitted on the inner validation split only."""
    K = r["info"]["K"]
    pos = Q.META[j["spec"]["ds"]]["pos"]
    T = Q.fit_temperature(z["ival_logit"], z["ival_y"])
    out = dict(T=T)
    for nm in ("own", "hold"):
        lg, y = z[f"{nm}_logit"], z[f"{nm}_y"]
        if len(y) == 0:
            continue
        out[nm] = Q.metrics(Q.softmax(lg, T), y, K, pos)
        out[nm + "_raw_ece"] = Q.ece_score(Q.softmax(lg), y)
        out[nm + "_raw_brier"] = float(np.mean(((Q.softmax(lg) - np.eye(K)[y]) ** 2).sum(1)))
    m = z["own_copy"] == 0
    if m.any() and (~m).any():
        out["own_orig"] = Q.metrics(Q.softmax(z["own_logit"][m], T), z["own_y"][m], K, pos)
    pi = Q.softmax(z["ival_logit"], T)[:, pos]
    yi = (z["ival_y"] == pos).astype(int)
    th = Q.thresholds_from(pi, yi)
    ph = Q.softmax(z["hold_logit"], T)[:, pos]
    yh = (z["hold_y"] == pos).astype(int)
    out["op"] = {k: Q.op_point(ph, yh, v) for k, v in th.items()}
    return out


RM = {rid: run_metrics(j, r, z) for rid, (j, r, z) in RUNS.items()}


def fold_key(s):
    return (s["outer"], s.get("rep", 0))


def tables_for(runs, part):
    """Per outer fold: cluster count tables summed over seeds (and the list of seeds)."""
    by = defaultdict(list)
    for j, r, z in runs:
        by[fold_key(j["spec"])].append((j, r, z))
    out = {}
    for fk, lst in by.items():
        ys, cs, ks = [], [], []
        for j, r, z in lst:
            ys.append(z[f"{part}_y"])
            cs.append(correct(z, part))
            ks.append(z[f"{part}_key"])
        K = lst[0][1]["info"]["K"]
        out[fk] = Q.cluster_table(np.concatenate(ys), np.concatenate(cs), np.concatenate(ks), K)
    return out


def paired_hold(runsA, runsB):
    """Per fold: shared cluster table n and correct counts of A and B on the same holdout images."""
    A, Bm = defaultdict(list), defaultdict(list)
    for j, r, z in runsA:
        A[fold_key(j["spec"])].append(z)
    for j, r, z in runsB:
        Bm[fold_key(j["spec"])].append(z)
    out = {}
    for fk in set(A) & set(Bm):
        za = {int(z0["hold_row"][0]) if len(z0["hold_row"]) else 0: z0 for z0 in A[fk]}
        y = A[fk][0]["hold_y"]
        key = A[fk][0]["hold_key"]
        K = int(max(y.max(), 1) + 1) if len(y) else 2
        ca = np.mean([correct(z, "hold") for z in A[fk]], 0)
        cb = np.mean([correct(z, "hold") for z in Bm[fk]], 0)
        n, c1 = Q.cluster_table(y, ca, key, K)
        _, c2 = Q.cluster_table(y, cb, key, K)
        out[fk] = (n, c1, c2)
    return out


def boot_folds(tabs, B, seed):
    """Within-fold cluster bootstrap: returns fold keys and matrix [F, B] of BA."""
    g = np.random.default_rng(seed)
    ks = sorted(tabs)
    return ks, np.stack([Q.boot_ba(tabs[k][0], tabs[k][1], B, g) for k in ks])


def point(tabs):
    return {k: float(Q.ba_counts(*tabs[k])) for k in tabs}


def two_stage(mat, B, seed):
    g = np.random.default_rng(seed)
    F = mat.shape[0]
    idx = g.integers(0, F, size=(B, F))
    return mat[idx, np.arange(B)[:, None]].mean(1)


def fmt(v, ci_=None, d=2):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    s = f"{100 * v:+.{d}f}" if ci_ is not None else f"{100 * v:.{d}f}"
    if ci_ is not None:
        s += f" [{100 * ci_[0]:+.{d}f}, {100 * ci_[1]:+.{d}f}]"
    return s


def pf(p):
    return "—" if p is None or not np.isfinite(p) else (f"{p:.1e}" if p < 1e-3 else f"{p:.4f}")


# ============================================================================ 1  E1 main experiment
Q.sub("1  E1 — OWN-FOLD vs CLEAN-HOLDOUT BALANCED ACCURACY  (primary backbone, all seeds, pooled over outer folds)")
E1 = {}
rows, csvrows = [], []
avail = [d for d in Q.DATASETS if sel(exp="E1", ds=d, backbone=BB0, recipe="std")]
for ds in avail:
    arms = P["exps"]["E1"]["datasets"][ds]["arms"]
    res = {}
    for arm in arms:
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std")
        if not R_:
            continue
        to, th = tables_for(R_, "own"), tables_for(R_, "hold")
        common = sorted(set(to) & set(th))
        to, th = {k: to[k] for k in common}, {k: th[k] for k in common}
        ko, mo = boot_folds(to, NB, Q.shash(ds, arm, "own") % 2 ** 32)
        kh, mh = boot_folds(th, NB, Q.shash(ds, arm, "hold") % 2 ** 32)
        po, ph_ = point(to), point(th)
        infl_pt = float(np.mean([po[k] - ph_[k] for k in common]))
        infl_w = (mo - mh).mean(0)                                        # within-fold (sensitivity)
        infl_b = Q.hier([mo - mh], NB, Q.shash(ds, arm, "hier") % 2 ** 32)[0]   # two-stage (primary)
        seeds = sorted({j["spec"]["seed"] for j, _, _ in R_})
        per_seed = []
        for sd in seeds:
            Rs = [x for x in R_ if x[0]["spec"]["seed"] == sd]
            a, b = point(tables_for(Rs, "own")), point(tables_for(Rs, "hold"))
            kk = set(a) & set(b)
            if kk:
                per_seed.append(float(np.mean([a[k] - b[k] for k in kk])))
        vals, outs, sds = [], [], []
        for j, r, z in R_:
            m = RM[j["id"]]
            if "own" in m and "hold" in m:
                vals.append(m["own"]["bal_acc"] - m["hold"]["bal_acc"])
                outs.append(f"{j['spec']['outer']}-{j['spec'].get('rep', 0)}")
                sds.append(j["spec"]["seed"])
        mix = Q.mixed_intercept(vals, outs, sds) if len(vals) >= 4 else None
        mean_m = lambda part, key, sub=None: float(np.nanmean([(RM[j["id"]][part][sub][key] if sub else RM[j["id"]][part][key])
                                                               for j, _, _ in R_ if part in RM[j["id"]] and (sub is None or sub in RM[j["id"]][part])
                                                               and (RM[j["id"]][part][sub] if sub else RM[j["id"]][part]).get(key) is not None])) \
            if R_ else float("nan")
        res[arm] = dict(n_runs=len(R_), folds=len(common), seeds=seeds, own=float(np.mean(list(po.values()))),
                        hold=float(np.mean(list(ph_.values()))), infl=infl_pt, infl_ci=Q.ci(infl_b), infl_p=Q.boot_p(infl_b),
                        infl_ci_within=Q.ci(infl_w), infl_p_within=Q.boot_p(infl_w), seed_infl=per_seed,
                        seed_sd=float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else None, mixed=mix,
                        own_acc=mean_m("own", "acc"), hold_acc=mean_m("hold", "acc"), own_auroc=mean_m("own", "auroc"),
                        hold_auroc=mean_m("hold", "auroc"), own_f1=mean_m("own", "macro_f1"), hold_f1=mean_m("hold", "macro_f1"),
                        hold_ece=mean_m("hold", "ece"), hold_brier=mean_m("hold", "brier"),
                        own_pos_sens=mean_m("own", "sens", "pos"), own_pos_spec=mean_m("own", "spec", "pos"),
                        own_pos_prec=mean_m("own", "prec", "pos"), own_pos_f1=mean_m("own", "f1", "pos"),
                        hold_pos_sens=mean_m("hold", "sens", "pos"), hold_pos_spec=mean_m("hold", "spec", "pos"),
                        hold_pos_prec=mean_m("hold", "prec", "pos"), hold_pos_f1=mean_m("hold", "f1", "pos"),
                        own_orig=float(np.nanmean([RM[j["id"]]["own_orig"]["bal_acc"] for j, _, _ in R_ if "own_orig" in RM[j["id"]]]))
                        if any("own_orig" in RM[j["id"]] for j, _, _ in R_) else None,
                        _mo=mo, _mh=mh, _k=common)
    if "L" in res and "C" in res:
        L, C = res["L"], res["C"]
        kk = sorted(set(L["_k"]) & set(C["_k"]))
        iL = [L["_k"].index(k) for k in kk]
        iC = [C["_k"].index(k) for k in kk]
        did_b = Q.hier([(L["_mo"] - L["_mh"])[iL] - (C["_mo"] - C["_mh"])[iC]], NB, Q.shash(ds, "did") % 2 ** 32)[0]
        ph_tab = paired_hold(sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std"),
                             sel(exp="E1", ds=ds, arm="C", backbone=BB0, recipe="std"))
        g = np.random.default_rng(Q.shash(ds, "paired") % 2 ** 32)
        dmat = np.stack([Q.boot_ba_paired(ph_tab[k][0], ph_tab[k][1], ph_tab[k][2], NB, g) for k in sorted(ph_tab)])
        dh = Q.hier([dmat], NB, Q.shash(ds, "dh") % 2 ** 32)[0]
        dh_pt = float(np.mean([Q.ba_counts(ph_tab[k][0], ph_tab[k][1]) - Q.ba_counts(ph_tab[k][0], ph_tab[k][2]) for k in ph_tab]))
        ca, cb, cl = [], [], []
        for fk in sorted(ph_tab):
            za = [z for j, r, z in sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std") if fold_key(j["spec"]) == fk]
            zb = [z for j, r, z in sel(exp="E1", ds=ds, arm="C", backbone=BB0, recipe="std") if fold_key(j["spec"]) == fk]
            for x, y_ in zip(za, zb):
                ca.append(correct(x, "hold"))
                cb.append(correct(y_, "hold"))
                cl.append(x["hold_key"] + 10 ** 7 * fk[1])
        gee = Q.gee_paired(np.concatenate(ca), np.concatenate(cb), np.concatenate(cl)) if ca else {}
        E1[ds] = dict(arms={a: {k: v for k, v in d.items() if not k.startswith("_")} for a, d in res.items()},
                      did=float(L["infl"] - C["infl"]), did_ci=Q.ci(did_b), did_p=Q.boot_p(did_b),
                      dhold=dh_pt, dhold_ci=Q.ci(dh), dhold_p=Q.boot_p(dh), gee=gee)
    else:
        E1[ds] = dict(arms={a: {k: v for k, v in d.items() if not k.startswith("_")} for a, d in res.items()})
    for arm, d in res.items():
        rows.append([ds, arm, f"{d['n_runs']}/{d['folds']}", fmt(d["own"]), fmt(d["hold"]),
                     fmt(d["infl"], d["infl_ci"]), pf(d["infl_p"]),
                     "—" if d["seed_sd"] is None else f"{100 * d['seed_sd']:.2f}"])
        csvrows.append([ds, arm, d["n_runs"], d["folds"], d["own"], d["hold"], d["infl"], *d["infl_ci"], d["infl_p"],
                        *d["infl_ci_within"], d["seed_sd"], d["own_acc"], d["hold_acc"], d["own_auroc"], d["hold_auroc"],
                        d["own_f1"], d["hold_f1"], d["hold_ece"], d["hold_brier"], d["own_pos_sens"], d["own_pos_spec"],
                        d["own_pos_prec"], d["own_pos_f1"], d["hold_pos_sens"], d["hold_pos_spec"], d["hold_pos_prec"],
                        d["hold_pos_f1"], d["own_orig"]])
Q.table(["dataset", "arm", "runs/folds", "own BA", "hold BA", "inflation [95% CI, two-stage]", "p", "seed SD"], rows,
        [12, 6, 11, 9, 9, 30, 10, 9])
write_csv("E1_main", ["dataset", "arm", "runs", "folds", "own_ba", "hold_ba", "inflation", "ci_lo_2stage", "ci_hi_2stage",
                      "p_2stage", "ci_lo_within", "ci_hi_within", "seed_sd", "own_acc", "hold_acc", "own_auroc", "hold_auroc", "own_macro_f1", "hold_macro_f1",
                      "hold_ece_cal", "hold_brier_cal", "own_pos_sens", "own_pos_spec", "own_pos_prec", "own_pos_f1",
                      "hold_pos_sens", "hold_pos_spec", "hold_pos_prec", "hold_pos_f1", "own_ba_originals_only"], csvrows)

# primary family: H1 with Holm
fam = [(ds, E1[ds]["arms"]["L"]["infl_p"]) for ds in avail if "L" in E1[ds]["arms"]]
if fam:
    adj = Q.holm([p for _, p in fam])
    Q.sub("2  PRIMARY FAMILY (H1): leaky-arm inflation > 0, Holm-adjusted across datasets")
    rows = []
    for (ds, p), pa in zip(fam, adj):
        a = E1[ds]["arms"]["L"]
        E1[ds]["arms"]["L"]["infl_p_holm"] = pa
        mx = a.get("mixed") or {}
        mixs = (f"{100 * mx['est']:+.2f} [{100 * mx['lo']:+.2f},{100 * mx['hi']:+.2f}]" + ("*" if mx.get("note") else "")
                if mx.get("lo") is not None else "—")
        rows.append([ds, fmt(a["infl"], a["infl_ci"]), fmt(a["infl"], a["infl_ci_within"]), mixs, pf(p), pf(pa),
                     "supported" if pa < 0.05 and a["infl"] > 0 else "not supported"])
    Q.table(["dataset", "inflation L [two-stage CI]", "[within-fold CI]", "mixed model [CI]", "p", "p Holm", "H1"], rows,
            [12, 28, 28, 24, 9, 9, 14])
    print("  p and Holm use the two-stage bootstrap (primary). * = mixed model not estimable (one run per fold, singular fit")
    print("  or no convergence), so a t-interval over fold-level means is shown instead.")

Q.sub("3  SECONDARY (H2-H4): difference-in-differences, controlled-arm bias, holdout L - C")
rows = []
for ds in avail:
    e = E1[ds]
    if "did" not in e:
        continue
    C = e["arms"]["C"]
    gee = e.get("gee", {})
    rows.append([ds, fmt(e["did"], e["did_ci"]), fmt(C["infl"], C["infl_ci"]), fmt(e["dhold"], e["dhold_ci"]), pf(e["dhold_p"]),
                 f"{gee.get('or_', float('nan')):.2f} [{gee.get('lo', float('nan')):.2f},{gee.get('hi', float('nan')):.2f}] p {pf(gee.get('p'))}"
                 if "or_" in gee else "—"])
Q.table(["dataset", "DiD (H2) [CI]", "bias of C (H3) [CI]", "hold L - C (H4) [CI]", "p", "GEE odds ratio L vs C"], rows,
        [12, 28, 28, 28, 9, 30])
dh_p = [(ds, E1[ds]["dhold_p"]) for ds in avail if "dhold_p" in E1[ds]]
if dh_p:
    for (ds, _), pa in zip(dh_p, Q.holm([p for _, p in dh_p])):
        E1[ds]["dhold_p_holm"] = pa

Q.sub("4  WHAT A PAPER WOULD REPORT (own fold, leaky arm) vs THE TRUTH (clean holdout) — full metric panel")
rows = []
for ds in avail:
    for arm in ("L", "C"):
        a = E1[ds]["arms"].get(arm)
        if not a:
            continue
        rows.append([ds, arm, fmt(a["own_acc"]), fmt(a["hold_acc"]), f"{a['own_auroc']:.3f}/{a['hold_auroc']:.3f}",
                     f"{fmt(a['own_pos_sens'])}/{fmt(a['hold_pos_sens'])}", f"{fmt(a['own_pos_spec'])}/{fmt(a['hold_pos_spec'])}",
                     f"{fmt(a['own_pos_f1'])}/{fmt(a['hold_pos_f1'])}", f"{a['hold_ece']:.3f}"])
Q.table(["dataset", "arm", "own acc", "hold acc", "AUROC own/hold", "pos sens own/hold", "pos spec own/hold",
         "pos F1 own/hold", "ECE hold"], rows, [12, 5, 9, 9, 16, 18, 18, 16, 9])
print("  'pos' = positive class one-vs-rest (tumour / OSCC / cancer / malignant); macro metrics are in tables/E1_main.csv.")
print("  temperature and ECE: temperature fitted on each run's inner-validation split, applied to the holdout.")

# ICC / design effect and OCDC per slide
Q.sub("5  CLUSTERING OF ERRORS  (ICC of holdout correctness within groups, L arm, seed 0)")
rows = []
for ds in avail:
    R_ = sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std", seed=0)
    if not R_:
        continue
    c = np.concatenate([correct(z, "hold") for _, _, z in R_])
    k = np.concatenate([z["hold_key"] + 10 ** 7 * j["spec"].get("rep", 0) for j, _, z in R_])
    ic = Q.icc1(c, k)
    E1[ds]["icc"] = ic
    rows.append([ds, f"{ic['icc']:.3f}", f"{ic['deff']:.2f}", ic["clusters"], f"{ic['mean_size']:.1f}"])
Q.table(["dataset", "ICC", "design effect", "clusters", "mean size"], rows, [12, 8, 15, 10, 10])
print("  a design effect > 1 means image-level tests (e.g. McNemar) overstate the evidence by that factor in variance.")

if "he_stained" in avail:
    Q.sub("6  OCDC LEAVE-ONE-SLIDE-OUT: per held-out slide (seed-averaged) + exact slide-level sign-flip tests")
    rows, dL, dH = [], [], []
    for o in sorted({j["spec"]["outer"] for j, _, _ in sel(exp="E1", ds="he_stained", backbone=BB0, recipe="std")}):
        v = {}
        for arm in ("L", "C"):
            R_ = sel(exp="E1", ds="he_stained", arm=arm, outer=o, backbone=BB0, recipe="std")
            if R_:
                v[arm] = (np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_]),
                          np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_]))
        if len(v) == 2:
            dd = Q.DS.get("he_stained")
            slide = sorted(set(dd.slide[dd.base]))[o]
            dL.append(v["L"][0] - v["L"][1])
            dH.append(v["L"][1] - v["C"][1])
            rows.append([slide, fmt(v["L"][0]), fmt(v["L"][1]), fmt(v["C"][0]), fmt(v["C"][1]), fmt(v["L"][0] - v["L"][1])])
    Q.table(["held-out slide", "L own", "L hold", "C own", "C hold", "L inflation"], rows, [18, 9, 9, 9, 9, 12])
    if dL:
        pL, pH = Q.sign_flip_p(dL), Q.sign_flip_p(dH)
        E1["he_stained"]["slide_level"] = dict(inflation=dL, dhold=dH, p_inflation=pL, p_dhold=pH)
        Q.kv("sign-flip p (L inflation)", f"{pf(pL)}  over {len(dL)} slides (smallest possible two-sided p = {2 / 2 ** len(dL):.4f})")
        Q.kv("sign-flip p (hold L - C)", pf(pH))
OUT["E1"] = E1

# factorial (clinical) and grouping sensitivity (NDB)
Q.sub("7  FACTORIAL (clinical) AND GROUPING SENSITIVITY (NDB-UFES coarse blocks), seed 0")
rows = []
for ds in avail:
    ex = P["exps"]["E1"]["datasets"][ds]
    for arm in ex.get("extra", []):
        a = E1[ds]["arms"].get(arm)
        if a:
            f_ = ex["arms"][arm]
            rows.append([ds, arm, f"S={f_.get('S', 0)} Dup={f_.get('Dup', 0)} Grp={f_.get('Grp', 0)} coarse={f_.get('coarse', 0)}",
                         fmt(a["own"]), fmt(a["hold"]), fmt(a["infl"], a["infl_ci"])])
if rows:
    Q.table(["dataset", "arm", "factors", "own BA", "hold BA", "inflation [CI]"], rows, [12, 6, 30, 9, 9, 30])
else:
    print("  no finished runs yet — these arms are trained in the 'general' phase (BLOCK 24)")
if "clinical" in avail and all(a in E1["clinical"]["arms"] for a in ("L", "C", "S0D1", "S1D0")):
    A = E1["clinical"]["arms"]
    fx = dict(S=0.5 * ((A["L"]["infl"] - A["S0D1"]["infl"]) + (A["S1D0"]["infl"] - A["C"]["infl"])),
              Dup=0.5 * ((A["L"]["infl"] - A["S1D0"]["infl"]) + (A["S0D1"]["infl"] - A["C"]["infl"])),
              SxDup=(A["L"]["infl"] - A["S0D1"]["infl"]) - (A["S1D0"]["infl"] - A["C"]["infl"]))
    OUT["factorial_clinical"] = fx
    Q.kv("main effect of augment-before-split", fmt(fx["S"]) + " points of inflation (clinical, seed 0)")
    Q.kv("main effect of keeping duplicates", fmt(fx["Dup"]) + " points")
    Q.kv("interaction S x Dup", fmt(fx["SxDup"]) + " points (exploratory, no CI: one seed per cell)")

# ============================================================================ 8  published protocol comparison
Q.sub("8  L ARM = PUBLISHED PROTOCOL: our replication numbers next to the published ones (definitions stated)")
PUB = {"histopath": [("Abd El-Aziz 2025", dict(acc=99.84, spec=99.92, prec=97.94, sens=97.93, f1=97.93), "not stated")],
       "ndb_ufes": [("SciRep 2025", dict(acc=97.16, prec=97.16, sens=97.16, f1=97.16, spec=98.58), "multi-class, averaging not stated")],
       "clinical": [("Liu & Bagi 2025", dict(acc=99.54, sens=95.73, spec=96.21, prec=96.34, f1=96.03, auroc=60.26), "not stated"),
                    ("Alzahrani 2025", dict(acc=98.47, sens=97.73, spec=97.73, f1=98.27), "not stated")]}
rows = []
for ds, lst in PUB.items():
    a = E1.get(ds, {}).get("arms", {}).get("L")
    c = E1.get(ds, {}).get("arms", {}).get("C")
    if not a:
        continue
    for name, vals, dfn in lst:
        ours = dict(acc=a["own_acc"], sens=a["own_pos_sens"], spec=a["own_pos_spec"], prec=a["own_pos_prec"],
                    f1=a["own_pos_f1"], auroc=a["own_auroc"])
        truth = dict(acc=c["hold_acc"], sens=c["hold_pos_sens"], spec=c["hold_pos_spec"], prec=c["hold_pos_prec"],
                     f1=c["hold_pos_f1"], auroc=c["hold_auroc"]) if c else {}
        for k, v in vals.items():
            rows.append([ds, name, k, f"{v:.2f}", fmt(ours.get(k)), fmt(truth.get(k)), dfn])
Q.table(["dataset", "paper", "metric", "published", "ours, L own fold", "ours, clean holdout", "published definition"],
        rows, [12, 18, 7, 11, 17, 20, 30])
print("  ours = positive-class (one-vs-rest) values; for NDB-UFES the positive class is OSCC. AUROC shown x100.")
write_csv("published_vs_ours", ["dataset", "paper", "metric", "published", "ours_L_own", "ours_clean_holdout", "definition"], rows)

# ============================================================================ 9  PRE, RL
_nh = Q.DS.get("histopath").n if "histopath" in avail else 0
for ex_name, title in (("PRE", f"9  PRE-AUGMENTED CORPUS ({_nh:,} files as distributed): L = random file split, C = grouped by recovered parent"),
                       ("RL", "10 RANDOM-LABEL CONTROL (H6): labels drawn per group; chance = 1/K")):
    Rx = sel(exp=ex_name, backbone=BB0, recipe="std")
    if not Rx:
        continue
    Q.sub(title)
    rows = []
    blk = {}
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for arm in ("L", "C"):
            R_ = [x for x in Rx if x[0]["spec"]["ds"] == ds and x[0]["spec"]["arm"] == arm]
            if not R_:
                continue
            to, th = tables_for(R_, "own"), tables_for(R_, "hold")
            _, mo = boot_folds(to, NB, 1)
            _, mh = boot_folds(th, NB, 2)
            own_b = mo.mean(0)
            d = dict(own=float(np.mean(list(point(to).values()))), hold=float(np.mean(list(point(th).values()))),
                     own_ci=Q.ci(own_b), infl_ci=Q.ci((mo - mh).mean(0)), n=len(R_))
            chance = 1.0 / Q.META[ds]["K"]
            d["p_vs_chance"] = Q.boot_p(own_b, chance)
            blk[f"{ds}|{arm}"] = d
            rows.append([ds, arm, len(R_), f"{fmt(d['own'])} [{fmt(d['own_ci'][0])},{fmt(d['own_ci'][1])}]", fmt(d["hold"]),
                         f"{100 * chance:.1f}", pf(d["p_vs_chance"]) if ex_name == "RL" else fmt(d["own"] - d["hold"], d["infl_ci"])])
    Q.table(["dataset", "arm", "runs", "own BA [CI]", "hold BA", "chance", "p vs chance" if ex_name == "RL" else "inflation [CI]"],
            rows, [12, 5, 6, 26, 9, 8, 30])
    OUT[ex_name] = blk

# ============================================================================ 11 dose-response
Q.sub("11 DOSE-RESPONSE (H5): inflation vs augmentation factor k, injected duplicates p, group overlap q")
DOSE = {}
rows = []
for ex_name, xform, lab in (("DOSEK", lambda d: math.log2(d), "log2 k"), ("DUP", lambda d: d, "p"), ("OVL", lambda d: d, "q")):
    Rx = sel(exp=ex_name, backbone=BB0, recipe="std")
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for arm in sorted({j["spec"]["arm"] for j, _, _ in Rx if j["spec"]["ds"] == ds}):
            pts = []
            cells = []
            for dose in P["exps"][ex_name]["doses"]:
                R_ = [x for x in Rx if x[0]["spec"]["ds"] == ds and x[0]["spec"]["arm"] == arm and x[0]["spec"]["dose"] == dose]
                if not R_:
                    cells.append("  —  ")
                    continue
                inf = [RM[j["id"]]["own"]["bal_acc"] - RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_]
                for (j, _, _), v in zip(R_, inf):
                    pts.append((xform(dose), v, j["spec"]["outer"]))
                cells.append(f"{100 * np.mean(inf):+5.1f}")
            if len(pts) >= 4:
                x, yv, g = map(np.array, zip(*pts))
                try:
                    import pandas as pd
                    import statsmodels.formula.api as smf
                    with Q.quiet():
                        mm = smf.mixedlm("y ~ x", pd.DataFrame(dict(x=x, y=yv, g=g)), groups=g).fit(reml=True)
                    slope, se, p = float(mm.params["x"]), float(mm.bse["x"]), float(mm.pvalues["x"])
                except Exception:
                    b = np.polyfit(x, yv, 1)
                    slope, se, p = float(b[0]), float("nan"), float("nan")
                from scipy.stats import spearmanr
                rho = float(spearmanr(x, yv).correlation) if len(set(x)) > 1 else float("nan")
                DOSE[f"{ex_name}|{ds}|{arm}"] = dict(points=pts, slope=slope, se=se, p=p, spearman=rho)
                rows.append([ex_name, ds, arm, "  ".join(cells), f"{100 * slope:+.2f}/{lab}", pf(p), f"{rho:+.2f}"])
Q.table(["exp", "dataset", "arm", "inflation at each dose (points)", "slope", "p (mixed)", "Spearman"], rows,
        [7, 12, 5, 44, 14, 10, 9])
for ex_name in ("DOSEK", "DUP", "OVL"):
    print(f"  {ex_name}: doses {P['exps'][ex_name]['doses']}")
OUT["dose"] = DOSE
Rx = sel(exp="DUP", backbone=BB0, recipe="std")
if Rx:
    rows = []
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for dose in P["exps"]["DUP"]["doses"][1:]:
            inj, non = [], []
            for j, r, z in Rx:
                if j["spec"]["ds"] != ds or j["spec"]["dose"] != dose or "own_injected" not in z:
                    continue
                m = z["own_injected"].astype(bool)
                c = correct(z, "own")
                if m.any():
                    inj.append(c[m].mean())
                if (~m).any():
                    non.append(c[~m].mean())
            if inj:
                rows.append([ds, dose, f"{100 * np.mean(inj):.1f}", f"{100 * np.mean(non):.1f}"])
    Q.table(["dataset", "p", "acc on injected (copy in training)", "acc on the rest"], rows, [12, 6, 36, 16])

# ============================================================================ 12 generality (backbones) + capacity
Q.sub("12 GENERALITY (H7): inflation by backbone family, seed 0, outer folds 0-2 (same splits for every backbone)")
G = P["generality"]
bbs = [BB0] + G["backbones"]
GEN = {}
rows = []
for ds in G["datasets"]:
    for bb in bbs:
        v = {}
        for arm in ("L", "C"):
            R_ = [x for x in sel(exp="E1", ds=ds, arm=arm, backbone=bb, recipe="std", seed=0) if x[0]["spec"]["outer"] in G["outer"]
                  and x[0]["spec"].get("rep", 0) == 0]
            if R_:
                v[arm] = (float(np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_])), len(R_))
        if "L" in v:
            GEN[f"{ds}|{bb}"] = v
            rows.append([ds, Q.BACKBONES[bb][0], f"{Q.BACKBONES[bb][1]:.1f}M", f"{fmt(v['L'][0])}/{fmt(v['L'][1])}",
                         fmt(v["L"][0] - v["L"][1]).replace("+", "+"),
                         f"{fmt(v['C'][0])}/{fmt(v['C'][1])}" if "C" in v else "—",
                         fmt(v["C"][0] - v["C"][1]) if "C" in v else "—", v["L"][2]])
Q.table(["dataset", "backbone", "params", "L own/hold", "L infl.", "C own/hold", "C infl.", "folds"], rows,
        [12, 12, 8, 16, 9, 16, 9, 6])
probes = Q.jload(Q.D["eval"] / "probes.json", []) or []
if probes and GEN:
    from scipy.stats import spearmanr
    rows = []
    for ds in G["datasets"]:
        pts = []
        for bb in bbs:
            v = GEN.get(f"{ds}|{bb}")
            if v:
                pts.append((Q.BACKBONES[bb][1], v["L"][0] - v["L"][1]))
        pr = [p_ for p_ in probes if p_["exp"] == "E1" and p_["ds"] == ds and p_["arm"] == "L" and p_["seed"] == 0
              and p_["outer"] in G["outer"] and "logreg" in p_]
        lin = float(np.mean([p_["logreg"]["own"] - p_["logreg"]["hold"] for p_ in pr])) if pr else float("nan")
        nn1 = float(np.mean([p_["1nn"]["own"] - p_["1nn"]["hold"] for p_ in pr])) if pr else float("nan")
        rho = float(spearmanr(*zip(*pts)).correlation) if len(pts) >= 3 else float("nan")
        rows.append([ds, fmt(lin), fmt(nn1), f"{rho:+.2f}" if np.isfinite(rho) else "— (needs 3 models)", len(pts)])
        GEN[f"{ds}|capacity"] = dict(linear_probe=lin, one_nn=nn1, spearman_params=rho)
    Q.table(["dataset", "linear probe infl.", "1-NN infl.", "Spearman(params, infl.)", "fine-tuned models"], rows,
            [12, 18, 12, 24, 18])
OUT["generality"] = GEN

# ============================================================================ 13 reimplementations
Rx = [x for x in RUNS.values() if x[0]["spec"]["recipe"] != "std"]
Q.sub("13 PUBLISHED PIPELINES REIMPLEMENTED (EfficientNet-B3): same method under the leaky and the controlled protocol")
if not Rx:
    print("  no finished runs yet — the reimplementations are trained in the 'general' phase (BLOCK 24)")
else:
    rows = []
    REI = {}
    for r_ in P["reimp"]:
        v = {}
        for arm in r_["arms"]:
            R_ = [x for x in Rx if x[0]["spec"]["recipe"] == r_["recipe"] and x[0]["spec"]["ds"] == r_["ds"]
                  and x[0]["spec"]["exp"] == r_["exp"] and x[0]["spec"]["arm"] == arm]
            if R_:
                v[arm] = (float(np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["own"]["acc"] for j, _, _ in R_])), len(R_))
        if v:
            REI[f"{r_['paper']}|{r_['exp']}|{r_['ds']}"] = v
            for arm, t in v.items():
                rows.append([r_["paper"][:40], arm, fmt(t[2]), fmt(t[0]), fmt(t[1]), fmt(t[0] - t[1]), t[3]])
    Q.table(["pipeline", "arm", "own acc", "own BA", "hold BA", "inflation", "folds"], rows, [42, 5, 9, 9, 9, 10, 6])
    OUT["reimp"] = REI

# ============================================================================ 14 cross-dataset
Q.sub("14 CROSS-DATASET TRANSFER (E1 seed 0): models trained on one corpus, tested on the other binary H&E corpora")
rows = []
XD = {}
for ds in Q.XEVAL:
    for arm in ("L", "C"):
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std", seed=0)
        for t in Q.XEVAL[ds]:
            au, bal = [], []
            for j, r, z in R_:
                if f"x_{t}_logit" not in z:
                    continue
                p_ = Q.softmax(z[f"x_{t}_logit"])
                score = p_[:, 1] if p_.shape[1] == 2 else p_[:, 2] / (p_[:, 0] + p_[:, 2] + 1e-9)
                y = z[f"x_{t}_y"]
                from sklearn.metrics import roc_auc_score
                if len(np.unique(y)) == 2:
                    au.append(roc_auc_score(y, score))
                    pr = (score >= 0.5).astype(int)
                    bal.append(0.5 * (np.mean(pr[y == 1] == 1) + np.mean(pr[y == 0] == 0)))
            if au:
                XD[f"{ds}|{arm}|{t}"] = dict(auroc=float(np.mean(au)), bal_acc=float(np.mean(bal)), n=len(au))
                rows.append([ds, arm, t, f"{np.mean(au):.3f}", fmt(np.mean(bal)), len(au)])
if rows:
    Q.table(["trained on", "arm", "tested on", "AUROC", "BA (thr 0.5)", "models"], rows, [12, 5, 12, 8, 13, 8])
OUT["cross_dataset"] = XD

# ============================================================================ 15 mechanism: NN similarity vs correctness
Q.sub("15 MECHANISM: similarity to the nearest training image and correctness (E1 seed 0, frozen DINOv2 space)")
audit = Q.jload(Q.D["index"] / "audit.json", {}) or {}
EMB = audit.get("_meta", {}).get("embed", "dinov2_s")
rows = []
MECH = {}
for ds in avail:
    fe = Q.D["store"] / f"{ds}_emb_{EMB}.npy"
    if not fe.exists():
        continue
    E = np.load(fe).astype(np.float32)
    pol = P["exps"]["E1"]["datasets"][ds].get("policy")
    sib = None
    if pol:
        fs = Q.D["store"] / f"{ds}_sib_{pol}_{EMB}.npz"
        if fs.exists():
            zz = np.load(fs)
            pos_ = np.full(len(E), -1)
            pos_[zz["rows"]] = np.arange(len(zz["rows"]))
            sib = (pos_, zz["emb"].astype(np.float32), int(zz["kmax"]))

    def fx(rows_, copies):
        X = E[rows_].copy()
        if sib is not None:
            m = (copies > 0) & (copies <= sib[2])
            X[m] = sib[1][sib[0][rows_[m]], copies[m] - 1]
        return X

    for arm in ("L", "C"):
        sims, cors = defaultdict(list), defaultdict(list)
        for j, r, z in sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std", seed=0):
            B = Q.build_job(j["spec"], check=False)
            Xt = fx(B["train"]["row"], B["train"]["copy"])
            for part in ("own", "hold"):
                Xe = fx(z[f"{part}_row"], z[f"{part}_copy"])
                s_ = np.concatenate([(Xe[i:i + 2048] @ Xt.T).max(1) for i in range(0, len(Xe), 2048)]) if len(Xe) else np.zeros(0)
                sims[part].append(s_)
                cors[part].append(correct(z, part))
        if sims:
            so, sh = np.concatenate(sims["own"]), np.concatenate(sims["hold"])
            co, ch = np.concatenate(cors["own"]), np.concatenate(cors["hold"])
            allc, alls = np.concatenate([co, ch]), np.concatenate([so, sh])
            r_pb = float(np.corrcoef(alls, allc)[0, 1]) if allc.std() > 0 else float("nan")
            MECH[f"{ds}|{arm}"] = dict(sim_own=float(so.mean()), sim_hold=float(sh.mean()), r_sim_correct=r_pb,
                                       acc_top_decile=float(allc[alls >= np.percentile(alls, 90)].mean()),
                                       acc_bottom_decile=float(allc[alls <= np.percentile(alls, 10)].mean()))
            rows.append([ds, arm, f"{so.mean():.3f}", f"{sh.mean():.3f}", f"{r_pb:+.3f}",
                         fmt(MECH[f'{ds}|{arm}']['acc_top_decile']), fmt(MECH[f'{ds}|{arm}']['acc_bottom_decile'])])
Q.table(["dataset", "arm", "sim own", "sim hold", "r(sim, correct)", "acc top-10% sim", "acc bottom-10% sim"], rows,
        [12, 5, 9, 9, 16, 16, 18])
OUT["mechanism_nn"] = MECH

# ============================================================================ 16 leakage-risk predictor
Q.sub("16 LEAKAGE-RISK PREDICTOR (exploratory): does the provenance audit predict a run's inflation?")
FEAT = ["same_original", "identical_file", "near_dup", "struct_group", "ctrl_group"]
Xr, yr, gr = [], [], []
for rid, (j, r, z) in RUNS.items():
    if j["spec"]["recipe"] != "std" or j["spec"]["backbone"] != BB0 or j["spec"]["exp"] in ("RL",):
        continue
    lk = r["info"].get("leak", {})
    if not lk or "own" not in RM[rid]:
        continue
    Xr.append([lk.get(k, 0) for k in FEAT])
    yr.append(RM[rid]["own"]["bal_acc"] - RM[rid]["hold"]["bal_acc"])
    gr.append(j["spec"]["ds"])
if len(set(gr)) >= 3:
    from sklearn.linear_model import LinearRegression, RidgeCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    Xr, yr, gr = np.array(Xr, float), np.array(yr, float), np.array(gr)
    models = {"null (training mean)": None,
              "one predictor: any relative in training": [4],
              "ridge: all five audit shares": [0, 1, 2, 3, 4]}
    RISK = {}
    rows = []
    for name, cols in models.items():
        pred = np.zeros_like(yr)
        for ds in sorted(set(gr)):
            m = gr == ds
            if cols is None:
                pred[m] = yr[~m].mean()
            elif len(cols) == 1:
                pred[m] = LinearRegression().fit(Xr[~m][:, cols], yr[~m]).predict(Xr[m][:, cols])
            else:
                mdl = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 12)))
                pred[m] = mdl.fit(Xr[~m][:, cols], yr[~m]).predict(Xr[m][:, cols])
        r2 = 1 - np.sum((yr - pred) ** 2) / np.sum((yr - yr.mean()) ** 2)
        mae = float(np.mean(np.abs(yr - pred)))
        RISK[name] = dict(lodo_r2=float(r2), mae=mae)
        rows.append([name, f"{r2:+.2f}", f"{100 * mae:.1f}"])
    Q.table(["model (leave-one-dataset-out)", "R²", "MAE (points)"], rows, [44, 8, 14])
    one = LinearRegression().fit(Xr[:, [4]], yr)
    RISK["slope_any_relative"] = float(one.coef_[0])
    RISK["n_runs"] = int(len(yr))
    RISK["datasets"] = sorted(set(gr))
    OUT["risk_predictor"] = RISK
    Q.kv("slope, any-relative share", f"{100 * one.coef_[0]:+.1f} points of inflation per 100% share (all runs pooled)")
    print(f"  n = {len(yr)} runs from {len(set(gr))} datasets. A negative R² means the model predicts worse than the mean of")
    print("  the other datasets; that result is reported as it is.")
else:
    print("  fewer than 3 datasets with finished runs — skipped")

# ============================================================================ 17 provenance-stratified own-fold accuracy (Idea B)
Q.sub("17 PROVENANCE-STRATIFIED OWN FOLD (L arm, all seeds): accuracy by the closest relative each test image has in training")
STRATA = ["same source image (sibling)", "identical / near-duplicate file", "same slide / patient / source group", "no relative"]
STR = {}
rows, csvrows = [], []
for ds in avail:
    R_ = sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std")
    if not R_:
        continue
    dd = Q.DS.get(ds)
    cor_all, st_all, key_all, hold_acc = [], [], [], []
    for j, r, z in R_:
        B = Q.build_job(dict(j["spec"], xeval=False), check=False)
        tr = B["train"]["row"]
        s_row, s_md5 = set(tr.tolist()), set(dd.md5[tr].tolist())
        s_near, s_grp = set(dd.g_near[tr].tolist()), set(dd.g_ctrl[tr].tolist())
        rows_o = z["own_row"]
        st = np.full(len(rows_o), 3)
        for i, rr in enumerate(rows_o):
            if rr in s_row:
                st[i] = 0
            elif dd.md5[rr] in s_md5 or dd.g_near[rr] in s_near:
                st[i] = 1
            elif dd.g_ctrl[rr] in s_grp:
                st[i] = 2
        cor_all.append(correct(z, "own"))
        st_all.append(st)
        key_all.append(z["own_key"] + 10 ** 7 * j["spec"].get("rep", 0) + 10 ** 8 * j["spec"]["outer"])
        hold_acc.append(float(correct(z, "hold").mean()))
    cor, st, key = map(np.concatenate, (cor_all, st_all, key_all))
    cl, inv = np.unique(key, return_inverse=True)
    g = np.random.default_rng(Q.shash(ds, "strata") % 2 ** 32)
    W = list(Q.boot_weights(len(cl), NB, g))
    d = dict(hold_acc=float(np.mean(hold_acc)), n_runs=len(R_), strata={})
    boots = {}
    for k, name in enumerate(STRATA):
        m = st == k
        if not m.any():
            continue
        cc = np.bincount(inv, cor * m, len(cl))
        nn_ = np.bincount(inv, m.astype(float), len(cl))
        bs = np.concatenate([(Wt @ cc) / np.maximum(Wt @ nn_, 1e-9) for Wt in W])
        boots[k] = bs
        d["strata"][name] = dict(share=float(m.mean()), n=int(m.sum()), acc=float(cor[m].mean()), ci=Q.ci(bs))
        rows.append([ds, name, f"{100 * m.mean():.1f}", int(m.sum()), f"{100 * cor[m].mean():.2f} [{100 * Q.ci(bs)[0]:.2f},"
                     f"{100 * Q.ci(bs)[1]:.2f}]", f"{100 * d['hold_acc']:.2f}"])
        csvrows.append([ds, name, m.mean(), int(m.sum()), cor[m].mean(), *Q.ci(bs), d["hold_acc"]])
    if 0 in boots and 3 in boots:
        diff = boots[0] - boots[3]
        d["sibling_minus_none"] = dict(est=d["strata"][STRATA[0]]["acc"] - d["strata"][STRATA[3]]["acc"], ci=Q.ci(diff),
                                       p=Q.boot_p(diff))
    STR[ds] = d
if rows:
    Q.table(["dataset", "closest relative in training", "% of own fold", "images", "own-fold accuracy [CI]", "holdout acc"], rows,
            [12, 38, 14, 8, 26, 12])
    for ds, d in STR.items():
        if "sibling_minus_none" in d:
            e = d["sibling_minus_none"]
            Q.kv(f"{ds}: sibling - no relative", f"{100 * e['est']:+.2f} points [{100 * e['ci'][0]:+.2f}, {100 * e['ci'][1]:+.2f}] "
                 f"p {pf(e['p'])}")
    print("  images are pooled over folds and seeds (an image counted once per run); CIs resample groups (fold x group clusters).")
    print("  A stratum with very few images (e.g. 'no relative' under heavy augmentation) gives a wide CI — read it with its n.")
    write_csv("provenance_strata", ["dataset", "stratum", "share", "n", "acc", "ci_lo", "ci_hi", "holdout_acc"], csvrows)
OUT["provenance_strata"] = STR

Q.jsave(Q.D["eval"] / "q1_results.json", OUT)
print(f"\n[BLOCK 25 ✓]  results in q1/eval/q1_results.json · tables in q1/tables/ · {Q.hms(time.time() - t_all)} · run BLOCK 26 next")

In [ ]:
# ============================================================================
# BLOCK 26 — CLINICAL SECTION: OPERATING POINTS · SELECTIVE PREDICTION · UNCERTAINTY · GRAD-CAM DELETION METRICS
# ============================================================================
import os, sys, json, time, math, importlib, warnings, csv
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("26")

t_all = time.time()
P = Q.plan()
BB0 = P["primary_backbone"]
NB = 4000
Q.hdr("BLOCK 26 — CLINICAL SECTION · OPERATING POINTS · SELECTIVE PREDICTION · UNCERTAINTY · SALIENCY CHECK")
jobs = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
E1 = defaultdict(list)
for j in jobs:
    s = j["spec"]
    if s["exp"] == "E1" and s["backbone"] == BB0 and s["recipe"] == "std" and s["arm"] in ("L", "C") and Q.job_done(j):
        r, z = Q.load_run(j["id"])
        E1[(s["ds"], s["arm"])].append((j, r, {k: z[k] for k in z.files}))
OUT = {}


def ent(p):
    return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1)


def fmt(v, d=2):
    return "—" if v is None or not np.isfinite(v) else f"{100 * v:.{d}f}"


# ------------------------------------------------------------------ 1  operating points (thresholds from inner validation)
Q.sub("1  OPERATING POINTS ON THE CLEAN HOLDOUT  (threshold chosen on each run's inner validation; every column from that threshold)")
rows, crow = [], []
OP = {}
for (ds, arm), R_ in sorted(E1.items()):
    pos = Q.META[ds]["pos"]
    for name in ("youden", "sens95", "sens99"):
        TP = FN = TN = FP = 0
        per_img = []
        for j, r, z in R_:
            T = Q.fit_temperature(z["ival_logit"], z["ival_y"])
            pi = Q.softmax(z["ival_logit"], T)[:, pos]
            th = Q.thresholds_from(pi, (z["ival_y"] == pos).astype(int))[name]
            ph = Q.softmax(z["hold_logit"], T)[:, pos]
            yh = (z["hold_y"] == pos).astype(int)
            pr = ph >= th
            TP += int((pr & (yh == 1)).sum()); FN += int((~pr & (yh == 1)).sum())
            TN += int((~pr & (yh == 0)).sum()); FP += int((pr & (yh == 0)).sum())
            per_img.append((yh, pr, z["hold_key"] + 10 ** 7 * j["spec"].get("rep", 0) + 10 ** 8 * j["spec"]["outer"]))
        se, sp = TP / max(1, TP + FN), TN / max(1, TN + FP)
        y_, p_, k_ = map(np.concatenate, zip(*per_img))
        cl, inv = np.unique(k_, return_inverse=True)
        g = np.random.default_rng(1)
        tp_c = np.bincount(inv, (p_ & (y_ == 1)).astype(float), len(cl))
        pos_c = np.bincount(inv, (y_ == 1).astype(float), len(cl))
        tn_c = np.bincount(inv, (~p_ & (y_ == 0)).astype(float), len(cl))
        neg_c = np.bincount(inv, (y_ == 0).astype(float), len(cl))
        bs_se, bs_sp = [], []
        for Wt in Q.boot_weights(len(cl), NB, g):
            bs_se.append((Wt @ tp_c) / np.maximum(Wt @ pos_c, 1e-9))
            bs_sp.append((Wt @ tn_c) / np.maximum(Wt @ neg_c, 1e-9))
        bs_se, bs_sp = np.concatenate(bs_se), np.concatenate(bs_sp)
        lrp = se / (1 - sp) if sp < 1 else float("inf")
        lrn = (1 - se) / sp if sp > 0 else float("inf")
        OP[f"{ds}|{arm}|{name}"] = dict(sens=se, spec=sp, sens_ci=Q.ci(bs_se), spec_ci=Q.ci(bs_sp), ppv=TP / max(1, TP + FP),
                                       npv=TN / max(1, TN + FN), lr_pos=lrp, lr_neg=lrn, fnr=1 - se, TP=TP, FN=FN, TN=TN, FP=FP)
        miss = f"1 in {1 / (1 - se):.1f}" if se < 1 else "none"
        rows.append([ds, arm, name, f"{fmt(se)} [{fmt(Q.ci(bs_se)[0])},{fmt(Q.ci(bs_se)[1])}]",
                     f"{fmt(sp)} [{fmt(Q.ci(bs_sp)[0])},{fmt(Q.ci(bs_sp)[1])}]", fmt(TP / max(1, TP + FP)), fmt(TN / max(1, TN + FN)),
                     f"{lrp:.2f}" if np.isfinite(lrp) else "inf", f"{lrn:.3f}", miss])
        crow.append([ds, arm, name, se, *Q.ci(bs_se), sp, *Q.ci(bs_sp), TP / max(1, TP + FP), TN / max(1, TN + FN), lrp, lrn, 1 - se,
                     TP, FN, TN, FP])
Q.table(["dataset", "arm", "point", "sens [CI]", "spec [CI]", "PPV", "NPV", "LR+", "LR-", "missed positives"], rows,
        [12, 5, 8, 23, 23, 8, 8, 8, 8, 16])
print("  pooled over outer folds and seeds; CIs resample groups; NDB-UFES: OSCC vs rest. 'sens95' / 'sens99' = the threshold")
print("  that gave >= 95% / >= 99% sensitivity on inner validation (the holdout sensitivity shows how well that transfers).")
with open(Q.D["tables"] / "operating_points.csv", "w", newline="") as f:
    csv.writer(f).writerows([["dataset", "arm", "point", "sens", "sens_lo", "sens_hi", "spec", "spec_lo", "spec_hi", "ppv", "npv",
                              "lr_pos", "lr_neg", "fnr", "TP", "FN", "TN", "FP"]] + crow)
OUT["operating_points"] = OP

# ------------------------------------------------------------------ 2  seed ensembles: uncertainty decomposition
Q.sub("2  UNCERTAINTY DECOMPOSITION ON THE HOLDOUT  (seed ensembles: total = aleatoric + epistemic)")
rows = []
UNC = {}
ENS = {}
missing = []
for (ds, arm), R_ in sorted(E1.items()):
    by = defaultdict(list)
    for j, r, z in R_:
        T = Q.fit_temperature(z["ival_logit"], z["ival_y"])
        by[(j["spec"]["outer"], j["spec"].get("rep", 0))].append((j["spec"]["seed"], Q.softmax(z["hold_logit"], T), z["hold_y"],
                                                                  z["hold_key"]))
    ensemble = all(len(v) >= 2 for v in by.values())
    tot, ale, epi, cor, key, ys, pm = [], [], [], [], [], [], []
    ba_single, ba_ens = [], []
    for fk, lst in sorted(by.items()):
        lst = sorted(lst, key=lambda x: x[0])
        if not ensemble:
            lst = lst[:1]                                   # seed-0 model only, same rule for every fold
        Pm = np.stack([x[1] for x in lst])
        pbar = Pm.mean(0)
        y = lst[0][2]
        t_ = ent(pbar)
        a_ = np.mean([ent(p) for p in Pm], 0)
        tot.append(t_); ale.append(a_); epi.append(t_ - a_)
        cor.append((pbar.argmax(1) == y).astype(float))
        key.append(lst[0][3] + 10 ** 7 * fk[1] + 10 ** 8 * fk[0])
        ys.append(y); pm.append(pbar)
        K = pbar.shape[1]
        ba = lambda pr: np.mean([np.mean(pr[y == c] == c) for c in range(K) if (y == c).any()])
        ba_single.append(np.mean([ba(p.argmax(1)) for p in Pm]))
        ba_ens.append(ba(pbar.argmax(1)))
    tot, ale, epi, cor = map(np.concatenate, (tot, ale, epi, cor))
    key, ys, pm = np.concatenate(key), np.concatenate(ys), np.concatenate(pm)
    ENS[(ds, arm)] = dict(p=pm, y=ys, key=key, H=tot, cor=cor, ensemble=ensemble)
    if not ensemble:
        missing.append(f"{ds}/{arm}")
        continue
    UNC[f"{ds}|{arm}"] = dict(total=float(tot.mean()), aleatoric=float(ale.mean()), epistemic=float(epi.mean()),
                              epi_share=float(epi.mean() / max(1e-9, tot.mean())),
                              epi_correct=float(epi[cor == 1].mean()) if (cor == 1).any() else None,
                              epi_wrong=float(epi[cor == 0].mean()) if (cor == 0).any() else None,
                              ba_single=float(np.mean(ba_single)), ba_ensemble=float(np.mean(ba_ens)), n=int(len(tot)),
                              seeds=int(min(len(v) for v in by.values())))
    u = UNC[f"{ds}|{arm}"]
    rows.append([ds, arm, u["n"], u["seeds"], f"{u['total']:.3f}", f"{u['aleatoric']:.3f}", f"{u['epistemic']:.4f}",
                 f"{100 * u['epi_share']:.1f}%", "/".join("—" if u[k] is None else f"{u[k]:.4f}" for k in ("epi_correct", "epi_wrong")),
                 f"{fmt(u['ba_single'])} -> {fmt(u['ba_ensemble'])}"])
if rows:
    Q.table(["dataset", "arm", "images", "seeds", "total H", "aleatoric", "epistemic", "epi share", "epi correct/wrong",
             "BA single -> ensemble"], rows, [12, 5, 8, 6, 9, 10, 10, 10, 18, 22])
    print("  entropies in nats; the ensemble gain compares single seeds with their seed average on the SAME holdout images.")
if missing:
    print(f"  not available yet for {', '.join(missing)}: needs the seeds phase (BLOCK 23) so that every fold has >= 2 seeds.")
OUT["uncertainty"] = UNC

# ------------------------------------------------------------------ 3  selective prediction with CIs
Q.sub("3  SELECTIVE PREDICTION ON THE HOLDOUT  (refer the least certain cases; CIs resample groups)")
rows = []
SEL = {}
for (ds, arm), e in sorted(ENS.items()):
    y, H, cor, key = e["y"], e["H"], e["cor"], e["key"]
    order = np.argsort(H, kind="stable")
    n = len(y)
    cl, inv = np.unique(key, return_inverse=True)
    res = {}
    for cov in (1.0, 0.9, 0.8, 0.7):
        keep = np.zeros(n, bool)
        keep[order[:int(math.ceil(cov * n))]] = True
        acc = cor[keep].mean()
        g = np.random.default_rng(2)
        cc = np.bincount(inv, cor * keep, len(cl))
        nn_ = np.bincount(inv, keep.astype(float), len(cl))
        bs = np.concatenate([(Wt @ cc) / np.maximum(Wt @ nn_, 1e-9) for Wt in Q.boot_weights(len(cl), NB, g)])
        res[cov] = (float(acc), Q.ci(bs), int(keep.sum()))
    risk = 1 - cor[order]
    aurc = float(np.mean(np.cumsum(risk) / np.arange(1, n + 1)))
    src = "seed ensemble" if e["ensemble"] else "single model"
    SEL[f"{ds}|{arm}"] = dict(cov={str(k): dict(acc=v[0], ci=v[1], n=v[2]) for k, v in res.items()}, aurc=aurc,
                              gain_80=res[0.8][0] - res[1.0][0], ranking=src)
    rows.append([ds, arm, n, src] + [f"{fmt(res[c][0])} [{fmt(res[c][1][0])},{fmt(res[c][1][1])}]" for c in (1.0, 0.9, 0.8, 0.7)]
                + [f"{100 * (res[0.8][0] - res[1.0][0]):+.2f}", f"{aurc:.4f}"])
Q.table(["dataset", "arm", "n", "ranking", "acc 100%", "acc 90%", "acc 80%", "acc 70%", "gain@80", "AURC"], rows,
        [12, 5, 6, 14, 21, 21, 21, 21, 9, 8])
print("  cases ranked by predictive entropy; accuracy of the retained cases; gain@80 = effect of referring 20% (points).")
OUT["selective"] = SEL

# ------------------------------------------------------------------ 4  calibration data for reliability diagrams
CAL = {}
for (ds, arm), R_ in sorted(E1.items()):
    raw, cal, ys = [], [], []
    for j, r, z in R_:
        T = Q.fit_temperature(z["ival_logit"], z["ival_y"])
        raw.append(Q.softmax(z["hold_logit"]))
        cal.append(Q.softmax(z["hold_logit"], T))
        ys.append(z["hold_y"])
    raw, cal, ys = np.concatenate(raw), np.concatenate(cal), np.concatenate(ys)
    bins = np.linspace(0, 1, 11)
    d = {}
    for nm, p in (("raw", raw), ("cal", cal)):
        conf, acc = p.max(1), (p.argmax(1) == ys).astype(float)
        idx = np.clip(np.digitize(conf, bins) - 1, 0, 9)
        d[nm] = dict(conf=[float(conf[idx == b].mean()) if (idx == b).any() else None for b in range(10)],
                     acc=[float(acc[idx == b].mean()) if (idx == b).any() else None for b in range(10)],
                     n=[int((idx == b).sum()) for b in range(10)], ece=Q.ece_score(p, ys))
    CAL[f"{ds}|{arm}"] = d
Q.sub("4  CALIBRATION ON THE HOLDOUT  (ECE before / after temperature scaling fitted on inner validation)")
Q.table(["dataset", "arm", "ECE raw", "ECE calibrated"],
        [[k.split("|")[0], k.split("|")[1], f"{v['raw']['ece']:.4f}", f"{v['cal']['ece']:.4f}"] for k, v in CAL.items()],
        [12, 5, 10, 16])
OUT["calibration"] = CAL

# ------------------------------------------------------------------ 5  Grad-CAM with deletion / insertion metrics
Q.sub("5  GRAD-CAM FAITHFULNESS  (deletion / insertion AUC vs a random-order baseline, up to 100 holdout images, E1 seed 0 fold 0)")
import torch
import torch.nn.functional as F
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAL = {}
examples = {}
rows = []
for (ds, arm), R_ in sorted(E1.items()):
    wr = [x for x in R_ if (Q.D["runs"] / x[0]["id"] / "weights.pt").exists()]
    if not wr:
        continue
    j, r, z = wr[0]
    K = r["info"]["K"]
    img = r["img"]
    model, _, _ = Q.make_model(j["spec"]["backbone"], K, img, 0.0, "linear", pretrained=False)
    sd = torch.load(Q.D["runs"] / j["id"] / "weights.pt", map_location="cpu")
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in sd.items()})
    model.to(dev).eval()
    mean, std = Q.norm_consts(model, dev)
    n = min(100, len(z["hold_row"]))
    sel_i = np.random.default_rng(0).choice(len(z["hold_row"]), n, replace=False)
    rows_ = z["hold_row"][sel_i]
    feed = Q.Feeder({ds: rows_}, dev, pin=False)
    x0 = Q.eval_prep(feed.batch(ds, rows_, np.zeros(n, int), None, None), img)
    blur = F.avg_pool2d(F.pad(x0, (7, 7, 7, 7), mode="reflect"), 15, stride=1)
    G = 14 if img >= 112 else 4
    cell = img // G
    del_cam, del_rnd, ins_cam, ins_rnd, cams = [], [], [], [], []
    for i in range(n):
        xi = ((x0[i:i + 1] - mean) / std).requires_grad_(False)
        feats = model.forward_features(xi)
        if feats.ndim == 3:
            continue
        feats.retain_grad()
        logits = model.forward_head(feats)
        c = int(logits.argmax(1))
        model.zero_grad()
        logits[0, c].backward()
        w = feats.grad.mean((2, 3), keepdim=True)
        cam = F.relu((w * feats).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(G, G), mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy().ravel()
        cams.append(cam)
        orders = {"cam": np.argsort(-cam), "rnd": np.random.default_rng(i).permutation(G * G)}
        for nm, od in orders.items():
            masks = torch.zeros(11, G * G, device=dev)
            for step in range(11):
                k = int(round(step / 10 * G * G))
                masks[step, torch.from_numpy(od[:k].copy()).to(dev)] = 1
            m = F.interpolate(masks.view(11, 1, G, G), size=(img, img), mode="nearest")
            for mode in ("del", "ins"):
                xm = x0[i:i + 1] * (1 - m) + blur[i:i + 1] * m if mode == "del" else blur[i:i + 1] * (1 - m) + x0[i:i + 1] * m
                with torch.no_grad(), torch.autocast(dev.type, enabled=dev.type == "cuda"):
                    probs = torch.softmax(model((xm - mean) / std).float(), 1)[:, c].cpu().numpy()
                auc = float((getattr(np, "trapezoid", None) or np.trapz)(probs, dx=0.1))
                {"camdel": del_cam, "rnddel": del_rnd, "camins": ins_cam, "rndins": ins_rnd}[nm + mode].append(auc)
        if len(examples.get((ds, arm), [])) < 4:
            examples.setdefault((ds, arm), []).append((x0[i].permute(1, 2, 0).cpu().numpy(), cam.reshape(G, G), c, int(z["hold_y"][sel_i[i]])))
    if not del_cam:
        continue
    d_ = np.array(del_rnd) - np.array(del_cam)
    i_ = np.array(ins_cam) - np.array(ins_rnd)
    g = np.random.default_rng(3)
    bd = [d_[g.integers(0, len(d_), len(d_))].mean() for _ in range(2000)]
    bi = [i_[g.integers(0, len(i_), len(i_))].mean() for _ in range(2000)]
    SAL[f"{ds}|{arm}"] = dict(n=len(d_), del_cam=float(np.mean(del_cam)), del_rnd=float(np.mean(del_rnd)),
                              ins_cam=float(np.mean(ins_cam)), ins_rnd=float(np.mean(ins_rnd)),
                              del_gain=float(d_.mean()), del_gain_ci=Q.ci(bd), ins_gain=float(i_.mean()), ins_gain_ci=Q.ci(bi))
    s_ = SAL[f"{ds}|{arm}"]
    verdict = "faithful" if s_["del_gain_ci"][0] > 0 and s_["ins_gain_ci"][0] > 0 else "illustration only"
    rows.append([ds, arm, s_["n"], f"{s_['del_cam']:.3f} vs {s_['del_rnd']:.3f}", f"{s_['del_gain']:+.3f} [{s_['del_gain_ci'][0]:+.3f},{s_['del_gain_ci'][1]:+.3f}]",
                 f"{s_['ins_cam']:.3f} vs {s_['ins_rnd']:.3f}", f"{s_['ins_gain']:+.3f} [{s_['ins_gain_ci'][0]:+.3f},{s_['ins_gain_ci'][1]:+.3f}]", verdict])
    del model
if rows:
    Q.table(["dataset", "arm", "n", "deletion AUC cam/rnd", "deletion gain [CI]", "insertion AUC cam/rnd", "insertion gain [CI]",
             "verdict"], rows, [12, 5, 5, 21, 24, 22, 24, 17])
    print("  deletion: lower AUC = the highlighted regions matter; gain = random - cam. insertion: gain = cam - random.")
else:
    print("  no saved weights found (they are kept for E1 seed 0, outer fold 0).")
OUT["saliency"] = SAL
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ex = [(k, v) for k, v in examples.items()]
    if ex:
        nrow = len(ex)
        fig, ax = plt.subplots(nrow, 8, figsize=(16, 2.1 * nrow))
        ax = np.atleast_2d(ax)
        for r_i, ((ds, arm), lst) in enumerate(ex):
            for c_i in range(4):
                for a_ in ax[r_i, 2 * c_i: 2 * c_i + 2]:
                    a_.axis("off")
                if c_i >= len(lst):
                    continue
                im, cam, pc, yc = lst[c_i]
                ax[r_i, 2 * c_i].imshow(np.clip(im, 0, 1))
                ax[r_i, 2 * c_i].set_title(f"{Q.SHORT[ds]} {arm} y={Q.META[ds]['classes'][yc]}", fontsize=7)
                ax[r_i, 2 * c_i + 1].imshow(np.clip(im, 0, 1))
                ax[r_i, 2 * c_i + 1].imshow(np.kron(cam, np.ones((im.shape[0] // cam.shape[0], im.shape[1] // cam.shape[1]))),
                                            cmap="magma", alpha=0.45)
                ax[r_i, 2 * c_i + 1].set_title(f"pred={Q.META[ds]['classes'][pc]}", fontsize=7)
        fig.tight_layout()
        fig.savefig(Q.D["figs"] / "gradcam_examples.png", dpi=160)
        plt.close(fig)
except Exception as e:
    print("  figure skipped:", e)
Q.jsave(Q.D["eval"] / "clinical.json", OUT)
print(f"\n[BLOCK 26 ✓]  results in q1/eval/clinical.json · {Q.hms(time.time() - t_all)} · run BLOCK 27 next")

In [ ]:
# ============================================================================
# BLOCK 27 — PAPER FIGURES (PNG, 300 dpi) + RESULTS SUMMARY (Markdown)
# ============================================================================
import os, sys, json, time, math, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("27")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

t_all = time.time()
P = Q.plan()
R = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
CL = Q.jload(Q.D["eval"] / "clinical.json", {}) or {}
AU = Q.jload(Q.D["index"] / "audit.json", {}) or {}
PR = Q.jload(Q.D["eval"] / "probes.json", []) or []
FIG = Q.D["figs"]
COL = {"L": "#eb6834", "C": "#2a78d6", "probe": "#1baf7a"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
NAME = {"he_stained": "OCDC (H&E)", "histopath": "Histopathology", "ndb_ufes": "NDB-UFES", "clinical": "Clinical photos",
        "breakhis": "BreakHis"}
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 300, "figure.dpi": 110})
made = []
Q.hdr("BLOCK 27 — PAPER FIGURES + RESULTS SUMMARY")


def save(fig, name, note):
    fig.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    made.append((name, note))


# ---------------------------------------------------------------- Figure A: own fold vs clean holdout
E1 = R.get("E1", {})
ds_list = [d for d in Q.DATASETS if d in E1 and "L" in E1[d].get("arms", {}) and "C" in E1[d]["arms"]]
if ds_list:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 0.75 * len(ds_list) + 1.6), gridspec_kw=dict(width_ratios=[1.25, 1]))
    for i, ds in enumerate(ds_list):
        y0 = len(ds_list) - 1 - i
        for k, arm in enumerate(("L", "C")):
            a = E1[ds]["arms"][arm]
            yy = y0 + (0.15 if arm == "L" else -0.15)
            a1.plot([100 * a["hold"], 100 * a["own"]], [yy, yy], color=COL[arm], lw=2, solid_capstyle="round")
            a1.scatter([100 * a["hold"]], [yy], s=40, color="white", edgecolor=COL[arm], lw=2, zorder=3)
            a1.scatter([100 * a["own"]], [yy], s=40, color=COL[arm], zorder=3)
            lo, hi = a["infl_ci"]
            a2.errorbar([100 * a["infl"]], [yy], xerr=[[100 * (a["infl"] - lo)], [100 * (hi - a["infl"])]], fmt="o",
                        color=COL[arm], ms=6, lw=2, capsize=0)
    for ax in (a1, a2):
        ax.set_yticks(range(len(ds_list)))
        ax.set_yticklabels([NAME[d] for d in ds_list][::-1])
        ax.grid(axis="y", visible=False)
    a1.set_xlabel("Balanced accuracy (%)")
    a1.set_title("A  What the protocol reports (filled) vs clean holdout (open)", loc="left", fontsize=9, color=INK)
    a2.axvline(0, color=INK2, lw=1)
    a2.set_xlabel("Inflation = own fold − clean holdout (points, 95% CI)")
    a2.set_title("B  Inflation by arm", loc="left", fontsize=9, color=INK)
    h = [plt.Line2D([], [], color=COL["L"], lw=2, marker="o", label="leaky protocol (as published)"),
         plt.Line2D([], [], color=COL["C"], lw=2, marker="o", label="leakage-controlled, matched")]
    fig.legend(handles=h, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save(fig, "Fig_Q1_main_own_vs_holdout.png", "E1: own-fold vs clean-holdout balanced accuracy and inflation with 95% CI")

# ---------------------------------------------------------------- Figure B: dose-response
D_ = R.get("dose", {})
panels = [(k.split("|")) for k in D_]
keys = sorted({(e, d) for e, d, a in panels}, key=lambda t: (["DOSEK", "DUP", "OVL"].index(t[0]), t[1]))
if keys:
    nc = min(4, len(keys))
    nr = int(math.ceil(len(keys) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.1 * nc, 2.6 * nr), squeeze=False)
    xl = {"DOSEK": "augmentation factor k (log scale)", "DUP": "share of own-fold images with a copy in training",
          "OVL": "share of group-mates moved into training"}
    for ax, (ex, ds) in zip(axs.ravel(), keys):
        for arm in ("L", "C"):
            v = D_.get(f"{ex}|{ds}|{arm}")
            if not v:
                continue
            pts = np.array(v["points"], float)
            xs = 2 ** pts[:, 0] if ex == "DOSEK" else pts[:, 0]
            ax.scatter(xs, 100 * pts[:, 1], s=10, color=COL[arm], alpha=0.35, lw=0)
            ux = np.unique(xs)
            lab_ = "leaky" if arm == "L" else "controlled"
            if v.get("p") is not None and np.isfinite(v["p"]):
                lab_ += f"  slope p={v['p']:.3g}"
            ax.plot(ux, [100 * pts[xs == u, 1].mean() for u in ux], color=COL[arm], lw=2, marker="o", ms=4, label=lab_)
        if ex == "DOSEK":
            ax.set_xscale("log", base=2)
        ax.axhline(0, color=INK2, lw=1)
        ax.set_title(f"{ex} · {NAME[ds]}", loc="left", fontsize=8.5, color=INK)
        ax.set_xlabel(xl[ex], fontsize=7.5)
        ax.set_ylabel("inflation (points)", fontsize=7.5)
        ax.legend(fontsize=6.5, loc="upper left")
    for ax in axs.ravel()[len(keys):]:
        ax.axis("off")
    fig.tight_layout()
    save(fig, "Fig_Q1_dose_response.png", "dose-response: augmentation factor, injected duplicates, group overlap (fold points + means)")

# ---------------------------------------------------------------- Figure C: random-label control
RL = R.get("RL", {})
if RL:
    dss = sorted({k.split("|")[0] for k in RL})
    fig, ax = plt.subplots(figsize=(1.6 * len(dss) + 2, 3))
    for i, ds in enumerate(dss):
        for k, arm in enumerate(("L", "C")):
            v = RL.get(f"{ds}|{arm}")
            if not v:
                continue
            x = i + (-0.18 if arm == "L" else 0.18)
            ax.bar(x, 100 * v["own"], width=0.34, color=COL[arm], label=("leaky" if arm == "L" else "controlled") if i == 0 else None)
            ax.errorbar(x, 100 * v["own"], yerr=[[100 * (v["own"] - v["own_ci"][0])], [100 * (v["own_ci"][1] - v["own"])]],
                        color=INK, lw=1, capsize=2)
        ch = 100 / Q.META[ds]["K"]
        ax.plot([i - 0.4, i + 0.4], [ch, ch], color=INK2, lw=1.2, ls="--")
    ax.set_xticks(range(len(dss)))
    ax.set_xticklabels([NAME[d] for d in dss])
    ax.set_ylabel("own-fold balanced accuracy (%)")
    ax.set_title("Labels randomised per group: own-fold balanced accuracy with 95% CI (dashed = chance)", loc="left", fontsize=9)
    ax.legend(loc="upper right")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    save(fig, "Fig_Q1_random_label.png", "random-label control (labels per source/slide/patient)")

# ---------------------------------------------------------------- Figure D: capacity
GEN = R.get("generality", {})
dss = [d for d in P["generality"]["datasets"] if any(k.startswith(d + "|") and "|capacity" not in k for k in GEN)]
if dss:
    nc = min(3, len(dss))
    nr = int(np.ceil(len(dss) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(4.3 * nc, 3.6 * nr), squeeze=False)
    for ax in axs.ravel()[len(dss):]:
        ax.axis("off")
    for ax, ds in zip(axs.ravel(), dss):
        cap = GEN.get(f"{ds}|capacity", {})
        ticks, labels = [], []
        pos = 0
        if cap and np.isfinite(cap.get("one_nn", np.nan)):
            ax.scatter([pos], [100 * cap["one_nn"]], marker="^", s=34, color=INK2, zorder=3, label="1-NN on frozen features")
            ticks.append(pos); labels.append("1-NN\nfrozen")
            pos += 1
        if cap and np.isfinite(cap.get("linear_probe", np.nan)):
            ax.scatter([pos], [100 * cap["linear_probe"]], marker="s", s=30, color=COL["probe"], zorder=3,
                       label="linear probe, frozen")
            ticks.append(pos); labels.append("linear\nprobe")
            pos += 1
        pts = []
        for bb, (short, par, _) in Q.BACKBONES.items():
            v = GEN.get(f"{ds}|{bb}")
            if v and "L" in v:
                pts.append((par, short, 100 * (v["L"][0] - v["L"][1])))
        pts.sort()
        if pts:
            xs = list(range(pos, pos + len(pts)))
            ax.plot(xs, [p_[2] for p_ in pts], color=COL["L"], lw=1.5, marker="o", ms=5, zorder=3, label="fine-tuned")
            ticks += xs
            labels += [f"{p_[1]}\n{p_[0]:.0f}M" for p_ in pts]
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels, fontsize=6.5, rotation=35, ha="right", rotation_mode="anchor")
        ax.set_xlim(-0.6, max(ticks) + 0.6 if ticks else 1)
        ax.axhline(0, color=INK2, lw=1)
        ax.set_title(NAME[ds], loc="left", fontsize=8.5)
        ax.set_ylabel("leaky-arm inflation (points)", fontsize=7.5)
    axs[0][0].legend(fontsize=6.5, loc="best")
    fig.tight_layout()
    save(fig, "Fig_Q1_capacity.png", "inflation of the leaky arm across backbone families and frozen-feature probes")

# ---------------------------------------------------------------- Figure E: audit (NDB order + histopathology parents)
if (Q.D["index"] / "ndb_ufes.json").exists():
    dd = Q.DS.get("ndb_ufes")
    emb = Q.D["store"] / f"ndb_ufes_emb_{AU.get('_meta', {}).get('embed', 'dinov2_s')}.npy"
    if emb.exists():
        E = np.load(emb).astype(np.float32)
        order = np.argsort([r["pidx"] for r in dd.R])[:600]
        nshow = len(order)
        s = (E[order][:-1] * E[order][1:]).sum(1)
        seg = dd.g_struct[order]
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 2.8), gridspec_kw=dict(width_ratios=[2.2, 1]))
        a1.plot(np.arange(len(s)), s, color=COL["C"], lw=0.8)
        for b in np.nonzero(seg[1:] != seg[:-1])[0]:
            a1.axvline(b + 0.5, color=COL["L"], lw=0.5, alpha=0.6)
        tau = AU.get("ndb_ufes", {}).get("segmentation", {}).get("tau")
        if tau:
            a1.axhline(tau, color=INK2, ls="--", lw=1)
        a1.set_xlabel(f"patch index p (first {nshow} in index order)")
        a1.set_ylabel("cosine to next patch")
        a1.set_title("A  NDB-UFES: similarity of consecutive patches in file-index order (orange = recovered boundaries)", loc="left", fontsize=8.5)
        allc = (E[np.argsort([r["pidx"] for r in dd.R])][:-1] * E[np.argsort([r["pidx"] for r in dd.R])][1:]).sum(1)
        a2.hist(allc, bins=60, color=COL["C"])
        if tau:
            a2.axvline(tau, color=COL["L"], lw=1.5, label=f"Otsu threshold {tau:.3f}")
        a2.set_xlabel("adjacent-patch cosine")
        a2.set_title("B  distribution", loc="left", fontsize=8.5)
        a2.legend(fontsize=7)
        fig.tight_layout()
        save(fig, "Fig_Q1_ndb_order.png", "evidence that NDB-UFES patch indices follow source images (group recovery)")

# ---------------------------------------------------------------- Figure F: reliability + selective prediction
CAL = CL.get("calibration", {})
SEL = CL.get("selective", {})
dss = [d for d in Q.DATASETS if f"{d}|L" in CAL and f"{d}|C" in CAL]
if dss:
    fig, axs = plt.subplots(2, len(dss), figsize=(2.6 * len(dss), 5.2), squeeze=False)
    for i, ds in enumerate(dss):
        ax = axs[0][i]
        ax.plot([0, 1], [0, 1], color=INK2, lw=1, ls="--")
        for arm in ("L", "C"):
            c = CAL[f"{ds}|{arm}"]["cal"]
            pts = [(x, y) for x, y, n in zip(c["conf"], c["acc"], c["n"]) if x is not None and n >= 5]
            if pts:
                ax.plot(*zip(*pts), color=COL[arm], lw=2, marker="o", ms=3,
                        label=f"{'leaky' if arm == 'L' else 'controlled'} ECE {c['ece']:.3f}")
        ax.set_title(NAME[ds], loc="left", fontsize=8.5)
        ax.set_xlabel("confidence", fontsize=7.5)
        ax.set_ylabel("accuracy", fontsize=7.5)
        ax.legend(fontsize=6)
        ax = axs[1][i]
        rk = None
        for arm in ("L", "C"):
            v = SEL.get(f"{ds}|{arm}")
            if not v:
                continue
            cov = sorted(v["cov"], key=float, reverse=True)
            x = [100 * float(c) for c in cov]
            y = [100 * v["cov"][c]["acc"] for c in cov]
            lo = [100 * v["cov"][c]["ci"][0] for c in cov]
            hi = [100 * v["cov"][c]["ci"][1] for c in cov]
            ax.fill_between(x, lo, hi, color=COL[arm], alpha=0.15, lw=0)
            ax.plot(x, y, color=COL[arm], lw=2, marker="o", ms=3, label="leaky" if arm == "L" else "controlled")
            rk = v.get("ranking")
        if rk:
            ax.text(0.02, 0.02, f"ranked by {rk} entropy", transform=ax.transAxes, fontsize=6, color=INK2)
        if i == 0:
            ax.legend(fontsize=6, loc="upper left")
        ax.invert_xaxis()
        ax.set_xlabel("coverage (% auto-reported)", fontsize=7.5)
        ax.set_ylabel("holdout accuracy (%)", fontsize=7.5)
    fig.tight_layout()
    save(fig, "Fig_Q1_calibration_selective.png", "holdout reliability (temperature from inner validation) and selective prediction with 95% CI")

# ---------------------------------------------------------------- results summary markdown
L = ["# Q1 results summary (generated by Block 27)", "", f"Plan SHA-256: `{R.get('meta', {}).get('plan_sha')}` · runs: "
     f"{R.get('meta', {}).get('n_runs')} · bootstrap resamples: {R.get('meta', {}).get('boot')}", "",
     "## E1: own fold vs clean holdout (balanced accuracy, %)", "",
     "Inflation CIs and p-values: two-stage cluster bootstrap. p is Holm-adjusted for the L arm (primary family H1) and "
     "unadjusted for the other arms.", "",
     "| Dataset | Arm | Own | Holdout | Inflation [95% CI] | p | seed SD |", "|---|---|---|---|---|---|---|"]
for ds in ds_list:
    for arm, a in E1[ds]["arms"].items():
        sdv = "" if a.get("seed_sd") is None else f"{100 * a['seed_sd']:.2f}"
        pv = a.get("infl_p_holm", a["infl_p"])
        L.append(f"| {NAME[ds]} | {arm} | {100 * a['own']:.2f} | {100 * a['hold']:.2f} | {100 * a['infl']:+.2f} "
                 f"[{100 * a['infl_ci'][0]:+.2f}, {100 * a['infl_ci'][1]:+.2f}] | {pv:.2g} | {sdv} |")
L += ["", "## Figures", ""] + [f"- `{n}` — {d}" for n, d in made]
(Q.D["tables"] / "RESULTS_Q1.md").write_text("\n".join(L))
for n, d in made:
    Q.kv(n, d)
print(f"\n[BLOCK 27 ✓]  {len(made)} figures in q1/figs/ · summary q1/tables/RESULTS_Q1.md · {Q.hms(time.time() - t_all)} · run BLOCK 28 next")

In [ ]:
# ============================================================================
# BLOCK 28 — RELEASE PACKAGE: audit toolkit · fixed leakage-controlled splits · manifests · computational status
# (run last, after BLOCK 25-27; nothing is uploaded anywhere — the zip stays in /kaggle/working/q1/release/)
# ============================================================================
import os, sys, json, time, shutil, csv, hashlib, importlib, warnings, subprocess, zipfile, platform
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("28")

INCLUDE_WEIGHTS = False      # True also puts every kept weights.pt into the zip (about 110 MB each)
AGREE_MAX_PAIRS = 150000     # the toolkit re-derives near-duplicate groups for datasets with at most this many candidate pairs

TOOLKIT_SRC = {
    '.pytest_cache/README.md': r'''# pytest cache directory #

This directory contains data from the pytest's cache plugin,
which provides the `--lf` and `--ff` options, as well as the `cache` fixture.

**Do not** commit this to version control.

See [the docs](https://docs.pytest.org/en/stable/how-to/cache.html) for more information.
''',
    'README.md': r'''# oralleak

A small audit toolkit for data leakage in medical image classification datasets. It was written for the
public oral-cancer image datasets studied in the accompanying article, but it works on any folder of images.

## What it does

- **Hashes** every file (MD5, SHA-256) and computes perceptual hashes (pHash from a 32x32 DCT, dHash from
  a 9x8 gradient) on a 288-px copy of each image.
- **Finds duplicates**: identical files (same MD5) and near duplicates (pHash distance <= 8 and SSIM >= 0.90
  on a 128-px grey copy). If one near-duplicate group grows larger than max(25, 2% of the images), the
  stricter fallback rule (pHash <= 4 and SSIM >= 0.95) is used, so that textures common to many patches do
  not chain unrelated images together.
- **Recovers source groups from file order** (`segment_order`): when patches cut from one source image are
  stored consecutively, the similarity of neighbouring patches drops at source boundaries; the cut-off is an
  Otsu threshold. Embeddings are supplied by the user (any L2-normalised feature vectors).
- **Builds grouped splits** (`grouped_split`, stratified group k-fold) and **checks splits**
  (`check_split`): it lists every relation (identical file, near duplicate, slide/patient/source group,
  augmented copy of the same source) that occurs on both sides of a split.
- **Reports provenance leakage** (`leak_report`): the share of test images that have a relative in the
  training set, by type of relation.

## Command line

```
pip install .
python -m oralleak audit --images /path/to/dataset --out files.csv      # class = parent folder name
python -m oralleak split --files files.csv --group g_near --k 5 --out split.csv
python -m oralleak check --files files.csv --split split.csv --test-fold 0
```

`check` exits with code 1 when any relation crosses the split, so it can be used in a CI job.

## Tests

`python -m pytest -q tests` runs unit tests on synthetic images and, when the package is inside the
release folder, checks every released split in `../splits`.

## Limits

Near-duplicate detection finds re-encoded, resized and lightly edited copies. It does not find crops of
different regions of the same slide; those need slide or patient identifiers (`g_struct`), or the
order-based recovery above when file order follows the source.
''',
    'oralleak/__init__.py': r'''"""oralleak - a small data-leakage audit toolkit for medical image classification datasets.

Functions cover file hashing, exact and near-duplicate detection (pHash + SSIM), recovery of source
groups from file order, grouped splitting, cross-partition checks and a provenance leakage report.
"""
from .core import (load_image, perceptual_hashes, hamming_pairs, gray_small, near_duplicate_groups,
                   union_labels, otsu, segment_order, grouped_split, check_split, leak_report,
                   audit_folder, read_csv, write_csv)

__version__ = "0.1.0"
''',
    'oralleak/__main__.py': r'''"""Command line:  python -m oralleak {audit,check,split} ...

  audit  --images DIR --out files.csv            hash every image, find exact and near duplicates
  split  --files files.csv --group g_near --k 5  stratified group k-fold (writes split.csv: relpath,fold)
  check  --files files.csv --split split.csv     cross-partition checks + leakage report; exit code 1 on leakage
         (split.csv needs relpath and role (train/test) columns, or relpath and fold with --test-fold F)
"""
import argparse
import json
import sys

from . import core


def main(argv=None):
    ap = argparse.ArgumentParser(prog="oralleak")
    sp = ap.add_subparsers(dest="cmd", required=True)
    a = sp.add_parser("audit")
    a.add_argument("--images", required=True)
    a.add_argument("--out", required=True)
    s = sp.add_parser("split")
    s.add_argument("--files", required=True)
    s.add_argument("--group", default="g_ctrl")
    s.add_argument("--label", default="label")
    s.add_argument("--k", type=int, default=5)
    s.add_argument("--seed", type=int, default=7)
    s.add_argument("--out", required=True)
    c = sp.add_parser("check")
    c.add_argument("--files", required=True)
    c.add_argument("--split", required=True)
    c.add_argument("--test-fold", type=int, default=None)
    c.add_argument("--train-roles", default="train")
    c.add_argument("--test-roles", default="test")
    ar = ap.parse_args(argv)

    if ar.cmd == "audit":
        rows, info = core.audit_folder(ar.images)
        core.write_csv(ar.out, rows)
        print(json.dumps(info, indent=1, default=str))
        return 0
    files = core.read_csv(ar.files)
    if ar.cmd == "split":
        key = ar.group if ar.group in files[0] else "g_near"
        fold = core.grouped_split([r[ar.label] for r in files], [r[key] for r in files], ar.k, ar.seed)
        core.write_csv(ar.out, [dict(relpath=r["relpath"], fold=int(f)) for r, f in zip(files, fold)])
        print(f"{len(files)} images -> {ar.k} folds grouped by {key}")
        return 0
    sp_rows = core.read_csv(ar.split)
    if ar.test_fold is not None:
        roles = {r["relpath"]: ("test" if int(r["fold"]) == ar.test_fold else "train") for r in sp_rows}
        trr, ter = ("train",), ("test",)
    else:
        roles = {r["relpath"]: r["role"] for r in sp_rows}
        trr, ter = tuple(ar.train_roles.split(",")), tuple(ar.test_roles.split(","))
    bad = core.check_split(files, roles, trr, ter)
    rep = core.leak_report(files, roles, trr, ter)
    print(json.dumps(dict(violations=bad, leakage=rep), indent=1))
    return 1 if bad else 0


if __name__ == "__main__":
    sys.exit(main())
''',
    'oralleak/core.py': r'''"""Core functions of oralleak.  Dependencies: numpy, scipy, pillow, scikit-image, scikit-learn."""
import csv
import hashlib
import io
import os
from collections import Counter, defaultdict

import numpy as np

STORE_PX = 288          # every image is decoded once and resized to this square size
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
NEAR_RULE = dict(phash_max=8, ssim_min=0.90, cap_frac=0.02, cap_min=25, fallback_phash=4, fallback_ssim=0.95)


# ----------------------------------------------------------------------------- files
def load_image(path, px=STORE_PX):
    """(md5, sha256, uint8 array px x px x 3).  JPEG files are decoded at reduced size first."""
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None
    b = open(path, "rb").read()
    md5, sha = hashlib.md5(b).hexdigest(), hashlib.sha256(b).hexdigest()
    im = Image.open(io.BytesIO(b))
    if im.format == "JPEG":
        im.draft("RGB", (px * 2, px * 2))
    im = im.convert("RGB").resize((px, px), Image.BILINEAR, reducing_gap=2.0)
    return md5, sha, np.asarray(im, dtype=np.uint8).copy()


def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path, rows, header=None):
    header = header or (list(rows[0].keys()) if rows else [])
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=header)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in header})


# ----------------------------------------------------------------------------- perceptual hashes
_POP8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)


def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return _POP8[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)


def _gray(x):
    x = np.asarray(x, np.float32)
    return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]


def _adaptive_mean(g, oh, ow):
    """Same bins as torch.nn.functional.adaptive_avg_pool2d.  g: [N, H, W]."""
    n, h, w = g.shape
    hs = np.array([(i * h) // oh for i in range(oh)])
    he = np.array([-(-((i + 1) * h) // oh) for i in range(oh)])
    ws = np.array([(j * w) // ow for j in range(ow)])
    we = np.array([-(-((j + 1) * w) // ow) for j in range(ow)])
    c = np.zeros((n, h + 1, w + 1), np.float64)
    c[:, 1:, 1:] = g.astype(np.float64).cumsum(1).cumsum(2)
    tot = c[:, he][:, :, we] - c[:, hs][:, :, we] - c[:, he][:, :, ws] + c[:, hs][:, :, ws]
    return (tot / ((he - hs)[:, None] * (we - ws)[None, :])).astype(np.float32)


def perceptual_hashes(arr, bs=256):
    """pHash (32x32 DCT, 8x8 low band vs median) and dHash (9x8 gradient) as uint64 arrays."""
    from scipy.fft import dctn
    n = len(arr)
    ph, dh = np.zeros(n, np.uint64), np.zeros(n, np.uint64)
    w64 = (1 << np.arange(64, dtype=np.uint64)).astype(np.uint64)
    for s in range(0, n, bs):
        g = _gray(arr[s:s + bs])
        g32 = _adaptive_mean(g, 32, 32)
        c = dctn(g32, type=2, norm="ortho", axes=(1, 2))[:, :8, :8].reshape(len(g32), 64)
        med = np.median(c[:, 1:], axis=1, keepdims=True)
        ph[s:s + bs] = ((c > med).astype(np.uint64) * w64).sum(1).astype(np.uint64)
        g98 = _adaptive_mean(g, 8, 9)
        bits = (g98[:, :, 1:] > g98[:, :, :-1]).reshape(len(g98), 64).astype(np.uint64)
        dh[s:s + bs] = (bits * w64).sum(1).astype(np.uint64)
    return ph, dh


def hamming_pairs(h, max_d=8, chunk=512):
    """All pairs i < j with Hamming distance <= max_d."""
    h = np.asarray(h, np.uint64)
    I, J, Dd = [], [], []
    for s in range(0, len(h), chunk):
        d = popcount64(h[s:s + chunk, None] ^ h[None, :])
        ii, jj = np.nonzero(d <= max_d)
        keep = jj > ii + s
        ii, jj = ii[keep], jj[keep]
        I.append(ii + s)
        J.append(jj)
        Dd.append(d[ii, jj])
    if not I:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0, int)
    return np.concatenate(I), np.concatenate(J), np.concatenate(Dd)


def gray_small(arr, px=128):
    return _adaptive_mean(_gray(arr), px, px)


# ----------------------------------------------------------------------------- grouping
def union_labels(n, pairs=(), label_arrays=()):
    """Connected components over n records: explicit pairs plus 'same value in any label array' (-1 = none)."""
    p = np.arange(n)

    def find(a):
        r = a
        while p[r] != r:
            r = p[r]
        while p[a] != r:
            p[a], a = r, p[a]
        return r

    def union(a, b):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb:
            lo, hi = (ra, rb) if ra < rb else (rb, ra)
            p[hi] = lo

    for a, b in pairs:
        union(a, b)
    for lab in label_arrays:
        first = {}
        for i in range(n):
            v = lab[i]
            if v is None or v == -1 or v == "":
                continue
            if v in first:
                union(i, first[v])
            else:
                first[v] = i
    return np.array([find(i) for i in range(n)])


def near_duplicate_groups(md5, ph, grays, rule=None):
    """Group records that are identical files (MD5) or near duplicates (pHash <= phash_max AND SSIM >= ssim_min).
    If the largest group exceeds max(cap_min, cap_frac * n) the stricter fallback rule is used.
    Returns (group id per record, info dict with pair counts)."""
    from skimage.metrics import structural_similarity
    rule = dict(NEAR_RULE, **(rule or {}))
    n = len(md5)
    ii, jj, dd = hamming_pairs(ph, max(rule["phash_max"], rule["fallback_phash"]))
    ss = np.array([structural_similarity(grays[a], grays[b], data_range=255.0) for a, b in zip(ii, jj)])
    by = defaultdict(list)
    for i, m in enumerate(md5):
        by[m].append(i)
    ex = [(g[0], x) for g in by.values() if len(g) > 1 for x in g[1:]]
    info = dict(exact_groups=sum(len(g) > 1 for g in by.values()), exact_redundant=sum(len(g) - 1 for g in by.values()),
                pairs_phash_le={t: int((dd <= t).sum()) for t in (0, 4, 8)},
                pairs_ssim_ge={t: int((ss >= t).sum()) for t in (0.90, 0.95)})
    m = (dd <= rule["phash_max"]) & (ss >= rule["ssim_min"]) if len(dd) else np.zeros(0, bool)
    roots = union_labels(n, pairs=list(zip(ii[m], jj[m])) + ex)
    big = max(Counter(roots.tolist()).values()) if n else 1
    cap = max(rule["cap_min"], rule["cap_frac"] * n)
    info["rule"] = "main"
    if big > cap:
        m = (dd <= rule["fallback_phash"]) & (ss >= rule["fallback_ssim"])
        roots = union_labels(n, pairs=list(zip(ii[m], jj[m])) + ex)
        info["rule"] = f"fallback (largest group {big} > cap {cap:.0f})"
    info["near_pairs"] = int(m.sum())
    info["redundant"] = int(n - len(set(roots.tolist())))
    return roots, info


def otsu(v, bins=256):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 4 or v.max() - v.min() < 1e-9:
        return float(np.median(v)) if len(v) else 0.5
    hist, edges = np.histogram(v, bins=bins)
    cen = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    s0 = np.cumsum(hist * cen)
    mu0 = s0 / np.maximum(w0, 1)
    mu1 = (s0[-1] - s0) / np.maximum(w1, 1)
    k = int(np.argmax((w0 * w1 * (mu0 - mu1) ** 2)[:-1]))
    return float(edges[k + 1])


def segment_order(order, labels, emb):
    """Recover source groups from file order: cut an ordered list wherever the class changes or the cosine
    similarity of consecutive L2-normalised embeddings drops below an Otsu threshold."""
    order = np.asarray(order)
    labels = np.asarray(labels)
    e = np.asarray(emb, np.float32)[order]
    s = (e[:-1] * e[1:]).sum(1)
    same = labels[order][:-1] == labels[order][1:]
    tau = otsu(s[same])
    seg_pos = np.concatenate([[0], np.cumsum((~same) | (s < tau))])
    seg = np.full(len(labels), -1)
    seg[order] = seg_pos
    return seg, dict(tau=tau, n_segments=int(seg_pos.max() + 1))


# ----------------------------------------------------------------------------- splits and checks
def grouped_split(labels, groups, k=5, seed=7):
    """Stratified group k-fold: fold id per record, no group in two folds."""
    from sklearn.model_selection import StratifiedGroupKFold
    labels, groups = np.asarray(labels), np.asarray(groups)
    fold = np.full(len(labels), -1)
    for f, (_, te) in enumerate(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(
            np.zeros(len(labels)), labels, groups)):
        fold[te] = f
    return fold


def _col(rows, key):
    return [r.get(key, "") for r in rows]


def check_split(files, roles, train_roles=("train",), test_roles=("test",),
                keys=("md5", "g_near", "g_struct", "g_ctrl", "parent_group")):
    """Cross-partition checks.  files: list of dicts (relpath + key columns).  roles: dict relpath -> role.
    Returns a list of (key, number of test images whose key value also occurs in training); empty = clean."""
    tr = [r for r in files if roles.get(r["relpath"]) in train_roles]
    te = [r for r in files if roles.get(r["relpath"]) in test_roles]
    out = []
    for k in keys:
        if not files or k not in files[0]:
            continue
        s = {v for v in _col(tr, k) if v not in ("", "-1")}
        bad = sum(1 for v in _col(te, k) if v in s)
        if bad:
            out.append((k, bad))
    return out


def leak_report(files, roles, train_roles=("train",), test_roles=("test",)):
    """Provenance leakage score: share of test images with a relative in training, by relation type."""
    tr = [r for r in files if roles.get(r["relpath"]) in train_roles]
    te = [r for r in files if roles.get(r["relpath"]) in test_roles]
    if not te:
        return {}
    rep = {}
    for name, k in (("identical_file", "md5"), ("near_duplicate", "g_near"), ("same_slide_patient_source", "g_struct"),
                    ("any_relative", "g_ctrl")):
        if k in te[0]:
            s = {v for v in _col(tr, k) if v not in ("", "-1")}
            rep[name] = float(np.mean([v in s for v in _col(te, k)]))
    rep["n_train"], rep["n_test"] = len(tr), len(te)
    return rep


# ----------------------------------------------------------------------------- folder audit
def audit_folder(root, label_from_parent=True, px=STORE_PX, rule=None, log=print):
    """Hash and group every image under root.  The class label is the name of the parent folder."""
    paths = sorted(os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs
                   if os.path.splitext(f)[1].lower() in IMG_EXT)
    if not paths:
        raise SystemExit(f"no images under {root}")
    md5, sha, arr, bad = [], [], np.zeros((len(paths), px, px, 3), np.uint8), []
    for i, p in enumerate(paths):
        try:
            a, b, x = load_image(p, px)
            md5.append(a); sha.append(b); arr[i] = x
        except Exception as e:
            md5.append(f"unreadable-{i}"); sha.append(""); bad.append((p, str(e)[:80]))
        if log and (i + 1) % 2000 == 0:
            log(f"  decoded {i + 1}/{len(paths)}")
    ph, dh = perceptual_hashes(arr)
    grays = gray_small(arr)
    near, info = near_duplicate_groups(md5, ph, grays, rule)
    exact = union_labels(len(paths), label_arrays=[md5])
    rows = []
    for i, p in enumerate(paths):
        rows.append(dict(relpath=os.path.relpath(p, root), label=os.path.basename(os.path.dirname(p)) if label_from_parent else "",
                         md5=md5[i], sha256=sha[i], phash=f"{int(ph[i]):016x}", dhash=f"{int(dh[i]):016x}",
                         g_exact=int(exact[i]), g_near=int(near[i])))
    info.update(n=len(paths), unreadable=len(bad))
    return rows, info
''',
    'pyproject.toml': r'''[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "oralleak"
version = "0.1.0"
description = "Data-leakage audit for medical image classification datasets: hashing, near duplicates, group recovery, split checks"
requires-python = ">=3.9"
dependencies = ["numpy", "scipy", "pillow", "scikit-image", "scikit-learn"]

[tool.setuptools]
packages = ["oralleak"]
''',
    'tests/test_oralleak.py': r'''"""pytest tests for oralleak.  The last two tests also check the released splits when they are present."""
import glob
import os

import numpy as np
import pytest

import oralleak as ol

HERE = os.path.dirname(os.path.abspath(__file__))
SPLITS = os.path.join(HERE, "..", "..", "splits")
# relations that must never cross partitions in the released splits (see splits/README.md)
KEYS = ("md5", "g_near", "g_struct", "parent_group")


def _img(seed, size=160):
    g = np.random.default_rng(seed)
    base = g.integers(0, 255, (8, 8, 3)).astype(np.uint8)
    return np.kron(base, np.ones((size // 8, size // 8, 1), np.uint8))


@pytest.fixture()
def folder(tmp_path):
    from PIL import Image
    for c in ("a", "b"):
        (tmp_path / c).mkdir()
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1.png")
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1_copy.png")                  # identical file content
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1_q90.jpg", quality=90)       # re-encoded near duplicate
    Image.fromarray(_img(2)).save(tmp_path / "a" / "x2.png")
    Image.fromarray(_img(3)).save(tmp_path / "b" / "y1.png")
    Image.fromarray(_img(4)).save(tmp_path / "b" / "y2.png")
    return tmp_path


def test_audit_groups(folder):
    rows, info = ol.audit_folder(str(folder), log=None)
    g = {r["relpath"]: r["g_near"] for r in rows}
    assert g["a/x1.png"] == g["a/x1_copy.png"] == g["a/x1_q90.jpg"]
    assert len({g["a/x2.png"], g["b/y1.png"], g["b/y2.png"], g["a/x1.png"]}) == 4
    assert info["exact_groups"] == 1 and info["redundant"] == 2


def test_check_flags_leak_and_passes_clean(folder):
    rows, _ = ol.audit_folder(str(folder), log=None)
    for r in rows:
        r["g_ctrl"] = r["g_near"]
    leaky = {r["relpath"]: "train" for r in rows}
    leaky["a/x1_copy.png"] = "test"
    assert ol.check_split(rows, leaky), "a duplicate across partitions must be reported"
    assert ol.leak_report(rows, leaky)["identical_file"] == 1.0
    fold = ol.grouped_split([r["label"] for r in rows], [r["g_ctrl"] for r in rows], k=2, seed=0)
    clean = {r["relpath"]: ("test" if f == 0 else "train") for r, f in zip(rows, fold)}
    assert ol.check_split(rows, clean) == []


def test_segment_order_recovers_blocks():
    g = np.random.default_rng(0)
    centers = g.normal(size=(6, 32))
    emb = np.concatenate([c + 0.05 * g.normal(size=(10, 32)) for c in centers])
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    seg, info = ol.segment_order(np.arange(60), np.zeros(60, int), emb)
    assert info["n_segments"] == 6
    assert all(len(set(seg[i * 10:(i + 1) * 10])) == 1 for i in range(6))


@pytest.mark.skipif(not glob.glob(os.path.join(SPLITS, "*_files.csv")), reason="released splits not present")
def test_released_holdouts_are_clean():
    for fcsv in sorted(glob.glob(os.path.join(SPLITS, "*_files.csv"))):
        ds = os.path.basename(fcsv)[:-len("_files.csv")]
        files = ol.read_csv(fcsv)
        for scsv in sorted(glob.glob(os.path.join(SPLITS, f"{ds}_controlled_*.csv"))):
            rows = ol.read_csv(scsv)
            for key in sorted({(r["rep"], r["outer"]) for r in rows}):
                roles = {r["relpath"]: r["role"] for r in rows if (r["rep"], r["outer"]) == key}
                assert ol.check_split(files, roles, ("train", "ival", "own"), ("hold",), KEYS) == [], (scsv, key)


@pytest.mark.skipif(not glob.glob(os.path.join(SPLITS, "*_files.csv")), reason="released splits not present")
def test_released_controlled_arms_are_clean():
    for scsv in sorted(glob.glob(os.path.join(SPLITS, "*_controlled_*.csv"))):
        ds = os.path.basename(scsv).split("_controlled_")[0]
        files = ol.read_csv(os.path.join(SPLITS, f"{ds}_files.csv"))
        rows = ol.read_csv(scsv)
        for key in sorted({(r["rep"], r["outer"]) for r in rows}):
            roles = {r["relpath"]: r["role"] for r in rows if (r["rep"], r["outer"]) == key}
            assert ol.check_split(files, roles, ("train",), ("own", "ival"), KEYS) == [], (scsv, key)
''',
}

t_all = time.time()
P = Q.plan()
BB0 = P["primary_backbone"]
Q.hdr("BLOCK 28 — RELEASE PACKAGE + COMPUTATIONAL STATUS")
stamp = time.strftime("%Y%m%d_%H%M", time.gmtime())
PKG = Q.D["release"] / "package"
if PKG.exists():
    shutil.rmtree(PKG)                     # the package folder is regenerated from q1/ every time
for sub in ("toolkit", "splits", "manifests", "results/eval", "results/tables", "results/figures", "plan", "code"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)
jobs = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
R = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
CL = Q.jload(Q.D["eval"] / "clinical.json", {}) or {}
AU = Q.jload(Q.D["index"] / "audit.json", {}) or {}
avail = Q.available_datasets()
if not avail:
    raise SystemExit("no dataset index found - run BLOCK 16 first")


def wcsv(path, header, rows):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


# ------------------------------------------------------------------ 1  toolkit
Q.sub("1  AUDIT TOOLKIT (oralleak): hashing · near duplicates · group recovery · split checks · leakage report")
for rel, src in TOOLKIT_SRC.items():
    f = PKG / "toolkit" / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(src)
sys.path.insert(0, str(PKG / "toolkit"))
for m in [k for k in list(sys.modules) if k == "oralleak" or k.startswith("oralleak.")]:
    del sys.modules[m]
import oralleak as OL
Q.kv("package", f"oralleak {OL.__version__} · {len(TOOLKIT_SRC)} files in release/package/toolkit")
Q.kv("command line", "python -m oralleak audit | split | check   (see toolkit/README.md)")

# ------------------------------------------------------------------ 2  toolkit agrees with the pipeline?
Q.sub("2  TOOLKIT vs PIPELINE: does the stand-alone code reproduce the hashes and the near-duplicate groups of BLOCK 16?")
rows, AGREE = [], {}
for ds in sorted(avail, key=lambda d: Q.DS.get(d).n):
    dd = Q.DS.get(ds)
    st = Q.open_store(ds)
    H = np.load(Q.D["store"] / f"{ds}_hash.npz")
    use = np.nonzero(dd.ok & dd.orig)[0]
    ph, dh, gr = [], [], []
    for s in range(0, len(use), 512):
        blk = np.asarray(st[use[s:s + 512]])
        a, b = OL.perceptual_hashes(blk)
        ph.append(a); dh.append(b)
    ph, dh = np.concatenate(ph), np.concatenate(dh)
    eq_p = float((ph == H["ph"][use]).mean())
    eq_d = float((dh == H["dh"][use]).mean()) if "dh" in H.files else float("nan")
    ncand = len(OL.hamming_pairs(ph, 8)[0])
    verdict, red = f"skipped ({ncand:,} candidate pairs)", "—"
    if ncand <= AGREE_MAX_PAIRS:
        gr = np.concatenate([OL.gray_small(np.asarray(st[use[s:s + 512]])) for s in range(0, len(use), 512)])
        roots, info = OL.near_duplicate_groups(dd.md5[use], ph, gr)
        g = dd.g_near[use]
        same = len(set(zip(roots.tolist(), g.tolist()))) == len(set(roots.tolist())) == len(set(g.tolist()))
        verdict = "identical partition" if same else "DIFFERENT partition"
        red = f"{info['redundant']} / {len(use) - len(set(g.tolist()))}"
        AGREE[ds] = dict(phash_equal=eq_p, dhash_equal=eq_d, partition_identical=bool(same), rule=info["rule"],
                         redundant_toolkit=info["redundant"], redundant_pipeline=int(len(use) - len(set(g.tolist()))))
    else:
        AGREE[ds] = dict(phash_equal=eq_p, dhash_equal=eq_d, partition_identical=None, candidate_pairs=ncand)
    rows.append([ds, len(use), f"{100 * eq_p:.2f}%", f"{100 * eq_d:.2f}%", verdict, red])
Q.table(["dataset", "originals", "pHash equal", "dHash equal", "near-duplicate groups", "redundant toolkit/pipeline"], rows,
        [12, 10, 12, 12, 34, 26])

# ------------------------------------------------------------------ 3  file manifests with SHA-256
Q.sub("3  FILE MANIFESTS AND HASH LISTS  (relative path, class, MD5, SHA-256, every group label used by the experiments)")
cache_p = Q.D["release"] / "sha256_cache.json"
cache = Q.jload(cache_p, {}) or {}


def sha_one(p):
    try:
        s_ = os.stat(p)
        key = f"{p}|{s_.st_size}|{int(s_.st_mtime)}"
        if key in cache:
            return key, cache[key]
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(1 << 20), b""):
                h.update(b)
        return key, h.hexdigest()
    except Exception:
        return None, ""


FILES, REL = {}, {}
rows = []
t0 = time.time()
for ds in avail:
    dd = Q.DS.get(ds)
    paths = [r["path"] for r in dd.R]
    with ThreadPoolExecutor(16) as ex:
        res = list(ex.map(sha_one, paths))
    for k, h in res:
        if k:
            cache[k] = h
    rel = [os.path.relpath(p, Q.INPUT) for p in paths]
    REL[ds] = rel
    cls = Q.META[ds]["classes"]
    F_ = []
    for i, r in enumerate(dd.R):
        par = int(dd.parent[i])
        F_.append(dict(relpath=rel[i], label=cls[int(dd.y[i])], label_id=int(dd.y[i]), md5=dd.md5[i], sha256=res[i][1],
                       readable=int(dd.ok[i]), original=int(dd.orig[i]), used_in_experiments=int(dd.base[i]),
                       parent_relpath=rel[par] if par >= 0 and dd.child_ok[i] else "", parent_group=int(dd.parent_link[i]),
                       g_exact=int(dd.g_exact[i]), g_near=int(dd.g_near[i]), g_struct=int(dd.g_struct[i]),
                       g_coarse=int(dd.g_coarse[i]), g_ctrl=int(dd.g_ctrl[i]), slide_or_patient=dd.slide[i],
                       native_split=r.get("native", "")))
    FILES[ds] = F_
    OL.write_csv(PKG / "splits" / f"{ds}_files.csv", F_)
    rows.append([ds, len(F_), int(dd.base.sum()), len(set(dd.g_ctrl[dd.base].tolist())), sum(1 for x in F_ if x["sha256"])])
Q.jsave(cache_p, cache)
Q.table(["dataset", "files", "used", "groups (g_ctrl)", "SHA-256 computed"], rows, [12, 8, 8, 16, 18])
Q.kv("hashing time", Q.hms(time.time() - t0))
cx = AU.get("cross")
if cx:
    Q.jsave(PKG / "manifests" / "cross_dataset_near_duplicates.json", cx)

# ------------------------------------------------------------------ 4  fixed leakage-controlled splits
Q.sub("4  FIXED LEAKAGE-CONTROLLED SPLITS  (controlled arm C of E1, seed 0, every outer fold) + independent toolkit checks")
KEYS = ("md5", "g_near", "g_struct", "parent_group")
rows, SPL = [], {}
for ds in avail:
    dd = Q.DS.get(ds)
    sch = P["outer"][ds]
    rel = REL[ds]
    fold_rows, long_rows = [], []
    viol_h = viol_c = 0
    leak_any = []
    nsplit = 0
    for rep in P["reps"].get(ds, P["reps"]["default"]):
        fold, _ = Q.outer_folds(dd, dict(sch, rep=rep))
        fold_rows += [[rel[i], rep, int(fold[i])] for i in np.nonzero(fold >= 0)[0]]
        for o in range(Q.n_outer(dd, dict(sch, rep=rep))):
            J = [j for j in jobs if j["spec"]["exp"] == "E1" and j["spec"]["ds"] == ds and j["spec"]["arm"] == "C"
                 and j["spec"].get("rep", 0) == rep and j["spec"]["outer"] == o and j["spec"]["seed"] == 0
                 and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std"]
            if not J:
                continue
            B = Q.build_job(dict(J[0]["spec"], xeval=False), check=True)   # the pipeline's own disjointness assertions
            roles = {}
            for part in ("train", "ival", "own", "hold"):
                for i in np.unique(B[part]["row"]):
                    roles[rel[i]] = part
                    long_rows.append([rel[i], rep, o, part])
            nsplit += 1
            viol_h += len(OL.check_split(FILES[ds], roles, ("train", "ival", "own"), ("hold",), KEYS))
            viol_c += len(OL.check_split(FILES[ds], roles, ("train",), ("own", "ival"), KEYS))
            lk = OL.leak_report(FILES[ds], roles, ("train",), ("own",))
            leak_any.append(lk.get("same_slide_patient_source", 0.0))
    wcsv(PKG / "splits" / f"{ds}_outer_folds.csv", ["relpath", "rep", "holdout_fold"], fold_rows)
    wcsv(PKG / "splits" / f"{ds}_controlled_E1_seed0.csv", ["relpath", "rep", "outer", "role"], long_rows)
    SPL[ds] = dict(splits=nsplit, holdout_violations=viol_h, controlled_violations=viol_c,
                   max_same_group_share=float(max(leak_any)) if leak_any else None)
    rows.append([ds, sch["kind"], nsplit, viol_h, viol_c, f"{100 * max(leak_any):.1f}" if leak_any else "—"])
Q.table(["dataset", "outer scheme", "splits", "holdout violations", "train/own violations", "max own share with group-mate %"],
        rows, [12, 13, 8, 19, 21, 30])
print("  checked relations: identical file (MD5), near duplicate, slide/patient/source group, augmented copy of the same")
print("  source. Every count must be 0; the pipeline's own assertions (build_job check=True) also passed for every split.")
(PKG / "splits" / "README.md").write_text("\n".join([
    "# Fixed leakage-controlled splits", "",
    "No images are included. Paths are relative to the Kaggle input root; download each dataset from its original source.", "",
    "- `<dataset>_files.csv`: one row per file: class, MD5, SHA-256, and the group labels used by the experiments",
    "  (`g_exact` identical files, `g_near` near duplicates [pHash <= 8 and SSIM >= 0.90, or identical MD5],",
    "  `g_struct` slide / patient / recovered source group, `parent_group` publisher-augmented copy -> source image,",
    "  `g_ctrl` union of all relations). `used_in_experiments` = readable original that represents its identical-file group.",
    "- `<dataset>_outer_folds.csv`: the clean-holdout fold of each used image (`rep` = repetition of the outer split).",
    "- `<dataset>_controlled_E1_seed0.csv`: for every outer fold, the role of each image in the leakage-controlled arm:",
    "  `train`, `ival` (inner validation: model selection, early stopping, temperature, thresholds), `own` (evaluation fold",
    "  of the arm) and `hold` (clean holdout). Images that are not listed for a fold were purged (relatives of holdout",
    "  images) or removed by the per-class size matching.",
    "- Check any split with: `python -m oralleak check --files <dataset>_files.csv --split <split.csv>`.", ""]))

# ------------------------------------------------------------------ 5  runs, weights, environment
Q.sub("5  RUN MANIFEST, WEIGHTS MANIFEST, ENVIRONMENT")
mrows, wrows = [], []
codes, vers = Counter(), Counter()
overlap_bad, n_checked = 0, 0
for j in jobs:
    s = j["spec"]
    rp = Q.D["runs"] / j["id"] / "result.json"
    r = Q.jload(rp) if rp.exists() else None
    st_ = "done" if (r and r.get("status") == "done") else ("failed" if Q.fail_count(j["id"]) else "not run")
    if st_ == "done":
        codes[json.dumps(r.get("code", {}), sort_keys=True)] += 1
        vers[json.dumps(r.get("versions", {}), sort_keys=True)] += 1
        z = np.load(Q.D["runs"] / j["id"] / "preds.npz")
        iv = set(zip(z["ival_row"].tolist(), z["ival_copy"].tolist()))
        ev = set(zip(z["own_row"].tolist(), z["own_copy"].tolist())) | set(zip(z["hold_row"].tolist(), z["hold_copy"].tolist()))
        overlap_bad += int(bool(iv & ev))
        n_checked += 1
    mrows.append([j["id"], j["phase"], s["exp"], s["ds"], s["arm"], s["backbone"], s["recipe"], s["outer"], s.get("rep", 0),
                  s["seed"], s.get("dose"), st_, *(([f"{r['own_bal_acc']:.6f}", f"{r['hold_bal_acc']:.6f}", r["steps_done"],
                  r["best_step"], f"{r['T_ival']:.4f}", r["info"]["sizes"]["train"], r["info"]["sizes"]["ival"],
                  r["info"]["sizes"]["own"], r["info"]["sizes"]["hold"], r.get("img"), f"{r['time_s']:.0f}",
                  r.get("finished_utc"), (r.get("code") or {}).get("q1lib.py"), (r.get("code") or {}).get("q1worker.py")])
                  if st_ == "done" else [""] * 14), j["fp"]])
    wp = Q.D["runs"] / j["id"] / "weights.pt"
    if wp.exists():
        wrows.append([j["id"], f"runs/{j['id']}/weights.pt", f"{wp.stat().st_size / 1e6:.1f}", Q.sha256_file(wp)])
wcsv(PKG / "manifests" / "runs_manifest.csv",
     ["job_id", "phase", "exp", "dataset", "arm", "backbone", "recipe", "outer", "rep", "seed", "dose", "status", "own_bal_acc",
      "hold_bal_acc", "steps_done", "best_step", "temperature_ival", "n_train", "n_ival", "n_own", "n_hold", "img_px",
      "seconds", "finished_utc", "q1lib_hash", "q1worker_hash", "job_fingerprint"], mrows)
wcsv(PKG / "manifests" / "weights_manifest.csv", ["job_id", "file", "MB", "sha256"], wrows)
env = dict(python=platform.python_version(), platform=platform.platform(), run_versions=[json.loads(k) for k in vers],
           run_code_hashes=[json.loads(k) for k in codes], plan_sha256=(Q.jload(Q.D["plan"] / "plan.sha256.json") or {}).get("sha256"),
           lib_version=Q.LIB_VERSION)
try:
    import torch, timm
    env.update(torch=torch.__version__, timm=timm.__version__, cuda=torch.version.cuda,
               gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception:
    pass
Q.jsave(PKG / "manifests" / "environment.json", env)
st_count = Counter(m[11] for m in mrows)
Q.kv("jobs", f"{len(jobs)} · done {st_count.get('done', 0)} · failed {st_count.get('failed', 0)} · not run {st_count.get('not run', 0)}")
Q.kv("code versions among runs", f"{len(codes)} distinct (q1lib/q1worker hashes)" + ("" if len(codes) <= 1 else
     " — runs were trained with different code; see runs_manifest.csv"))
Q.kv("inner validation vs evaluation", f"{n_checked} runs checked · {overlap_bad} with an image unit in both (must be 0)")
Q.kv("weights", f"{len(wrows)} files listed in manifests/weights_manifest.csv" + (" (included in the zip)" if INCLUDE_WEIGHTS
     else " (not zipped; set INCLUDE_WEIGHTS = True to include them)"))

# ------------------------------------------------------------------ (copies) results, plan, code
for src_dir, dst in ((Q.D["eval"], "results/eval"), (Q.D["tables"], "results/tables"), (Q.D["figs"], "results/figures"),
                     (Q.D["plan"], "plan"), (Q.D["code"], "code")):
    if src_dir.exists():
        for f in sorted(src_dir.iterdir()):
            if f.is_file():
                shutil.copy2(f, PKG / dst / f.name)
for f in ("q1lib.py", "q1worker.py"):
    if (Q.D["lib"] / f).exists():
        shutil.copy2(Q.D["lib"] / f, PKG / "code" / f)
shutil.copy2(Q.D["index"] / "audit.json", PKG / "manifests" / "audit.json") if (Q.D["index"] / "audit.json").exists() else None
blocks_kept = sorted(p.name for p in (PKG / "code").glob("block*.py"))

# ------------------------------------------------------------------ (copies) leaderboard baseline on the controlled splits
E1 = R.get("E1", {})
lb = ["# Baseline on the fixed leakage-controlled splits", "",
      f"Model: {Q.BACKBONES[BB0][0]} ({BB0}), {P['img']} px, recipe 'std'. Balanced accuracy (%), mean over outer folds and "
      "seeds. 'Own fold' = the controlled arm's evaluation fold; 'clean holdout' = images whose slide / patient / source "
      "never entered development. Numbers are copied from results/eval/q1_results.json.", "",
      "| Dataset | Outer scheme | Runs | Own fold | Clean holdout | Inflation [95% CI] |", "|---|---|---|---|---|---|"]
for ds in avail:
    a = E1.get(ds, {}).get("arms", {}).get("C")
    if a:
        lb.append(f"| {Q.SHORT.get(ds, ds)} | {P['outer'][ds]['kind']} | {a['n_runs']} | {100 * a['own']:.2f} | {100 * a['hold']:.2f} | "
                  f"{100 * a['infl']:+.2f} [{100 * a['infl_ci'][0]:+.2f}, {100 * a['infl_ci'][1]:+.2f}] |")
(PKG / "LEADERBOARD.md").write_text("\n".join(lb) + "\n")

# ------------------------------------------------------------------ (copies) README, citation and licence placeholders
TODO = "[TO BE COMPLETED BY THE AUTHORS]"
(PKG / "README.md").write_text("\n".join([
    "# Leakage audit of public oral-cancer image datasets: toolkit, fixed splits and results", "",
    f"Package generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 28 of the analysis notebook.", "",
    f"Repository: {TODO}  ·  Archive DOI: {TODO} (no DOI has been minted)  ·  Code licence: {TODO}", "",
    "## Contents", "",
    "- `toolkit/` — `oralleak`, a stand-alone audit package (file hashing, pHash/dHash, SSIM-confirmed near duplicates,",
    "  source recovery from file order, grouped splits, cross-partition checks, provenance leakage report) with pytest tests.",
    "- `splits/` — per-file manifests with MD5/SHA-256 and group labels, clean-holdout folds and the fixed",
    "  leakage-controlled splits (no images; see `splits/README.md`).",
    "- `manifests/` — run manifest (every job, its split sizes, results and code hash), weights manifest, data audit,",
    "  environment.",
    "- `results/` — every table, figure and JSON file produced by BLOCKS 25-27.",
    "- `plan/` — the analysis plan with its local timestamp and SHA-256 (written before any training run).",
    "- `code/` — the pipeline library, the training worker and the source of each notebook block that was run.",
    "- `LEADERBOARD.md` — baseline results on the controlled splits. `FINAL_STATUS_compute.md` — what was and was not done.", "",
    "## Reproducing", "",
    "1. Attach the datasets listed in `splits/*_files.csv` (paths are relative to the Kaggle input root).",
    "2. Run the notebook blocks in the order given in `FINAL_STATUS_compute.md`.",
    "3. Check a split: `cd toolkit && pip install . && python -m oralleak check --files ../splits/<ds>_files.csv "
    "--split <split.csv>`; run the tests with `python -m pytest -q tests`.", "",
    "## Data", "",
    "Images are not redistributed. Each dataset keeps its own licence and terms of use, which the authors must confirm:",
    ""] + [f"- {Q.SHORT.get(ds, ds)}: licence {TODO}" for ds in avail] + [""]))
(PKG / "CITATION.cff").write_text("\n".join([
    "cff-version: 1.2.0", "message: If you use this toolkit or these splits, please cite the article.",
    f'title: "{TODO}"', "authors:", f'  - name: "{TODO}"', f'doi: "{TODO}"', f'repository-code: "{TODO}"', "version: 0.1.0",
    f"date-released: {time.strftime('%Y-%m-%d', time.gmtime())}", ""]))
(PKG / "LICENSE_NOTE.md").write_text("\n".join([
    "# Licence", "", f"Code licence: {TODO} (choose one, e.g. MIT or Apache-2.0, and replace this file with its text).",
    "Split files contain only relative paths, hashes and group labels derived from the public datasets; check that each",
    "dataset's terms allow publishing such derived metadata.", ""]))

# ------------------------------------------------------------------ 6  FINAL_STATUS_compute.md
Q.sub("6  DEFINITION-OF-DONE CHECKLIST (compute side) — also written to FINAL_STATUS_compute.md")
done_ph = Counter(j["phase"] for j in jobs if Q.job_done(j))
all_ph = Counter(j["phase"] for j in jobs)


def seeds_done(ds, arm):
    c = Counter((j["spec"]["outer"], j["spec"].get("rep", 0)) for j in jobs if Q.job_done(j) and j["spec"]["exp"] == "E1"
                and j["spec"]["ds"] == ds and j["spec"]["arm"] == arm and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std")
    return min(c.values()) if c else 0


bb_done = sorted({j["spec"]["backbone"] for j in jobs if Q.job_done(j) and j["spec"]["exp"] == "E1" and j["spec"]["recipe"] == "std"})
fam = {"convnext_tiny.fb_in22k_ft_in1k": "CNN", "resnet50.a1_in1k": "CNN", "efficientnet_b0.ra_in1k": "small CNN",
       "convnext_base.fb_in22k_ft_in1k": "CNN (large)", "vit_small_patch16_224.augreg_in21k_ft_in1k": "transformer"}
fams = sorted({fam.get(b, b) for b in bb_done})
probes_ok = bool(Q.jload(Q.D["eval"] / "probes.json", []))
min_seeds = min([seeds_done(ds, arm) for ds in avail for arm in ("L", "C")] or [0])
dose = R.get("dose", {})
dose_ds = sorted({k.split("|")[1] for k in dose})
reimp = R.get("reimp", {})
reimp_both = [k for k, v in reimp.items() if len(v) >= 2]
strata = R.get("provenance_strata", {})
strata_contrast = [d for d, v in strata.items() if "sibling_minus_none" in v]
h1 = {ds: E1[ds]["arms"]["L"].get("infl_p_holm") for ds in E1 if "L" in E1[ds].get("arms", {})}
plan_meta = Q.jload(Q.D["plan"] / "plan.sha256.json", {}) or {}
first_run = min([Q.jload(Q.D["runs"] / j["id"] / "result.json", {}).get("finished_utc", "9999") for j in jobs if Q.job_done(j)]
                or ["—"])
tests_file = PKG / "toolkit" / "tests" / "test_oralleak.py"


def item(name, status, evidence):
    return [name, status, evidence]


ITEMS = [
    item("Tier 0 errors fixed (213->313, Table 13, comparison counts, '31 studies', placeholders)", "WRITING PHASE",
         "new operating points: results/tables/operating_points.csv; text errata are fixed in the manuscript"),
    item("Selection and early stopping never use evaluation data",
         "DONE" if n_checked and overlap_bad == 0 else ("CHECK" if n_checked else "NOT RUN"),
         f"inner validation carved from training (plan.selection_rules); {n_checked} runs checked, {overlap_bad} overlaps"),
    item("Temperature scaling is fitted on disjoint data", "DONE" if R else "NOT RUN",
         "temperature from inner validation only (BLOCK 25 run_metrics, BLOCK 26); ECE/Brier on own fold and clean holdout"),
    item("Sensitivity and specificity defined consistently, Table 12 rebuilt",
         "DONE (numbers)" if (Q.D["tables"] / "published_vs_ours.csv").exists() else "NOT RUN",
         "positive-class columns separate from macro: results/tables/published_vs_ours.csv, E1_main.csv"),
    item("One evaluation mode per paired comparison", "DONE" if R else "NOT RUN",
         "every comparison = single models, own fold vs clean holdout of the same models; ensembles only in BLOCK 26, labelled"),
    item("Cluster-level CIs and tests, with Holm correction", "DONE" if h1 else "NOT RUN",
         "two-stage cluster bootstrap + within-fold bootstrap + mixed model + GEE; Holm over H1 (q1_results.json E1.*.arms.L)"),
    item("'Pre-specified' claims backed by a timestamp", "PARTIAL" if plan_meta else "NOT RUN",
         f"plan written {plan_meta.get('created_utc', '?')} (SHA-256 {str(plan_meta.get('sha256', '?'))[:12]}) before the "
         f"first finished run ({first_run}); a public timestamp (saved Kaggle version, Git tag or OSF) is human-only"),
    item("Matched-training control and, where possible, a factorial design",
         "DONE" if "factorial_clinical" in R else ("PARTIAL" if E1 else "NOT RUN"),
         "L and C matched per class and in steps (jobs.json Tc); factorial S x Dup on clinical: "
         + ("present" if "factorial_clinical" in R else "general phase not finished")),
    item("Provenance-stratified analysis (sibling vs no sibling)", "DONE" if strata else "NOT RUN",
         f"results/tables/provenance_strata.csv; sibling-vs-none contrast estimable for: {', '.join(strata_contrast) or 'none'}"
         + ("" if strata_contrast else " (no leaky own fold has both strata: under augment-before-split nearly every test image "
            "has a sibling in training, so the clean holdout carries the comparison)")),
    item("Dose-response curves on at least two datasets", "DONE" if len(dose_ds) >= 2 else ("PARTIAL" if dose_ds else "NOT RUN"),
         f"datasets with dose curves: {', '.join(dose_ds) or 'none'} (DOSEK / DUP / OVL, q1_results.json dose)"),
    item("Perceptual and embedding near-duplicate audit, with positive and negative controls",
         "DONE" if AU and "RL" in R and any(k.startswith("DUP|") for k in dose) else ("PARTIAL" if AU else "NOT RUN"),
         "index/audit.json (MD5, pHash, dHash, SSIM, embedding cosine, cross-dataset); positive control DUP; negative controls: "
         "augment-after-split arm C and random labels (RL)"),
    item("At least 3 seeds and at least 3 architecture families",
         "DONE" if min_seeds >= 3 and "transformer" in fams and probes_ok else ("PARTIAL" if min_seeds or bb_done else "NOT RUN"),
         f"minimum finished seeds per E1 fold: {min_seeds}; fine-tuned families: {', '.join(fams) or 'none'}; "
         f"frozen DINOv2 linear probe: {'yes' if probes_ok else 'no'}"),
    item("At least 2 additional datasets, one with patient IDs", "PARTIAL" if "breakhis" in avail else "BLOCKED",
         "BreakHis (non-oral, patient IDs) added" if "breakhis" in avail else "BreakHis not attached ('ambarish/breakhis')"),
    item("At least 2 published methods reimplemented under both protocols",
         "DONE" if len(reimp_both) >= 2 else ("PARTIAL" if reimp else "NOT RUN"), f"{', '.join(reimp_both) or 'none finished'}"),
    item("Literature analysis of every surveyed study, with the true study count", "WRITING PHASE", "not a compute task"),
    item("Audit toolkit, corrected splits, manifests and weights released with a DOI", "PARTIAL",
         "toolkit, splits and manifests in this zip; deposit and DOI are human-only; weights listed, not zipped by default"),
    item("Novelty claims verified against prior leakage literature", "WRITING PHASE", "not a compute task"),
    item("Abstract and conclusion rewritten to match the evidence", "WRITING PHASE", "uses results/ from this package"),
    item("CLAIM checklist completed and dataset licences checked", "WRITING PHASE / HUMAN", "licence terms are human-only"),
]
Q.table(["item", "status", "evidence"], [[i[0][:56], i[1], i[2][:56]] for i in ITEMS], [58, 23, 58])

# honest scale-down and surprises (generated from the numbers, not written by hand)
scale = [P.get("deviations", "")]
for ph in P["phases"]:
    if all_ph.get(ph) and done_ph.get(ph, 0) < all_ph[ph]:
        scale.append(f"phase '{ph}': {done_ph.get(ph, 0)} of {all_ph[ph]} jobs finished; the missing runs are reported as missing.")
if "histopath" in avail and not AU.get("histopath", {}).get("pre_enabled", False):
    scale.append("PRE experiment disabled: parent recovery of the publisher-augmented files was below the pre-set rate.")
if "breakhis" not in avail:
    scale.append("BreakHis not attached: no dataset with patient identifiers.")
surprises = []
for ds, p in h1.items():
    if p is not None and p >= 0.05:
        surprises.append(f"H1 not supported for {ds}: leaky-arm inflation {100 * E1[ds]['arms']['L']['infl']:+.2f} points, Holm p = {p:.3g}.")
for ds in E1:
    c = E1[ds]["arms"].get("C")
    if c and (c["infl_ci"][0] > 0 or c["infl_ci"][1] < 0):
        surprises.append(f"H3 fails for {ds}: the controlled arm is itself biased (inflation {100 * c['infl']:+.2f} "
                         f"[{100 * c['infl_ci'][0]:+.2f}, {100 * c['infl_ci'][1]:+.2f}]).")
    if E1[ds].get("dhold_ci") and (E1[ds]["dhold_ci"][0] > 0 or E1[ds]["dhold_ci"][1] < 0):
        surprises.append(f"H4 for {ds}: clean-holdout BA differs between leaky and controlled training "
                         f"({100 * E1[ds]['dhold']:+.2f} [{100 * E1[ds]['dhold_ci'][0]:+.2f}, {100 * E1[ds]['dhold_ci'][1]:+.2f}]).")
for k, v in R.get("RL", {}).items():
    if k.endswith("|L") and (v.get("p_vs_chance") or 1) >= 0.05:
        surprises.append(f"random labels, {k.split('|')[0]}: the leaky arm is not above chance on its own fold (p = {v['p_vs_chance']:.3g}).")
for k, v in dose.items():
    exn, _, arm_ = k.split("|")
    if v.get("slope") is None:
        continue
    if exn == "DOSEK" and arm_ == "C":        # augmenting after the split should NOT raise inflation
        if v["slope"] > 0 and (v.get("p") or 1) < 0.05:
            surprises.append(f"dose-response {k}: inflation rises with k even though augmentation happens after the split "
                             f"(slope {100 * v['slope']:+.2f}, p = {v['p']:.4f}).")
        continue
    if v["slope"] <= 0 or (v.get("p") or 1) >= 0.05:
        surprises.append(f"dose-response {k}: no clear increase (slope {100 * v['slope']:+.2f}, p = {v.get('p') if v.get('p') is None else round(v['p'], 4)}).")
rp = R.get("risk_predictor", {})
for name, v in rp.items():
    if isinstance(v, dict) and "lodo_r2" in v and name.startswith("one") and v["lodo_r2"] <= 0:
        surprises.append(f"leakage-risk predictor: leave-one-dataset-out R² = {v['lodo_r2']:.2f} (no predictive value across datasets).")
nw = [k for k, v in CL.get("saliency", {}).items() if not (v["del_gain_ci"][0] > 0 and v["ins_gain_ci"][0] > 0)]
if nw:
    surprises.append(f"Grad-CAM not better than random order for {', '.join(nw)}: those maps are illustration only.")
worse = [k for k, v in CL.get("calibration", {}).items() if v["cal"]["ece"] > v["raw"]["ece"]]
if worse:
    surprises.append(f"temperature scaling (fitted on inner validation) increased holdout ECE for {', '.join(worse)}.")
for ds, a in AGREE.items():
    if a.get("partition_identical") is False:
        surprises.append(f"toolkit and pipeline near-duplicate groups differ for {ds}.")
human = ["repository URL", "archive DOI (Zenodo or similar) — none minted", "code licence", "licence terms of every dataset",
         "public timestamp of the analysis plan (saved Kaggle version, Git tag or OSF registration)", "funding statement",
         "author contributions", "ethics / data-provenance statement sign-off", "citation metadata (CITATION.cff)"]
order = ("First session: 15 -> Save Version -> 16 -> 17 -> 18 -> 19 -> 20 -> 21 -> 22.  Later sessions: 15 -> 19 -> next "
         "training block (23, then 24; re-run a block to resume it).  Final session: 15 -> 25 -> 26 -> 27 -> 28.")
md = ["# FINAL_STATUS (computational part)", "",
      f"Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 28 from the files in q1/. "
      f"Plan SHA-256 `{plan_meta.get('sha256', '?')}`. Jobs: {len(jobs)}, finished {st_count.get('done', 0)}, "
      f"failed {st_count.get('failed', 0)}, not run {st_count.get('not run', 0)}.", "",
      "## Definition of done", "", "| Item | Status | Evidence |", "|---|---|---|"]
md += [f"| {a} | {b} | {c} |" for a, b, c in ITEMS]
md += ["", "## Jobs by phase", "", "| Phase | Finished | Total |", "|---|---|---|"]
md += [f"| {ph} | {done_ph.get(ph, 0)} | {all_ph.get(ph, 0)} |" for ph in P["phases"] if all_ph.get(ph)]
md += ["", "## Scaled down or skipped", ""] + [f"- {x}" for x in scale if x]
md += ["", "## Results that go against the original thesis or the plan's expectations", ""]
md += [f"- {x}" for x in surprises] or ["- none detected automatically; read BLOCK 25-26 outputs before writing"]
md += ["", "## Toolkit agreement with the pipeline", ""]
md += [f"- {ds}: pHash equal {100 * a['phash_equal']:.2f}%, near-duplicate partition "
       f"{'identical' if a.get('partition_identical') else ('different' if a.get('partition_identical') is False else 'not re-derived (size)')}"
       for ds, a in AGREE.items()]
md += ["", "## Human-only items left open", ""] + [f"- {x}" for x in human]
md += ["", "## Block order used", "", order, "", f"Blocks whose source is stored in code/: {', '.join(blocks_kept) or 'none'} "
       "(a block's source is stored when it runs; re-run a block to store it)."]
(PKG / "FINAL_STATUS_compute.md").write_text("\n".join(md) + "\n")
if surprises:
    print("  results against expectations (all listed in FINAL_STATUS_compute.md):")
    for x in surprises[:12]:
        print("   -", x)
    if len(surprises) > 12:
        print(f"   ... and {len(surprises) - 12} more")

# ------------------------------------------------------------------ 7  toolkit tests on the packaged splits
Q.sub("7  TOOLKIT TESTS  (pytest on the package, including every released split)")
try:
    import pytest  # noqa
    pt = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=PKG / "toolkit",
                        capture_output=True, text=True, timeout=1800)
    last = [l for l in pt.stdout.strip().splitlines() if l.strip()][-1:] or ["(no output)"]
    Q.kv("pytest", last[0] + ("" if pt.returncode == 0 else "  <- FAILED, see release/pytest.log"))
    (Q.D["release"] / "pytest.log").write_text(pt.stdout + pt.stderr)
    TEST_OK = pt.returncode == 0
except ImportError:
    Q.kv("pytest", "not installed; the split checks in section 4 are the test result")
    TEST_OK = all(v["holdout_violations"] == 0 and v["controlled_violations"] == 0 for v in SPL.values())

# ------------------------------------------------------------------ 8  zip
Q.sub("8  ZIP")
zp = Q.D["release"] / f"oralleak_release_{stamp}.zip"
n_files = 0
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(PKG.rglob("*")):
        if f.is_file() and "__pycache__" not in f.parts:
            zf.write(f, f"oralleak_release/{f.relative_to(PKG)}")
            n_files += 1
    if INCLUDE_WEIGHTS:
        for w in wrows:
            zf.write(Q.D["runs"] / w[0] / "weights.pt", f"oralleak_release/weights/{w[0]}.pt")
            n_files += 1
Q.kv("archive", f"{zp.relative_to(Q1_ROOT.parent)} · {n_files} files · {zp.stat().st_size / 1e6:.1f} MB")
Q.kv("status file", "release/package/FINAL_STATUS_compute.md")
Q.kv("nothing uploaded", "the zip stays in /kaggle/working; publishing and a DOI are left to the authors")
print(f"\n[BLOCK 28 ✓]  release package built · toolkit tests {'passed' if TEST_OK else 'FAILED'} · {Q.hms(time.time() - t_all)}")

In [ ]:
# ============================================================================
# BLOCK 25 — EVALUATION + STATISTICS (all numbers for the rewritten Results section)
# ============================================================================
import os, sys, json, time, math, importlib, warnings, csv
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("25")

NB = 10000          # bootstrap resamples (pre-specified)
t_all = time.time()
P = Q.plan()
BB0 = P["primary_backbone"]
rng = np.random.default_rng(20260930)
Q.hdr("BLOCK 25 — EVALUATION + STATISTICS")
jobs = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
RUNS = {}
for j in jobs:
    if Q.job_done(j):
        r, z = Q.load_run(j["id"])
        if r is not None:
            RUNS[j["id"]] = (j, r, {k: z[k] for k in z.files})
Q.kv("finished runs", f"{len(RUNS)} of {len(jobs)} jobs")
if not RUNS:
    raise SystemExit("no finished runs yet")
OUT = dict(meta=dict(n_runs=len(RUNS), n_jobs=len(jobs), boot=NB, plan_sha=(Q.jload(Q.D["plan"] / "plan.sha256.json") or {}).get("sha256")))
TAB = Q.D["tables"]


def write_csv(name, header, rows):
    with open(TAB / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


def sel(**kw):
    out = []
    for j, r, z in RUNS.values():
        s = j["spec"]
        if all(s.get(k) == v for k, v in kw.items()):
            out.append((j, r, z))
    return out


def correct(z, name):
    return (z[f"{name}_logit"].argmax(1) == z[f"{name}_y"]).astype(float)


def run_metrics(j, r, z):
    """Per-run metrics with the temperature and thresholds fitted on the inner validation split only."""
    K = r["info"]["K"]
    pos = Q.META[j["spec"]["ds"]]["pos"]
    T = Q.fit_temperature(z["ival_logit"], z["ival_y"])
    out = dict(T=T)
    for nm in ("own", "hold"):
        lg, y = z[f"{nm}_logit"], z[f"{nm}_y"]
        if len(y) == 0:
            continue
        out[nm] = Q.metrics(Q.softmax(lg, T), y, K, pos)
        out[nm + "_raw_ece"] = Q.ece_score(Q.softmax(lg), y)
        out[nm + "_raw_brier"] = float(np.mean(((Q.softmax(lg) - np.eye(K)[y]) ** 2).sum(1)))
    m = z["own_copy"] == 0
    if m.any() and (~m).any():
        out["own_orig"] = Q.metrics(Q.softmax(z["own_logit"][m], T), z["own_y"][m], K, pos)
    pi = Q.softmax(z["ival_logit"], T)[:, pos]
    yi = (z["ival_y"] == pos).astype(int)
    th = Q.thresholds_from(pi, yi)
    ph = Q.softmax(z["hold_logit"], T)[:, pos]
    yh = (z["hold_y"] == pos).astype(int)
    out["op"] = {k: Q.op_point(ph, yh, v) for k, v in th.items()}
    return out


RM = {rid: run_metrics(j, r, z) for rid, (j, r, z) in RUNS.items()}


def fold_key(s):
    return (s["outer"], s.get("rep", 0))


def tables_for(runs, part):
    """Per outer fold: cluster count tables summed over seeds (and the list of seeds)."""
    by = defaultdict(list)
    for j, r, z in runs:
        by[fold_key(j["spec"])].append((j, r, z))
    out = {}
    for fk, lst in by.items():
        ys, cs, ks = [], [], []
        for j, r, z in lst:
            ys.append(z[f"{part}_y"])
            cs.append(correct(z, part))
            ks.append(z[f"{part}_key"])
        K = lst[0][1]["info"]["K"]
        out[fk] = Q.cluster_table(np.concatenate(ys), np.concatenate(cs), np.concatenate(ks), K)
    return out


def paired_hold(runsA, runsB):
    """Per fold: shared cluster table n and correct counts of A and B on the same holdout images."""
    A, Bm = defaultdict(list), defaultdict(list)
    for j, r, z in runsA:
        A[fold_key(j["spec"])].append(z)
    for j, r, z in runsB:
        Bm[fold_key(j["spec"])].append(z)
    out = {}
    for fk in set(A) & set(Bm):
        za = {int(z0["hold_row"][0]) if len(z0["hold_row"]) else 0: z0 for z0 in A[fk]}
        y = A[fk][0]["hold_y"]
        key = A[fk][0]["hold_key"]
        K = int(max(y.max(), 1) + 1) if len(y) else 2
        ca = np.mean([correct(z, "hold") for z in A[fk]], 0)
        cb = np.mean([correct(z, "hold") for z in Bm[fk]], 0)
        n, c1 = Q.cluster_table(y, ca, key, K)
        _, c2 = Q.cluster_table(y, cb, key, K)
        out[fk] = (n, c1, c2)
    return out


def boot_folds(tabs, B, seed):
    """Within-fold cluster bootstrap: returns fold keys and matrix [F, B] of BA."""
    g = np.random.default_rng(seed)
    ks = sorted(tabs)
    return ks, np.stack([Q.boot_ba(tabs[k][0], tabs[k][1], B, g) for k in ks])


def point(tabs):
    return {k: float(Q.ba_counts(*tabs[k])) for k in tabs}


def two_stage(mat, B, seed):
    g = np.random.default_rng(seed)
    F = mat.shape[0]
    idx = g.integers(0, F, size=(B, F))
    return mat[idx, np.arange(B)[:, None]].mean(1)


def fmt(v, ci_=None, d=2):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    s = f"{100 * v:+.{d}f}" if ci_ is not None else f"{100 * v:.{d}f}"
    if ci_ is not None:
        s += f" [{100 * ci_[0]:+.{d}f}, {100 * ci_[1]:+.{d}f}]"
    return s


def pf(p):
    return "—" if p is None or not np.isfinite(p) else (f"{p:.1e}" if p < 1e-3 else f"{p:.4f}")


# ============================================================================ 1  E1 main experiment
Q.sub("1  E1 — OWN-FOLD vs CLEAN-HOLDOUT BALANCED ACCURACY  (primary backbone, all seeds, pooled over outer folds)")
E1 = {}
rows, csvrows = [], []
avail = [d for d in Q.DATASETS if sel(exp="E1", ds=d, backbone=BB0, recipe="std")]
for ds in avail:
    arms = P["exps"]["E1"]["datasets"][ds]["arms"]
    res = {}
    for arm in arms:
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std")
        if not R_:
            continue
        to, th = tables_for(R_, "own"), tables_for(R_, "hold")
        common = sorted(set(to) & set(th))
        to, th = {k: to[k] for k in common}, {k: th[k] for k in common}
        ko, mo = boot_folds(to, NB, Q.shash(ds, arm, "own") % 2 ** 32)
        kh, mh = boot_folds(th, NB, Q.shash(ds, arm, "hold") % 2 ** 32)
        po, ph_ = point(to), point(th)
        infl_pt = float(np.mean([po[k] - ph_[k] for k in common]))
        infl_w = (mo - mh).mean(0)                                        # within-fold (sensitivity)
        infl_b = Q.hier([mo - mh], NB, Q.shash(ds, arm, "hier") % 2 ** 32)[0]   # two-stage (primary)
        seeds = sorted({j["spec"]["seed"] for j, _, _ in R_})
        per_seed = []
        for sd in seeds:
            Rs = [x for x in R_ if x[0]["spec"]["seed"] == sd]
            a, b = point(tables_for(Rs, "own")), point(tables_for(Rs, "hold"))
            kk = set(a) & set(b)
            if kk:
                per_seed.append(float(np.mean([a[k] - b[k] for k in kk])))
        vals, outs, sds = [], [], []
        for j, r, z in R_:
            m = RM[j["id"]]
            if "own" in m and "hold" in m:
                vals.append(m["own"]["bal_acc"] - m["hold"]["bal_acc"])
                outs.append(f"{j['spec']['outer']}-{j['spec'].get('rep', 0)}")
                sds.append(j["spec"]["seed"])
        mix = Q.mixed_intercept(vals, outs, sds) if len(vals) >= 4 else None
        mean_m = lambda part, key, sub=None: float(np.nanmean([(RM[j["id"]][part][sub][key] if sub else RM[j["id"]][part][key])
                                                               for j, _, _ in R_ if part in RM[j["id"]] and (sub is None or sub in RM[j["id"]][part])
                                                               and (RM[j["id"]][part][sub] if sub else RM[j["id"]][part]).get(key) is not None])) \
            if R_ else float("nan")
        res[arm] = dict(n_runs=len(R_), folds=len(common), seeds=seeds, own=float(np.mean(list(po.values()))),
                        hold=float(np.mean(list(ph_.values()))), infl=infl_pt, infl_ci=Q.ci(infl_b), infl_p=Q.boot_p(infl_b),
                        infl_ci_within=Q.ci(infl_w), infl_p_within=Q.boot_p(infl_w), seed_infl=per_seed,
                        seed_sd=float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else None, mixed=mix,
                        own_acc=mean_m("own", "acc"), hold_acc=mean_m("hold", "acc"), own_auroc=mean_m("own", "auroc"),
                        hold_auroc=mean_m("hold", "auroc"), own_f1=mean_m("own", "macro_f1"), hold_f1=mean_m("hold", "macro_f1"),
                        hold_ece=mean_m("hold", "ece"), hold_brier=mean_m("hold", "brier"),
                        own_pos_sens=mean_m("own", "sens", "pos"), own_pos_spec=mean_m("own", "spec", "pos"),
                        own_pos_prec=mean_m("own", "prec", "pos"), own_pos_f1=mean_m("own", "f1", "pos"),
                        hold_pos_sens=mean_m("hold", "sens", "pos"), hold_pos_spec=mean_m("hold", "spec", "pos"),
                        hold_pos_prec=mean_m("hold", "prec", "pos"), hold_pos_f1=mean_m("hold", "f1", "pos"),
                        own_orig=float(np.nanmean([RM[j["id"]]["own_orig"]["bal_acc"] for j, _, _ in R_ if "own_orig" in RM[j["id"]]]))
                        if any("own_orig" in RM[j["id"]] for j, _, _ in R_) else None,
                        _mo=mo, _mh=mh, _k=common)
    if "L" in res and "C" in res:
        L, C = res["L"], res["C"]
        kk = sorted(set(L["_k"]) & set(C["_k"]))
        iL = [L["_k"].index(k) for k in kk]
        iC = [C["_k"].index(k) for k in kk]
        did_b = Q.hier([(L["_mo"] - L["_mh"])[iL] - (C["_mo"] - C["_mh"])[iC]], NB, Q.shash(ds, "did") % 2 ** 32)[0]
        ph_tab = paired_hold(sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std"),
                             sel(exp="E1", ds=ds, arm="C", backbone=BB0, recipe="std"))
        g = np.random.default_rng(Q.shash(ds, "paired") % 2 ** 32)
        dmat = np.stack([Q.boot_ba_paired(ph_tab[k][0], ph_tab[k][1], ph_tab[k][2], NB, g) for k in sorted(ph_tab)])
        dh = Q.hier([dmat], NB, Q.shash(ds, "dh") % 2 ** 32)[0]
        dh_pt = float(np.mean([Q.ba_counts(ph_tab[k][0], ph_tab[k][1]) - Q.ba_counts(ph_tab[k][0], ph_tab[k][2]) for k in ph_tab]))
        ca, cb, cl = [], [], []
        for fk in sorted(ph_tab):
            za = [z for j, r, z in sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std") if fold_key(j["spec"]) == fk]
            zb = [z for j, r, z in sel(exp="E1", ds=ds, arm="C", backbone=BB0, recipe="std") if fold_key(j["spec"]) == fk]
            for x, y_ in zip(za, zb):
                ca.append(correct(x, "hold"))
                cb.append(correct(y_, "hold"))
                cl.append(x["hold_key"] + 10 ** 7 * fk[1])
        gee = Q.gee_paired(np.concatenate(ca), np.concatenate(cb), np.concatenate(cl)) if ca else {}
        E1[ds] = dict(arms={a: {k: v for k, v in d.items() if not k.startswith("_")} for a, d in res.items()},
                      did=float(L["infl"] - C["infl"]), did_ci=Q.ci(did_b), did_p=Q.boot_p(did_b),
                      dhold=dh_pt, dhold_ci=Q.ci(dh), dhold_p=Q.boot_p(dh), gee=gee)
    else:
        E1[ds] = dict(arms={a: {k: v for k, v in d.items() if not k.startswith("_")} for a, d in res.items()})
    for arm, d in res.items():
        rows.append([ds, arm, f"{d['n_runs']}/{d['folds']}", fmt(d["own"]), fmt(d["hold"]),
                     fmt(d["infl"], d["infl_ci"]), pf(d["infl_p"]),
                     "—" if d["seed_sd"] is None else f"{100 * d['seed_sd']:.2f}"])
        csvrows.append([ds, arm, d["n_runs"], d["folds"], d["own"], d["hold"], d["infl"], *d["infl_ci"], d["infl_p"],
                        *d["infl_ci_within"], d["seed_sd"], d["own_acc"], d["hold_acc"], d["own_auroc"], d["hold_auroc"],
                        d["own_f1"], d["hold_f1"], d["hold_ece"], d["hold_brier"], d["own_pos_sens"], d["own_pos_spec"],
                        d["own_pos_prec"], d["own_pos_f1"], d["hold_pos_sens"], d["hold_pos_spec"], d["hold_pos_prec"],
                        d["hold_pos_f1"], d["own_orig"]])
Q.table(["dataset", "arm", "runs/folds", "own BA", "hold BA", "inflation [95% CI, two-stage]", "p", "seed SD"], rows,
        [12, 6, 11, 9, 9, 30, 10, 9])
write_csv("E1_main", ["dataset", "arm", "runs", "folds", "own_ba", "hold_ba", "inflation", "ci_lo_2stage", "ci_hi_2stage",
                      "p_2stage", "ci_lo_within", "ci_hi_within", "seed_sd", "own_acc", "hold_acc", "own_auroc", "hold_auroc", "own_macro_f1", "hold_macro_f1",
                      "hold_ece_cal", "hold_brier_cal", "own_pos_sens", "own_pos_spec", "own_pos_prec", "own_pos_f1",
                      "hold_pos_sens", "hold_pos_spec", "hold_pos_prec", "hold_pos_f1", "own_ba_originals_only"], csvrows)

# primary family: H1 with Holm
fam = [(ds, E1[ds]["arms"]["L"]["infl_p"]) for ds in avail if "L" in E1[ds]["arms"]]
if fam:
    adj = Q.holm([p for _, p in fam])
    Q.sub("2  PRIMARY FAMILY (H1): leaky-arm inflation > 0, Holm-adjusted across datasets")
    rows = []
    for (ds, p), pa in zip(fam, adj):
        a = E1[ds]["arms"]["L"]
        E1[ds]["arms"]["L"]["infl_p_holm"] = pa
        mx = a.get("mixed") or {}
        mixs = (f"{100 * mx['est']:+.2f} [{100 * mx['lo']:+.2f},{100 * mx['hi']:+.2f}]" + ("*" if mx.get("note") else "")
                if mx.get("lo") is not None else "—")
        rows.append([ds, fmt(a["infl"], a["infl_ci"]), fmt(a["infl"], a["infl_ci_within"]), mixs, pf(p), pf(pa),
                     "supported" if pa < 0.05 and a["infl"] > 0 else "not supported"])
    Q.table(["dataset", "inflation L [two-stage CI]", "[within-fold CI]", "mixed model [CI]", "p", "p Holm", "H1"], rows,
            [12, 28, 28, 24, 9, 9, 14])
    print("  p and Holm use the two-stage bootstrap (primary). * = mixed model not estimable (one run per fold, singular fit")
    print("  or no convergence), so a t-interval over fold-level means is shown instead.")

Q.sub("3  SECONDARY (H2-H4): difference-in-differences, controlled-arm bias, holdout L - C")
rows = []
for ds in avail:
    e = E1[ds]
    if "did" not in e:
        continue
    C = e["arms"]["C"]
    gee = e.get("gee", {})
    rows.append([ds, fmt(e["did"], e["did_ci"]), fmt(C["infl"], C["infl_ci"]), fmt(e["dhold"], e["dhold_ci"]), pf(e["dhold_p"]),
                 f"{gee.get('or_', float('nan')):.2f} [{gee.get('lo', float('nan')):.2f},{gee.get('hi', float('nan')):.2f}] p {pf(gee.get('p'))}"
                 if "or_" in gee else "—"])
Q.table(["dataset", "DiD (H2) [CI]", "bias of C (H3) [CI]", "hold L - C (H4) [CI]", "p", "GEE odds ratio L vs C"], rows,
        [12, 28, 28, 28, 9, 30])
dh_p = [(ds, E1[ds]["dhold_p"]) for ds in avail if "dhold_p" in E1[ds]]
if dh_p:
    for (ds, _), pa in zip(dh_p, Q.holm([p for _, p in dh_p])):
        E1[ds]["dhold_p_holm"] = pa

Q.sub("4  WHAT A PAPER WOULD REPORT (own fold, leaky arm) vs THE TRUTH (clean holdout) — full metric panel")
rows = []
for ds in avail:
    for arm in ("L", "C"):
        a = E1[ds]["arms"].get(arm)
        if not a:
            continue
        rows.append([ds, arm, fmt(a["own_acc"]), fmt(a["hold_acc"]), f"{a['own_auroc']:.3f}/{a['hold_auroc']:.3f}",
                     f"{fmt(a['own_pos_sens'])}/{fmt(a['hold_pos_sens'])}", f"{fmt(a['own_pos_spec'])}/{fmt(a['hold_pos_spec'])}",
                     f"{fmt(a['own_pos_f1'])}/{fmt(a['hold_pos_f1'])}", f"{a['hold_ece']:.3f}"])
Q.table(["dataset", "arm", "own acc", "hold acc", "AUROC own/hold", "pos sens own/hold", "pos spec own/hold",
         "pos F1 own/hold", "ECE hold"], rows, [12, 5, 9, 9, 16, 18, 18, 16, 9])
print("  'pos' = positive class one-vs-rest (tumour / OSCC / cancer / malignant); macro metrics are in tables/E1_main.csv.")
print("  temperature and ECE: temperature fitted on each run's inner-validation split, applied to the holdout.")

# ICC / design effect and OCDC per slide
Q.sub("5  CLUSTERING OF ERRORS  (ICC of holdout correctness within groups, L arm, seed 0)")
rows = []
for ds in avail:
    R_ = sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std", seed=0)
    if not R_:
        continue
    c = np.concatenate([correct(z, "hold") for _, _, z in R_])
    k = np.concatenate([z["hold_key"] + 10 ** 7 * j["spec"].get("rep", 0) for j, _, z in R_])
    ic = Q.icc1(c, k)
    cnt = np.unique(k, return_counts=True)[1]
    multi = int((cnt >= 2).sum())
    share = float(cnt[cnt >= 2].sum() / max(1, cnt.sum()))
    ic.update(clusters_with_2plus=multi, share_in_multi=share)
    if multi < 2 or share < 0.2:         # ICC needs repeated observations within clusters
        ic.update(icc=None, deff=None)
    E1[ds]["icc"] = ic
    rows.append([ds, "—" if ic["icc"] is None else f"{ic['icc']:.3f}", "—" if ic["deff"] is None else f"{ic['deff']:.2f}",
                 ic["clusters"], multi, f"{ic['mean_size']:.1f}"])
Q.table(["dataset", "ICC", "design effect", "clusters", "with >= 2 images", "mean size"], rows, [12, 8, 15, 10, 18, 10])
print("  a design effect > 1 means image-level tests (e.g. McNemar) overstate the evidence by that factor in variance.")
print("  '—' = not estimable: fewer than 20% of holdout images share a cluster with another image (nearly all singletons).")

if "he_stained" in avail:
    Q.sub("6  OCDC LEAVE-ONE-SLIDE-OUT: per held-out slide (seed-averaged) + exact slide-level sign-flip tests")
    rows, dL, dH = [], [], []
    for o in sorted({j["spec"]["outer"] for j, _, _ in sel(exp="E1", ds="he_stained", backbone=BB0, recipe="std")}):
        v = {}
        for arm in ("L", "C"):
            R_ = sel(exp="E1", ds="he_stained", arm=arm, outer=o, backbone=BB0, recipe="std")
            if R_:
                v[arm] = (np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_]),
                          np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_]))
        if len(v) == 2:
            dd = Q.DS.get("he_stained")
            slide = sorted(set(dd.slide[dd.base]))[o]
            dL.append(v["L"][0] - v["L"][1])
            dH.append(v["L"][1] - v["C"][1])
            rows.append([slide, fmt(v["L"][0]), fmt(v["L"][1]), fmt(v["C"][0]), fmt(v["C"][1]), fmt(v["L"][0] - v["L"][1])])
    Q.table(["held-out slide", "L own", "L hold", "C own", "C hold", "L inflation"], rows, [18, 9, 9, 9, 9, 12])
    if dL:
        pL, pH = Q.sign_flip_p(dL), Q.sign_flip_p(dH)
        E1["he_stained"]["slide_level"] = dict(inflation=dL, dhold=dH, p_inflation=pL, p_dhold=pH)
        Q.kv("sign-flip p (L inflation)", f"{pf(pL)}  over {len(dL)} slides (smallest possible two-sided p = {2 / 2 ** len(dL):.4f})")
        Q.kv("sign-flip p (hold L - C)", pf(pH))
OUT["E1"] = E1

# factorial (clinical) and grouping sensitivity (NDB)
Q.sub("7  FACTORIAL (clinical) AND GROUPING SENSITIVITY (NDB-UFES coarse blocks), seed 0")
rows = []
for ds in avail:
    ex = P["exps"]["E1"]["datasets"][ds]
    for arm in ex.get("extra", []):
        a = E1[ds]["arms"].get(arm)
        if a:
            f_ = ex["arms"][arm]
            rows.append([ds, arm, f"S={f_.get('S', 0)} Dup={f_.get('Dup', 0)} Grp={f_.get('Grp', 0)} coarse={f_.get('coarse', 0)}",
                         fmt(a["own"]), fmt(a["hold"]), fmt(a["infl"], a["infl_ci"])])
if rows:
    Q.table(["dataset", "arm", "factors", "own BA", "hold BA", "inflation [CI]"], rows, [12, 6, 30, 9, 9, 30])
else:
    print("  no finished runs yet — these arms are trained in the 'general' phase (BLOCK 24)")
if "clinical" in avail and all(a in E1["clinical"]["arms"] for a in ("L", "C", "S0D1", "S1D0")):
    A = E1["clinical"]["arms"]
    fx = dict(S=0.5 * ((A["L"]["infl"] - A["S0D1"]["infl"]) + (A["S1D0"]["infl"] - A["C"]["infl"])),
              Dup=0.5 * ((A["L"]["infl"] - A["S1D0"]["infl"]) + (A["S0D1"]["infl"] - A["C"]["infl"])),
              SxDup=(A["L"]["infl"] - A["S0D1"]["infl"]) - (A["S1D0"]["infl"] - A["C"]["infl"]))
    OUT["factorial_clinical"] = fx
    Q.kv("main effect of augment-before-split", fmt(fx["S"]) + " points of inflation (clinical, seed 0)")
    Q.kv("main effect of keeping duplicates", fmt(fx["Dup"]) + " points")
    Q.kv("interaction S x Dup", fmt(fx["SxDup"]) + " points (exploratory, no CI: one seed per cell)")

# ============================================================================ 8  published protocol comparison
Q.sub("8  L ARM = PUBLISHED PROTOCOL: our replication numbers next to the published ones (definitions stated)")
PUB = {"histopath": [("Abd El-Aziz 2025", dict(acc=99.84, spec=99.92, prec=97.94, sens=97.93, f1=97.93), "not stated")],
       "ndb_ufes": [("SciRep 2025", dict(acc=97.16, prec=97.16, sens=97.16, f1=97.16, spec=98.58), "multi-class, averaging not stated")],
       "clinical": [("Liu & Bagi 2025", dict(acc=99.54, sens=95.73, spec=96.21, prec=96.34, f1=96.03, auroc=60.26), "not stated"),
                    ("Alzahrani 2025", dict(acc=98.47, sens=97.73, spec=97.73, f1=98.27), "not stated")]}
rows = []
for ds, lst in PUB.items():
    a = E1.get(ds, {}).get("arms", {}).get("L")
    c = E1.get(ds, {}).get("arms", {}).get("C")
    if not a:
        continue
    for name, vals, dfn in lst:
        ours = dict(acc=a["own_acc"], sens=a["own_pos_sens"], spec=a["own_pos_spec"], prec=a["own_pos_prec"],
                    f1=a["own_pos_f1"], auroc=a["own_auroc"])
        truth = dict(acc=c["hold_acc"], sens=c["hold_pos_sens"], spec=c["hold_pos_spec"], prec=c["hold_pos_prec"],
                     f1=c["hold_pos_f1"], auroc=c["hold_auroc"]) if c else {}
        for k, v in vals.items():
            rows.append([ds, name, k, f"{v:.2f}", fmt(ours.get(k)), fmt(truth.get(k)), dfn])
Q.table(["dataset", "paper", "metric", "published", "ours, L own fold", "ours, clean holdout", "published definition"],
        rows, [12, 18, 7, 11, 17, 20, 30])
print("  ours = positive-class (one-vs-rest) values; for NDB-UFES the positive class is OSCC. AUROC shown x100.")
write_csv("published_vs_ours", ["dataset", "paper", "metric", "published", "ours_L_own", "ours_clean_holdout", "definition"], rows)

# ============================================================================ 9  PRE, RL
_nh = Q.DS.get("histopath").n if "histopath" in avail else 0


def rl_ceiling(j, z):
    """BA reachable on the holdout by a rule that sees only the TRUE class: each true class is mapped to the random
    label it most often carries in the development pool.  Random labels are drawn per group, so with few groups they
    can correlate with the true classes by chance; this ceiling shows how much of an above-chance score that explains."""
    dd = Q.DS.get(j["spec"]["ds"])
    y_rl = Q._labels(dd, j["spec"])
    hold = z["hold_row"]
    dev = np.setdiff1d(np.nonzero(dd.base)[0], hold)
    K = dd.K
    f = {c: int(np.bincount(y_rl[dev][dd.y[dev] == c], minlength=K).argmax()) if (dd.y[dev] == c).any() else 0
         for c in range(K)}
    pred = np.array([f[int(c)] for c in dd.y[hold]])
    yt = y_rl[hold]
    return float(np.mean([np.mean(pred[yt == k] == k) for k in range(K) if (yt == k).any()]))


for ex_name, title in (("PRE", f"9  PRE-AUGMENTED CORPUS ({_nh:,} files as distributed): L = random file split, C = grouped by recovered parent"),
                       ("RL", "10 RANDOM-LABEL CONTROL (H6): labels drawn per group; chance = 1/K")):
    Rx = sel(exp=ex_name, backbone=BB0, recipe="std")
    if not Rx:
        continue
    Q.sub(title)
    rows = []
    blk = {}
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for arm in ("L", "C"):
            R_ = [x for x in Rx if x[0]["spec"]["ds"] == ds and x[0]["spec"]["arm"] == arm]
            if not R_:
                continue
            to, th = tables_for(R_, "own"), tables_for(R_, "hold")
            _, mo = boot_folds(to, NB, 1)
            _, mh = boot_folds(th, NB, 2)
            own_b = mo.mean(0)
            d = dict(own=float(np.mean(list(point(to).values()))), hold=float(np.mean(list(point(th).values()))),
                     own_ci=Q.ci(own_b), infl_ci=Q.ci((mo - mh).mean(0)), infl_p=Q.boot_p((mo - mh).mean(0)), n=len(R_))
            chance = 1.0 / Q.META[ds]["K"]
            d["p_vs_chance"] = Q.boot_p(own_b, chance)
            if ex_name == "RL":
                d["true_class_ceiling"] = float(np.mean([rl_ceiling(j, z) for j, _, z in R_]))
                rows.append([ds, arm, len(R_), f"{fmt(d['own'])} [{fmt(d['own_ci'][0])},{fmt(d['own_ci'][1])}]", fmt(d["hold"]),
                             fmt(d["own"] - d["hold"], d["infl_ci"]), f"{100 * chance:.1f}", fmt(d["true_class_ceiling"]),
                             pf(d["p_vs_chance"])])
            else:
                rows.append([ds, arm, len(R_), f"{fmt(d['own'])} [{fmt(d['own_ci'][0])},{fmt(d['own_ci'][1])}]", fmt(d["hold"]),
                             fmt(d["own"] - d["hold"], d["infl_ci"])])
            blk[f"{ds}|{arm}"] = d
    if ex_name == "RL":
        Q.table(["dataset", "arm", "runs", "own BA [CI]", "hold BA", "own - hold [CI]", "chance", "true-class ceiling",
                 "p own vs chance"], rows, [12, 5, 6, 24, 9, 26, 8, 19, 16])
        print("  true-class ceiling = holdout BA of a rule that knows only the true class (random labels are drawn per group,")
        print("  so with few groups they can correlate with the true classes by chance). A controlled arm whose own-fold BA")
        print("  sits at its holdout BA and near this ceiling has learned no group identity; the leaky arm's excess over its")
        print("  holdout (own - hold) is the memorised share.")
    else:
        Q.table(["dataset", "arm", "runs", "own BA [CI]", "hold BA", "inflation [CI]"], rows, [12, 5, 6, 26, 9, 30])
    OUT[ex_name] = blk

# ============================================================================ 11 dose-response
Q.sub("11 DOSE-RESPONSE (H5): inflation vs augmentation factor k, injected duplicates p, group overlap q")
DOSE = {}
rows = []
for ex_name, xform, lab in (("DOSEK", lambda d: math.log2(d), "log2 k"), ("DUP", lambda d: d, "p"), ("OVL", lambda d: d, "q")):
    Rx = sel(exp=ex_name, backbone=BB0, recipe="std")
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for arm in sorted({j["spec"]["arm"] for j, _, _ in Rx if j["spec"]["ds"] == ds}):
            pts = []
            cells = []
            for dose in P["exps"][ex_name]["doses"]:
                R_ = [x for x in Rx if x[0]["spec"]["ds"] == ds and x[0]["spec"]["arm"] == arm and x[0]["spec"]["dose"] == dose]
                if not R_:
                    cells.append("  —  ")
                    continue
                inf = [RM[j["id"]]["own"]["bal_acc"] - RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_]
                for (j, _, _), v in zip(R_, inf):
                    pts.append((xform(dose), v, j["spec"]["outer"]))
                cells.append(f"{100 * np.mean(inf):+5.1f}")
            if len(pts) >= 4:
                x, yv, g = map(np.array, zip(*pts))
                try:
                    import pandas as pd
                    import statsmodels.formula.api as smf
                    with Q.quiet():
                        mm = smf.mixedlm("y ~ x", pd.DataFrame(dict(x=x, y=yv, g=g)), groups=g).fit(reml=True)
                    slope, se, p = float(mm.params["x"]), float(mm.bse["x"]), float(mm.pvalues["x"])
                except Exception:
                    b = np.polyfit(x, yv, 1)
                    slope, se, p = float(b[0]), float("nan"), float("nan")
                from scipy.stats import spearmanr
                rho = float(spearmanr(x, yv).correlation) if len(set(x)) > 1 else float("nan")
                DOSE[f"{ex_name}|{ds}|{arm}"] = dict(points=pts, slope=slope, se=se, p=p, spearman=rho)
                rows.append([ex_name, ds, arm, "  ".join(cells), f"{100 * slope:+.2f}/{lab}", pf(p), f"{rho:+.2f}"])
Q.table(["exp", "dataset", "arm", "inflation at each dose (points)", "slope", "p (mixed)", "Spearman"], rows,
        [7, 12, 5, 44, 14, 10, 9])
for ex_name in ("DOSEK", "DUP", "OVL"):
    print(f"  {ex_name}: doses {P['exps'][ex_name]['doses']}")
OUT["dose"] = DOSE
Rx = sel(exp="DUP", backbone=BB0, recipe="std")
if Rx:
    rows = []
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for dose in P["exps"]["DUP"]["doses"][1:]:
            inj, non = [], []
            for j, r, z in Rx:
                if j["spec"]["ds"] != ds or j["spec"]["dose"] != dose or "own_injected" not in z:
                    continue
                m = z["own_injected"].astype(bool)
                c = correct(z, "own")
                if m.any():
                    inj.append(c[m].mean())
                if (~m).any():
                    non.append(c[~m].mean())
            if inj:
                rows.append([ds, dose, f"{100 * np.mean(inj):.1f}", f"{100 * np.mean(non):.1f}"])
    Q.table(["dataset", "p", "acc on injected (copy in training)", "acc on the rest"], rows, [12, 6, 36, 16])

# ============================================================================ 12 generality (backbones) + capacity
Q.sub("12 GENERALITY (H7): inflation by backbone family, seed 0, outer folds 0-2 (same splits for every backbone)")
G = P["generality"]
bbs = [BB0] + G["backbones"]
GEN = {}
rows = []
for ds in G["datasets"]:
    for bb in bbs:
        v = {}
        for arm in ("L", "C"):
            R_ = [x for x in sel(exp="E1", ds=ds, arm=arm, backbone=bb, recipe="std", seed=0) if x[0]["spec"]["outer"] in G["outer"]
                  and x[0]["spec"].get("rep", 0) == 0]
            if R_:
                v[arm] = (float(np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_])), len(R_))
        if "L" in v:
            GEN[f"{ds}|{bb}"] = v
            rows.append([ds, Q.BACKBONES[bb][0], f"{Q.BACKBONES[bb][1]:.1f}M", f"{fmt(v['L'][0])}/{fmt(v['L'][1])}",
                         fmt(v["L"][0] - v["L"][1]).replace("+", "+"),
                         f"{fmt(v['C'][0])}/{fmt(v['C'][1])}" if "C" in v else "—",
                         fmt(v["C"][0] - v["C"][1]) if "C" in v else "—", v["L"][2]])
Q.table(["dataset", "backbone", "params", "L own/hold", "L infl.", "C own/hold", "C infl.", "folds"], rows,
        [12, 12, 8, 16, 9, 16, 9, 6])
probes = Q.jload(Q.D["eval"] / "probes.json", []) or []
if probes and GEN:
    from scipy.stats import spearmanr
    rows = []
    for ds in G["datasets"]:
        pts = []
        for bb in bbs:
            v = GEN.get(f"{ds}|{bb}")
            if v:
                pts.append((Q.BACKBONES[bb][1], v["L"][0] - v["L"][1]))
        pr = [p_ for p_ in probes if p_["exp"] == "E1" and p_["ds"] == ds and p_["arm"] == "L" and p_["seed"] == 0
              and p_["outer"] in G["outer"] and "logreg" in p_]
        lin = float(np.mean([p_["logreg"]["own"] - p_["logreg"]["hold"] for p_ in pr])) if pr else float("nan")
        nn1 = float(np.mean([p_["1nn"]["own"] - p_["1nn"]["hold"] for p_ in pr])) if pr else float("nan")
        rho = float(spearmanr(*zip(*pts)).correlation) if len(pts) >= 3 else float("nan")
        rows.append([ds, fmt(lin), fmt(nn1), f"{rho:+.2f}" if np.isfinite(rho) else "— (needs 3 models)", len(pts)])
        GEN[f"{ds}|capacity"] = dict(linear_probe=lin, one_nn=nn1, spearman_params=rho)
    Q.table(["dataset", "linear probe infl.", "1-NN infl.", "Spearman(params, infl.)", "fine-tuned models"], rows,
            [12, 18, 12, 24, 18])
OUT["generality"] = GEN

# ============================================================================ 13 reimplementations
Rx = [x for x in RUNS.values() if x[0]["spec"]["recipe"] != "std"]
Q.sub("13 PUBLISHED PIPELINES REIMPLEMENTED (EfficientNet-B3): same method under the leaky and the controlled protocol")
if not Rx:
    print("  no finished runs yet — the reimplementations are trained in the 'general' phase (BLOCK 24)")
else:
    rows = []
    REI = {}
    for r_ in P["reimp"]:
        v = {}
        for arm in r_["arms"]:
            R_ = [x for x in Rx if x[0]["spec"]["recipe"] == r_["recipe"] and x[0]["spec"]["ds"] == r_["ds"]
                  and x[0]["spec"]["exp"] == r_["exp"] and x[0]["spec"]["arm"] == arm]
            if R_:
                v[arm] = (float(np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["own"]["acc"] for j, _, _ in R_])), len(R_))
        if v:
            REI[f"{r_['paper']}|{r_['exp']}|{r_['ds']}"] = v
            for arm, t in v.items():
                rows.append([r_["paper"][:40], arm, fmt(t[2]), fmt(t[0]), fmt(t[1]), fmt(t[0] - t[1]), t[3]])
    Q.table(["pipeline", "arm", "own acc", "own BA", "hold BA", "inflation", "folds"], rows, [42, 5, 9, 9, 9, 10, 6])
    OUT["reimp"] = REI

# ============================================================================ 14 cross-dataset
Q.sub("14 CROSS-DATASET TRANSFER (E1 seed 0): models trained on one corpus, tested on the other binary H&E corpora")
rows = []
XD = {}
for ds in Q.XEVAL:
    for arm in ("L", "C"):
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std", seed=0)
        for t in Q.XEVAL[ds]:
            au, bal = [], []
            for j, r, z in R_:
                if f"x_{t}_logit" not in z:
                    continue
                p_ = Q.softmax(z[f"x_{t}_logit"])
                score = p_[:, 1] if p_.shape[1] == 2 else p_[:, 2] / (p_[:, 0] + p_[:, 2] + 1e-9)
                y = z[f"x_{t}_y"]
                from sklearn.metrics import roc_auc_score
                if len(np.unique(y)) == 2:
                    au.append(roc_auc_score(y, score))
                    pr = (score >= 0.5).astype(int)
                    bal.append(0.5 * (np.mean(pr[y == 1] == 1) + np.mean(pr[y == 0] == 0)))
            if au:
                XD[f"{ds}|{arm}|{t}"] = dict(auroc=float(np.mean(au)), bal_acc=float(np.mean(bal)), n=len(au))
                rows.append([ds, arm, t, f"{np.mean(au):.3f}", fmt(np.mean(bal)), len(au)])
if rows:
    Q.table(["trained on", "arm", "tested on", "AUROC", "BA (thr 0.5)", "models"], rows, [12, 5, 12, 8, 13, 8])
OUT["cross_dataset"] = XD

# ============================================================================ 15 mechanism: NN similarity vs correctness
Q.sub("15 MECHANISM: similarity to the nearest training image and correctness (E1 seed 0, frozen DINOv2 space)")
audit = Q.jload(Q.D["index"] / "audit.json", {}) or {}
EMB = audit.get("_meta", {}).get("embed", "dinov2_s")
rows = []
MECH = {}
for ds in avail:
    fe = Q.D["store"] / f"{ds}_emb_{EMB}.npy"
    if not fe.exists():
        continue
    E = np.load(fe).astype(np.float32)
    pol = P["exps"]["E1"]["datasets"][ds].get("policy")
    sib = None
    if pol:
        fs = Q.D["store"] / f"{ds}_sib_{pol}_{EMB}.npz"
        if fs.exists():
            zz = np.load(fs)
            pos_ = np.full(len(E), -1)
            pos_[zz["rows"]] = np.arange(len(zz["rows"]))
            sib = (pos_, zz["emb"].astype(np.float32), int(zz["kmax"]))

    def fx(rows_, copies):
        X = E[rows_].copy()
        if sib is not None:
            m = (copies > 0) & (copies <= sib[2])
            X[m] = sib[1][sib[0][rows_[m]], copies[m] - 1]
        return X

    for arm in ("L", "C"):
        sims, cors = defaultdict(list), defaultdict(list)
        for j, r, z in sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std", seed=0):
            B = Q.build_job(j["spec"], check=False)
            Xt = fx(B["train"]["row"], B["train"]["copy"])
            for part in ("own", "hold"):
                Xe = fx(z[f"{part}_row"], z[f"{part}_copy"])
                s_ = np.concatenate([(Xe[i:i + 2048] @ Xt.T).max(1) for i in range(0, len(Xe), 2048)]) if len(Xe) else np.zeros(0)
                sims[part].append(s_)
                cors[part].append(correct(z, part))
        if sims:
            so, sh = np.concatenate(sims["own"]), np.concatenate(sims["hold"])
            co, ch = np.concatenate(cors["own"]), np.concatenate(cors["hold"])
            allc, alls = np.concatenate([co, ch]), np.concatenate([so, sh])
            r_pb = float(np.corrcoef(alls, allc)[0, 1]) if allc.std() > 0 else float("nan")
            MECH[f"{ds}|{arm}"] = dict(sim_own=float(so.mean()), sim_hold=float(sh.mean()), r_sim_correct=r_pb,
                                       acc_top_decile=float(allc[alls >= np.percentile(alls, 90)].mean()),
                                       acc_bottom_decile=float(allc[alls <= np.percentile(alls, 10)].mean()))
            rows.append([ds, arm, f"{so.mean():.3f}", f"{sh.mean():.3f}", f"{r_pb:+.3f}",
                         fmt(MECH[f'{ds}|{arm}']['acc_top_decile']), fmt(MECH[f'{ds}|{arm}']['acc_bottom_decile'])])
Q.table(["dataset", "arm", "sim own", "sim hold", "r(sim, correct)", "acc top-10% sim", "acc bottom-10% sim"], rows,
        [12, 5, 9, 9, 16, 16, 18])
OUT["mechanism_nn"] = MECH

# ============================================================================ 16 leakage-risk predictor
Q.sub("16 LEAKAGE-RISK PREDICTOR (exploratory): does the provenance audit predict a run's inflation?")
FEAT = ["same_original", "identical_file", "near_dup", "struct_group", "ctrl_group"]
Xr, yr, gr = [], [], []
for rid, (j, r, z) in RUNS.items():
    if j["spec"]["recipe"] != "std" or j["spec"]["backbone"] != BB0 or j["spec"]["exp"] in ("RL",):
        continue
    lk = r["info"].get("leak", {})
    if not lk or "own" not in RM[rid]:
        continue
    Xr.append([lk.get(k, 0) for k in FEAT])
    yr.append(RM[rid]["own"]["bal_acc"] - RM[rid]["hold"]["bal_acc"])
    gr.append(j["spec"]["ds"])
if len(set(gr)) >= 3:
    from sklearn.linear_model import LinearRegression, RidgeCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    Xr, yr, gr = np.array(Xr, float), np.array(yr, float), np.array(gr)
    models = {"null (training mean)": None,
              "one predictor: any relative in training": [4],
              "ridge: all five audit shares": [0, 1, 2, 3, 4]}
    RISK = {}
    rows = []
    for name, cols in models.items():
        pred = np.zeros_like(yr)
        for ds in sorted(set(gr)):
            m = gr == ds
            if cols is None:
                pred[m] = yr[~m].mean()
            elif len(cols) == 1:
                pred[m] = LinearRegression().fit(Xr[~m][:, cols], yr[~m]).predict(Xr[m][:, cols])
            else:
                mdl = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 12)))
                pred[m] = mdl.fit(Xr[~m][:, cols], yr[~m]).predict(Xr[m][:, cols])
        r2 = 1 - np.sum((yr - pred) ** 2) / np.sum((yr - yr.mean()) ** 2)
        mae = float(np.mean(np.abs(yr - pred)))
        RISK[name] = dict(lodo_r2=float(r2), mae=mae)
        rows.append([name, f"{r2:+.2f}", f"{100 * mae:.1f}"])
    Q.table(["model (leave-one-dataset-out)", "R²", "MAE (points)"], rows, [44, 8, 14])
    one = LinearRegression().fit(Xr[:, [4]], yr)
    RISK["slope_any_relative"] = float(one.coef_[0])
    RISK["n_runs"] = int(len(yr))
    RISK["datasets"] = sorted(set(gr))
    OUT["risk_predictor"] = RISK
    Q.kv("slope, any-relative share", f"{100 * one.coef_[0]:+.1f} points of inflation per 100% share (all runs pooled)")
    print(f"  n = {len(yr)} runs from {len(set(gr))} datasets. A negative R² means the model predicts worse than the mean of")
    print("  the other datasets; that result is reported as it is.")
else:
    print("  fewer than 3 datasets with finished runs — skipped")

# ============================================================================ 17 provenance-stratified own-fold accuracy (Idea B)
Q.sub("17 PROVENANCE-STRATIFIED OWN FOLD (L arm, all seeds): accuracy by the closest relative each test image has in training")
STRATA = ["same source image (sibling)", "identical / near-duplicate file", "same slide / patient / source group", "no relative"]
STR = {}
rows, csvrows = [], []
for ds in avail:
    R_ = sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std")
    if not R_:
        continue
    dd = Q.DS.get(ds)
    cor_all, st_all, key_all, hold_acc = [], [], [], []
    for j, r, z in R_:
        B = Q.build_job(dict(j["spec"], xeval=False), check=False)
        tr = B["train"]["row"]
        s_row, s_md5 = set(tr.tolist()), set(dd.md5[tr].tolist())
        s_near, s_grp = set(dd.g_near[tr].tolist()), set(dd.g_ctrl[tr].tolist())
        rows_o = z["own_row"]
        st = np.full(len(rows_o), 3)
        for i, rr in enumerate(rows_o):
            if rr in s_row:
                st[i] = 0
            elif dd.md5[rr] in s_md5 or dd.g_near[rr] in s_near:
                st[i] = 1
            elif dd.g_ctrl[rr] in s_grp:
                st[i] = 2
        cor_all.append(correct(z, "own"))
        st_all.append(st)
        key_all.append(z["own_key"] + 10 ** 7 * j["spec"].get("rep", 0) + 10 ** 8 * j["spec"]["outer"])
        hold_acc.append(float(correct(z, "hold").mean()))
    cor, st, key = map(np.concatenate, (cor_all, st_all, key_all))
    cl, inv = np.unique(key, return_inverse=True)
    g = np.random.default_rng(Q.shash(ds, "strata") % 2 ** 32)
    W = list(Q.boot_weights(len(cl), NB, g))
    d = dict(hold_acc=float(np.mean(hold_acc)), n_runs=len(R_), strata={})
    boots = {}
    for k, name in enumerate(STRATA):
        m = st == k
        if not m.any():
            continue
        cc = np.bincount(inv, cor * m, len(cl))
        nn_ = np.bincount(inv, m.astype(float), len(cl))
        bs = np.concatenate([(Wt @ cc) / np.maximum(Wt @ nn_, 1e-9) for Wt in W])
        boots[k] = bs
        d["strata"][name] = dict(share=float(m.mean()), n=int(m.sum()), acc=float(cor[m].mean()), ci=Q.ci(bs))
        rows.append([ds, name, f"{100 * m.mean():.1f}", int(m.sum()), f"{100 * cor[m].mean():.2f} [{100 * Q.ci(bs)[0]:.2f},"
                     f"{100 * Q.ci(bs)[1]:.2f}]", f"{100 * d['hold_acc']:.2f}"])
        csvrows.append([ds, name, m.mean(), int(m.sum()), cor[m].mean(), *Q.ci(bs), d["hold_acc"]])
    if 0 in boots and 3 in boots:
        diff = boots[0] - boots[3]
        d["sibling_minus_none"] = dict(est=d["strata"][STRATA[0]]["acc"] - d["strata"][STRATA[3]]["acc"], ci=Q.ci(diff),
                                       p=Q.boot_p(diff))
    STR[ds] = d
if rows:
    Q.table(["dataset", "closest relative in training", "% of own fold", "images", "own-fold accuracy [CI]", "holdout acc"], rows,
            [12, 38, 14, 8, 26, 12])
    for ds, d in STR.items():
        if "sibling_minus_none" in d:
            e = d["sibling_minus_none"]
            Q.kv(f"{ds}: sibling - no relative", f"{100 * e['est']:+.2f} points [{100 * e['ci'][0]:+.2f}, {100 * e['ci'][1]:+.2f}] "
                 f"p {pf(e['p'])}")
    print("  images are pooled over folds and seeds (an image counted once per run); CIs resample groups (fold x group clusters).")
    print("  A stratum with very few images (e.g. 'no relative' under heavy augmentation) gives a wide CI — read it with its n.")
    write_csv("provenance_strata", ["dataset", "stratum", "share", "n", "acc", "ci_lo", "ci_hi", "holdout_acc"], csvrows)
OUT["provenance_strata"] = STR

# ============================================================================ 18 confusion matrices (multi-class)
CONF = {}
for ds in [d for d in avail if Q.META[d]["K"] > 2]:
    K = Q.META[ds]["K"]
    cls = Q.META[ds]["classes"]
    Q.sub(f"18 CONFUSION MATRICES, {ds} (E1, all seeds, pooled over outer folds; rows = true class, % of the row)")
    crow = []
    for arm, part in (("L", "own"), ("L", "hold"), ("C", "own"), ("C", "hold")):
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std")
        if not R_:
            continue
        M = np.zeros((K, K), int)
        for j, r, z in R_:
            yt, yp = z[f"{part}_y"], z[f"{part}_logit"].argmax(1)
            np.add.at(M, (yt, yp), 1)
        rowp = M / np.maximum(M.sum(1, keepdims=True), 1)
        CONF[f"{ds}|{arm}|{part}"] = dict(counts=M.tolist(), row_pct=rowp.tolist(), classes=cls)
        print(f"  arm {arm} · {'own fold (what a paper would report)' if part == 'own' else 'clean holdout'} · "
              f"{int(M.sum()):,} predictions")
        Q.table(["true \\ predicted"] + cls + ["recall"],
                [[cls[i]] + [f"{100 * rowp[i, k]:.1f}" for k in range(K)] + [f"{100 * rowp[i, i]:.1f}"] for i in range(K)],
                [20] + [14] * K + [10])
        off = M.sum() - np.trace(M)
        for i in range(K):
            for k in range(K):
                crow.append([ds, arm, part, cls[i], cls[k], int(M[i, k]), float(rowp[i, k])])
        if off:
            pair = max(((i, k) for i in range(K) for k in range(K) if i != k), key=lambda t: M[t] + M[t[::-1]])
            Q.kv("most confused pair", f"{cls[pair[0]]} <-> {cls[pair[1]]}: {100 * (M[pair] + M[pair[::-1]]) / off:.1f}% "
                 f"of all {int(off):,} errors")
    write_csv(f"confusion_{ds}", ["dataset", "arm", "part", "true", "predicted", "count", "row_share"], crow)
OUT["confusion"] = CONF

Q.jsave(Q.D["eval"] / "q1_results.json", OUT)
print(f"\n[BLOCK 25 ✓]  results in q1/eval/q1_results.json · tables in q1/tables/ · {Q.hms(time.time() - t_all)} · run BLOCK 26 next")

In [ ]:
# ============================================================================
# BLOCK 27 — PAPER FIGURES (PNG, 300 dpi) + RESULTS SUMMARY (Markdown)
# ============================================================================
import os, sys, json, time, math, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("27")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

t_all = time.time()
P = Q.plan()
R = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
CL = Q.jload(Q.D["eval"] / "clinical.json", {}) or {}
AU = Q.jload(Q.D["index"] / "audit.json", {}) or {}
PR = Q.jload(Q.D["eval"] / "probes.json", []) or []
FIG = Q.D["figs"]
COL = {"L": "#eb6834", "C": "#2a78d6", "probe": "#1baf7a"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
NAME = {"he_stained": "OCDC (H&E)", "histopath": "Histopathology", "ndb_ufes": "NDB-UFES", "clinical": "Clinical photos",
        "breakhis": "BreakHis"}
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 300, "figure.dpi": 110})
made = []
Q.hdr("BLOCK 27 — PAPER FIGURES + RESULTS SUMMARY")


def save(fig, name, note):
    fig.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    made.append((name, note))


# ---------------------------------------------------------------- Figure A: own fold vs clean holdout
E1 = R.get("E1", {})
ds_list = [d for d in Q.DATASETS if d in E1 and "L" in E1[d].get("arms", {}) and "C" in E1[d]["arms"]]
if ds_list:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 0.75 * len(ds_list) + 1.6), gridspec_kw=dict(width_ratios=[1.25, 1]))
    for i, ds in enumerate(ds_list):
        y0 = len(ds_list) - 1 - i
        for k, arm in enumerate(("L", "C")):
            a = E1[ds]["arms"][arm]
            yy = y0 + (0.15 if arm == "L" else -0.15)
            a1.plot([100 * a["hold"], 100 * a["own"]], [yy, yy], color=COL[arm], lw=2, solid_capstyle="round")
            a1.scatter([100 * a["hold"]], [yy], s=40, color="white", edgecolor=COL[arm], lw=2, zorder=3)
            a1.scatter([100 * a["own"]], [yy], s=40, color=COL[arm], zorder=3)
            lo, hi = a["infl_ci"]
            a2.errorbar([100 * a["infl"]], [yy], xerr=[[100 * (a["infl"] - lo)], [100 * (hi - a["infl"])]], fmt="o",
                        color=COL[arm], ms=6, lw=2, capsize=0)
    for ax in (a1, a2):
        ax.set_yticks(range(len(ds_list)))
        ax.set_yticklabels([NAME[d] for d in ds_list][::-1])
        ax.grid(axis="y", visible=False)
    a1.set_xlabel("Balanced accuracy (%)")
    a1.set_title("A  What the protocol reports (filled) vs clean holdout (open)", loc="left", fontsize=9, color=INK)
    a2.axvline(0, color=INK2, lw=1)
    a2.set_xlabel("Inflation = own fold − clean holdout (points, 95% CI)")
    a2.set_title("B  Inflation by arm", loc="left", fontsize=9, color=INK)
    h = [plt.Line2D([], [], color=COL["L"], lw=2, marker="o", label="leaky protocol (as published)"),
         plt.Line2D([], [], color=COL["C"], lw=2, marker="o", label="leakage-controlled, matched")]
    fig.legend(handles=h, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save(fig, "Fig_Q1_main_own_vs_holdout.png", "E1: own-fold vs clean-holdout balanced accuracy and inflation with 95% CI")

# ---------------------------------------------------------------- Figure B: dose-response
D_ = R.get("dose", {})
panels = [(k.split("|")) for k in D_]
keys = sorted({(e, d) for e, d, a in panels}, key=lambda t: (["DOSEK", "DUP", "OVL"].index(t[0]), t[1]))
if keys:
    nc = min(4, len(keys))
    nr = int(math.ceil(len(keys) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.1 * nc, 2.6 * nr), squeeze=False)
    xl = {"DOSEK": "augmentation factor k (log scale)", "DUP": "share of own-fold images with a copy in training",
          "OVL": "share of group-mates moved into training"}
    for ax, (ex, ds) in zip(axs.ravel(), keys):
        for arm in ("L", "C"):
            v = D_.get(f"{ex}|{ds}|{arm}")
            if not v:
                continue
            pts = np.array(v["points"], float)
            xs = 2 ** pts[:, 0] if ex == "DOSEK" else pts[:, 0]
            ax.scatter(xs, 100 * pts[:, 1], s=10, color=COL[arm], alpha=0.35, lw=0)
            ux = np.unique(xs)
            lab_ = "leaky" if arm == "L" else "controlled"
            if v.get("p") is not None and np.isfinite(v["p"]):
                lab_ += f"  slope p={v['p']:.3g}"
            ax.plot(ux, [100 * pts[xs == u, 1].mean() for u in ux], color=COL[arm], lw=2, marker="o", ms=4, label=lab_)
        if ex == "DOSEK":
            ax.set_xscale("log", base=2)
        ax.axhline(0, color=INK2, lw=1)
        ax.set_title(f"{ex} · {NAME[ds]}", loc="left", fontsize=8.5, color=INK)
        ax.set_xlabel(xl[ex], fontsize=7.5)
        ax.set_ylabel("inflation (points)", fontsize=7.5)
        ax.legend(fontsize=6.5, loc="upper left")
    for ax in axs.ravel()[len(keys):]:
        ax.axis("off")
    fig.tight_layout()
    save(fig, "Fig_Q1_dose_response.png", "dose-response: augmentation factor, injected duplicates, group overlap (fold points + means)")

# ---------------------------------------------------------------- Figure C: random-label control
RL = R.get("RL", {})
if RL:
    dss = sorted({k.split("|")[0] for k in RL})
    fig, ax = plt.subplots(figsize=(1.6 * len(dss) + 2, 3))
    for i, ds in enumerate(dss):
        for k, arm in enumerate(("L", "C")):
            v = RL.get(f"{ds}|{arm}")
            if not v:
                continue
            x = i + (-0.18 if arm == "L" else 0.18)
            ax.bar(x, 100 * v["own"], width=0.34, color=COL[arm], label=("leaky" if arm == "L" else "controlled") if i == 0 else None)
            ax.errorbar(x, 100 * v["own"], yerr=[[100 * (v["own"] - v["own_ci"][0])], [100 * (v["own_ci"][1] - v["own"])]],
                        color=INK, lw=1, capsize=2)
        ch = 100 / Q.META[ds]["K"]
        ax.plot([i - 0.4, i + 0.4], [ch, ch], color=INK2, lw=1.2, ls="--")
    ax.set_xticks(range(len(dss)))
    ax.set_xticklabels([NAME[d] for d in dss])
    ax.set_ylabel("own-fold balanced accuracy (%)")
    ax.set_title("Labels randomised per group: own-fold balanced accuracy with 95% CI (dashed = chance)", loc="left", fontsize=9)
    ax.legend(loc="upper right")
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    save(fig, "Fig_Q1_random_label.png", "random-label control (labels per source/slide/patient)")

# ---------------------------------------------------------------- Figure D: capacity
GEN = R.get("generality", {})
dss = [d for d in P["generality"]["datasets"] if any(k.startswith(d + "|") and "|capacity" not in k for k in GEN)]
if dss:
    nc = min(3, len(dss))
    nr = int(np.ceil(len(dss) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(4.3 * nc, 3.6 * nr), squeeze=False)
    for ax in axs.ravel()[len(dss):]:
        ax.axis("off")
    for ax, ds in zip(axs.ravel(), dss):
        cap = GEN.get(f"{ds}|capacity", {})
        ticks, labels = [], []
        pos = 0
        if cap and np.isfinite(cap.get("one_nn", np.nan)):
            ax.scatter([pos], [100 * cap["one_nn"]], marker="^", s=34, color=INK2, zorder=3, label="1-NN on frozen features")
            ticks.append(pos); labels.append("1-NN\nfrozen")
            pos += 1
        if cap and np.isfinite(cap.get("linear_probe", np.nan)):
            ax.scatter([pos], [100 * cap["linear_probe"]], marker="s", s=30, color=COL["probe"], zorder=3,
                       label="linear probe, frozen")
            ticks.append(pos); labels.append("linear\nprobe")
            pos += 1
        pts = []
        for bb, (short, par, _) in Q.BACKBONES.items():
            v = GEN.get(f"{ds}|{bb}")
            if v and "L" in v:
                pts.append((par, short, 100 * (v["L"][0] - v["L"][1])))
        pts.sort()
        if pts:
            xs = list(range(pos, pos + len(pts)))
            ax.plot(xs, [p_[2] for p_ in pts], color=COL["L"], lw=1.5, marker="o", ms=5, zorder=3, label="fine-tuned")
            ticks += xs
            labels += [f"{p_[1]}\n{p_[0]:.0f}M" for p_ in pts]
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels, fontsize=6.5, rotation=35, ha="right", rotation_mode="anchor")
        ax.set_xlim(-0.6, max(ticks) + 0.6 if ticks else 1)
        ax.axhline(0, color=INK2, lw=1)
        ax.set_title(NAME[ds], loc="left", fontsize=8.5)
        ax.set_ylabel("leaky-arm inflation (points)", fontsize=7.5)
    axs[0][0].legend(fontsize=6.5, loc="best")
    fig.tight_layout()
    save(fig, "Fig_Q1_capacity.png", "inflation of the leaky arm across backbone families and frozen-feature probes")

# ---------------------------------------------------------------- Figure E: audit (NDB order + histopathology parents)
if (Q.D["index"] / "ndb_ufes.json").exists():
    dd = Q.DS.get("ndb_ufes")
    emb = Q.D["store"] / f"ndb_ufes_emb_{AU.get('_meta', {}).get('embed', 'dinov2_s')}.npy"
    if emb.exists():
        E = np.load(emb).astype(np.float32)
        order = np.argsort([r["pidx"] for r in dd.R])[:600]
        nshow = len(order)
        s = (E[order][:-1] * E[order][1:]).sum(1)
        seg = dd.g_struct[order]
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 2.8), gridspec_kw=dict(width_ratios=[2.2, 1]))
        a1.plot(np.arange(len(s)), s, color=COL["C"], lw=0.8)
        for b in np.nonzero(seg[1:] != seg[:-1])[0]:
            a1.axvline(b + 0.5, color=COL["L"], lw=0.5, alpha=0.6)
        tau = AU.get("ndb_ufes", {}).get("segmentation", {}).get("tau")
        if tau:
            a1.axhline(tau, color=INK2, ls="--", lw=1)
        a1.set_xlabel(f"patch index p (first {nshow} in index order)")
        a1.set_ylabel("cosine to next patch")
        a1.set_title("A  NDB-UFES: similarity of consecutive patches in file-index order (orange = recovered boundaries)", loc="left", fontsize=8.5)
        allc = (E[np.argsort([r["pidx"] for r in dd.R])][:-1] * E[np.argsort([r["pidx"] for r in dd.R])][1:]).sum(1)
        a2.hist(allc, bins=60, color=COL["C"])
        if tau:
            a2.axvline(tau, color=COL["L"], lw=1.5, label=f"Otsu threshold {tau:.3f}")
        a2.set_xlabel("adjacent-patch cosine")
        a2.set_title("B  distribution", loc="left", fontsize=8.5)
        a2.legend(fontsize=7)
        fig.tight_layout()
        save(fig, "Fig_Q1_ndb_order.png", "evidence that NDB-UFES patch indices follow source images (group recovery)")

# ---------------------------------------------------------------- Figure F: reliability + selective prediction
CAL = CL.get("calibration", {})
SEL = CL.get("selective", {})
dss = [d for d in Q.DATASETS if f"{d}|L" in CAL and f"{d}|C" in CAL]
if dss:
    fig, axs = plt.subplots(2, len(dss), figsize=(2.6 * len(dss), 5.2), squeeze=False)
    for i, ds in enumerate(dss):
        ax = axs[0][i]
        ax.plot([0, 1], [0, 1], color=INK2, lw=1, ls="--")
        for arm in ("L", "C"):
            c = CAL[f"{ds}|{arm}"]["cal"]
            pts = [(x, y) for x, y, n in zip(c["conf"], c["acc"], c["n"]) if x is not None and n >= 5]
            if pts:
                ax.plot(*zip(*pts), color=COL[arm], lw=2, marker="o", ms=3,
                        label=f"{'leaky' if arm == 'L' else 'controlled'} ECE {c['ece']:.3f}")
        ax.set_title(NAME[ds], loc="left", fontsize=8.5)
        ax.set_xlabel("confidence", fontsize=7.5)
        ax.set_ylabel("accuracy", fontsize=7.5)
        ax.legend(fontsize=6)
        ax = axs[1][i]
        rk = None
        for arm in ("L", "C"):
            v = SEL.get(f"{ds}|{arm}")
            if not v:
                continue
            cov = sorted(v["cov"], key=float, reverse=True)
            x = [100 * float(c) for c in cov]
            y = [100 * v["cov"][c]["acc"] for c in cov]
            lo = [100 * v["cov"][c]["ci"][0] for c in cov]
            hi = [100 * v["cov"][c]["ci"][1] for c in cov]
            ax.fill_between(x, lo, hi, color=COL[arm], alpha=0.15, lw=0)
            ax.plot(x, y, color=COL[arm], lw=2, marker="o", ms=3, label="leaky" if arm == "L" else "controlled")
            rk = v.get("ranking")
        if rk:
            ax.text(0.02, 0.02, f"ranked by {rk} entropy", transform=ax.transAxes, fontsize=6, color=INK2)
        if i == 0:
            ax.legend(fontsize=6, loc="upper left")
        ax.invert_xaxis()
        ax.set_xlabel("coverage (% auto-reported)", fontsize=7.5)
        ax.set_ylabel("holdout accuracy (%)", fontsize=7.5)
    fig.tight_layout()
    save(fig, "Fig_Q1_calibration_selective.png", "holdout reliability (temperature from inner validation) and selective prediction with 95% CI")

# ---------------------------------------------------------------- results summary markdown
L = ["# Q1 results summary (generated by Block 27)", "", f"Plan SHA-256: `{R.get('meta', {}).get('plan_sha')}` · runs: "
     f"{R.get('meta', {}).get('n_runs')} · bootstrap resamples: {R.get('meta', {}).get('boot')}", "",
     "## E1: own fold vs clean holdout (balanced accuracy, %)", "",
     "Inflation CIs and p-values: two-stage cluster bootstrap. p is Holm-adjusted for the L arm (primary family H1) and "
     "unadjusted for the other arms.", "",
     "| Dataset | Arm | Own | Holdout | Inflation [95% CI] | p | seed SD |", "|---|---|---|---|---|---|---|"]
for ds in ds_list:
    for arm, a in E1[ds]["arms"].items():
        sdv = "" if a.get("seed_sd") is None else f"{100 * a['seed_sd']:.2f}"
        pv = a.get("infl_p_holm", a["infl_p"])
        L.append(f"| {NAME[ds]} | {arm} | {100 * a['own']:.2f} | {100 * a['hold']:.2f} | {100 * a['infl']:+.2f} "
                 f"[{100 * a['infl_ci'][0]:+.2f}, {100 * a['infl_ci'][1]:+.2f}] | {pv:.2g} | {sdv} |")
L += ["", "## Figures", ""] + [f"- `{n}` — {d}" for n, d in made]
(Q.D["tables"] / "RESULTS_Q1.md").write_text("\n".join(L))
for n, d in made:
    Q.kv(n, d)
print(f"\n[BLOCK 27 ✓]  {len(made)} figures in q1/figs/ · summary q1/tables/RESULTS_Q1.md · {Q.hms(time.time() - t_all)} · run BLOCK 28 next")

In [ ]:
# ============================================================================
# BLOCK 28 — RELEASE PACKAGE: audit toolkit · fixed leakage-controlled splits · manifests · computational status
# (run last, after BLOCK 25-27; nothing is uploaded anywhere — the zip stays in /kaggle/working/q1/release/)
# ============================================================================
import os, sys, json, time, shutil, csv, hashlib, importlib, warnings, subprocess, zipfile, platform
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("28")

INCLUDE_WEIGHTS = False      # True also puts every kept weights.pt into the zip (about 110 MB each)
AGREE_MAX_PAIRS = 150000     # the toolkit re-derives near-duplicate groups for datasets with at most this many candidate pairs

TOOLKIT_SRC = {
    '.pytest_cache/README.md': r'''# pytest cache directory #

This directory contains data from the pytest's cache plugin,
which provides the `--lf` and `--ff` options, as well as the `cache` fixture.

**Do not** commit this to version control.

See [the docs](https://docs.pytest.org/en/stable/how-to/cache.html) for more information.
''',
    'README.md': r'''# oralleak

A small audit toolkit for data leakage in medical image classification datasets. It was written for the
public oral-cancer image datasets studied in the accompanying article, but it works on any folder of images.

## What it does

- **Hashes** every file (MD5, SHA-256) and computes perceptual hashes (pHash from a 32x32 DCT, dHash from
  a 9x8 gradient) on a 288-px copy of each image.
- **Finds duplicates**: identical files (same MD5) and near duplicates (pHash distance <= 8 and SSIM >= 0.90
  on a 128-px grey copy). If one near-duplicate group grows larger than max(25, 2% of the images), the
  stricter fallback rule (pHash <= 4 and SSIM >= 0.95) is used, so that textures common to many patches do
  not chain unrelated images together.
- **Recovers source groups from file order** (`segment_order`): when patches cut from one source image are
  stored consecutively, the similarity of neighbouring patches drops at source boundaries; the cut-off is an
  Otsu threshold. Embeddings are supplied by the user (any L2-normalised feature vectors).
- **Builds grouped splits** (`grouped_split`, stratified group k-fold) and **checks splits**
  (`check_split`): it lists every relation (identical file, near duplicate, slide/patient/source group,
  augmented copy of the same source) that occurs on both sides of a split.
- **Reports provenance leakage** (`leak_report`): the share of test images that have a relative in the
  training set, by type of relation.

## Command line

```
pip install .
python -m oralleak audit --images /path/to/dataset --out files.csv      # class = parent folder name
python -m oralleak split --files files.csv --group g_near --k 5 --out split.csv
python -m oralleak check --files files.csv --split split.csv --test-fold 0
```

`check` exits with code 1 when any relation crosses the split, so it can be used in a CI job.

## Tests

`python -m pytest -q tests` runs unit tests on synthetic images and, when the package is inside the
release folder, checks every released split in `../splits`.

## Limits

Near-duplicate detection finds re-encoded, resized and lightly edited copies. It does not find crops of
different regions of the same slide; those need slide or patient identifiers (`g_struct`), or the
order-based recovery above when file order follows the source.
''',
    'oralleak/__init__.py': r'''"""oralleak - a small data-leakage audit toolkit for medical image classification datasets.

Functions cover file hashing, exact and near-duplicate detection (pHash + SSIM), recovery of source
groups from file order, grouped splitting, cross-partition checks and a provenance leakage report.
"""
from .core import (load_image, perceptual_hashes, hamming_pairs, gray_small, near_duplicate_groups,
                   union_labels, otsu, segment_order, grouped_split, check_split, leak_report,
                   audit_folder, read_csv, write_csv)

__version__ = "0.1.0"
''',
    'oralleak/__main__.py': r'''"""Command line:  python -m oralleak {audit,check,split} ...

  audit  --images DIR --out files.csv            hash every image, find exact and near duplicates
  split  --files files.csv --group g_near --k 5  stratified group k-fold (writes split.csv: relpath,fold)
  check  --files files.csv --split split.csv     cross-partition checks + leakage report; exit code 1 on leakage
         (split.csv needs relpath and role (train/test) columns, or relpath and fold with --test-fold F)
"""
import argparse
import json
import sys

from . import core


def main(argv=None):
    ap = argparse.ArgumentParser(prog="oralleak")
    sp = ap.add_subparsers(dest="cmd", required=True)
    a = sp.add_parser("audit")
    a.add_argument("--images", required=True)
    a.add_argument("--out", required=True)
    s = sp.add_parser("split")
    s.add_argument("--files", required=True)
    s.add_argument("--group", default="g_ctrl")
    s.add_argument("--label", default="label")
    s.add_argument("--k", type=int, default=5)
    s.add_argument("--seed", type=int, default=7)
    s.add_argument("--out", required=True)
    c = sp.add_parser("check")
    c.add_argument("--files", required=True)
    c.add_argument("--split", required=True)
    c.add_argument("--test-fold", type=int, default=None)
    c.add_argument("--train-roles", default="train")
    c.add_argument("--test-roles", default="test")
    ar = ap.parse_args(argv)

    if ar.cmd == "audit":
        rows, info = core.audit_folder(ar.images)
        core.write_csv(ar.out, rows)
        print(json.dumps(info, indent=1, default=str))
        return 0
    files = core.read_csv(ar.files)
    if ar.cmd == "split":
        key = ar.group if ar.group in files[0] else "g_near"
        fold = core.grouped_split([r[ar.label] for r in files], [r[key] for r in files], ar.k, ar.seed)
        core.write_csv(ar.out, [dict(relpath=r["relpath"], fold=int(f)) for r, f in zip(files, fold)])
        print(f"{len(files)} images -> {ar.k} folds grouped by {key}")
        return 0
    sp_rows = core.read_csv(ar.split)
    if ar.test_fold is not None:
        roles = {r["relpath"]: ("test" if int(r["fold"]) == ar.test_fold else "train") for r in sp_rows}
        trr, ter = ("train",), ("test",)
    else:
        roles = {r["relpath"]: r["role"] for r in sp_rows}
        trr, ter = tuple(ar.train_roles.split(",")), tuple(ar.test_roles.split(","))
    bad = core.check_split(files, roles, trr, ter)
    rep = core.leak_report(files, roles, trr, ter)
    print(json.dumps(dict(violations=bad, leakage=rep), indent=1))
    return 1 if bad else 0


if __name__ == "__main__":
    sys.exit(main())
''',
    'oralleak/core.py': r'''"""Core functions of oralleak.  Dependencies: numpy, scipy, pillow, scikit-image, scikit-learn."""
import csv
import hashlib
import io
import os
from collections import Counter, defaultdict

import numpy as np

STORE_PX = 288          # every image is decoded once and resized to this square size
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
NEAR_RULE = dict(phash_max=8, ssim_min=0.90, cap_frac=0.02, cap_min=25, fallback_phash=4, fallback_ssim=0.95)


# ----------------------------------------------------------------------------- files
def load_image(path, px=STORE_PX):
    """(md5, sha256, uint8 array px x px x 3).  JPEG files are decoded at reduced size first."""
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None
    b = open(path, "rb").read()
    md5, sha = hashlib.md5(b).hexdigest(), hashlib.sha256(b).hexdigest()
    im = Image.open(io.BytesIO(b))
    if im.format == "JPEG":
        im.draft("RGB", (px * 2, px * 2))
    im = im.convert("RGB").resize((px, px), Image.BILINEAR, reducing_gap=2.0)
    return md5, sha, np.asarray(im, dtype=np.uint8).copy()


def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path, rows, header=None):
    header = header or (list(rows[0].keys()) if rows else [])
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=header)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in header})


# ----------------------------------------------------------------------------- perceptual hashes
_POP8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)


def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return _POP8[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)


def _gray(x):
    x = np.asarray(x, np.float32)
    return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]


def _adaptive_mean(g, oh, ow):
    """Same bins as torch.nn.functional.adaptive_avg_pool2d.  g: [N, H, W]."""
    n, h, w = g.shape
    hs = np.array([(i * h) // oh for i in range(oh)])
    he = np.array([-(-((i + 1) * h) // oh) for i in range(oh)])
    ws = np.array([(j * w) // ow for j in range(ow)])
    we = np.array([-(-((j + 1) * w) // ow) for j in range(ow)])
    c = np.zeros((n, h + 1, w + 1), np.float64)
    c[:, 1:, 1:] = g.astype(np.float64).cumsum(1).cumsum(2)
    tot = c[:, he][:, :, we] - c[:, hs][:, :, we] - c[:, he][:, :, ws] + c[:, hs][:, :, ws]
    return (tot / ((he - hs)[:, None] * (we - ws)[None, :])).astype(np.float32)


def perceptual_hashes(arr, bs=256):
    """pHash (32x32 DCT, 8x8 low band vs median) and dHash (9x8 gradient) as uint64 arrays."""
    from scipy.fft import dctn
    n = len(arr)
    ph, dh = np.zeros(n, np.uint64), np.zeros(n, np.uint64)
    w64 = (1 << np.arange(64, dtype=np.uint64)).astype(np.uint64)
    for s in range(0, n, bs):
        g = _gray(arr[s:s + bs])
        g32 = _adaptive_mean(g, 32, 32)
        c = dctn(g32, type=2, norm="ortho", axes=(1, 2))[:, :8, :8].reshape(len(g32), 64)
        med = np.median(c[:, 1:], axis=1, keepdims=True)
        ph[s:s + bs] = ((c > med).astype(np.uint64) * w64).sum(1).astype(np.uint64)
        g98 = _adaptive_mean(g, 8, 9)
        bits = (g98[:, :, 1:] > g98[:, :, :-1]).reshape(len(g98), 64).astype(np.uint64)
        dh[s:s + bs] = (bits * w64).sum(1).astype(np.uint64)
    return ph, dh


def hamming_pairs(h, max_d=8, chunk=512):
    """All pairs i < j with Hamming distance <= max_d."""
    h = np.asarray(h, np.uint64)
    I, J, Dd = [], [], []
    for s in range(0, len(h), chunk):
        d = popcount64(h[s:s + chunk, None] ^ h[None, :])
        ii, jj = np.nonzero(d <= max_d)
        keep = jj > ii + s
        ii, jj = ii[keep], jj[keep]
        I.append(ii + s)
        J.append(jj)
        Dd.append(d[ii, jj])
    if not I:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0, int)
    return np.concatenate(I), np.concatenate(J), np.concatenate(Dd)


def gray_small(arr, px=128):
    return _adaptive_mean(_gray(arr), px, px)


# ----------------------------------------------------------------------------- grouping
def union_labels(n, pairs=(), label_arrays=()):
    """Connected components over n records: explicit pairs plus 'same value in any label array' (-1 = none)."""
    p = np.arange(n)

    def find(a):
        r = a
        while p[r] != r:
            r = p[r]
        while p[a] != r:
            p[a], a = r, p[a]
        return r

    def union(a, b):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb:
            lo, hi = (ra, rb) if ra < rb else (rb, ra)
            p[hi] = lo

    for a, b in pairs:
        union(a, b)
    for lab in label_arrays:
        first = {}
        for i in range(n):
            v = lab[i]
            if v is None or v == -1 or v == "":
                continue
            if v in first:
                union(i, first[v])
            else:
                first[v] = i
    return np.array([find(i) for i in range(n)])


def near_duplicate_groups(md5, ph, grays, rule=None):
    """Group records that are identical files (MD5) or near duplicates (pHash <= phash_max AND SSIM >= ssim_min).
    If the largest group exceeds max(cap_min, cap_frac * n) the stricter fallback rule is used.
    Returns (group id per record, info dict with pair counts)."""
    from skimage.metrics import structural_similarity
    rule = dict(NEAR_RULE, **(rule or {}))
    n = len(md5)
    ii, jj, dd = hamming_pairs(ph, max(rule["phash_max"], rule["fallback_phash"]))
    ss = np.array([structural_similarity(grays[a], grays[b], data_range=255.0) for a, b in zip(ii, jj)])
    by = defaultdict(list)
    for i, m in enumerate(md5):
        by[m].append(i)
    ex = [(g[0], x) for g in by.values() if len(g) > 1 for x in g[1:]]
    info = dict(exact_groups=sum(len(g) > 1 for g in by.values()), exact_redundant=sum(len(g) - 1 for g in by.values()),
                pairs_phash_le={t: int((dd <= t).sum()) for t in (0, 4, 8)},
                pairs_ssim_ge={t: int((ss >= t).sum()) for t in (0.90, 0.95)})
    m = (dd <= rule["phash_max"]) & (ss >= rule["ssim_min"]) if len(dd) else np.zeros(0, bool)
    roots = union_labels(n, pairs=list(zip(ii[m], jj[m])) + ex)
    big = max(Counter(roots.tolist()).values()) if n else 1
    cap = max(rule["cap_min"], rule["cap_frac"] * n)
    info["rule"] = "main"
    if big > cap:
        m = (dd <= rule["fallback_phash"]) & (ss >= rule["fallback_ssim"])
        roots = union_labels(n, pairs=list(zip(ii[m], jj[m])) + ex)
        info["rule"] = f"fallback (largest group {big} > cap {cap:.0f})"
    info["near_pairs"] = int(m.sum())
    info["redundant"] = int(n - len(set(roots.tolist())))
    return roots, info


def otsu(v, bins=256):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 4 or v.max() - v.min() < 1e-9:
        return float(np.median(v)) if len(v) else 0.5
    hist, edges = np.histogram(v, bins=bins)
    cen = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    s0 = np.cumsum(hist * cen)
    mu0 = s0 / np.maximum(w0, 1)
    mu1 = (s0[-1] - s0) / np.maximum(w1, 1)
    k = int(np.argmax((w0 * w1 * (mu0 - mu1) ** 2)[:-1]))
    return float(edges[k + 1])


def segment_order(order, labels, emb):
    """Recover source groups from file order: cut an ordered list wherever the class changes or the cosine
    similarity of consecutive L2-normalised embeddings drops below an Otsu threshold."""
    order = np.asarray(order)
    labels = np.asarray(labels)
    e = np.asarray(emb, np.float32)[order]
    s = (e[:-1] * e[1:]).sum(1)
    same = labels[order][:-1] == labels[order][1:]
    tau = otsu(s[same])
    seg_pos = np.concatenate([[0], np.cumsum((~same) | (s < tau))])
    seg = np.full(len(labels), -1)
    seg[order] = seg_pos
    return seg, dict(tau=tau, n_segments=int(seg_pos.max() + 1))


# ----------------------------------------------------------------------------- splits and checks
def grouped_split(labels, groups, k=5, seed=7):
    """Stratified group k-fold: fold id per record, no group in two folds."""
    from sklearn.model_selection import StratifiedGroupKFold
    labels, groups = np.asarray(labels), np.asarray(groups)
    fold = np.full(len(labels), -1)
    for f, (_, te) in enumerate(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(
            np.zeros(len(labels)), labels, groups)):
        fold[te] = f
    return fold


def _col(rows, key):
    return [r.get(key, "") for r in rows]


def check_split(files, roles, train_roles=("train",), test_roles=("test",),
                keys=("md5", "g_near", "g_struct", "g_ctrl", "parent_group")):
    """Cross-partition checks.  files: list of dicts (relpath + key columns).  roles: dict relpath -> role.
    Returns a list of (key, number of test images whose key value also occurs in training); empty = clean."""
    tr = [r for r in files if roles.get(r["relpath"]) in train_roles]
    te = [r for r in files if roles.get(r["relpath"]) in test_roles]
    out = []
    for k in keys:
        if not files or k not in files[0]:
            continue
        s = {v for v in _col(tr, k) if v not in ("", "-1")}
        bad = sum(1 for v in _col(te, k) if v in s)
        if bad:
            out.append((k, bad))
    return out


def leak_report(files, roles, train_roles=("train",), test_roles=("test",)):
    """Provenance leakage score: share of test images with a relative in training, by relation type."""
    tr = [r for r in files if roles.get(r["relpath"]) in train_roles]
    te = [r for r in files if roles.get(r["relpath"]) in test_roles]
    if not te:
        return {}
    rep = {}
    for name, k in (("identical_file", "md5"), ("near_duplicate", "g_near"), ("same_slide_patient_source", "g_struct"),
                    ("any_relative", "g_ctrl")):
        if k in te[0]:
            s = {v for v in _col(tr, k) if v not in ("", "-1")}
            rep[name] = float(np.mean([v in s for v in _col(te, k)]))
    rep["n_train"], rep["n_test"] = len(tr), len(te)
    return rep


# ----------------------------------------------------------------------------- folder audit
def audit_folder(root, label_from_parent=True, px=STORE_PX, rule=None, log=print):
    """Hash and group every image under root.  The class label is the name of the parent folder."""
    paths = sorted(os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs
                   if os.path.splitext(f)[1].lower() in IMG_EXT)
    if not paths:
        raise SystemExit(f"no images under {root}")
    md5, sha, arr, bad = [], [], np.zeros((len(paths), px, px, 3), np.uint8), []
    for i, p in enumerate(paths):
        try:
            a, b, x = load_image(p, px)
            md5.append(a); sha.append(b); arr[i] = x
        except Exception as e:
            md5.append(f"unreadable-{i}"); sha.append(""); bad.append((p, str(e)[:80]))
        if log and (i + 1) % 2000 == 0:
            log(f"  decoded {i + 1}/{len(paths)}")
    ph, dh = perceptual_hashes(arr)
    grays = gray_small(arr)
    near, info = near_duplicate_groups(md5, ph, grays, rule)
    exact = union_labels(len(paths), label_arrays=[md5])
    rows = []
    for i, p in enumerate(paths):
        rows.append(dict(relpath=os.path.relpath(p, root), label=os.path.basename(os.path.dirname(p)) if label_from_parent else "",
                         md5=md5[i], sha256=sha[i], phash=f"{int(ph[i]):016x}", dhash=f"{int(dh[i]):016x}",
                         g_exact=int(exact[i]), g_near=int(near[i])))
    info.update(n=len(paths), unreadable=len(bad))
    return rows, info
''',
    'pyproject.toml': r'''[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "oralleak"
version = "0.1.0"
description = "Data-leakage audit for medical image classification datasets: hashing, near duplicates, group recovery, split checks"
requires-python = ">=3.9"
dependencies = ["numpy", "scipy", "pillow", "scikit-image", "scikit-learn"]

[tool.setuptools]
packages = ["oralleak"]
''',
    'tests/test_oralleak.py': r'''"""pytest tests for oralleak.  The last two tests also check the released splits when they are present."""
import glob
import os

import numpy as np
import pytest

import oralleak as ol

HERE = os.path.dirname(os.path.abspath(__file__))
SPLITS = os.path.join(HERE, "..", "..", "splits")
# relations that must never cross partitions in the released splits (see splits/README.md)
KEYS = ("md5", "g_near", "g_struct", "parent_group")


def _img(seed, size=160):
    g = np.random.default_rng(seed)
    base = g.integers(0, 255, (8, 8, 3)).astype(np.uint8)
    return np.kron(base, np.ones((size // 8, size // 8, 1), np.uint8))


@pytest.fixture()
def folder(tmp_path):
    from PIL import Image
    for c in ("a", "b"):
        (tmp_path / c).mkdir()
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1.png")
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1_copy.png")                  # identical file content
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1_q90.jpg", quality=90)       # re-encoded near duplicate
    Image.fromarray(_img(2)).save(tmp_path / "a" / "x2.png")
    Image.fromarray(_img(3)).save(tmp_path / "b" / "y1.png")
    Image.fromarray(_img(4)).save(tmp_path / "b" / "y2.png")
    return tmp_path


def test_audit_groups(folder):
    rows, info = ol.audit_folder(str(folder), log=None)
    g = {r["relpath"]: r["g_near"] for r in rows}
    assert g["a/x1.png"] == g["a/x1_copy.png"] == g["a/x1_q90.jpg"]
    assert len({g["a/x2.png"], g["b/y1.png"], g["b/y2.png"], g["a/x1.png"]}) == 4
    assert info["exact_groups"] == 1 and info["redundant"] == 2


def test_check_flags_leak_and_passes_clean(folder):
    rows, _ = ol.audit_folder(str(folder), log=None)
    for r in rows:
        r["g_ctrl"] = r["g_near"]
    leaky = {r["relpath"]: "train" for r in rows}
    leaky["a/x1_copy.png"] = "test"
    assert ol.check_split(rows, leaky), "a duplicate across partitions must be reported"
    assert ol.leak_report(rows, leaky)["identical_file"] == 1.0
    fold = ol.grouped_split([r["label"] for r in rows], [r["g_ctrl"] for r in rows], k=2, seed=0)
    clean = {r["relpath"]: ("test" if f == 0 else "train") for r, f in zip(rows, fold)}
    assert ol.check_split(rows, clean) == []


def test_segment_order_recovers_blocks():
    g = np.random.default_rng(0)
    centers = g.normal(size=(6, 32))
    emb = np.concatenate([c + 0.05 * g.normal(size=(10, 32)) for c in centers])
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    seg, info = ol.segment_order(np.arange(60), np.zeros(60, int), emb)
    assert info["n_segments"] == 6
    assert all(len(set(seg[i * 10:(i + 1) * 10])) == 1 for i in range(6))


@pytest.mark.skipif(not glob.glob(os.path.join(SPLITS, "*_files.csv")), reason="released splits not present")
def test_released_holdouts_are_clean():
    for fcsv in sorted(glob.glob(os.path.join(SPLITS, "*_files.csv"))):
        ds = os.path.basename(fcsv)[:-len("_files.csv")]
        files = ol.read_csv(fcsv)
        for scsv in sorted(glob.glob(os.path.join(SPLITS, f"{ds}_controlled_*.csv"))):
            rows = ol.read_csv(scsv)
            for key in sorted({(r["rep"], r["outer"]) for r in rows}):
                roles = {r["relpath"]: r["role"] for r in rows if (r["rep"], r["outer"]) == key}
                assert ol.check_split(files, roles, ("train", "ival", "own"), ("hold",), KEYS) == [], (scsv, key)


@pytest.mark.skipif(not glob.glob(os.path.join(SPLITS, "*_files.csv")), reason="released splits not present")
def test_released_controlled_arms_are_clean():
    for scsv in sorted(glob.glob(os.path.join(SPLITS, "*_controlled_*.csv"))):
        ds = os.path.basename(scsv).split("_controlled_")[0]
        files = ol.read_csv(os.path.join(SPLITS, f"{ds}_files.csv"))
        rows = ol.read_csv(scsv)
        for key in sorted({(r["rep"], r["outer"]) for r in rows}):
            roles = {r["relpath"]: r["role"] for r in rows if (r["rep"], r["outer"]) == key}
            assert ol.check_split(files, roles, ("train",), ("own", "ival"), KEYS) == [], (scsv, key)
''',
}

t_all = time.time()
P = Q.plan()
BB0 = P["primary_backbone"]
Q.hdr("BLOCK 28 — RELEASE PACKAGE + COMPUTATIONAL STATUS")
stamp = time.strftime("%Y%m%d_%H%M", time.gmtime())
PKG = Q.D["release"] / "package"
if PKG.exists():
    shutil.rmtree(PKG)                     # the package folder is regenerated from q1/ every time
for sub in ("toolkit", "splits", "manifests", "results/eval", "results/tables", "results/figures", "plan", "code"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)
jobs = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
R = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
CL = Q.jload(Q.D["eval"] / "clinical.json", {}) or {}
AU = Q.jload(Q.D["index"] / "audit.json", {}) or {}
avail = Q.available_datasets()
if not avail:
    raise SystemExit("no dataset index found - run BLOCK 16 first")


def wcsv(path, header, rows):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


# ------------------------------------------------------------------ 1  toolkit
Q.sub("1  AUDIT TOOLKIT (oralleak): hashing · near duplicates · group recovery · split checks · leakage report")
for rel, src in TOOLKIT_SRC.items():
    f = PKG / "toolkit" / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(src)
sys.path.insert(0, str(PKG / "toolkit"))
for m in [k for k in list(sys.modules) if k == "oralleak" or k.startswith("oralleak.")]:
    del sys.modules[m]
import oralleak as OL
Q.kv("package", f"oralleak {OL.__version__} · {len(TOOLKIT_SRC)} files in release/package/toolkit")
Q.kv("command line", "python -m oralleak audit | split | check   (see toolkit/README.md)")

# ------------------------------------------------------------------ 2  toolkit agrees with the pipeline?
Q.sub("2  TOOLKIT vs PIPELINE: does the stand-alone code reproduce the hashes and the near-duplicate groups of BLOCK 16?")
rows, AGREE = [], {}
for ds in sorted(avail, key=lambda d: Q.DS.get(d).n):
    dd = Q.DS.get(ds)
    st = Q.open_store(ds)
    H = np.load(Q.D["store"] / f"{ds}_hash.npz")
    use = np.nonzero(dd.ok & dd.orig)[0]
    ph, dh, gr = [], [], []
    for s in range(0, len(use), 512):
        blk = np.asarray(st[use[s:s + 512]])
        a, b = OL.perceptual_hashes(blk)
        ph.append(a); dh.append(b)
    ph, dh = np.concatenate(ph), np.concatenate(dh)
    eq_p = float((ph == H["ph"][use]).mean())
    eq_d = float((dh == H["dh"][use]).mean()) if "dh" in H.files else float("nan")
    ncand = len(OL.hamming_pairs(ph, 8)[0])
    verdict, red = f"skipped ({ncand:,} candidate pairs)", "—"
    if ncand <= AGREE_MAX_PAIRS:
        gr = np.concatenate([OL.gray_small(np.asarray(st[use[s:s + 512]])) for s in range(0, len(use), 512)])
        roots, info = OL.near_duplicate_groups(dd.md5[use], ph, gr)
        g = dd.g_near[use]
        same = len(set(zip(roots.tolist(), g.tolist()))) == len(set(roots.tolist())) == len(set(g.tolist()))
        verdict = "identical partition" if same else "DIFFERENT partition"
        red = f"{info['redundant']} / {len(use) - len(set(g.tolist()))}"
        AGREE[ds] = dict(phash_equal=eq_p, dhash_equal=eq_d, partition_identical=bool(same), rule=info["rule"],
                         redundant_toolkit=info["redundant"], redundant_pipeline=int(len(use) - len(set(g.tolist()))))
    else:
        AGREE[ds] = dict(phash_equal=eq_p, dhash_equal=eq_d, partition_identical=None, candidate_pairs=ncand)
    rows.append([ds, len(use), f"{100 * eq_p:.2f}%", f"{100 * eq_d:.2f}%", verdict, red])
Q.table(["dataset", "originals", "pHash equal", "dHash equal", "near-duplicate groups", "redundant toolkit/pipeline"], rows,
        [12, 10, 12, 12, 34, 26])

# ------------------------------------------------------------------ 3  file manifests with SHA-256
Q.sub("3  FILE MANIFESTS AND HASH LISTS  (relative path, class, MD5, SHA-256, every group label used by the experiments)")
cache_p = Q.D["release"] / "sha256_cache.json"
cache = Q.jload(cache_p, {}) or {}


def sha_one(p):
    try:
        s_ = os.stat(p)
        key = f"{p}|{s_.st_size}|{int(s_.st_mtime)}"
        if key in cache:
            return key, cache[key]
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(1 << 20), b""):
                h.update(b)
        return key, h.hexdigest()
    except Exception:
        return None, ""


FILES, REL = {}, {}
rows = []
t0 = time.time()
for ds in avail:
    dd = Q.DS.get(ds)
    paths = [r["path"] for r in dd.R]
    with ThreadPoolExecutor(16) as ex:
        res = list(ex.map(sha_one, paths))
    for k, h in res:
        if k:
            cache[k] = h
    rel = [os.path.relpath(p, Q.INPUT) for p in paths]
    REL[ds] = rel
    cls = Q.META[ds]["classes"]
    F_ = []
    for i, r in enumerate(dd.R):
        par = int(dd.parent[i])
        F_.append(dict(relpath=rel[i], label=cls[int(dd.y[i])], label_id=int(dd.y[i]), md5=dd.md5[i], sha256=res[i][1],
                       readable=int(dd.ok[i]), original=int(dd.orig[i]), used_in_experiments=int(dd.base[i]),
                       parent_relpath=rel[par] if par >= 0 and dd.child_ok[i] else "", parent_group=int(dd.parent_link[i]),
                       g_exact=int(dd.g_exact[i]), g_near=int(dd.g_near[i]), g_struct=int(dd.g_struct[i]),
                       g_coarse=int(dd.g_coarse[i]), g_ctrl=int(dd.g_ctrl[i]), slide_or_patient=dd.slide[i],
                       native_split=r.get("native", "")))
    FILES[ds] = F_
    OL.write_csv(PKG / "splits" / f"{ds}_files.csv", F_)
    rows.append([ds, len(F_), int(dd.base.sum()), len(set(dd.g_ctrl[dd.base].tolist())), sum(1 for x in F_ if x["sha256"])])
Q.jsave(cache_p, cache)
Q.table(["dataset", "files", "used", "groups (g_ctrl)", "SHA-256 computed"], rows, [12, 8, 8, 16, 18])
Q.kv("hashing time", Q.hms(time.time() - t0))
cx = AU.get("cross")
if cx:
    Q.jsave(PKG / "manifests" / "cross_dataset_near_duplicates.json", cx)

# ------------------------------------------------------------------ 4  fixed leakage-controlled splits
Q.sub("4  FIXED LEAKAGE-CONTROLLED SPLITS  (controlled arm C of E1, seed 0, every outer fold) + independent toolkit checks")
KEYS = ("md5", "g_near", "g_struct", "parent_group")
rows, SPL = [], {}
for ds in avail:
    dd = Q.DS.get(ds)
    sch = P["outer"][ds]
    rel = REL[ds]
    fold_rows, long_rows = [], []
    viol_h = viol_c = 0
    leak_any = []
    nsplit = 0
    for rep in P["reps"].get(ds, P["reps"]["default"]):
        fold, _ = Q.outer_folds(dd, dict(sch, rep=rep))
        fold_rows += [[rel[i], rep, int(fold[i])] for i in np.nonzero(fold >= 0)[0]]
        for o in range(Q.n_outer(dd, dict(sch, rep=rep))):
            J = [j for j in jobs if j["spec"]["exp"] == "E1" and j["spec"]["ds"] == ds and j["spec"]["arm"] == "C"
                 and j["spec"].get("rep", 0) == rep and j["spec"]["outer"] == o and j["spec"]["seed"] == 0
                 and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std"]
            if not J:
                continue
            B = Q.build_job(dict(J[0]["spec"], xeval=False), check=True)   # the pipeline's own disjointness assertions
            roles = {}
            for part in ("train", "ival", "own", "hold"):
                for i in np.unique(B[part]["row"]):
                    roles[rel[i]] = part
                    long_rows.append([rel[i], rep, o, part])
            nsplit += 1
            viol_h += len(OL.check_split(FILES[ds], roles, ("train", "ival", "own"), ("hold",), KEYS))
            viol_c += len(OL.check_split(FILES[ds], roles, ("train",), ("own", "ival"), KEYS))
            lk = OL.leak_report(FILES[ds], roles, ("train",), ("own",))
            leak_any.append(lk.get("same_slide_patient_source", 0.0))
    wcsv(PKG / "splits" / f"{ds}_outer_folds.csv", ["relpath", "rep", "holdout_fold"], fold_rows)
    wcsv(PKG / "splits" / f"{ds}_controlled_E1_seed0.csv", ["relpath", "rep", "outer", "role"], long_rows)
    SPL[ds] = dict(splits=nsplit, holdout_violations=viol_h, controlled_violations=viol_c,
                   max_same_group_share=float(max(leak_any)) if leak_any else None)
    rows.append([ds, sch["kind"], nsplit, viol_h, viol_c, f"{100 * max(leak_any):.1f}" if leak_any else "—"])
Q.table(["dataset", "outer scheme", "splits", "holdout violations", "train/own violations", "max own share with group-mate %"],
        rows, [12, 13, 8, 19, 21, 30])
print("  checked relations: identical file (MD5), near duplicate, slide/patient/source group, augmented copy of the same")
print("  source. Every count must be 0; the pipeline's own assertions (build_job check=True) also passed for every split.")
(PKG / "splits" / "README.md").write_text("\n".join([
    "# Fixed leakage-controlled splits", "",
    "No images are included. Paths are relative to the Kaggle input root; download each dataset from its original source.", "",
    "- `<dataset>_files.csv`: one row per file: class, MD5, SHA-256, and the group labels used by the experiments",
    "  (`g_exact` identical files, `g_near` near duplicates [pHash <= 8 and SSIM >= 0.90, or identical MD5],",
    "  `g_struct` slide / patient / recovered source group, `parent_group` publisher-augmented copy -> source image,",
    "  `g_ctrl` union of all relations). `used_in_experiments` = readable original that represents its identical-file group.",
    "- `<dataset>_outer_folds.csv`: the clean-holdout fold of each used image (`rep` = repetition of the outer split).",
    "- `<dataset>_controlled_E1_seed0.csv`: for every outer fold, the role of each image in the leakage-controlled arm:",
    "  `train`, `ival` (inner validation: model selection, early stopping, temperature, thresholds), `own` (evaluation fold",
    "  of the arm) and `hold` (clean holdout). Images that are not listed for a fold were purged (relatives of holdout",
    "  images) or removed by the per-class size matching.",
    "- Check any split with: `python -m oralleak check --files <dataset>_files.csv --split <split.csv>`.", ""]))

# ------------------------------------------------------------------ 5  runs, weights, environment
Q.sub("5  RUN MANIFEST, WEIGHTS MANIFEST, ENVIRONMENT")
mrows, wrows = [], []
codes, vers = Counter(), Counter()
overlap_bad, n_checked = 0, 0
for j in jobs:
    s = j["spec"]
    rp = Q.D["runs"] / j["id"] / "result.json"
    r = Q.jload(rp) if rp.exists() else None
    st_ = "done" if (r and r.get("status") == "done") else ("failed" if Q.fail_count(j["id"]) else "not run")
    if st_ == "done":
        codes[json.dumps(r.get("code", {}), sort_keys=True)] += 1
        vers[json.dumps(r.get("versions", {}), sort_keys=True)] += 1
        z = np.load(Q.D["runs"] / j["id"] / "preds.npz")
        iv = set(zip(z["ival_row"].tolist(), z["ival_copy"].tolist()))
        ev = set(zip(z["own_row"].tolist(), z["own_copy"].tolist())) | set(zip(z["hold_row"].tolist(), z["hold_copy"].tolist()))
        overlap_bad += int(bool(iv & ev))
        n_checked += 1
    mrows.append([j["id"], j["phase"], s["exp"], s["ds"], s["arm"], s["backbone"], s["recipe"], s["outer"], s.get("rep", 0),
                  s["seed"], s.get("dose"), st_, *(([f"{r['own_bal_acc']:.6f}", f"{r['hold_bal_acc']:.6f}", r["steps_done"],
                  r["best_step"], f"{r['T_ival']:.4f}", r["info"]["sizes"]["train"], r["info"]["sizes"]["ival"],
                  r["info"]["sizes"]["own"], r["info"]["sizes"]["hold"], r.get("img"), f"{r['time_s']:.0f}",
                  r.get("finished_utc"), (r.get("code") or {}).get("q1lib.py"), (r.get("code") or {}).get("q1worker.py")])
                  if st_ == "done" else [""] * 14), j["fp"]])
    wp = Q.D["runs"] / j["id"] / "weights.pt"
    if wp.exists():
        wrows.append([j["id"], f"runs/{j['id']}/weights.pt", f"{wp.stat().st_size / 1e6:.1f}", Q.sha256_file(wp)])
wcsv(PKG / "manifests" / "runs_manifest.csv",
     ["job_id", "phase", "exp", "dataset", "arm", "backbone", "recipe", "outer", "rep", "seed", "dose", "status", "own_bal_acc",
      "hold_bal_acc", "steps_done", "best_step", "temperature_ival", "n_train", "n_ival", "n_own", "n_hold", "img_px",
      "seconds", "finished_utc", "q1lib_hash", "q1worker_hash", "job_fingerprint"], mrows)
wcsv(PKG / "manifests" / "weights_manifest.csv", ["job_id", "file", "MB", "sha256"], wrows)
env = dict(python=platform.python_version(), platform=platform.platform(), run_versions=[json.loads(k) for k in vers],
           run_code_hashes=[json.loads(k) for k in codes], plan_sha256=(Q.jload(Q.D["plan"] / "plan.sha256.json") or {}).get("sha256"),
           lib_version=Q.LIB_VERSION)
try:
    import torch, timm
    env.update(torch=torch.__version__, timm=timm.__version__, cuda=torch.version.cuda,
               gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception:
    pass
Q.jsave(PKG / "manifests" / "environment.json", env)
st_count = Counter(m[11] for m in mrows)
Q.kv("jobs", f"{len(jobs)} · done {st_count.get('done', 0)} · failed {st_count.get('failed', 0)} · not run {st_count.get('not run', 0)}")
Q.kv("code versions among runs", f"{len(codes)} distinct (q1lib/q1worker hashes)" + ("" if len(codes) <= 1 else
     " — runs were trained with different code; see runs_manifest.csv"))
Q.kv("inner validation vs evaluation", f"{n_checked} runs checked · {overlap_bad} with an image unit in both (must be 0)")
Q.kv("weights", f"{len(wrows)} files listed in manifests/weights_manifest.csv" + (" (included in the zip)" if INCLUDE_WEIGHTS
     else " (not zipped; set INCLUDE_WEIGHTS = True to include them)"))

# ------------------------------------------------------------------ (copies) results, plan, code
for src_dir, dst in ((Q.D["eval"], "results/eval"), (Q.D["tables"], "results/tables"), (Q.D["figs"], "results/figures"),
                     (Q.D["plan"], "plan"), (Q.D["code"], "code")):
    if src_dir.exists():
        for f in sorted(src_dir.iterdir()):
            if f.is_file():
                shutil.copy2(f, PKG / dst / f.name)
for f in ("q1lib.py", "q1worker.py"):
    if (Q.D["lib"] / f).exists():
        shutil.copy2(Q.D["lib"] / f, PKG / "code" / f)
shutil.copy2(Q.D["index"] / "audit.json", PKG / "manifests" / "audit.json") if (Q.D["index"] / "audit.json").exists() else None
blocks_kept = sorted(p.name for p in (PKG / "code").glob("block*.py"))

# ------------------------------------------------------------------ (copies) leaderboard baseline on the controlled splits
E1 = R.get("E1", {})
lb = ["# Baseline on the fixed leakage-controlled splits", "",
      f"Model: {Q.BACKBONES[BB0][0]} ({BB0}), {P['img']} px, recipe 'std'. Balanced accuracy (%), mean over outer folds and "
      "seeds. 'Own fold' = the controlled arm's evaluation fold; 'clean holdout' = images whose slide / patient / source "
      "never entered development. Numbers are copied from results/eval/q1_results.json.", "",
      "| Dataset | Outer scheme | Runs | Own fold | Clean holdout | Inflation [95% CI] |", "|---|---|---|---|---|---|"]
for ds in avail:
    a = E1.get(ds, {}).get("arms", {}).get("C")
    if a:
        lb.append(f"| {Q.SHORT.get(ds, ds)} | {P['outer'][ds]['kind']} | {a['n_runs']} | {100 * a['own']:.2f} | {100 * a['hold']:.2f} | "
                  f"{100 * a['infl']:+.2f} [{100 * a['infl_ci'][0]:+.2f}, {100 * a['infl_ci'][1]:+.2f}] |")
(PKG / "LEADERBOARD.md").write_text("\n".join(lb) + "\n")

# ------------------------------------------------------------------ (copies) README, citation and licence placeholders
TODO = "[TO BE COMPLETED BY THE AUTHORS]"
(PKG / "README.md").write_text("\n".join([
    "# Leakage audit of public oral-cancer image datasets: toolkit, fixed splits and results", "",
    f"Package generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 28 of the analysis notebook.", "",
    f"Repository: {TODO}  ·  Archive DOI: {TODO} (no DOI has been minted)  ·  Code licence: {TODO}", "",
    "## Contents", "",
    "- `toolkit/` — `oralleak`, a stand-alone audit package (file hashing, pHash/dHash, SSIM-confirmed near duplicates,",
    "  source recovery from file order, grouped splits, cross-partition checks, provenance leakage report) with pytest tests.",
    "- `splits/` — per-file manifests with MD5/SHA-256 and group labels, clean-holdout folds and the fixed",
    "  leakage-controlled splits (no images; see `splits/README.md`).",
    "- `manifests/` — run manifest (every job, its split sizes, results and code hash), weights manifest, data audit,",
    "  environment.",
    "- `results/` — every table, figure and JSON file produced by BLOCKS 25-27.",
    "- `plan/` — the analysis plan with its local timestamp and SHA-256 (written before any training run).",
    "- `code/` — the pipeline library, the training worker and the source of each notebook block that was run.",
    "- `LEADERBOARD.md` — baseline results on the controlled splits. `FINAL_STATUS_compute.md` — what was and was not done.", "",
    "## Reproducing", "",
    "1. Attach the datasets listed in `splits/*_files.csv` (paths are relative to the Kaggle input root).",
    "2. Run the notebook blocks in the order given in `FINAL_STATUS_compute.md`.",
    "3. Check a split: `cd toolkit && pip install . && python -m oralleak check --files ../splits/<ds>_files.csv "
    "--split <split.csv>`; run the tests with `python -m pytest -q tests`.", "",
    "## Data", "",
    "Images are not redistributed. Each dataset keeps its own licence and terms of use, which the authors must confirm:",
    ""] + [f"- {Q.SHORT.get(ds, ds)}: licence {TODO}" for ds in avail] + [""]))
(PKG / "CITATION.cff").write_text("\n".join([
    "cff-version: 1.2.0", "message: If you use this toolkit or these splits, please cite the article.",
    f'title: "{TODO}"', "authors:", f'  - name: "{TODO}"', f'doi: "{TODO}"', f'repository-code: "{TODO}"', "version: 0.1.0",
    f"date-released: {time.strftime('%Y-%m-%d', time.gmtime())}", ""]))
(PKG / "LICENSE_NOTE.md").write_text("\n".join([
    "# Licence", "", f"Code licence: {TODO} (choose one, e.g. MIT or Apache-2.0, and replace this file with its text).",
    "Split files contain only relative paths, hashes and group labels derived from the public datasets; check that each",
    "dataset's terms allow publishing such derived metadata.", ""]))

# ------------------------------------------------------------------ 6  FINAL_STATUS_compute.md
Q.sub("6  DEFINITION-OF-DONE CHECKLIST (compute side) — also written to FINAL_STATUS_compute.md")
done_ph = Counter(j["phase"] for j in jobs if Q.job_done(j))
all_ph = Counter(j["phase"] for j in jobs)


def seeds_done(ds, arm):
    c = Counter((j["spec"]["outer"], j["spec"].get("rep", 0)) for j in jobs if Q.job_done(j) and j["spec"]["exp"] == "E1"
                and j["spec"]["ds"] == ds and j["spec"]["arm"] == arm and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std")
    return min(c.values()) if c else 0


bb_done = sorted({j["spec"]["backbone"] for j in jobs if Q.job_done(j) and j["spec"]["exp"] == "E1" and j["spec"]["recipe"] == "std"})
fam = {"convnext_tiny.fb_in22k_ft_in1k": "CNN", "resnet50.a1_in1k": "CNN", "efficientnet_b0.ra_in1k": "small CNN",
       "convnext_base.fb_in22k_ft_in1k": "CNN (large)", "vit_small_patch16_224.augreg_in21k_ft_in1k": "transformer"}
fams = sorted({fam.get(b, b) for b in bb_done})
probes_ok = bool(Q.jload(Q.D["eval"] / "probes.json", []))
min_seeds = min([seeds_done(ds, arm) for ds in avail for arm in ("L", "C")] or [0])
dose = R.get("dose", {})
dose_ds = sorted({k.split("|")[1] for k in dose})
reimp = R.get("reimp", {})
reimp_both = [k for k, v in reimp.items() if len(v) >= 2]
strata = R.get("provenance_strata", {})
strata_contrast = [d for d, v in strata.items() if "sibling_minus_none" in v]
h1 = {ds: E1[ds]["arms"]["L"].get("infl_p_holm") for ds in E1 if "L" in E1[ds].get("arms", {})}
plan_meta = Q.jload(Q.D["plan"] / "plan.sha256.json", {}) or {}
first_run = min([Q.jload(Q.D["runs"] / j["id"] / "result.json", {}).get("finished_utc", "9999") for j in jobs if Q.job_done(j)]
                or ["—"])
tests_file = PKG / "toolkit" / "tests" / "test_oralleak.py"


def item(name, status, evidence):
    return [name, status, evidence]


ITEMS = [
    item("Tier 0 errors fixed (213->313, Table 13, comparison counts, '31 studies', placeholders)", "WRITING PHASE",
         "new operating points: results/tables/operating_points.csv; text errata are fixed in the manuscript"),
    item("Selection and early stopping never use evaluation data",
         "DONE" if n_checked and overlap_bad == 0 else ("CHECK" if n_checked else "NOT RUN"),
         f"inner validation carved from training (plan.selection_rules); {n_checked} runs checked, {overlap_bad} overlaps"),
    item("Temperature scaling is fitted on disjoint data", "DONE" if R else "NOT RUN",
         "temperature from inner validation only (BLOCK 25 run_metrics, BLOCK 26); ECE/Brier on own fold and clean holdout"),
    item("Sensitivity and specificity defined consistently, Table 12 rebuilt",
         "DONE (numbers)" if (Q.D["tables"] / "published_vs_ours.csv").exists() else "NOT RUN",
         "positive-class columns separate from macro: results/tables/published_vs_ours.csv, E1_main.csv"),
    item("One evaluation mode per paired comparison", "DONE" if R else "NOT RUN",
         "every comparison = single models, own fold vs clean holdout of the same models; ensembles only in BLOCK 26, labelled"),
    item("Cluster-level CIs and tests, with Holm correction", "DONE" if h1 else "NOT RUN",
         "two-stage cluster bootstrap + within-fold bootstrap + mixed model + GEE; Holm over H1 (q1_results.json E1.*.arms.L)"),
    item("'Pre-specified' claims backed by a timestamp", "PARTIAL" if plan_meta else "NOT RUN",
         f"plan written {plan_meta.get('created_utc', '?')} (SHA-256 {str(plan_meta.get('sha256', '?'))[:12]}) before the "
         f"first finished run ({first_run}); a public timestamp (saved Kaggle version, Git tag or OSF) is human-only"),
    item("Matched-training control and, where possible, a factorial design",
         "DONE" if "factorial_clinical" in R else ("PARTIAL" if E1 else "NOT RUN"),
         "L and C matched per class and in steps (jobs.json Tc); factorial S x Dup on clinical: "
         + ("present" if "factorial_clinical" in R else "general phase not finished")),
    item("Provenance-stratified analysis (sibling vs no sibling)", "DONE" if strata else "NOT RUN",
         f"results/tables/provenance_strata.csv; sibling-vs-none contrast estimable for: {', '.join(strata_contrast) or 'none'}"
         + ("" if strata_contrast else " (no leaky own fold has both strata: under augment-before-split nearly every test image "
            "has a sibling in training, so the clean holdout carries the comparison)")),
    item("Dose-response curves on at least two datasets", "DONE" if len(dose_ds) >= 2 else ("PARTIAL" if dose_ds else "NOT RUN"),
         f"datasets with dose curves: {', '.join(dose_ds) or 'none'} (DOSEK / DUP / OVL, q1_results.json dose)"),
    item("Perceptual and embedding near-duplicate audit, with positive and negative controls",
         "DONE" if AU and "RL" in R and any(k.startswith("DUP|") for k in dose) else ("PARTIAL" if AU else "NOT RUN"),
         "index/audit.json (MD5, pHash, dHash, SSIM, embedding cosine, cross-dataset); positive control DUP; negative controls: "
         "augment-after-split arm C and random labels (RL)"),
    item("At least 3 seeds and at least 3 architecture families",
         "DONE" if min_seeds >= 3 and "transformer" in fams and probes_ok else ("PARTIAL" if min_seeds or bb_done else "NOT RUN"),
         f"minimum finished seeds per E1 fold: {min_seeds}; fine-tuned families: {', '.join(fams) or 'none'}; "
         f"frozen DINOv2 linear probe: {'yes' if probes_ok else 'no'}"),
    item("At least 2 additional datasets, one with patient IDs", "PARTIAL" if "breakhis" in avail else "BLOCKED",
         "BreakHis (non-oral, patient IDs) added" if "breakhis" in avail else "BreakHis not attached ('ambarish/breakhis')"),
    item("At least 2 published methods reimplemented under both protocols",
         "DONE" if len(reimp_both) >= 2 else ("PARTIAL" if reimp else "NOT RUN"), f"{', '.join(reimp_both) or 'none finished'}"),
    item("Literature analysis of every surveyed study, with the true study count", "WRITING PHASE", "not a compute task"),
    item("Audit toolkit, corrected splits, manifests and weights released with a DOI", "PARTIAL",
         "toolkit, splits and manifests in this zip; deposit and DOI are human-only; weights listed, not zipped by default"),
    item("Novelty claims verified against prior leakage literature", "WRITING PHASE", "not a compute task"),
    item("Abstract and conclusion rewritten to match the evidence", "WRITING PHASE", "uses results/ from this package"),
    item("CLAIM checklist completed and dataset licences checked", "WRITING PHASE / HUMAN", "licence terms are human-only"),
]
Q.table(["item", "status", "evidence"], [[i[0][:56], i[1], i[2][:56]] for i in ITEMS], [58, 23, 58])

# honest scale-down and surprises (generated from the numbers, not written by hand)
scale = [P.get("deviations", "")]
for ph in P["phases"]:
    if all_ph.get(ph) and done_ph.get(ph, 0) < all_ph[ph]:
        scale.append(f"phase '{ph}': {done_ph.get(ph, 0)} of {all_ph[ph]} jobs finished; the missing runs are reported as missing.")
if "histopath" in avail and not AU.get("histopath", {}).get("pre_enabled", False):
    scale.append("PRE experiment disabled: parent recovery of the publisher-augmented files was below the pre-set rate.")
if "breakhis" not in avail:
    scale.append("BreakHis not attached: no dataset with patient identifiers.")
surprises = []
for ds, p in h1.items():
    if p is not None and p >= 0.05:
        surprises.append(f"H1 not supported for {ds}: leaky-arm inflation {100 * E1[ds]['arms']['L']['infl']:+.2f} points, Holm p = {p:.3g}.")
for ds in E1:
    c = E1[ds]["arms"].get("C")
    if c and (c["infl_ci"][0] > 0 or c["infl_ci"][1] < 0):
        surprises.append(f"H3 fails for {ds}: the controlled arm is itself biased (inflation {100 * c['infl']:+.2f} "
                         f"[{100 * c['infl_ci'][0]:+.2f}, {100 * c['infl_ci'][1]:+.2f}]).")
    if E1[ds].get("dhold_ci") and (E1[ds]["dhold_ci"][0] > 0 or E1[ds]["dhold_ci"][1] < 0):
        surprises.append(f"H4 for {ds}: clean-holdout BA differs between leaky and controlled training "
                         f"({100 * E1[ds]['dhold']:+.2f} [{100 * E1[ds]['dhold_ci'][0]:+.2f}, {100 * E1[ds]['dhold_ci'][1]:+.2f}]).")
for k, v in R.get("RL", {}).items():
    if k.endswith("|L") and (v.get("p_vs_chance") or 1) >= 0.05:
        surprises.append(f"random labels, {k.split('|')[0]}: the leaky arm is not above chance on its own fold (p = {v['p_vs_chance']:.3g}).")
for k, v in dose.items():
    exn, _, arm_ = k.split("|")
    if v.get("slope") is None:
        continue
    if exn == "DOSEK" and arm_ == "C":        # augmenting after the split should NOT raise inflation
        if v["slope"] > 0 and (v.get("p") or 1) < 0.05:
            surprises.append(f"dose-response {k}: inflation rises with k even though augmentation happens after the split "
                             f"(slope {100 * v['slope']:+.2f}, p = {v['p']:.4f}).")
        continue
    if v["slope"] <= 0 or (v.get("p") or 1) >= 0.05:
        surprises.append(f"dose-response {k}: no clear increase (slope {100 * v['slope']:+.2f}, p = {v.get('p') if v.get('p') is None else round(v['p'], 4)}).")
rp = R.get("risk_predictor", {})
for name, v in rp.items():
    if isinstance(v, dict) and "lodo_r2" in v and name.startswith("one") and v["lodo_r2"] <= 0:
        surprises.append(f"leakage-risk predictor: leave-one-dataset-out R² = {v['lodo_r2']:.2f} (no predictive value across datasets).")
nw = [k for k, v in CL.get("saliency", {}).items() if not (v["del_gain_ci"][0] > 0 and v["ins_gain_ci"][0] > 0)]
if nw:
    surprises.append(f"Grad-CAM not better than random order for {', '.join(nw)}: those maps are illustration only.")
worse = [k for k, v in CL.get("calibration", {}).items() if v["cal"]["ece"] > v["raw"]["ece"]]
if worse:
    surprises.append(f"temperature scaling (fitted on inner validation) increased holdout ECE for {', '.join(worse)}.")
for ds, a in AGREE.items():
    if a.get("partition_identical") is False:
        surprises.append(f"toolkit and pipeline near-duplicate groups differ for {ds}.")
human = ["repository URL", "archive DOI (Zenodo or similar) — none minted", "code licence", "licence terms of every dataset",
         "public timestamp of the analysis plan (saved Kaggle version, Git tag or OSF registration)", "funding statement",
         "author contributions", "ethics / data-provenance statement sign-off", "citation metadata (CITATION.cff)"]
order = ("First session: 15 -> Save Version -> 16 -> 17 -> 18 -> 19 -> 20 -> 21 -> 22.  Later sessions: 15 -> 19 -> next "
         "training block (23, then 24; re-run a block to resume it).  Final session: 15 -> 25 -> 26 -> 27 -> 28.")
md = ["# FINAL_STATUS (computational part)", "",
      f"Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 28 from the files in q1/. "
      f"Plan SHA-256 `{plan_meta.get('sha256', '?')}`. Jobs: {len(jobs)}, finished {st_count.get('done', 0)}, "
      f"failed {st_count.get('failed', 0)}, not run {st_count.get('not run', 0)}.", "",
      "## Definition of done", "", "| Item | Status | Evidence |", "|---|---|---|"]
md += [f"| {a} | {b} | {c} |" for a, b, c in ITEMS]
md += ["", "## Jobs by phase", "", "| Phase | Finished | Total |", "|---|---|---|"]
md += [f"| {ph} | {done_ph.get(ph, 0)} | {all_ph.get(ph, 0)} |" for ph in P["phases"] if all_ph.get(ph)]
md += ["", "## Scaled down or skipped", ""] + [f"- {x}" for x in scale if x]
md += ["", "## Results that go against the original thesis or the plan's expectations", ""]
md += [f"- {x}" for x in surprises] or ["- none detected automatically; read BLOCK 25-26 outputs before writing"]
md += ["", "## Toolkit agreement with the pipeline", ""]
md += [f"- {ds}: pHash equal {100 * a['phash_equal']:.2f}%, near-duplicate partition "
       f"{'identical' if a.get('partition_identical') else ('different' if a.get('partition_identical') is False else 'not re-derived (size)')}"
       for ds, a in AGREE.items()]
md += ["", "## Human-only items left open", ""] + [f"- {x}" for x in human]
md += ["", "## Block order used", "", order, "", f"Blocks whose source is stored in code/: {', '.join(blocks_kept) or 'none'} "
       "(a block's source is stored when it runs; re-run a block to store it)."]
(PKG / "FINAL_STATUS_compute.md").write_text("\n".join(md) + "\n")
if surprises:
    print("  results against expectations (all listed in FINAL_STATUS_compute.md):")
    for x in surprises[:12]:
        print("   -", x)
    if len(surprises) > 12:
        print(f"   ... and {len(surprises) - 12} more")

# ------------------------------------------------------------------ 7  toolkit tests on the packaged splits
Q.sub("7  TOOLKIT TESTS  (pytest on the package, including every released split)")
try:
    import pytest  # noqa
    pt = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=PKG / "toolkit",
                        capture_output=True, text=True, timeout=1800)
    last = [l for l in pt.stdout.strip().splitlines() if l.strip()][-1:] or ["(no output)"]
    Q.kv("pytest", last[0] + ("" if pt.returncode == 0 else "  <- FAILED, see release/pytest.log"))
    (Q.D["release"] / "pytest.log").write_text(pt.stdout + pt.stderr)
    TEST_OK = pt.returncode == 0
except ImportError:
    Q.kv("pytest", "not installed; the split checks in section 4 are the test result")
    TEST_OK = all(v["holdout_violations"] == 0 and v["controlled_violations"] == 0 for v in SPL.values())

# ------------------------------------------------------------------ 8  zip
Q.sub("8  ZIP")
zp = Q.D["release"] / f"oralleak_release_{stamp}.zip"
n_files = 0
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(PKG.rglob("*")):
        if f.is_file() and "__pycache__" not in f.parts:
            zf.write(f, f"oralleak_release/{f.relative_to(PKG)}")
            n_files += 1
    if INCLUDE_WEIGHTS:
        for w in wrows:
            zf.write(Q.D["runs"] / w[0] / "weights.pt", f"oralleak_release/weights/{w[0]}.pt")
            n_files += 1
Q.kv("archive", f"{zp.relative_to(Q1_ROOT.parent)} · {n_files} files · {zp.stat().st_size / 1e6:.1f} MB")
Q.kv("status file", "release/package/FINAL_STATUS_compute.md")
Q.kv("nothing uploaded", "the zip stays in /kaggle/working; publishing and a DOI are left to the authors")
print(f"\n[BLOCK 28 ✓]  release package built · toolkit tests {'passed' if TEST_OK else 'FAILED'} · {Q.hms(time.time() - t_all)}")

In [ ]:
# ============================================================================
# BLOCK 15 — Q1 SETUP: LIBRARY + ANALYSIS PLAN (fixed before training) + ENVIRONMENT
# ============================================================================
import os, sys, json, time, importlib, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
(Q1_ROOT / "lib").mkdir(parents=True, exist_ok=True)
Q1LIB_SRC = r'''"""
q1lib.py - shared library for the OralNet Q1 leakage study.
Written to disk by Block 15.  Do not edit by hand: re-run Block 15 to refresh it.

Sections
  1  paths, printing, json, hashing, session clock
  2  dataset registry and file parsing
  3  image store, exact hashes, perceptual hashes
  4  torch helpers: models, embeddings, GPU augmentation (siblings, train, eval)
  5  near-duplicate graph, union-find, index-order segmentation, parent recovery
  6  split builder: clean holdout, leaky / controlled arms, matching, leak stats, disjointness tests
  7  metrics, calibration, operating points, cluster bootstrap, GEE, ICC, Holm, mixed model
"""
import os, sys, io, re, json, time, math, hashlib, random, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

LIB_VERSION = "q1-1.0"

# =====================================================================================
# 1  paths, printing, json, hashing, session clock
# =====================================================================================
ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
INPUT = Path(os.environ.get("Q1_INPUT", "/kaggle/input"))
OFFLINE = os.environ.get("Q1_OFFLINE", "0") == "1"      # tests only: no pretrained downloads
TINY = os.environ.get("Q1_TINY", "0") == "1"            # tests only: tiny images and models
STORE_PX = 64 if TINY else 288

D = {k: ROOT / v for k, v in dict(
    lib="lib", plan="plan", index="index", store="store", jobs="jobs", claims="jobs/claims",
    fail="jobs/fail", runs="runs", superseded="runs_superseded", ckpt="ckpt", logs="logs",
    eval="eval", figs="figs", tables="tables", release="release", state="state", code="code").items()}


def ensure_dirs():
    for p in D.values():
        p.mkdir(parents=True, exist_ok=True)


W = 100


def hdr(t):
    print("=" * W)
    print(t)
    print("=" * W)


def sub(t):
    print()
    print("-" * W)
    print(t)
    print("-" * W)


def kv(k, v, w=30):
    print(f"  {k:<{w}} {v}")


def table(headers, rows, widths=None, indent=2):
    cols = len(headers)
    if widths is None:
        widths = []
        for i in range(cols):
            m = max([len(str(headers[i]))] + [len(str(r[i])) for r in rows]) if rows else len(str(headers[i]))
            widths.append(min(max(m + 2, 6), 44))
    pad = " " * indent
    print(pad + "".join(f"{str(h):<{w}}" for h, w in zip(headers, widths)))
    print(pad + "-" * min(sum(widths), W - indent))
    for r in rows:
        print(pad + "".join(f"{str(c):<{w}}" for c, w in zip(r, widths)))


def hms(sec):
    sec = max(0, int(sec))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return f"{h}h{m:02d}m" if h else (f"{m}m{s:02d}s" if m else f"{s}s")


def pct(x, d=2):
    return "—" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{100 * x:.{d}f}"


def _jdef(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, Path):
        return str(o)
    if isinstance(o, set):
        return sorted(o)
    raise TypeError(str(type(o)))


def _clean(o):
    if isinstance(o, float):
        return o if math.isfinite(o) else None
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.floating):
        v = float(o)
        return v if math.isfinite(v) else None
    return o


def jload(p, default=None):
    p = Path(p)
    if not p.exists():
        return default
    try:
        return json.loads(p.read_text())
    except Exception:
        return default


def jsave(p, obj, indent=1):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + f".tmp{os.getpid()}")
    tmp.write_text(json.dumps(_clean(obj), indent=indent, default=_jdef))
    os.replace(tmp, p)


def shash(*parts):
    h = hashlib.blake2b("|".join(map(str, parts)).encode(), digest_size=8)
    return int.from_bytes(h.digest(), "little") & ((1 << 62) - 1)


def fingerprint(obj):
    return hashlib.sha1(json.dumps(_clean(obj), sort_keys=True, default=_jdef).encode()).hexdigest()[:12]


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def code_hashes():
    out = {}
    for n in ("q1lib.py", "q1worker.py"):
        p = D["lib"] / n
        out[n] = sha256_file(p)[:16] if p.exists() else None
    return out


def keep_source(tag):
    """Save the source of the notebook cell that is running now to q1/code/block<tag>.py (IPython only), so the
    release package contains the exact code behind every table and figure."""
    try:
        from IPython import get_ipython
        ip = get_ipython()
        src = ip.user_ns.get("In", [""])[-1] if ip is not None else ""
        if src and f"BLOCK {tag}" in src[:600]:
            D["code"].mkdir(parents=True, exist_ok=True)
            (D["code"] / f"block{tag}.py").write_text(src)
            return True
    except Exception:
        pass
    return False


def disk_free_gb(p=None):
    import shutil
    p = p or (ROOT if ROOT.exists() else Path("/"))
    return shutil.disk_usage(p).free / 1e9


def kernel_start():
    """Start time of the current Python process (the notebook kernel when called from a cell)."""
    try:
        import psutil
        return psutil.Process(os.getpid()).create_time()
    except Exception:
        return time.time()


def session_clock(budget_min=660):
    """Session start = kernel start (first call per kernel is recorded).  Returns dict with deadline."""
    st = jload(D["state"] / "session.json", {}) or {}
    ks = kernel_start()
    if abs(st.get("kernel_start", 0) - ks) > 2:
        st = dict(kernel_start=ks, n=int(st.get("n", 0)) + 1, first_seen=time.time())
        jsave(D["state"] / "session.json", st)
    st["deadline"] = ks + budget_min * 60
    st["elapsed"] = time.time() - ks
    st["remaining"] = st["deadline"] - time.time()
    return st


def plan():
    p = jload(D["plan"] / "plan.json")
    if p is None:
        raise RuntimeError("plan.json missing - run Block 15 first")
    return p


# =====================================================================================
# 2  dataset registry and file parsing
# =====================================================================================
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
DATASETS = ["he_stained", "histopath", "ndb_ufes", "clinical", "breakhis"]
META = {
    "he_stained": dict(K=2, classes=["non_tumor", "tumor"], pos=1, modality="he", group="slide",
                       title="OCDC H&E patches (7 slides)"),
    "histopath": dict(K=2, classes=["Normal", "OSCC"], pos=1, modality="he", group="source image",
                      title="Oral histopathology, Normal vs OSCC (Kaggle mirror)"),
    "ndb_ufes": dict(K=3, classes=["Healthy", "Leukoplakia", "OSCC"], pos=2, modality="he",
                     group="recovered source image", title="NDB-UFES patches (Kaggle mirror)"),
    "clinical": dict(K=2, classes=["non-cancer", "cancer"], pos=1, modality="photo", group="duplicate cluster",
                     title="Oral cancer lip and tongue photographs"),
    "breakhis": dict(K=2, classes=["benign", "malignant"], pos=1, modality="he", group="patient",
                     title="BreakHis breast H&E (patient IDs)"),
}
SHORT = {"he_stained": "OCDC", "histopath": "histo", "ndb_ufes": "NDB", "clinical": "clin", "breakhis": "BrkHs"}


def load_registry():
    """Extension datasets (added by analysis-plan amendments) are registered in index/_registry.json.
    They get META / SHORT entries but are never part of DATASETS, so the pre-registered analyses are unchanged."""
    reg = jload(D["index"] / "_registry.json", {}) or {}
    for name, m in reg.items():
        META[name] = dict(m["meta"])
        SHORT[name] = m["short"]
    return reg


def ext_datasets():
    return [d for d in (jload(D["index"] / "_registry.json", {}) or {}) if (D["index"] / f"{d}.json").exists()]


load_registry()

NDB_CLASSES = {"healthy": 0, "leukoplakia": 1, "oral squamous cell carcinoma": 2}
RE_HIST_ORIG = re.compile(r"^(oscc|normal)_(\d+)x_(\d+)\.[a-z]+$", re.I)
RE_HIST_AUG = re.compile(r"^aug_(\d+)_(\d+)\.[a-z]+$", re.I)
RE_BH = re.compile(r"^SOB_([BM])_([A-Za-z]+)-(\d+)-([A-Za-z0-9]+)-(\d+)-(\d+)\.png$", re.I)


def classify_file(p):
    p = Path(p)
    name = p.name
    parts = [x.lower() for x in p.parts]
    par = parts[-2] if len(parts) > 1 else ""
    gpar = parts[-3] if len(parts) > 2 else ""
    m = RE_BH.match(name)
    if m:
        return "breakhis", dict(path=str(p), label=1 if m.group(1).upper() == "M" else 0,
                                patient=f"{m.group(3)}-{m.group(4)}", subtype=m.group(2).upper(),
                                mag=m.group(5) + "x", seq=int(m.group(6)))
    if par == "01-original" and ("01-roi" in parts or "02-non_roi" in parts):
        tum = "01-roi" in parts
        i = parts.index("01-roi" if tum else "02-non_roi")
        native = "test" if "testing" in parts else ("train" if "training" in parts else "")
        return "he_stained", dict(path=str(p), label=int(tum), slide=p.parts[i - 1], native=native)
    if par in NDB_CLASSES and re.match(r"^p\d+\.[a-z]+$", name, re.I):
        return "ndb_ufes", dict(path=str(p), label=NDB_CLASSES[par], pidx=int(re.findall(r"\d+", name)[0]))
    if par in ("oscc", "normal") and gpar in ("train", "val", "test"):
        lab = 1 if par == "oscc" else 0
        m = RE_HIST_ORIG.match(name)
        if m:
            return "histopath", dict(path=str(p), label=lab, orig=True, native=gpar, mag=m.group(2) + "x",
                                     pidx=int(m.group(3)))
        m = RE_HIST_AUG.match(name)
        a, b = (int(m.group(1)), int(m.group(2))) if m else (-1, -1)
        return "histopath", dict(path=str(p), label=lab, orig=False, native=gpar, aug_a=a, aug_b=b)
    if par in ("cancer", "non-cancer") and gpar == "oralcancer":
        return "clinical", dict(path=str(p), label=1 if par == "cancer" else 0)
    return None


def walk_input(root=None):
    root = Path(root or INPUT)
    found = defaultdict(list)
    n_img = 0
    for dp, dn, fn in os.walk(root):
        dn[:] = sorted(d for d in dn if not d.startswith("."))
        for f in sorted(fn):
            if os.path.splitext(f)[1].lower() not in IMG_EXT:
                continue
            n_img += 1
            r = classify_file(Path(dp) / f)
            if r:
                found[r[0]].append(r[1])
    for ds in found:
        found[ds].sort(key=lambda r: r["path"])
    return dict(found), n_img


# =====================================================================================
# 3  image store, exact hashes, perceptual hashes
# =====================================================================================
def _load_one(args):
    path, px = args
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None
    try:
        b = open(path, "rb").read()
    except Exception:
        return None, np.zeros((px, px, 3), np.uint8), (0, 0), "", False, 0
    md5 = hashlib.md5(b).hexdigest()
    try:
        im = Image.open(io.BytesIO(b))
        w, h = im.size
        mode = im.mode
        if im.format == "JPEG":
            im.draft("RGB", (px * 2, px * 2))
        im = im.convert("RGB").resize((px, px), Image.BILINEAR, reducing_gap=2.0)
        arr = np.asarray(im, dtype=np.uint8).copy()
        ok = True
    except Exception:
        arr, w, h, mode, ok = np.zeros((px, px, 3), np.uint8), 0, 0, "", False
    return md5, arr, (w, h), mode, ok, len(b)


def build_store(ds, recs, px=None, workers=4, log_every=2000):
    """Decode every file once: MD5 + resized uint8 array into store/<ds>.npy (memmap)."""
    from multiprocessing import get_context
    px = px or STORE_PX
    out = D["store"] / f"{ds}.npy"
    tmp = D["store"] / f"{ds}.npy.partial"
    n = len(recs)
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(n, px, px, 3))
    meta = []
    t0 = time.time()
    ctx = get_context("fork")
    with ctx.Pool(workers) as pool:
        for i, res in enumerate(pool.imap(_load_one, [(r["path"], px) for r in recs], chunksize=16)):
            md5, a, (w, h), mode, ok, nb = res
            arr[i] = a
            meta.append(dict(md5=md5 or f"missing{i}", w=w, h=h, mode=mode, ok=bool(ok), bytes=nb))
            if log_every and (i + 1) % log_every == 0:
                print(f"    {ds:<11} {i + 1}/{n}  ({time.time() - t0:.0f}s)", flush=True)
    arr.flush()
    del arr
    os.replace(tmp, out)
    return meta, time.time() - t0


def open_store(ds):
    return np.load(D["store"] / f"{ds}.npy", mmap_mode="r")


_POP8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)


def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return _POP8[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)


def perceptual_hashes(store, bs=512):
    """pHash (32x32 DCT, 8x8 low band vs median) and dHash (9x8 gradient) as uint64."""
    from scipy.fft import dctn
    import torch
    import torch.nn.functional as F
    n = store.shape[0]
    ph = np.zeros(n, np.uint64)
    dh = np.zeros(n, np.uint64)
    w64 = (1 << np.arange(64, dtype=np.uint64)).astype(np.uint64)
    for s in range(0, n, bs):
        x = torch.from_numpy(np.asarray(store[s:s + bs])).float()
        g = (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]).unsqueeze(1)
        g32 = F.adaptive_avg_pool2d(g, (32, 32))[:, 0].numpy()
        c = dctn(g32, type=2, norm="ortho", axes=(1, 2))[:, :8, :8].reshape(len(g32), 64)
        med = np.median(c[:, 1:], axis=1, keepdims=True)
        bits = (c > med).astype(np.uint64)
        ph[s:s + bs] = (bits * w64).sum(1).astype(np.uint64)
        g98 = F.adaptive_avg_pool2d(g, (8, 9))[:, 0].numpy()
        bits = (g98[:, :, 1:] > g98[:, :, :-1]).reshape(len(g98), 64).astype(np.uint64)
        dh[s:s + bs] = (bits * w64).sum(1).astype(np.uint64)
    return ph, dh


def hamming_pairs(ha, hb=None, max_d=12, chunk=512):
    """All pairs with Hamming distance <= max_d.  Within one set (hb None, i<j) or across two sets."""
    ha = np.asarray(ha, np.uint64)
    same = hb is None
    hb = ha if same else np.asarray(hb, np.uint64)
    I, J, Dd = [], [], []
    for s in range(0, len(ha), chunk):
        x = ha[s:s + chunk, None] ^ hb[None, :]
        d = popcount64(x)
        ii, jj = np.nonzero(d <= max_d)
        ii = ii + s
        if same:
            keep = jj > ii
            ii, jj = ii[keep], jj[keep]
        I.append(ii)
        J.append(jj)
        Dd.append(d[ii - s, jj])
    if not I:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0, int)
    return np.concatenate(I), np.concatenate(J), np.concatenate(Dd).astype(int)


def gray_small(store, idx, px=128):
    import torch
    import torch.nn.functional as F
    x = torch.from_numpy(np.asarray(store[np.sort(np.unique(idx))])).float()
    g = (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]).unsqueeze(1)
    g = F.adaptive_avg_pool2d(g, (px, px))[:, 0].numpy()
    return {int(i): g[k] for k, i in enumerate(np.sort(np.unique(idx)))}


def ssim_pairs(store_a, ia, store_b, ib, px=128, max_pairs=300000):
    from skimage.metrics import structural_similarity
    ia, ib = np.asarray(ia), np.asarray(ib)
    if len(ia) > max_pairs:
        ia, ib = ia[:max_pairs], ib[:max_pairs]
    if len(ia) == 0:
        return np.zeros(0)
    if store_b is store_a:
        ga = gray_small(store_a, np.concatenate([ia, ib]), px)
        gb_ = ga
    else:
        ga = gray_small(store_a, ia, px)
        gb_ = gray_small(store_b, ib, px)
    out = np.zeros(len(ia))
    for k, (i, j) in enumerate(zip(ia, ib)):
        out[k] = structural_similarity(ga[int(i)], gb_[int(j)], data_range=255.0)
    return out


# =====================================================================================
# 4  torch helpers: models, embeddings, GPU augmentation
# =====================================================================================
EMB_MODELS = {"dinov2_s": "vit_small_patch14_dinov2.lvd142m", "convnext_t": "convnext_tiny.fb_in22k_ft_in1k",
              "phikon": "owkin/phikon"}
BACKBONES = {  # name -> (short label, parameter count in millions, default img/s on one T4 at 224 px, fp16)
    "convnext_tiny.fb_in22k_ft_in1k": ("ConvNeXt-T", 28.6, 300),
    "resnet50.a1_in1k": ("ResNet-50", 25.6, 380),
    "vit_small_patch16_224.augreg_in21k_ft_in1k": ("ViT-S/16", 22.1, 380),
    "efficientnet_b0.ra_in1k": ("EffNet-B0", 5.3, 520),
    "convnext_base.fb_in22k_ft_in1k": ("ConvNeXt-B", 88.6, 100),
    "efficientnet_b3.ra2_in1k": ("EffNet-B3", 12.2, 170),
    "resnet18": ("ResNet-18", 11.7, 900),
}


def tiny_name(name):
    return "resnet18" if TINY else name


def make_model(name, K, img=224, drop_path=0.0, head="linear", pretrained=None):
    """Returns (model, head_params, dense_layer_for_regularisation or None)."""
    import timm
    import torch.nn as nn
    name = tiny_name(name)
    pretrained = (not OFFLINE) if pretrained is None else pretrained
    kw = {}
    if any(t in name for t in ("vit_", "dinov2", "deit", "eva")):
        kw["img_size"] = img
    if drop_path:
        kw["drop_path_rate"] = drop_path
    if head == "scirep":
        base = timm.create_model(name, pretrained=pretrained, num_classes=0, **kw)
        nf = base.num_features
        hd = nn.Sequential(nn.BatchNorm1d(nf), nn.Linear(nf, 256), nn.ReLU(inplace=True), nn.Dropout(0.45),
                           nn.Linear(256, K))

        class _Wrap(nn.Module):
            def __init__(s, b, h):
                super().__init__()
                s.base, s.head = b, h
                s.pretrained_cfg = getattr(b, "pretrained_cfg", {})

            def forward(s, x):
                return s.head(s.base(x))

        m = _Wrap(base, hd)
        return m, list(hd.parameters()), hd[1]
    m = timm.create_model(name, pretrained=pretrained, num_classes=K, **kw)
    return m, list(m.get_classifier().parameters()), None


def norm_consts(model, device):
    import torch
    cfg = getattr(model, "pretrained_cfg", None) or {}
    mean = cfg.get("mean", (0.485, 0.456, 0.406))
    std = cfg.get("std", (0.229, 0.224, 0.225))
    return (torch.tensor(mean, device=device).view(1, 3, 1, 1), torch.tensor(std, device=device).view(1, 3, 1, 1))


POLICIES = {
    # Abd El-Aziz 2025: rotation 10 deg, width/height shift 10%, shear 10%, zoom 10%, horizontal flip,
    # brightness 0.2-1.2, applied before the split (Keras ImageDataGenerator style, nearest fill)
    "abdelaziz": dict(rot=10, shift=0.10, shear=0.10, zoom=0.10, hflip=True, vflip=False,
                      bright=(0.2, 1.2), contrast=None, sat=None),
    # generic Keras-style policy used for the clinical photographs (augment-before-split replication)
    "keras_generic": dict(rot=20, shift=0.10, shear=0.10, zoom=0.10, hflip=True, vflip=False,
                          bright=(0.8, 1.2), contrast=None, sat=None),
    # H&E policy for the NDB-UFES dose-response arm: flips, rotation 30 deg, mild zoom and colour
    "he_d4": dict(rot=30, shift=0.0, shear=0.0, zoom=0.10, hflip=True, vflip=True,
                  bright=(0.9, 1.1), contrast=(0.9, 1.1), sat=(0.9, 1.1)),
}
TRAIN_PROFILES = {
    "std_he": dict(rrc=(0.7, 1.0), hflip=True, vflip=True, rot90=True, rot=0, jitter=0.1),
    "std_photo": dict(rrc=(0.7, 1.0), hflip=True, vflip=False, rot90=False, rot=0, jitter=0.1),
    "scirep_ndb": dict(rrc=None, hflip=True, vflip=True, rot90=False, rot=30, jitter=0.0),
    "scirep_hist": dict(rrc=None, hflip=True, vflip=False, rot90=False, rot=0, jitter=0.0),
    "none": dict(rrc=None, hflip=False, vflip=False, rot90=False, rot=0, jitter=0.0),
}
IDENT = np.array([0, 0, 0, 0, 1, 0, 0, 1, 1, 1], np.float32)


def sibling_params(ds, rows, copies, policy):
    """Deterministic augmentation parameters of sibling copy c of store row r (c = 0 is the original)."""
    n = len(rows)
    out = np.tile(IDENT, (n, 1))
    if policy is None:
        return out
    P = POLICIES[policy]
    salt = shash("sib", ds, policy)
    for k in range(n):
        c = int(copies[k])
        if c == 0:
            continue
        g = np.random.default_rng(shash(salt, int(rows[k]), c))
        a = g.uniform(-P["rot"], P["rot"])
        tx = g.uniform(-P["shift"], P["shift"]) if P["shift"] else 0.0
        ty = g.uniform(-P["shift"], P["shift"]) if P["shift"] else 0.0
        sh = g.uniform(-P["shear"], P["shear"]) if P["shear"] else 0.0
        z = g.uniform(1 - P["zoom"], 1 + P["zoom"]) if P["zoom"] else 1.0
        hf = float(g.random() < 0.5) if P["hflip"] else 0.0
        vf = float(g.random() < 0.5) if P["vflip"] else 0.0
        b = g.uniform(*P["bright"]) if P["bright"] else 1.0
        ct = g.uniform(*P["contrast"]) if P["contrast"] else 1.0
        st = g.uniform(*P["sat"]) if P["sat"] else 1.0
        out[k] = (a, tx, ty, sh, z, hf, vf, b, ct, st)
    return out


def _gray(x):
    return (0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3])


def apply_sibling(x, prm):
    """x: float [B,3,H,W] in [0,1]; prm: float tensor [B,10].  Geometric + photometric sibling transform."""
    import torch
    import torch.nn.functional as F
    B = x.shape[0]
    a = prm[:, 0] * (math.pi / 180.0)
    c, s = torch.cos(a), torch.sin(a)
    sh, z = prm[:, 3], prm[:, 4]
    fx = 1 - 2 * prm[:, 5]
    fy = 1 - 2 * prm[:, 6]
    th = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
    # output -> input:  R(a) . Shear . Zoom . Flip
    m00, m01 = c * z * fx, (c * sh - s) * z * fy
    m10, m11 = s * z * fx, (s * sh + c) * z * fy
    th[:, 0, 0], th[:, 0, 1], th[:, 1, 0], th[:, 1, 1] = m00, m01, m10, m11
    th[:, 0, 2], th[:, 1, 2] = 2 * prm[:, 1], 2 * prm[:, 2]
    grid = F.affine_grid(th, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    x = x * prm[:, 7].view(B, 1, 1, 1)
    m = _gray(x).mean((2, 3), keepdim=True)
    x = (x - m) * prm[:, 8].view(B, 1, 1, 1) + m
    g = _gray(x)
    x = g + (x - g) * prm[:, 9].view(B, 1, 1, 1)
    return x.clamp(0, 1)


def train_aug(x, out, prof, gen):
    """Random resized crop / flips / rotation / colour jitter on GPU, reproducible through `gen`."""
    import torch
    import torch.nn.functional as F
    P = TRAIN_PROFILES[prof]
    B, dev = x.shape[0], x.device

    def u(n=B):
        return torch.rand(n, generator=gen, device=dev)

    if P["rrc"]:
        lo, hi = P["rrc"]
        area = lo + (hi - lo) * u()
        r = torch.exp((u() * 2 - 1) * math.log(4 / 3))
        w = torch.sqrt(area * r).clamp(max=1.0)
        h = torch.sqrt(area / r).clamp(max=1.0)
        cx = (u() * 2 - 1) * (1 - w)
        cy = (u() * 2 - 1) * (1 - h)
    else:
        w = torch.ones(B, device=dev)
        h = torch.ones(B, device=dev)
        cx = torch.zeros(B, device=dev)
        cy = torch.zeros(B, device=dev)
        u(), u(), u(), u()
    fx = torch.where((u() < 0.5) & torch.tensor(P["hflip"], device=dev), -1.0, 1.0)
    fy = torch.where((u() < 0.5) & torch.tensor(P["vflip"], device=dev), -1.0, 1.0)
    ang = torch.zeros(B, device=dev)
    k90 = torch.floor(u() * 4)
    if P["rot90"]:
        ang = ang + k90 * (math.pi / 2)
    rr = (u() * 2 - 1) * (P["rot"] * math.pi / 180.0)
    ang = ang + rr
    c, s = torch.cos(ang), torch.sin(ang)
    th = torch.zeros(B, 2, 3, device=dev)
    th[:, 0, 0], th[:, 0, 1] = w * fx * c, -w * fx * s
    th[:, 1, 0], th[:, 1, 1] = h * fy * s, h * fy * c
    th[:, 0, 2], th[:, 1, 2] = cx, cy
    grid = F.affine_grid(th, [B, 3, out, out], align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    j = P["jitter"]
    bj, cj, sj = 1 + j * (u() * 2 - 1), 1 + j * (u() * 2 - 1), 1 + j * (u() * 2 - 1)
    if j > 0:
        x = x * bj.view(B, 1, 1, 1)
        m = _gray(x).mean((2, 3), keepdim=True)
        x = (x - m) * cj.view(B, 1, 1, 1) + m
        g = _gray(x)
        x = g + (x - g) * sj.view(B, 1, 1, 1)
    return x.clamp(0, 1)


def eval_prep(x, out):
    import torch.nn.functional as F
    if x.shape[-1] == out and x.shape[-2] == out:
        return x
    return F.interpolate(x, size=(out, out), mode="bilinear", antialias=True, align_corners=False)


class Feeder:
    """Keeps the needed store rows of one or more datasets in pinned CPU memory and assembles GPU batches."""

    def __init__(self, rows_by_ds, device, pin=True):
        import torch
        self.dev = device
        self.X, self.rows = {}, {}
        for ds, rows in rows_by_ds.items():
            rows = np.unique(np.asarray(rows, np.int64))
            st = open_store(ds)
            arr = np.ascontiguousarray(st[rows])
            t = torch.from_numpy(arr)
            if pin and device.type == "cuda":
                try:
                    t = t.pin_memory()
                except Exception:
                    pass
            self.X[ds], self.rows[ds] = t, rows

    def batch(self, ds, rows, copies, sib_prm, policy):
        import torch
        pos = np.searchsorted(self.rows[ds], rows)
        x = self.X[ds].index_select(0, torch.from_numpy(pos)).to(self.dev, non_blocking=True)
        x = x.permute(0, 3, 1, 2).float().div_(255.0)
        if policy is not None:
            m = np.nonzero(np.asarray(copies) > 0)[0]
            if len(m):
                mt = torch.from_numpy(m).to(self.dev)
                x[mt] = apply_sibling(x[mt], torch.from_numpy(sib_prm[m]).to(self.dev))
        return x


def _hf_vit_cls(repo, img):
    """Hugging Face ViT (e.g. Phikon) returning the CLS token, with timm-style normalisation constants."""
    import torch
    from transformers import ViTModel, ViTConfig

    class ViTCLS(torch.nn.Module):
        def __init__(self, vit):
            super().__init__()
            self.vit = vit
            self.pretrained_cfg = {"mean": (0.485, 0.456, 0.406), "std": (0.229, 0.224, 0.225)}

        def forward(self, x):
            return self.vit(pixel_values=x, interpolate_pos_encoding=True).last_hidden_state[:, 0]

    if OFFLINE or TINY:
        vit = ViTModel(ViTConfig(hidden_size=96, num_hidden_layers=2, num_attention_heads=3, intermediate_size=192,
                                 image_size=img, patch_size=8), add_pooling_layer=False)
    else:
        vit = ViTModel.from_pretrained(repo, add_pooling_layer=False)
    return ViTCLS(vit)


@__import__("functools").lru_cache(maxsize=4)
def _emb_model(key, img, device_str):
    import torch
    if key == "phikon":
        m = _hf_vit_cls(EMB_MODELS["phikon"], img).eval().to(torch.device(device_str))
        if device_str.startswith("cuda"):
            m.half()
        return m
    import timm
    name = tiny_name(EMB_MODELS.get(key, key))
    kw = {"img_size": img} if any(t in name for t in ("vit_", "dinov2")) else {}
    m = timm.create_model(name, pretrained=not OFFLINE, num_classes=0, **kw)
    m.eval().to(torch.device(device_str))
    if device_str.startswith("cuda"):
        m.half()
    return m


def embed(ds, rows, key="dinov2_s", img=224, bs=256, copies=None, policy=None, views="none", device=None):
    """L2-normalised embeddings of store rows (optionally sibling copies).  views='d4' returns [n,8,dim]."""
    import torch
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    img = 56 if TINY else img
    m = _emb_model(key, img, device)
    dev = torch.device(device)
    mean, std = norm_consts(m, dev)
    rows = np.asarray(rows, np.int64)
    copies = np.zeros(len(rows), int) if copies is None else np.asarray(copies)
    prm = sibling_params(ds, rows, copies, policy)
    st = open_store(ds)
    outs = []
    with torch.no_grad():
        for s in range(0, len(rows), bs):
            r = rows[s:s + bs]
            x = torch.from_numpy(np.ascontiguousarray(st[np.sort(r)])).to(dev)
            order = np.argsort(np.argsort(r))
            x = x[torch.from_numpy(order).to(dev)].permute(0, 3, 1, 2).float() / 255.0
            if policy is not None and (copies[s:s + bs] > 0).any():
                x = apply_sibling(x, torch.from_numpy(prm[s:s + bs]).to(dev))
            x = (eval_prep(x, img) - mean) / std
            vs = [x] if views == "none" else [torch.rot90(xx, k, (2, 3)) for xx in (x, x.flip(3)) for k in range(4)]
            fs = []
            for v in vs:
                v = v.half() if device.startswith("cuda") else v
                f = m(v).float()
                fs.append(torch.nn.functional.normalize(f, dim=1).cpu().numpy())
            outs.append(fs[0] if views == "none" else np.stack(fs, 1))
    return np.concatenate(outs, 0).astype(np.float32)


# =====================================================================================
# 5  near-duplicate graph, union-find, index-order segmentation, parent recovery
# =====================================================================================
class UF:
    def __init__(self, n):
        self.p = np.arange(n)

    def find(self, a):
        p = self.p
        r = a
        while p[r] != r:
            r = p[r]
        while p[a] != r:
            p[a], a = r, p[a]
        return r

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            lo, hi = (ra, rb) if ra < rb else (rb, ra)
            self.p[hi] = lo

    def roots(self):
        return np.array([self.find(i) for i in range(len(self.p))])


def union_labels(n, pairs=(), label_arrays=(), subset=None):
    """Connected components over records: explicit pairs plus 'same value in any label array'."""
    uf = UF(n)
    for a, b in pairs:
        uf.union(int(a), int(b))
    idx = np.arange(n) if subset is None else np.asarray(subset)
    for lab in label_arrays:
        first = {}
        for i in idx:
            v = int(lab[i])
            if v < 0:
                continue
            if v in first:
                uf.union(int(i), first[v])
            else:
                first[v] = int(i)
    return uf.roots()


def cos_pairs(ea, eb=None, thr=0.95, chunk=2048):
    same = eb is None
    eb = ea if same else eb
    I, J, S = [], [], []
    for s in range(0, len(ea), chunk):
        sim = ea[s:s + chunk] @ eb.T
        ii, jj = np.nonzero(sim >= thr)
        ii = ii + s
        if same:
            k = jj > ii
            ii, jj = ii[k], jj[k]
        I.append(ii)
        J.append(jj)
        S.append(sim[ii - s, jj])
    if not I:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0)
    return np.concatenate(I), np.concatenate(J), np.concatenate(S)


def otsu(v, bins=256):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 4 or v.max() - v.min() < 1e-9:
        return float(np.median(v)) if len(v) else 0.5
    hist, edges = np.histogram(v, bins=bins)
    cen = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    s0 = np.cumsum(hist * cen)
    mu0 = s0 / np.maximum(w0, 1)
    mu1 = (s0[-1] - s0) / np.maximum(w1, 1)
    var = w0 * w1 * (mu0 - mu1) ** 2
    k = int(np.argmax(var[:-1]))
    return float(edges[k + 1])


def segment_order(order, labels, emb):
    """Split an ordered list of records into contiguous segments at class changes or similarity drops."""
    order = np.asarray(order)
    e = emb[order]
    s = (e[:-1] * e[1:]).sum(1)
    same = labels[order][:-1] == labels[order][1:]
    tau = otsu(s[same])
    boundary = (~same) | (s < tau)
    seg_pos = np.concatenate([[0], np.cumsum(boundary)])
    seg = np.full(len(labels), -1)
    seg[order] = seg_pos
    sizes = np.bincount(seg_pos)
    return seg, dict(tau=tau, n_seg=int(len(sizes)), size_med=float(np.median(sizes)),
                     size_q1=float(np.percentile(sizes, 25)), size_q3=float(np.percentile(sizes, 75)),
                     size_max=int(sizes.max()), adj_cos_mean=float(s[same].mean()) if same.any() else None)


def coarse_blocks(order, seg, labels, min_size):
    order = np.asarray(order)
    blk = np.full(len(labels), -1)
    b, cur = 0, 0
    for k, r in enumerate(order):
        if k > 0:
            prev = order[k - 1]
            if labels[r] != labels[prev] or (seg[r] != seg[prev] and cur >= min_size):
                b += 1
                cur = 0
        blk[r] = b
        cur += 1
    return blk


def adjacency_lift(order, labels, emb, rng, n_rand=20000):
    order = np.asarray(order)
    e = emb[order]
    same = labels[order][:-1] == labels[order][1:]
    adj = (e[:-1] * e[1:]).sum(1)[same]
    a = rng.integers(0, len(order), n_rand)
    b = rng.integers(0, len(order), n_rand)
    k = (labels[order][a] == labels[order][b]) & (a != b)
    rnd = (e[a[k]] * e[b[k]]).sum(1)
    return float(adj.mean() - rnd.mean()), float(adj.mean()), float(rnd.mean())


def recover_parents(emb_child_views, emb_orig, y_child, y_orig, near_of_orig, cos_min, margin_min):
    """For each publisher-augmented child: best matching original (max over 8 dihedral views)."""
    nc = len(emb_child_views)
    V = emb_child_views.reshape(nc * 8, -1)
    S = (V @ emb_orig.T).reshape(nc, 8, -1).max(1)
    S[y_child[:, None] != y_orig[None, :]] = -2
    top = np.argmax(S, 1)
    s1 = S[np.arange(nc), top]
    S2 = S.copy()
    same_cluster = near_of_orig[None, :] == near_of_orig[top][:, None]
    S2[same_cluster] = -2
    s2 = S2.max(1)
    ok = (s1 >= cos_min) & ((s1 - s2) >= margin_min)
    return top, s1, s2, ok


# =====================================================================================
# 6  split builder
#    clean holdout (outer fold over the strictest grouping g_ctrl, originals only, one per exact duplicate)
#    development pool -> arm split into train / inner validation / own evaluation (70/10/20)
#    arm factors: S   = siblings created before the split and allowed to straddle it
#                 Dup = duplicate files kept and allowed to straddle the split
#                 Grp = structural groups (slide / patient / recovered source image) ignored by the split
#    all arms of one experiment are matched on per-class training-set size
# =====================================================================================
XEVAL = {"histopath": ["he_stained", "ndb_ufes"], "he_stained": ["histopath", "ndb_ufes"],
         "ndb_ufes": ["histopath", "he_stained"]}


class DS:
    _cache = {}

    @classmethod
    def get(cls, ds):
        p = D["index"] / f"{ds}.json"
        pr = D["index"] / f"{ds}.recov.json"
        key = (ds, p.stat().st_mtime if p.exists() else 0, pr.stat().st_mtime if pr.exists() else 0)
        if key not in cls._cache:
            cls._cache[key] = DS(ds)
        return cls._cache[key]

    def __init__(self, ds):
        J = jload(D["index"] / f"{ds}.json")
        if J is None:
            raise FileNotFoundError(f"index for {ds} missing - run Block 16")
        R = J["records"]
        self.ds, self.R, self.n = ds, R, len(R)
        self.meta = META[ds]
        self.K = self.meta["K"]
        self.y = np.array([r["label"] for r in R], int)
        self.orig = np.array([bool(r.get("orig", True)) for r in R])
        self.ok = np.array([bool(r.get("ok", True)) for r in R])
        self.parent = np.array([int(r.get("parent", -1)) for r in R])
        self.child_ok = np.array([bool(r.get("parent_ok", False)) for r in R])
        self.dup_rep = np.array([bool(r.get("dup_rep", True)) for r in R])
        self.md5 = np.array([r.get("md5", str(i)) for i, r in enumerate(R)])
        ar = np.arange(self.n)
        for g in ("g_exact", "g_near", "g_struct", "g_ctrl", "g_coarse"):
            setattr(self, g, np.array([int(r.get(g, i)) for i, r in enumerate(R)]))
        self.parent_link = np.where((self.parent >= 0) & self.child_ok, self.parent, ar)
        self.slide = np.array([str(r.get("slide", r.get("patient", ""))) for r in R])
        self.base = self.ok & self.orig & self.dup_rep
        self.summary = J.get("summary", {})
        X = jload(D["index"] / f"{ds}.recov.json") or {}
        self.extra = {k: np.asarray(v, np.int64) for k, v in (X.get("groups") or {}).items()}


_OUTER = {}


def outer_folds(dd, scheme):
    """(fold id per record, holdout grouping key per record).  fold -1 = never used.
    loso: one slide per fold (grouping key = slide); near duplicates of holdout images are purged from the
    development pool by build_job instead of merging slides.  sgkf: stratified group k-fold on g_ctrl."""
    key = (dd.ds, dd.n, json.dumps(scheme, sort_keys=True))
    if key in _OUTER:
        return _OUTER[key]
    base = np.nonzero(dd.base)[0]
    if scheme["kind"] == "loso":
        slides = sorted(set(dd.slide[base]))
        rank = {s_: i for i, s_ in enumerate(slides)}
        fold = np.array([rank.get(dd.slide[i], -1) if dd.ok[i] else -1 for i in range(dd.n)])
        gh = dd.g_struct.copy()
    else:
        from sklearn.model_selection import StratifiedGroupKFold
        ctrl = dd.g_ctrl
        f_ctrl = {}
        sg = StratifiedGroupKFold(n_splits=scheme["n"], shuffle=True,
                                  random_state=(1000 * int(scheme.get("rep", 0)) + 7))
        for f, (_, te) in enumerate(sg.split(np.zeros(len(base)), dd.y[base], ctrl[base])):
            for r in base[te]:
                f_ctrl[ctrl[r]] = f
        fold = np.array([f_ctrl.get(ctrl[i], -1) if dd.ok[i] else -1 for i in range(dd.n)])
        gh = dd.g_ctrl.copy()
    _OUTER[key] = (fold, gh)
    return fold, gh


def n_outer(dd, scheme):
    return len(set(dd.slide[dd.base])) if scheme["kind"] == "loso" else scheme["n"]


def sg_split(keys, y, frac, seed):
    """Boolean mask of the 'test' side of one stratified group split with target fraction `frac`."""
    keys = np.asarray(keys)
    y = np.asarray(y)
    n = len(keys)
    mask = np.zeros(n, bool)
    if n == 0:
        return mask
    ug = np.unique(keys)
    n_splits = max(2, int(round(1.0 / frac)))
    rng = np.random.default_rng(seed)
    if len(ug) == n:  # every item is its own group: plain stratified random split
        for c in np.unique(y):
            ic = rng.permutation(np.nonzero(y == c)[0])
            mask[ic[:int(round(frac * len(ic)))]] = True
        if 0 < mask.sum() < n:
            return mask
        mask[:] = False
    if len(ug) >= 2 * n_splits:
        from sklearn.model_selection import StratifiedGroupKFold
        sg = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=int(seed % (2 ** 31 - 1)))
        folds = list(sg.split(np.zeros(n), y, keys))
        te = folds[int(rng.integers(0, n_splits))][1]
        mask[te] = True
        if 0 < mask.sum() < n:
            return mask
        mask[:] = False
    # few groups: greedy random group selection closest to the target fraction (at least one group)
    order = rng.permutation(ug)
    size = {g: int((keys == g).sum()) for g in ug}
    target, cur, chosen = frac * n, 0, []
    for g in order:
        if not chosen or abs(cur + size[g] - target) < abs(cur - target):
            chosen.append(g)
            cur += size[g]
        if cur >= target:
            break
    if len(chosen) == len(ug) and len(ug) > 1:
        chosen = chosen[:-1]
    mask[np.isin(keys, chosen)] = True
    return mask


def _distribute(n_items, total, rng):
    """copies per item summing to max(total, n_items); every item keeps its original (>=1)."""
    if n_items == 0:
        return np.zeros(0, int)
    total = max(int(total), n_items)
    base, extra = divmod(total, n_items)
    k = np.full(n_items, base, int)
    k[rng.permutation(n_items)[:extra]] += 1
    return k


class Units(dict):
    """Evaluation / training units: row (store row), copy (sibling index), y, key (bootstrap cluster = g_ctrl)."""

    @staticmethod
    def make(rows, copies, y, key, ds):
        u = Units(row=np.asarray(rows, np.int64), copy=np.asarray(copies, np.int64),
                  y=np.asarray(y, np.int64), key=np.asarray(key, np.int64))
        u.ds = ds
        return u

    def take(self, m):
        u = Units(**{k: v[m] for k, v in self.items()})
        u.ds = self.ds
        return u

    def cat(self, other):
        u = Units(**{k: np.concatenate([self[k], other[k]]) for k in self})
        u.ds = self.ds
        return u

    def __len__(self):
        return len(self["row"])


def _exp_cfg(spec):
    P = plan()
    ex = P["exps"][spec["exp"]]
    return P, ex, ex["datasets"][spec["ds"]]


def _labels(dd, spec):
    if not str(spec["exp"]).endswith("RL"):
        return dd.y
    rng = np.random.default_rng(shash("RL", dd.ds, spec["seed"]))
    roots = np.unique(dd.g_ctrl[dd.ok])
    perm = rng.permutation(len(roots))
    lab = {int(g): int(perm[i] % dd.K) for i, g in enumerate(roots)}
    return np.array([lab.get(int(dd.g_ctrl[i]), 0) for i in range(dd.n)])


def _build_arm(dd, y, fold, o, fac, dcfg, spec, name, purge):
    S, Dup, Grp = int(fac.get("S", 0)), int(fac.get("Dup", 0)), int(fac.get("Grp", 0))
    coarse = int(fac.get("coarse", 0))
    seed = int(spec["seed"])
    # the split never depends on the dose, so dose levels share one base split (nested designs)
    rng = np.random.default_rng(shash("arm", dd.ds, spec["exp"], o, spec.get("rep", 0), seed, name))
    use_children = bool(dcfg.get("children", False))
    dev = dd.ok & (fold >= 0) & (fold != o) & ~purge
    dev &= (dd.orig | (dd.child_ok & use_children))
    if not Dup:
        dev &= dd.dup_rep
    idx = np.nonzero(dev)[0]
    arrays = []
    if not Dup:
        arrays.append(dd.g_near)
    if not Grp:
        rv = fac.get("recov")
        if rv:
            if rv not in dd.extra:
                raise KeyError(f"recovered grouping '{rv}' missing for {dd.ds} - run the group-recovery block first")
            arrays.append(dd.extra[rv])
        else:
            arrays.append(dd.g_coarse if coarse else dd.g_struct)
    if not S:
        arrays.append(dd.parent_link)
    rk = union_labels(dd.n, label_arrays=arrays, subset=idx)
    policy = dcfg.get("policy")
    K = dd.K
    # per-record copy counts (siblings) for the development pool
    phi = (dd.base & (fold >= 0) & (fold != o)).sum() / max(1, (dd.base & (fold >= 0)).sum())
    if policy is not None:
        if spec["exp"] == "DOSEK":
            kmap = {int(r): int(spec["dose"]) for r in idx}
        else:
            kmap = {}
            for c in range(K):
                rc = idx[y[idx] == c]
                U = int(round(dcfg["target"] * phi))
                for r, k in zip(rc, _distribute(len(rc), U, rng)):
                    kmap[int(r)] = int(k)
    s_own = shash("own", dd.ds, spec["exp"], o, spec.get("rep", 0), seed, name)
    s_iv = shash("ival", dd.ds, spec["exp"], o, spec.get("rep", 0), seed, name)
    if policy is not None and S:
        rows = np.concatenate([np.full(kmap[int(r)], r) for r in idx]) if len(idx) else np.zeros(0, int)
        cps = np.concatenate([np.arange(kmap[int(r)]) for r in idx]) if len(idx) else np.zeros(0, int)
        keys = rk[rows] * 1000 + cps
        U = Units.make(rows, cps, y[rows], dd.g_ctrl[rows], dd.ds)
        own_m = sg_split(keys, U["y"], 0.20, s_own)
        rest = ~own_m
        iv_m = np.zeros(len(U), bool)
        iv_m[np.nonzero(rest)[0][sg_split(keys[rest], U["y"][rest], 0.125, s_iv)]] = True
        parts = dict(own=U.take(own_m), ival=U.take(iv_m), train=U.take(rest & ~iv_m))
    else:
        U = Units.make(idx, np.zeros(len(idx), int), y[idx], dd.g_ctrl[idx], dd.ds)
        keys = rk[idx]
        own_m = sg_split(keys, U["y"], 0.20, s_own)
        rest = ~own_m
        iv_m = np.zeros(len(U), bool)
        iv_m[np.nonzero(rest)[0][sg_split(keys[rest], U["y"][rest], 0.125, s_iv)]] = True
        parts = dict(own=U.take(own_m), ival=U.take(iv_m), train=U.take(rest & ~iv_m))
        if policy is not None:  # augment after the split, same average factor per class
            for pn, P_ in list(parts.items()):
                R_, C_, Y_, G_ = [], [], [], []
                for c in range(K):
                    rc = P_["row"][P_["y"] == c]
                    tot = int(sum(kmap[int(r)] for r in rc))
                    for r, k in zip(rc, _distribute(len(rc), tot, rng)):
                        R_.append(np.full(k, r))
                        C_.append(np.arange(k))
                rows = np.concatenate(R_) if R_ else np.zeros(0, int)
                cps = np.concatenate(C_) if C_ else np.zeros(0, int)
                parts[pn] = Units.make(rows, cps, y[rows], dd.g_ctrl[rows], dd.ds)
    parts["rec_key"] = rk
    parts["factors"] = dict(S=S, Dup=Dup, Grp=Grp, coarse=coarse)
    return parts


def _subsample_to(U, Tc, rng):
    keep = []
    for c, t in Tc.items():
        ic = np.nonzero(U["y"] == c)[0]
        keep.append(ic if len(ic) <= t else np.sort(rng.choice(ic, t, replace=False)))
    return U.take(np.sort(np.concatenate(keep)) if keep else np.zeros(0, int))


def leak_stats(dd, train, own):
    if len(own) == 0 or len(train) == 0:
        return {}
    tr = train["row"]
    ow = own["row"]
    s_rec = set(tr.tolist())
    f = lambda a: float(np.mean([v in s_rec for v in a])) if len(a) else 0.0
    out = dict(same_original=f(ow))
    for name, arr in (("near_dup", dd.g_near), ("struct_group", dd.g_struct), ("ctrl_group", dd.g_ctrl)):
        s = set(arr[tr].tolist())
        out[name] = float(np.mean([v in s for v in arr[ow]]))
    s = set(dd.md5[tr].tolist())
    out["identical_file"] = float(np.mean([v in s for v in dd.md5[ow]]))
    out["own_units_from_originals"] = float(np.mean(own["copy"] == 0))
    return out


def check_disjoint(dd, B, controlled, injected=False):
    """Raises AssertionError when the clean holdout or a controlled arm is contaminated."""
    hold = B["hold"]
    gh = B["ghold"]
    dev_rows = np.concatenate([B[k]["row"] for k in ("train", "ival", "own")])
    hr = set(hold["row"].tolist())
    assert not (hr & set(dev_rows.tolist())), "holdout image also in train/ival/own"
    assert not (set(gh[hold["row"]].tolist()) & set(gh[dev_rows].tolist())), "holdout group (slide/patient/source) in development data"
    assert not (set(dd.g_near[hold["row"]].tolist()) & set(dd.g_near[dev_rows].tolist())), "near duplicate of a holdout image in development data"
    hm = set(dd.md5[hold["row"]].tolist())
    assert not (hm & set(dd.md5[dev_rows].tolist())), "holdout file hash also in development data"
    assert not (set(dd.parent_link[hold["row"]].tolist()) & set(dd.parent_link[dev_rows].tolist())), "augmented copy of a holdout image in development data"
    if controlled:
        rk = B["rec_key"]
        tr = B["train"]["row"]
        for part in ("own", "ival"):
            pr = B[part]["row"]
            if not injected or part == "ival":
                assert not (set(rk[tr].tolist()) & set(rk[pr].tolist())), f"controlled arm: train/{part} share a group"
                assert not (set(dd.md5[tr].tolist()) & set(dd.md5[pr].tolist())), f"controlled arm: train/{part} share a file hash"
                assert not (set(dd.parent_link[tr].tolist()) & set(dd.parent_link[pr].tolist())), f"controlled arm: train/{part} share a source image"
    return True


def xeval_units(src):
    out = {}
    for t in XEVAL.get(src, []):
        if not (D["index"] / f"{t}.json").exists():
            continue
        dd = DS.get(t)
        r = np.nonzero(dd.base)[0]
        y = dd.y[r]
        if t == "ndb_ufes":
            keep = y != 1
            r, y = r[keep], (y[keep] == 2).astype(int)
        out[t] = Units.make(r, np.zeros(len(r), int), y, dd.g_ctrl[r], t)
    return out


def build_job(spec, check=True):
    """Deterministic data for one job.  Returns dict(train, ival, own, hold, xeval, info)."""
    P, ex, dcfg = _exp_cfg(spec)
    dd = DS.get(spec["ds"])
    y = _labels(dd, spec)
    scheme = P["outer"][spec["ds"]]
    scheme = dict(scheme, rep=int(spec.get("rep", 0)))
    fold, ghold = outer_folds(dd, scheme)
    o = int(spec["outer"])
    hr = np.nonzero(dd.base & (fold == o))[0]
    if len(hr) == 0:
        raise ValueError(f"empty holdout for {dd.ds} outer fold {o}")
    hold = Units.make(hr, np.zeros(len(hr), int), y[hr], dd.g_ctrl[hr], dd.ds)
    # purge: development records that are identical / near-duplicate / same source as any holdout image
    hn = set(dd.g_near[fold == o].tolist()) | set(dd.md5[fold == o].tolist()) | set(dd.parent_link[fold == o].tolist())
    purge = np.array([(dd.g_near[i] in hn or dd.md5[i] in hn or dd.parent_link[i] in hn) and fold[i] != o
                      for i in range(dd.n)])
    # arms split by a recovered grouping that could not be formed (k > n/3) are left out of the size matching
    arms = {nm: fac for nm, fac in dcfg["arms"].items()
            if not fac.get("recov") or fac["recov"] in dd.extra or nm == spec["arm"]}
    built = {nm: _build_arm(dd, y, fold, o, fac, dcfg, spec, nm, purge) for nm, fac in arms.items()}
    Tc = {c: min(int((b["train"]["y"] == c).sum()) for b in built.values()) for c in range(dd.K)}
    rng = np.random.default_rng(shash("match", dd.ds, spec["exp"], o, spec.get("rep", 0), spec["seed"], spec.get("dose")))
    B = built[spec["arm"]]
    B["train"] = _subsample_to(B["train"], Tc, rng)
    injected = False
    info = dict(n_dev=None, Tc=Tc)
    if spec["exp"] == "DUP" and float(spec["dose"]) > 0:
        own0 = np.nonzero(B["own"]["copy"] == 0)[0]
        n_inj = int(round(float(spec["dose"]) * len(own0)))
        rg = np.random.default_rng(shash("dup", dd.ds, o, spec["seed"]))   # nested across doses
        pick = np.sort(rg.permutation(own0)[:n_inj])
        add = B["own"].take(pick)
        drop = []
        for c in range(dd.K):
            nc_ = int((add["y"] == c).sum())
            ic = np.nonzero(B["train"]["y"] == c)[0]
            drop.extend(np.random.default_rng(shash("dupdrop", dd.ds, o, spec["seed"], c)).permutation(ic)[:nc_].tolist())
        keep = np.setdiff1d(np.arange(len(B["train"])), np.array(drop, int))
        B["train"] = B["train"].take(keep).cat(add)
        injected = True
        info["injected"] = int(n_inj)
        B["own_injected_mask"] = np.isin(np.arange(len(B["own"])), pick)
    if spec["exp"] == "OVL":
        rk = B["rec_key"]
        own = B["own"]
        rg = np.random.default_rng(shash("ovl", dd.ds, o, spec["seed"]))
        g_own = rk[own["row"]]
        ev_m = np.zeros(len(own), bool)
        res_by_g = {}
        for g in np.unique(g_own):
            ii = rg.permutation(np.nonzero(g_own == g)[0])
            h = len(ii) // 2
            ev_m[ii[:max(1, len(ii) - h)]] = True
            res_by_g[g] = ii[max(1, len(ii) - h):]
        q = float(spec["dose"])
        add_idx = []
        for g, ii in res_by_g.items():
            add_idx.extend(ii[:int(math.ceil(q * len(ii)))].tolist())
        add = own.take(np.array(sorted(add_idx), int))
        drop = []
        for c in range(dd.K):
            nc_ = int((add["y"] == c).sum())
            ic = np.nonzero(B["train"]["y"] == c)[0]
            drop.extend(np.random.default_rng(shash("ovldrop", dd.ds, o, spec["seed"], c)).permutation(ic)[:nc_].tolist())
        keep = np.setdiff1d(np.arange(len(B["train"])), np.array(drop, int))
        B["train"] = B["train"].take(keep).cat(add)
        B["own"] = own.take(ev_m)
        injected = q > 0
        info["overlap_added"] = int(len(add))
    B["hold"] = hold
    B["ghold"] = ghold
    B["xeval"] = xeval_units(spec["ds"]) if spec.get("xeval") else {}
    fac = B["factors"]
    controlled = (fac["S"] == 0 and fac["Dup"] == 0 and fac["Grp"] == 0)
    if check:
        check_disjoint(dd, B, controlled, injected)
    info.update(sizes={k: int(len(B[k])) for k in ("train", "ival", "own", "hold")}, purged=int(purge.sum()),
                train_per_class=[int((B["train"]["y"] == c).sum()) for c in range(dd.K)],
                hold_per_class=[int((hold["y"] == c).sum()) for c in range(dd.K)],
                leak=leak_stats(dd, B["train"], B["own"]), controlled=controlled,
                policy=dcfg.get("policy"), K=dd.K)
    B["info"] = info
    return B


# =====================================================================================
# 7  metrics, calibration, operating points, cluster statistics
# =====================================================================================
def softmax(z, T=1.0):
    z = np.asarray(z, np.float64) / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits, y):
    """Temperature minimising NLL on the given (inner-validation) predictions."""
    logits, y = np.asarray(logits, np.float64), np.asarray(y, int)
    if len(y) < 5 or len(np.unique(y)) < 2:
        return 1.0

    def nll(T):
        p = softmax(logits, T)
        return -np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))

    grid = np.exp(np.linspace(np.log(0.05), np.log(20), 121))
    v = [nll(t) for t in grid]
    k = int(np.argmin(v))
    lo, hi = grid[max(0, k - 1)], grid[min(len(grid) - 1, k + 1)]
    for _ in range(40):
        m1, m2 = lo + (hi - lo) / 3, hi - (hi - lo) / 3
        if nll(m1) < nll(m2):
            hi = m2
        else:
            lo = m1
    return float((lo + hi) / 2)


def ece_score(prob, y, bins=15):
    conf = prob.max(1)
    pred = prob.argmax(1)
    acc = (pred == y).astype(float)
    e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for a, b in zip(edges[:-1], edges[1:]):
        m = (conf > a) & (conf <= b)
        if m.any():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)


def metrics(prob, y, K, pos=None, pred=None):
    """Full metric set.  Macro metrics AND explicit positive-class (one-vs-rest) metrics are kept apart."""
    from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef, cohen_kappa_score
    prob = np.asarray(prob, np.float64)
    y = np.asarray(y, int)
    n = len(y)
    if n == 0:
        return {}
    pred = prob.argmax(1) if pred is None else np.asarray(pred, int)
    cm = np.zeros((K, K), int)
    np.add.at(cm, (y, pred), 1)
    tp = np.diag(cm).astype(float)
    rows, cols = cm.sum(1).astype(float), cm.sum(0).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        rec = np.where(rows > 0, tp / rows, np.nan)
        prec = np.where(cols > 0, tp / cols, np.nan)
        f1 = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0.0)
        tn = n - rows - cols + tp
        spec = np.where((n - rows) > 0, tn / (n - rows), np.nan)
        npv = np.where((n - cols) > 0, tn / (n - cols), np.nan)
    present = rows > 0
    out = dict(n=n, acc=float(tp.sum() / n), bal_acc=float(np.nanmean(rec[present])),
               macro_f1=float(np.mean(f1[present])), macro_prec=float(np.nanmean(prec[present])),
               macro_rec=float(np.nanmean(rec[present])), macro_spec=float(np.nanmean(spec[present])),
               per_class=dict(recall=rec.tolist(), precision=prec.tolist(), f1=f1.tolist(),
                              specificity=spec.tolist(), npv=npv.tolist(), n=rows.astype(int).tolist()),
               cm=cm.tolist())
    try:
        if K == 2 and present.all():
            out["auroc"] = float(roc_auc_score(y, prob[:, 1]))
            out["auprc"] = float(average_precision_score(y, prob[:, 1]))
        elif present.all():
            out["auroc"] = float(roc_auc_score(y, prob, multi_class="ovr", average="macro"))
            out["auprc"] = float(np.mean([average_precision_score(y == k, prob[:, k]) for k in range(K)]))
    except Exception:
        pass
    try:
        out["mcc"] = float(matthews_corrcoef(y, pred))
        out["kappa"] = float(cohen_kappa_score(y, pred))
    except Exception:
        pass
    out["nll"] = float(-np.mean(np.log(np.clip(prob[np.arange(n), y], 1e-12, 1))))
    oh = np.eye(K)[y]
    out["brier"] = float(np.mean(((prob - oh) ** 2).sum(1)))
    out["ece"] = ece_score(prob, y)
    if pos is not None:
        out["pos"] = dict(cls=int(pos), sens=float(rec[pos]), spec=float(spec[pos]), prec=float(prec[pos]),
                          f1=float(f1[pos]), npv=float(npv[pos]))
    return out


def thresholds_from(p_pos, y_bin):
    """Operating points chosen on inner-validation data: Youden, 'sensitivity >= 95%' and 'sensitivity >= 99%'
    (the highest threshold that still reaches the target sensitivity on inner validation)."""
    p_pos, y_bin = np.asarray(p_pos, float), np.asarray(y_bin, int)
    out = dict(default=0.5)
    if len(np.unique(y_bin)) < 2:
        out.update(youden=0.5, sens95=0.5, sens99=0.5)
        return out
    cand = np.unique(np.concatenate([p_pos, [0.0, 1.0 + 1e-9]]))
    P, N = (y_bin == 1).sum(), (y_bin == 0).sum()
    best, bt = -1, 0.5
    s95 = s99 = None
    for t in cand:
        pr = p_pos >= t
        se = (pr & (y_bin == 1)).sum() / P
        sp = (~pr & (y_bin == 0)).sum() / N
        if se + sp - 1 > best:
            best, bt = se + sp - 1, float(t)
        if se >= 0.95:
            s95 = float(t)
        if se >= 0.99:
            s99 = float(t)
    out.update(youden=bt, sens95=s95 if s95 is not None else float(cand.min()),
               sens99=s99 if s99 is not None else float(cand.min()))
    return out


def op_point(p_pos, y_bin, thr):
    """Every column computed from the same threshold."""
    p_pos, y_bin = np.asarray(p_pos, float), np.asarray(y_bin, int)
    pr = p_pos >= thr
    TP = int((pr & (y_bin == 1)).sum()); FN = int((~pr & (y_bin == 1)).sum())
    TN = int((~pr & (y_bin == 0)).sum()); FP = int((pr & (y_bin == 0)).sum())
    se = TP / max(1, TP + FN)
    sp = TN / max(1, TN + FP)
    return dict(thr=float(thr), sens=se, spec=sp, ppv=TP / max(1, TP + FP), npv=TN / max(1, TN + FN),
                lr_pos=(se / (1 - sp)) if sp < 1 else float("inf"), lr_neg=((1 - se) / sp) if sp > 0 else float("inf"),
                fnr=1 - se, TP=TP, FN=FN, TN=TN, FP=FP)


# ---------- cluster bootstrap (balanced accuracy via per-cluster count tables) ----------
def cluster_table(y, correct, cluster, K):
    cl, inv = np.unique(np.asarray(cluster), return_inverse=True)
    n = np.zeros((len(cl), K))
    c = np.zeros((len(cl), K))
    np.add.at(n, (inv, np.asarray(y, int)), 1)
    np.add.at(c, (inv, np.asarray(y, int)), np.asarray(correct, float))
    return n, c


def ba_counts(n, c):
    tot = n.sum(-2)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(tot > 0, c.sum(-2) / tot, np.nan)
    return np.nanmean(r, -1)


def boot_weights(C, B, rng, chunk=1000):
    for s in range(0, B, chunk):
        b = min(chunk, B - s)
        yield rng.multinomial(C, np.full(C, 1.0 / C), size=b).astype(np.float32)


def boot_ba(n, c, B, rng):
    out = []
    for Wt in boot_weights(len(n), B, rng):
        nn_ = np.einsum("bc,ck->bk", Wt, n)
        cc = np.einsum("bc,ck->bk", Wt, c)
        with np.errstate(divide="ignore", invalid="ignore"):
            r = np.where(nn_ > 0, cc / nn_, np.nan)
        out.append(np.nanmean(r, 1))
    return np.concatenate(out)


def boot_ba_paired(n, cA, cB, B, rng):
    """Paired difference BA_A - BA_B on the same images, resampling shared clusters."""
    out = []
    for Wt in boot_weights(len(n), B, rng):
        nn_ = np.einsum("bc,ck->bk", Wt, n)
        with np.errstate(divide="ignore", invalid="ignore"):
            ra = np.where(nn_ > 0, np.einsum("bc,ck->bk", Wt, cA) / nn_, np.nan)
            rb = np.where(nn_ > 0, np.einsum("bc,ck->bk", Wt, cB) / nn_, np.nan)
        out.append(np.nanmean(ra, 1) - np.nanmean(rb, 1))
    return np.concatenate(out)


def hier(mats, B, seed):
    """Two-stage bootstrap: resample outer folds with replacement, then take each chosen fold's
    within-fold cluster replicate.  mats: list of [F, B] arrays in the same fold order."""
    g = np.random.default_rng(seed)
    F = mats[0].shape[0]
    idx = g.integers(0, F, size=(B, F))
    cols = np.arange(B)[:, None]
    return [m[idx, cols].mean(1) for m in mats]


def ci(v, a=0.05):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return (np.nan, np.nan)
    return (float(np.percentile(v, 100 * a / 2)), float(np.percentile(v, 100 * (1 - a / 2))))


def boot_p(v, null=0.0):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return np.nan
    lo = (np.sum(v <= null) + 1) / (len(v) + 1)
    hi = (np.sum(v >= null) + 1) / (len(v) + 1)
    return float(min(1.0, 2 * min(lo, hi)))


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i]))
        adj[i] = run
    return adj.tolist()


def icc1(correct, cluster):
    """One-way random-effects ICC(1) of a binary outcome, and the design effect."""
    y = np.asarray(correct, float)
    cl, inv = np.unique(np.asarray(cluster), return_inverse=True)
    C, N = len(cl), len(y)
    if C < 2 or C == N:
        return dict(icc=0.0, deff=1.0, clusters=C, mean_size=N / max(1, C))
    ni = np.bincount(inv).astype(float)
    mi = np.bincount(inv, weights=y) / ni
    gm = y.mean()
    msb = np.sum(ni * (mi - gm) ** 2) / (C - 1)
    msw = np.sum((y - mi[inv]) ** 2) / (N - C)
    m0 = (N - np.sum(ni ** 2) / N) / (C - 1)
    icc = (msb - msw) / (msb + (m0 - 1) * msw) if (msb + (m0 - 1) * msw) > 0 else 0.0
    icc = float(max(0.0, icc))
    return dict(icc=icc, deff=float(1 + (ni.mean() - 1) * icc), clusters=int(C), mean_size=float(ni.mean()))


class quiet:
    """Silence warnings inside a block (statsmodels re-enables its own warnings at import)."""

    def __enter__(self):
        try:  # import first: statsmodels installs 'always' filters on first import
            import statsmodels.api  # noqa
            import statsmodels.formula.api  # noqa
        except Exception:
            pass
        self._c = warnings.catch_warnings()
        self._c.__enter__()
        warnings.simplefilter("ignore")
        return self

    def __exit__(self, *a):
        return self._c.__exit__(*a)


def gee_paired(correct_a, correct_b, cluster):
    """GEE (logit, exchangeable) for P(correct) of arm A vs arm B on the same images, clustered."""
    with quiet():
        return _gee_paired(correct_a, correct_b, cluster)


def _gee_paired(correct_a, correct_b, cluster):
    try:
        import statsmodels.api as sm
        ya = np.asarray(correct_a, float)
        yb = np.asarray(correct_b, float)
        yy = np.concatenate([ya, yb])
        arm = np.concatenate([np.ones(len(ya)), np.zeros(len(yb))])
        g = np.concatenate([cluster, cluster])
        X = sm.add_constant(arm)
        res = sm.GEE(yy, X, groups=g, family=sm.families.Binomial(),
                     cov_struct=sm.cov_struct.Exchangeable()).fit()
        b, se = float(res.params[1]), float(res.bse[1])
        return dict(log_or=b, or_=math.exp(b), lo=math.exp(b - 1.96 * se), hi=math.exp(b + 1.96 * se),
                    p=float(res.pvalues[1]))
    except Exception as e:
        return dict(error=str(e)[:120])


def mixed_intercept(values, outer, seed):
    """Random-intercept model  value ~ 1 + (1|outer fold) + seed variance component."""
    with quiet():
        return _mixed_intercept(values, outer, seed)


def _t_interval(v, note):
    from scipy import stats as st
    v = np.asarray(v, float)
    m = float(v.mean())
    if len(v) < 2:
        return dict(est=m, lo=None, hi=None, p=None, note=note)
    se = float(v.std(ddof=1) / np.sqrt(len(v)))
    q = float(st.t.ppf(0.975, len(v) - 1))
    p = float(2 * st.t.sf(abs(m) / se, len(v) - 1)) if se > 0 else None
    return dict(est=m, lo=m - q * se, hi=m + q * se, p=p, note=note)


def _mixed_intercept(values, outer, seed):
    import pandas as pd
    df = pd.DataFrame(dict(v=np.asarray(values, float), outer=outer, seed=seed))
    if df["outer"].nunique() < 2:
        return _t_interval(df["v"], "one outer fold")
    if df.groupby("outer").size().max() < 2:
        return _t_interval(df["v"], "one run per fold: t-interval over folds")
    try:
        import statsmodels.formula.api as smf
        vc = {"seed": "0 + C(seed)"} if df["seed"].nunique() > 1 else None
        res = smf.mixedlm("v ~ 1", df, groups=df["outer"], vc_formula=vc).fit(reml=True)
        b, se = float(res.params["Intercept"]), float(res.bse["Intercept"])
        if not np.isfinite(se) or se < 1e-4 or not res.converged:     # a zero SE means a singular (boundary) fit
            raise ValueError("singular fit" if np.isfinite(se) and res.converged else "did not converge")
        return dict(est=b, lo=b - 1.96 * se, hi=b + 1.96 * se, p=float(res.pvalues["Intercept"]), note=None)
    except Exception as e:
        agg = df.groupby("outer")["v"].mean()
        return _t_interval(agg.values, "mixed model failed (" + str(e)[:40] + "): t-interval over fold means")


def sign_flip_p(d):
    """Exact two-sided sign-flip permutation test on a small vector of paired differences."""
    d = np.asarray(d, float)
    n = len(d)
    if n == 0 or n > 16:
        return np.nan
    obs = abs(d.mean())
    cnt = 0
    for m in range(2 ** n):
        s = np.array([1 if (m >> i) & 1 else -1 for i in range(n)])
        cnt += abs((s * d).mean()) >= obs - 1e-12
    return cnt / 2 ** n


# ---------- run results ----------
def load_run(rid):
    r = jload(D["runs"] / rid / "result.json")
    if r is None or r.get("status") != "done":
        return None, None
    z = np.load(D["runs"] / rid / "preds.npz", allow_pickle=False)
    return r, z


def runs_index():
    out = []
    for p in sorted(D["runs"].glob("*/result.json")):
        r = jload(p)
        if r and r.get("status") == "done":
            out.append(r)
    return out


def set_preds(z, name, T=1.0):
    if f"{name}_logit" not in z.files:
        return None
    lg = z[f"{name}_logit"].astype(np.float64)
    return dict(prob=softmax(lg, T), logit=lg, y=z[f"{name}_y"], key=z[f"{name}_key"],
                row=z[f"{name}_row"], copy=z[f"{name}_copy"])


# =====================================================================================
# 8  job planner (used by Block 18 and the launcher)
# =====================================================================================
BB_SHORT = {"convnext_tiny.fb_in22k_ft_in1k": "cnxT", "resnet50.a1_in1k": "r50",
            "vit_small_patch16_224.augreg_in21k_ft_in1k": "vitS", "efficientnet_b0.ra_in1k": "eb0",
            "convnext_base.fb_in22k_ft_in1k": "cnxB", "efficientnet_b3.ra2_in1k": "eb3"}
PHASE_DESC = {"core": "E1 clean-holdout experiment, primary backbone, seed 0",
              "controls": "random-label, duplicate-injection, group-overlap, pre-augmented corpus",
              "dose": "augmentation-factor dose-response",
              "seeds": "E1 seeds 1 and 2 (seed as random effect)",
              "general": "extra backbones, factorial extras, published-pipeline reimplementations",
              "orchid": "ORCHID (oral H&E with patient IDs): leaky vs patient-grouped vs recovered-group arms",
              "recover": "arms split by metadata-free recovered groups (BreakHis, OCDC, NDB-UFES, histopathology)"}


def job_id(s):
    t = f"{s['exp']}.{SHORT[s['ds']]}.{s['arm']}.o{s['outer']}r{s.get('rep', 0)}.s{s['seed']}.{BB_SHORT.get(s['backbone'], 'bb')}"
    if s.get("recipe", "std") != "std":
        t += f".{s['recipe']}"
    if s.get("dose") is not None:
        t += f".d{s['dose']}"
    return t


def job_fp(s, P):
    ex = P["exps"][s["exp"]]
    dc = ex["datasets"][s["ds"]]
    return fingerprint(dict(spec=s, arms=dc["arms"], policy=dc.get("policy"), target=dc.get("target"),
                            children=dc.get("children"), recipe=P["recipes"][s.get("recipe", "std")],
                            outer=P["outer"][s["ds"]], near=P["near_dup"], lib=LIB_VERSION))


def recipe_steps(spec, R, n_train):
    if R.get("epochs"):
        spe = max(1, int(math.ceil(n_train / R["bs"])))
        if TINY:                                   # tests only: a few steps, same code path
            st = max(12, int(R["epochs"] * spe) // 600)
            return st, max(2, st // 6)
        return int(R["epochs"] * spe), spe
    st = int(spec["steps"])
    if TINY:
        st = max(6, st // 60)
    return st, max(1, int(round(st * R.get("eval_frac", 0.1))))


def estimate_minutes(spec, info, R, speed=None):
    img = R["img"]
    ips = (speed or {}).get(spec["backbone"]) or BACKBONES.get(spec["backbone"], ("", 0, 300))[2]
    ips = ips * (224.0 / img) ** 2
    n_tr = info["sizes"]["train"]
    steps, every = recipe_steps(spec, R, n_tr)
    n_evals = max(1, steps // every)
    ev = info["sizes"]["ival"] * n_evals + (info["sizes"]["own"] + info["sizes"]["hold"] + info.get("n_xeval", 0)) * (2 if R.get("tta") else 1)
    return (steps * R["bs"] / ips + ev / (3 * ips) + 30) / 60.0


def available_datasets():
    return [d for d in DATASETS if (D["index"] / f"{d}.json").exists()]


def enumerate_specs(P):
    avail = set(available_datasets())
    audit = jload(D["index"] / "audit.json", {}) or {}
    pre_ok = bool(audit.get("histopath", {}).get("pre_enabled", False))
    bb0 = P["primary_backbone"]
    out = []

    def add(phase, **s):
        s.setdefault("rep", 0)
        s.setdefault("backbone", bb0)
        s.setdefault("recipe", "std")
        s.setdefault("dose", None)
        s.setdefault("xeval", False)
        s.setdefault("keep_weights", False)
        s.setdefault("train_profile", None)
        out.append((phase, s))

    E1 = P["exps"]["E1"]
    for ds, dc in E1["datasets"].items():
        if ds not in avail:
            continue
        dd = DS.get(ds)
        sch = P["outer"][ds]
        for rep in P["reps"].get(ds, P["reps"]["default"]):
            for o in range(n_outer(dd, dict(sch, rep=rep))):
                for seed in E1["seeds"]:
                    for arm in dc["arms"]:
                        extra = arm in dc.get("extra", [])
                        if extra and seed != 0:
                            continue
                        phase = "general" if extra else ("core" if seed == 0 else "seeds")
                        add(phase, exp="E1", ds=ds, arm=arm, outer=o, rep=rep, seed=seed, steps=dc["steps"],
                            xeval=(seed == 0 and not extra and ds in XEVAL),
                            keep_weights=(seed == 0 and o == 0 and rep == 0 and not extra))
    for ex_name, phase in (("PRE", "controls"), ("RL", "controls"), ("DUP", "controls"), ("OVL", "controls"),
                           ("DOSEK", "dose")):
        ex = P["exps"][ex_name]
        for ds, dc in ex["datasets"].items():
            if ds not in avail or (ex_name == "PRE" and not pre_ok):
                continue
            dd = DS.get(ds)
            outs = list(range(n_outer(dd, P["outer"][ds]))) if dc.get("outer") == "all" else ex.get("outer", [0, 1, 2])
            if ex_name == "PRE":
                outs = list(range(n_outer(dd, P["outer"][ds])))
            for o in outs:
                for seed in ex.get("seeds", [0]):
                    for dose in (ex.get("doses") or [None]):
                        for arm in dc["arms"]:
                            add(phase, exp=ex_name, ds=ds, arm=arm, outer=o, seed=seed, steps=dc["steps"], dose=dose)
    G = P["generality"]
    for bb in G["backbones"]:
        for ds in G["datasets"]:
            if ds not in avail:
                continue
            for o in G["outer"]:
                for arm in G["arms"]:
                    add("general", exp="E1", ds=ds, arm=arm, outer=o, seed=G["seed"], backbone=bb,
                        steps=E1["datasets"][ds]["steps"])
    for r in P["reimp"]:
        if r["ds"] not in avail or (r["exp"] == "PRE" and not pre_ok):
            continue
        for o in r["outer"]:
            for arm in r["arms"]:
                add("general", exp=r["exp"], ds=r["ds"], arm=arm, outer=o, seed=0, backbone=r["backbone"],
                    recipe=r["recipe"], steps=None, train_profile=r["train_profile"])
    return out


def make_jobs(verbose=True):
    """Enumerate, build (with disjointness tests), estimate and prioritise every job.  Keeps finished runs."""
    P = plan()
    speed = merged_speed()
    specs = enumerate_specs(P)
    old = {j["id"]: j for j in (jload(D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])}
    jobs, bad = [], []
    order = {p: i for i, p in enumerate(P["phases"])}
    t0 = time.time()
    for k, (phase, s) in enumerate(specs):
        jid = job_id(s)
        fp = job_fp(s, P)
        try:
            B = build_job(s, check=True)
        except AssertionError as e:
            bad.append((jid, "DISJOINTNESS FAILED: " + str(e)))
            continue
        except Exception as e:
            bad.append((jid, f"{type(e).__name__}: {e}"))
            continue
        info = B["info"]
        info["n_xeval"] = int(sum(len(u) for u in B["xeval"].values()))
        R = P["recipes"][s["recipe"]]
        est = estimate_minutes(s, info, R, speed)
        steps, _ = recipe_steps(s, R, info["sizes"]["train"])
        prio = order[phase] * 1_000_000 + (s["seed"] * 10_000 if phase in ("seeds",) else 0) + s["outer"] * 1000 + int(est * 10)
        jobs.append(dict(id=jid, fp=fp, phase=phase, priority=prio, est_min=round(est, 2), steps=steps,
                         sizes=info["sizes"], leak=info["leak"], spec=s))
        if verbose and (k + 1) % 100 == 0:
            print(f"    built {k + 1}/{len(specs)} jobs ({time.time() - t0:.0f}s)", flush=True)
    jobs.sort(key=lambda j: j["priority"])
    # finished runs whose fingerprint no longer matches are moved aside, never silently reused
    moved = 0
    for j in jobs:
        r = jload(D["runs"] / j["id"] / "result.json")
        if r and r.get("fp") != j["fp"]:
            D["superseded"].mkdir(parents=True, exist_ok=True)
            import shutil
            shutil.move(str(D["runs"] / j["id"]), str(D["superseded"] / f"{j['id']}.{int(time.time())}"))
            moved += 1
    jsave(D["jobs"] / "jobs.json", dict(created=time.time(), plan_sha=jload(D["plan"] / "plan.sha256.json", {}).get("sha256"),
                                        jobs=jobs, rejected=bad))
    return jobs, bad, moved


def job_done(j):
    r = jload(D["runs"] / j["id"] / "result.json")
    return bool(r and r.get("status") == "done" and r.get("fp") == j["fp"])


def fail_count(jid):
    return int((jload(D["fail"] / f"{jid}.json", {}) or {}).get("count", 0))


def merged_speed():
    sp = {}
    for p in D["state"].glob("speed_*.json"):
        for k, v in (jload(p, {}) or {}).items():
            sp.setdefault(k, []).extend(v[-20:])
    return {k: float(np.median(v)) for k, v in sp.items() if v}
'''
(Q1_ROOT / "lib" / "q1lib.py").write_text(Q1LIB_SRC)
sys.path.insert(0, str(Q1_ROOT / "lib"))
import q1lib as Q
importlib.reload(Q)
Q.keep_source("15")
Q.ensure_dirs()

# ------------------------------------------------------------------ dependencies
need = {"timm": "timm==1.0.15", "statsmodels": "statsmodels", "skimage": "scikit-image", "psutil": "psutil"}
for mod, pkg in need.items():
    try:
        importlib.import_module(mod)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ------------------------------------------------------------------ analysis plan (fixed and timestamped before any training run)
PLAN = dict(
    version=1,
    title="Leakage in oral-cancer image classification: analysis plan fixed before the experiments (Q1 revision)",
    primary_backbone="convnext_tiny.fb_in22k_ft_in1k",
    primary_metric="bal_acc",
    store_px=288, img=224, budget_min=660,
    near_dup=dict(cand_phash=12, cand_cos=0.95, phash_max=8, ssim_min=0.90, report_cos=0.97,
                  cap_frac=0.02, cap_min=25, fallback_phash=4, fallback_ssim=0.95),
    segmentation=dict(embed="dinov2_s", hist_lift_min=0.10, coarse_min=48),
    parent_recovery=dict(cos_min=0.90, margin_min=0.02, enable_pre_min_rate=0.95),
    outer=dict(he_stained=dict(kind="loso"), histopath=dict(kind="sgkf", n=5), ndb_ufes=dict(kind="sgkf", n=5),
               clinical=dict(kind="sgkf", n=5), breakhis=dict(kind="sgkf", n=5)),
    reps=dict(default=[0], clinical=[0, 1]),
    recipes=dict(
        std=dict(opt="adamw", lr_backbone=1e-4, lr_head=1e-3, wd=0.05, sched="cosine", warmup_frac=0.1, warmup_max=300,
                 min_lr_frac=0.01, label_smoothing=0.1, drop_path=0.1, bs=64, img=224, eval_frac=0.1, select="bal_acc",
                 patience_evals=4, patience_after=0.5, clip=1.0, tta=True, head="linear"),
        scirep=dict(opt="adamax", lr_backbone=1e-4, lr_head=1e-4, wd=0.0, sched="plateau", plateau_factor=0.1,
                    plateau_patience=2, plateau_metric="nll", label_smoothing=0.0, drop_path=0.0, bs=32, img=300,
                    epochs=20, select="nll", patience_evals=10, patience_after=0.0, clip=0.0, tta=False, head="scirep",
                    l2_kernel=0.016, l1_bias=0.006, l1_activity=0.006),
        abdelaziz=dict(opt="adam", lr_backbone=1e-4, lr_head=1e-4, wd=0.0, sched="plateau", plateau_factor=0.5,
                       plateau_patience=2, plateau_metric="acc", label_smoothing=0.0, drop_path=0.0, bs=64, img=300,
                       epochs=30, select="acc", patience_evals=5, patience_after=0.0, clip=0.0, tta=False,
                       head="linear")),
    train_profile=dict(he_stained="std_he", histopath="std_he", ndb_ufes="std_he", clinical="std_photo",
                       breakhis="std_he"),
    exps=dict(
        E1=dict(desc="clean holdout; leaky (L) vs leakage-controlled (C) arms matched on training size, class balance and steps",
                seeds=[0, 1, 2], datasets=dict(
                histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz",
                               target=5000, steps=1300,
                               published="Abd El-Aziz 2025: 1,224 originals augmented to 10,000 before a random 70/15/15 split"),
                ndb_ufes=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                        Cc=dict(S=0, Dup=0, Grp=0, coarse=1)), policy=None, steps=800, extra=["Cc"],
                              published="SciRep 2025: random patch-level 70/15/15 split, augmentation of the training set only"),
                clinical=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                        S0D1=dict(S=0, Dup=1, Grp=0), S1D0=dict(S=1, Dup=0, Grp=0)),
                              policy="keras_generic", target=1200, steps=300, extra=["S0D1", "S1D0"],
                              published="131-image corpus with its duplicates, augmented before a random split"),
                he_stained=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=300,
                                published="shipped OCDC split: test patches come from training slides"),
                breakhis=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=1200,
                              published="random image-level split: patients shared by train and test"))),
        PRE=dict(desc="histopathology corpus as distributed (1,224 originals + 3,968 publisher-augmented copies)",
                 seeds=[0], datasets=dict(histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)),
                                                         policy=None, children=True, steps=900,
                                                         published="SciRep 2025 and Manoj Kumar 2025: the 5,192-file Kaggle corpus split at random"))),
        RL=dict(desc="random-label control: labels drawn per group (source image / slide / patient / duplicate cluster)",
                seeds=[0], outer=[0, 1, 2], datasets=dict(
                histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz", target=5000, steps=1300),
                ndb_ufes=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=800),
                clinical=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy="keras_generic", target=1200, steps=300),
                breakhis=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=1200))),
        DUP=dict(desc="positive control: exact copies of a fraction p of own-evaluation originals injected into training",
                 seeds=[0], outer=[0, 1, 2], doses=[0.0, 0.05, 0.10, 0.20, 0.50], datasets=dict(
                histopath=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz", target=5000, steps=1000),
                ndb_ufes=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=800))),
        OVL=dict(desc="group-overlap dose: fraction q of the evaluation groups' sister images moved into training",
                 seeds=[0], outer=[0, 1, 2], doses=[0.0, 0.25, 0.50, 0.75, 1.0], datasets=dict(
                he_stained=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=300, outer="all"),
                breakhis=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=1000),
                ndb_ufes=dict(arms=dict(C=dict(S=0, Dup=0, Grp=0)), policy=None, steps=800))),
        DOSEK=dict(desc="augmentation-factor dose: k copies per original before (L) or after (C) the split, fixed step budget",
                   seeds=[0], outer=[0, 1, 2], doses=[1, 2, 5, 10, 20], datasets=dict(
                histopath=dict(arms=dict(L=dict(S=1, Dup=0, Grp=0), C=dict(S=0, Dup=0, Grp=0)), policy="abdelaziz", steps=800),
                ndb_ufes=dict(arms=dict(L=dict(S=1, Dup=0, Grp=0), C=dict(S=0, Dup=0, Grp=0)), policy="he_d4", steps=800)))),
    generality=dict(backbones=["resnet50.a1_in1k", "vit_small_patch16_224.augreg_in21k_ft_in1k", "efficientnet_b0.ra_in1k",
                               "convnext_base.fb_in22k_ft_in1k"],
                    datasets=["histopath", "ndb_ufes", "he_stained", "breakhis", "clinical"], outer=[0, 1, 2],
                    arms=["L", "C"], seed=0),
    reimp=[dict(recipe="scirep", exp="E1", ds="ndb_ufes", backbone="efficientnet_b3.ra2_in1k", train_profile="scirep_ndb",
                arms=["L", "C"], outer=[0, 1, 2], paper="SciRep 2025 (NDB-UFES)"),
           dict(recipe="scirep", exp="PRE", ds="histopath", backbone="efficientnet_b3.ra2_in1k", train_profile="scirep_hist",
                arms=["L", "C"], outer=[0, 1, 2], paper="SciRep 2025 (5,192-file histopathology corpus)"),
           dict(recipe="abdelaziz", exp="E1", ds="histopath", backbone="efficientnet_b3.ra2_in1k", train_profile="none",
                arms=["L", "C"], outer=[0, 1, 2], paper="Abd El-Aziz 2025 (LRT EfficientNet-B3)")],
    probes=dict(backbones=["dinov2_s", "convnext_t"], heads=["logreg", "1nn"], seeds=[0, 1, 2, 3, 4], sib_max=20),
    phases=["core", "controls", "dose", "seeds", "general"],
    hypotheses=[
        "H1 (primary, one test per dataset): in the leaky arm L, balanced accuracy on its own evaluation fold exceeds "
        "balanced accuracy of the same models on the clean holdout (inflation_L > 0).",
        "H2: the difference-in-differences inflation_L - inflation_C is positive.",
        "H3: the leakage-controlled arm is approximately unbiased (95% CI of inflation_C contains 0).",
        "H4: leaky training does not improve true generalisation (holdout BA of L minus C; two-sided, no direction).",
        "H5: inflation increases with augmentation factor k (DOSEK), injected duplicate fraction p (DUP) and group "
        "overlap q (OVL); slope from a linear mixed model with outer fold as random intercept.",
        "H6: with labels randomised per group, L scores above chance on its own fold and C does not.",
        "H7 (descriptive): inflation_L is positive for every backbone family; its relation to parameter count is reported."],
    primary_family="H1 across the available datasets (he_stained, histopath, ndb_ufes, clinical, breakhis); Holm correction; "
                   "two-sided p-values and 95% CIs from a two-stage cluster bootstrap (10,000 resamples: outer folds resampled "
                   "with replacement, then groups resampled within each chosen fold); alpha 0.05. Sensitivity: within-fold "
                   "cluster bootstrap, linear mixed model with outer fold as random intercept and seed as variance component, "
                   "and for OCDC an exact sign-flip test over the seven held-out slides.",
    estimands="own_BA = mean over outer folds and seeds of balanced accuracy on the arm's own evaluation fold; "
              "hold_BA = same models on the clean holdout; inflation = own_BA - hold_BA.",
    selection_rules="Model selection, early stopping, temperature scaling and decision thresholds use the inner validation "
                    "split only (carved from the training portion with the arm's own grouping rules). The own evaluation fold "
                    "and the clean holdout are never used for any choice.",
    exclusions="No run is excluded after the fact. A run whose inner-validation balanced accuracy stays at chance with a flat "
               "loss after 30% of its steps is restarted with a new training seed on the same split (at most two restarts); "
               "runs that still fail are reported as missing.",
    exploratory="Everything not listed under H1-H7 is exploratory: pre-augmented corpus (PRE), published-pipeline "
                "reimplementations, cross-dataset transfer, frozen-feature probes, mechanism analyses, calibration, clinical "
                "operating points, uncertainty decomposition, deletion/insertion saliency metrics, factorial extras, NDB-UFES "
                "coarse grouping, provenance-stratified own-fold accuracy (closest relative in training: sibling, duplicate, "
                "group-mate, none), leakage-risk predictor (leave-one-dataset-out comparison of the training-set mean, the share "
                "of own-fold images with any relative in training, and a ridge model on all audit shares).",
    deviations="All Q1 experiments use 224-px inputs (300 px for the EfficientNet-B3 reimplementations) resized from a 288-px "
               "store, instead of 448-512 px in the original manuscript, to fit five backbones, three seeds and the "
               "dose-response grid into the available GPU time. ConvNeXt-T is the primary backbone; ConvNeXt-B is included "
               "as the high-capacity point.",
)
PLAN_SRC = json.dumps(PLAN, indent=1, sort_keys=True)

# amendments are appended here (never edit PLAN above once it has been saved)
AMENDMENTS = []

pj, ph = Q.D["plan"] / "plan.json", Q.D["plan"] / "plan.sha256.json"
created = False
if not pj.exists():
    stamp = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    base = dict(PLAN, created_utc=stamp, lib_version=Q.LIB_VERSION)
    (Q.D["plan"] / "plan_v1.json").write_text(json.dumps(base, indent=1, sort_keys=True))
    Q.jsave(pj, base)
    sha = Q.sha256_file(Q.D["plan"] / "plan_v1.json")
    Q.jsave(ph, dict(sha256=sha, created_utc=stamp, file="plan_v1.json"))
    md = [f"# {PLAN['title']}", "", f"Written and hashed: {stamp} (before any training run; this is a local timestamp, not a public registration)", f"SHA-256 of plan_v1.json: `{sha}`", "",
          "## Estimands", PLAN["estimands"], "", "## Hypotheses"] + [f"- {h}" for h in PLAN["hypotheses"]] + \
         ["", "## Primary family", PLAN["primary_family"], "", "## Selection rules", PLAN["selection_rules"], "",
          "## Exclusions", PLAN["exclusions"], "", "## Exploratory analyses", PLAN["exploratory"], "",
          "## Deviations from the published pipelines", PLAN["deviations"], "", "## Full configuration", "```json",
          PLAN_SRC, "```"]
    (Q.D["plan"] / "ANALYSIS_PLAN.md").write_text("\n".join(md))
    created = True
stored = Q.jload(pj)
shai = Q.jload(ph)
applied = {a["id"] for a in stored.get("amendments", [])}
for a in AMENDMENTS:
    if a["id"] in applied:
        continue
    for path, val in a["changes"].items():
        node = stored
        keys = path.split(".")
        for k in keys[:-1]:
            node = node[k]
        node[keys[-1]] = val
    stored.setdefault("amendments", []).append(dict(a, applied_utc=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())))
    Q.jsave(pj, stored)


def _drop(d, paths):
    """Copy of d without the dotted paths that an amendment changed (so amended keys do not count as drift)."""
    d = json.loads(json.dumps(d))
    for p_ in paths:
        node, ks = d, p_.split(".")
        for k in ks[:-1]:
            node = node.get(k, {}) if isinstance(node, dict) else {}
        if isinstance(node, dict):
            node.pop(ks[-1], None)
    return d


_amended = [p_ for a in stored.get("amendments", []) for p_ in (a.get("changed") or list(a.get("changes", {})))]
same = json.dumps(_drop({k: v for k, v in stored.items() if k not in ("created_utc", "lib_version", "amendments")}, _amended),
                  sort_keys=True) == json.dumps(_drop(json.loads(PLAN_SRC), _amended), sort_keys=True)

# ------------------------------------------------------------------ report
Q.hdr("BLOCK 15 — Q1 SETUP · LIBRARY · ANALYSIS PLAN (fixed before training)")
import numpy as np
try:
    import torch, timm
    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    tv = f"torch {torch.__version__} · timm {timm.__version__}"
except Exception as e:
    gpus, tv = [], f"torch unavailable ({e})"
clk = Q.session_clock(stored["budget_min"])
Q.kv("library", f"q1lib.py {Q.LIB_VERSION} · {(Q.D['lib'] / 'q1lib.py').stat().st_size // 1024} KB · sha {Q.code_hashes()['q1lib.py']}")
Q.kv("versions", f"{tv} · numpy {np.__version__} · python {sys.version.split()[0]}")
Q.kv("GPUs", " | ".join(gpus) if gpus else "none (CPU)")
Q.kv("work root", str(Q.ROOT))
Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")
Q.kv("session", f"#{clk['n']} · elapsed {Q.hms(clk['elapsed'])} · budget {stored['budget_min']} min · {Q.hms(clk['remaining'])} left")

Q.sub("ANALYSIS PLAN")
Q.kv("status", "WRITTEN NOW (timestamped + hashed)" if created else "already written (unchanged file)")
Q.kv("written (UTC)", shai["created_utc"])
Q.kv("SHA-256 (plan_v1.json)", shai["sha256"])
Q.kv("amendments", len(stored.get("amendments", [])) or "none")
for a in stored.get("amendments", []):
    Q.kv(f"  amendment {a['id']}", f"{a.get('applied_utc', '?')} · {a.get('title', '')[:70]}")
if not same and not created:
    Q.kv("note", "this block's PLAN differs from the stored plan outside the amended keys; the stored plan is kept")
Q.kv("hypotheses", f"{len(stored['hypotheses'])} (H1 primary, Holm across datasets)")
Q.kv("files", "plan/ANALYSIS_PLAN.md · plan/plan_v1.json · plan/plan.sha256.json")

Q.sub("DATASETS FOUND UNDER /kaggle/input")
found, n_img = Q.walk_input()
rows = []
for ds in Q.DATASETS:
    recs = found.get(ds, [])
    cls = np.bincount([r["label"] for r in recs], minlength=Q.META[ds]["K"]).tolist() if recs else []
    rows.append([ds, len(recs), " / ".join(f"{c}:{n}" for c, n in zip(Q.META[ds]["classes"], cls)) if recs else "—",
                 "ready" if recs else ("NOT ATTACHED (optional)" if ds == "breakhis" else "MISSING")])
Q.table(["dataset", "files", "classes", "status"], rows, [13, 8, 58, 22])
Q.kv("image files scanned", n_img)
if not found.get("breakhis"):
    print("\n  BreakHis is optional but needed for the patient-level experiments (report item T2.7).")
    print("  Add it with: Add Input -> search 'breakhis' -> 'ambarish/breakhis', then re-run this block.")
print()
if created:
    print("  >>> Save a Kaggle version NOW (Save Version -> Quick Save). Its timestamp is the external proof that")
    print("      the plan existed before any Q1 experiment was run.")
_nxt = "run BLOCK 16 next" if not (Q.D["index"] / "audit.json").exists() else (
    "run BLOCK 19 next (then the next unfinished block; amendment A1 starts with BLOCK 29)"
    if "A1" not in {a["id"] for a in stored.get("amendments", [])}
    else "run BLOCK 19 next, then the first A1 block not yet finished (30 -> 31 -> 32 -> 25 -> 27 -> 33 -> 28)")
print(f"\n[BLOCK 15 ✓]  plan {shai['sha256'][:12]} · {sum(1 for d in Q.DATASETS if found.get(d))}/5 datasets · {_nxt}")

In [ ]:
# ============================================================================
# BLOCK 19 — TRAINING WORKER + LAUNCHER  (writes q1worker.py, defines RUN(...) and STATUS())
# ============================================================================
import os, sys, json, time, signal, threading, queue, subprocess, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("19")
Q1WORKER_SRC = r'''"""
q1worker.py - one training worker per GPU for the OralNet Q1 study.
Written to disk by Block 19.  Pulls jobs from q1/jobs/jobs.json in priority order.

Guarantees
  * model selection, early stopping, temperature and thresholds use the inner-validation split only
  * exact resume: step-indexed batches and augmentation seeds, atomic checkpoint with .prev rollback
  * OOM -> micro-batching with gradient accumulation (effective batch unchanged)
  * SIGTERM/SIGINT -> checkpoint at the next step and exit (signals in the first 90 s are ignored)
  * collapse -> retry the same split with a new training seed (max 2 retries)
"""
import os, sys, time, math, json, signal, argparse, traceback, gc, warnings
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
warnings.filterwarnings("ignore")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
import numpy as np
import q1lib as Q

T0 = time.time()
GRACE = int(os.environ.get("Q1_GRACE", "90"))
STOP = {"flag": False, "why": ""}
TEST_PAUSE_AT = int(os.environ.get("Q1_TEST_PAUSE_AT", "-1"))   # tests only: pause once after this step
TAG = "g?"


def log(msg):
    print(f"[{TAG}] {msg}", flush=True)


def _sig(signum, frame):
    if time.time() - T0 < GRACE:
        log(f"signal {signum} ignored (first {GRACE}s after start)")
        return
    STOP["flag"] = True
    STOP["why"] = f"signal {signum}"


class Collapse(Exception):
    pass


def claim(jid):
    p = Q.D["claims"] / f"{jid}.claim"
    try:
        fd = os.open(str(p), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.write(fd, json.dumps(dict(pid=os.getpid(), tag=TAG, t=time.time())).encode())
        os.close(fd)
        return True
    except FileExistsError:
        return False


def release(jid):
    try:
        (Q.D["claims"] / f"{jid}.claim").unlink()
    except FileNotFoundError:
        pass


def note_fail(jid, msg, tb=""):
    p = Q.D["fail"] / f"{jid}.json"
    r = Q.jload(p, {}) or {}
    r["count"] = int(r.get("count", 0)) + 1
    r.setdefault("msgs", []).append(msg[:400])
    r.setdefault("tracebacks", []).append(tb[-2000:])
    Q.jsave(p, r)


def stop_requested():
    return STOP["flag"] or (Q.D["state"] / "STOP").exists()


def save_speed(gpu, bb, ips):
    p = Q.D["state"] / f"speed_{gpu}.json"
    s = Q.jload(p, {}) or {}
    s.setdefault(bb, []).append(float(ips))
    s[bb] = s[bb][-30:]
    Q.jsave(p, s)


def atomic_torch_save(obj, path):
    import torch
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    if path.exists():
        os.replace(path, path.with_name(path.name + ".prev"))
    os.replace(tmp, path)


def load_ckpt(path, fp):
    import torch
    for p in (path, path.with_name(path.name + ".prev")):
        if p.exists():
            try:
                ck = torch.load(p, map_location="cpu", weights_only=False)
                if ck.get("fp") == fp:
                    return ck
            except Exception:
                continue
    return None


def run_job(job, dev, gpu, deadline, nth, total):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    spec, rid, fp = job["spec"], job["id"], job["fp"]
    P = Q.plan()
    R = dict(P["recipes"][spec["recipe"]])
    ds = spec["ds"]
    B = Q.build_job(spec, check=True)
    info = B["info"]
    K, pos = info["K"], Q.META[ds]["pos"]
    policy = info["policy"]
    prof = spec.get("train_profile") or P["train_profile"][ds]
    img = 48 if Q.TINY else R["img"]
    bs = min(R["bs"], 16) if Q.TINY else R["bs"]
    tr, iv = B["train"], B["ival"]
    n_tr = len(tr)
    steps, every = Q.recipe_steps(spec, dict(R, bs=bs), n_tr)
    sets = dict(ival=B["ival"], own=B["own"], hold=B["hold"])
    rows_by_ds = {ds: np.concatenate([tr["row"]] + [u["row"] for u in sets.values()])}
    for t, u in B["xeval"].items():
        rows_by_ds[t] = u["row"]
    feed = Q.Feeder(rows_by_ds, dev)
    sib = {k: Q.sibling_params(ds, u["row"], u["copy"], policy) for k, u in list(sets.items()) + [("train", tr)]}
    ckp = Q.D["ckpt"] / f"{rid}.pt"
    ck = load_ckpt(ckp, fp)
    attempt = int(ck["attempt"]) if ck else 0
    short = f"{spec['exp']}·{Q.SHORT[ds]}·{spec['arm']}·o{spec['outer']}{'r' + str(spec['rep']) if spec.get('rep') else ''}·s{spec['seed']}" + \
            (f"·d{spec['dose']}" if spec.get("dose") is not None else "")
    bbl = Q.BACKBONES.get(spec["backbone"], (spec["backbone"],))[0] + ("" if spec["recipe"] == "std" else f"/{spec['recipe']}")
    while True:
        try:
            status = _train_once(job, spec, R, B, info, K, pos, policy, prof, img, bs, steps, every, feed, sib,
                                 dev, gpu, deadline, ck, attempt, short, bbl, nth, total)
            return status
        except Collapse as e:
            attempt += 1
            ck = None
            if ckp.exists():
                ckp.unlink()
            if attempt > 2:
                raise RuntimeError(f"collapsed on 3 training seeds: {e}")
            log(f"  {short}  collapse ({e}) -> retry {attempt}/2 with a new training seed (same split)")


def _train_once(job, spec, R, B, info, K, pos, policy, prof, img, bs, steps, every, feed, sib, dev, gpu,
                deadline, ck, attempt, short, bbl, nth, total):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    rid, fp, ds = job["id"], job["fp"], spec["ds"]
    tr = B["train"]
    n_tr = len(tr)
    tseed = Q.shash("train", rid, attempt)
    torch.manual_seed(tseed % (2 ** 31))
    model, head_params, dense = Q.make_model(spec["backbone"], K, img, R.get("drop_path", 0.0), R.get("head", "linear"))
    model = model.to(dev).to(memory_format=torch.channels_last)
    mean, std = Q.norm_consts(model, dev)
    hid = {id(p) for p in head_params}
    groups = {"hd": [], "hn": [], "bd": [], "bn": []}
    for n_, p in model.named_parameters():
        if not p.requires_grad:
            continue
        h = "h" if id(p) in hid else "b"
        d = "n" if (p.ndim <= 1 or n_.endswith(".bias")) else "d"
        groups[h + d].append(p)
    wd = R.get("wd", 0.0)
    pg = [dict(params=groups["bd"], lr=R["lr_backbone"], weight_decay=wd, base=R["lr_backbone"]),
          dict(params=groups["bn"], lr=R["lr_backbone"], weight_decay=0.0, base=R["lr_backbone"]),
          dict(params=groups["hd"], lr=R["lr_head"], weight_decay=wd, base=R["lr_head"]),
          dict(params=groups["hn"], lr=R["lr_head"], weight_decay=0.0, base=R["lr_head"])]
    pg = [g for g in pg if g["params"]]
    if R["opt"] == "adamw":
        opt = torch.optim.AdamW(pg, betas=(0.9, 0.999))
    elif R["opt"] == "adamax":
        opt = torch.optim.Adamax(pg)
    else:
        opt = torch.optim.Adam(pg)
    use_amp = dev.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    act = {}
    if dense is not None:
        dense.register_forward_hook(lambda m, i, o: act.__setitem__("a", o))
    st = dict(step=0, lr_mult=1.0, best=-1e9, best_step=-1, best_state=None, since=0, plat_best=-1e9, plat_since=0,
              hist=[], micro=1, loss0=None, loss_recent=[], t_train=0.0, n_seen=0)
    if ck:
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        st.update(ck["st"])
        log(f"  {short}  RESUMED at step {st['step']}/{steps}")
    ls = R.get("label_smoothing", 0.0)
    warm = min(int(R.get("warmup_frac", 0) * steps), int(R.get("warmup_max", 0))) if R.get("sched") == "cosine" else 0
    spe = max(1, n_tr // bs)
    gen = torch.Generator(device=dev)
    ytr = torch.from_numpy(tr["y"]).to(dev)
    perm_cache = {}

    def lr_factor(s):
        if R.get("sched") == "cosine":
            if s < warm:
                return (s + 1) / max(1, warm)
            t = (s - warm) / max(1, steps - warm)
            mn = R.get("min_lr_frac", 0.01)
            return mn + (1 - mn) * 0.5 * (1 + math.cos(math.pi * t))
        return st["lr_mult"]

    def predict(u, name, dsx=None, pol=None, tta=None):
        dsx = dsx or ds
        tta = R.get("tta", False) if tta is None else tta
        prm = sib.get(name) if pol is not None else None
        model.eval()
        out = []
        eb = 128 if not Q.TINY else 32
        with torch.no_grad():
            for s in range(0, len(u), eb):
                sl = slice(s, s + eb)
                x = feed.batch(dsx, u["row"][sl], u["copy"][sl], None if prm is None else prm[sl], pol)
                x = ((Q.eval_prep(x, img) - mean) / std).contiguous(memory_format=torch.channels_last)
                with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    lg = model(x).float()
                    if tta:
                        lg = (lg + model(x.flip(3)).float()) / 2
                out.append(lg.cpu().numpy())
        model.train()
        return np.concatenate(out) if out else np.zeros((0, K), np.float32)

    def evaluate_ival():
        lg = predict(B["ival"], "ival", pol=policy, tta=False)
        p = Q.softmax(lg)
        m = Q.metrics(p, B["ival"]["y"], K, pos)
        sel = R.get("select", "bal_acc")
        score = {"bal_acc": m["bal_acc"] - 1e-4 * m["nll"], "nll": -m["nll"], "acc": m["acc"]}[sel]
        plat = {"nll": -m["nll"], "acc": m["acc"], "bal_acc": m["bal_acc"]}[R.get("plateau_metric", "nll")]
        return m, score, plat

    def save(final=False):
        atomic_torch_save(dict(fp=fp, attempt=attempt, model=model.state_dict(), opt=opt.state_dict(),
                               scaler=scaler.state_dict(), st=st), Q.D["ckpt"] / f"{rid}.pt")

    if st["step"] == 0:
        log(f"▶ {short}  {bbl}  train {n_tr} · ival {len(B['ival'])} · own {len(B['own'])} · hold {len(B['hold'])}"
            f"  · {steps} steps  [{nth}/{total}]" + (f"  attempt {attempt + 1}" if attempt else ""))
    model.train()
    last_ck = time.time()
    t_start = time.time()
    marks = {int(steps * f) for f in (0.25, 0.5, 0.75)}
    while st["step"] < steps:
        s = st["step"]
        torch.manual_seed(int(Q.shash(tseed, "step", s) % (2 ** 31)))   # dropout / drop-path masks depend on the step only
        e, b = divmod(s, spe)
        if e not in perm_cache:
            perm_cache.clear()
            perm_cache[e] = np.random.default_rng(Q.shash(tseed, "perm", e)).permutation(n_tr)
        idx = perm_cache[e][b * bs:(b + 1) * bs]
        if len(idx) < bs:
            idx = perm_cache[e][:bs]
        for g in opt.param_groups:
            g["lr"] = g["base"] * lr_factor(s)
        t1 = time.time()
        while True:
            try:
                opt.zero_grad(set_to_none=True)
                parts = np.array_split(idx, st["micro"])
                tot = 0.0
                for pi, part in enumerate(parts):
                    x = feed.batch(ds, tr["row"][part], tr["copy"][part], sib["train"][part], policy)
                    gen.manual_seed(int(Q.shash(tseed, "aug", s, pi) % (2 ** 62)))
                    x = Q.train_aug(x, img, prof, gen)
                    x = ((x - mean) / std).contiguous(memory_format=torch.channels_last)
                    yb = ytr[torch.from_numpy(part).to(dev)]
                    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                        lg = model(x)
                        loss = F.cross_entropy(lg.float(), yb, label_smoothing=ls)
                        if dense is not None:
                            loss = loss + R.get("l2_kernel", 0) * dense.weight.float().pow(2).sum() \
                                   + R.get("l1_bias", 0) * dense.bias.float().abs().sum() \
                                   + R.get("l1_activity", 0) * act["a"].float().abs().sum(1).mean()
                    scaler.scale(loss * (len(part) / len(idx))).backward()
                    tot += float(loss) * len(part) / len(idx)
                if R.get("clip", 0) > 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), R["clip"])
                scaler.step(opt)
                scaler.update()
                break
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True)
                gc.collect()
                torch.cuda.empty_cache()
                if bs // (st["micro"] * 2) < 2:
                    raise
                st["micro"] *= 2
                log(f"  {short}  OOM -> micro-batch {bs // st['micro']} x {st['micro']} (effective batch {bs} kept)")
        st["t_train"] += time.time() - t1
        st["n_seen"] += len(idx)
        if not math.isfinite(tot):
            raise Collapse("non-finite loss")
        if st["loss0"] is None and s < 20:
            st.setdefault("_l0", []).append(tot)
            if len(st["_l0"]) >= min(20, steps // 5 + 1):
                st["loss0"] = float(np.mean(st["_l0"]))
        st["loss_recent"] = (st["loss_recent"] + [tot])[-50:]
        st["step"] = s + 1
        s1 = st["step"]
        if s1 % every == 0 or s1 == steps:
            m, score, plat = evaluate_ival()
            st["hist"].append(dict(step=s1, lr=opt.param_groups[0]["lr"], loss=float(np.mean(st["loss_recent"])),
                                   ival_ba=m["bal_acc"], ival_acc=m["acc"], ival_nll=m["nll"]))
            if score > st["best"]:
                st["best"], st["best_step"], st["since"] = score, s1, 0
                st["best_ba"] = m["bal_acc"]
                st["best_state"] = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
            else:
                st["since"] += 1
            if R.get("sched") == "plateau":
                if plat > st["plat_best"]:
                    st["plat_best"], st["plat_since"] = plat, 0
                else:
                    st["plat_since"] += 1
                    if st["plat_since"] >= R.get("plateau_patience", 2):
                        st["lr_mult"] *= R.get("plateau_factor", 0.1)
                        st["plat_since"] = 0
            guard = not str(spec["exp"]).endswith("RL") and steps >= 100 and not Q.TINY
            if guard and s1 >= 0.3 * steps and len(st["hist"]) >= 2 and not st.get("collapse_checked"):
                st["collapse_checked"] = True
                l0 = st["loss0"] or st["hist"][0]["loss"]
                if m["bal_acc"] <= 1.0 / K + 0.02 and np.mean(st["loss_recent"]) > 0.97 * l0:
                    raise Collapse(f"inner-val BA {100 * m['bal_acc']:.1f}% at chance, loss flat")
            ips = st["n_seen"] / max(1e-6, st["t_train"])
            mk = max([mm for mm in marks if mm <= s1], default=None)
            if mk is not None and s1 - every < mk <= s1 or s1 == steps:
                mem = torch.cuda.max_memory_allocated(dev) / 1e9 if dev.type == "cuda" else 0.0
                log(f"  {short}  step {s1}/{steps} · loss {np.mean(st['loss_recent']):.3f} · ival BA {100 * m['bal_acc']:.1f}"
                    f" (best {100 * st.get('best_ba', m['bal_acc']):.1f}) · lr {opt.param_groups[0]['lr']:.1e}"
                    f" · {ips:.0f} img/s · {mem:.1f}GB")
            if st["since"] >= R.get("patience_evals", 4) and s1 >= R.get("patience_after", 0.5) * steps:
                log(f"  {short}  early stop at step {s1} (no inner-val gain for {st['since']} evaluations)")
                break
        if time.time() - last_ck > 60 and st["step"] < steps:     # checkpoint about once a minute, between any two steps
            save()
            last_ck = time.time()
        if stop_requested() or time.time() > deadline - 90 or st["step"] == TEST_PAUSE_AT:
            save()
            why = STOP["why"] or ("stop requested" if (Q.D["state"] / "STOP").exists() else
                                  ("test pause" if st["step"] == TEST_PAUSE_AT else "session deadline"))
            log(f"  {short}  PAUSED at step {st['step']}/{steps} ({why}) - resumable")
            return "paused"
    # ---------------------------------------------------------------- final predictions
    if st["best_state"] is not None:
        model.load_state_dict(st["best_state"])
    out = {}
    for name in ("ival", "own", "hold"):
        u = B[name]
        lg = predict(u, name, pol=policy if name != "hold" else None)
        for k in ("row", "copy", "y", "key"):
            out[f"{name}_{k}"] = u[k]
        out[f"{name}_logit"] = lg.astype(np.float32)
    for t, u in B["xeval"].items():
        lg = predict(u, "x_" + t, dsx=t, pol=None)
        for k in ("row", "copy", "y", "key"):
            out[f"x_{t}_{k}"] = u[k]
        out[f"x_{t}_logit"] = lg.astype(np.float32)
    if "own_injected_mask" in B:
        out["own_injected"] = B["own_injected_mask"].astype(np.int8)
    rdir = Q.D["runs"] / rid
    rdir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(rdir / "preds.npz", **out)
    if spec.get("keep_weights"):
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in model.state_dict().items()}, rdir / "weights.pt")
    T = Q.fit_temperature(out["ival_logit"], out["ival_y"])
    m_own = Q.metrics(Q.softmax(out["own_logit"], T), out["own_y"], K, pos)
    m_hold = Q.metrics(Q.softmax(out["hold_logit"], T), out["hold_y"], K, pos)
    ips = st["n_seen"] / max(1e-6, st["t_train"])
    save_speed(gpu, spec["backbone"], ips * (img / 224.0) ** 2)
    res = dict(id=rid, fp=fp, status="done", spec=spec, recipe=R, train_profile=prof, img=img, bs=bs, steps=steps,
               steps_done=st["step"], best_step=st["best_step"], hist=st["hist"], attempts=attempt + 1, micro=st["micro"],
               info=info, T_ival=T, own_bal_acc=m_own.get("bal_acc"), hold_bal_acc=m_hold.get("bal_acc"),
               own_acc=m_own.get("acc"), hold_acc=m_hold.get("acc"), time_s=time.time() - t_start, img_per_s=ips,
               device=(torch.cuda.get_device_name(0) if dev.type == "cuda" else "cpu"),
               versions=dict(torch=torch.__version__, timm=__import__("timm").__version__, numpy=np.__version__),
               code=Q.code_hashes(), finished_utc=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()))
    Q.jsave(rdir / "result.json", res)
    ckp = Q.D["ckpt"] / f"{rid}.pt"
    for p in (ckp, ckp.with_name(ckp.name + ".prev")):
        if p.exists():
            p.unlink()
    gap = (m_own.get("bal_acc", 0) - m_hold.get("bal_acc", 0)) * 100
    log(f"✓ {short}  own BA {100 * m_own['bal_acc']:.2f} · hold BA {100 * m_hold['bal_acc']:.2f} · gap {gap:+.2f}"
        f" · best@{st['best_step']} · {Q.hms(time.time() - t_start)}")
    del model, opt
    gc.collect()
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    return "done"


def main():
    global TAG
    ap = argparse.ArgumentParser()
    ap.add_argument("--gpu", type=int, default=0)
    ap.add_argument("--phase", default="core")
    ap.add_argument("--deadline", type=float, default=time.time() + 10 * 3600)
    ap.add_argument("--datasets", default="")
    ap.add_argument("--max_jobs", type=int, default=100000)
    a = ap.parse_args()
    TAG = f"g{a.gpu}"
    signal.signal(signal.SIGTERM, _sig)
    signal.signal(signal.SIGINT, _sig)
    import torch
    torch.backends.cudnn.benchmark = True
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    Q.ensure_dirs()
    phases = [p for p in a.phase.split(",") if p]
    dsf = [d for d in a.datasets.split(",") if d]
    log(f"worker up · pid {os.getpid()} · {torch.cuda.get_device_name(0) if dev.type == 'cuda' else 'cpu'} · phases {','.join(phases)}"
        f" · {Q.hms(a.deadline - time.time())} to deadline")
    n_done = n_fail = n_pause = 0
    skipped = set()
    while not stop_requested() and n_done + n_fail < a.max_jobs:
        J = Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}
        jobs = [j for j in J.get("jobs", []) if j["phase"] in phases and (not dsf or j["spec"]["ds"] in dsf)]
        total = len(jobs)
        pending = [j for j in jobs if j["id"] not in skipped and not Q.job_done(j) and Q.fail_count(j["id"]) < 3]
        speed = Q.merged_speed()
        picked = None
        for j in pending:
            est = j["est_min"]
            bb = j["spec"]["backbone"]
            if bb in speed and Q.BACKBONES.get(bb):
                est = est * Q.BACKBONES[bb][2] / max(1.0, speed[bb])
            if Q.TINY:
                est = 0.05
            if time.time() + est * 60 * 1.15 + 120 > a.deadline and not (Q.D["ckpt"] / f"{j['id']}.pt").exists():
                skipped.add(j["id"])
                continue
            if claim(j["id"]):
                picked = j
                break
        if picked is None:
            break
        nth = 1 + sum(1 for j in jobs if Q.job_done(j))
        try:
            st = run_job(picked, dev, a.gpu, a.deadline, nth, total)
            if st == "done":
                n_done += 1
            else:
                n_pause += 1
                release(picked["id"])
                break
        except Exception as e:
            n_fail += 1
            msg = f"{type(e).__name__}: {e}"
            note_fail(picked["id"], msg, traceback.format_exc())
            skipped.add(picked["id"])
            log(f"✗ {picked['id']} FAILED · {msg[:300]}")
            gc.collect()
            if dev.type == "cuda":
                torch.cuda.empty_cache()
        release(picked["id"])
    left = [j for j in (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
            if j["phase"] in phases and (not dsf or j["spec"]["ds"] in dsf) and not Q.job_done(j)]
    log(f"worker exit · {n_done} done · {n_pause} paused · {n_fail} failed · {len(left)} left in phase"
        f" · {Q.hms(a.deadline - time.time())} to deadline")


if __name__ == "__main__":
    main()
'''
(Q.D["lib"] / "q1worker.py").write_text(Q1WORKER_SRC)
compile(Q1WORKER_SRC, "q1worker.py", "exec")
WORKER = Q.D["lib"] / "q1worker.py"


def _alive(pid):
    try:
        os.kill(int(pid), 0)
        return True
    except Exception:
        return False


def _clear_stale_claims():
    n = 0
    for c in Q.D["claims"].glob("*.claim"):
        info = Q.jload(c, {}) or {}
        if not _alive(info.get("pid", -1)):
            c.unlink()
            n += 1
    return n


def _prefetch(backbones):
    if Q.OFFLINE:
        return
    import timm
    for bb in sorted(set(backbones)):
        try:
            m = timm.create_model(Q.tiny_name(bb), pretrained=True, num_classes=0)
            del m
        except Exception as e:
            print(f"  prefetch {bb}: {type(e).__name__} {str(e)[:80]}")


def _phase_table(phases, datasets=None):
    J = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
    rows, agg = [], {}
    for j in J:
        if j["phase"] not in phases or (datasets and j["spec"]["ds"] not in datasets):
            continue
        s = j["spec"]
        k = (s["exp"], s["ds"], s["arm"], Q.BACKBONES.get(s["backbone"], (s["backbone"],))[0] +
             ("" if s["recipe"] == "std" else "/" + s["recipe"]))
        a = agg.setdefault(k, dict(n=0, done=0, own=[], hold=[], fail=0, paused=0))
        a["n"] += 1
        r = Q.jload(Q.D["runs"] / j["id"] / "result.json")
        if r and r.get("status") == "done" and r.get("fp") == j["fp"]:
            a["done"] += 1
            if r.get("own_bal_acc") is not None:
                a["own"].append(r["own_bal_acc"])
                a["hold"].append(r["hold_bal_acc"])
        elif Q.fail_count(j["id"]) >= 3:
            a["fail"] += 1
        elif (Q.D["ckpt"] / f"{j['id']}.pt").exists():
            a["paused"] += 1
    for (ex, ds, arm, bb), a in sorted(agg.items()):
        o = 100 * np.mean(a["own"]) if a["own"] else None
        h = 100 * np.mean(a["hold"]) if a["hold"] else None
        rows.append([ex, ds, arm, bb, f"{a['done']}/{a['n']}", "—" if o is None else f"{o:.2f}",
                     "—" if h is None else f"{h:.2f}", "—" if o is None else f"{o - h:+.2f}",
                     (f"{a['fail']} failed " if a["fail"] else "") + (f"{a['paused']} paused" if a["paused"] else "")])
    return rows


def STATUS():
    importlib.reload(Q)
    P = Q.plan()
    J = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
    Q.hdr("Q1 STATUS")
    rows = []
    for ph in P["phases"]:
        jj = [j for j in J if j["phase"] == ph]
        if not jj:
            continue
        d = sum(Q.job_done(j) for j in jj)
        f = sum(Q.fail_count(j["id"]) >= 3 for j in jj)
        left = sum(j["est_min"] for j in jj if not Q.job_done(j)) / 60
        rows.append([ph, f"{d}/{len(jj)}", f, f"{left:.1f}", f"{left / 2:.1f}"])
    Q.table(["phase", "done", "failed", "GPU-h left", "wall h (2 GPU)"], rows, [10, 12, 8, 12, 15])
    Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")


def RUN(phase, datasets=None, budget_min=None):
    """Train every pending job of `phase` (comma-separated phases allowed) on all GPUs, resumably."""
    importlib.reload(Q)
    P = Q.plan()
    phases = [p for p in phase.split(",") if p]
    clk = Q.session_clock(P["budget_min"] if budget_min is None else budget_min)
    deadline = clk["deadline"]
    J = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
    if not J:
        print("jobs.json missing - run BLOCK 18 first")
        return
    JJ = [j for j in J if j["phase"] in phases and (not datasets or j["spec"]["ds"] in datasets)]
    stale = _clear_stale_claims()
    stopf = Q.D["state"] / "STOP"
    if stopf.exists():
        stopf.unlink()
    pend = [j for j in JJ if not Q.job_done(j) and Q.fail_count(j["id"]) < 3]
    resum = [j for j in pend if (Q.D["ckpt"] / f"{j['id']}.pt").exists()]
    try:
        import torch
        ngpu = max(1, torch.cuda.device_count())
        gname = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    except Exception:
        ngpu, gname = 1, "cpu"
    Q.hdr(f"TRAINING · phase {'+'.join(phases).upper()} · session {clk['n']}")
    Q.kv("content", " | ".join(Q.PHASE_DESC.get(p, p) for p in phases))
    Q.kv("jobs", f"{len(JJ)} in phase · {len(JJ) - len(pend)} done · {len(pend)} to run · {len(resum)} resume mid-run")
    Q.kv("estimated work", f"{sum(j['est_min'] for j in pend) / 60:.1f} GPU-h -> ~{sum(j['est_min'] for j in pend) / 60 / ngpu:.1f} h on {ngpu} x {gname}")
    Q.kv("session", f"{Q.hms(clk['elapsed'])} elapsed · {Q.hms(deadline - time.time())} until the stop line (budget {P['budget_min']} min)")
    Q.kv("safety", "inner-val selection only · checkpoint ~every minute · OOM -> micro-batches · collapse -> reseed x2")
    Q.kv("interrupt", "press Stop: workers checkpoint within ~1 step and exit (allow up to 2 minutes)")
    Q.kv("stale claims cleared", stale)
    Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")
    if not pend:
        print("\n  nothing to run in this phase.")
    else:
        _prefetch([j["spec"]["backbone"] for j in pend])
        env = dict(os.environ, PYTHONWARNINGS="ignore", HF_HUB_DISABLE_PROGRESS_BARS="1", TOKENIZERS_PARALLELISM="false",
                   OMP_NUM_THREADS="2", PYTHONUNBUFFERED="1")
        procs, q_ = [], queue.Queue()
        for g in range(ngpu):
            e = dict(env, CUDA_VISIBLE_DEVICES=str(g)) if gname != "cpu" else env
            p = subprocess.Popen([sys.executable, "-u", str(WORKER), "--gpu", str(g), "--phase", ",".join(phases),
                                  "--deadline", str(deadline), "--datasets", ",".join(datasets or [])],
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e,
                                 start_new_session=True)
            procs.append(p)
            threading.Thread(target=lambda pp=p: [q_.put(l) for l in pp.stdout], daemon=True).start()
            time.sleep(3)
        Q.kv("launched", f"{len(procs)} worker(s) · pids {[p.pid for p in procs]}")
        print("=" * Q.W, flush=True)

        def pump(timeout=0.5):
            try:
                while True:
                    l = q_.get(timeout=timeout).rstrip()
                    if l.startswith("[g") or "Error" in l or "Traceback" in l:
                        print(l, flush=True)
                    timeout = 0.01
            except queue.Empty:
                pass

        try:
            while any(p.poll() is None for p in procs):
                pump()
            pump(0.2)
        except KeyboardInterrupt:
            print("\n  interrupt — asking workers to checkpoint and exit ...", flush=True)
            stopf.write_text(str(time.time()))
            for p in procs:
                try:
                    os.killpg(os.getpgid(p.pid), signal.SIGTERM)
                except Exception:
                    pass
            t0 = time.time()
            while any(p.poll() is None for p in procs) and time.time() - t0 < 150:
                pump()
            for p in procs:
                if p.poll() is None:
                    os.killpg(os.getpgid(p.pid), signal.SIGKILL)
            pump(0.2)
            print("  stopped; progress is checkpointed.", flush=True)
    _clear_stale_claims()
    if stopf.exists():
        stopf.unlink()
    # ------------------------------------------------ auto-save summary
    Q.hdr(f"AUTO-SAVE · phase {'+'.join(phases)}")
    rows = _phase_table(phases, datasets)
    Q.table(["exp", "dataset", "arm", "backbone", "done", "own BA", "hold BA", "gap", "notes"], rows,
            [7, 12, 6, 22, 9, 9, 9, 9, 22])
    print("  own BA = balanced accuracy on the arm's own evaluation fold (what a paper would report);")
    print("  hold BA = the same models on the clean holdout; gap = own - hold (percentage points).")
    fails = [(j["id"], (Q.jload(Q.D["fail"] / f"{j['id']}.json", {}) or {}).get("msgs", [""])[-1])
             for j in JJ if Q.fail_count(j["id"]) >= 1 and not Q.job_done(j)]
    if fails:
        Q.sub("FAILURES (retried automatically on the next run, up to 3 attempts)")
        for jid, m in fails[:15]:
            print(f"  {jid}: {m[:160] if m else ''}")
    left = [j for j in JJ if not Q.job_done(j) and Q.fail_count(j["id"]) < 3]
    Q.kv("remaining in phase", f"{len(left)} jobs · {sum(j['est_min'] for j in left) / 60:.1f} GPU-h")
    Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB")
    print("  Before closing Kaggle: Persistence = 'Files only', then Save Version.")
    print(f"\n[AUTO-SAVE ✓]  {'phase complete' if not left else 're-run this block to continue (next session: 15 -> 19 -> this block)'}")


Q.hdr("BLOCK 19 — WORKER + LAUNCHER READY")
Q.kv("worker", f"q1worker.py · {WORKER.stat().st_size // 1024} KB · syntax PASS · sha {Q.code_hashes()['q1worker.py']}")
Q.kv("rules", "selection / early stop / temperature / thresholds on inner validation only")
Q.kv("resume", "same batches, augmentations and dropout masks at every step as an uninterrupted run; .prev rollback")
Q.kv("launch", "RUN('core') · RUN('controls') · RUN('dose') · RUN('seeds') · RUN('general') · RUN('orchid,recover') · STATUS()")
STATUS()
print("\n[BLOCK 19 ✓]  run the training blocks 20-24, or BLOCK 32 for amendment A1 (each can be re-run to resume)")

In [ ]:
# ============================================================================
# BLOCK 29 — AMENDMENT A1 TO THE ANALYSIS PLAN (written and hashed before any new run)
# metadata-free group recovery · Phikon pathology foundation model · ORCHID (oral H&E with patient IDs)
# needs: BLOCK 15 (new library) run in this session; the finished Q1 runs, jobs.json and plan.json on disk
# ============================================================================
import os, sys, json, time, copy, hashlib, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("29")

Q.hdr("BLOCK 29 — ANALYSIS-PLAN AMENDMENT A1 (before any new experiment)")
P = Q.plan()
REC = dict(S=0, Dup=0, Grp=0)
A1_DOC = dict(
    id="A1",
    title="Metadata-free group recovery, a pathology foundation model, and an oral dataset with patient identifiers",
    rationale=("Three of the four oral datasets carry no patient key, so their leakage-controlled arms rely on groups "
               "recovered from file order or duplicates. A1 adds (i) an oral H&E dataset with patient identifiers "
               "(ORCHID) to measure patient-level leakage directly, and (ii) a metadata-free grouping recovered from "
               "image content alone, validated against true patients/slides, so that practitioners without metadata "
               "can build leakage-safe splits."),
    hypotheses=[
        "H8 (descriptive): Ward clusters of frozen image embeddings recover patient or slide identity; AMI and ARI against "
        "the true groups of BreakHis (patients), OCDC (slides) and ORCHID (patients) are reported for every feature set "
        "and cluster count.",
        "H9 (confirmatory, single test, alpha 0.05, two-sided two-stage cluster bootstrap over outer folds and patients): "
        "on ORCHID the image-level split (arm L) over-estimates patient-disjoint balanced accuracy (inflation_L > 0).",
        "H10 (estimation): arms split by recovered groups at the automatically chosen cluster count (Rauto) have an "
        "inflation whose 95% CI contains 0 on BreakHis, OCDC and ORCHID; their residual leak share (own-fold images whose "
        "true group is in training) is reported next to that of L. Fixed counts k=25 and k=100 are reported as well.",
        "H11 (descriptive): frozen-probe inflation decreases as recovered clusters get coarser (Spearman between images per "
        "cluster and inflation), with the image-level split and the true grouping as reference points.",
        "H12 (descriptive): frozen features of a pathology foundation model (Phikon) show the same leaky-versus-controlled "
        "inflation pattern as DINOv2 on the E1 seed-0 splits."],
    orchid=dict(source="Zenodo record 12646943, file test.zip (all 1,592 PNG members; the released test partition, which "
                       "contains every patient); members verified with the CRC32 of the zip central directory",
                classes="normal -> Normal; osmf -> OSMF; wdoscc, mdoscc, pdoscc -> OSCC (positive class)",
                group="patient = slide folder name (o-<class>-<source>-<sample>)",
                audit=("patient overlap of the released train/val/test partitions from the three central directories; "
                       "members identical across partitions (same CRC32 and size, confirmed by MD5 of the downloaded "
                       "bytes) and whether they sit in different patient folders or grade folders; inside the corpus, "
                       "identical and near-duplicate images that link different patient folders"),
                outer=("stratified group 5-fold on patients unioned with identical-file and near-duplicate groups (so "
                       "patient folders linked by a shared image stay together); clean holdout = patient-disjoint"),
                arms="L = image-level split (patients shared); C = patient-grouped; R25/R100/Rauto = recovered groups",
                seeds="L and C: 0, 1, 2; recovered-group arms: 0", steps=400),
    recovery=dict(features=["dinov2_s", "phikon", "phikon+stain"],
                  fallback="if Phikon cannot be loaded: dinov2_s and dinov2_s+stain",
                  stain="per-image mean and standard deviation of CIELAB channels (z-scored), scaled by 1/sqrt(6)",
                  method="Ward linkage on L2-normalised features of all usable images (labels never used)",
                  k_grid=[10, 15, 25, 40, 60, 100, 150, 250, 400], k_max="n/3",
                  auto="k in the grid within [10, n/10] with the highest cosine silhouette (<= 3000 sampled images, seed 0)",
                  selection="feature set with the highest mean AMI at its automatic k over BreakHis, OCDC and ORCHID",
                  units="recovered clusters are unioned with identical-file and near-duplicate groups before splitting",
                  fine_tuned="BreakHis, OCDC, ORCHID: k25, k100, kauto; NDB-UFES and histopathology: kauto; clinical: none "
                             "(108 images, groups are duplicate clusters)",
                  probes="logistic regression and 1-NN on DINOv2 features; dev pool of each E1 outer fold split 80/20 "
                         "by: single images, every k of the grid, the audit grouping; seeds 0-2"),
    controls=dict(orchid_random_labels="X1RL: labels drawn per patient group, L and C, outer folds 0-2",
                  ndb_random_labels="X2RL: NDB-UFES Rauto with labels drawn per group (tests whether the controlled-arm "
                                    "excess under random labels disappears with recovered grouping)"),
    reporting=("Same estimands, metrics and two-stage cluster bootstrap as the original plan. H9 is a separate "
               "confirmatory family of one test; H8, H10, H11 and H12 are estimation/descriptive. The original H1-H7 "
               "family is not changed; ORCHID is not added to it."),
    exclusions="Same collapse rule as the original plan; failed runs are reported as missing.")

CHANGES = {
    "exps.X1": dict(desc="ORCHID: image-level (L) vs patient-grouped (C) vs recovered-group arms", seeds=[0, 1, 2],
                    seeds_recovered=[0],
                    datasets=dict(orchid=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                                        R25=dict(REC, recov="k25"), R100=dict(REC, recov="k100"),
                                                        Rauto=dict(REC, recov="kauto")),
                                              policy=None, steps=400, train=["L", "C", "R25", "R100", "Rauto"]))),
    "exps.X1RL": dict(desc="ORCHID random-label control (labels per patient group)", seeds=[0], outer=[0, 1, 2],
                      datasets=dict(orchid=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0)),
                                                policy=None, steps=400, train=["L", "C"]))),
    "exps.X2": dict(desc="arms split by metadata-free recovered groups (L and C are built only to match training size)",
                    seeds=[0],
                    datasets=dict(
                        breakhis=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                                R25=dict(REC, recov="k25"), R100=dict(REC, recov="k100"),
                                                Rauto=dict(REC, recov="kauto")),
                                      policy=None, steps=1200, train=["R25", "R100", "Rauto"]),
                        he_stained=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                                  R25=dict(REC, recov="k25"), R100=dict(REC, recov="k100"),
                                                  Rauto=dict(REC, recov="kauto")),
                                        policy=None, steps=300, train=["R25", "R100", "Rauto"]),
                        ndb_ufes=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                                Rauto=dict(REC, recov="kauto")),
                                      policy=None, steps=800, train=["Rauto"]),
                        histopath=dict(arms=dict(L=dict(S=1, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                                 Rauto=dict(REC, recov="kauto")),
                                       policy="abdelaziz", target=5000, steps=1300, train=["Rauto"]))),
    "exps.X2RL": dict(desc="NDB-UFES random-label control with recovered grouping", seeds=[0], outer=[0, 1, 2],
                      datasets=dict(ndb_ufes=dict(arms=dict(L=dict(S=0, Dup=1, Grp=1), C=dict(S=0, Dup=0, Grp=0),
                                                            Rauto=dict(REC, recov="kauto")),
                                                  policy=None, steps=800, train=["Rauto"]))),
    "outer.orchid": dict(kind="sgkf", n=5),
    "train_profile.orchid": "std_he",
    "phases": list(P["phases"]) + [p for p in ("orchid", "recover") if p not in P["phases"]],
    "amendment_A1": A1_DOC,
}


def apply_changes(base, changes):
    out = copy.deepcopy(base)
    for path, val in changes.items():
        node = out
        keys = path.split(".")
        for k in keys[:-1]:
            node = node.setdefault(k, {})
        node[keys[-1]] = copy.deepcopy(val)
    return out


doc_json = json.dumps(dict(changes=CHANGES), sort_keys=True, indent=1)
doc_sha = hashlib.sha256(doc_json.encode()).hexdigest()
applied = {a["id"]: a for a in P.get("amendments", [])}
jobs_all = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
if "A1" in applied:
    st = "already applied" + (" (identical content)" if applied["A1"].get("sha256") == doc_sha else
                              " (this cell's text differs from the stored amendment; the STORED version stays in force)")
    Q.kv("amendment A1", st)
    Q.kv("written (UTC)", applied["A1"]["applied_utc"])
    Q.kv("SHA-256", applied["A1"]["sha256"])
else:
    P2 = apply_changes(P, CHANGES)
    bad = [j["id"] for j in jobs_all if j["spec"]["exp"] in P["exps"] and Q.job_fp(j["spec"], P2) != j["fp"]]
    if bad:
        raise SystemExit(f"amendment would change the fingerprint of {len(bad)} existing jobs (e.g. {bad[:3]}) - not applied")
    stamp = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    (Q.D["plan"] / "amendment_A1.json").write_text(doc_json)
    rec = dict(id="A1", applied_utc=stamp, sha256=doc_sha, file="amendment_A1.json", title=A1_DOC["title"],
               changed=sorted(CHANGES))
    P2.setdefault("amendments", []).append(rec)
    Q.jsave(Q.D["plan"] / "plan.json", P2)
    md = Q.D["plan"] / "ANALYSIS_PLAN.md"
    txt = md.read_text() if md.exists() else ""
    add = ["", f"## Amendment A1 — {A1_DOC['title']}", "",
           f"Written and hashed: {stamp}, before any A1 experiment was run (local timestamp). "
           f"SHA-256 of amendment_A1.json: `{doc_sha}`.", "", A1_DOC["rationale"], "", "### Hypotheses"] + \
          [f"- {h}" for h in A1_DOC["hypotheses"]] + ["", "### ORCHID"] + [f"- {k}: {v}" for k, v in A1_DOC["orchid"].items()] + \
          ["", "### Group recovery"] + [f"- {k}: {v}" for k, v in A1_DOC["recovery"].items()] + \
          ["", "### Controls"] + [f"- {k}: {v}" for k, v in A1_DOC["controls"].items()] + \
          ["", "### Reporting", A1_DOC["reporting"], "", "### Exclusions", A1_DOC["exclusions"], ""]
    md.write_text(txt + "\n".join(add))
    Q.kv("amendment A1", "WRITTEN NOW (timestamped + hashed) — Save Version right after this block")
    Q.kv("written (UTC)", stamp)
    Q.kv("SHA-256", doc_sha)
    Q.kv("existing jobs", f"{len(jobs_all)} checked · fingerprints unchanged by the amendment")
P = Q.plan()
Q.kv("hypotheses added", "H8-H12 (H9 confirmatory: ORCHID patient-level inflation)")
Q.kv("new phases", ", ".join(p for p in P["phases"] if p in ("orchid", "recover")))

# snapshot of the frozen Q1 results, used later to prove that nothing in them changed
snap = Q.D["eval"] / "q1_results.pre_A1.json"
if (Q.D["eval"] / "q1_results.json").exists() and not snap.exists():
    snap.write_text((Q.D["eval"] / "q1_results.json").read_text())
Q.kv("Q1 results snapshot", "eval/q1_results.pre_A1.json" if snap.exists() else "no q1_results.json yet (run BLOCK 25 first)")

Q.sub("ENVIRONMENT CHECKS FOR A1")
try:
    import transformers
    Q.kv("transformers", transformers.__version__)
except Exception as e:
    Q.kv("transformers", f"missing ({e}) -> installing")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=False)
try:
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m = Q._emb_model("phikon", 56 if Q.TINY else 224, dev)
    x = torch.zeros(2, 3, 56 if Q.TINY else 224, 56 if Q.TINY else 224, device=dev)
    with torch.no_grad():
        f = m(x.half() if dev == "cuda" else x)
    Q.kv("Phikon (owkin/phikon)", f"loaded · embedding dim {f.shape[1]}")
except Exception as e:
    Q.kv("Phikon (owkin/phikon)", f"NOT available ({type(e).__name__}: {str(e)[:90]}) -> BLOCK 31 falls back to DINOv2")
try:
    import requests
    src = json.loads(os.environ.get("Q1_ORCHID_SOURCES", "{}")) or {
        "test": "https://zenodo.org/records/12646943/files/test.zip?download=1"}
    r = requests.get(src["test"], headers={"Range": "bytes=0-0"}, timeout=60, allow_redirects=True)
    Q.kv("Zenodo (ORCHID test.zip)", f"HTTP {r.status_code} · " + ("range requests supported" if r.status_code == 206 else
                                                                    "range requests NOT supported"))
except Exception as e:
    Q.kv("Zenodo (ORCHID test.zip)", f"not reachable ({type(e).__name__}) — turn Internet on; BLOCK 30 will retry")
Q.kv("disk free", f"{Q.disk_free_gb():.2f} GB (ORCHID needs about 0.5 GB; nothing large is written)")
print("\n[BLOCK 29 ✓]  amendment A1 in force · run BLOCK 30 next")

In [ ]:
# ============================================================================
# BLOCK 30 — ORCHID: ORAL H&E WITH PATIENT IDs (Zenodo) · INGESTION + AUDIT
# reads only the needed bytes of the Zenodo zip files (HTTP range requests); resumable; nothing large on disk
# needs: BLOCK 15 + BLOCK 29 in this session (Internet on)
# ============================================================================
import os, sys, io, re, json, time, zlib, struct, hashlib, threading, importlib, warnings
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import requests
import q1lib as Q
importlib.reload(Q)
Q.keep_source("30")
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

DS = "orchid"
SOURCES = json.loads(os.environ.get("Q1_ORCHID_SOURCES", "{}")) or {
    "test": "https://zenodo.org/records/12646943/files/test.zip?download=1",
    "val": "https://zenodo.org/records/12646943/files/val.zip?download=1",
    "train": "https://zenodo.org/records/12636426/files/train.zip?download=1"}
USE = "test"                                     # corpus = every member of the released test partition (A1)
CLASS_MAP = {"normal": 0, "osmf": 1, "wdoscc": 2, "mdoscc": 2, "pdoscc": 2}
CLASSES = ["Normal", "OSMF", "OSCC"]
THREADS = 8
BATCH_BYTES = 48 * 2 ** 20
PX = Q.STORE_PX
P = Q.plan()
ND = P["near_dup"]
EMB = (Q.jload(Q.D["index"] / "audit.json", {}) or {}).get("_meta", {}).get("embed", "dinov2_s")
t_all = time.time()
Q.hdr("BLOCK 30 — ORCHID INGESTION + AUDIT")
if "amendment_A1" not in P:
    raise SystemExit("amendment A1 not found in the plan - run BLOCK 29 first")
_tl = threading.local()


def _sess():
    if not hasattr(_tl, "s"):
        _tl.s = requests.Session()
    return _tl.s


def get_range(url, a, b, tries=8):
    """Bytes a..b (inclusive) of a remote file; retries with back-off (honours HTTP 429 Retry-After); requires 206."""
    last = None
    for k in range(tries):
        wait = min(60, 2 ** k)
        try:
            with _sess().get(url, headers={"Range": f"bytes={a}-{b}"}, timeout=(30, 180), allow_redirects=True,
                             stream=True) as r:
                if r.status_code == 200:
                    raise RuntimeError("server ignores range requests")
                if r.status_code == 206:
                    data = r.content
                    if len(data) == b - a + 1:
                        return data, r.headers
                    last = f"short read ({len(data)} of {b - a + 1} bytes)"
                else:
                    last = f"HTTP {r.status_code}"
                    if r.status_code == 429:
                        try:
                            wait = max(wait, min(300, int(float(r.headers.get("retry-after", "60")))))
                        except ValueError:
                            wait = 60
        except RuntimeError:
            raise
        except Exception as e:
            last = f"{type(e).__name__}: {str(e)[:80]}"
        time.sleep(wait)
    raise IOError(f"range {a}-{b} failed after {tries} tries ({last})")


def central_directory(url):
    """Entries of a remote zip (ZIP64 aware): name, method, crc, csize, usize, offset."""
    _, h = get_range(url, 0, 0)
    total = int(h["Content-Range"].split("/")[-1])
    a = max(0, total - 70000)
    tail, _ = get_range(url, a, total - 1)
    i = tail.rfind(b"PK\x05\x06")
    if i < 0:
        raise IOError("end of central directory not found")
    _, _, _, _, n_tot, cd_size, cd_off, _ = struct.unpack("<IHHHHIIH", tail[i:i + 22])
    j = tail.rfind(b"PK\x06\x07")
    if j >= 0:
        z64_off = struct.unpack("<Q", tail[j + 8:j + 16])[0]
        z64, _ = get_range(url, z64_off, z64_off + 55)
        n_tot, cd_size, cd_off = struct.unpack("<QQQ", z64[32:56])
    cd, _ = get_range(url, cd_off, cd_off + cd_size - 1)
    out, p = [], 0
    while p + 46 <= len(cd) and cd[p:p + 4] == b"PK\x01\x02":
        (_, _, _, flag, method, _, _, crc, csize, usize, nlen, xlen, clen, _, _, _, off) = \
            struct.unpack("<IHHHHHHIIIHHHHHII", cd[p:p + 46])
        name = cd[p + 46:p + 46 + nlen].decode("utf-8", "replace")
        extra = cd[p + 46 + nlen:p + 46 + nlen + xlen]
        q = 0
        while q + 4 <= len(extra):
            hid, hsz = struct.unpack("<HH", extra[q:q + 4])
            data = extra[q + 4:q + 4 + hsz]
            if hid == 1:
                k = 0
                if usize == 0xFFFFFFFF:
                    usize = struct.unpack("<Q", data[k:k + 8])[0]; k += 8
                if csize == 0xFFFFFFFF:
                    csize = struct.unpack("<Q", data[k:k + 8])[0]; k += 8
                if off == 0xFFFFFFFF:
                    off = struct.unpack("<Q", data[k:k + 8])[0]; k += 8
            q += 4 + hsz
        out.append(dict(name=name, method=method, crc=crc, csize=csize, usize=usize, off=off, flag=flag))
        p += 46 + nlen + xlen + clen
    return out, total


def member_from(buf, e):
    """Raw bytes of one zip member from a buffer that starts at its local header; None if the buffer is too short."""
    if buf[:4] != b"PK\x03\x04":
        raise IOError("bad local header")
    n, m = struct.unpack("<HH", buf[26:30])
    s0 = 30 + n + m
    if s0 + e["csize"] > len(buf):
        return None
    return inflate(buf[s0:s0 + e["csize"]], e)


def fetch_member(url, e, total):
    """Raw bytes of one zip member, CRC32-checked (one or two range requests)."""
    pad = 30 + len(e["name"].encode()) + 1024
    a = e["off"]
    b = min(total - 1, a + pad + e["csize"])
    buf, _ = get_range(url, a, b)
    raw = member_from(buf, e)
    if raw is not None:
        return raw
    n, m = struct.unpack("<HH", buf[26:30])
    more, _ = get_range(url, a + len(buf), a + 30 + n + m + e["csize"] - 1)
    return member_from(buf + more, e)


def inflate(data, e):
    if e["method"] == 8:
        d_ = zlib.decompressobj(-15)
        raw = d_.decompress(data) + d_.flush()
    elif e["method"] == 0:
        raw = data
    else:
        raise IOError(f"unsupported compression {e['method']}")
    if (zlib.crc32(raw) & 0xFFFFFFFF) != e["crc"]:
        raise IOError("CRC mismatch")
    return raw


def parse_name(name):
    parts = [x for x in name.split("/") if x]
    if len(parts) < 3 or Path(parts[-1]).suffix.lower() not in Q.IMG_EXT:
        return None
    cls = parts[-3].lower()
    if cls not in CLASS_MAP:
        return None
    return dict(cls5=cls, label=CLASS_MAP[cls], patient=parts[-2], file=parts[-1])


def decode(raw):
    md5, sha = hashlib.md5(raw).hexdigest(), hashlib.sha256(raw).hexdigest()
    im = Image.open(io.BytesIO(raw))
    w, h = im.size
    mode = im.mode
    if im.format == "JPEG":
        im.draft("RGB", (PX * 2, PX * 2))
    im = im.convert("RGB").resize((PX, PX), Image.BILINEAR, reducing_gap=2.0)
    return md5, sha, np.asarray(im, dtype=np.uint8).copy(), (w, h), mode


# ------------------------------------------------------------------ 1  central directories + official-split audit
Q.sub("1  ZENODO CENTRAL DIRECTORIES  (file lists of the released partitions; no image is downloaded yet)")
cds, ok_src = {}, True
for part, url in SOURCES.items():
    try:
        ents, total = central_directory(url)
        cds[part] = (ents, total)
        files = [e for e in ents if parse_name(e["name"])]
        pats = {parse_name(e["name"])["patient"] for e in files}
        Q.kv(f"{part}.zip", f"{total / 1e9:.2f} GB · {len(files):,} images · {len(pats)} patients")
    except Exception as e:
        Q.kv(f"{part}.zip", f"FAILED ({type(e).__name__}: {str(e)[:90]})")
        if part == USE:
            ok_src = False
if not ok_src:
    print("\n  ORCHID could not be read (network or server problem). Nothing was written; the A1 analyses run without")
    print("  ORCHID. Re-run this block later to add it.")
    print(f"\n[BLOCK 30 ✗]  ORCHID unavailable · continue with BLOCK 31")
else:
    split_pat = {p: {parse_name(e["name"])["patient"] for e in v[0] if parse_name(e["name"])} for p, v in cds.items()}
    audit_x = dict(orchid=dict(source=SOURCES[USE], partitions={p: dict(images=sum(1 for e in v[0] if parse_name(e["name"])),
                                                                         patients=len(split_pat[p]))
                                                                 for p, v in cds.items()}))
    rows = []
    names = list(split_pat)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            sh = len(split_pat[a] & split_pat[b])
            rows.append([f"{a} ∩ {b}", len(split_pat[a]), len(split_pat[b]), sh])
            audit_x["orchid"].setdefault("patients_shared", {})[f"{a}|{b}"] = sh
    if rows:
        Q.table(["released partitions", "patients A", "patients B", "patients in both"], rows, [22, 12, 12, 18])
        print("  a patient in two released partitions means that partition is not a patient-disjoint test set.")

    # identical members across released partitions: same CRC32 and size in the central directories, confirmed by MD5
    def _ckey(e):
        return (e["crc"], e["usize"])

    def _md5_member(part, e):
        return hashlib.md5(fetch_member(SOURCES[part], e, cds[part][1])).hexdigest()

    xrows = []
    xcache_f = Q.D["store"] / f"{DS}.partitions.json"
    xcache = Q.jload(xcache_f, {}) or {}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            ka = defaultdict(list)
            for e in cds[a][0]:
                if parse_name(e["name"]):
                    ka[_ckey(e)].append(e)
            prs = [(ka[_ckey(e)][0], e) for e in cds[b][0] if parse_name(e["name"]) and _ckey(e) in ka]

            def _same(pe):
                try:
                    return _md5_member(a, pe[0]) == _md5_member(b, pe[1])
                except Exception:
                    return None

            ck = hashlib.sha256(json.dumps([[x["name"], y["name"]] for x, y in prs]).encode()).hexdigest()[:16]
            if xcache.get(f"{a}|{b}", {}).get("key") == ck:
                same_ = xcache[f"{a}|{b}"]["same"]
            else:
                with ThreadPoolExecutor(4) as ex:
                    same_ = list(ex.map(_same, prs))
                if None not in same_:                      # cache only complete checks
                    xcache[f"{a}|{b}"] = dict(key=ck, same=same_)
                    Q.jsave(xcache_f, xcache)
            conf = [pe for pe, v in zip(prs, same_) if v]
            pa = lambda e: parse_name(e["name"])

            def _differs(field):                       # y differs from ANY identical member of partition a
                return sum(1 for x, y in conf if any(pa(z)[field] != pa(y)[field] for z in ka[_ckey(x)]))

            d_pat, d_cls, d_lab = _differs("patient"), _differs("cls5"), _differs("label")
            audit_x["orchid"].setdefault("identical_across_partitions", {})[f"{a}|{b}"] = dict(
                candidates=len(prs), confirmed=len(conf), not_checked=sum(1 for v in same_ if v is None),
                different_patient_folder=d_pat, different_grade_folder=d_cls, different_3class_label=d_lab,
                examples=[[x["name"], y["name"]] for x, y in conf[:8]])
            xrows.append([f"{a} ∩ {b}", len(prs), len(conf), d_pat, d_cls, d_lab])
    if xrows:
        Q.table(["released partitions", "same CRC32+size", "MD5-confirmed", "other patient folder", "other grade folder",
                 "other 3-class label"], xrows, [22, 16, 14, 21, 19, 20])
        print("  identical bytes filed under two different patient (slide) folders mean those folders are not independent.")

    # ------------------------------------------------------------------ 2  download + decode (resumable)
    Q.sub(f"2  IMAGES OF {USE}.zip  (range requests, CRC32-checked, decoded once to a {PX}-px store)")
    ents, total = cds[USE]
    sel = sorted([e for e in ents if parse_name(e["name"])], key=lambda e: e["name"])
    n = len(sel)
    sig = hashlib.sha256("\n".join(e["name"] + str(e["crc"]) for e in sel).encode()).hexdigest()[:16]
    store_f = Q.D["store"] / f"{DS}.npy"
    part_f = Q.D["store"] / f"{DS}.npy.partial"
    state_f = Q.D["store"] / f"{DS}.ingest.json"
    state = Q.jload(state_f, {}) or {}
    if store_f.exists() and state.get("sig") == sig and state.get("complete"):
        Q.kv("store", f"reused ({n} images)")
        meta = state["meta"]
    else:
        if state.get("sig") != sig or not part_f.exists():
            state = dict(sig=sig, meta=[None] * n, complete=False)
            arr = np.lib.format.open_memmap(part_f, mode="w+", dtype=np.uint8, shape=(n, PX, PX, 3))
            del arr
        arr = np.lib.format.open_memmap(part_f, mode="r+")
        meta = state["meta"]
        todo = [i for i in range(n) if meta[i] is None or not meta[i]["ok"]]      # failed ones are retried too
        todo_bytes = sum(sel[i]["csize"] for i in todo)
        # neighbouring members are fetched together (one range request per ~48 MB) to stay far below the server's
        # request-rate limit; each member is still CRC32-checked on its own
        batches, cur = [], []
        for i in sorted(todo, key=lambda i: sel[i]["off"]):
            if cur and sel[i]["off"] + sel[i]["csize"] - sel[cur[0]]["off"] > BATCH_BYTES:
                batches.append(cur)
                cur = []
            cur.append(i)
        if cur:
            batches.append(cur)
        Q.kv("to fetch", f"{len(todo):,} of {n:,} images · {todo_bytes / 1e9:.2f} GB · {len(batches)} range requests · "
             f"{THREADS} streams")
        t0, got, nbytes, fails, lost = time.time(), 0, 0, 0, 0

        def work(batch):
            first, last = sel[batch[0]], sel[batch[-1]]
            a0 = first["off"]
            b0 = min(total - 1, last["off"] + 30 + len(last["name"].encode()) + 1024 + last["csize"])
            mv = memoryview(get_range(SOURCES[USE], a0, b0)[0])
            out = []
            for i in batch:
                e = sel[i]
                try:
                    raw = member_from(mv[e["off"] - a0:], e)
                    if raw is None:
                        raw = fetch_member(SOURCES[USE], e, total)
                    md5, sha, a_, wh, mode = decode(raw)
                    out.append((i, dict(md5=md5, sha256=sha, w=wh[0], h=wh[1], mode=mode, bytes=len(raw), ok=True), a_))
                except Exception as ex_:
                    out.append((i, dict(md5=f"failed-{i}", w=0, h=0, mode="", bytes=0, ok=False, error=str(ex_)[:120]), None))
            return out

        last_print = 0.0
        with ThreadPoolExecutor(THREADS) as ex:
            futs = {ex.submit(work, bt): bt for bt in batches}
            for f in as_completed(futs):
                bt = futs[f]
                try:
                    for i, m_, a_ in f.result():
                        meta[i] = m_
                        if a_ is not None:
                            arr[i] = a_
                        else:
                            fails += 1
                except Exception as ex_:                    # the whole request failed: retried on the next run
                    lost += len(bt)
                    print(f"    range request failed ({type(ex_).__name__}: {str(ex_)[:90]}); {len(bt)} images left for "
                          f"the next run", flush=True)
                got += len(bt)
                nbytes += sum(sel[i]["csize"] for i in bt)
                arr.flush()
                Q.jsave(state_f, dict(state, meta=meta))
                if time.time() - last_print > 20 or got == len(todo):
                    last_print = time.time()
                    rate = nbytes / 1e6 / max(1e-6, time.time() - t0)
                    eta = (todo_bytes - nbytes) / 1e6 / max(rate, 1e-6)
                    print(f"    {got:,}/{len(todo):,} images · {rate:.1f} MB/s · ETA {Q.hms(eta)} · unreadable {fails}"
                          + (f" · not fetched {lost}" if lost else ""), flush=True)
        arr.flush()
        del arr
        missing = sum(1 for m in meta if m is None)
        bad = sum(1 for m in meta if m is not None and not m["ok"])
        if missing or bad > 0.05 * n:
            Q.jsave(state_f, dict(state, meta=meta))
            raise SystemExit(f"{missing} images not fetched and {bad} unreadable of {n} - re-run this block to continue "
                             f"(finished images are kept)")
        os.replace(part_f, store_f)
        state = dict(sig=sig, meta=meta, complete=True)
        Q.jsave(state_f, state)
        Q.kv("downloaded", f"{n - bad:,} images in {Q.hms(time.time() - t0)} · {bad} unreadable (excluded)")

    # ------------------------------------------------------------------ 3  records, hashes, embeddings, duplicates
    Q.sub("3  AUDIT  (same rules as BLOCK 16: MD5, pHash/dHash, SSIM-confirmed near duplicates, DINOv2 embeddings)")
    recs = []
    for i, (e, m) in enumerate(zip(sel, meta)):
        pn = parse_name(e["name"])
        recs.append(dict(path=f"zenodo:{USE}.zip/{e['name']}", label=pn["label"], cls5=pn["cls5"], patient=pn["patient"],
                         native=USE, md5=m["md5"], sha256=m.get("sha256", ""), w=m["w"], h=m["h"], mode=m["mode"], ok=bool(m["ok"]), bytes=m["bytes"],
                         uid=i, orig=True))
    st = Q.open_store(DS)
    fh = Q.D["store"] / f"{DS}_hash.npz"
    fe = Q.D["store"] / f"{DS}_emb_{EMB}.npy"
    if not fh.exists():
        ph, dh = Q.perceptual_hashes(st)
        np.savez(fh, ph=ph, dh=dh)
    if not fe.exists():
        np.save(fe, Q.embed(DS, np.arange(n), key=EMB).astype(np.float16))
    H = np.load(fh)
    ph = H["ph"]
    E = np.load(fe).astype(np.float32)
    ok = np.array([r["ok"] for r in recs])
    use = np.nonzero(ok)[0]
    md5 = np.array([r["md5"] for r in recs])
    ii, jj, dd_ = Q.hamming_pairs(ph[use], max_d=ND["cand_phash"])
    ci_, cj_, cs_ = Q.cos_pairs(E[use], thr=ND["cand_cos"])
    cand = {}
    for a, b, d in zip(use[ii], use[jj], dd_):
        cand[(a, b)] = [int(d), None]
    for a, b, s_ in zip(use[ci_], use[cj_], cs_):
        cand.setdefault((a, b), [None, None])[1] = float(s_)
    pairs = np.array(list(cand.keys()), int).reshape(-1, 2)
    ham = np.array([v[0] if v[0] is not None else int(Q.popcount64(np.array([ph[a] ^ ph[b]]))[0])
                    for (a, b), v in cand.items()], int)
    cosv = np.array([v[1] if v[1] is not None else float(E[a] @ E[b]) for (a, b), v in cand.items()])
    ss = np.full(len(pairs), np.nan)
    need = np.nonzero(ham <= max(ND["phash_max"], ND["fallback_phash"]))[0]
    if len(need):
        ss[need] = Q.ssim_pairs(st, pairs[need, 0], st, pairs[need, 1], max_pairs=10 ** 9)
    by = defaultdict(list)
    for i in use:
        by[md5[i]].append(i)
    ex_pairs = [(g[0], x) for g in by.values() if len(g) > 1 for x in g[1:]]
    ssf = np.nan_to_num(ss, nan=-1.0)
    rule = (ham <= ND["phash_max"]) & (ssf >= ND["ssim_min"]) if len(pairs) else np.zeros(0, bool)
    roots = Q.union_labels(n, pairs=[tuple(p) for p in pairs[rule]] + ex_pairs)
    big = max(Counter(roots[use].tolist()).values()) if len(use) else 1
    cap = max(ND["cap_min"], ND["cap_frac"] * len(use))
    near_rule = "main"
    if big > cap:
        rule = (ham <= ND["fallback_phash"]) & (ssf >= ND["fallback_ssim"])
        roots = Q.union_labels(n, pairs=[tuple(p) for p in pairs[rule]] + ex_pairs)
        near_rule = f"fallback (largest component {big} > cap {cap:.0f})"
    pid = {p: k for k, p in enumerate(sorted({r["patient"] for r in recs}))}
    struct_ = np.array([pid[r["patient"]] for r in recs])
    first = {}
    exact = np.array([first.setdefault(m, i) for i, m in enumerate(md5)])
    ctrl = Q.union_labels(n, label_arrays=[roots, struct_])
    rep_ = np.zeros(n, bool)
    seen = set()
    for i in np.argsort(exact, kind="stable"):
        if ok[i] and exact[i] not in seen:
            rep_[i] = True
            seen.add(exact[i])
    for i, r in enumerate(recs):
        r.update(g_exact=int(exact[i]), g_near=int(roots[i]), g_struct=int(struct_[i]), g_coarse=int(struct_[i]),
                 g_ctrl=int(ctrl[i]), dup_rep=bool(rep_[i]))
    base = ok & rep_
    labs = np.array([r["label"] for r in recs])
    gl = defaultdict(set)
    for i in np.nonzero(base)[0]:
        gl[ctrl[i]].add(int(labs[i]))
    n_ex_groups = sum(1 for g in by.values() if len(g) > 1)
    n_ex_red = sum(len(g) - 1 for g in by.values() if len(g) > 1)
    summ = dict(n_files=n, n_unique_originals=int(base.sum()), n_ctrl_groups=len(set(ctrl[base].tolist())),
                mixed_label_groups=sum(1 for v in gl.values() if len(v) > 1), near_rule=near_rule, classes=CLASSES,
                n_patients=len(pid))
    Q.jsave(Q.D["index"] / f"{DS}.json", dict(summary=summ, records=recs))
    reg = Q.jload(Q.D["index"] / "_registry.json", {}) or {}
    reg[DS] = dict(short="ORCH", meta=dict(K=3, classes=CLASSES, pos=2, modality="he", group="patient",
                                           title="ORCHID oral H&E (patient IDs, Zenodo)"))
    Q.jsave(Q.D["index"] / "_registry.json", reg)
    Q.load_registry()
    rows = []
    for c, name in enumerate(CLASSES):
        m_ = base & (labs == c)
        rows.append([name, int(m_.sum()), len({recs[i]["patient"] for i in np.nonzero(m_)[0]})])
    Q.table(["class", "images", "patients"], rows, [10, 10, 10])
    sub5 = Counter(r["cls5"] for r in recs if r["ok"])
    Q.kv("original folders", ", ".join(f"{k} {v}" for k, v in sorted(sub5.items())))
    ppi = Counter(struct_[base].tolist())
    Q.kv("images per patient", f"min {min(ppi.values())} · median {int(np.median(list(ppi.values())))} · max {max(ppi.values())}")
    Q.kv("identical files", f"{n_ex_groups} groups · {n_ex_red} redundant")
    Q.kv("near duplicates", f"{int(rule.sum())} pairs (pHash<=8 and SSIM>=0.90) · rule {near_rule}")
    Q.kv("candidate pairs", f"pHash<=8: {int((ham <= 8).sum())} · DINOv2 cosine>=0.97: {int((cosv >= .97).sum())}")
    Q.kv("groups (g_ctrl)", f"{summ['n_ctrl_groups']} · mixed-label groups {summ['mixed_label_groups']}")
    pat_of = np.array([r["patient"] for r in recs])
    c5_of = np.array([r["cls5"] for r in recs])
    cross = {}
    for gname, garr in (("identical", exact), ("near_duplicate", roots)):
        mem = defaultdict(list)
        for i in use:
            mem[int(garr[i])].append(i)
        multi = [v for v in mem.values() if len(set(pat_of[v].tolist())) > 1]
        cross[gname] = dict(groups_spanning_patients=len(multi), images=int(sum(len(v) for v in multi)),
                            patients_involved=len({p_ for v in multi for p_ in pat_of[v].tolist()}),
                            groups_spanning_grades=sum(1 for v in multi if len(set(c5_of[v].tolist())) > 1),
                            examples=[sorted(set(pat_of[v].tolist())) for v in multi[:8]])
    n_pat_groups = len(set(ctrl[base].tolist()))
    Q.kv("images shared by patient folders", f"identical: {cross['identical']['groups_spanning_patients']} groups, "
         f"{cross['identical']['patients_involved']} folders · near duplicates: {cross['near_duplicate']['groups_spanning_patients']} "
         f"groups, {cross['near_duplicate']['patients_involved']} folders · {len(pid)} folders -> {n_pat_groups} split units")
    audit_x["orchid"].update(summ, exact_groups=n_ex_groups, exact_redundant=n_ex_red, near_pairs=int(rule.sum()),
                             cross_patient=cross, split_units=n_pat_groups,
                             pairs_phash_le8=int((ham <= 8).sum()), pairs_cos_ge97=int((cosv >= .97).sum()),
                             images_per_patient=dict(min=int(min(ppi.values())), median=float(np.median(list(ppi.values()))),
                                                     max=int(max(ppi.values()))), folders=dict(sub5))

    # ------------------------------------------------------------------ 4  cross-dataset near duplicates
    Q.sub("4  CROSS-DATASET NEAR DUPLICATES (ORCHID vs the five Q1 datasets)")
    xrows = []
    for b in Q.available_datasets():
        hb = np.load(Q.D["store"] / f"{b}_hash.npz")["ph"]
        Eb = np.load(Q.D["store"] / f"{b}_emb_{EMB}.npy").astype(np.float32)
        i1, j1, _ = Q.hamming_pairs(ph, hb, max_d=8)
        i2, j2, _ = Q.cos_pairs(E, Eb, thr=0.97)
        pr = set(zip(i1.tolist(), j1.tolist())) | set(zip(i2.tolist(), j2.tolist()))
        conf = 0
        if pr:
            pa = np.array(sorted(pr))
            conf = int((Q.ssim_pairs(st, pa[:, 0], Q.open_store(b), pa[:, 1], max_pairs=20000) >= 0.90).sum())
        xrows.append([f"orchid × {b}", len(i1), len(i2), conf])
        audit_x.setdefault("cross", {})[f"orchid|{b}"] = dict(phash_le8=int(len(i1)), cos_ge97=int(len(i2)), ssim_confirmed=conf)
    Q.table(["pair", "pHash<=8", "cos>=0.97", "confirmed (SSIM>=0.90)"], xrows, [24, 12, 12, 24])
    audit_x["_meta"] = dict(finished=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()), embed=EMB, near_dup=ND)
    Q.jsave(Q.D["index"] / "audit_ext.json", audit_x)
    print(f"\n[BLOCK 30 ✓]  ORCHID indexed ({summ['n_unique_originals']} images, {len(pid)} patients) · "
          f"{Q.hms(time.time() - t_all)} · run BLOCK 31 next")

In [ ]:
# ============================================================================
# BLOCK 31 — METADATA-FREE GROUP RECOVERY · PHIKON FEATURES · PROBE SWEEPS · A1 JOBS
# needs: BLOCK 15 + 29 (and 30 for ORCHID) in this session; DINOv2 embeddings and stores from BLOCK 16
# ============================================================================
import os, sys, json, time, math, importlib, warnings
from pathlib import Path
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("31")
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import adjusted_mutual_info_score, adjusted_rand_score, homogeneity_score, completeness_score
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from joblib import Parallel, delayed

t_all = time.time()
P = Q.plan()
if "amendment_A1" not in P:
    raise SystemExit("amendment A1 not found in the plan - run BLOCK 29 first")
RC = P["amendment_A1"]["recovery"]
BB0 = P["primary_backbone"]
EMB = (Q.jload(Q.D["index"] / "audit.json", {}) or {}).get("_meta", {}).get("embed", "dinov2_s")
DSETS = Q.available_datasets() + [d for d in Q.ext_datasets() if d not in Q.DATASETS]
TRUTH = [d for d in ("breakhis", "he_stained", "orchid") if d in DSETS]
GRID = RC["k_grid"]
SEEDS = [0, 1, 2]
Q.hdr("BLOCK 31 — GROUP RECOVERY · PHIKON · PROBE SWEEPS · A1 JOBS")
Q.kv("datasets", ", ".join(DSETS) + ("" if "orchid" in DSETS else "   (ORCHID not available: A1 runs without it)"))
Q.kv("ground truth for recovery", ", ".join(f"{d} ({Q.META[d]['group']})" for d in TRUTH))


def unit(x):
    x = np.asarray(x, np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)


# ------------------------------------------------------------------ 1  frozen features
Q.sub("1  FROZEN FEATURES  (DINOv2 from BLOCK 16/30 · Phikon pathology foundation model · CIELAB stain statistics)")
FE = {}
phikon_ok = True
for ds in DSETS:
    dd = Q.DS.get(ds)
    rows = np.nonzero(dd.ok & dd.orig)[0]
    t0 = time.time()
    d = dict(dinov2_s=np.load(Q.D["store"] / f"{ds}_emb_{EMB}.npy").astype(np.float32))
    fp_ = Q.D["store"] / f"{ds}_emb_phikon.npy"
    if phikon_ok and not fp_.exists():
        try:
            e = Q.embed(ds, rows, key="phikon", img=224, bs=128)
            full = np.zeros((dd.n, e.shape[1]), np.float16)
            full[rows] = e
            np.save(fp_, full)
        except Exception as ex:
            phikon_ok = False
            print(f"  Phikon unavailable ({type(ex).__name__}: {str(ex)[:90]}) -> DINOv2 feature sets only")
    if phikon_ok and fp_.exists():
        d["phikon"] = np.load(fp_).astype(np.float32)
    fs_ = Q.D["store"] / f"{ds}_stain.npy"
    if not fs_.exists():
        from skimage.color import rgb2lab
        st = Q.open_store(ds)
        sf = np.zeros((dd.n, 6), np.float32)
        for s in range(0, len(rows), 512):
            r = rows[s:s + 512]
            x = np.asarray(st[np.sort(r)])[:, ::4, ::4].astype(np.float32) / 255.0
            lab = rgb2lab(x)
            sf[np.sort(r)] = np.concatenate([lab.mean((1, 2)), lab.std((1, 2))], 1)
        np.save(fs_, sf)
    d["stain"] = np.load(fs_)
    FE[ds] = d
    Q.kv(ds, f"{len(rows):,} images · " + " · ".join(f"{k} {v.shape[1]}" for k, v in d.items()) + f" · {time.time() - t0:.0f}s")
FEATS = ["dinov2_s", "phikon", "phikon+stain"] if phikon_ok else ["dinov2_s", "dinov2_s+stain"]


def feature(ds, name, rows):
    d = FE[ds]
    if name.endswith("+stain"):
        b = unit(d[name.split("+")[0]][rows])
        s = d["stain"][rows]
        s = (s - s.mean(0)) / np.maximum(s.std(0), 1e-6) / math.sqrt(s.shape[1])
        return np.concatenate([b, s], 1)
    return unit(d[name][rows])


# ------------------------------------------------------------------ 2  Ward clustering, recovery quality, automatic k
Q.sub("2  RECOVERY QUALITY  (Ward clusters of frozen features vs true patients/slides; labels never used)")
REC = {}
for ds in DSETS:
    dd = Q.DS.get(ds)
    base = np.nonzero(dd.base)[0]
    nb = len(base)
    ks = [k for k in GRID if k <= nb / 3]
    for fn in FEATS:
        X = feature(ds, fn, base)
        Z = linkage(X.astype(np.float64), method="ward")
        labs = {k: fcluster(Z, t=k, criterion="maxclust") for k in ks}
        g = np.random.default_rng(0)
        sub = np.sort(g.choice(nb, min(nb, 3000), replace=False))
        sil = {k: float(silhouette_score(X[sub], labs[k][sub], metric="cosine"))
               for k in ks if 10 <= k <= max(10, nb / 10) and len(np.unique(labs[k][sub])) > 1}
        kauto = max(sil, key=sil.get) if sil else ks[0]
        q = {}
        if ds in TRUTH:
            tr = dd.g_struct[base]
            for k in ks:
                q[k] = dict(ami=float(adjusted_mutual_info_score(tr, labs[k])), ari=float(adjusted_rand_score(tr, labs[k])),
                            homogeneity=float(homogeneity_score(tr, labs[k])), completeness=float(completeness_score(tr, labs[k])))
        REC[(ds, fn)] = dict(ks=ks, labels=labs, sil=sil, kauto=int(kauto), quality=q, n=nb)
rows = []
for ds in TRUTH:
    for fn in FEATS:
        r = REC[(ds, fn)]
        q = r["quality"]
        ka = r["kauto"]
        rows.append([ds, fn, len(set(Q.DS.get(ds).g_struct[Q.DS.get(ds).base].tolist())), ka,
                     f"{q[ka]['ami']:.3f}", f"{q[ka]['ari']:.3f}", f"{q[ka]['homogeneity']:.3f}",
                     f"{q[ka]['completeness']:.3f}", f"{max(v['ami'] for v in q.values()):.3f}"])
Q.table(["dataset", "features", "true groups", "k auto", "AMI", "ARI", "homogeneity", "completeness", "best AMI any k"], rows,
        [12, 14, 12, 8, 7, 7, 13, 14, 15])
print("  homogeneity = each cluster holds one true group (what leakage-safe splitting needs);")
print("  completeness = each true group sits in one cluster. k auto = best cosine silhouette in [10, n/10].")
score = {fn: float(np.mean([REC[(ds, fn)]["quality"][REC[(ds, fn)]["kauto"]]["ami"] for ds in TRUTH])) if TRUTH else 0.0
         for fn in FEATS}
SEL = max(FEATS, key=lambda f: (score[f], -FEATS.index(f)))
Q.kv("selected feature set", f"{SEL}  (mean AMI at automatic k: " + ", ".join(f"{f} {score[f]:.3f}" for f in FEATS) + ")")

for ds in DSETS:
    dd = Q.DS.get(ds)
    base = np.nonzero(dd.base)[0]
    r = REC[(ds, SEL)]
    groups, ksel = {}, {}
    for name, k in (("k25", 25), ("k100", 100), ("kauto", r["kauto"])):
        if k not in r["labels"]:
            continue
        arr = np.full(dd.n, -1, np.int64)
        arr[base] = r["labels"][k]
        groups[name] = arr.tolist()
        ksel[name] = int(k)
    Q.jsave(Q.D["index"] / f"{ds}.recov.json", dict(feature=SEL, k=ksel, groups=groups,
                                                    created=time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())))
Q.kv("recovered groupings", " · ".join(f"{ds}: kauto={REC[(ds, SEL)]['kauto']}" for ds in DSETS))


# ------------------------------------------------------------------ 3  residual leakage of recovered groupings (simulation)
Q.sub("3  RESIDUAL LEAK SHARE  (5-fold splits of each grouping: share of test images whose TRUE group is in training)")


def leak_share(keys, y, truth, seed):
    keys, y = np.asarray(keys), np.asarray(y)
    if len(np.unique(keys)) == len(keys):
        sp = StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
    else:
        sp = StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y, keys)
    out = []
    for tr, te in sp:
        s = set(truth[tr].tolist())
        out.append(np.mean([t in s for t in truth[te]]))
    return float(np.mean(out))


LEAK = {}
rows = []
for ds in TRUTH:
    dd = Q.DS.get(ds)
    base = np.nonzero(dd.base)[0]
    y, tr_ = dd.y[base], dd.g_struct[base]
    near = dd.g_near[base]
    ref_img = float(np.mean([leak_share(np.arange(len(base)), y, tr_, s) for s in SEEDS]))
    for fn in FEATS:
        r = REC[(ds, fn)]
        curve = {}
        for k in r["ks"]:
            keys = Q.union_labels(len(base), label_arrays=[near, r["labels"][k]])
            curve[k] = float(np.mean([leak_share(keys, y, tr_, s) for s in SEEDS]))
        LEAK[f"{ds}|{fn}"] = dict(image_level=ref_img, true_groups=0.0, curve={str(k): v for k, v in curve.items()},
                                  kauto=r["kauto"])
        if fn == SEL:
            pick = [k for k in dict.fromkeys((25, 100, r["kauto"])) if k in curve]
            rows.append([ds, f"{100 * ref_img:.1f}", " · ".join(f"k={k}{' (auto)' if k == r['kauto'] else ''}: "
                                                                f"{100 * curve[k]:.1f}" for k in pick)])
Q.table(["dataset", "image-level split %", "recovered groupings, selected features (%)"], rows, [12, 20, 60])
print("  the true-group split has 0% by construction; lower = less patient/slide leakage.")


# ------------------------------------------------------------------ 4  frozen-probe sweep: inflation vs recovered-cluster size
Q.sub("4  PROBE SWEEP  (DINOv2 logistic regression + 1-NN; own fold vs clean holdout of every E1 outer fold)")


def dev_hold(dd, o, scheme):
    fold, _ = Q.outer_folds(dd, dict(scheme, rep=0))
    hold = np.nonzero(dd.base & (fold == o))[0]
    hn = set(dd.g_near[fold == o].tolist()) | set(dd.md5[fold == o].tolist()) | set(dd.parent_link[fold == o].tolist())
    purge = np.array([(dd.g_near[i] in hn or dd.md5[i] in hn or dd.parent_link[i] in hn) and fold[i] != o
                      for i in range(dd.n)])
    dev = np.nonzero(dd.base & (fold >= 0) & (fold != o) & ~purge)[0]
    return dev, hold


def bal_acc(y, p, K):
    return float(np.mean([np.mean(p[y == c] == c) for c in range(K) if (y == c).any()]))


def probe_eval(Xtr, ytr, Xev_list, K, seed):
    g = np.random.default_rng(seed)
    if len(ytr) > 6000:
        keep = np.sort(g.choice(len(ytr), 6000, replace=False))
        Xtr, ytr = Xtr[keep], ytr[keep]
    out = {}
    if len(np.unique(ytr)) < 2:
        return None
    sc = StandardScaler().fit(Xtr)
    lr = LogisticRegression(max_iter=3000, C=1.0).fit(sc.transform(Xtr), ytr)
    Xn = unit(Xtr)
    for name, (Xe, ye) in Xev_list.items():
        if len(ye) == 0:
            continue
        p_lr = lr.predict(sc.transform(Xe))
        p_nn = ytr[np.argmax(unit(Xe) @ Xn.T, 1)]
        out[name] = dict(logreg=bal_acc(ye, p_lr, K), nn=bal_acc(ye, p_nn, K))
    return out


def sweep_task(ds, o, grp, keys_dev, seed, dev, hold, Xd, y, K, truth):
    own_m = Q.sg_split(keys_dev, y[dev], 0.2, Q.shash("sweep", ds, o, grp, seed) % 2 ** 32)
    own, tr = dev[own_m], dev[~own_m]
    r = probe_eval(Xd[tr], y[tr], {"own": (Xd[own], y[own]), "hold": (Xd[hold], y[hold])}, K, seed)
    if r is None or "own" not in r or "hold" not in r:
        return None
    s = set(truth[tr].tolist())
    return dict(ds=ds, outer=int(o), group=grp, seed=seed, own_lr=r["own"]["logreg"], hold_lr=r["hold"]["logreg"],
                own_nn=r["own"]["nn"], hold_nn=r["hold"]["nn"], leak=float(np.mean([t in s for t in truth[own]])),
                n_groups=int(len(np.unique(keys_dev))), n_dev=int(len(dev)))


SWEEP = []
for ds in DSETS:
    dd = Q.DS.get(ds)
    scheme = P["outer"][ds]
    K = dd.K
    Xd = unit(FE[ds]["dinov2_s"])
    y = dd.y
    base = np.nonzero(dd.base)[0]
    r = REC[(ds, SEL)]
    truth = dd.g_struct
    keys_all = {"image": np.arange(dd.n),
                "audit": Q.union_labels(dd.n, label_arrays=[dd.g_near, dd.g_struct, dd.parent_link])}
    for k in r["ks"]:
        lab = np.full(dd.n, -1, np.int64)
        lab[base] = r["labels"][k]
        keys_all[f"k{k}"] = Q.union_labels(dd.n, label_arrays=[dd.g_near, lab, dd.parent_link])
    tasks = []
    for o in range(Q.n_outer(dd, dict(scheme, rep=0))):
        dev, hold = dev_hold(dd, o, scheme)
        if len(dev) < 20 or len(hold) == 0:
            continue
        for grp, ka in keys_all.items():
            for sd in SEEDS:
                tasks.append((ds, o, grp, ka[dev], sd, dev, hold, Xd, y, K, truth))
    t0 = time.time()
    res = Parallel(n_jobs=4, prefer="threads")(delayed(sweep_task)(*t) for t in tasks)
    res = [x for x in res if x]
    SWEEP.extend(res)
    by = defaultdict(list)
    for x in res:
        by[x["group"]].append(x)
    line = []
    for grp in ["image"] + [f"k{k}" for k in r["ks"]] + ["audit"]:
        if by.get(grp):
            v = by[grp]
            line.append(f"{grp} {100 * np.mean([a['own_lr'] - a['hold_lr'] for a in v]):+.1f}")
    Q.kv(f"{ds} ({len(res)} fits, {time.time() - t0:.0f}s)", "logreg inflation: " + " | ".join(line))
print("  image = random image-level split of the dev pool; kN = recovered clusters (selected features) + near duplicates;")
print("  audit = the grouping of BLOCK 16 (true patients/slides where they exist).")


# ------------------------------------------------------------------ 5  Phikon vs DINOv2 frozen probes on the E1 seed-0 splits (H12)
Q.sub("5  FOUNDATION-MODEL PROBES  (E1 seed-0 splits, leaky L vs controlled C; ORCHID uses its X1 splits)")
FMP = []
fm_keys = ["dinov2_s"] + (["phikon"] if phikon_ok else [])
for ds in DSETS:
    exp = "X1" if ds == "orchid" else "E1"
    if ds not in P["exps"][exp]["datasets"]:
        continue
    dd = Q.DS.get(ds)
    scheme = P["outer"][ds]
    dc = P["exps"][exp]["datasets"][ds]
    sib_cache = {k: {} for k in fm_keys}
    t0 = time.time()
    for o in range(Q.n_outer(dd, dict(scheme, rep=0))):
        for arm in ("L", "C"):
            spec = dict(exp=exp, ds=ds, arm=arm, outer=o, rep=0, seed=0, steps=dc["steps"], backbone=BB0, recipe="std",
                        dose=None, xeval=False, keep_weights=False, train_profile=None)
            try:
                B = Q.build_job(spec, check=False)
            except Exception as ex:
                print(f"    {ds} {arm} o{o}: {type(ex).__name__} {str(ex)[:80]}")
                continue
            pol = B["info"]["policy"]
            for fk in fm_keys:
                base_f = FE[ds]["dinov2_s"] if fk == "dinov2_s" else FE[ds]["phikon"]

                def get(u):
                    rows_, cps = u["row"], u["copy"]
                    X = base_f[rows_].copy()
                    need = [(int(a), int(c)) for a, c in zip(rows_, cps) if c > 0 and (int(a), int(c)) not in sib_cache[fk]]
                    if need:
                        nr = np.array([a for a, _ in need])
                        nc = np.array([c for _, c in need])
                        e = Q.embed(ds, nr, key=fk, img=224, bs=128, copies=nc, policy=pol)
                        for (a, c), v in zip(need, e):
                            sib_cache[fk][(a, c)] = v
                    m = cps > 0
                    if m.any():
                        X[m] = np.stack([sib_cache[fk][(int(a), int(c))] for a, c in zip(rows_[m], cps[m])])
                    return X

                Xtr = get(B["train"])
                r = probe_eval(Xtr, B["train"]["y"], {"own": (get(B["own"]), B["own"]["y"]),
                                                      "hold": (get(B["hold"]), B["hold"]["y"])}, dd.K, o)
                if r:
                    FMP.append(dict(ds=ds, outer=o, arm=arm, feat=fk, own_lr=r["own"]["logreg"], hold_lr=r["hold"]["logreg"],
                                    own_nn=r["own"]["nn"], hold_nn=r["hold"]["nn"]))
    rows = []
    for fk in fm_keys:
        line = [ds, fk]
        for arm in ("L", "C"):
            v = [x for x in FMP if x["ds"] == ds and x["feat"] == fk and x["arm"] == arm]
            line += [f"{100 * np.mean([x['own_lr'] for x in v]):.1f}/{100 * np.mean([x['hold_lr'] for x in v]):.1f}"
                     if v else "—",
                     f"{100 * np.mean([x['own_lr'] - x['hold_lr'] for x in v]):+.1f}" if v else "—",
                     f"{100 * np.mean([x['own_nn'] - x['hold_nn'] for x in v]):+.1f}" if v else "—"]
        rows.append(line)
    Q.table(["dataset", "features", "L own/hold", "L infl LR", "L infl 1NN", "C own/hold", "C infl LR", "C infl 1NN"], rows,
            [12, 10, 13, 10, 11, 13, 10, 11])
Q.jsave(Q.D["eval"] / "recovery.json", dict(
    selected=SEL, features=FEATS, phikon=phikon_ok, score=score,
    quality={f"{ds}|{fn}": dict(n=r["n"], kauto=r["kauto"], sil={str(k): v for k, v in r["sil"].items()},
                                quality={str(k): v for k, v in r["quality"].items()}, ks=r["ks"])
             for (ds, fn), r in REC.items()},
    leak=LEAK, sweep=SWEEP, fm_probes=FMP))


# ------------------------------------------------------------------ 6  A1 jobs (built + tested for disjointness, appended to jobs.json)
Q.sub("6  A1 TRAINING JOBS  (every split built and checked; appended to jobs/jobs.json)")
speed = Q.merged_speed()
order = {p: i for i, p in enumerate(P["phases"])}
new, bad, skipped = [], [], []
for exp in ("X1", "X1RL", "X2", "X2RL"):
    ex = P["exps"][exp]
    for ds, dc in ex["datasets"].items():
        if ds not in DSETS:
            continue
        dd = Q.DS.get(ds)
        scheme = P["outer"][ds]
        outs = ex.get("outer") or list(range(Q.n_outer(dd, dict(scheme, rep=0))))
        for o in outs:
            for arm in dc["train"]:
                rv = dc["arms"][arm].get("recov")
                if rv and rv not in dd.extra:
                    if o == outs[0]:
                        skipped.append(f"{exp} {ds} {arm}: grouping {rv} not available (k > n/3)")
                    continue
                seeds = ex.get("seeds_recovered", ex["seeds"]) if arm.startswith("R") else ex["seeds"]
                for seed in seeds:
                    s = dict(exp=exp, ds=ds, arm=arm, outer=int(o), rep=0, seed=int(seed), steps=dc["steps"], backbone=BB0,
                             recipe="std", dose=None, xeval=False, keep_weights=False, train_profile=None)
                    jid = Q.job_id(s)
                    try:
                        B = Q.build_job(s, check=True)
                    except AssertionError as e:
                        bad.append((jid, "DISJOINTNESS FAILED: " + str(e)))
                        continue
                    except Exception as e:
                        bad.append((jid, f"{type(e).__name__}: {e}"))
                        continue
                    info = B["info"]
                    info["n_xeval"] = 0
                    R = P["recipes"]["std"]
                    est = Q.estimate_minutes(s, info, R, speed)
                    steps, _ = Q.recipe_steps(s, R, info["sizes"]["train"])
                    phase = "orchid" if exp.startswith("X1") else "recover"
                    prio = order[phase] * 1_000_000 + (0 if arm in ("L", "C") else 100_000) + seed * 10_000 + o * 1000 + int(est * 10)
                    new.append(dict(id=jid, fp=Q.job_fp(s, P), phase=phase, priority=prio, est_min=round(est, 2), steps=steps,
                                    sizes=info["sizes"], leak=info["leak"], spec=s))
new.sort(key=lambda j: j["priority"])
J = Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}
ids = {j["id"] for j in new}
J["jobs"] = [j for j in J.get("jobs", []) if j["id"] not in ids] + new
Q.jsave(Q.D["jobs"] / "jobs.json", J)
Q.jsave(Q.D["jobs"] / "jobs_A1.json", dict(created=time.time(), jobs=new, rejected=bad))
rows = []
for (ph, exp, ds, arm), v in sorted(Counter((j["phase"], j["spec"]["exp"], j["spec"]["ds"], j["spec"]["arm"]) for j in new).items()):
    J_ = [j for j in new if (j["phase"], j["spec"]["exp"], j["spec"]["ds"], j["spec"]["arm"]) == (ph, exp, ds, arm)]
    rows.append([ph, exp, ds, arm, v, f"{np.mean([j['sizes']['train'] for j in J_]):.0f}",
                 f"{100 * np.mean([j['leak'].get('struct_group', 0) for j in J_]):.1f}", f"{sum(j['est_min'] for j in J_) / 60:.2f}"])
Q.table(["phase", "exp", "dataset", "arm", "jobs", "train n", "own share with true group-mate %", "GPU-h"], rows,
        [9, 6, 12, 7, 6, 9, 32, 7])
for jid, msg in bad[:10]:
    print(f"    ✗ {jid}: {msg[:120]}")
for msg in skipped:
    print(f"    skipped: {msg}")
Q.kv("A1 jobs", f"{len(new)} built · {len(bad)} rejected · {sum(j['est_min'] for j in new) / 60:.1f} GPU-h "
     f"(~{sum(j['est_min'] for j in new) / 120:.1f} h on 2 GPUs)")
print(f"\n[BLOCK 31 ✓]  recovery + probes in q1/eval/recovery.json · {Q.hms(time.time() - t_all)} · "
      f"run BLOCK 19 (if not yet run in this session), then BLOCK 32")

In [ ]:
# ============================================================================
# BLOCK 32 — TRAINING · AMENDMENT A1: ORCHID (L / C / recovered-group arms, random labels) + RECOVERED-GROUP ARMS ON FIVE DATASETS
# (needs BLOCKS 15 -> 19 in this session and BLOCK 31 run once; re-run to resume; press Stop to pause safely)
# ============================================================================
RUN("orchid,recover")

In [ ]:
# ============================================================================
# BLOCK 25 — EVALUATION + STATISTICS (all numbers for the rewritten Results section)
# ============================================================================
import os, sys, json, time, math, importlib, warnings, csv
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("25")

NB = 10000          # bootstrap resamples (pre-specified)
t_all = time.time()
P = Q.plan()
BB0 = P["primary_backbone"]
rng = np.random.default_rng(20260930)
Q.hdr("BLOCK 25 — EVALUATION + STATISTICS")
jobs = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
_A1 = {j["id"] for j in (Q.jload(Q.D["jobs"] / "jobs_A1.json", {}) or {}).get("jobs", [])}
jobs = [j for j in jobs if j["id"] not in _A1]          # amendment-A1 runs are analysed in BLOCK 33, never here
RUNS = {}
for j in jobs:
    if Q.job_done(j):
        r, z = Q.load_run(j["id"])
        if r is not None:
            RUNS[j["id"]] = (j, r, {k: z[k] for k in z.files})
Q.kv("finished runs", f"{len(RUNS)} of {len(jobs)} jobs")
if not RUNS:
    raise SystemExit("no finished runs yet")
OUT = dict(meta=dict(n_runs=len(RUNS), n_jobs=len(jobs), boot=NB, plan_sha=(Q.jload(Q.D["plan"] / "plan.sha256.json") or {}).get("sha256")))
TAB = Q.D["tables"]


def write_csv(name, header, rows):
    with open(TAB / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


def sel(**kw):
    out = []
    for j, r, z in RUNS.values():
        s = j["spec"]
        if all(s.get(k) == v for k, v in kw.items()):
            out.append((j, r, z))
    return out


def correct(z, name):
    return (z[f"{name}_logit"].argmax(1) == z[f"{name}_y"]).astype(float)


def run_metrics(j, r, z):
    """Per-run metrics with the temperature and thresholds fitted on the inner validation split only."""
    K = r["info"]["K"]
    pos = Q.META[j["spec"]["ds"]]["pos"]
    T = Q.fit_temperature(z["ival_logit"], z["ival_y"])
    out = dict(T=T)
    for nm in ("own", "hold"):
        lg, y = z[f"{nm}_logit"], z[f"{nm}_y"]
        if len(y) == 0:
            continue
        out[nm] = Q.metrics(Q.softmax(lg, T), y, K, pos)
        out[nm + "_raw_ece"] = Q.ece_score(Q.softmax(lg), y)
        out[nm + "_raw_brier"] = float(np.mean(((Q.softmax(lg) - np.eye(K)[y]) ** 2).sum(1)))
    m = z["own_copy"] == 0
    if m.any() and (~m).any():
        out["own_orig"] = Q.metrics(Q.softmax(z["own_logit"][m], T), z["own_y"][m], K, pos)
    pi = Q.softmax(z["ival_logit"], T)[:, pos]
    yi = (z["ival_y"] == pos).astype(int)
    th = Q.thresholds_from(pi, yi)
    ph = Q.softmax(z["hold_logit"], T)[:, pos]
    yh = (z["hold_y"] == pos).astype(int)
    out["op"] = {k: Q.op_point(ph, yh, v) for k, v in th.items()}
    return out


RM = {rid: run_metrics(j, r, z) for rid, (j, r, z) in RUNS.items()}


def fold_key(s):
    return (s["outer"], s.get("rep", 0))


def tables_for(runs, part):
    """Per outer fold: cluster count tables summed over seeds (and the list of seeds)."""
    by = defaultdict(list)
    for j, r, z in runs:
        by[fold_key(j["spec"])].append((j, r, z))
    out = {}
    for fk, lst in by.items():
        ys, cs, ks = [], [], []
        for j, r, z in lst:
            ys.append(z[f"{part}_y"])
            cs.append(correct(z, part))
            ks.append(z[f"{part}_key"])
        K = lst[0][1]["info"]["K"]
        out[fk] = Q.cluster_table(np.concatenate(ys), np.concatenate(cs), np.concatenate(ks), K)
    return out


def paired_hold(runsA, runsB):
    """Per fold: shared cluster table n and correct counts of A and B on the same holdout images."""
    A, Bm = defaultdict(list), defaultdict(list)
    for j, r, z in runsA:
        A[fold_key(j["spec"])].append(z)
    for j, r, z in runsB:
        Bm[fold_key(j["spec"])].append(z)
    out = {}
    for fk in set(A) & set(Bm):
        za = {int(z0["hold_row"][0]) if len(z0["hold_row"]) else 0: z0 for z0 in A[fk]}
        y = A[fk][0]["hold_y"]
        key = A[fk][0]["hold_key"]
        K = int(max(y.max(), 1) + 1) if len(y) else 2
        ca = np.mean([correct(z, "hold") for z in A[fk]], 0)
        cb = np.mean([correct(z, "hold") for z in Bm[fk]], 0)
        n, c1 = Q.cluster_table(y, ca, key, K)
        _, c2 = Q.cluster_table(y, cb, key, K)
        out[fk] = (n, c1, c2)
    return out


def boot_folds(tabs, B, seed):
    """Within-fold cluster bootstrap: returns fold keys and matrix [F, B] of BA."""
    g = np.random.default_rng(seed)
    ks = sorted(tabs)
    return ks, np.stack([Q.boot_ba(tabs[k][0], tabs[k][1], B, g) for k in ks])


def point(tabs):
    return {k: float(Q.ba_counts(*tabs[k])) for k in tabs}


def two_stage(mat, B, seed):
    g = np.random.default_rng(seed)
    F = mat.shape[0]
    idx = g.integers(0, F, size=(B, F))
    return mat[idx, np.arange(B)[:, None]].mean(1)


def fmt(v, ci_=None, d=2):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    s = f"{100 * v:+.{d}f}" if ci_ is not None else f"{100 * v:.{d}f}"
    if ci_ is not None:
        s += f" [{100 * ci_[0]:+.{d}f}, {100 * ci_[1]:+.{d}f}]"
    return s


def pf(p):
    return "—" if p is None or not np.isfinite(p) else (f"{p:.1e}" if p < 1e-3 else f"{p:.4f}")


# ============================================================================ 1  E1 main experiment
Q.sub("1  E1 — OWN-FOLD vs CLEAN-HOLDOUT BALANCED ACCURACY  (primary backbone, all seeds, pooled over outer folds)")
E1 = {}
rows, csvrows = [], []
avail = [d for d in Q.DATASETS if sel(exp="E1", ds=d, backbone=BB0, recipe="std")]
for ds in avail:
    arms = P["exps"]["E1"]["datasets"][ds]["arms"]
    res = {}
    for arm in arms:
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std")
        if not R_:
            continue
        to, th = tables_for(R_, "own"), tables_for(R_, "hold")
        common = sorted(set(to) & set(th))
        to, th = {k: to[k] for k in common}, {k: th[k] for k in common}
        ko, mo = boot_folds(to, NB, Q.shash(ds, arm, "own") % 2 ** 32)
        kh, mh = boot_folds(th, NB, Q.shash(ds, arm, "hold") % 2 ** 32)
        po, ph_ = point(to), point(th)
        infl_pt = float(np.mean([po[k] - ph_[k] for k in common]))
        infl_w = (mo - mh).mean(0)                                        # within-fold (sensitivity)
        infl_b = Q.hier([mo - mh], NB, Q.shash(ds, arm, "hier") % 2 ** 32)[0]   # two-stage (primary)
        seeds = sorted({j["spec"]["seed"] for j, _, _ in R_})
        per_seed = []
        for sd in seeds:
            Rs = [x for x in R_ if x[0]["spec"]["seed"] == sd]
            a, b = point(tables_for(Rs, "own")), point(tables_for(Rs, "hold"))
            kk = set(a) & set(b)
            if kk:
                per_seed.append(float(np.mean([a[k] - b[k] for k in kk])))
        vals, outs, sds = [], [], []
        for j, r, z in R_:
            m = RM[j["id"]]
            if "own" in m and "hold" in m:
                vals.append(m["own"]["bal_acc"] - m["hold"]["bal_acc"])
                outs.append(f"{j['spec']['outer']}-{j['spec'].get('rep', 0)}")
                sds.append(j["spec"]["seed"])
        mix = Q.mixed_intercept(vals, outs, sds) if len(vals) >= 4 else None
        mean_m = lambda part, key, sub=None: float(np.nanmean([(RM[j["id"]][part][sub][key] if sub else RM[j["id"]][part][key])
                                                               for j, _, _ in R_ if part in RM[j["id"]] and (sub is None or sub in RM[j["id"]][part])
                                                               and (RM[j["id"]][part][sub] if sub else RM[j["id"]][part]).get(key) is not None])) \
            if R_ else float("nan")
        res[arm] = dict(n_runs=len(R_), folds=len(common), seeds=seeds, own=float(np.mean(list(po.values()))),
                        hold=float(np.mean(list(ph_.values()))), infl=infl_pt, infl_ci=Q.ci(infl_b), infl_p=Q.boot_p(infl_b),
                        infl_ci_within=Q.ci(infl_w), infl_p_within=Q.boot_p(infl_w), seed_infl=per_seed,
                        seed_sd=float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else None, mixed=mix,
                        own_acc=mean_m("own", "acc"), hold_acc=mean_m("hold", "acc"), own_auroc=mean_m("own", "auroc"),
                        hold_auroc=mean_m("hold", "auroc"), own_f1=mean_m("own", "macro_f1"), hold_f1=mean_m("hold", "macro_f1"),
                        hold_ece=mean_m("hold", "ece"), hold_brier=mean_m("hold", "brier"),
                        own_pos_sens=mean_m("own", "sens", "pos"), own_pos_spec=mean_m("own", "spec", "pos"),
                        own_pos_prec=mean_m("own", "prec", "pos"), own_pos_f1=mean_m("own", "f1", "pos"),
                        hold_pos_sens=mean_m("hold", "sens", "pos"), hold_pos_spec=mean_m("hold", "spec", "pos"),
                        hold_pos_prec=mean_m("hold", "prec", "pos"), hold_pos_f1=mean_m("hold", "f1", "pos"),
                        own_orig=float(np.nanmean([RM[j["id"]]["own_orig"]["bal_acc"] for j, _, _ in R_ if "own_orig" in RM[j["id"]]]))
                        if any("own_orig" in RM[j["id"]] for j, _, _ in R_) else None,
                        _mo=mo, _mh=mh, _k=common)
    if "L" in res and "C" in res:
        L, C = res["L"], res["C"]
        kk = sorted(set(L["_k"]) & set(C["_k"]))
        iL = [L["_k"].index(k) for k in kk]
        iC = [C["_k"].index(k) for k in kk]
        did_b = Q.hier([(L["_mo"] - L["_mh"])[iL] - (C["_mo"] - C["_mh"])[iC]], NB, Q.shash(ds, "did") % 2 ** 32)[0]
        ph_tab = paired_hold(sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std"),
                             sel(exp="E1", ds=ds, arm="C", backbone=BB0, recipe="std"))
        g = np.random.default_rng(Q.shash(ds, "paired") % 2 ** 32)
        dmat = np.stack([Q.boot_ba_paired(ph_tab[k][0], ph_tab[k][1], ph_tab[k][2], NB, g) for k in sorted(ph_tab)])
        dh = Q.hier([dmat], NB, Q.shash(ds, "dh") % 2 ** 32)[0]
        dh_pt = float(np.mean([Q.ba_counts(ph_tab[k][0], ph_tab[k][1]) - Q.ba_counts(ph_tab[k][0], ph_tab[k][2]) for k in ph_tab]))
        ca, cb, cl = [], [], []
        for fk in sorted(ph_tab):
            za = [z for j, r, z in sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std") if fold_key(j["spec"]) == fk]
            zb = [z for j, r, z in sel(exp="E1", ds=ds, arm="C", backbone=BB0, recipe="std") if fold_key(j["spec"]) == fk]
            for x, y_ in zip(za, zb):
                ca.append(correct(x, "hold"))
                cb.append(correct(y_, "hold"))
                cl.append(x["hold_key"] + 10 ** 7 * fk[1])
        gee = Q.gee_paired(np.concatenate(ca), np.concatenate(cb), np.concatenate(cl)) if ca else {}
        E1[ds] = dict(arms={a: {k: v for k, v in d.items() if not k.startswith("_")} for a, d in res.items()},
                      did=float(L["infl"] - C["infl"]), did_ci=Q.ci(did_b), did_p=Q.boot_p(did_b),
                      dhold=dh_pt, dhold_ci=Q.ci(dh), dhold_p=Q.boot_p(dh), gee=gee)
    else:
        E1[ds] = dict(arms={a: {k: v for k, v in d.items() if not k.startswith("_")} for a, d in res.items()})
    for arm, d in res.items():
        rows.append([ds, arm, f"{d['n_runs']}/{d['folds']}", fmt(d["own"]), fmt(d["hold"]),
                     fmt(d["infl"], d["infl_ci"]), pf(d["infl_p"]),
                     "—" if d["seed_sd"] is None else f"{100 * d['seed_sd']:.2f}"])
        csvrows.append([ds, arm, d["n_runs"], d["folds"], d["own"], d["hold"], d["infl"], *d["infl_ci"], d["infl_p"],
                        *d["infl_ci_within"], d["seed_sd"], d["own_acc"], d["hold_acc"], d["own_auroc"], d["hold_auroc"],
                        d["own_f1"], d["hold_f1"], d["hold_ece"], d["hold_brier"], d["own_pos_sens"], d["own_pos_spec"],
                        d["own_pos_prec"], d["own_pos_f1"], d["hold_pos_sens"], d["hold_pos_spec"], d["hold_pos_prec"],
                        d["hold_pos_f1"], d["own_orig"]])
Q.table(["dataset", "arm", "runs/folds", "own BA", "hold BA", "inflation [95% CI, two-stage]", "p", "seed SD"], rows,
        [12, 6, 11, 9, 9, 30, 10, 9])
write_csv("E1_main", ["dataset", "arm", "runs", "folds", "own_ba", "hold_ba", "inflation", "ci_lo_2stage", "ci_hi_2stage",
                      "p_2stage", "ci_lo_within", "ci_hi_within", "seed_sd", "own_acc", "hold_acc", "own_auroc", "hold_auroc", "own_macro_f1", "hold_macro_f1",
                      "hold_ece_cal", "hold_brier_cal", "own_pos_sens", "own_pos_spec", "own_pos_prec", "own_pos_f1",
                      "hold_pos_sens", "hold_pos_spec", "hold_pos_prec", "hold_pos_f1", "own_ba_originals_only"], csvrows)

# primary family: H1 with Holm
fam = [(ds, E1[ds]["arms"]["L"]["infl_p"]) for ds in avail if "L" in E1[ds]["arms"]]
if fam:
    adj = Q.holm([p for _, p in fam])
    Q.sub("2  PRIMARY FAMILY (H1): leaky-arm inflation > 0, Holm-adjusted across datasets")
    rows = []
    for (ds, p), pa in zip(fam, adj):
        a = E1[ds]["arms"]["L"]
        E1[ds]["arms"]["L"]["infl_p_holm"] = pa
        mx = a.get("mixed") or {}
        mixs = (f"{100 * mx['est']:+.2f} [{100 * mx['lo']:+.2f},{100 * mx['hi']:+.2f}]" + ("*" if mx.get("note") else "")
                if mx.get("lo") is not None else "—")
        rows.append([ds, fmt(a["infl"], a["infl_ci"]), fmt(a["infl"], a["infl_ci_within"]), mixs, pf(p), pf(pa),
                     "supported" if pa < 0.05 and a["infl"] > 0 else "not supported"])
    Q.table(["dataset", "inflation L [two-stage CI]", "[within-fold CI]", "mixed model [CI]", "p", "p Holm", "H1"], rows,
            [12, 28, 28, 24, 9, 9, 14])
    print("  p and Holm use the two-stage bootstrap (primary). * = mixed model not estimable (one run per fold, singular fit")
    print("  or no convergence), so a t-interval over fold-level means is shown instead.")

Q.sub("3  SECONDARY (H2-H4): difference-in-differences, controlled-arm bias, holdout L - C")
rows = []
for ds in avail:
    e = E1[ds]
    if "did" not in e:
        continue
    C = e["arms"]["C"]
    gee = e.get("gee", {})
    rows.append([ds, fmt(e["did"], e["did_ci"]), fmt(C["infl"], C["infl_ci"]), fmt(e["dhold"], e["dhold_ci"]), pf(e["dhold_p"]),
                 f"{gee.get('or_', float('nan')):.2f} [{gee.get('lo', float('nan')):.2f},{gee.get('hi', float('nan')):.2f}] p {pf(gee.get('p'))}"
                 if "or_" in gee else "—"])
Q.table(["dataset", "DiD (H2) [CI]", "bias of C (H3) [CI]", "hold L - C (H4) [CI]", "p", "GEE odds ratio L vs C"], rows,
        [12, 28, 28, 28, 9, 30])
dh_p = [(ds, E1[ds]["dhold_p"]) for ds in avail if "dhold_p" in E1[ds]]
if dh_p:
    for (ds, _), pa in zip(dh_p, Q.holm([p for _, p in dh_p])):
        E1[ds]["dhold_p_holm"] = pa

Q.sub("4  WHAT A PAPER WOULD REPORT (own fold, leaky arm) vs THE TRUTH (clean holdout) — full metric panel")
rows = []
for ds in avail:
    for arm in ("L", "C"):
        a = E1[ds]["arms"].get(arm)
        if not a:
            continue
        rows.append([ds, arm, fmt(a["own_acc"]), fmt(a["hold_acc"]), f"{a['own_auroc']:.3f}/{a['hold_auroc']:.3f}",
                     f"{fmt(a['own_pos_sens'])}/{fmt(a['hold_pos_sens'])}", f"{fmt(a['own_pos_spec'])}/{fmt(a['hold_pos_spec'])}",
                     f"{fmt(a['own_pos_f1'])}/{fmt(a['hold_pos_f1'])}", f"{a['hold_ece']:.3f}"])
Q.table(["dataset", "arm", "own acc", "hold acc", "AUROC own/hold", "pos sens own/hold", "pos spec own/hold",
         "pos F1 own/hold", "ECE hold"], rows, [12, 5, 9, 9, 16, 18, 18, 16, 9])
print("  'pos' = positive class one-vs-rest (tumour / OSCC / cancer / malignant); macro metrics are in tables/E1_main.csv.")
print("  temperature and ECE: temperature fitted on each run's inner-validation split, applied to the holdout.")

# ICC / design effect and OCDC per slide
Q.sub("5  CLUSTERING OF ERRORS  (ICC of holdout correctness within groups, L arm, seed 0)")
rows = []
for ds in avail:
    R_ = sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std", seed=0)
    if not R_:
        continue
    c = np.concatenate([correct(z, "hold") for _, _, z in R_])
    k = np.concatenate([z["hold_key"] + 10 ** 7 * j["spec"].get("rep", 0) for j, _, z in R_])
    ic = Q.icc1(c, k)
    cnt = np.unique(k, return_counts=True)[1]
    multi = int((cnt >= 2).sum())
    share = float(cnt[cnt >= 2].sum() / max(1, cnt.sum()))
    ic.update(clusters_with_2plus=multi, share_in_multi=share)
    if multi < 2 or share < 0.2:         # ICC needs repeated observations within clusters
        ic.update(icc=None, deff=None)
    E1[ds]["icc"] = ic
    rows.append([ds, "—" if ic["icc"] is None else f"{ic['icc']:.3f}", "—" if ic["deff"] is None else f"{ic['deff']:.2f}",
                 ic["clusters"], multi, f"{ic['mean_size']:.1f}"])
Q.table(["dataset", "ICC", "design effect", "clusters", "with >= 2 images", "mean size"], rows, [12, 8, 15, 10, 18, 10])
print("  a design effect > 1 means image-level tests (e.g. McNemar) overstate the evidence by that factor in variance.")
print("  '—' = not estimable: fewer than 20% of holdout images share a cluster with another image (nearly all singletons).")

if "he_stained" in avail:
    Q.sub("6  OCDC LEAVE-ONE-SLIDE-OUT: per held-out slide (seed-averaged) + exact slide-level sign-flip tests")
    rows, dL, dH = [], [], []
    for o in sorted({j["spec"]["outer"] for j, _, _ in sel(exp="E1", ds="he_stained", backbone=BB0, recipe="std")}):
        v = {}
        for arm in ("L", "C"):
            R_ = sel(exp="E1", ds="he_stained", arm=arm, outer=o, backbone=BB0, recipe="std")
            if R_:
                v[arm] = (np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_]),
                          np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_]))
        if len(v) == 2:
            dd = Q.DS.get("he_stained")
            slide = sorted(set(dd.slide[dd.base]))[o]
            dL.append(v["L"][0] - v["L"][1])
            dH.append(v["L"][1] - v["C"][1])
            rows.append([slide, fmt(v["L"][0]), fmt(v["L"][1]), fmt(v["C"][0]), fmt(v["C"][1]), fmt(v["L"][0] - v["L"][1])])
    Q.table(["held-out slide", "L own", "L hold", "C own", "C hold", "L inflation"], rows, [18, 9, 9, 9, 9, 12])
    if dL:
        pL, pH = Q.sign_flip_p(dL), Q.sign_flip_p(dH)
        E1["he_stained"]["slide_level"] = dict(inflation=dL, dhold=dH, p_inflation=pL, p_dhold=pH)
        Q.kv("sign-flip p (L inflation)", f"{pf(pL)}  over {len(dL)} slides (smallest possible two-sided p = {2 / 2 ** len(dL):.4f})")
        Q.kv("sign-flip p (hold L - C)", pf(pH))
OUT["E1"] = E1

# factorial (clinical) and grouping sensitivity (NDB)
Q.sub("7  FACTORIAL (clinical) AND GROUPING SENSITIVITY (NDB-UFES coarse blocks), seed 0")
rows = []
for ds in avail:
    ex = P["exps"]["E1"]["datasets"][ds]
    for arm in ex.get("extra", []):
        a = E1[ds]["arms"].get(arm)
        if a:
            f_ = ex["arms"][arm]
            rows.append([ds, arm, f"S={f_.get('S', 0)} Dup={f_.get('Dup', 0)} Grp={f_.get('Grp', 0)} coarse={f_.get('coarse', 0)}",
                         fmt(a["own"]), fmt(a["hold"]), fmt(a["infl"], a["infl_ci"])])
if rows:
    Q.table(["dataset", "arm", "factors", "own BA", "hold BA", "inflation [CI]"], rows, [12, 6, 30, 9, 9, 30])
else:
    print("  no finished runs yet — these arms are trained in the 'general' phase (BLOCK 24)")
if "clinical" in avail and all(a in E1["clinical"]["arms"] for a in ("L", "C", "S0D1", "S1D0")):
    A = E1["clinical"]["arms"]
    fx = dict(S=0.5 * ((A["L"]["infl"] - A["S0D1"]["infl"]) + (A["S1D0"]["infl"] - A["C"]["infl"])),
              Dup=0.5 * ((A["L"]["infl"] - A["S1D0"]["infl"]) + (A["S0D1"]["infl"] - A["C"]["infl"])),
              SxDup=(A["L"]["infl"] - A["S0D1"]["infl"]) - (A["S1D0"]["infl"] - A["C"]["infl"]))
    OUT["factorial_clinical"] = fx
    Q.kv("main effect of augment-before-split", fmt(fx["S"]) + " points of inflation (clinical, seed 0)")
    Q.kv("main effect of keeping duplicates", fmt(fx["Dup"]) + " points")
    Q.kv("interaction S x Dup", fmt(fx["SxDup"]) + " points (exploratory, no CI: one seed per cell)")

# ============================================================================ 8  published protocol comparison
Q.sub("8  L ARM = PUBLISHED PROTOCOL: our replication numbers next to the published ones (definitions stated)")
PUB = {"histopath": [("Abd El-Aziz 2025", dict(acc=99.84, spec=99.92, prec=97.94, sens=97.93, f1=97.93), "not stated")],
       "ndb_ufes": [("SciRep 2025", dict(acc=97.16, prec=97.16, sens=97.16, f1=97.16, spec=98.58), "multi-class, averaging not stated")],
       "clinical": [("Liu & Bagi 2025", dict(acc=99.54, sens=95.73, spec=96.21, prec=96.34, f1=96.03, auroc=60.26), "not stated"),
                    ("Alzahrani 2025", dict(acc=98.47, sens=97.73, spec=97.73, f1=98.27), "not stated")]}
rows = []
for ds, lst in PUB.items():
    a = E1.get(ds, {}).get("arms", {}).get("L")
    c = E1.get(ds, {}).get("arms", {}).get("C")
    if not a:
        continue
    for name, vals, dfn in lst:
        ours = dict(acc=a["own_acc"], sens=a["own_pos_sens"], spec=a["own_pos_spec"], prec=a["own_pos_prec"],
                    f1=a["own_pos_f1"], auroc=a["own_auroc"])
        truth = dict(acc=c["hold_acc"], sens=c["hold_pos_sens"], spec=c["hold_pos_spec"], prec=c["hold_pos_prec"],
                     f1=c["hold_pos_f1"], auroc=c["hold_auroc"]) if c else {}
        for k, v in vals.items():
            rows.append([ds, name, k, f"{v:.2f}", fmt(ours.get(k)), fmt(truth.get(k)), dfn])
Q.table(["dataset", "paper", "metric", "published", "ours, L own fold", "ours, clean holdout", "published definition"],
        rows, [12, 18, 7, 11, 17, 20, 30])
print("  ours = positive-class (one-vs-rest) values; for NDB-UFES the positive class is OSCC. AUROC shown x100.")
write_csv("published_vs_ours", ["dataset", "paper", "metric", "published", "ours_L_own", "ours_clean_holdout", "definition"], rows)

# ============================================================================ 9  PRE, RL
_nh = Q.DS.get("histopath").n if "histopath" in avail else 0


def rl_ceiling(j, z):
    """BA reachable on the holdout by a rule that sees only the TRUE class: each true class is mapped to the random
    label it most often carries in the development pool.  Random labels are drawn per group, so with few groups they
    can correlate with the true classes by chance; this ceiling shows how much of an above-chance score that explains."""
    dd = Q.DS.get(j["spec"]["ds"])
    y_rl = Q._labels(dd, j["spec"])
    hold = z["hold_row"]
    dev = np.setdiff1d(np.nonzero(dd.base)[0], hold)
    K = dd.K
    f = {c: int(np.bincount(y_rl[dev][dd.y[dev] == c], minlength=K).argmax()) if (dd.y[dev] == c).any() else 0
         for c in range(K)}
    pred = np.array([f[int(c)] for c in dd.y[hold]])
    yt = y_rl[hold]
    return float(np.mean([np.mean(pred[yt == k] == k) for k in range(K) if (yt == k).any()]))


for ex_name, title in (("PRE", f"9  PRE-AUGMENTED CORPUS ({_nh:,} files as distributed): L = random file split, C = grouped by recovered parent"),
                       ("RL", "10 RANDOM-LABEL CONTROL (H6): labels drawn per group; chance = 1/K")):
    Rx = sel(exp=ex_name, backbone=BB0, recipe="std")
    if not Rx:
        continue
    Q.sub(title)
    rows = []
    blk = {}
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for arm in ("L", "C"):
            R_ = [x for x in Rx if x[0]["spec"]["ds"] == ds and x[0]["spec"]["arm"] == arm]
            if not R_:
                continue
            to, th = tables_for(R_, "own"), tables_for(R_, "hold")
            _, mo = boot_folds(to, NB, 1)
            _, mh = boot_folds(th, NB, 2)
            own_b = mo.mean(0)
            d = dict(own=float(np.mean(list(point(to).values()))), hold=float(np.mean(list(point(th).values()))),
                     own_ci=Q.ci(own_b), infl_ci=Q.ci((mo - mh).mean(0)), infl_p=Q.boot_p((mo - mh).mean(0)), n=len(R_))
            chance = 1.0 / Q.META[ds]["K"]
            d["p_vs_chance"] = Q.boot_p(own_b, chance)
            if ex_name == "RL":
                d["true_class_ceiling"] = float(np.mean([rl_ceiling(j, z) for j, _, z in R_]))
                rows.append([ds, arm, len(R_), f"{fmt(d['own'])} [{fmt(d['own_ci'][0])},{fmt(d['own_ci'][1])}]", fmt(d["hold"]),
                             fmt(d["own"] - d["hold"], d["infl_ci"]), f"{100 * chance:.1f}", fmt(d["true_class_ceiling"]),
                             pf(d["p_vs_chance"])])
            else:
                rows.append([ds, arm, len(R_), f"{fmt(d['own'])} [{fmt(d['own_ci'][0])},{fmt(d['own_ci'][1])}]", fmt(d["hold"]),
                             fmt(d["own"] - d["hold"], d["infl_ci"])])
            blk[f"{ds}|{arm}"] = d
    if ex_name == "RL":
        Q.table(["dataset", "arm", "runs", "own BA [CI]", "hold BA", "own - hold [CI]", "chance", "true-class ceiling",
                 "p own vs chance"], rows, [12, 5, 6, 24, 9, 26, 8, 19, 16])
        print("  true-class ceiling = holdout BA of a rule that knows only the true class (random labels are drawn per group,")
        print("  so with few groups they can correlate with the true classes by chance). A controlled arm whose own-fold BA")
        print("  sits at its holdout BA and near this ceiling has learned no group identity; the leaky arm's excess over its")
        print("  holdout (own - hold) is the memorised share.")
    else:
        Q.table(["dataset", "arm", "runs", "own BA [CI]", "hold BA", "inflation [CI]"], rows, [12, 5, 6, 26, 9, 30])
    OUT[ex_name] = blk

# ============================================================================ 11 dose-response
Q.sub("11 DOSE-RESPONSE (H5): inflation vs augmentation factor k, injected duplicates p, group overlap q")
DOSE = {}
rows = []
for ex_name, xform, lab in (("DOSEK", lambda d: math.log2(d), "log2 k"), ("DUP", lambda d: d, "p"), ("OVL", lambda d: d, "q")):
    Rx = sel(exp=ex_name, backbone=BB0, recipe="std")
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for arm in sorted({j["spec"]["arm"] for j, _, _ in Rx if j["spec"]["ds"] == ds}):
            pts = []
            cells = []
            for dose in P["exps"][ex_name]["doses"]:
                R_ = [x for x in Rx if x[0]["spec"]["ds"] == ds and x[0]["spec"]["arm"] == arm and x[0]["spec"]["dose"] == dose]
                if not R_:
                    cells.append("  —  ")
                    continue
                inf = [RM[j["id"]]["own"]["bal_acc"] - RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_]
                for (j, _, _), v in zip(R_, inf):
                    pts.append((xform(dose), v, j["spec"]["outer"]))
                cells.append(f"{100 * np.mean(inf):+5.1f}")
            if len(pts) >= 4:
                x, yv, g = map(np.array, zip(*pts))
                try:
                    import pandas as pd
                    import statsmodels.formula.api as smf
                    with Q.quiet():
                        mm = smf.mixedlm("y ~ x", pd.DataFrame(dict(x=x, y=yv, g=g)), groups=g).fit(reml=True)
                    slope, se, p = float(mm.params["x"]), float(mm.bse["x"]), float(mm.pvalues["x"])
                except Exception:
                    b = np.polyfit(x, yv, 1)
                    slope, se, p = float(b[0]), float("nan"), float("nan")
                from scipy.stats import spearmanr
                rho = float(spearmanr(x, yv).correlation) if len(set(x)) > 1 else float("nan")
                DOSE[f"{ex_name}|{ds}|{arm}"] = dict(points=pts, slope=slope, se=se, p=p, spearman=rho)
                rows.append([ex_name, ds, arm, "  ".join(cells), f"{100 * slope:+.2f}/{lab}", pf(p), f"{rho:+.2f}"])
Q.table(["exp", "dataset", "arm", "inflation at each dose (points)", "slope", "p (mixed)", "Spearman"], rows,
        [7, 12, 5, 44, 14, 10, 9])
for ex_name in ("DOSEK", "DUP", "OVL"):
    print(f"  {ex_name}: doses {P['exps'][ex_name]['doses']}")
OUT["dose"] = DOSE
Rx = sel(exp="DUP", backbone=BB0, recipe="std")
if Rx:
    rows = []
    for ds in sorted({j["spec"]["ds"] for j, _, _ in Rx}):
        for dose in P["exps"]["DUP"]["doses"][1:]:
            inj, non = [], []
            for j, r, z in Rx:
                if j["spec"]["ds"] != ds or j["spec"]["dose"] != dose or "own_injected" not in z:
                    continue
                m = z["own_injected"].astype(bool)
                c = correct(z, "own")
                if m.any():
                    inj.append(c[m].mean())
                if (~m).any():
                    non.append(c[~m].mean())
            if inj:
                rows.append([ds, dose, f"{100 * np.mean(inj):.1f}", f"{100 * np.mean(non):.1f}"])
    Q.table(["dataset", "p", "acc on injected (copy in training)", "acc on the rest"], rows, [12, 6, 36, 16])

# ============================================================================ 12 generality (backbones) + capacity
Q.sub("12 GENERALITY (H7): inflation by backbone family, seed 0, outer folds 0-2 (same splits for every backbone)")
G = P["generality"]
bbs = [BB0] + G["backbones"]
GEN = {}
rows = []
for ds in G["datasets"]:
    for bb in bbs:
        v = {}
        for arm in ("L", "C"):
            R_ = [x for x in sel(exp="E1", ds=ds, arm=arm, backbone=bb, recipe="std", seed=0) if x[0]["spec"]["outer"] in G["outer"]
                  and x[0]["spec"].get("rep", 0) == 0]
            if R_:
                v[arm] = (float(np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_])), len(R_))
        if "L" in v:
            GEN[f"{ds}|{bb}"] = v
            rows.append([ds, Q.BACKBONES[bb][0], f"{Q.BACKBONES[bb][1]:.1f}M", f"{fmt(v['L'][0])}/{fmt(v['L'][1])}",
                         fmt(v["L"][0] - v["L"][1]).replace("+", "+"),
                         f"{fmt(v['C'][0])}/{fmt(v['C'][1])}" if "C" in v else "—",
                         fmt(v["C"][0] - v["C"][1]) if "C" in v else "—", v["L"][2]])
Q.table(["dataset", "backbone", "params", "L own/hold", "L infl.", "C own/hold", "C infl.", "folds"], rows,
        [12, 12, 8, 16, 9, 16, 9, 6])
probes = Q.jload(Q.D["eval"] / "probes.json", []) or []
if probes and GEN:
    from scipy.stats import spearmanr
    rows = []
    for ds in G["datasets"]:
        pts = []
        for bb in bbs:
            v = GEN.get(f"{ds}|{bb}")
            if v:
                pts.append((Q.BACKBONES[bb][1], v["L"][0] - v["L"][1]))
        pr = [p_ for p_ in probes if p_["exp"] == "E1" and p_["ds"] == ds and p_["arm"] == "L" and p_["seed"] == 0
              and p_["outer"] in G["outer"] and "logreg" in p_]
        lin = float(np.mean([p_["logreg"]["own"] - p_["logreg"]["hold"] for p_ in pr])) if pr else float("nan")
        nn1 = float(np.mean([p_["1nn"]["own"] - p_["1nn"]["hold"] for p_ in pr])) if pr else float("nan")
        rho = float(spearmanr(*zip(*pts)).correlation) if len(pts) >= 3 else float("nan")
        rows.append([ds, fmt(lin), fmt(nn1), f"{rho:+.2f}" if np.isfinite(rho) else "— (needs 3 models)", len(pts)])
        GEN[f"{ds}|capacity"] = dict(linear_probe=lin, one_nn=nn1, spearman_params=rho)
    Q.table(["dataset", "linear probe infl.", "1-NN infl.", "Spearman(params, infl.)", "fine-tuned models"], rows,
            [12, 18, 12, 24, 18])
OUT["generality"] = GEN

# ============================================================================ 13 reimplementations
Rx = [x for x in RUNS.values() if x[0]["spec"]["recipe"] != "std"]
Q.sub("13 PUBLISHED PIPELINES REIMPLEMENTED (EfficientNet-B3): same method under the leaky and the controlled protocol")
if not Rx:
    print("  no finished runs yet — the reimplementations are trained in the 'general' phase (BLOCK 24)")
else:
    rows = []
    REI = {}
    for r_ in P["reimp"]:
        v = {}
        for arm in r_["arms"]:
            R_ = [x for x in Rx if x[0]["spec"]["recipe"] == r_["recipe"] and x[0]["spec"]["ds"] == r_["ds"]
                  and x[0]["spec"]["exp"] == r_["exp"] and x[0]["spec"]["arm"] == arm]
            if R_:
                v[arm] = (float(np.mean([RM[j["id"]]["own"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["hold"]["bal_acc"] for j, _, _ in R_])),
                          float(np.mean([RM[j["id"]]["own"]["acc"] for j, _, _ in R_])), len(R_))
        if v:
            REI[f"{r_['paper']}|{r_['exp']}|{r_['ds']}"] = v
            for arm, t in v.items():
                rows.append([r_["paper"][:40], arm, fmt(t[2]), fmt(t[0]), fmt(t[1]), fmt(t[0] - t[1]), t[3]])
    Q.table(["pipeline", "arm", "own acc", "own BA", "hold BA", "inflation", "folds"], rows, [42, 5, 9, 9, 9, 10, 6])
    OUT["reimp"] = REI

# ============================================================================ 14 cross-dataset
Q.sub("14 CROSS-DATASET TRANSFER (E1 seed 0): models trained on one corpus, tested on the other binary H&E corpora")
rows = []
XD = {}
for ds in Q.XEVAL:
    for arm in ("L", "C"):
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std", seed=0)
        for t in Q.XEVAL[ds]:
            au, bal = [], []
            for j, r, z in R_:
                if f"x_{t}_logit" not in z:
                    continue
                p_ = Q.softmax(z[f"x_{t}_logit"])
                score = p_[:, 1] if p_.shape[1] == 2 else p_[:, 2] / (p_[:, 0] + p_[:, 2] + 1e-9)
                y = z[f"x_{t}_y"]
                from sklearn.metrics import roc_auc_score
                if len(np.unique(y)) == 2:
                    au.append(roc_auc_score(y, score))
                    pr = (score >= 0.5).astype(int)
                    bal.append(0.5 * (np.mean(pr[y == 1] == 1) + np.mean(pr[y == 0] == 0)))
            if au:
                XD[f"{ds}|{arm}|{t}"] = dict(auroc=float(np.mean(au)), bal_acc=float(np.mean(bal)), n=len(au))
                rows.append([ds, arm, t, f"{np.mean(au):.3f}", fmt(np.mean(bal)), len(au)])
if rows:
    Q.table(["trained on", "arm", "tested on", "AUROC", "BA (thr 0.5)", "models"], rows, [12, 5, 12, 8, 13, 8])
OUT["cross_dataset"] = XD

# ============================================================================ 15 mechanism: NN similarity vs correctness
Q.sub("15 MECHANISM: similarity to the nearest training image and correctness (E1 seed 0, frozen DINOv2 space)")
audit = Q.jload(Q.D["index"] / "audit.json", {}) or {}
EMB = audit.get("_meta", {}).get("embed", "dinov2_s")
rows = []
MECH = {}
for ds in avail:
    fe = Q.D["store"] / f"{ds}_emb_{EMB}.npy"
    if not fe.exists():
        continue
    E = np.load(fe).astype(np.float32)
    pol = P["exps"]["E1"]["datasets"][ds].get("policy")
    sib = None
    if pol:
        fs = Q.D["store"] / f"{ds}_sib_{pol}_{EMB}.npz"
        if fs.exists():
            zz = np.load(fs)
            pos_ = np.full(len(E), -1)
            pos_[zz["rows"]] = np.arange(len(zz["rows"]))
            sib = (pos_, zz["emb"].astype(np.float32), int(zz["kmax"]))

    def fx(rows_, copies):
        X = E[rows_].copy()
        if sib is not None:
            m = (copies > 0) & (copies <= sib[2])
            X[m] = sib[1][sib[0][rows_[m]], copies[m] - 1]
        return X

    for arm in ("L", "C"):
        sims, cors = defaultdict(list), defaultdict(list)
        for j, r, z in sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std", seed=0):
            B = Q.build_job(j["spec"], check=False)
            Xt = fx(B["train"]["row"], B["train"]["copy"])
            for part in ("own", "hold"):
                Xe = fx(z[f"{part}_row"], z[f"{part}_copy"])
                s_ = np.concatenate([(Xe[i:i + 2048] @ Xt.T).max(1) for i in range(0, len(Xe), 2048)]) if len(Xe) else np.zeros(0)
                sims[part].append(s_)
                cors[part].append(correct(z, part))
        if sims:
            so, sh = np.concatenate(sims["own"]), np.concatenate(sims["hold"])
            co, ch = np.concatenate(cors["own"]), np.concatenate(cors["hold"])
            allc, alls = np.concatenate([co, ch]), np.concatenate([so, sh])
            r_pb = float(np.corrcoef(alls, allc)[0, 1]) if allc.std() > 0 else float("nan")
            MECH[f"{ds}|{arm}"] = dict(sim_own=float(so.mean()), sim_hold=float(sh.mean()), r_sim_correct=r_pb,
                                       acc_top_decile=float(allc[alls >= np.percentile(alls, 90)].mean()),
                                       acc_bottom_decile=float(allc[alls <= np.percentile(alls, 10)].mean()))
            rows.append([ds, arm, f"{so.mean():.3f}", f"{sh.mean():.3f}", f"{r_pb:+.3f}",
                         fmt(MECH[f'{ds}|{arm}']['acc_top_decile']), fmt(MECH[f'{ds}|{arm}']['acc_bottom_decile'])])
Q.table(["dataset", "arm", "sim own", "sim hold", "r(sim, correct)", "acc top-10% sim", "acc bottom-10% sim"], rows,
        [12, 5, 9, 9, 16, 16, 18])
OUT["mechanism_nn"] = MECH

# ============================================================================ 16 leakage-risk predictor
Q.sub("16 LEAKAGE-RISK PREDICTOR (exploratory): does the provenance audit predict a run's inflation?")
FEAT = ["same_original", "identical_file", "near_dup", "struct_group", "ctrl_group"]
Xr, yr, gr = [], [], []
for rid, (j, r, z) in RUNS.items():
    if j["spec"]["recipe"] != "std" or j["spec"]["backbone"] != BB0 or j["spec"]["exp"] in ("RL",):
        continue
    lk = r["info"].get("leak", {})
    if not lk or "own" not in RM[rid]:
        continue
    Xr.append([lk.get(k, 0) for k in FEAT])
    yr.append(RM[rid]["own"]["bal_acc"] - RM[rid]["hold"]["bal_acc"])
    gr.append(j["spec"]["ds"])
if len(set(gr)) >= 3:
    from sklearn.linear_model import LinearRegression, RidgeCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    Xr, yr, gr = np.array(Xr, float), np.array(yr, float), np.array(gr)
    models = {"null (training mean)": None,
              "one predictor: any relative in training": [4],
              "ridge: all five audit shares": [0, 1, 2, 3, 4]}
    RISK = {}
    rows = []
    for name, cols in models.items():
        pred = np.zeros_like(yr)
        for ds in sorted(set(gr)):
            m = gr == ds
            if cols is None:
                pred[m] = yr[~m].mean()
            elif len(cols) == 1:
                pred[m] = LinearRegression().fit(Xr[~m][:, cols], yr[~m]).predict(Xr[m][:, cols])
            else:
                mdl = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 12)))
                pred[m] = mdl.fit(Xr[~m][:, cols], yr[~m]).predict(Xr[m][:, cols])
        r2 = 1 - np.sum((yr - pred) ** 2) / np.sum((yr - yr.mean()) ** 2)
        mae = float(np.mean(np.abs(yr - pred)))
        RISK[name] = dict(lodo_r2=float(r2), mae=mae)
        rows.append([name, f"{r2:+.2f}", f"{100 * mae:.1f}"])
    Q.table(["model (leave-one-dataset-out)", "R²", "MAE (points)"], rows, [44, 8, 14])
    one = LinearRegression().fit(Xr[:, [4]], yr)
    RISK["slope_any_relative"] = float(one.coef_[0])
    RISK["n_runs"] = int(len(yr))
    RISK["datasets"] = sorted(set(gr))
    OUT["risk_predictor"] = RISK
    Q.kv("slope, any-relative share", f"{100 * one.coef_[0]:+.1f} points of inflation per 100% share (all runs pooled)")
    print(f"  n = {len(yr)} runs from {len(set(gr))} datasets. A negative R² means the model predicts worse than the mean of")
    print("  the other datasets; that result is reported as it is.")
else:
    print("  fewer than 3 datasets with finished runs — skipped")

# ============================================================================ 17 provenance-stratified own-fold accuracy (Idea B)
Q.sub("17 PROVENANCE-STRATIFIED OWN FOLD (L arm, all seeds): accuracy by the closest relative each test image has in training")
STRATA = ["same source image (sibling)", "identical / near-duplicate file", "same slide / patient / source group", "no relative"]
STR = {}
rows, csvrows = [], []
for ds in avail:
    R_ = sel(exp="E1", ds=ds, arm="L", backbone=BB0, recipe="std")
    if not R_:
        continue
    dd = Q.DS.get(ds)
    cor_all, st_all, key_all, hold_acc = [], [], [], []
    for j, r, z in R_:
        B = Q.build_job(dict(j["spec"], xeval=False), check=False)
        tr = B["train"]["row"]
        s_row, s_md5 = set(tr.tolist()), set(dd.md5[tr].tolist())
        s_near, s_grp = set(dd.g_near[tr].tolist()), set(dd.g_ctrl[tr].tolist())
        rows_o = z["own_row"]
        st = np.full(len(rows_o), 3)
        for i, rr in enumerate(rows_o):
            if rr in s_row:
                st[i] = 0
            elif dd.md5[rr] in s_md5 or dd.g_near[rr] in s_near:
                st[i] = 1
            elif dd.g_ctrl[rr] in s_grp:
                st[i] = 2
        cor_all.append(correct(z, "own"))
        st_all.append(st)
        key_all.append(z["own_key"] + 10 ** 7 * j["spec"].get("rep", 0) + 10 ** 8 * j["spec"]["outer"])
        hold_acc.append(float(correct(z, "hold").mean()))
    cor, st, key = map(np.concatenate, (cor_all, st_all, key_all))
    cl, inv = np.unique(key, return_inverse=True)
    g = np.random.default_rng(Q.shash(ds, "strata") % 2 ** 32)
    W = list(Q.boot_weights(len(cl), NB, g))
    d = dict(hold_acc=float(np.mean(hold_acc)), n_runs=len(R_), strata={})
    boots = {}
    for k, name in enumerate(STRATA):
        m = st == k
        if not m.any():
            continue
        cc = np.bincount(inv, cor * m, len(cl))
        nn_ = np.bincount(inv, m.astype(float), len(cl))
        bs = np.concatenate([(Wt @ cc) / np.maximum(Wt @ nn_, 1e-9) for Wt in W])
        boots[k] = bs
        d["strata"][name] = dict(share=float(m.mean()), n=int(m.sum()), acc=float(cor[m].mean()), ci=Q.ci(bs))
        rows.append([ds, name, f"{100 * m.mean():.1f}", int(m.sum()), f"{100 * cor[m].mean():.2f} [{100 * Q.ci(bs)[0]:.2f},"
                     f"{100 * Q.ci(bs)[1]:.2f}]", f"{100 * d['hold_acc']:.2f}"])
        csvrows.append([ds, name, m.mean(), int(m.sum()), cor[m].mean(), *Q.ci(bs), d["hold_acc"]])
    if 0 in boots and 3 in boots:
        diff = boots[0] - boots[3]
        d["sibling_minus_none"] = dict(est=d["strata"][STRATA[0]]["acc"] - d["strata"][STRATA[3]]["acc"], ci=Q.ci(diff),
                                       p=Q.boot_p(diff))
    STR[ds] = d
if rows:
    Q.table(["dataset", "closest relative in training", "% of own fold", "images", "own-fold accuracy [CI]", "holdout acc"], rows,
            [12, 38, 14, 8, 26, 12])
    for ds, d in STR.items():
        if "sibling_minus_none" in d:
            e = d["sibling_minus_none"]
            Q.kv(f"{ds}: sibling - no relative", f"{100 * e['est']:+.2f} points [{100 * e['ci'][0]:+.2f}, {100 * e['ci'][1]:+.2f}] "
                 f"p {pf(e['p'])}")
    print("  images are pooled over folds and seeds (an image counted once per run); CIs resample groups (fold x group clusters).")
    print("  A stratum with very few images (e.g. 'no relative' under heavy augmentation) gives a wide CI — read it with its n.")
    write_csv("provenance_strata", ["dataset", "stratum", "share", "n", "acc", "ci_lo", "ci_hi", "holdout_acc"], csvrows)
OUT["provenance_strata"] = STR

# ============================================================================ 18 confusion matrices (multi-class)
CONF = {}
for ds in [d for d in avail if Q.META[d]["K"] > 2]:
    K = Q.META[ds]["K"]
    cls = Q.META[ds]["classes"]
    Q.sub(f"18 CONFUSION MATRICES, {ds} (E1, all seeds, pooled over outer folds; rows = true class, % of the row)")
    crow = []
    for arm, part in (("L", "own"), ("L", "hold"), ("C", "own"), ("C", "hold")):
        R_ = sel(exp="E1", ds=ds, arm=arm, backbone=BB0, recipe="std")
        if not R_:
            continue
        M = np.zeros((K, K), int)
        for j, r, z in R_:
            yt, yp = z[f"{part}_y"], z[f"{part}_logit"].argmax(1)
            np.add.at(M, (yt, yp), 1)
        rowp = M / np.maximum(M.sum(1, keepdims=True), 1)
        CONF[f"{ds}|{arm}|{part}"] = dict(counts=M.tolist(), row_pct=rowp.tolist(), classes=cls)
        print(f"  arm {arm} · {'own fold (what a paper would report)' if part == 'own' else 'clean holdout'} · "
              f"{int(M.sum()):,} predictions")
        Q.table(["true \\ predicted"] + cls + ["recall"],
                [[cls[i]] + [f"{100 * rowp[i, k]:.1f}" for k in range(K)] + [f"{100 * rowp[i, i]:.1f}"] for i in range(K)],
                [20] + [14] * K + [10])
        off = M.sum() - np.trace(M)
        for i in range(K):
            for k in range(K):
                crow.append([ds, arm, part, cls[i], cls[k], int(M[i, k]), float(rowp[i, k])])
        if off:
            pair = max(((i, k) for i in range(K) for k in range(K) if i != k), key=lambda t: M[t] + M[t[::-1]])
            Q.kv("most confused pair", f"{cls[pair[0]]} <-> {cls[pair[1]]}: {100 * (M[pair] + M[pair[::-1]]) / off:.1f}% "
                 f"of all {int(off):,} errors")
    write_csv(f"confusion_{ds}", ["dataset", "arm", "part", "true", "predicted", "count", "row_share"], crow)
OUT["confusion"] = CONF

Q.jsave(Q.D["eval"] / "q1_results.json", OUT)
_nxt = "run BLOCK 27 next (amendment A1 order: 27 -> 33 -> 28)" if _A1 else "run BLOCK 26 next"
print(f"\n[BLOCK 25 ✓]  results in q1/eval/q1_results.json · tables in q1/tables/ · {Q.hms(time.time() - t_all)} · {_nxt}")

In [ ]:
# ============================================================================
# BLOCK 27 — PAPER FIGURES (PNG, 300 dpi) + RESULTS SUMMARY (Markdown)
# ============================================================================
import os, sys, json, time, math, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("27")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

t_all = time.time()
P = Q.plan()
R = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
CL = Q.jload(Q.D["eval"] / "clinical.json", {}) or {}
AU = Q.jload(Q.D["index"] / "audit.json", {}) or {}
PR = Q.jload(Q.D["eval"] / "probes.json", []) or []
FIG = Q.D["figs"]
COL = {"L": "#eb6834", "C": "#2a78d6", "probe": "#1baf7a"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
NAME = {"he_stained": "OCDC (H&E)", "histopath": "Histopathology", "ndb_ufes": "NDB-UFES", "clinical": "Clinical photos",
        "breakhis": "BreakHis"}
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 300, "figure.dpi": 110})
made = []
Q.hdr("BLOCK 27 — PAPER FIGURES + RESULTS SUMMARY")


def save(fig, name, note):
    fig.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    made.append((name, note))


# ---------------------------------------------------------------- Figure A: own fold vs clean holdout
E1 = R.get("E1", {})
ds_list = [d for d in Q.DATASETS if d in E1 and "L" in E1[d].get("arms", {}) and "C" in E1[d]["arms"]]
if ds_list:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 0.75 * len(ds_list) + 1.6), gridspec_kw=dict(width_ratios=[1.25, 1]))
    for i, ds in enumerate(ds_list):
        y0 = len(ds_list) - 1 - i
        for k, arm in enumerate(("L", "C")):
            a = E1[ds]["arms"][arm]
            yy = y0 + (0.15 if arm == "L" else -0.15)
            a1.plot([100 * a["hold"], 100 * a["own"]], [yy, yy], color=COL[arm], lw=2, solid_capstyle="round")
            a1.scatter([100 * a["hold"]], [yy], s=40, color="white", edgecolor=COL[arm], lw=2, zorder=3)
            a1.scatter([100 * a["own"]], [yy], s=40, color=COL[arm], zorder=3)
            lo, hi = a["infl_ci"]
            a2.errorbar([100 * a["infl"]], [yy], xerr=[[100 * (a["infl"] - lo)], [100 * (hi - a["infl"])]], fmt="o",
                        color=COL[arm], ms=6, lw=2, capsize=0)
    for ax in (a1, a2):
        ax.set_yticks(range(len(ds_list)))
        ax.set_yticklabels([NAME[d] for d in ds_list][::-1])
        ax.grid(axis="y", visible=False)
    a1.set_xlabel("Balanced accuracy (%)")
    a1.set_title("A  What the protocol reports (filled) vs clean holdout (open)", loc="left", fontsize=9, color=INK)
    a2.axvline(0, color=INK2, lw=1)
    a2.set_xlabel("Inflation = own fold − clean holdout (points, 95% CI)")
    a2.set_title("B  Inflation by arm", loc="left", fontsize=9, color=INK)
    h = [plt.Line2D([], [], color=COL["L"], lw=2, marker="o", label="leaky protocol (as published)"),
         plt.Line2D([], [], color=COL["C"], lw=2, marker="o", label="leakage-controlled, matched")]
    fig.legend(handles=h, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save(fig, "Fig_Q1_main_own_vs_holdout.png", "E1: own-fold vs clean-holdout balanced accuracy and inflation with 95% CI")

# ---------------------------------------------------------------- Figure B: dose-response
D_ = R.get("dose", {})
panels = [(k.split("|")) for k in D_]
keys = sorted({(e, d) for e, d, a in panels}, key=lambda t: (["DOSEK", "DUP", "OVL"].index(t[0]), t[1]))
if keys:
    nc = min(4, len(keys))
    nr = int(math.ceil(len(keys) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.1 * nc, 2.6 * nr), squeeze=False)
    xl = {"DOSEK": "augmentation factor k (log scale)", "DUP": "share of own-fold images with a copy in training",
          "OVL": "share of group-mates moved into training"}
    for ax, (ex, ds) in zip(axs.ravel(), keys):
        for arm in ("L", "C"):
            v = D_.get(f"{ex}|{ds}|{arm}")
            if not v:
                continue
            pts = np.array(v["points"], float)
            xs = 2 ** pts[:, 0] if ex == "DOSEK" else pts[:, 0]
            ax.scatter(xs, 100 * pts[:, 1], s=10, color=COL[arm], alpha=0.35, lw=0)
            ux = np.unique(xs)
            lab_ = "leaky" if arm == "L" else "controlled"
            if v.get("p") is not None and np.isfinite(v["p"]):
                lab_ += f"  slope p={v['p']:.3g}"
            ax.plot(ux, [100 * pts[xs == u, 1].mean() for u in ux], color=COL[arm], lw=2, marker="o", ms=4, label=lab_)
        if ex == "DOSEK":
            ax.set_xscale("log", base=2)
        ax.axhline(0, color=INK2, lw=1)
        ax.set_title(f"{ex} · {NAME[ds]}", loc="left", fontsize=8.5, color=INK)
        ax.set_xlabel(xl[ex], fontsize=7.5)
        ax.set_ylabel("inflation (points)", fontsize=7.5)
        ax.legend(fontsize=6.5, loc="upper left")
    for ax in axs.ravel()[len(keys):]:
        ax.axis("off")
    fig.tight_layout()
    save(fig, "Fig_Q1_dose_response.png", "dose-response: augmentation factor, injected duplicates, group overlap (fold points + means)")

# ---------------------------------------------------------------- Figure C: random-label control
RL = R.get("RL", {})
if RL:
    dss = sorted({k.split("|")[0] for k in RL})
    fig, ax = plt.subplots(figsize=(1.6 * len(dss) + 2, 3))
    for i, ds in enumerate(dss):
        for k, arm in enumerate(("L", "C")):
            v = RL.get(f"{ds}|{arm}")
            if not v:
                continue
            x = i + (-0.18 if arm == "L" else 0.18)
            ax.bar(x, 100 * v["own"], width=0.34, color=COL[arm], label=("leaky" if arm == "L" else "controlled") if i == 0 else None)
            ax.errorbar(x, 100 * v["own"], yerr=[[100 * (v["own"] - v["own_ci"][0])], [100 * (v["own_ci"][1] - v["own"])]],
                        color=INK, lw=1, capsize=2)
            ax.scatter([x], [100 * v["hold"]], marker="D", s=26, color="white", edgecolor=INK, lw=1.2, zorder=4,
                       label="clean holdout" if (i == 0 and arm == "L") else None)
        ch = 100 / Q.META[ds]["K"]
        ax.plot([i - 0.4, i + 0.4], [ch, ch], color=INK2, lw=1.2, ls="--", label="chance" if i == 0 else None)
    ax.set_xticks(range(len(dss)))
    ax.set_xticklabels([NAME[d] for d in dss])
    ax.set_ylabel("own-fold balanced accuracy (%)")
    ax.set_title("Labels randomised per group: own-fold BA with 95% CI (bars) and clean-holdout BA (diamonds)",
                 loc="left", fontsize=9)
    ax.set_ylim(0, 112)
    ax.legend(loc="upper center", ncol=4, fontsize=7.5, frameon=False, bbox_to_anchor=(0.5, 1.0))
    ax.grid(axis="x", visible=False)
    fig.tight_layout()
    save(fig, "Fig_Q1_random_label.png", "random-label control (labels per source/slide/patient)")

# ---------------------------------------------------------------- Figure D: capacity
GEN = R.get("generality", {})
dss = [d for d in P["generality"]["datasets"] if any(k.startswith(d + "|") and "|capacity" not in k for k in GEN)]
if dss:
    nc = min(3, len(dss))
    nr = int(np.ceil(len(dss) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(4.3 * nc, 3.6 * nr), squeeze=False)
    for ax in axs.ravel()[len(dss):]:
        ax.axis("off")
    for ax, ds in zip(axs.ravel(), dss):
        cap = GEN.get(f"{ds}|capacity", {})
        ticks, labels = [], []
        pos = 0
        if cap and np.isfinite(cap.get("one_nn", np.nan)):
            ax.scatter([pos], [100 * cap["one_nn"]], marker="^", s=34, color=INK2, zorder=3, label="1-NN on frozen features")
            ticks.append(pos); labels.append("1-NN\nfrozen")
            pos += 1
        if cap and np.isfinite(cap.get("linear_probe", np.nan)):
            ax.scatter([pos], [100 * cap["linear_probe"]], marker="s", s=30, color=COL["probe"], zorder=3,
                       label="linear probe, frozen")
            ticks.append(pos); labels.append("linear\nprobe")
            pos += 1
        pts = []
        for bb, (short, par, _) in Q.BACKBONES.items():
            v = GEN.get(f"{ds}|{bb}")
            if v and "L" in v:
                pts.append((par, short, 100 * (v["L"][0] - v["L"][1])))
        pts.sort()
        if pts:
            xs = list(range(pos, pos + len(pts)))
            ax.plot(xs, [p_[2] for p_ in pts], color=COL["L"], lw=1.5, marker="o", ms=5, zorder=3, label="fine-tuned")
            ticks += xs
            labels += [f"{p_[1]}\n{p_[0]:.0f}M" for p_ in pts]
        ax.set_xticks(ticks)
        ax.set_xticklabels(labels, fontsize=6.5, rotation=35, ha="right", rotation_mode="anchor")
        ax.set_xlim(-0.6, max(ticks) + 0.6 if ticks else 1)
        ax.axhline(0, color=INK2, lw=1)
        ax.set_title(NAME[ds], loc="left", fontsize=8.5)
        ax.set_ylabel("leaky-arm inflation (points)", fontsize=7.5)
    axs[0][0].legend(fontsize=6.5, loc="best")
    fig.tight_layout()
    save(fig, "Fig_Q1_capacity.png", "inflation of the leaky arm across backbone families and frozen-feature probes")

# ---------------------------------------------------------------- Figure E: audit (NDB order + histopathology parents)
if (Q.D["index"] / "ndb_ufes.json").exists():
    dd = Q.DS.get("ndb_ufes")
    emb = Q.D["store"] / f"ndb_ufes_emb_{AU.get('_meta', {}).get('embed', 'dinov2_s')}.npy"
    if emb.exists():
        E = np.load(emb).astype(np.float32)
        order = np.argsort([r["pidx"] for r in dd.R])[:600]
        nshow = len(order)
        s = (E[order][:-1] * E[order][1:]).sum(1)
        seg = dd.g_struct[order]
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 2.8), gridspec_kw=dict(width_ratios=[2.2, 1]))
        a1.plot(np.arange(len(s)), s, color=COL["C"], lw=0.8)
        for b in np.nonzero(seg[1:] != seg[:-1])[0]:
            a1.axvline(b + 0.5, color=COL["L"], lw=0.5, alpha=0.6)
        tau = AU.get("ndb_ufes", {}).get("segmentation", {}).get("tau")
        if tau:
            a1.axhline(tau, color=INK2, ls="--", lw=1)
        a1.set_xlabel(f"patch index p (first {nshow} in index order)")
        a1.set_ylabel("cosine to next patch")
        a1.set_title("A  NDB-UFES: similarity of consecutive patches in file-index order (orange = recovered boundaries)", loc="left", fontsize=8.5)
        allc = (E[np.argsort([r["pidx"] for r in dd.R])][:-1] * E[np.argsort([r["pidx"] for r in dd.R])][1:]).sum(1)
        a2.hist(allc, bins=60, color=COL["C"])
        if tau:
            a2.axvline(tau, color=COL["L"], lw=1.5, label=f"Otsu threshold {tau:.3f}")
        a2.set_xlabel("adjacent-patch cosine")
        a2.set_title("B  distribution", loc="left", fontsize=8.5)
        a2.legend(fontsize=7)
        fig.tight_layout()
        save(fig, "Fig_Q1_ndb_order.png", "evidence that NDB-UFES patch indices follow source images (group recovery)")

# ---------------------------------------------------------------- Figure F: reliability + selective prediction
CAL = CL.get("calibration", {})
SEL = CL.get("selective", {})
dss = [d for d in Q.DATASETS if f"{d}|L" in CAL and f"{d}|C" in CAL]
if dss:
    fig, axs = plt.subplots(2, len(dss), figsize=(2.6 * len(dss), 5.2), squeeze=False)
    for i, ds in enumerate(dss):
        ax = axs[0][i]
        ax.plot([0, 1], [0, 1], color=INK2, lw=1, ls="--")
        for arm in ("L", "C"):
            c = CAL[f"{ds}|{arm}"]["cal"]
            pts = [(x, y) for x, y, n in zip(c["conf"], c["acc"], c["n"]) if x is not None and n >= 5]
            if pts:
                ax.plot(*zip(*pts), color=COL[arm], lw=2, marker="o", ms=3,
                        label=f"{'leaky' if arm == 'L' else 'controlled'} ECE {c['ece']:.3f}")
        ax.set_title(NAME[ds], loc="left", fontsize=8.5)
        ax.set_xlabel("confidence", fontsize=7.5)
        ax.set_ylabel("accuracy", fontsize=7.5)
        ax.legend(fontsize=6)
        ax = axs[1][i]
        rk = None
        for arm in ("L", "C"):
            v = SEL.get(f"{ds}|{arm}")
            if not v:
                continue
            cov = sorted(v["cov"], key=float, reverse=True)
            x = [100 * float(c) for c in cov]
            y = [100 * v["cov"][c]["acc"] for c in cov]
            lo = [100 * v["cov"][c]["ci"][0] for c in cov]
            hi = [100 * v["cov"][c]["ci"][1] for c in cov]
            ax.fill_between(x, lo, hi, color=COL[arm], alpha=0.15, lw=0)
            ax.plot(x, y, color=COL[arm], lw=2, marker="o", ms=3, label="leaky" if arm == "L" else "controlled")
            rk = v.get("ranking")
        if rk:
            ax.text(0.02, 0.02, f"ranked by {rk} entropy", transform=ax.transAxes, fontsize=6, color=INK2)
        if i == 0:
            ax.legend(fontsize=6, loc="upper left")
        ax.invert_xaxis()
        ax.set_xlabel("coverage (% auto-reported)", fontsize=7.5)
        ax.set_ylabel("holdout accuracy (%)", fontsize=7.5)
    fig.tight_layout()
    save(fig, "Fig_Q1_calibration_selective.png", "holdout reliability (temperature from inner validation) and selective prediction with 95% CI")

# ---------------------------------------------------------------- results summary markdown
L = ["# Q1 results summary (generated by Block 27)", "", f"Plan SHA-256: `{R.get('meta', {}).get('plan_sha')}` · runs: "
     f"{R.get('meta', {}).get('n_runs')} · bootstrap resamples: {R.get('meta', {}).get('boot')}", "",
     "## E1: own fold vs clean holdout (balanced accuracy, %)", "",
     "Inflation CIs and p-values: two-stage cluster bootstrap. p is Holm-adjusted for the L arm (primary family H1) and "
     "unadjusted for the other arms.", "",
     "| Dataset | Arm | Own | Holdout | Inflation [95% CI] | p | seed SD |", "|---|---|---|---|---|---|---|"]
for ds in ds_list:
    for arm, a in E1[ds]["arms"].items():
        sdv = "" if a.get("seed_sd") is None else f"{100 * a['seed_sd']:.2f}"
        pv = a.get("infl_p_holm", a["infl_p"])
        L.append(f"| {NAME[ds]} | {arm} | {100 * a['own']:.2f} | {100 * a['hold']:.2f} | {100 * a['infl']:+.2f} "
                 f"[{100 * a['infl_ci'][0]:+.2f}, {100 * a['infl_ci'][1]:+.2f}] | {pv:.2g} | {sdv} |")
L += ["", "## Figures", ""] + [f"- `{n}` — {d}" for n, d in made]
(Q.D["tables"] / "RESULTS_Q1.md").write_text("\n".join(L))
for n, d in made:
    Q.kv(n, d)
_nxt = "run BLOCK 33 next, then 28" if (Q.D["jobs"] / "jobs_A1.json").exists() else "run BLOCK 28 next"
print(f"\n[BLOCK 27 ✓]  {len(made)} figures in q1/figs/ · summary q1/tables/RESULTS_Q1.md · {Q.hms(time.time() - t_all)} · {_nxt}")

In [ ]:
# ============================================================================
# BLOCK 33 — AMENDMENT A1 RESULTS: ORCHID · RECOVERED-GROUP ARMS · RANDOM LABELS · RECOVERY · PHIKON · FIGURES
# needs: BLOCK 31 + the A1 training of BLOCK 32 finished (missing runs are reported as missing)
# ============================================================================
import os, sys, json, time, math, csv, importlib, warnings
from pathlib import Path
from collections import defaultdict, Counter
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("33")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker
from scipy.stats import spearmanr

NB = 10000
t_all = time.time()
P = Q.plan()
if "amendment_A1" not in P:
    raise SystemExit("amendment A1 not found - run BLOCK 29 first")
BB0 = P["primary_backbone"]
Q.hdr("BLOCK 33 — AMENDMENT A1 RESULTS")
JA = (Q.jload(Q.D["jobs"] / "jobs_A1.json", {}) or {}).get("jobs", [])
RUNS = {}
for j in JA:
    if Q.job_done(j):
        r, z = Q.load_run(j["id"])
        if r is not None:
            RUNS[j["id"]] = (j, r, {k: z[k] for k in z.files})
Q.kv("A1 runs", f"{len(RUNS)} of {len(JA)} finished" + ("" if len(RUNS) == len(JA) else " (missing runs are reported as missing)"))
REF = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
RCV = Q.jload(Q.D["eval"] / "recovery.json", {}) or {}
AUX = Q.jload(Q.D["index"] / "audit_ext.json", {}) or {}
ALLJ = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
OUT = dict(meta=dict(n_runs=len(RUNS), n_jobs=len(JA), boot=NB,
                     amendment=[a for a in P.get("amendments", []) if a["id"] == "A1"]))
TAB, FIG = Q.D["tables"], Q.D["figs"]
COL = {"L": "#eb6834", "C": "#2a78d6", "probe": "#1baf7a", "R": "#8b5cf6"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
NAME = {"he_stained": "OCDC (H&E)", "histopath": "Histopathology", "ndb_ufes": "NDB-UFES", "clinical": "Clinical photos",
        "breakhis": "BreakHis", "orchid": "ORCHID (oral H&E)"}
plt.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "legend.frameon": False, "savefig.dpi": 300, "figure.dpi": 110})
made = []


def save(fig, name, note):
    fig.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    made.append((name, note))


def write_csv(name, header, rows):
    with open(TAB / f"{name}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


def sel(**kw):
    return [(j, r, z) for j, r, z in RUNS.values() if all(j["spec"].get(k) == v for k, v in kw.items())]


def correct(z, name):
    return (z[f"{name}_logit"].argmax(1) == z[f"{name}_y"]).astype(float)


def fold_key(s):
    return (s["outer"], s.get("rep", 0))


def tables_for(runs, part):
    by = defaultdict(list)
    for j, r, z in runs:
        by[fold_key(j["spec"])].append((j, r, z))
    out = {}
    for fk, lst in by.items():
        K = lst[0][1]["info"]["K"]
        out[fk] = Q.cluster_table(np.concatenate([z[f"{part}_y"] for _, _, z in lst]),
                                  np.concatenate([correct(z, part) for _, _, z in lst]),
                                  np.concatenate([z[f"{part}_key"] for _, _, z in lst]), K)
    return out


def boot_folds(tabs, B, seed):
    g = np.random.default_rng(seed)
    ks = sorted(tabs)
    return ks, np.stack([Q.boot_ba(tabs[k][0], tabs[k][1], B, g) for k in ks])


def point(tabs):
    return {k: float(Q.ba_counts(*tabs[k])) for k in tabs}


def fmt(v, ci_=None, d=2):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    s = f"{100 * v:+.{d}f}" if ci_ is not None else f"{100 * v:.{d}f}"
    if ci_ is not None:
        s += f" [{100 * ci_[0]:+.{d}f}, {100 * ci_[1]:+.{d}f}]"
    return s


def pf(p):
    return "—" if p is None or not np.isfinite(p) else (f"{p:.1e}" if p < 1e-3 else f"{p:.4f}")


def arm_stats(R_, tag):
    """Own BA, holdout BA and inflation with two-stage (fold + cluster) bootstrap CI, as in BLOCK 25."""
    to, th = tables_for(R_, "own"), tables_for(R_, "hold")
    common = sorted(set(to) & set(th))
    if not common:
        return None
    to, th = {k: to[k] for k in common}, {k: th[k] for k in common}
    _, mo = boot_folds(to, NB, Q.shash(tag, "own") % 2 ** 32)
    _, mh = boot_folds(th, NB, Q.shash(tag, "hold") % 2 ** 32)
    po, ph_ = point(to), point(th)
    infl_b = Q.hier([mo - mh], NB, Q.shash(tag, "hier") % 2 ** 32)[0]
    seeds = sorted({j["spec"]["seed"] for j, _, _ in R_})
    per_seed = []
    for sd in seeds:
        Rs = [x for x in R_ if x[0]["spec"]["seed"] == sd]
        a, b = point(tables_for(Rs, "own")), point(tables_for(Rs, "hold"))
        kk = set(a) & set(b)
        if kk:
            per_seed.append(float(np.mean([a[k] - b[k] for k in kk])))
    leak = float(np.mean([j["leak"].get("struct_group", 0.0) for j, _, _ in R_]))
    pos_s = lambda part: float(np.nanmean([Q.metrics(Q.softmax(z[f"{part}_logit"]), z[f"{part}_y"], r["info"]["K"],
                                                     Q.META[j["spec"]["ds"]]["pos"])["pos"]["sens"] for j, r, z in R_]))
    return dict(n_runs=len(R_), folds=len(common), seeds=seeds, own=float(np.mean(list(po.values()))),
                hold=float(np.mean(list(ph_.values()))), infl=float(np.mean([po[k] - ph_[k] for k in common])),
                infl_ci=Q.ci(infl_b), infl_p=Q.boot_p(infl_b), seed_sd=float(np.std(per_seed, ddof=1)) if len(per_seed) > 1 else None,
                leak=leak, own_pos_sens=pos_s("own"), hold_pos_sens=pos_s("hold"), _mo=mo, _mh=mh, _k=common)


def strip(d):
    return {k: v for k, v in d.items() if not k.startswith("_")} if d else d


# ============================================================================ 1  ORCHID audit
A1 = {}
if "orchid" in AUX:
    Q.sub("1  ORCHID AUDIT  (released partitions and patients)")
    ax = AUX["orchid"]
    for p, v in ax.get("partitions", {}).items():
        Q.kv(f"released {p} partition", f"{v['images']:,} images · {v['patients']} patients")
    for k, v in ax.get("patients_shared", {}).items():
        Q.kv(f"patients shared {k.replace('|', ' & ')}", v)
    for k, v in ax.get("identical_across_partitions", {}).items():
        Q.kv(f"identical files {k.replace('|', ' & ')}", f"{v['confirmed']} MD5-confirmed of {v['candidates']} candidates · "
             f"{v['different_patient_folder']} in another patient folder · {v['different_grade_folder']} in another grade folder")
    Q.kv("our corpus", f"{ax.get('n_unique_originals')} images · {ax.get('n_patients')} patient folders · "
         f"identical files {ax.get('exact_redundant')} · near-duplicate pairs {ax.get('near_pairs')}")
    for k, v in (ax.get("cross_patient") or {}).items():
        Q.kv(f"{k.replace('_', '-')} across folders", f"{v['groups_spanning_patients']} groups · {v['images']} images · "
             f"{v['patients_involved']} folders · {v['groups_spanning_grades']} span two grades")
    if ax.get("split_units"):
        Q.kv("split units", f"{ax['split_units']} (patient folders joined by shared images)")
    A1["orchid_audit"] = ax

# ============================================================================ 2  ORCHID main experiment (H9)
ORC = {}
if sel(exp="X1", ds="orchid"):
    Q.sub("2  ORCHID: OWN FOLD vs PATIENT-DISJOINT HOLDOUT  (H9: image-level split over-estimates; two-stage bootstrap)")
    rows, csvrows = [], []
    for arm in ("L", "C", "R25", "R100", "Rauto"):
        R_ = sel(exp="X1", ds="orchid", arm=arm)
        if not R_:
            continue
        a = arm_stats(R_, f"orchid|{arm}")
        ORC[arm] = a
        rows.append([arm, f"{a['n_runs']}/{a['folds']}", fmt(a["own"]), fmt(a["hold"]), fmt(a["infl"], a["infl_ci"]),
                     pf(a["infl_p"]), "—" if a["seed_sd"] is None else f"{100 * a['seed_sd']:.2f}", f"{100 * a['leak']:.1f}",
                     f"{fmt(a['own_pos_sens'])}/{fmt(a['hold_pos_sens'])}"])
        csvrows.append(["orchid", arm, a["n_runs"], a["folds"], a["own"], a["hold"], a["infl"], *a["infl_ci"], a["infl_p"],
                        a["seed_sd"], a["leak"]])
    Q.table(["arm", "runs/folds", "own BA", "hold BA", "inflation [95% CI]", "p", "seed SD", "own share w/ patient in train %",
             "OSCC sens own/hold"], rows, [6, 11, 8, 8, 27, 9, 8, 33, 18])
    write_csv("A1_orchid", ["dataset", "arm", "runs", "folds", "own_ba", "hold_ba", "inflation", "ci_lo", "ci_hi", "p",
                            "seed_sd", "leak_share"], csvrows)
    if "L" in ORC:
        h9 = ORC["L"]["infl_p"] < 0.05 and ORC["L"]["infl"] > 0
        Q.kv("H9 (confirmatory)", ("SUPPORTED" if h9 else "NOT supported") + f" · inflation {fmt(ORC['L']['infl'], ORC['L']['infl_ci'])}"
             f" points · p {pf(ORC['L']['infl_p'])}")
        A1["H9"] = dict(supported=bool(h9), infl=ORC["L"]["infl"], ci=ORC["L"]["infl_ci"], p=ORC["L"]["infl_p"])
    if "L" in ORC and "C" in ORC:
        L_, C_ = ORC["L"], ORC["C"]
        kk = sorted(set(L_["_k"]) & set(C_["_k"]))
        iL = [L_["_k"].index(k) for k in kk]
        iC = [C_["_k"].index(k) for k in kk]
        did = Q.hier([(L_["_mo"] - L_["_mh"])[iL] - (C_["_mo"] - C_["_mh"])[iC]], NB, Q.shash("orchid", "did") % 2 ** 32)[0]
        Q.kv("difference-in-differences L - C", f"{fmt(L_['infl'] - C_['infl'], Q.ci(did))} points · p {pf(Q.boot_p(did))}")
        Q.kv("holdout BA L - C", f"{100 * (L_['hold'] - C_['hold']):+.2f} points (leaky training does not buy generalisation "
             f"if this is near 0)")
        A1["orchid_did"] = dict(est=L_["infl"] - C_["infl"], ci=Q.ci(did), p=Q.boot_p(did), hold_diff=L_["hold"] - C_["hold"])
    A1["orchid"] = {k: strip(v) for k, v in ORC.items()}

# ============================================================================ 3  recovered-group arms (H10)
Q.sub("3  ARMS SPLIT BY METADATA-FREE RECOVERED GROUPS vs LEAKY (L) AND TRUE-GROUP (C) REFERENCES")
REC_ARMS = {}
rows, csvrows = [], []
E1R = REF.get("E1", {})
for ds in ("breakhis", "he_stained", "orchid", "ndb_ufes", "histopath"):
    refs = {}
    if ds == "orchid":
        for arm in ("L", "C"):
            if arm in ORC:
                refs[arm] = dict(infl=ORC[arm]["infl"], ci=ORC[arm]["infl_ci"], leak=ORC[arm]["leak"], src="X1")
    elif ds in E1R:
        for arm in ("L", "C"):
            a = E1R[ds]["arms"].get(arm)
            if a:
                jl = [j for j in ALLJ if j["spec"]["exp"] == "E1" and j["spec"]["ds"] == ds and j["spec"]["arm"] == arm
                      and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std" and j["spec"]["seed"] == 0]
                refs[arm] = dict(infl=a["infl"], ci=a["infl_ci"], leak=float(np.mean([j["leak"].get("struct_group", 0)
                                                                                      for j in jl])) if jl else None, src="E1")
    exp = "X1" if ds == "orchid" else "X2"
    arms = {}
    for arm in ("R25", "R100", "Rauto"):
        R_ = sel(exp=exp, ds=ds, arm=arm)
        if R_:
            arms[arm] = strip(arm_stats(R_, f"{ds}|{exp}|{arm}"))
    if not arms:
        continue
    kinfo = (Q.jload(Q.D["index"] / f"{ds}.recov.json", {}) or {}).get("k", {})
    REC_ARMS[ds] = dict(refs=refs, arms=arms, k=kinfo)
    for arm, v in list(refs.items()) + list(arms.items()):
        lab = arm if arm in ("L", "C") else f"{arm} (k={kinfo.get('k' + arm[1:].lower(), kinfo.get('kauto') if arm == 'Rauto' else '?')})"
        h10 = ""
        if arm == "Rauto" and ds in ("breakhis", "he_stained", "orchid"):
            h10 = "CI contains 0" if v["infl_ci"][0] <= 0 <= v["infl_ci"][1] else "CI excludes 0"
        rows.append([ds, lab, fmt(v["infl"], v.get("infl_ci", v.get("ci"))),
                     "—" if v.get("leak") is None else f"{100 * v['leak']:.1f}", h10])
        csvrows.append([ds, arm, v["infl"], *(v.get("infl_ci", v.get("ci"))), v.get("leak")])
Q.table(["dataset", "arm", "inflation [95% CI]", "own share w/ true group-mate %", "H10"], rows, [12, 18, 28, 33, 14])
print("  L/C references: E1 (three seeds) or ORCHID X1; recovered arms: seed 0, matched in training size to L and C.")
print("  For NDB-UFES and histopathology the 'true group' is the audit grouping (no patient IDs exist).")
write_csv("A1_recovered_arms", ["dataset", "arm", "inflation", "ci_lo", "ci_hi", "leak_share"], csvrows)
A1["recovered_arms"] = REC_ARMS

# ============================================================================ 4  random-label controls
Q.sub("4  RANDOM-LABEL CONTROLS  (labels drawn per group; own - holdout isolates memorised group identity)")


def rl_ceiling(j, z):
    dd = Q.DS.get(j["spec"]["ds"])
    y_rl = Q._labels(dd, j["spec"])
    hold = z["hold_row"]
    dev = np.setdiff1d(np.nonzero(dd.base)[0], hold)
    f = {c: int(np.bincount(y_rl[dev][dd.y[dev] == c], minlength=dd.K).argmax()) if (dd.y[dev] == c).any() else 0
         for c in range(dd.K)}
    pred = np.array([f[int(c)] for c in dd.y[hold]])
    yt = y_rl[hold]
    return float(np.mean([np.mean(pred[yt == k] == k) for k in range(dd.K) if (yt == k).any()]))


RLX = {}
rows = []
for exp, ds in (("X1RL", "orchid"), ("X2RL", "ndb_ufes")):
    for arm in ("L", "C", "Rauto"):
        R_ = sel(exp=exp, ds=ds, arm=arm)
        if not R_:
            continue
        to, th = tables_for(R_, "own"), tables_for(R_, "hold")
        _, mo = boot_folds(to, NB, 1)
        _, mh = boot_folds(th, NB, 2)
        own_b = mo.mean(0)
        chance = 1.0 / Q.META[ds]["K"]
        d = dict(own=float(np.mean(list(point(to).values()))), hold=float(np.mean(list(point(th).values()))),
                 own_ci=Q.ci(own_b), infl_ci=Q.ci((mo - mh).mean(0)), p_vs_chance=Q.boot_p(own_b, chance),
                 ceiling=float(np.mean([rl_ceiling(j, z) for j, _, z in R_])), n=len(R_))
        RLX[f"{exp}|{ds}|{arm}"] = d
        rows.append([exp, ds, arm, d["n"], f"{fmt(d['own'])} [{fmt(d['own_ci'][0])},{fmt(d['own_ci'][1])}]", fmt(d["hold"]),
                     fmt(d["own"] - d["hold"], d["infl_ci"]), f"{100 * chance:.1f}", fmt(d["ceiling"]), pf(d["p_vs_chance"])])
ref_rl = REF.get("RL", {})
for k in ("ndb_ufes|L", "ndb_ufes|C"):
    if k in ref_rl:
        v = ref_rl[k]
        rows.append(["RL (Q1)", "ndb_ufes", k.split("|")[1], v.get("n"), f"{fmt(v['own'])} [{fmt(v['own_ci'][0])},{fmt(v['own_ci'][1])}]",
                     fmt(v["hold"]), fmt(v["own"] - v["hold"], v["infl_ci"]), "33.3", fmt(v.get("true_class_ceiling")),
                     pf(v["p_vs_chance"])])
if rows:
    Q.table(["exp", "dataset", "arm", "runs", "own BA [CI]", "hold BA", "own - hold [CI]", "chance", "true-class ceiling",
             "p own vs chance"], rows, [9, 10, 6, 5, 24, 8, 26, 7, 20, 15])
A1["random_labels"] = RLX

# ============================================================================ 5  recovery quality and probe sweep (H8, H11)
SW = RCV.get("sweep", [])
if RCV:
    Q.sub("5  RECOVERY QUALITY (H8) AND PROBE SWEEP (H11)")
    sel_f = RCV.get("selected")
    Q.kv("feature set used for recovered groups", f"{sel_f} (rule: highest mean AMI at the automatic k)")
    rows = []
    for key, v in RCV.get("quality", {}).items():
        ds, fn = key.split("|")
        if not v.get("quality"):
            continue
        ka = str(v["kauto"])
        q = v["quality"].get(ka, {})
        best = max(v["quality"].items(), key=lambda t: t[1]["ami"])
        rows.append([ds, fn, v["kauto"], f"{q.get('ami', float('nan')):.3f}", f"{q.get('ari', float('nan')):.3f}",
                     f"{q.get('homogeneity', float('nan')):.3f}", f"{best[1]['ami']:.3f} (k={best[0]})"])
    Q.table(["dataset", "features", "k auto", "AMI", "ARI", "homogeneity", "best AMI"], rows, [12, 14, 8, 7, 7, 12, 18])
    rows, H11 = [], {}
    for ds in sorted({x["ds"] for x in SW}):
        nb = RCV["quality"].get(f"{ds}|{sel_f}", {}).get("n")
        by = defaultdict(list)
        for x in SW:
            if x["ds"] == ds:
                by[x["group"]].append(x)
        ks = sorted(int(g[1:]) for g in by if g.startswith("k"))
        inf_lr = {g: float(np.mean([a["own_lr"] - a["hold_lr"] for a in v])) for g, v in by.items()}
        inf_nn = {g: float(np.mean([a["own_nn"] - a["hold_nn"] for a in v])) for g, v in by.items()}
        lk = {g: float(np.mean([a["leak"] for a in v])) for g, v in by.items()}
        sizes = [nb / k for k in ks] if nb else ks
        rho = spearmanr(sizes, [inf_lr[f"k{k}"] for k in ks]).correlation if len(ks) >= 3 else float("nan")
        H11[ds] = dict(spearman_size_vs_inflation=float(rho), image=inf_lr.get("image"), audit=inf_lr.get("audit"),
                       curve={str(k): inf_lr[f"k{k}"] for k in ks}, curve_nn={str(k): inf_nn[f"k{k}"] for k in ks},
                       leak={g: lk[g] for g in lk}, n=nb)
        sg = lambda v: "—" if v is None else f"{100 * v:+.1f}"
        rows.append([ds, sg(inf_lr.get("image")), " ".join(f"{k}:{100 * inf_lr[f'k{k}']:+.1f}" for k in ks),
                     sg(inf_lr.get("audit")), f"{rho:+.2f}" if np.isfinite(rho) else "—"])
    Q.table(["dataset", "image-level", "recovered k: inflation (points)", "audit grouping", "Spearman(size, infl)"], rows,
            [12, 12, 70, 15, 20])
    A1["H8_quality"] = RCV.get("quality")
    A1["H11"] = H11

# ============================================================================ 6  Phikon vs DINOv2 frozen probes (H12)
FMP = RCV.get("fm_probes", [])
if FMP:
    Q.sub("6  PATHOLOGY FOUNDATION MODEL (Phikon) vs DINOv2 FROZEN PROBES ON THE E1 SPLITS (H12)")
    rows, H12 = [], {}
    for ds in sorted({x["ds"] for x in FMP}):
        for fk in sorted({x["feat"] for x in FMP}):
            line = [ds, fk]
            for arm in ("L", "C"):
                v = [x for x in FMP if x["ds"] == ds and x["feat"] == fk and x["arm"] == arm]
                if not v:
                    line += ["—", "—", "—"]
                    continue
                il = [x["own_lr"] - x["hold_lr"] for x in v]
                inn = [x["own_nn"] - x["hold_nn"] for x in v]
                H12[f"{ds}|{fk}|{arm}"] = dict(hold_lr=float(np.mean([x["hold_lr"] for x in v])), infl_lr=float(np.mean(il)),
                                               infl_lr_sd=float(np.std(il, ddof=1)) if len(il) > 1 else None,
                                               infl_nn=float(np.mean(inn)), folds=len(v))
                line += [fmt(np.mean([x["hold_lr"] for x in v])), f"{100 * np.mean(il):+.1f} ± {100 * np.std(il, ddof=1) if len(il) > 1 else 0:.1f}",
                         f"{100 * np.mean(inn):+.1f}"]
            rows.append(line)
    Q.table(["dataset", "features", "L hold BA", "L infl LR (±SD folds)", "L infl 1NN", "C hold BA", "C infl LR",
             "C infl 1NN"], rows, [12, 10, 10, 22, 11, 10, 14, 11])
    A1["H12"] = H12

# ============================================================================ 7  ORCHID confusion matrices
if sel(exp="X1", ds="orchid"):
    Q.sub("7  ORCHID CONFUSION MATRICES  (rows = true class, % of the row; all seeds, pooled over folds)")
    cls = Q.META["orchid"]["classes"]
    CM = {}
    for arm, part in (("L", "own"), ("L", "hold"), ("C", "hold")):
        R_ = sel(exp="X1", ds="orchid", arm=arm)
        if not R_:
            continue
        M = np.zeros((3, 3), int)
        for j, r, z in R_:
            np.add.at(M, (z[f"{part}_y"], z[f"{part}_logit"].argmax(1)), 1)
        rp = M / np.maximum(M.sum(1, keepdims=True), 1)
        CM[f"{arm}|{part}"] = dict(counts=M.tolist(), row_pct=rp.tolist())
        print(f"  arm {arm} · {'own fold' if part == 'own' else 'patient-disjoint holdout'} · {int(M.sum()):,} predictions")
        Q.table(["true \\ predicted"] + cls + ["recall"],
                [[cls[i]] + [f"{100 * rp[i, k]:.1f}" for k in range(3)] + [f"{100 * rp[i, i]:.1f}"] for i in range(3)],
                [18, 10, 10, 10, 10])
    A1["orchid_confusion"] = CM

# ============================================================================ 8  frozen Q1 results unchanged?
Q.sub("8  REPRODUCIBILITY CHECK  (Q1 results before and after amendment A1)")
pre = Q.jload(Q.D["eval"] / "q1_results.pre_A1.json", {}) or {}
if pre and REF:
    diffs = []
    for ds, e in pre.get("E1", {}).items():
        for arm, a in e.get("arms", {}).items():
            b = REF.get("E1", {}).get(ds, {}).get("arms", {}).get(arm)
            if b is None or abs(a["infl"] - b["infl"]) > 1e-12 or abs(a["infl_p"] - b["infl_p"]) > 1e-12:
                diffs.append(f"{ds}/{arm}")
    Q.kv("E1 inflation and p-values", "identical" if not diffs else f"CHANGED for {', '.join(diffs)}")
    A1["q1_unchanged"] = not diffs
else:
    Q.kv("E1 inflation and p-values", "snapshot or current results missing (run BLOCK 25 after BLOCK 29)")

# ============================================================================ figures
Q.sub("FIGURES")
if ORC:
    arms = [a for a in ("L", "C", "R25", "R100", "Rauto") if a in ORC]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 0.55 * len(arms) + 1.6), gridspec_kw=dict(width_ratios=[1.2, 1]))
    for i, arm in enumerate(arms):
        a = ORC[arm]
        yy = len(arms) - 1 - i
        c = COL.get(arm, COL["R"])
        a1.plot([100 * a["hold"], 100 * a["own"]], [yy, yy], color=c, lw=2, solid_capstyle="round")
        a1.scatter([100 * a["hold"]], [yy], s=40, color="white", edgecolor=c, lw=2, zorder=3)
        a1.scatter([100 * a["own"]], [yy], s=40, color=c, zorder=3, marker="o" if arm in ("L", "C") else "D")
        lo, hi = a["infl_ci"]
        a2.errorbar([100 * a["infl"]], [yy], xerr=[[100 * (a["infl"] - lo)], [100 * (hi - a["infl"])]], fmt="o" if arm in ("L", "C") else "D",
                    color=c, ms=6, lw=2, capsize=0)
    lab = {"L": "L: image-level split", "C": "C: patient-grouped", "R25": "R: recovered, k=25", "R100": "R: recovered, k=100",
           "Rauto": "R: recovered, automatic k"}
    for ax in (a1, a2):
        ax.set_yticks(range(len(arms)))
        ax.set_yticklabels([lab[a] for a in arms][::-1])
        ax.grid(axis="y", visible=False)
    a2.axvline(0, color=INK2, lw=1)
    a1.set_xlabel("Balanced accuracy (%)")
    a2.set_xlabel("Own fold − holdout (points, 95% CI)")
    a1.set_title("A  ORCHID: own fold (filled) vs patient-disjoint holdout (open)", loc="left", fontsize=9)
    a2.set_title("B  Inflation by arm", loc="left", fontsize=9)
    a2.set_yticklabels([])
    fig.tight_layout()
    save(fig, "Fig_A1_orchid.png", "ORCHID: own fold vs patient-disjoint holdout for leaky, patient-grouped and recovered-group arms")

if REC_ARMS:
    dss = [d for d in ("breakhis", "he_stained", "orchid", "ndb_ufes", "histopath") if d in REC_ARMS]
    fig, ax = plt.subplots(figsize=(7.8, 0.42 * sum(len(REC_ARMS[d]["refs"]) + len(REC_ARMS[d]["arms"]) for d in dss) + 1.2))
    y, ticks, labels = 0, [], []
    for ds in dss[::-1]:
        items = list(REC_ARMS[ds]["arms"].items())[::-1] + [(a, REC_ARMS[ds]["refs"][a]) for a in ("C", "L") if a in REC_ARMS[ds]["refs"]]
        for arm, v in items:
            ci_ = v.get("infl_ci", v.get("ci"))
            c = COL.get(arm, COL["R"])
            ax.errorbar([100 * v["infl"]], [y], xerr=[[100 * (v["infl"] - ci_[0])], [100 * (ci_[1] - v["infl"])]],
                        fmt="o" if arm in ("L", "C") else "D", color=c, ms=5, lw=1.8, capsize=0)
            ticks.append(y)
            labels.append(f"{NAME[ds]} · {arm}")
            y += 1
        y += 0.6
    ax.axvline(0, color=INK2, lw=1)
    ax.set_yticks(ticks)
    ax.set_yticklabels(labels, fontsize=7)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Inflation = own fold − clean holdout (points, 95% CI)")
    ax.set_title("Splits by metadata-free recovered groups (diamonds) vs leaky (orange) and true-group (blue) splits",
                 loc="left", fontsize=8.5)
    fig.tight_layout()
    save(fig, "Fig_A1_recovered_arms.png", "inflation of arms split by recovered groups next to leaky and true-group references")

if A1.get("H11"):
    H11 = A1["H11"]
    dss = [d for d in ("breakhis", "he_stained", "orchid", "ndb_ufes", "histopath", "clinical") if d in H11]
    nc = min(3, len(dss))
    nr = int(math.ceil(len(dss) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.1 * nr), squeeze=False)
    for ax in axs.ravel()[len(dss):]:
        ax.axis("off")
    for ax, ds in zip(axs.ravel(), dss):
        h = H11[ds]
        ks = sorted(int(k) for k in h["curve"])
        nb = h["n"] or 1
        xs = [nb / k for k in ks]
        ax.plot(xs, [100 * h["curve"][str(k)] for k in ks], color=COL["R"], marker="D", ms=4, lw=1.8, label="recovered clusters (LR)")
        ax.plot(xs, [100 * h["curve_nn"][str(k)] for k in ks], color=COL["R"], marker="D", ms=3, lw=1, ls=":",
                label="recovered clusters (1-NN)")
        if h.get("image") is not None:
            ax.axhline(100 * h["image"], color=COL["L"], lw=1.4, ls="--", label="image-level split")
        if h.get("audit") is not None:
            ax.axhline(100 * h["audit"], color=COL["C"], lw=1.4, ls="--",
                       label="true groups (audit grouping where no IDs exist)")
        ax.axhline(0, color=INK2, lw=0.8)
        ax.set_xscale("log")
        ax.set_xticks(xs)
        ax.set_xticklabels([f"{x_:.0f}" if x_ >= 10 else f"{x_:.1f}" for x_ in xs], fontsize=7)
        ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
        ax.grid(axis="x", visible=False)
        ax.set_title(NAME[ds], loc="left", fontsize=8.5)
        ax.set_xlabel("images per recovered cluster (log)", fontsize=7.5)
        ax.set_ylabel("probe inflation (points)", fontsize=7.5)
    axs[0][0].legend(fontsize=6.5, loc="best")
    fig.tight_layout()
    save(fig, "Fig_A1_recovery_sweep.png", "frozen-probe inflation as recovered clusters get coarser, with image-level and true-group references")

for n_, d_ in made:
    Q.kv(n_, d_)
Q.jsave(Q.D["eval"] / "a1_results.json", A1 | dict(meta=OUT["meta"]))
L = ["# Amendment A1 results (generated by BLOCK 33)", ""]
if "H9" in A1:
    h = A1["H9"]
    L.append(f"- H9 (ORCHID, image-level split vs patient-disjoint holdout): inflation {100 * h['infl']:+.2f} points "
             f"[{100 * h['ci'][0]:+.2f}, {100 * h['ci'][1]:+.2f}], p = {pf(h['p'])} -> {'supported' if h['supported'] else 'not supported'}")
for ds, v in REC_ARMS.items():
    for arm, a in v["arms"].items():
        L.append(f"- {NAME[ds]} {arm}: inflation {100 * a['infl']:+.2f} [{100 * a['infl_ci'][0]:+.2f}, {100 * a['infl_ci'][1]:+.2f}], "
                 f"residual true-group share {100 * a['leak']:.1f}%")
L += ["", "Figures:"] + [f"- `{n}` — {d}" for n, d in made]
(TAB / "RESULTS_A1.md").write_text("\n".join(L) + "\n")
print(f"\n[BLOCK 33 ✓]  results in q1/eval/a1_results.json · {len(made)} figures · {Q.hms(time.time() - t_all)} · run BLOCK 28 next")

In [ ]:
# ============================================================================
# BLOCK 28 — RELEASE PACKAGE: audit toolkit · fixed leakage-controlled splits · manifests · computational status
# (run last, after BLOCK 25-27; nothing is uploaded anywhere — the zip stays in /kaggle/working/q1/release/)
# ============================================================================
import os, sys, json, time, shutil, csv, hashlib, importlib, warnings, subprocess, zipfile, platform
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
Q1_ROOT = Path(os.environ.get("Q1_ROOT", "/kaggle/working/q1"))
sys.path.insert(0, str(Q1_ROOT / "lib"))
import numpy as np
import q1lib as Q
importlib.reload(Q)
Q.keep_source("28")

INCLUDE_WEIGHTS = False      # True also puts every kept weights.pt into the zip (about 110 MB each)
AGREE_MAX_PAIRS = 150000     # the toolkit re-derives near-duplicate groups for datasets with at most this many candidate pairs

TOOLKIT_SRC = {
    '.pytest_cache/README.md': r'''# pytest cache directory #

This directory contains data from the pytest's cache plugin,
which provides the `--lf` and `--ff` options, as well as the `cache` fixture.

**Do not** commit this to version control.

See [the docs](https://docs.pytest.org/en/stable/how-to/cache.html) for more information.
''',
    'README.md': r'''# oralleak

A small audit toolkit for data leakage in medical image classification datasets. It was written for the
public oral-cancer image datasets studied in the accompanying article, but it works on any folder of images.

## What it does

- **Hashes** every file (MD5, SHA-256) and computes perceptual hashes (pHash from a 32x32 DCT, dHash from
  a 9x8 gradient) on a 288-px copy of each image.
- **Finds duplicates**: identical files (same MD5) and near duplicates (pHash distance <= 8 and SSIM >= 0.90
  on a 128-px grey copy). If one near-duplicate group grows larger than max(25, 2% of the images), the
  stricter fallback rule (pHash <= 4 and SSIM >= 0.95) is used, so that textures common to many patches do
  not chain unrelated images together.
- **Recovers source groups from file order** (`segment_order`): when patches cut from one source image are
  stored consecutively, the similarity of neighbouring patches drops at source boundaries; the cut-off is an
  Otsu threshold. Embeddings are supplied by the user (any L2-normalised feature vectors).
- **Recovers source groups without any metadata** (`recover_groups`): Ward clustering of L2-normalised frozen
  image features (DINOv2, or a pathology foundation model such as Phikon), optionally with CIELAB stain
  statistics; the number of clusters is chosen automatically by the best cosine silhouette in [10, n/10].
  Class labels are never used. In the accompanying article this grouping was validated against true patient
  and slide identifiers (BreakHis, OCDC, ORCHID) before being used to split datasets that have none.
- **Builds grouped splits** (`grouped_split`, stratified group k-fold) and **checks splits**
  (`check_split`): it lists every relation (identical file, near duplicate, slide/patient/source group,
  augmented copy of the same source) that occurs on both sides of a split.
- **Reports provenance leakage** (`leak_report`): the share of test images that have a relative in the
  training set, by type of relation.

## Command line

```
pip install .
python -m oralleak audit --images /path/to/dataset --out files.csv      # class = parent folder name
python -m oralleak recover --files files.csv --emb emb.npy --out files_recov.csv    # rows of emb.npy = rows of files.csv
python -m oralleak split --files files_recov.csv --group g_recov_ctrl --k 5 --out split.csv
python -m oralleak check --files files.csv --split split.csv --test-fold 0
```

`check` exits with code 1 when any relation crosses the split, so it can be used in a CI job.

## Tests

`python -m pytest -q tests` runs unit tests on synthetic images and, when the package is inside the
release folder, checks every released split in `../splits`.

## Limits

Near-duplicate detection finds re-encoded, resized and lightly edited copies. It does not find crops of
different regions of the same slide; those need slide or patient identifiers (`g_struct`), the order-based
recovery above when file order follows the source, or the content-based recovery (`recover`). Recovered groups
are an approximation: check their agreement with true identifiers wherever some are available.
''',
    'oralleak/__init__.py': r'''"""oralleak - a small data-leakage audit toolkit for medical image classification datasets.

Functions cover file hashing, exact and near-duplicate detection (pHash + SSIM), recovery of source
groups from file order, metadata-free recovery of source groups from image content (Ward clustering of frozen
features with an automatic cluster count), grouped splitting, cross-partition checks and a provenance leakage report.
"""
from .core import (load_image, perceptual_hashes, hamming_pairs, gray_small, near_duplicate_groups,
                   union_labels, otsu, segment_order, recover_groups, grouped_split, check_split, leak_report,
                   audit_folder, read_csv, write_csv)

__version__ = "0.2.0"
''',
    'oralleak/__main__.py': r'''"""Command line:  python -m oralleak {audit,recover,split,check} ...

  audit    --images DIR --out files.csv          hash every image, find exact and near duplicates
  recover  --files files.csv --emb emb.npy --out files_recov.csv [--stain stain.npy] [--k K]
           metadata-free source groups from image features (rows of emb.npy follow files.csv); adds g_recov and
           g_recov_ctrl (= g_recov united with g_near), which can then be passed to split --group g_recov_ctrl
  split  --files files.csv --group g_near --k 5  stratified group k-fold (writes split.csv: relpath,fold)
  check  --files files.csv --split split.csv     cross-partition checks + leakage report; exit code 1 on leakage
         (split.csv needs relpath and role (train/test) columns, or relpath and fold with --test-fold F)
"""
import argparse
import json
import sys

from . import core


def main(argv=None):
    ap = argparse.ArgumentParser(prog="oralleak")
    sp = ap.add_subparsers(dest="cmd", required=True)
    a = sp.add_parser("audit")
    a.add_argument("--images", required=True)
    a.add_argument("--out", required=True)
    s = sp.add_parser("split")
    s.add_argument("--files", required=True)
    s.add_argument("--group", default="g_ctrl")
    s.add_argument("--label", default="label")
    s.add_argument("--k", type=int, default=5)
    s.add_argument("--seed", type=int, default=7)
    s.add_argument("--out", required=True)
    r = sp.add_parser("recover")
    r.add_argument("--files", required=True)
    r.add_argument("--emb", required=True)
    r.add_argument("--stain", default=None)
    r.add_argument("--k", type=int, default=None)
    r.add_argument("--out", required=True)
    c = sp.add_parser("check")
    c.add_argument("--files", required=True)
    c.add_argument("--split", required=True)
    c.add_argument("--test-fold", type=int, default=None)
    c.add_argument("--train-roles", default="train")
    c.add_argument("--test-roles", default="test")
    ar = ap.parse_args(argv)

    if ar.cmd == "audit":
        rows, info = core.audit_folder(ar.images)
        core.write_csv(ar.out, rows)
        print(json.dumps(info, indent=1, default=str))
        return 0
    files = core.read_csv(ar.files)
    if ar.cmd == "recover":
        import numpy as np
        emb = np.load(ar.emb)
        if len(emb) != len(files):
            raise SystemExit(f"{ar.emb} has {len(emb)} rows, {ar.files} has {len(files)}")
        stain = np.load(ar.stain) if ar.stain else None
        lab, info = core.recover_groups(emb, k=ar.k, stain=stain)
        near = [r.get("g_near", str(i)) for i, r in enumerate(files)]
        ctrl = core.union_labels(len(files), label_arrays=[np.unique(near, return_inverse=True)[1], lab])
        for r, a, b in zip(files, lab, ctrl):
            r["g_recov"], r["g_recov_ctrl"] = int(a), int(b)
        core.write_csv(ar.out, files)
        print(json.dumps(info, indent=1))
        return 0
    if ar.cmd == "split":
        key = ar.group if ar.group in files[0] else "g_near"
        fold = core.grouped_split([r[ar.label] for r in files], [r[key] for r in files], ar.k, ar.seed)
        core.write_csv(ar.out, [dict(relpath=r["relpath"], fold=int(f)) for r, f in zip(files, fold)])
        print(f"{len(files)} images -> {ar.k} folds grouped by {key}")
        return 0
    sp_rows = core.read_csv(ar.split)
    if ar.test_fold is not None:
        roles = {r["relpath"]: ("test" if int(r["fold"]) == ar.test_fold else "train") for r in sp_rows}
        trr, ter = ("train",), ("test",)
    else:
        roles = {r["relpath"]: r["role"] for r in sp_rows}
        trr, ter = tuple(ar.train_roles.split(",")), tuple(ar.test_roles.split(","))
    bad = core.check_split(files, roles, trr, ter)
    rep = core.leak_report(files, roles, trr, ter)
    print(json.dumps(dict(violations=bad, leakage=rep), indent=1))
    return 1 if bad else 0


if __name__ == "__main__":
    sys.exit(main())
''',
    'oralleak/core.py': r'''"""Core functions of oralleak.  Dependencies: numpy, scipy, pillow, scikit-image, scikit-learn."""
import csv
import hashlib
import io
import os
from collections import Counter, defaultdict

import numpy as np

STORE_PX = 288          # every image is decoded once and resized to this square size
IMG_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
NEAR_RULE = dict(phash_max=8, ssim_min=0.90, cap_frac=0.02, cap_min=25, fallback_phash=4, fallback_ssim=0.95)


# ----------------------------------------------------------------------------- files
def load_image(path, px=STORE_PX):
    """(md5, sha256, uint8 array px x px x 3).  JPEG files are decoded at reduced size first."""
    from PIL import Image
    Image.MAX_IMAGE_PIXELS = None
    b = open(path, "rb").read()
    md5, sha = hashlib.md5(b).hexdigest(), hashlib.sha256(b).hexdigest()
    im = Image.open(io.BytesIO(b))
    if im.format == "JPEG":
        im.draft("RGB", (px * 2, px * 2))
    im = im.convert("RGB").resize((px, px), Image.BILINEAR, reducing_gap=2.0)
    return md5, sha, np.asarray(im, dtype=np.uint8).copy()


def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path, rows, header=None):
    header = header or (list(rows[0].keys()) if rows else [])
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=header)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in header})


# ----------------------------------------------------------------------------- perceptual hashes
_POP8 = np.array([bin(i).count("1") for i in range(256)], np.uint8)


def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    return _POP8[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)


def _gray(x):
    x = np.asarray(x, np.float32)
    return 0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2]


def _adaptive_mean(g, oh, ow):
    """Same bins as torch.nn.functional.adaptive_avg_pool2d.  g: [N, H, W]."""
    n, h, w = g.shape
    hs = np.array([(i * h) // oh for i in range(oh)])
    he = np.array([-(-((i + 1) * h) // oh) for i in range(oh)])
    ws = np.array([(j * w) // ow for j in range(ow)])
    we = np.array([-(-((j + 1) * w) // ow) for j in range(ow)])
    c = np.zeros((n, h + 1, w + 1), np.float64)
    c[:, 1:, 1:] = g.astype(np.float64).cumsum(1).cumsum(2)
    tot = c[:, he][:, :, we] - c[:, hs][:, :, we] - c[:, he][:, :, ws] + c[:, hs][:, :, ws]
    return (tot / ((he - hs)[:, None] * (we - ws)[None, :])).astype(np.float32)


def perceptual_hashes(arr, bs=256):
    """pHash (32x32 DCT, 8x8 low band vs median) and dHash (9x8 gradient) as uint64 arrays."""
    from scipy.fft import dctn
    n = len(arr)
    ph, dh = np.zeros(n, np.uint64), np.zeros(n, np.uint64)
    w64 = (1 << np.arange(64, dtype=np.uint64)).astype(np.uint64)
    for s in range(0, n, bs):
        g = _gray(arr[s:s + bs])
        g32 = _adaptive_mean(g, 32, 32)
        c = dctn(g32, type=2, norm="ortho", axes=(1, 2))[:, :8, :8].reshape(len(g32), 64)
        med = np.median(c[:, 1:], axis=1, keepdims=True)
        ph[s:s + bs] = ((c > med).astype(np.uint64) * w64).sum(1).astype(np.uint64)
        g98 = _adaptive_mean(g, 8, 9)
        bits = (g98[:, :, 1:] > g98[:, :, :-1]).reshape(len(g98), 64).astype(np.uint64)
        dh[s:s + bs] = (bits * w64).sum(1).astype(np.uint64)
    return ph, dh


def hamming_pairs(h, max_d=8, chunk=512):
    """All pairs i < j with Hamming distance <= max_d."""
    h = np.asarray(h, np.uint64)
    I, J, Dd = [], [], []
    for s in range(0, len(h), chunk):
        d = popcount64(h[s:s + chunk, None] ^ h[None, :])
        ii, jj = np.nonzero(d <= max_d)
        keep = jj > ii + s
        ii, jj = ii[keep], jj[keep]
        I.append(ii + s)
        J.append(jj)
        Dd.append(d[ii, jj])
    if not I:
        return np.zeros(0, int), np.zeros(0, int), np.zeros(0, int)
    return np.concatenate(I), np.concatenate(J), np.concatenate(Dd)


def gray_small(arr, px=128):
    return _adaptive_mean(_gray(arr), px, px)


# ----------------------------------------------------------------------------- grouping
def union_labels(n, pairs=(), label_arrays=()):
    """Connected components over n records: explicit pairs plus 'same value in any label array' (-1 = none)."""
    p = np.arange(n)

    def find(a):
        r = a
        while p[r] != r:
            r = p[r]
        while p[a] != r:
            p[a], a = r, p[a]
        return r

    def union(a, b):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb:
            lo, hi = (ra, rb) if ra < rb else (rb, ra)
            p[hi] = lo

    for a, b in pairs:
        union(a, b)
    for lab in label_arrays:
        first = {}
        for i in range(n):
            v = lab[i]
            if v is None or v == -1 or v == "":
                continue
            if v in first:
                union(i, first[v])
            else:
                first[v] = i
    return np.array([find(i) for i in range(n)])


def near_duplicate_groups(md5, ph, grays, rule=None):
    """Group records that are identical files (MD5) or near duplicates (pHash <= phash_max AND SSIM >= ssim_min).
    If the largest group exceeds max(cap_min, cap_frac * n) the stricter fallback rule is used.
    Returns (group id per record, info dict with pair counts)."""
    from skimage.metrics import structural_similarity
    rule = dict(NEAR_RULE, **(rule or {}))
    n = len(md5)
    ii, jj, dd = hamming_pairs(ph, max(rule["phash_max"], rule["fallback_phash"]))
    ss = np.array([structural_similarity(grays[a], grays[b], data_range=255.0) for a, b in zip(ii, jj)])
    by = defaultdict(list)
    for i, m in enumerate(md5):
        by[m].append(i)
    ex = [(g[0], x) for g in by.values() if len(g) > 1 for x in g[1:]]
    info = dict(exact_groups=sum(len(g) > 1 for g in by.values()), exact_redundant=sum(len(g) - 1 for g in by.values()),
                pairs_phash_le={t: int((dd <= t).sum()) for t in (0, 4, 8)},
                pairs_ssim_ge={t: int((ss >= t).sum()) for t in (0.90, 0.95)})
    m = (dd <= rule["phash_max"]) & (ss >= rule["ssim_min"]) if len(dd) else np.zeros(0, bool)
    roots = union_labels(n, pairs=list(zip(ii[m], jj[m])) + ex)
    big = max(Counter(roots.tolist()).values()) if n else 1
    cap = max(rule["cap_min"], rule["cap_frac"] * n)
    info["rule"] = "main"
    if big > cap:
        m = (dd <= rule["fallback_phash"]) & (ss >= rule["fallback_ssim"])
        roots = union_labels(n, pairs=list(zip(ii[m], jj[m])) + ex)
        info["rule"] = f"fallback (largest group {big} > cap {cap:.0f})"
    info["near_pairs"] = int(m.sum())
    info["redundant"] = int(n - len(set(roots.tolist())))
    return roots, info


def otsu(v, bins=256):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) < 4 or v.max() - v.min() < 1e-9:
        return float(np.median(v)) if len(v) else 0.5
    hist, edges = np.histogram(v, bins=bins)
    cen = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    s0 = np.cumsum(hist * cen)
    mu0 = s0 / np.maximum(w0, 1)
    mu1 = (s0[-1] - s0) / np.maximum(w1, 1)
    k = int(np.argmax((w0 * w1 * (mu0 - mu1) ** 2)[:-1]))
    return float(edges[k + 1])


def segment_order(order, labels, emb):
    """Recover source groups from file order: cut an ordered list wherever the class changes or the cosine
    similarity of consecutive L2-normalised embeddings drops below an Otsu threshold."""
    order = np.asarray(order)
    labels = np.asarray(labels)
    e = np.asarray(emb, np.float32)[order]
    s = (e[:-1] * e[1:]).sum(1)
    same = labels[order][:-1] == labels[order][1:]
    tau = otsu(s[same])
    seg_pos = np.concatenate([[0], np.cumsum((~same) | (s < tau))])
    seg = np.full(len(labels), -1)
    seg[order] = seg_pos
    return seg, dict(tau=tau, n_segments=int(seg_pos.max() + 1))


# ----------------------------------------------------------------------------- metadata-free group recovery
K_GRID = (10, 15, 25, 40, 60, 100, 150, 250, 400)


def recover_groups(emb, k=None, k_grid=K_GRID, k_min=10, k_max_frac=0.10, sample=3000, seed=0, stain=None):
    """Recover source groups (patients, slides) from image content alone, without any metadata.

    Ward clustering of L2-normalised frozen image features (for example DINOv2 or a pathology foundation model).
    When k is None it is chosen automatically: the grid value in [k_min, n * k_max_frac] (and <= n/3) with the
    highest cosine silhouette, computed on at most `sample` images drawn with `seed`.  Optional per-image stain
    statistics (n x d, e.g. CIELAB channel means and standard deviations) are z-scored, scaled by 1/sqrt(d) and
    appended to the features.  Class labels are never used.  Union the result with the identical-file and
    near-duplicate groups before splitting (column g_recov_ctrl of the `recover` command).

    Returns (labels 1..k as an int array, info dict with k, silhouette per k and n)."""
    from scipy.cluster.hierarchy import linkage, fcluster
    from sklearn.metrics import silhouette_score
    X = np.asarray(emb, np.float32)
    X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-8)
    if stain is not None:
        st = np.asarray(stain, np.float32)
        st = (st - st.mean(0)) / np.maximum(st.std(0), 1e-6) / float(np.sqrt(st.shape[1]))
        X = np.concatenate([X, st.astype(np.float32)], 1)
    n = len(X)
    ks = [int(k)] if k is not None else [int(kk) for kk in k_grid if kk <= n / 3]
    if not ks:
        raise ValueError(f"no cluster count in the grid fits {n} images (need k <= n/3)")
    Z = linkage(X.astype(np.float64), method="ward")
    labs = {kk: fcluster(Z, t=kk, criterion="maxclust") for kk in ks}
    sil = {}
    if k is None:
        g = np.random.default_rng(seed)
        sub = np.sort(g.choice(n, min(n, sample), replace=False))
        sil = {kk: float(silhouette_score(X[sub], labs[kk][sub], metric="cosine"))
               for kk in ks if k_min <= kk <= max(k_min, n * k_max_frac) and len(np.unique(labs[kk][sub])) > 1}
        kbest = max(sil, key=sil.get) if sil else ks[0]
    else:
        kbest = int(k)
    return np.asarray(labs[kbest], np.int64), dict(k=int(kbest), n=int(n), silhouette={str(a): b for a, b in sil.items()})


# ----------------------------------------------------------------------------- splits and checks
def grouped_split(labels, groups, k=5, seed=7):
    """Stratified group k-fold: fold id per record, no group in two folds."""
    from sklearn.model_selection import StratifiedGroupKFold
    labels, groups = np.asarray(labels), np.asarray(groups)
    fold = np.full(len(labels), -1)
    for f, (_, te) in enumerate(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(
            np.zeros(len(labels)), labels, groups)):
        fold[te] = f
    return fold


def _col(rows, key):
    return [r.get(key, "") for r in rows]


def check_split(files, roles, train_roles=("train",), test_roles=("test",),
                keys=("md5", "g_near", "g_struct", "g_ctrl", "parent_group")):
    """Cross-partition checks.  files: list of dicts (relpath + key columns).  roles: dict relpath -> role.
    Returns a list of (key, number of test images whose key value also occurs in training); empty = clean."""
    tr = [r for r in files if roles.get(r["relpath"]) in train_roles]
    te = [r for r in files if roles.get(r["relpath"]) in test_roles]
    out = []
    for k in keys:
        if not files or k not in files[0]:
            continue
        s = {v for v in _col(tr, k) if v not in ("", "-1")}
        bad = sum(1 for v in _col(te, k) if v in s)
        if bad:
            out.append((k, bad))
    return out


def leak_report(files, roles, train_roles=("train",), test_roles=("test",)):
    """Provenance leakage score: share of test images with a relative in training, by relation type."""
    tr = [r for r in files if roles.get(r["relpath"]) in train_roles]
    te = [r for r in files if roles.get(r["relpath"]) in test_roles]
    if not te:
        return {}
    rep = {}
    for name, k in (("identical_file", "md5"), ("near_duplicate", "g_near"), ("same_slide_patient_source", "g_struct"),
                    ("any_relative", "g_ctrl")):
        if k in te[0]:
            s = {v for v in _col(tr, k) if v not in ("", "-1")}
            rep[name] = float(np.mean([v in s for v in _col(te, k)]))
    rep["n_train"], rep["n_test"] = len(tr), len(te)
    return rep


# ----------------------------------------------------------------------------- folder audit
def audit_folder(root, label_from_parent=True, px=STORE_PX, rule=None, log=print):
    """Hash and group every image under root.  The class label is the name of the parent folder."""
    paths = sorted(os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs
                   if os.path.splitext(f)[1].lower() in IMG_EXT)
    if not paths:
        raise SystemExit(f"no images under {root}")
    md5, sha, arr, bad = [], [], np.zeros((len(paths), px, px, 3), np.uint8), []
    for i, p in enumerate(paths):
        try:
            a, b, x = load_image(p, px)
            md5.append(a); sha.append(b); arr[i] = x
        except Exception as e:
            md5.append(f"unreadable-{i}"); sha.append(""); bad.append((p, str(e)[:80]))
        if log and (i + 1) % 2000 == 0:
            log(f"  decoded {i + 1}/{len(paths)}")
    ph, dh = perceptual_hashes(arr)
    grays = gray_small(arr)
    near, info = near_duplicate_groups(md5, ph, grays, rule)
    exact = union_labels(len(paths), label_arrays=[md5])
    rows = []
    for i, p in enumerate(paths):
        rows.append(dict(relpath=os.path.relpath(p, root), label=os.path.basename(os.path.dirname(p)) if label_from_parent else "",
                         md5=md5[i], sha256=sha[i], phash=f"{int(ph[i]):016x}", dhash=f"{int(dh[i]):016x}",
                         g_exact=int(exact[i]), g_near=int(near[i])))
    info.update(n=len(paths), unreadable=len(bad))
    return rows, info
''',
    'pyproject.toml': r'''[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "oralleak"
version = "0.2.0"
description = "Data-leakage audit for medical image classification datasets: hashing, near duplicates, group recovery, split checks"
requires-python = ">=3.9"
dependencies = ["numpy", "scipy", "pillow", "scikit-image", "scikit-learn"]

[tool.setuptools]
packages = ["oralleak"]
''',
    'tests/test_oralleak.py': r'''"""pytest tests for oralleak.  The last two tests also check the released splits when they are present."""
import glob
import os

import numpy as np
import pytest

import oralleak as ol

HERE = os.path.dirname(os.path.abspath(__file__))
SPLITS = os.path.join(HERE, "..", "..", "splits")
# relations that must never cross partitions in the released splits (see splits/README.md)
KEYS = ("md5", "g_near", "g_struct", "parent_group")


def _img(seed, size=160):
    g = np.random.default_rng(seed)
    base = g.integers(0, 255, (8, 8, 3)).astype(np.uint8)
    return np.kron(base, np.ones((size // 8, size // 8, 1), np.uint8))


@pytest.fixture()
def folder(tmp_path):
    from PIL import Image
    for c in ("a", "b"):
        (tmp_path / c).mkdir()
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1.png")
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1_copy.png")                  # identical file content
    Image.fromarray(_img(1)).save(tmp_path / "a" / "x1_q90.jpg", quality=90)       # re-encoded near duplicate
    Image.fromarray(_img(2)).save(tmp_path / "a" / "x2.png")
    Image.fromarray(_img(3)).save(tmp_path / "b" / "y1.png")
    Image.fromarray(_img(4)).save(tmp_path / "b" / "y2.png")
    return tmp_path


def test_audit_groups(folder):
    rows, info = ol.audit_folder(str(folder), log=None)
    g = {r["relpath"]: r["g_near"] for r in rows}
    assert g["a/x1.png"] == g["a/x1_copy.png"] == g["a/x1_q90.jpg"]
    assert len({g["a/x2.png"], g["b/y1.png"], g["b/y2.png"], g["a/x1.png"]}) == 4
    assert info["exact_groups"] == 1 and info["redundant"] == 2


def test_check_flags_leak_and_passes_clean(folder):
    rows, _ = ol.audit_folder(str(folder), log=None)
    for r in rows:
        r["g_ctrl"] = r["g_near"]
    leaky = {r["relpath"]: "train" for r in rows}
    leaky["a/x1_copy.png"] = "test"
    assert ol.check_split(rows, leaky), "a duplicate across partitions must be reported"
    assert ol.leak_report(rows, leaky)["identical_file"] == 1.0
    fold = ol.grouped_split([r["label"] for r in rows], [r["g_ctrl"] for r in rows], k=2, seed=0)
    clean = {r["relpath"]: ("test" if f == 0 else "train") for r, f in zip(rows, fold)}
    assert ol.check_split(rows, clean) == []


def test_segment_order_recovers_blocks():
    g = np.random.default_rng(0)
    centers = g.normal(size=(6, 32))
    emb = np.concatenate([c + 0.05 * g.normal(size=(10, 32)) for c in centers])
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    seg, info = ol.segment_order(np.arange(60), np.zeros(60, int), emb)
    assert info["n_segments"] == 6
    assert all(len(set(seg[i * 10:(i + 1) * 10])) == 1 for i in range(6))


def _blobs(n_groups=12, per=20, dim=32, noise=0.05, seed=0):
    g = np.random.default_rng(seed)
    centers = g.normal(size=(n_groups, dim))
    emb = np.concatenate([c + noise * g.normal(size=(per, dim)) for c in centers])
    return emb, np.repeat(np.arange(n_groups), per)


def test_recover_groups_finds_sources_without_metadata():
    from sklearn.metrics import adjusted_mutual_info_score
    emb, truth = _blobs()
    lab, info = ol.recover_groups(emb, k_grid=(6, 12, 24, 48), k_min=2, k_max_frac=0.25)
    assert info["k"] == 12
    assert adjusted_mutual_info_score(truth, lab) > 0.99
    fixed, info2 = ol.recover_groups(emb, k=6)
    assert info2["k"] == 6 and len(set(fixed.tolist())) == 6
    # a split on recovered groups keeps every true source on one side
    fold = ol.grouped_split(truth % 2, lab, k=3, seed=0)
    assert all(len(set(fold[truth == t].tolist())) == 1 for t in range(12))


def test_cli_recover(tmp_path):
    from oralleak.__main__ import main
    emb, truth = _blobs(n_groups=12, per=20)
    rows = [dict(relpath=f"img{i}.png", label=str(int(t % 2)), g_near=str(i)) for i, t in enumerate(truth)]
    ol.write_csv(str(tmp_path / "files.csv"), rows)
    np.save(tmp_path / "emb.npy", emb)
    assert main(["recover", "--files", str(tmp_path / "files.csv"), "--emb", str(tmp_path / "emb.npy"), "--k", "12",
                 "--out", str(tmp_path / "rec.csv")]) == 0
    out = ol.read_csv(str(tmp_path / "rec.csv"))
    assert len({r["g_recov"] for r in out}) == 12
    assert all(len({out[i]["g_recov_ctrl"] for i in np.nonzero(truth == t)[0]}) == 1 for t in range(12))


@pytest.mark.skipif(not glob.glob(os.path.join(SPLITS, "*_files.csv")), reason="released splits not present")
def test_released_holdouts_are_clean():
    for fcsv in sorted(glob.glob(os.path.join(SPLITS, "*_files.csv"))):
        ds = os.path.basename(fcsv)[:-len("_files.csv")]
        files = ol.read_csv(fcsv)
        for scsv in sorted(glob.glob(os.path.join(SPLITS, f"{ds}_controlled_*.csv"))):
            rows = ol.read_csv(scsv)
            for key in sorted({(r["rep"], r["outer"]) for r in rows}):
                roles = {r["relpath"]: r["role"] for r in rows if (r["rep"], r["outer"]) == key}
                assert ol.check_split(files, roles, ("train", "ival", "own"), ("hold",), KEYS) == [], (scsv, key)


@pytest.mark.skipif(not glob.glob(os.path.join(SPLITS, "*_files.csv")), reason="released splits not present")
def test_released_controlled_arms_are_clean():
    for scsv in sorted(glob.glob(os.path.join(SPLITS, "*_controlled_*.csv"))):
        ds = os.path.basename(scsv).split("_controlled_")[0]
        files = ol.read_csv(os.path.join(SPLITS, f"{ds}_files.csv"))
        rows = ol.read_csv(scsv)
        for key in sorted({(r["rep"], r["outer"]) for r in rows}):
            roles = {r["relpath"]: r["role"] for r in rows if (r["rep"], r["outer"]) == key}
            assert ol.check_split(files, roles, ("train",), ("own", "ival"), KEYS) == [], (scsv, key)
''',
}

t_all = time.time()
P = Q.plan()
BB0 = P["primary_backbone"]
Q.hdr("BLOCK 28 — RELEASE PACKAGE + COMPUTATIONAL STATUS")
stamp = time.strftime("%Y%m%d_%H%M", time.gmtime())
PKG = Q.D["release"] / "package"
if PKG.exists():
    shutil.rmtree(PKG)                     # the package folder is regenerated from q1/ every time
for sub in ("toolkit", "splits", "manifests", "results/eval", "results/tables", "results/figures", "plan", "code"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)
jobs = (Q.jload(Q.D["jobs"] / "jobs.json", {}) or {}).get("jobs", [])
R = Q.jload(Q.D["eval"] / "q1_results.json", {}) or {}
CL = Q.jload(Q.D["eval"] / "clinical.json", {}) or {}
AU = Q.jload(Q.D["index"] / "audit.json", {}) or {}
A1R = Q.jload(Q.D["eval"] / "a1_results.json", {}) or {}
RCV = Q.jload(Q.D["eval"] / "recovery.json", {}) or {}
AUX = Q.jload(Q.D["index"] / "audit_ext.json", {}) or {}
avail = Q.available_datasets()
if not avail:
    raise SystemExit("no dataset index found - run BLOCK 16 first")
EXT = [d for d in Q.ext_datasets() if d not in Q.DATASETS]          # amendment A1 (ORCHID)
ALL = avail + EXT
SPLIT_EXP = {d: ("X1" if d == "orchid" else "E1") for d in ALL}


def relpath_of(p):
    """Paths of the Kaggle datasets are relative to the input root; ORCHID members keep their zip path."""
    return p.split(":", 1)[1] if p.startswith("zenodo:") else os.path.relpath(p, Q.INPUT)


def wcsv(path, header, rows):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)


# ------------------------------------------------------------------ 1  toolkit
Q.sub("1  AUDIT TOOLKIT (oralleak): hashing · near duplicates · group recovery · split checks · leakage report")
for rel, src in TOOLKIT_SRC.items():
    f = PKG / "toolkit" / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(src)
sys.path.insert(0, str(PKG / "toolkit"))
for m in [k for k in list(sys.modules) if k == "oralleak" or k.startswith("oralleak.")]:
    del sys.modules[m]
import oralleak as OL
Q.kv("package", f"oralleak {OL.__version__} · {len(TOOLKIT_SRC)} files in release/package/toolkit")
Q.kv("command line", "python -m oralleak audit | recover | split | check   (see toolkit/README.md)")

# ------------------------------------------------------------------ 2  toolkit agrees with the pipeline?
Q.sub("2  TOOLKIT vs PIPELINE: does the stand-alone code reproduce the hashes and the near-duplicate groups of BLOCK 16?")
rows, AGREE = [], {}
for ds in sorted(ALL, key=lambda d: Q.DS.get(d).n):
    dd = Q.DS.get(ds)
    st = Q.open_store(ds)
    H = np.load(Q.D["store"] / f"{ds}_hash.npz")
    use = np.nonzero(dd.ok & dd.orig)[0]
    ph, dh, gr = [], [], []
    for s in range(0, len(use), 512):
        blk = np.asarray(st[use[s:s + 512]])
        a, b = OL.perceptual_hashes(blk)
        ph.append(a); dh.append(b)
    ph, dh = np.concatenate(ph), np.concatenate(dh)
    eq_p = float((ph == H["ph"][use]).mean())
    eq_d = float((dh == H["dh"][use]).mean()) if "dh" in H.files else float("nan")
    ncand = len(OL.hamming_pairs(ph, 8)[0])
    verdict, red = f"skipped ({ncand:,} candidate pairs)", "—"
    if ncand <= AGREE_MAX_PAIRS:
        gr = np.concatenate([OL.gray_small(np.asarray(st[use[s:s + 512]])) for s in range(0, len(use), 512)])
        roots, info = OL.near_duplicate_groups(dd.md5[use], ph, gr)
        g = dd.g_near[use]
        same = len(set(zip(roots.tolist(), g.tolist()))) == len(set(roots.tolist())) == len(set(g.tolist()))
        verdict = "identical partition" if same else "DIFFERENT partition"
        red = f"{info['redundant']} / {len(use) - len(set(g.tolist()))}"
        AGREE[ds] = dict(phash_equal=eq_p, dhash_equal=eq_d, partition_identical=bool(same), rule=info["rule"],
                         redundant_toolkit=info["redundant"], redundant_pipeline=int(len(use) - len(set(g.tolist()))))
    else:
        AGREE[ds] = dict(phash_equal=eq_p, dhash_equal=eq_d, partition_identical=None, candidate_pairs=ncand)
    rows.append([ds, len(use), f"{100 * eq_p:.2f}%", f"{100 * eq_d:.2f}%", verdict, red])
Q.table(["dataset", "originals", "pHash equal", "dHash equal", "near-duplicate groups", "redundant toolkit/pipeline"], rows,
        [12, 10, 12, 12, 34, 26])

# the toolkit's recover_groups must reproduce the recovered groups of BLOCK 31 (amendment A1) from the same features
REC_AGREE = {}
EMB = (AU.get("_meta") or {}).get("embed", "dinov2_s")
rows = []
for ds in ALL:
    rj = Q.jload(Q.D["index"] / f"{ds}.recov.json")
    if not rj or "kauto" not in rj.get("groups", {}):
        continue
    dd = Q.DS.get(ds)
    base = np.nonzero(dd.base)[0]
    fname = rj["feature"]
    bname = fname.split("+")[0]
    fpath = Q.D["store"] / f"{ds}_emb_{EMB if bname == 'dinov2_s' else bname}.npy"
    if not fpath.exists():
        continue
    emb = np.load(fpath)[base]
    stain = np.load(Q.D["store"] / f"{ds}_stain.npy")[base] if fname.endswith("+stain") else None
    lab, inf_ = OL.recover_groups(emb, stain=stain)
    ref = dd.extra["kauto"][base]
    same = inf_["k"] == rj["k"]["kauto"] and len(set(zip(lab.tolist(), ref.tolist()))) == len(set(lab.tolist())) == len(set(ref.tolist()))
    REC_AGREE[ds] = dict(feature=fname, k_toolkit=inf_["k"], k_pipeline=rj["k"]["kauto"], partition_identical=bool(same))
    rows.append([ds, fname, len(base), f"{inf_['k']} / {rj['k']['kauto']}", "identical partition" if same else "DIFFERENT partition"])
if rows:
    Q.table(["dataset", "features", "images", "k toolkit / pipeline", "recovered groups"], rows, [12, 14, 8, 22, 22])

# ------------------------------------------------------------------ 3  file manifests with SHA-256
Q.sub("3  FILE MANIFESTS AND HASH LISTS  (relative path, class, MD5, SHA-256, every group label used by the experiments)")
cache_p = Q.D["release"] / "sha256_cache.json"
cache = Q.jload(cache_p, {}) or {}


def sha_one(p):
    try:
        s_ = os.stat(p)
        key = f"{p}|{s_.st_size}|{int(s_.st_mtime)}"
        if key in cache:
            return key, cache[key]
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for b in iter(lambda: f.read(1 << 20), b""):
                h.update(b)
        return key, h.hexdigest()
    except Exception:
        return None, ""


FILES, REL = {}, {}
rows = []
t0 = time.time()
for ds in ALL:
    dd = Q.DS.get(ds)
    paths = [r["path"] for r in dd.R]
    if ds in EXT:
        res = [(None, r.get("sha256", "")) for r in dd.R]           # hashed from the downloaded bytes in BLOCK 30
    else:
        with ThreadPoolExecutor(16) as ex:
            res = list(ex.map(sha_one, paths))
    for k, h in res:
        if k:
            cache[k] = h
    rel = [relpath_of(p) for p in paths]
    REL[ds] = rel
    cls = Q.META[ds]["classes"]
    F_ = []
    for i, r in enumerate(dd.R):
        par = int(dd.parent[i])
        F_.append(dict(relpath=rel[i], label=cls[int(dd.y[i])], label_id=int(dd.y[i]), md5=dd.md5[i], sha256=res[i][1],
                       readable=int(dd.ok[i]), original=int(dd.orig[i]), used_in_experiments=int(dd.base[i]),
                       parent_relpath=rel[par] if par >= 0 and dd.child_ok[i] else "", parent_group=int(dd.parent_link[i]),
                       g_exact=int(dd.g_exact[i]), g_near=int(dd.g_near[i]), g_struct=int(dd.g_struct[i]),
                       g_coarse=int(dd.g_coarse[i]), g_ctrl=int(dd.g_ctrl[i]), slide_or_patient=dd.slide[i],
                       native_split=r.get("native", ""),
                       **{f"g_recov_{k}": int(v[i]) for k, v in sorted(dd.extra.items())}))
    FILES[ds] = F_
    OL.write_csv(PKG / "splits" / f"{ds}_files.csv", F_)
    rows.append([ds, len(F_), int(dd.base.sum()), len(set(dd.g_ctrl[dd.base].tolist())), sum(1 for x in F_ if x["sha256"])])
Q.jsave(cache_p, cache)
Q.table(["dataset", "files", "used", "groups (g_ctrl)", "SHA-256 computed"], rows, [12, 8, 8, 16, 18])
Q.kv("hashing time", Q.hms(time.time() - t0))
cx = AU.get("cross")
if cx:
    Q.jsave(PKG / "manifests" / "cross_dataset_near_duplicates.json", cx)
for f in [Q.D["index"] / "audit_ext.json", Q.D["index"] / "_registry.json"] + sorted(Q.D["index"].glob("*.recov.json")):
    if f.exists():
        shutil.copy2(f, PKG / "manifests" / f.name)

# ------------------------------------------------------------------ 4  fixed leakage-controlled splits
Q.sub("4  FIXED LEAKAGE-CONTROLLED SPLITS  (controlled arm C of E1 [ORCHID: X1], seed 0, every outer fold) + toolkit checks")
KEYS = ("md5", "g_near", "g_struct", "parent_group")
rows, SPL = [], {}
for ds in ALL:
    dd = Q.DS.get(ds)
    sch = P["outer"][ds]
    rel = REL[ds]
    fold_rows, long_rows = [], []
    viol_h = viol_c = 0
    leak_any = []
    nsplit = 0
    for rep in P["reps"].get(ds, P["reps"]["default"]):
        fold, _ = Q.outer_folds(dd, dict(sch, rep=rep))
        fold_rows += [[rel[i], rep, int(fold[i])] for i in np.nonzero(fold >= 0)[0]]
        for o in range(Q.n_outer(dd, dict(sch, rep=rep))):
            J = [j for j in jobs if j["spec"]["exp"] == SPLIT_EXP[ds] and j["spec"]["ds"] == ds and j["spec"]["arm"] == "C"
                 and j["spec"].get("rep", 0) == rep and j["spec"]["outer"] == o and j["spec"]["seed"] == 0
                 and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std"]
            if not J:
                continue
            B = Q.build_job(dict(J[0]["spec"], xeval=False), check=True)   # the pipeline's own disjointness assertions
            roles = {}
            for part in ("train", "ival", "own", "hold"):
                for i in np.unique(B[part]["row"]):
                    roles[rel[i]] = part
                    long_rows.append([rel[i], rep, o, part])
            nsplit += 1
            viol_h += len(OL.check_split(FILES[ds], roles, ("train", "ival", "own"), ("hold",), KEYS))
            viol_c += len(OL.check_split(FILES[ds], roles, ("train",), ("own", "ival"), KEYS))
            lk = OL.leak_report(FILES[ds], roles, ("train",), ("own",))
            leak_any.append(lk.get("same_slide_patient_source", 0.0))
    wcsv(PKG / "splits" / f"{ds}_outer_folds.csv", ["relpath", "rep", "holdout_fold"], fold_rows)
    wcsv(PKG / "splits" / f"{ds}_controlled_{SPLIT_EXP[ds]}_seed0.csv", ["relpath", "rep", "outer", "role"], long_rows)
    SPL[ds] = dict(splits=nsplit, holdout_violations=viol_h, controlled_violations=viol_c,
                   max_same_group_share=float(max(leak_any)) if leak_any else None)
    rows.append([ds, sch["kind"], nsplit, viol_h, viol_c, f"{100 * max(leak_any):.1f}" if leak_any else "—"])
Q.table(["dataset", "outer scheme", "splits", "holdout violations", "train/own violations", "max own share with group-mate %"],
        rows, [12, 13, 8, 19, 21, 30])
print("  checked relations: identical file (MD5), near duplicate, slide/patient/source group, augmented copy of the same")
print("  source. Every count must be 0; the pipeline's own assertions (build_job check=True) also passed for every split.")

# splits of the arms grouped by recovered clusters (amendment A1): the holdout must be clean; the arm's own fold is
# allowed to share true groups with training (that residual share is what A1 measures), so it is reported, not tested
RSPL, rows = {}, []
for j in jobs:
    s_ = j["spec"]
    if not (s_["exp"] in ("X1", "X2") and s_["arm"].startswith("R") and s_["seed"] == 0 and s_["ds"] in REL):
        continue
    ds = s_["ds"]
    B = Q.build_job(dict(s_, xeval=False), check=True)
    roles = {}
    out_rows = []
    for part in ("train", "ival", "own", "hold"):
        for i in np.unique(B[part]["row"]):
            roles[REL[ds][i]] = part
            out_rows.append([REL[ds][i], s_.get("rep", 0), s_["outer"], part])
    key = f"{ds}|{s_['arm']}"
    a = RSPL.setdefault(key, dict(rows=[], viol_h=0, share=[]))
    a["rows"] += out_rows
    a["viol_h"] += len(OL.check_split(FILES[ds], roles, ("train", "ival", "own"), ("hold",), KEYS))
    a["share"].append(OL.leak_report(FILES[ds], roles, ("train",), ("own",)).get("same_slide_patient_source", 0.0))
for key, a in sorted(RSPL.items()):
    ds, arm = key.split("|")
    exp = "X1" if ds == "orchid" else "X2"
    wcsv(PKG / "splits" / f"{ds}_recovered_{arm}_{exp}_seed0.csv", ["relpath", "rep", "outer", "role"], a["rows"])
    SPL[key] = dict(holdout_violations=a["viol_h"], mean_own_share_true_group=float(np.mean(a["share"])))
    rows.append([ds, arm, len(a["share"]), a["viol_h"], f"{100 * np.mean(a['share']):.1f}"])
if rows:
    Q.table(["dataset", "recovered arm", "splits", "holdout violations", "own share with true group-mate %"], rows,
            [12, 14, 8, 19, 33])
(PKG / "splits" / "README.md").write_text("\n".join([
    "# Fixed leakage-controlled splits", "",
    "No images are included. Paths are relative to the Kaggle input root; download each dataset from its original source.", "",
    "- `<dataset>_files.csv`: one row per file: class, MD5, SHA-256, and the group labels used by the experiments",
    "  (`g_exact` identical files, `g_near` near duplicates [pHash <= 8 and SSIM >= 0.90, or identical MD5],",
    "  `g_struct` slide / patient / recovered source group, `parent_group` publisher-augmented copy -> source image,",
    "  `g_ctrl` union of all relations). `used_in_experiments` = readable original that represents its identical-file group.",
    "- `<dataset>_outer_folds.csv`: the clean-holdout fold of each used image (`rep` = repetition of the outer split).",
    "- `<dataset>_controlled_E1_seed0.csv` (ORCHID: `orchid_controlled_X1_seed0.csv`): for every outer fold, the role of",
    "  each image in the leakage-controlled arm:",
    "  `train`, `ival` (inner validation: model selection, early stopping, temperature, thresholds), `own` (evaluation fold",
    "  of the arm) and `hold` (clean holdout). Images that are not listed for a fold were purged (relatives of holdout",
    "  images) or removed by the per-class size matching.",
    "- `<dataset>_recovered_<arm>_<exp>_seed0.csv` (amendment A1): the same roles for the arms grouped by clusters",
    "  recovered from image content alone (`g_recov_*` columns of `<dataset>_files.csv`; k25, k100 or the automatic k).",
    "  Their holdout is clean; their own fold may share true patients/slides with training, which is what A1 measures.",
    "- ORCHID paths are member paths inside the Zenodo archive (`test.zip/...`, record 12646943); SHA-256 values are",
    "  computed from the downloaded bytes.",
    "- Check any split with: `python -m oralleak check --files <dataset>_files.csv --split <split.csv>`.", ""]))

# ------------------------------------------------------------------ 5  runs, weights, environment
Q.sub("5  RUN MANIFEST, WEIGHTS MANIFEST, ENVIRONMENT")
mrows, wrows = [], []
codes, vers = Counter(), Counter()
overlap_bad, n_checked = 0, 0
for j in jobs:
    s = j["spec"]
    rp = Q.D["runs"] / j["id"] / "result.json"
    r = Q.jload(rp) if rp.exists() else None
    st_ = "done" if (r and r.get("status") == "done") else ("failed" if Q.fail_count(j["id"]) else "not run")
    if st_ == "done":
        codes[json.dumps(r.get("code", {}), sort_keys=True)] += 1
        vers[json.dumps(r.get("versions", {}), sort_keys=True)] += 1
        z = np.load(Q.D["runs"] / j["id"] / "preds.npz")
        iv = set(zip(z["ival_row"].tolist(), z["ival_copy"].tolist()))
        ev = set(zip(z["own_row"].tolist(), z["own_copy"].tolist())) | set(zip(z["hold_row"].tolist(), z["hold_copy"].tolist()))
        overlap_bad += int(bool(iv & ev))
        n_checked += 1
    mrows.append([j["id"], j["phase"], s["exp"], s["ds"], s["arm"], s["backbone"], s["recipe"], s["outer"], s.get("rep", 0),
                  s["seed"], s.get("dose"), st_, *(([f"{r['own_bal_acc']:.6f}", f"{r['hold_bal_acc']:.6f}", r["steps_done"],
                  r["best_step"], f"{r['T_ival']:.4f}", r["info"]["sizes"]["train"], r["info"]["sizes"]["ival"],
                  r["info"]["sizes"]["own"], r["info"]["sizes"]["hold"], r.get("img"), f"{r['time_s']:.0f}",
                  r.get("finished_utc"), (r.get("code") or {}).get("q1lib.py"), (r.get("code") or {}).get("q1worker.py")])
                  if st_ == "done" else [""] * 14), j["fp"]])
    wp = Q.D["runs"] / j["id"] / "weights.pt"
    if wp.exists():
        wrows.append([j["id"], f"runs/{j['id']}/weights.pt", f"{wp.stat().st_size / 1e6:.1f}", Q.sha256_file(wp)])
wcsv(PKG / "manifests" / "runs_manifest.csv",
     ["job_id", "phase", "exp", "dataset", "arm", "backbone", "recipe", "outer", "rep", "seed", "dose", "status", "own_bal_acc",
      "hold_bal_acc", "steps_done", "best_step", "temperature_ival", "n_train", "n_ival", "n_own", "n_hold", "img_px",
      "seconds", "finished_utc", "q1lib_hash", "q1worker_hash", "job_fingerprint"], mrows)
wcsv(PKG / "manifests" / "weights_manifest.csv", ["job_id", "file", "MB", "sha256"], wrows)
env = dict(python=platform.python_version(), platform=platform.platform(), run_versions=[json.loads(k) for k in vers],
           run_code_hashes=[json.loads(k) for k in codes], plan_sha256=(Q.jload(Q.D["plan"] / "plan.sha256.json") or {}).get("sha256"),
           lib_version=Q.LIB_VERSION)
try:
    import torch, timm
    env.update(torch=torch.__version__, timm=timm.__version__, cuda=torch.version.cuda,
               gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception:
    pass
Q.jsave(PKG / "manifests" / "environment.json", env)
st_count = Counter(m[11] for m in mrows)
Q.kv("jobs", f"{len(jobs)} · done {st_count.get('done', 0)} · failed {st_count.get('failed', 0)} · not run {st_count.get('not run', 0)}")
Q.kv("code versions among runs", f"{len(codes)} distinct (q1lib/q1worker hashes)" + ("" if len(codes) <= 1 else
     " — runs were trained with different code; see runs_manifest.csv"))
Q.kv("inner validation vs evaluation", f"{n_checked} runs checked · {overlap_bad} with an image unit in both (must be 0)")
Q.kv("weights", f"{len(wrows)} files listed in manifests/weights_manifest.csv" + (" (included in the zip)" if INCLUDE_WEIGHTS
     else " (not zipped; set INCLUDE_WEIGHTS = True to include them)"))

# ------------------------------------------------------------------ (copies) results, plan, code
for src_dir, dst in ((Q.D["eval"], "results/eval"), (Q.D["tables"], "results/tables"), (Q.D["figs"], "results/figures"),
                     (Q.D["plan"], "plan"), (Q.D["code"], "code")):
    if src_dir.exists():
        for f in sorted(src_dir.iterdir()):
            if f.is_file():
                shutil.copy2(f, PKG / dst / f.name)
for f in ("q1lib.py", "q1worker.py"):
    if (Q.D["lib"] / f).exists():
        shutil.copy2(Q.D["lib"] / f, PKG / "code" / f)
shutil.copy2(Q.D["index"] / "audit.json", PKG / "manifests" / "audit.json") if (Q.D["index"] / "audit.json").exists() else None
blocks_kept = sorted(p.name for p in (PKG / "code").glob("block*.py"))

# ------------------------------------------------------------------ (copies) leaderboard baseline on the controlled splits
E1 = R.get("E1", {})
lb = ["# Baseline on the fixed leakage-controlled splits", "",
      f"Model: {Q.BACKBONES[BB0][0]} ({BB0}), {P['img']} px, recipe 'std'. Balanced accuracy (%), mean over outer folds and "
      "seeds. 'Own fold' = the controlled arm's evaluation fold; 'clean holdout' = images whose slide / patient / source "
      "never entered development. Numbers are copied from results/eval/q1_results.json.", "",
      "| Dataset | Outer scheme | Runs | Own fold | Clean holdout | Inflation [95% CI] |", "|---|---|---|---|---|---|"]
for ds in ALL:
    a = E1.get(ds, {}).get("arms", {}).get("C") if ds in avail else (A1R.get("orchid") or {}).get("C")
    if a:
        lb.append(f"| {Q.SHORT.get(ds, ds)} | {P['outer'][ds]['kind']} | {a['n_runs']} | {100 * a['own']:.2f} | {100 * a['hold']:.2f} | "
                  f"{100 * a['infl']:+.2f} [{100 * a['infl_ci'][0]:+.2f}, {100 * a['infl_ci'][1]:+.2f}] |")
(PKG / "LEADERBOARD.md").write_text("\n".join(lb) + "\n")

# ------------------------------------------------------------------ (copies) README, citation and licence placeholders
TODO = "[TO BE COMPLETED BY THE AUTHORS]"
(PKG / "README.md").write_text("\n".join([
    "# Leakage audit of public oral-cancer image datasets: toolkit, fixed splits and results", "",
    f"Package generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 28 of the analysis notebook.", "",
    f"Repository: {TODO}  ·  Archive DOI: {TODO} (no DOI has been minted)  ·  Code licence: {TODO}", "",
    "## Contents", "",
    "- `toolkit/` — `oralleak`, a stand-alone audit package (file hashing, pHash/dHash, SSIM-confirmed near duplicates,",
    "  source recovery from file order, metadata-free source recovery from image content, grouped splits, cross-partition",
    "  checks, provenance leakage report) with pytest tests.",
    "- `splits/` — per-file manifests with MD5/SHA-256 and group labels, clean-holdout folds and the fixed",
    "  leakage-controlled splits (no images; see `splits/README.md`).",
    "- `manifests/` — run manifest (every job, its split sizes, results and code hash), weights manifest, data audit,",
    "  environment.",
    "- `results/` — every table, figure and JSON file produced by BLOCKS 25-27 and, for amendment A1, BLOCKS 31 and 33",
    "  (`a1_results.json`, `recovery.json`, `RESULTS_A1.md`, `Fig_A1_*.png`).",
    "- `plan/` — the analysis plan with its local timestamp and SHA-256 (written before any training run) and amendment",
    "  A1 with its own timestamp and SHA-256 (written before any A1 run).",
    "- `code/` — the pipeline library, the training worker and the source of each notebook block that was run.",
    "- `LEADERBOARD.md` — baseline results on the controlled splits. `FINAL_STATUS_compute.md` — what was and was not done.", "",
    "## Reproducing", "",
    "1. Attach the datasets listed in `splits/*_files.csv` (paths are relative to the Kaggle input root).",
    "2. Run the notebook blocks in the order given in `FINAL_STATUS_compute.md`.",
    "3. Check a split: `cd toolkit && pip install . && python -m oralleak check --files ../splits/<ds>_files.csv "
    "--split <split.csv>`; run the tests with `python -m pytest -q tests`.", "",
    "## Data", "",
    "Images are not redistributed. Each dataset keeps its own licence and terms of use, which the authors must confirm:",
    ""] + [f"- {Q.SHORT.get(ds, ds)}: licence {TODO}" for ds in ALL] + [""]))
(PKG / "CITATION.cff").write_text("\n".join([
    "cff-version: 1.2.0", "message: If you use this toolkit or these splits, please cite the article.",
    f'title: "{TODO}"', "authors:", f'  - name: "{TODO}"', f'doi: "{TODO}"', f'repository-code: "{TODO}"', "version: 0.1.0",
    f"date-released: {time.strftime('%Y-%m-%d', time.gmtime())}", ""]))
(PKG / "LICENSE_NOTE.md").write_text("\n".join([
    "# Licence", "", f"Code licence: {TODO} (choose one, e.g. MIT or Apache-2.0, and replace this file with its text).",
    "Split files contain only relative paths, hashes and group labels derived from the public datasets; check that each",
    "dataset's terms allow publishing such derived metadata.", ""]))

# ------------------------------------------------------------------ 6  FINAL_STATUS_compute.md
Q.sub("6  DEFINITION-OF-DONE CHECKLIST (compute side) — also written to FINAL_STATUS_compute.md")
done_ph = Counter(j["phase"] for j in jobs if Q.job_done(j))
all_ph = Counter(j["phase"] for j in jobs)


def seeds_done(ds, arm):
    c = Counter((j["spec"]["outer"], j["spec"].get("rep", 0)) for j in jobs if Q.job_done(j) and j["spec"]["exp"] == "E1"
                and j["spec"]["ds"] == ds and j["spec"]["arm"] == arm and j["spec"]["backbone"] == BB0 and j["spec"]["recipe"] == "std")
    return min(c.values()) if c else 0


bb_done = sorted({j["spec"]["backbone"] for j in jobs if Q.job_done(j) and j["spec"]["exp"] == "E1" and j["spec"]["recipe"] == "std"})
fam = {"convnext_tiny.fb_in22k_ft_in1k": "CNN", "resnet50.a1_in1k": "CNN", "efficientnet_b0.ra_in1k": "small CNN",
       "convnext_base.fb_in22k_ft_in1k": "CNN (large)", "vit_small_patch16_224.augreg_in21k_ft_in1k": "transformer"}
fams = sorted({fam.get(b, b) for b in bb_done})
probes_ok = bool(Q.jload(Q.D["eval"] / "probes.json", []))
min_seeds = min([seeds_done(ds, arm) for ds in avail for arm in ("L", "C")] or [0])
dose = R.get("dose", {})
dose_ds = sorted({k.split("|")[1] for k in dose})
reimp = R.get("reimp", {})
reimp_both = [k for k, v in reimp.items() if len(v) >= 2]
strata = R.get("provenance_strata", {})
strata_contrast = [d for d, v in strata.items() if "sibling_minus_none" in v]
h1 = {ds: E1[ds]["arms"]["L"].get("infl_p_holm") for ds in E1 if "L" in E1[ds].get("arms", {})}
plan_meta = Q.jload(Q.D["plan"] / "plan.sha256.json", {}) or {}
first_run = min([Q.jload(Q.D["runs"] / j["id"] / "result.json", {}).get("finished_utc", "9999") for j in jobs if Q.job_done(j)]
                or ["—"])
tests_file = PKG / "toolkit" / "tests" / "test_oralleak.py"
a1_rec = next((a for a in P.get("amendments", []) if a["id"] == "A1"), None)
a1_jobs = (Q.jload(Q.D["jobs"] / "jobs_A1.json", {}) or {}).get("jobs", [])
a1_first = min([Q.jload(Q.D["runs"] / j["id"] / "result.json", {}).get("finished_utc", "9999") for j in a1_jobs if Q.job_done(j)]
               or ["—"])
if a1_rec is None:
    a1_status, a1_evidence = "NOT RUN", "BLOCK 29 not run"
else:
    a1_status = "DONE (local timestamp)" if a1_first == "—" or a1_rec["applied_utc"] < a1_first else "CHECK"
    a1_evidence = (f"plan/amendment_A1.json written {a1_rec['applied_utc']} (SHA-256 {a1_rec['sha256'][:12]}); first finished "
                   f"A1 run {a1_first}; a public timestamp is human-only")
_q = (RCV.get("quality") or {})
_sel = RCV.get("selected")
_h8 = [(k.split("|")[0], v) for k, v in _q.items() if k.endswith(f"|{_sel}") and v.get("quality")]
h8_txt = ("; ".join(f"{d}: AMI {v['quality'][str(v['kauto'])]['ami']:.2f} at k={v['kauto']}" for d, v in _h8)
          + f" (features {_sel})") if _h8 else ""
h9 = A1R.get("H9")


def item(name, status, evidence):
    return [name, status, evidence]


ITEMS = [
    item("Tier 0 errors fixed (213->313, Table 13, comparison counts, '31 studies', placeholders)", "WRITING PHASE",
         "new operating points: results/tables/operating_points.csv; text errata are fixed in the manuscript"),
    item("Selection and early stopping never use evaluation data",
         "DONE" if n_checked and overlap_bad == 0 else ("CHECK" if n_checked else "NOT RUN"),
         f"inner validation carved from training (plan.selection_rules); {n_checked} runs checked, {overlap_bad} overlaps"),
    item("Temperature scaling is fitted on disjoint data", "DONE" if R else "NOT RUN",
         "temperature from inner validation only (BLOCK 25 run_metrics, BLOCK 26); ECE/Brier on own fold and clean holdout"),
    item("Sensitivity and specificity defined consistently, Table 12 rebuilt",
         "DONE (numbers)" if (Q.D["tables"] / "published_vs_ours.csv").exists() else "NOT RUN",
         "positive-class columns separate from macro: results/tables/published_vs_ours.csv, E1_main.csv"),
    item("One evaluation mode per paired comparison", "DONE" if R else "NOT RUN",
         "every comparison = single models, own fold vs clean holdout of the same models; ensembles only in BLOCK 26, labelled"),
    item("Cluster-level CIs and tests, with Holm correction", "DONE" if h1 else "NOT RUN",
         "two-stage cluster bootstrap + within-fold bootstrap + mixed model + GEE; Holm over H1 (q1_results.json E1.*.arms.L)"),
    item("'Pre-specified' claims backed by a timestamp", "PARTIAL" if plan_meta else "NOT RUN",
         f"plan written {plan_meta.get('created_utc', '?')} (SHA-256 {str(plan_meta.get('sha256', '?'))[:12]}) before the "
         f"first finished run ({first_run}); a public timestamp (saved Kaggle version, Git tag or OSF) is human-only"),
    item("Matched-training control and, where possible, a factorial design",
         "DONE" if "factorial_clinical" in R else ("PARTIAL" if E1 else "NOT RUN"),
         "L and C matched per class and in steps (jobs.json Tc); factorial S x Dup on clinical: "
         + ("present" if "factorial_clinical" in R else "general phase not finished")),
    item("Provenance-stratified analysis (sibling vs no sibling)", "DONE" if strata else "NOT RUN",
         f"results/tables/provenance_strata.csv; sibling-vs-none contrast estimable for: {', '.join(strata_contrast) or 'none'}"
         + ("" if strata_contrast else " (no leaky own fold has both strata: under augment-before-split nearly every test image "
            "has a sibling in training, so the clean holdout carries the comparison)")),
    item("Dose-response curves on at least two datasets", "DONE" if len(dose_ds) >= 2 else ("PARTIAL" if dose_ds else "NOT RUN"),
         f"datasets with dose curves: {', '.join(dose_ds) or 'none'} (DOSEK / DUP / OVL, q1_results.json dose)"),
    item("Perceptual and embedding near-duplicate audit, with positive and negative controls",
         "DONE" if AU and "RL" in R and any(k.startswith("DUP|") for k in dose) else ("PARTIAL" if AU else "NOT RUN"),
         "index/audit.json (MD5, pHash, dHash, SSIM, embedding cosine, cross-dataset); positive control DUP; negative controls: "
         "augment-after-split arm C and random labels (RL)"),
    item("At least 3 seeds and at least 3 architecture families",
         "DONE" if min_seeds >= 3 and "transformer" in fams and probes_ok else ("PARTIAL" if min_seeds or bb_done else "NOT RUN"),
         f"minimum finished seeds per E1 fold: {min_seeds}; fine-tuned families: {', '.join(fams) or 'none'}; "
         f"frozen DINOv2 linear probe: {'yes' if probes_ok else 'no'}"),
    item("At least 2 additional datasets, one with patient IDs",
         "DONE" if "breakhis" in avail and "orchid" in ALL else ("PARTIAL" if "breakhis" in avail or "orchid" in ALL else "BLOCKED"),
         ", ".join(x for x in ["BreakHis (non-oral, patient IDs)" if "breakhis" in avail else "",
                               "ORCHID (oral H&E, patient IDs; amendment A1)" if "orchid" in ALL else ""] if x) or "none attached"),
    item("Amendment A1 written and hashed before any A1 run", a1_status, a1_evidence),
    item("Metadata-free group recovery validated against true patients/slides (H8)", "DONE" if h8_txt else "NOT RUN",
         h8_txt or "run BLOCK 31"),
    item("Oral dataset with patient IDs: image-level vs patient-disjoint evaluation (H9)", "DONE" if h9 else "NOT RUN",
         (f"ORCHID inflation {100 * h9['infl']:+.2f} [{100 * h9['ci'][0]:+.2f}, {100 * h9['ci'][1]:+.2f}] points, p = {h9['p']:.3g}"
          if h9 else "run BLOCKS 30-33")),
    item("Pathology foundation model (Phikon) frozen-feature check (H12)",
         "DONE" if any("|phikon|" in k for k in (A1R.get("H12") or {})) else ("PARTIAL (DINOv2 only)" if A1R.get("H12") else "NOT RUN"),
         "results/eval/a1_results.json H12; frozen probes on the E1 seed-0 splits"),
    item("At least 2 published methods reimplemented under both protocols",
         "DONE" if len(reimp_both) >= 2 else ("PARTIAL" if reimp else "NOT RUN"), f"{', '.join(reimp_both) or 'none finished'}"),
    item("Literature analysis of every surveyed study, with the true study count", "WRITING PHASE", "not a compute task"),
    item("Audit toolkit, corrected splits, manifests and weights released with a DOI", "PARTIAL",
         "toolkit, splits and manifests in this zip; deposit and DOI are human-only; weights listed, not zipped by default"),
    item("Novelty claims verified against prior leakage literature", "WRITING PHASE", "not a compute task"),
    item("Abstract and conclusion rewritten to match the evidence", "WRITING PHASE", "uses results/ from this package"),
    item("CLAIM checklist completed and dataset licences checked", "WRITING PHASE / HUMAN", "licence terms are human-only"),
]
Q.table(["item", "status", "evidence"], [[i[0][:56], i[1], i[2][:56]] for i in ITEMS], [58, 23, 58])

# honest scale-down and surprises (generated from the numbers, not written by hand)
scale = [P.get("deviations", "")]
for ph in P["phases"]:
    if all_ph.get(ph) and done_ph.get(ph, 0) < all_ph[ph]:
        scale.append(f"phase '{ph}': {done_ph.get(ph, 0)} of {all_ph[ph]} jobs finished; the missing runs are reported as missing.")
if "histopath" in avail and not AU.get("histopath", {}).get("pre_enabled", False):
    scale.append("PRE experiment disabled: parent recovery of the publisher-augmented files was below the pre-set rate.")
if "breakhis" not in avail:
    scale.append("BreakHis not attached: no dataset with patient identifiers.")
surprises = []
for ds, p in h1.items():
    if p is not None and p >= 0.05:
        surprises.append(f"H1 not supported for {ds}: leaky-arm inflation {100 * E1[ds]['arms']['L']['infl']:+.2f} points, Holm p = {p:.3g}.")
for ds in E1:
    c = E1[ds]["arms"].get("C")
    if c and (c["infl_ci"][0] > 0 or c["infl_ci"][1] < 0):
        surprises.append(f"H3 fails for {ds}: the controlled arm is itself biased (inflation {100 * c['infl']:+.2f} "
                         f"[{100 * c['infl_ci'][0]:+.2f}, {100 * c['infl_ci'][1]:+.2f}]).")
    if E1[ds].get("dhold_ci") and (E1[ds]["dhold_ci"][0] > 0 or E1[ds]["dhold_ci"][1] < 0):
        surprises.append(f"H4 for {ds}: clean-holdout BA differs between leaky and controlled training "
                         f"({100 * E1[ds]['dhold']:+.2f} [{100 * E1[ds]['dhold_ci'][0]:+.2f}, {100 * E1[ds]['dhold_ci'][1]:+.2f}]).")
for k, v in R.get("RL", {}).items():
    if k.endswith("|L") and (v.get("p_vs_chance") or 1) >= 0.05:
        surprises.append(f"random labels, {k.split('|')[0]}: the leaky arm is not above chance on its own fold (p = {v['p_vs_chance']:.3g}).")
    if k.endswith("|C") and (v.get("p_vs_chance") or 1) < 0.05:
        surprises.append(f"random labels, {k.split('|')[0]}: the controlled arm is above chance on its own fold "
                         f"(own {100 * v['own']:.2f}, holdout {100 * v['hold']:.2f}, true-class ceiling "
                         f"{100 * v.get('true_class_ceiling', float('nan')):.2f}, chance p = {v['p_vs_chance']:.3g}).")
for k, v in dose.items():
    exn, _, arm_ = k.split("|")
    if v.get("slope") is None:
        continue
    if exn == "DOSEK" and arm_ == "C":        # augmenting after the split should NOT raise inflation
        if v["slope"] > 0 and (v.get("p") or 1) < 0.05:
            surprises.append(f"dose-response {k}: inflation rises with k even though augmentation happens after the split "
                             f"(slope {100 * v['slope']:+.2f}, p = {v['p']:.4f}).")
        continue
    if v["slope"] <= 0 or (v.get("p") or 1) >= 0.05:
        surprises.append(f"dose-response {k}: no clear increase (slope {100 * v['slope']:+.2f}, p = {v.get('p') if v.get('p') is None else round(v['p'], 4)}).")
rp = R.get("risk_predictor", {})
for name, v in rp.items():
    if isinstance(v, dict) and "lodo_r2" in v and name.startswith("one") and v["lodo_r2"] <= 0:
        surprises.append(f"leakage-risk predictor: leave-one-dataset-out R² = {v['lodo_r2']:.2f} (no predictive value across datasets).")
nw = [k for k, v in CL.get("saliency", {}).items() if not (v["del_gain_ci"][0] > 0 and v["ins_gain_ci"][0] > 0)]
if nw:
    surprises.append(f"Grad-CAM not better than random order for {', '.join(nw)}: those maps are illustration only.")
worse = [k for k, v in CL.get("calibration", {}).items() if v["cal"]["ece"] > v["raw"]["ece"]]
if worse:
    surprises.append(f"temperature scaling (fitted on inner validation) increased holdout ECE for {', '.join(worse)}.")
for ds, a in AGREE.items():
    if a.get("partition_identical") is False:
        surprises.append(f"toolkit and pipeline near-duplicate groups differ for {ds}.")
if h9 and not h9.get("supported"):
    surprises.append(f"H9 not supported on ORCHID: inflation {100 * h9['infl']:+.2f} points, p = {h9['p']:.3g}.")
for ds, v in (A1R.get("recovered_arms") or {}).items():
    ra = v.get("arms", {}).get("Rauto")
    if ra and ds in ("breakhis", "he_stained", "orchid") and (ra["infl_ci"][0] > 0 or ra["infl_ci"][1] < 0):
        surprises.append(f"H10 fails for {ds}: the split on recovered groups (automatic k) is still biased (inflation "
                         f"{100 * ra['infl']:+.2f} [{100 * ra['infl_ci'][0]:+.2f}, {100 * ra['infl_ci'][1]:+.2f}]).")
for ds, v in (A1R.get("H11") or {}).items():
    rho = v.get("spearman_size_vs_inflation")
    if rho is not None and np.isfinite(rho) and rho >= 0:
        surprises.append(f"H11 for {ds}: probe inflation does not fall as recovered clusters get coarser (Spearman {rho:+.2f}).")
for k, v in (A1R.get("random_labels") or {}).items():
    exn, ds, arm = k.split("|")
    if arm in ("C", "Rauto") and (v.get("p_vs_chance") or 1) < 0.05:
        surprises.append(f"random labels {exn} {ds} arm {arm}: own fold above chance (own {100 * v['own']:.2f}, holdout "
                         f"{100 * v['hold']:.2f}, true-class ceiling {100 * v['ceiling']:.2f}, p = {v['p_vs_chance']:.3g}).")
if A1R and A1R.get("q1_unchanged") is False:
    surprises.append("Q1 results changed after amendment A1 (see BLOCK 33 section 8) - investigate before writing.")
for ds, a in REC_AGREE.items():
    if not a["partition_identical"]:
        surprises.append(f"toolkit recover_groups and the pipeline give different recovered groups for {ds}.")
human = ["repository URL", "archive DOI (Zenodo or similar) — none minted", "code licence", "licence terms of every dataset",
         "public timestamp of the analysis plan (saved Kaggle version, Git tag or OSF registration)", "funding statement",
         "author contributions", "ethics / data-provenance statement sign-off", "citation metadata (CITATION.cff)"]
order = ("First session: 15 -> Save Version -> 16 -> 17 -> 18 -> 19 -> 20 -> 21 -> 22.  Later sessions: 15 -> 19 -> next "
         "training block (23, then 24; re-run a block to resume it).  Final session: 15 -> 25 -> 26 -> 27 -> 28.  "
         "Amendment A1: 15 -> 19 -> 29 -> Save Version -> 30 -> 31 -> 32 -> 25 -> 27 -> 33 -> 28 (BLOCK 18 is never re-run; "
         "if a session ends, continue with 15 -> 19 -> 32 and the remaining blocks).")
md = ["# FINAL_STATUS (computational part)", "",
      f"Generated {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())} by BLOCK 28 from the files in q1/. "
      f"Plan SHA-256 `{plan_meta.get('sha256', '?')}`. Jobs: {len(jobs)}, finished {st_count.get('done', 0)}, "
      f"failed {st_count.get('failed', 0)}, not run {st_count.get('not run', 0)}.", "",
      "## Definition of done", "", "| Item | Status | Evidence |", "|---|---|---|"]
md += [f"| {a} | {b} | {c} |" for a, b, c in ITEMS]
md += ["", "## Jobs by phase", "", "| Phase | Finished | Total |", "|---|---|---|"]
md += [f"| {ph} | {done_ph.get(ph, 0)} | {all_ph.get(ph, 0)} |" for ph in P["phases"] if all_ph.get(ph)]
md += ["", "## Scaled down or skipped", ""] + [f"- {x}" for x in scale if x]
md += ["", "## Results that go against the original thesis or the plan's expectations", ""]
md += [f"- {x}" for x in surprises] or ["- none detected automatically; read BLOCK 25-26 outputs before writing"]
md += ["", "## Toolkit agreement with the pipeline", ""]
md += [f"- {ds}: pHash equal {100 * a['phash_equal']:.2f}%, near-duplicate partition "
       f"{'identical' if a.get('partition_identical') else ('different' if a.get('partition_identical') is False else 'not re-derived (size)')}"
       + (f", recovered groups {'identical' if REC_AGREE[ds]['partition_identical'] else 'DIFFERENT'} (k={REC_AGREE[ds]['k_toolkit']})"
          if ds in REC_AGREE else "")
       for ds, a in AGREE.items()]
md += ["", "## Human-only items left open", ""] + [f"- {x}" for x in human]
md += ["", "## Block order used", "", order, "", f"Blocks whose source is stored in code/: {', '.join(blocks_kept) or 'none'} "
       "(a block's source is stored when it runs; re-run a block to store it)."]
(PKG / "FINAL_STATUS_compute.md").write_text("\n".join(md) + "\n")
if surprises:
    print("  results against expectations (all listed in FINAL_STATUS_compute.md):")
    for x in surprises[:12]:
        print("   -", x)
    if len(surprises) > 12:
        print(f"   ... and {len(surprises) - 12} more")

# ------------------------------------------------------------------ 7  toolkit tests on the packaged splits
Q.sub("7  TOOLKIT TESTS  (pytest on the package, including every released split)")
try:
    import pytest  # noqa
    pt = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"], cwd=PKG / "toolkit",
                        capture_output=True, text=True, timeout=1800)
    last = [l for l in pt.stdout.strip().splitlines() if l.strip()][-1:] or ["(no output)"]
    Q.kv("pytest", last[0] + ("" if pt.returncode == 0 else "  <- FAILED, see release/pytest.log"))
    (Q.D["release"] / "pytest.log").write_text(pt.stdout + pt.stderr)
    TEST_OK = pt.returncode == 0
except ImportError:
    Q.kv("pytest", "not installed; the split checks in section 4 are the test result")
    TEST_OK = all(v["holdout_violations"] == 0 and v["controlled_violations"] == 0 for v in SPL.values())

# ------------------------------------------------------------------ 8  zip
Q.sub("8  ZIP")
zp = Q.D["release"] / f"oralleak_release_{stamp}.zip"
n_files = 0
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(PKG.rglob("*")):
        if f.is_file() and "__pycache__" not in f.parts:
            zf.write(f, f"oralleak_release/{f.relative_to(PKG)}")
            n_files += 1
    if INCLUDE_WEIGHTS:
        for w in wrows:
            zf.write(Q.D["runs"] / w[0] / "weights.pt", f"oralleak_release/weights/{w[0]}.pt")
            n_files += 1
Q.kv("archive", f"{zp.relative_to(Q1_ROOT.parent)} · {n_files} files · {zp.stat().st_size / 1e6:.1f} MB")
Q.kv("status file", "release/package/FINAL_STATUS_compute.md")
Q.kv("nothing uploaded", "the zip stays in /kaggle/working; publishing and a DOI are left to the authors")
print(f"\n[BLOCK 28 ✓]  release package built · toolkit tests {'passed' if TEST_OK else 'FAILED'} · {Q.hms(time.time() - t_all)}")

In [ ]:
import os
from pathlib import Path

# Search locations
search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/kaggle/temp")
]

print("=" * 80)
print("FILE LOCATIONS")
print("=" * 80)

# 1. Find per-run prediction files
print("\n[1] PER-RUN PREDICTION FILES (preds.npz)")
print("-" * 80)

prediction_files = []

for root in search_roots:
    if root.exists():
        prediction_files.extend(root.glob("**/q1/runs/*/preds.npz"))

prediction_files = sorted(set(prediction_files))

if prediction_files:
    for file in prediction_files:
        print(f"FOUND: {file}")
        print(f"SIZE:  {file.stat().st_size / (1024**2):.2f} MB")
else:
    print("NOT FOUND: No q1/runs/*/preds.npz files detected.")

# 2. Find Zenodo archive
print("\n[2] ZENODO ARCHIVE")
print("-" * 80)

zenodo_files = []

archive_extensions = {".zip", ".tar", ".gz", ".tgz", ".7z"}

for root in search_roots:
    if root.exists():
        for file in root.rglob("*"):
            if file.is_file():
                name = file.name.lower()
                if "zenodo" in name or (
                    file.suffix.lower() in archive_extensions
                    and "zenodo" in str(file.parent).lower()
                ):
                    zenodo_files.append(file)

zenodo_files = sorted(set(zenodo_files))

if zenodo_files:
    for file in zenodo_files:
        print(f"FOUND: {file}")
        print(f"SIZE:  {file.stat().st_size / (1024**2):.2f} MB")
else:
    print("NOT FOUND: No Zenodo archive detected.")

# Summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"Prediction files found: {len(prediction_files)}")
print(f"Zenodo archives found:  {len(zenodo_files)}")